# WIMB: train your own model on a free Colab GPU

**How to run it**
1. Check the top right says **T4**. If it doesn't: **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Runtime → Run all**.
3. In the first minute Google asks to connect **Google Drive**. Click **Connect** and **Allow**
   (the trained model is saved there).
4. Keep this tab open. It takes about 30–40 minutes.
5. When it's done: **File → Save a copy in GitHub** → repository `pratik9yadav/yok`,
   branch `main`, file path `train_colab.ipynb` → **OK**.
   Then tell Claude it's done. Claude reads the results from GitHub.

**What it does:** takes a free model (Qwen3 4B), tests it on shop calls, trains it on
the training set chosen in cell 3 (`TRAIN_SET`: v2 by default, or v4 or v1), tests it again, then tests the
saved Ollama file too. Held-out test sets, never trained on: 30 calls, 56 harder ones written in styles the
training data does not have, 60 written blind, and (when `data/asr_test.jsonl` is there) calls as speech-to-text
writes them. The model goes to Google Drive.

It also runs on Kaggle's free GPU: `run_on_kaggle.py` starts it there with no clicks.

In [ ]:
#@title 1. Check the GPU
import os, shutil, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle gives two GPUs; training uses one
# nvidia-smi is not always installed, so also ask torch (in its own process: nothing is imported here yet).
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.strip() == "True"
if not (cuda or shutil.which("nvidia-smi")):
    # What the machine has, to see why there is no GPU (no secret values printed).
    print("nvidia devices:", [d for d in os.listdir("/dev") if d.startswith("nvidia")])
    print("Kaggle settings:", {k: v for k, v in os.environ.items()
                               if k.startswith("KAGGLE") and any(w in k for w in ("ACCELERATOR", "GPU", "MACHINE", "RUN_TYPE"))})
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU > Save, then Run all. "
                       "Kaggle: verify your phone, then turn on GPU T4 once in the notebook editor (Settings > Accelerator).")
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
                     capture_output=True, text=True).stdout or "GPU found")

In [ ]:
#@title 2. Install the training tools (about 3–5 minutes)
!pip install -q unsloth

In [ ]:
#@title 3. Settings (and connect Google Drive now, so nothing waits for you later)
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
EPOCHS = 2  #@param {type:"integer"}
# v4: 6,000 rows written as speech-to-text writes them (Hindi, Hinglish and Marathi, mostly in Devanagari, with
#     speech-to-text errors), plus the 151 real Hindi answers. About 1 hour of training for 2 epochs.
# v2: training data built from real products, prices and spellings, plus 151 real Hindi answers.
# v1: the first 400 made-up examples (kept to compare against).
TRAIN_SET = "v2"  #@param ["v4", "v2", "v1"]
OUTPUT_NAME = "wimb-shop-model"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
MAKE_OLLAMA_FILE = True  #@param {type:"boolean"}
# This notebook is set up for Qwen3 instruct models (their chat format is used below).

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/tmp" if ON_KAGGLE else "/content"  # big temporary files (Kaggle saves all of /kaggle/working)
if ON_KAGGLE:
    SAVE_DIR = f"/kaggle/working/{OUTPUT_NAME}"  # shown in the notebook's Output tab
elif SAVE_TO_DRIVE:
    SAVE_DIR = f"/content/drive/MyDrive/{OUTPUT_NAME}"
    from google.colab import drive
    drive.mount("/content/drive")
else:
    SAVE_DIR = None

In [ ]:
#@title 4. Training data and scoring (hidden; built from data/ and score.py)
"""Shared by the notebook and the tests: the task's prompt, and how an answer is scored.

The task: read what a shopkeeper said on a phone call and return JSON with four fields.
"""

import json
import re
import time

SYSTEM = (
    "You read what a shopkeeper said on a phone call about a product. "
    "Reply with only a JSON object with these keys: "
    "in_stock (true, false or null), price_inr (a number or null), "
    "delivery (true, false or null), alternative (text or null). "
    "price_inr is the price of the product that was asked for. "
    "Use null when the shopkeeper did not say."
)

FIELDS = ("in_stock", "price_inr", "delivery", "alternative")


def user_message(product: str, said: str) -> str:
    return f"Product: {product}\nShopkeeper: {said}"


def answer_text(answer: dict) -> str:
    """The exact text the model is trained to reply with."""
    return json.dumps({k: answer[k] for k in FIELDS}, ensure_ascii=False)


def parse(text: str) -> dict | None:
    """The last JSON object in the reply that has exactly the four keys, or None.

    Every "{" is tried, so braces inside a value ("M-seal {2 pack}") or other text around the
    answer do not hide it.
    """
    decoder = json.JSONDecoder()
    found = None
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text, match.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and set(value) == set(FIELDS):
            found = value
    return found


def _tokens(text: str) -> set[str]:
    """Lower-case words, with "40mm" read as "40 mm" and punctuation dropped."""
    text = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", text.lower())
    return set(re.sub(r"[-_.,;:!?()\[\]{}\"'/]", " ", text).split())


def same(field: str, got, want) -> bool:
    if want is None or got is None:
        return got is None and want is None
    if field in ("in_stock", "delivery"):
        return isinstance(got, bool) and got == want  # 1 and 0 are not true and false
    if field == "price_inr":
        if isinstance(got, bool):
            return False
        try:
            return float(got) == float(want)
        except (TypeError, ValueError):
            return False
    # alternative: the wanted words must all be there, with at most two extra ("40 mm padlock").
    if not isinstance(got, str):
        return False
    have, need = _tokens(got), _tokens(want)
    return bool(have) and need <= have and len(have - need) <= 2


def score(replies: list[str], examples: list[dict]) -> dict:
    """Per-field accuracy, all-four-right accuracy, and the mistakes.

    A reply with no valid answer object (or a missing or extra key) gets every field wrong.
    """
    right = {f: 0 for f in FIELDS}
    all_right = json_ok = 0
    mistakes = []
    for reply, ex in zip(replies, examples, strict=True):
        got = parse(reply)
        if got is not None:
            json_ok += 1
        wrong = list(FIELDS) if got is None else [f for f in FIELDS if not same(f, got[f], ex["answer"][f])]
        for f in FIELDS:
            right[f] += f not in wrong
        if wrong:
            mistakes.append({"said": ex["said"], "want": ex["answer"], "reply": reply, "wrong": wrong})
        else:
            all_right += 1
    n = len(examples)
    return {
        "examples": n,
        "json_ok": json_ok / n,
        "all_correct": all_right / n,
        **{f: right[f] / n for f in FIELDS},
        "mistakes": mistakes,
    }


KEYS = ("all_correct", "json_ok", *FIELDS)


def evaluate(ask, sets: dict[str, list[dict]], deadline: float | None = None) -> dict:
    """Run ask(product, said) over every test set and score it. Every reply is kept.

    After `deadline` (a time.time() value) no more questions are asked; what was answered is
    scored, and "of" says how many there were.
    """
    out = {}
    for name, rows in sets.items():
        replies = []
        for ex in rows:
            if deadline is not None and time.time() > deadline:
                break
            replies.append(ask(ex["product"], ex["said"]))
        done = rows[: len(replies)]
        scored = score(replies, done) if done else {"examples": 0, "mistakes": []}
        out[name] = {**scored, "replies": replies, "of": len(rows)}
    return out


def show(label: str, results: dict, max_mistakes: int = 8) -> None:
    for name, res in results.items():
        n = res["examples"]
        if not n:
            print(f"== {label} [{name}]: no calls answered")
            continue
        partial = f" (only {n} of {res['of']} answered)" if n < res["of"] else ""
        print(f"== {label} [{name}]: all 4 fields right in {res['all_correct']:.0%} of {n} calls{partial}")
        print("   " + "   ".join(f"{k} {res[k]:.0%}" for k in KEYS[1:]))
        for m in res["mistakes"][:max_mistakes]:
            print("\n   said: ", m["said"])
            print("   want: ", answer_text(m["want"]))
            print("   reply:", m["reply"][:200])
            print("   wrong:", ", ".join(m["wrong"]))


def results_payload(model: str, epochs: int, before: dict, after: dict) -> dict:
    """What results.json holds: scores, mistakes and every raw reply, so scores can be recomputed."""
    sets = {}
    for name, a in after.items():
        b = before[name]
        sets[name] = {
            "examples": a["of"],
            "before": {k: round(b[k], 3) for k in KEYS},
            "after": {k: round(a[k], 3) for k in KEYS},
            "mistakes_before": b["mistakes"],
            "mistakes_after": a["mistakes"],
            "replies_before": b["replies"],
            "replies_after": a["replies"],
        }
    return {"model": model, "epochs": epochs, "sets": sets}


def make_modelfile(gguf_name: str) -> str:
    """An Ollama Modelfile whose template is exactly the text the model was trained on.

    Unsloth's own Qwen3 template differs a little (a blank line after the system message, and
    thinking-mode syntax that needs a newer Ollama), so it is not used.
    """
    return (
        f"FROM ./{gguf_name}\n"
        'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
        '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
        '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        "PARAMETER temperature 0\n"
        # Calls are short (training used 1,024 tokens). Ollama's default context (32,768 on 0.40) takes about
        # 7.5 GB of memory with this model; 2,048 needs about 3 GB, so it fits a laptop with 8 GB.
        "PARAMETER num_ctx 2048\n"
        f'SYSTEM """{SYSTEM}"""\n'
    )


TRAIN_V4 = [json.loads(line) for line in r'''
{"product": "silicone sealant", "said": "जी उपलब्ध है हाँ तो अभी रेट दो सौ तीस हो गया पिछले महीने दो सौ पाँच था कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "Stock madhe aahe, sutta savvashe la, panchvis nagancha box ghetla tar don hajar sahashe", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हेलो बेसिकली हाँ है एक मिनट नहीं वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "जी उपलब्ध है २७० लगेगा डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 270, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "sorry sir, abhi out of stock hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "मशीन नही है खत्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "nahi bhai, yahan nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "बेसिकली अभी बिज़ी हूँ, call बैक करता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉकमध्ये आहे एका नगाचे ४०, १० नगांचा बॉक्स ३४० ला", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "हाँ, माल पड़ा है, दो सौ पचानवे लगा देंगे, वैसे 360 का है.", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "मशीन आहे मिळेल चार हजार सातशे सांगितलं का चुकलो सव्वाएकोणचाळीसशे आहे पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 3925, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "सवा अट्ठाईस सौ रुपये हां रखा है", "answer": {"in_stock": true, "price_inr": 2825, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "चार सौ फ़ाइनल. स्टॉक में है", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "हाँ, रखा है, 305 का है, आपके लिए दो सौ पचास", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Haan haan, hai, ek ka 65, 25 ka dabba 1390 ka, address bhej do, bhej dete hain", "answer": {"in_stock": true, "price_inr": 65, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "ha sir, available hai, list price 190, discount ke bad 150, sham tak ghar pe aa jayega", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी संपली नाही, आहे, अं तुम्हाला सव्वाबाराशेला देऊ, 8 वाजता दुकान बंद बंद होतं, पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 1225, "delivery": false, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "हाँ हाँ, है, लिए छह सौ पैंसठ, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 665, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "haan sir boliye dekho haan ji mil jayega do sao pachasi mein ho jayega upar mrp teen sau paintees likha hai", "answer": {"in_stock": true, "price_inr": 285, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "हाँ, कवर है, दो सौ पैंतालीस लगेगा, हम्म घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 245, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "haan ji, geyser available hai, kya hai na pichle mahine 2010 tha, ab pachis sau pachas", "answer": {"in_stock": true, "price_inr": 2550, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "हाँ, रखा है, चार सौ चालीस रुपये लगेंगे, दस बजे तक मैं यहीं हूँ.", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "जी, उपलब्ध है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan sir available hai aapke liye pachis", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "घर पर भेज देंगे बताइए क्या करना है", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "बघा हो, ठेवतो आम्ही, आहे, पूर्ण पॅकेट 710, सुटे पण आहेत 455 ला एक, पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 710, "delivery": false, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "hello? aawaj yetoy ka? haan bola. ek hazar charshe pannas rupayanna padel. ho dada, aahe", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "साढे चार सौ लगेगा, हाँ, रेडी stock है, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "जी बोलिए, पौने दो सौ फाइनल प्राइस है, है भाई, मिल जाएगा, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "sorry सर अभी आउट ऑफ stock है एटमबर्ग एग्जॉस्ट फैन रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एटमबर्ग एग्जॉस्ट फैन"}}
{"product": "Rolson screwdriver bit set", "said": "helo, haan bolo, dekho han han, hai, 1300 likha hai, discount karke gyaarah sau, ladka ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हां है बेसिकली प्राइस कन्फर्म करके बताता हूं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "जी, उपलब्ध है। सौ लगेगालगेगा। होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी, मिल जाएगा, 61... रुकिए, लिस्ट देखता हूँ... पचास, होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "गलत नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "एक मिनिट हां... बोला. पक्कड पडलीये. आमचा रेट १९०, MRP ३१५. येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ रखा है डिलीवरी की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "आहे सर, रेडी स्टॉक आहे. घरपोच करून देतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा है, मिल जाएगा, सिंगल पीस 90, 20 का फुल बॉक्स 1670.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "हाँ, माल पड़ा है, सवा छह हज़ार फाइनल प्राइस है, हम्म, ओके?", "answer": {"in_stock": true, "price_inr": 6250, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "मतलब हाँ, है, साढ़े सात सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "कल फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "गलत नंबर लगाया है आपने।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हॅलो आवाज येतोय का हां बोला म्हणजे ऑइल आहे मिळेल तुम्हाला साडेचारशेला देतो", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "मतलब है भाई, मिल जाएगा। एमआरपी पाँच सौ नब्बे है, डिस्काउंट के बाद तीन सौ साठ पड़ेगा।", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Meco multimeter", "said": "हाँ सर, वो मॉडल अवेलेबल है. उन्नीस सौ फ़ाइनल. एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 1900, "delivery": true, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "त्योहार का सीज़न है ना. stock में है sir. home डिलीवरी available है, चालीस चार्ज. अं सामान पौने तीन सौ का है, delivery अलग से. कंपनी ने फिर रेट बढ़ा दिया. लड़का चाय लेने गया है. और कुछ sir?", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "बरं आमच्याकडे आहे, एका नगाचे दोनशे पंधरा, 20 नगांचा बॉक्स तीन हजार सहाशेला", "answer": {"in_stock": true, "price_inr": 215, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "लड़का चाय लेने गया है। आवाज़ कट रही है, ज़ोर से बोलिए। अच्छा। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। हाँ है। डिलीवरी हो जाएगी, पचास रुपीज़ चार्ज एक्स्ट्रा। आइटम प्राइस सवा दो सौ, डिलीवरी चार्ज अलग।", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "ho, geyser aahe 2850 padel, 2720 cha rate juna ahe, to nahi.", "answer": {"in_stock": true, "price_inr": 2850, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हाँ, माल पड़ा है, आपके लिए ५२०, Royal Enfield lube भी है, वो ५१० का.", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग पड़ा है. देखो. 220 में दे देंगे.", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "जी, उपलब्ध है, रेट बदल गया है, अब सवा सात सौ है, पहले 600 था, डिलीवरी नहीं है.", "answer": {"in_stock": true, "price_inr": 725, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हैलो, कौन? जी, उपलब्ध है। सौ का पड़ेगा। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji boliye, abhi nahi hai, market mein maal ki kami chal rahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "ji haan, rakha hai. 265 mein ho jayega, upar MRP 340 likha hai.", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "बघा आधी पावणेसहाशेला होता आता आहे की नाही बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "Dr.Fixit sampla, Anabond silicone sealant aahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond silicone sealant"}}
{"product": "4 inch door hinge", "said": "यस सर, बोलिए. अच्छा स्टॉक में है. 70 लगा देंगे, वैसे 77 का है.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "han haan bolo maal khatm hai orient 1200 mm ceiling fan chalega sadhe pachis sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "orient 1200 mm ceiling fan"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हो, बल्ब आहे, MRP सहाशे पंचेचाळीस आहे, पण तुम्हाला चारशे तीसला देऊ, अं डिलिव्हरी नाही, शॉपवरून पिकअप करा", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "स्टॉक में है, पूरा पैकेट ५९५, अलग से एक पीस ४१० का, कूरियर कर देंगे.", "answer": {"in_stock": true, "price_inr": 595, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji, available hai, arre ek ka 150, 6 ka dabba 810 ka", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "Ho, bulb milel, kay aahe na baraa ghetle tar 110 pramane, ek piece ekshe vees, dukanatun nyava lagel.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ जी, मिल जाएगा। हाँ तो कंपनी का रेट दो सौ दस है, मैं एक सौ नब्बे लूँगा। कंपनी ने फिर रेट बढ़ा दिया। एक ग्राहक खड़ा है, एक मिनट। बाज़ार में माल की कमी चल रही है। लड़का चाय लेने गया है। और कुछ?", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "price साढ़े पाँच सौ है। हाँ, रेडी stock है", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "आहे आहे आहे एक पीस दोनशेला मिळतो पॅकेट तीनशे चाळीसला", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हां आहे आहे. 140 फिक्स. पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ सर, वो मॉडल अवेलेबल है। भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "नहीं है. हम्म Supreme 3/4 inch elbow दे सकता हूँ, 10 रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Supreme 3/4 inch elbow"}}
{"product": "ESN 2000W immersion rod", "said": "हैलो? हां, आ रही है, अभी अवेलेबल नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "आहे सर, रेडी स्टॉक आहे, बारा घेतले तर 81 प्रमाणे, एक पीस 90.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Nahi hai... ruko ruko, hai, ek pada hai, rate 520 hai sir", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "बरं हो, आहे शिल्लक, पावणे तेवीसशे रुपये होतील.", "answer": {"in_stock": true, "price_inr": 2275, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "हॅलो, हां बोला। हो दादा, आहे। म्हणजे दोन हजार नऊशे पन्नास, जीएसटी एक्स्ट्रा।", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "पहले दस का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हां हां जी, सॉरी सर, अभी आउट ऑफ स्टॉक है, लोकल सील किट है, सेम क्वालिटी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल सील किट"}}
{"product": "Vista 500 ml chain lube", "said": "abhi khatam hai bhai royal enfield chain lube hai chaar sau nabbe ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "royal enfield chain lube"}}
{"product": "Raaisin car cover", "said": "कवर पडा है तीन हज़ार चार सौ नही नही चार हज़ार चार सौ है", "answer": {"in_stock": true, "price_inr": 4400, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "बोला ना, आहे आहे, अं 390 होतील, डिलिव्हरी करत नाही", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे sir, रेडी stock आहे, किंमत नंतर सांगतो, आत्ता लिस्ट नाही.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "हाँ जी, मिल जाएगा, पाँच हज़ार नौ सौ MRP लिखा है, सवा छत्तीस सौ लगा देंगे, आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 3625, "delivery": false, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "bulb khatam nahin hua, hai, dhai sau ka padega, delivery free hai.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "हाँ बोलिए. देखो चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "2050 का पड़ेगा, मिल जाएगा, है अपने पास", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "woh hum nahin rakhte.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "है ना, कितने चाहिए? भेज देंगे, तीस रुपये डिलीवरी चार्ज लगेगा. माल का साढ़े चार सौ, डिलीवरी अलग", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "iBELL drill machine", "said": "मशीन पड़ी है रेट रेट अभी पता नहीं ओनर से पूछना पड़ेगा आके ले जाओ हम्म", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पता नहीं भाई।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "यस, अवेलेबल है, 98 बोला क्या? माफ़ करना, 90 है", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हाँ जी, मिल जाएगा, अं डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Raaisin car cover", "said": "अं नहीं मिलेगा भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "नहीं मिलेगा भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्जा पड़ा है. डिलीवरी नहीं होती, पिकअप करना पडेगा. और कुछ? देखो", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ, माल पड़ा है, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "325 का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "अं नमस्कार बोला बघा नाही ना संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "acha available hai, koi problem nahi our price teen sau pachas, MRP 370 hai.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "नहीं है, मंगवा देंगे, दो दिन लगेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "बरं 2699, मशीन पडलीये, कळवा मग मग.", "answer": {"in_stock": true, "price_inr": 2699, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "मतलब कल सुबह आजाएगा, अभी नहीं, देखो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "haan ahe haan tar tumhala sadesatashe la deto ajun kahi", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "Haan sir, wo model available hai sava chhah sau lagega ab, pehle 525 ka tha.", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "hai bhai, do sao painsath ka packet hai, khule bhi hain 185 wale", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "हाँ बोलिए। Kirloskar वाला ख़त्म है, हैवेल्स वॉटर पंप है, चलेगा? लड़का भेज देता हूँ।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "हैवेल्स वॉटर पंप"}}
{"product": "10W-30 engine oil 1 L", "said": "चुकीचा नंबर लागलाय", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "मिल जाएगा, है अपने पास. डब्बा चार हज़ार चार सौ का है, पचास पीस वाला, एक का सौ", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ हाँ हाँ, है, 30 है, 31 नहीं, मतलब लड़का नहीं है भेजने को, आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "कल सुबह आ जाएगा, अभी नहीं, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ, नौ डिब्बे पड़े हैं, बाज़ार में माल की कमी चल रही है, आज सुबह से लाइट नहीं है, आवाज़ कट रही है, ज़ोर से बोलिए, त्योहार का सीज़न है ना, सवा पाँच सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "hai ji, mil jayega, ek piece 170 ka milta hai, packet do sau paintalis ka, pick up karna padega", "answer": {"in_stock": true, "price_inr": 245, "delivery": false, "alternative": null}}
{"product": "Easy4buy horn", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी सर, बताइए. वो क्या है हाँ, रेडी स्टॉक है. साढ़े तेरह सौ रुपये. देखो", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "आपके लिए तीन सौ दस. मतलब. हां हां, है. शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "जी, उपलब्ध है, सवा तीन सौ लगा देंगे, वैसे चार सौ बीस का है.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "जी जी उपलब्ध है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "yes sir, boliye, haan, ready stock hai, ek so nabbe rupaye", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "हाँ हाँ, है, लड़का चाय लेने गया है, 1300 फ़ाइनल, भेज देंगे, 60 रुपये डिलीवरी चार्ज लगेगा, आवाज़ कट रही है, ज़ोर से बोलिए, कंपनी ने फिर रेट बढ़ा दिया, ठीक है?", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब संपला नाही, आहे, पाठवायला काही प्रॉब्लेम नाही.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "आमच्याकडे आहे, दोन हजार तीनशे सांगितलं का? चुकलो, चोवीसशे पन्नास आहे.", "answer": {"in_stock": true, "price_inr": 2450, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Paune teen sau rupaye lagenge, ji haan, rakha hai, aapke ghar tak bhijwa denge, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Woh hum nahi rakhte, steel hinge rakha hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "Modx steering cover for i10", "said": "एक मिनट... हाँ बोलिए। यस, अवेलेबल है। आपके लिए पाँच सौ पंद्रह। लड़का घर पे दे जाएगा।", "answer": {"in_stock": true, "price_inr": 515, "delivery": true, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "लाइट है, मिल जाएगी. पहले तीन हज़ार तीन सौ पचास था, अब सत्ताईस सौ पचास हो गया, सस्ता हुआ है", "answer": {"in_stock": true, "price_inr": 2750, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, आहे, बरं अँकर एलईडी बॅटन आहे 555 ला, पण हा पावणे पाचशेला मिळेल, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Khadija angle grinder", "said": "हाँ जी, बताइए, वो हम नहीं रखते, बाज़ार में माल की कमी चल रही है, त्योहार का सीज़न है ना, बॉश ग्राइंडर है, पौने ग्यारह सौ का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश ग्राइंडर"}}
{"product": "R-tek multimeter", "said": "बेसिकली मिल जाएगा, है अपने पास, आपको 395 में मिलेगा, MRP 645 है", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "त्योहार का सीज़न है ना, आज सुबह से लाइट नहीं है, नहीं है जी, ख़त्म, आवाज़ कट रही है, ज़ोर से बोलिए, बाज़ार में माल की कमी चल रही है, बजाज 3 L geyser वाला चलेगा? उसका 6800 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बजाज 3 L geyser"}}
{"product": "Electronicspices 60W soldering iron", "said": "है भाई मिल जाएगा एक सौ पचास रुपीज़ है हां पन्द्रह piece पडे हैं", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म नहीं हुई, है, साढ़े पंद्रह सौ फ़ाइनल, ठीक है?", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है जी, ख़त्म, वो क्या है बिड़ला वाला है, आठ सौ साठ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बिड़ला"}}
{"product": "wall putty 20 kg", "said": "yes sir, boliye, yes, available hai, haan toh aath so nabbe ka padega.", "answer": {"in_stock": true, "price_inr": 890, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "savva bechalisshe fix machine padliye porga gharee sodun deil", "answer": {"in_stock": true, "price_inr": 4225, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "Matlab stock khatam ho gaya hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां हां पुट्टी पडलीये.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan wahi wala hai paune do sau last rate hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "यस सर, बोलिए, हाँ, बैटरी है, पौने ग्यारह सौ फ़ाइनल, डिलीवरी बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 1075, "delivery": false, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "porga chaha aanayla gelay, kay sangu, sakal pasun gardi ahe, ek giraik ahe, ek minute, company ne parat rate vadhavle, aatta stock nahi, udya yeil, ajun kahi?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "hai na, kitne chahiye? panch sau chalis rupees hai, bhej denge, pachas rupaye delivery charge lagega", "answer": {"in_stock": true, "price_inr": 540, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हां बोला. हो, आहे. रेट दीडशे आहे. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हो sir, बोला। म्हणजे हे दुकान नाही, चुकीचा नंबर।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "नहीं मिलेगा भाई. अच्छा Mastech है, ढाई सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Mastech"}}
{"product": "Makita drill machine", "said": "aahe ki, milel kay aahe na sattavisshe padel aata, magchya mahinyat don hazar tinashe la hota", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉकमध्ये आहे, चारशे तीसला लावतो, एमआरपी 515 आहे.", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "मिल जाएगा, है अपने पास, home delivery available है, 50 चार्ज, आइटम price छह सौ, डिलीवरी चार्ज अलग", "answer": {"in_stock": true, "price_inr": 600, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हैलो? हाँ, आवाज़ आ रही है। त्योहार का सीज़न है ना। लड़का चाय लेने गया है। ख़त्म हो गया। आवाज़ कट रही है, ज़ोर से बोलिए। mechanical seal दे सकता हूँ, सवा दो सौ रुपये का। और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "mechanical seal"}}
{"product": "Castrol 10W-40 engine oil", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, मोबिल 5 watt इंजन ऑयल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मोबिल 5 watt इंजन ऑयल"}}
{"product": "Philips 5W LED bulb", "said": "hai na, kitne chahiye? chhah ka dabba 545 ka, ek pees 110", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "mhanje stock madhe aahe, 50 final, 22 nag shillak ahet.", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "शेठ बाहेर गेलेत, ते आले की सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "rate das hazar saat sau ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते. त्योहार का सीज़न है ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "थोडी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हैलो? हाँ, आवाज़ आ रही है. रेट 110 है, पर अभी स्टॉक नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "यस sir, बोलिए, अभी स्टॉक नहीं है, कल तक आ जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "रेट दस है, पर अभी स्टॉक नहीं है, अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ बोलिए सर। अं हाँ जी, स्टॉक में पड़ा है। सवा सौ फ़ाइनल।", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "season time hai na, demand zyada hai. haan, ready stock hai. 5400... ek minute, rate list dekhta hun... haan sava chhah hazar. network issue hai, aawaz break ho rahi hai. sir aaj bahut busy day hai. bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 6250, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "नमस्ते बोलिए जी उपलब्ध है साढ़े तीन सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "hai... arre nahi, wo dusra wala hai, ye wala nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "stock nahi hai abhi, Havells switch chalega? 290 ka hai, courier kar denge sir", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Havells switch"}}
{"product": "4 inch door hinge", "said": "hello, ha boliye, nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "पाठवायची सोय नाही.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "हाँ, रखा है, एक कस्टमर खड़ा है, होल्ड करो, 5880... नहीं नहीं, 5168 है, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है.", "answer": {"in_stock": true, "price_inr": 5168, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "matlab ruko dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "haan hai... ek minute, nahi, wo kal khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Matlab pauna atarah sau ka hai, isse kam nahi hoga light padi hai.", "answer": {"in_stock": true, "price_inr": 1775, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "हाँ, आवाज़ आ रही है, हाँ, माल पड़ा है, बयालीस सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 4200, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "देखिए जी, उपलब्ध है। पता भेज दो, लड़का दे आएगा। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "umm haan stock mein hai rate ek sau sath hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "माल खत्म है, बजाज 3 L गीज़र वाला है, 2250 का, हम्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बजाज 3 L गीज़र"}}
{"product": "Kaiweets multimeter", "said": "हैलो, कौन? नहीं है, Fluke multimeter है, 2700 का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fluke multimeter"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अच्छा कल फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "Haan ji, mil jayega, 705 bola kya? galti se bola, 750 hai, bhejne ka intezaam nahi hai", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Paune solah sau final price hai, network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, season time hai na, demand jada hai, ha, wahi wala hai, address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 1575, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "सवा सत्रह सौ में दे देंगे। है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": 1725, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "हॅलो, कोण बोलतंय? मी नवीन आहे, मला माहीत नाही. आवाज तुटतोय, मोठ्याने बोला.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "बोला, काय पाहिजे? अं, स्टॉक संपलाय, हां हां, ऐकतोय, ठीक आहे?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "नमस्कार, बोला। हॅलो? आवाज येत नाही ये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "एक मिनिट हां... बोला, हो, ठेवतो आम्ही, आहे, दोनशे होतील, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "अं हाँ सर, available है, चार सौ पचहत्तर में दे देंगे, और कुछ?", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "नहीं है जी, ख़त्म.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "पहले सात हज़ार चार सौ का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "बल्ब पड़ा है. अं आपके लिए लिए 285. और कुछ?", "answer": {"in_stock": true, "price_inr": 285, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "हाँ, २४ पीस पड़े हैं, १७०० का पड़ेगा", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ जी मिल जाएगा डब्बा दो हज़ार तीन सौ पचास का है 10 पीस वाला एक का दो सौ सत्तर", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "मतलब १ kg नही है local glue है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local glue"}}
{"product": "Speedwav LED headlight", "said": "लाईट संपली नाही, आहे। 1870 ला देतो। बरं पोरगा पाठवतो, 80 रुपये रिक्षाचे। हां", "answer": {"in_stock": true, "price_inr": 1870, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Haan sir, boliye, kya hai na ladka ghar pe chhod dega", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "available hai, koi problem nahi, chalis ka ek, 10 loge toh taintis ke hisaab se laga dunga, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 40, "delivery": false, "alternative": null}}
{"product": "Crispy LED torch", "said": "हाँ, रेडी stock है, 360 लगेगा, delivery नहीं होती, pickup करना पडेगा", "answer": {"in_stock": true, "price_inr": 360, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ भाई, बोलो, स्टॉक में नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "मतलब हाँ रखा है भेज देंगे 40 रुपये डिलीवरी चार्ज लगेगा आइटम प्राइस पंद्रह सौ delivery चार्ज अलग", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हाँ जी, बताइए, हाँ तो टेप ख़त्म हो गई, big teflon roll है, पंद्रह का, भेजने का इंतज़ाम नहीं है, आज सुबह से लाइट नहीं है, कंपनी ने फिर रेट बढ़ा दिया, त्योहार का सीज़न है ना, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "big teflon roll"}}
{"product": "10W-30 engine oil 1 L", "said": "Ha ji, boliye, pata nahi bhai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "kay aahe na nahi kasa, ahe na.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "Tyre padlay dada", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "जी उपलब्ध है 210 रुकिए लिस्ट देखता हूँ एक सौ अस्सी शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Bosch drill machine", "said": "हेलो? हाँ, बोलो भाई, मशीन का पता नहीं, गोदाम में देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "मशीन आहे, मिळेल, अठ्ठावीसशे पन्नास होतील, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 2850, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "Ha sir, wo model available hai, courier kar denge sir", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "त्योहार का सीज़न है ना, हैलो? आवाज़ नहीं आ रही, बाद में करना, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, आज सुबह से लाइट नहीं है, बाज़ार में माल की कमी चल रही है, क्या करना है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "नहीं है भाई, खत्म हो गया. वो क्या है Crompton फैन अवेलेबल है, उसका प्राइस सत्ताईस सौ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton फैन"}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "han, maal pada hai, 1560 ka hai, abhi delivery band hai, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 1560, "delivery": false, "alternative": null}}
{"product": "Avita switch", "said": "आमच्याकडे आहे, हां, तुम्हाला तीनशे चाळीसला देतो, 8 वाजेपर्यंत मी इथेच आहे, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "Retina nahi hai, V-GRIP steering cover hai, bhejne ka intezaam nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "V-GRIP steering cover"}}
{"product": "CEAT 3.00-18 tyre", "said": "hai bhai. 1850 ka padega", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई। आपके लिए सौ।", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हेलो हाँ बोलो सॉरी सर अभी आउट ऑफ स्टॉक है वो Havells 3.15 mfd capacitor पड़ा है वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 3.15 mfd capacitor"}}
{"product": "Candes 1200W room heater", "said": "देखो स्टॉक में नहीं है. ओके?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aamhi te thevat nahi, Passion brake shoe aahe, chalel ka? pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Passion brake shoe"}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, bolo, mil jayega, hai, bhaiya aap ruko do minute, phone pe hun, chorasi... ek minute, rate list dekhta hun... haan sau, awaaz kat rahi hai, zor se bolo, ek customer khada hai, ek minute.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Stock men hai sir, price sadhe chhah sau hai, khud aake uthana padega", "answer": {"in_stock": true, "price_inr": 650, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Aatta stock nahi, udya yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "आमच्याकडे आहे, चारशे पन्नासला देऊ, छापील किंमत 800 आहे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hai abhi bees piece bache hain khud aake uthana padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "हो, बोला दादा, माल नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "देखो हाँ, रखा है, 115... रुकिए, लिस्ट देखता हूँ... एक सौ", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "dekho nahin hai ji, khatm, chahiye toh Anabond silicone sealant de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond silicone sealant"}}
{"product": "12V 7Ah battery", "said": "abhi nahi hai, Amaron 12V 9Ah battery rakha hai, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "1 inch PVC elbow", "said": "aahe na", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हो, मिळेल, सुटा एकशे चाळीसला, 20 नगांचा बॉक्स घेतला तर दोन हजार पाचशे पन्नास.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "यस सर बोलिए सीज़न टाइम है ना डिमांड ज़्यादा है हैलो आवाज़ नहीं आ रही बाद में करना नेटवर्क इशू है आवाज़ ब्रेक हो रही है सर आज बहुत बिज़ी डे है एक कस्टमर खड़ा है होल्ड करो बताओ क्या करना है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "देखो हाँ जी, मिल जाएगा, एक सौ साठ लगेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "है भाई, हाँ तो भेज देता, पर आज बॉय छुट्टी पे है, आके ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "देखो बल्ब नहीं है खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ऐसा नहीं है कि नही है है डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "लाईट संपलीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो दादा, आहे. हां पावणेतीनशे फिक्स. पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "hello, haan bola, mhanje rate pavane tinashe ahe, pakkad sampli nahi, aahe", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "आहे सर, रेडी stock आहे, rate बाराशे पन्नास आहे sir, delivery नाही, शॉपवरून पिकअप करा.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "जी, उपलब्ध है, 320 का है, इससे कम नहीं होगा, शॉप 10 बजे तक ओपन है.", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "हाँ, रखा है, २६६०... नहीं नहीं, २२५० है, शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 2250, "delivery": true, "alternative": null}}
{"product": "Carzex LED headlight", "said": "rate sadhe unnees sau hai sir, haan, light hai", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ, माल पड़ा है. सवा चार सौ में दे देंगे. घर पर भेज देंगे, बीस अलग से", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "है ना, कितने चाहिए? 10 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हां एकशे पंचावन्न फिक्स, हो, आहे शिल्लक", "answer": {"in_stock": true, "price_inr": 155, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan ji boliye tala rakha hai mil jayega teen sau nabbe ka milega aapke ghar tak bhijwa denge", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ट्यूब आहे, 515 MRP आहे, पण साडेचारशेला देऊ", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "नमस्ते, बोलिए, अं कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "आत्ता नाहीये, परवापर्यंत येईल, तर संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "अं हाँ, है, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नहीं है भाई खत्म हो गया अमरोन 12V 5Ah बैटरी है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अमरोन 12V 5Ah बैटरी"}}
{"product": "MYRA helmet lock", "said": "है भाई, मिल जाएगा, अं आपको 179, MRP तो 210 है", "answer": {"in_stock": true, "price_inr": 179, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "aatta nahiye. Type R horn aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Type R horn"}}
{"product": "4 inch door hinge", "said": "हाँ है, 110 का है, आपके लिए सौ.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "आधी 400 ला होता आता आहे की नाही बघावं लागेल हां", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ, माल पड़ा है, 25 का पड़ेगा, अच्छा भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Dow silicone sealant", "said": "home delivery available hai", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "Matlab no no, available hai, 70 rupaye ka hai.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "हाँ हाँ, है. Havells switch का रेट 160 है, ये वाला 190.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हो आहे शिल्लक तीनशे पासष्ट थांबा लिस्ट बघतो चारशे वीस", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "हो, ऑइल शिल्लक आहे, रेट साडेतीनशे आहे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नमस्ते, बोलिए। मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो, बल्ब शिल्लक आहे। रेट २२० आहे", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ जी, मिल जाएगा, होम डिलीवरी हो जाएगी.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है, सवा सौ का है, इससे कम नहीं होगा, ठीक है?", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "sir aaj bahut busy day hai season time hai na, demand zyada hai haan sir, available hai 215 mein de dunga ladka de aayega, 40 rupaye auto ka network issue hai, aawaz break ho rahi hai bata dena", "answer": {"in_stock": true, "price_inr": 215, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "है ना, कितने चाहिए? साढे छह सौ फ़ाइनल, घर पर भेज देंगे, और कुछ?", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ भाई, बोलो, हाँ तो हाँ, माल पड़ा है, एक सौ दस रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "हां हां मशीन पडलीये। सात हजार चारशे... नाही नाही, आठ हजार।", "answer": {"in_stock": true, "price_inr": 8000, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "हाँ, है, सवा आठ सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां बोला साहेब, पावणे आठशेला देतो, हां प्लेट संपली नाही, आहे", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "स्टॉक में है सर 2550 रुपये लगेंगे भेज देता पर आज बॉय छुट्टी पे है आके ले जाओ", "answer": {"in_stock": true, "price_inr": 2550, "delivery": false, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "Ho, motor milel, bara 3540 lihila aahe, discount karun 2950, porga pathavto.", "answer": {"in_stock": true, "price_inr": 2950, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "अं ये दुकान का नंबर नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "आमच्याकडे आहे। पावणेबाराशे फिक्स। संध्याकाळपर्यंत पाठवून देतो।", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "जी बोलिए, 480 रुपये, हाँ, रखा है, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 480, "delivery": true, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "abhi stock men nahi hai, naya maal Monday ko aaega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "ahe sir, ready stock aahe, tumhala sadesahashe la deu, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 650, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Haan bola, kay pahije? mhanje stock madhe aahe, tumhala tinashe tis la deto", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "म्हणजे हो, मशीन शिल्लक आहे, सव्वापाच हजार रुपये, कमी नाही होणार, घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 5250, "delivery": false, "alternative": null}}
{"product": "Raaisin car cover", "said": "स्टॉकमध्ये आहे म्हणजे रेट शेठ सांगतील कुरिअर करतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, bijagri ahe, mhanje sutta navvad la, panchvis nagancha box ghetla tar ek hazar aathshe pannas", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं नाही, आहे। हां तर पाचशेला देतो", "answer": {"in_stock": true, "price_inr": 500, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हम्म, है भाई, मिल जाएगा, बेस्ट price 80, अपना लडका छोड देगा घर पे.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "कव्हर संपलं नाही, आहे। पावणेसहाशे रुपयांना पडेल।", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "हाँ हाँ, है, प्राइस 215, देखो लड़का छोड़ देगा घर पर.", "answer": {"in_stock": true, "price_inr": 215, "delivery": true, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "हां बोला साहेब पिकअप करावा लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "han haan, hai. aapke liye dedh sau. delivery ka 50 extra", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हो मशीन शिल्लक आहे छापील एक हजार सहाशे डिस्काउंट करून तेराशे पन्नास घरपोच नाही तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हाँ, रेडी stock है. 1740, sorry sorry, पौने सत्रह सौ.", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "अच्छा जी, उपलब्ध है, पाँच सौ चालीस का पड़ेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 540, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "नॉट अवेलेबल सर mechanical seal है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "mechanical seal"}}
{"product": "Makita drill machine", "said": "है अरे नहीं वो दूसरा वाला है ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "अं AmiciFlo संपलं, Lakshmi 0.5 HP water pump आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Lakshmi 0.5 HP water pump"}}
{"product": "Tata Agrico plier", "said": "Sampala hota... nahi, ek ahe ajun.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ जी, बताइए. प्लेट है कि नहीं, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello? haan, awaaz aa rahi hai, bolo 4 inch wala khatam hai, 3 inch hinge hai, chalega? home delivery nahi hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3 inch hinge"}}
{"product": "AmiciFlo water pump", "said": "अच्छा मोटर है कि नहीं, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "नमस्कार, बोला, हो, ठेवतो आम्ही, आहे, घरपोच देतो, एक गिऱ्हाईक आहे, एक मिनिट, आवाज तुटतोय, मोठ्याने बोला.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan, hai apne paas, 400 hai, 335 nahi.", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "हो, बोला दादा, आहे आहे, तुम्हाला एक हजार सहाशे पन्नासला देतो, दुकानातून न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "हाँ हाँ, है, तीन सौ बीस में दे देंगे, लड़का भेज देता हूँ, अच्छा और कुछ? तो.", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हां बोला साहेब. स्टॉकमध्ये आहे. 140 आणि 18 टक्के GST वेगळा.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हो सर, अवेलेबल आहे। अं 245चं आहे, तुम्हाला २२० ला देतो", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, रखा है, आपके लिए चालीस.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "umm haan mil jayega bhai 1480 mein ho jayega upar mrp 2110 likha hai pick up karna padega", "answer": {"in_stock": true, "price_inr": 1480, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ हाँ, है, MRP उनतालीस है पर पैंतीस लगा देंगे", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "हाँ जी, गीज़र रखा है. अट्ठाईस सौ है, GST अलग से. घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 2800, "delivery": true, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "हाँ जी sir, बताइए, हाँ हाँ, है, लास्ट मंथ 160 था, अब पौने दो सौ है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "आहे, अजून बावीस पीस पडलेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "प्राइस दीडशे आहे, हो, बल्ब आहे, डिलिव्हरी नाही, शॉपवरून पिकअप करा, बघा अजून काही?", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "म्हणजे। बोला, काय पाहिजे? हो, ठेवतो आम्ही, आहे। चाळीस रुपये होतील", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "अं घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "मोटर है, मिल जाएगी, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "acha mil jayega, hai, courier kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बघा हो, बल्ब शिल्लक आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "नमस्ते, बोलिए, हाँ, माल पड़ा है, सवा नौ सौ लगेगा.", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ है लूज़ पीस २८० का फुल पैकेट चार सौ दस", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "haan bhaiya bataiye dukaan pe aake lena padega", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "Abhi stock mein nahi hai, naya maal Monday ko aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "machine sampli nahi, aahe, 4150 la deto, ghari pathavto.", "answer": {"in_stock": true, "price_inr": 4150, "delivery": true, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हो, मिळेल, सहाशे तीस होतील", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "जी उपलब्ध है चार हज़ार आठ सौ पचास है होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 4850, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हाँ तो हैलो? आवाज़ नहीं आ रही, बाद में करना। कंपनी ने रेट बढ़ादिया", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Ho, bulb milel, vis ghetle tar ekshe vees pramane, ek piece ekshe tis", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, बोला दादा, दुकानातून न्यावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "हेलो, हाँ बोलो, मिल जाएगा, है अपने पास, देखो, बेसिकली साढ़े बारह सौ का पड़ेगा, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Anchor LED batten", "said": "मिल जाएगा है अपने पास पहले 160 था अब 195 हो गया", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "लाइट ख़त्म नहीं हुई, है, एमआरपी 1880 है पर 1600 लगा देंगे, बेसिकली सीज़न टाइम है ना, डिमांड ज़्यादा है, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, sir आज बहुत बिज़ी डे है, एक कस्टमर खड़ा है, होल्ड करो.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Rate savaa sau hai sir, arre haa, ready stock hai", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "R-tek multimeter", "said": "हाँ, है, आपके लिए सवा चार सौ, देखिए डिलीवरी फ़्री है, ठीक है?", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ जी, मिल जाएगा, देखिए अस्सी फ़ाइनल", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है भाई, 380 लिखा है, छूट देकर तीन सौ तीस.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "helmet khatm nahi hua, hai, saat sao pandrah rupaye ka hai", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ti sampli ho, kay aahe na Bosch Jupiter clutch plate aahe, 530 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Jupiter clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "Haan toh stock khatam ho gaya hai, Shell 20W-40 engine oil chalega? 440 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Shell 20W-40 engine oil"}}
{"product": "50 mm padlock", "said": "bara kulup sampala nahi, ahe, rate pavaneteenshe ahe sir, self pickup ahe.", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "MK 16A switch", "said": "हाँ, माल पड़ा है, आपको एक सौ पैंतालीस में दे देंगे, MRP एक सौ अस्सी है", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बघा आमच्याकडे आहे, चौदाशे पन्नास रुपये, कमी नाही होणार.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "टायर संपला नाही, आहे, आज सकाळपासून लाईट नाही, हां हां, ऐकतोय, कंपनीने परत रेट वाढवले, तीन हजार शंभर पडेल, दोन हजार पाचशेचा रेट जुना आहे, तो नाही.", "answer": {"in_stock": true, "price_inr": 3100, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "अं आहे... एक मिनिट, नाही, ते काल संपलं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "नमस्ते, बोलिए, हाँ जी, मिल जाएगा, रेट सवा दो सौ है, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "हाँ, रखा है, पौने सात सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हां सर वो मॉडल अवेलेबल है अं फुल पैकेट सौ लूज पीस बारह का", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "kal phone karna kal sunday tha toh dukaan band thi", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli nahi aahe kay aahe na rate 90 aahe yeun gheun ja", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "मतलब बल्ब पड़ा है, रेट 290 है सर", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ रखते हैं, पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "हां तर हो, ठेवतो आम्ही, पण आत्ता संपलंय, तर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "हां, माल पडा है, देखो 319 और 18 परसेंट GST अलग, हम्म.", "answer": {"in_stock": true, "price_inr": 319, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ है हम्म आपके घर तक भिजवा देंगे बताइए करना है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "एक गिऱ्हाईक आहे, एक मिनिट, काय सांगू, सकाळपासून गर्दी आहे, बाजारात माल कमी आहे सध्या, बिजागरी आत्ता नाहीये, संपली, आवाज तुटतोय, मोठ्याने बोला, कळवा मग.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "कम नहीं है, बहुत पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "milel na, aahe, rate savvadonshe ahe", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hallo, ha boliye, matlab owner abhi available nahin hain, baad men call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ जी सर, बताइए। यस, अवेलेबल है। दस रुपये लगेंगे। डिलीवरी नहीं होती, पिकअप करना पडेगा।", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "haa, pankha hai, ek hazar pachas hai, 835 nahi", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "haan sir, available hai, aapko 590 me milega, MRP 715 hai, ghar tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 590, "delivery": true, "alternative": null}}
{"product": "AmiciTools water pump", "said": "नमस्ते बोलिए हां तो वो हम नही रखते उसकी जगह हैवेल्स 1.0 HP water पंप दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 1.0 HP water पंप"}}
{"product": "teflon tape", "said": "haan bola... nahi, te sadhya nahi, tyachya aivaji thread seal tape deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "thread seal tape"}}
{"product": "Anchor LED batten", "said": "han ha, bolo han sir, wo model available hai ek sao sattar ka padega home delivery nahi hai bhai", "answer": {"in_stock": true, "price_inr": 170, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "acha ha, mil jayega bhai, aapke liye 60.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "बिजागरी आहे, मिळेल, तुम्हाला एकशे वीस, छापील किंमत एकशे पन्नास.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "हाँ तो मशीन पड़ी है, सवा तेईस सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 2325, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब आहे मिळेल पॅकेट सहाशे वीसला आहे सुटा एक चारशे पंधराला", "answer": {"in_stock": true, "price_inr": 620, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "नाही वाटतं थांबा आहे एक उरलाय येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "MYRA helmet lock", "said": "यस sir, बोलिए. १८५ में दे देंगे. मिल जाएगा, है अपने पास. home डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ, रखा है, साढ़े चार सौ लगेगा, तीन सौ पैंसठ वाला रेट पुराना है, वो नहीं.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "बोला ना, आज सकाळपासून लाईट नाही, हां हेल्मेट आहे, मिळेल, दिवाळीची गर्दी आहे ना, छापील ५२५, डिस्काउंट करून ४७०", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "हाँ जी, पंखा रखा है, प्राइस इनक्रीज़ हो गया, पहले पाँच सौ पचास था, अब सात सौ पैंतीस, वो क्या है डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 735, "delivery": false, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "hai na, kitne chahiye? 190 hai, 225 nahi", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ti sampli ho birla wala aahe tyacha price sahashe sath", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "birla"}}
{"product": "Windek tyre", "said": "bas chaubis piece bache hain kya hai na rate abhi pata nahi, malik se puchna padega bolo kya karna hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "है ना, कितने चाहिए? पौने चार सौ लगेगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "हाँ बोलिए। हाँ, ऑयल है। हमारा रेट 295, MRP तो 330 है। पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 295, "delivery": true, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "helmet nahiye, sampla, Vega aahe aamchyakade", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega"}}
{"product": "Havells Oro switch", "said": "हाँ जी स्टॉक में पड़ा है एमआरपी 590 है पर चार सौ पैंसठ लगा देंगे", "answer": {"in_stock": true, "price_inr": 465, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Haan, mil jayega bhai, sava panch sau final.", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Out of stock nahin hai, available hai, 35 rupaye hai.", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "bara stock madhe ahe, haan haan, aiktoy, ekshe panchyaainshi la lavto, MRP 235 ahe, porga chaha aanayla gelay, kay sangu, sakal pasun gardi ahe", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "hello? aawaj yetoy ka? haan bola, thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "han ha bolo geyser pada hai pone sat hajar mein de dunga aur kuch", "answer": {"in_stock": true, "price_inr": 6750, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "pata nahi bhai. are kya bataun, subah se bahut rush hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "4 socket sampla, Anchor 3 socket board wala aahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 3 socket board"}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण बोलतंय? एक गिऱ्हाईक आहे, एक मिनिट, हां हां, ऐकतोय, आज सकाळपासून लाईट नाही, आत्ता गडबडीत आहे, अर्ध्या तासाने फोन करा, ठीक आहे?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "है भाई, प्राइस 360 है है, डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 360, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हो, मिळेल। तीस होतील। दुकानातून न्यावं लागेल। कळवा मग।", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Stanley angle grinder", "said": "हैलो? हाँ, आवाज़ आ रही है, आठ पीस हैं स्टॉक में, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ho, oil aahe, charshe chalis... thamba, list baghto... sadepachshe", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "kya hai na rate chaudah sau pachas ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "नहीं मिलेगा भाई भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "out of stock nahi hai, available hai, same day delivery kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "अं शाम तक माल आ जाएगा, अभी नहीं है, सौ फ़ाइनल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ भाई, बोलो, हाँ, रखा है, आपके लिए 900, सॉरी, डिलीवरी सर्विस नहीं है, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 900, "delivery": false, "alternative": null}}
{"product": "iBELL drill machine", "said": "stock khatam ho gaya hai, bata dena.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "हाँ जी मिल जाएगा सत्ताईस सौ इन्क्लूसिव ऑफ जीएसटी खुद आके ले जाना पडेगा", "answer": {"in_stock": true, "price_inr": 2700, "delivery": false, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "haan aahe savvaakarashe rupaye", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Available hai, koi problem nahi kya hai na 10 loge toh 200 ke hisaab se, ek ka 240 ladka bhej deta hun.", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "Machine hai, mil jayegi, price sade terah sau hai, dukaan se le jana padega.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Ho sir, bola, stock nahi ahe sadhya, bagha AmiciTools 60W iron available aahe, chaalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools 60W iron"}}
{"product": "Bosch screwdriver bit set", "said": "Haan haan, bolo, hai na, kitne chahiye? kal Sunday tha toh dukaan band thi, 340 lagega ab, pehle 255 ka tha, haan haan, sun raha hun, company ne is mahine phir rate badhaya, address bhej do, bhej dete hain", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "Ahe... ek minute, nahi, te kal sampala Autoplus lock ahe, chaalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autoplus lock"}}
{"product": "Adima LED torch", "said": "kay sangu, sakal pasun gardi aahe porga chaha aanayla gelay Diwali chi gardi ahe na kal Ravivar hota mhanun dukan band hota te model available nahi Havells torch ahe, tyacha price 175 delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells torch"}}
{"product": "2 inch paint brush", "said": "stock में है sir। भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ख़त्म हो गया था... नहीं, एक बचा है, पचास रुपये का है.", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "Delivery nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "hai bhai khud aake uthana padega aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ, है. एक पीस 435 का मिलता है, पैकेट 659 का", "answer": {"in_stock": true, "price_inr": 659, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "अभी बिज़ी हूं कॉल बैक करता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां हां हां बोला साहेब सध्या नाही मागवावं लागेल हां घरपोच देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "पहले साढे पैंतीस सौ का था अभी है कि नही देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "yes, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "है भाई. सवा चार सौ का पडेगा. ठीक है?", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नमस्ते, बोलिए. नहीं है, बोलो तो मंगवा देता हूँ, दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, बॅटरी आहे, बाराशे पन्नासला देतो, Exide 12V 5Ah battery हवं असेल तर ते पण आहे, 1230", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, माल पड़ा है। अब 120 है, पहले 92 था। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "एक मिनट... हाँ बोलिए, कंपनी ने फिर रेट बढ़ा दिया, आज नहीं है, कल आएगा, लड़का चाय लेने गया है, अं अरे क्या बताऊँ, सुबह से बहुत भीड़ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "नहीं है जी, ख़त्म, ओसराम एलईडी हेडलाइट बल्ब चलेगा? उसका 140 है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओसराम एलईडी हेडलाइट बल्ब"}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ जी, लाइट रखी है. पूरा पैकेट 450, अलग से एक पीस 310 का", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bolo... nahi, wo nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "है भाई. एक हज़ार पचास का... नहीं नहीं, गलत बोला, चौदह सौ का है.", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब ख़त्म नहीं हुई, है, अच्छा खुला एक तीन सौ नब्बे का, पूरा पैकेट छह सौ पच्चीस", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "हॅलो आवाज का हां बोला लाईट संपली नाही आहे MRP पाचशे आहे तुम्हाला पावणेपाचशेला देतो", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "नहीं है मंगवा देंगे दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Matlab haan, tube hai. full packet chhah sau nabbe, loose piece 400 ka", "answer": {"in_stock": true, "price_inr": 690, "delivery": null, "alternative": null}}
{"product": "Goodyear bolt cutter", "said": "हेलो, हाँ बोलो। हाँ sir, वो मॉडल अवेलेबल है। साढ़े आठ सौ रुपये का है। वो क्या है लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "स्टॉक में सिर्फ 24 पीस है, भेजने का इंतज़ाम नहीं है, अं.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "म्हणजे मी नवीन आहे मला माहीत नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "हाँ है लास्ट मंथ एक हज़ार एक सौ था अब चौदह सौ है आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery padliye, porga pathavto, ainshi rupaye rikshache, vastuche sadenaushe rupaye, delivery vegli", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Mal khatm hai, Astral 3/4 inch elbow hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Astral 3/4 inch elbow"}}
{"product": "Hella horn", "said": "है ना, कितने चाहिए? जी। अच्छा", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "स्टॉक में है। दस का डिब्बा दो हज़ार एक सौ पचास का, एक पीस दो सौ साठ", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "एक मिनट हाँ बोलिए नहीं है Rolson screwdriver bit set रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rolson screwdriver bit set"}}
{"product": "Vormir 200A welding machine", "said": "नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, sir आज बहुत बिज़ी डे है, Vormir वाला फिनिश है, iBELL 200A वेल्डिंग मशीन available है, और कुछ चाहिए सर?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "iBELL 200A वेल्डिंग मशीन"}}
{"product": "Usha 1250 mm ceiling fan", "said": "ho sir, bola pankha samplay Diwali chi gardi aahe na kalva mag", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग ख़त्म नहीं हुआ, है, 140 लगेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "एक मिनट... हाँ बोलिए। हाँ हाँ, है। आपसे 520 लेंगे, लिखा 615 है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 520, "delivery": false, "alternative": null}}
{"product": "Goodyear tyre", "said": "अभी १५ पीस ही बचे हैं, रेट ३३५० है.", "answer": {"in_stock": true, "price_inr": 3350, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "हेल्मेट आहे, मिळेल। सहाशे पंच्याऐंशी, जीएसटी एक्स्ट्रा। पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 685, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, रखा है, एक सौ बीस का है, इससे कम नहीं होगा, 8 एमएम बोल्ट चाहिए तो वो भी है, 145 वाला, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "self pickup hai, delivery nahi", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "जी बोलिए. वो आवाज कट रही है, ज़ोर से बोलिए. लडका चाय लेने गया है. कंपनी ने फिर रेट बढा दिया दिया. अभी बिजी हूं, आधे घंटे बाद फोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "स्टॉक में है सर पहले १६२० था अब १४५० हो गया सस्ता हुआ है", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ सर, अवेलेबल है, बीस का फुल बॉक्स एक सौ पैंसठ, सिंगल पीस दस", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan bhaiya, bataiye. hai ji, mil jayega. sadhe saat sau last rate hai.", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "हाँ भाई बोलो हाँ जी मिल जाएगा पौने तीन सौ लगेगा लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "bara ho ahe... nahi nahi, to dusara ahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "जी जी, उपलब्ध है। दो सौ तीस है। अं। वो घर तक पहुँचा देंगे।", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "हाँ, माल पड़ा है। 385 फ़ाइनल। डिलीवरी फ़्री है। दुकान 9 बजे तक खुली है।", "answer": {"in_stock": true, "price_inr": 385, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "है भाई, मिल जाएगा, दस का डिब्बा 1220 का, एक पीस एक सौ पैंतीस.", "answer": {"in_stock": true, "price_inr": 135, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "नहीं है भाई खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "namaste ji, boliye, putty nahi hai, khatam ho gayi, uski jagah Birla wall putty 40 kg le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Birla wall putty 40 kg"}}
{"product": "RSCT 9W LED bulb", "said": "हाँ रखा है १० लोगे तो १९० के हिसाब से एक का २२५", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "जी बोलिए मिल जाएगा है अपने पास अरे क्या बताऊँ सुबह से बहुत भीड़ है कंपनी ने फिर रेट बढ़ा दिया आपके लिए सवा चार सौ", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "तर ते मॉडेल available नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "यस, अवेलेबल है, रेट 330 है सर, आके ले जाओ.", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "han maal pada hai arre sadhe teen sau de dena mrp toh teen so assi hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "बॉय पाठवतो", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "हाँ बोलिए sir. हां जी, स्टॉक में पडा है. बारह सौ पचास का, इससे कम नही होगा. खुद आके ले जाना पडेगा.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "ha, maal pada hai. saadhe chaudah sau men de dunga. ladka ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "संपले सगळे। त्याच्याऐवजी MYRA helmet lock देऊ शकतो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "MYRA helmet lock"}}
{"product": "Accessoreez helmet lock", "said": "Haan ji, mil jayega, khula dhaai sau ka, 10 ka poora box 2150 mein.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ho sir, available aahe, bagha courier karto sir.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हाँ जी सर, बताइए, हाँ जी, स्टॉक में पड़ा है, वो क्या है चार सौ बीस का है", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ sir, available है, आपके तीस.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "एक मिनट हाँ बोलिए हाँ रखा है पिछले महीने 285 था अभी 340 तो दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "स्टॉक में है सर। सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "पक्कड संपली, JCB plier आहे, सेम क्वालिटी, बघा पोरगा घरी आणून देईल", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "JCB plier"}}
{"product": "Osram LED headlight", "said": "लाईट आहे, मिळेल, 380 ला देतो, AutoPowerz हेडलाईट पण आहे, ते 465 ला, पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, माल पड़ा है। डेढ़ सौ लगेगा, 190 वाला रेट पुराना है, वो नहीं। शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "फ्री डिलीवरी है अच्छा", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Rainfun car cover", "said": "रेट छः सौ है, पर अभी स्टॉक नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, बॅटरी आहे। म्हणजे एक हजार तीनशे... नाही नाही, एक हजार शंभर", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "ha, wahi vala hai aapke liye pone sola sau", "answer": {"in_stock": true, "price_inr": 1575, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "nahi hai... ruko ruko, hai, ek pada hai, umm tis ka hai, isse kam nahi hoga, address bhej do, bhej dete hain", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो कोण बोलतंय आमच्याकडे नसतं ऑर्डर करून मागवतो घरपोच देतो बघा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "कव्हर संपलं। Viral Shopping steering cover आहे, त्याचा price 235।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Viral Shopping steering cover"}}
{"product": "Spaiko LED headlight", "said": "हां उद्या फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "अं ट्यूब पडलीये पूर्ण पॅकेट तीनशे चाळीस सुटे पण आहेत दोनशे दहाला एक", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "अरे क्या बताऊँ, सुबह से बहुत भीड़ है। मिल जाएगा, है अपने पास। पंद्रह रुपये। त्योहार का सीज़न है ना। चौदह पीस हैं स्टॉक में। अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "Havells Oro switch", "said": "helo? han, awaaz aa rahi hai, bolo. hai na, kitne chahiye? aapke liye sadhe panch sau. khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "स्टॉकमध्ये आहे १० नगांचा बॉक्स दोन हजार पाचशे पन्नासला सुटा एक पावणेतीनशे", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "जी बोलिए, देखिए हाँ, रखा है, साढ़े तीन सौ लगेगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "आहे sir, रेडी stock आहे। तुम्हाला सव्वातीनशेला देऊ। home delivery आहे", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "hai, bilkul hai, paune teen sau last rate hai, local air filter bhi hai, wo do sau pandrah ka", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "हाँ जी, stock में पड़ा है, एक कस्टमर खड़ा है, होल्ड करो, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, sir आज बहुत बिज़ी डे है, घर तक पहुँचा देंगे, सीज़न टाइम है ना, डिमांड ज़्यादा है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "pankha aahe, milel, terashe la lavto, MRP 1570 aahe, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "ji boliye malik abhi nahi hain, baad mein phone karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ho, plug aahe, vis ghetle tar donshe pach pramane, ek piece adichshe, delivery deto sir", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "कव्हर संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो मिळेल दीडशे रुपये कमी नाही होणार हां घरपोच देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "rate पच्चीस है sir, हाँ sir, वो मॉडल available है, बेसिकली और कुछ चाहिए sir?", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "hai bhai, rate do sau sath hai, delivery ka 60 extra", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "Haan, mil jayega bhai, rate pone chah sau hai.", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हैलो, कौन? लड़का चाय लेने गया है, बाज़ार में माल की कमी चल रही है, आवाज़ कट रही है, ज़ोर से बोलिए, त्योहार का सीज़न है ना, है ना, कितने चाहिए? तीन सौ दस का है, आपके लिए ढाई सौ.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म हो गई. देखो. मतलब आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "वो मॉडल अवेलेबल नहीं है. Passion brake shoe वाला पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Passion brake shoe"}}
{"product": "Orient 400 mm pedestal fan", "said": "पौने छत्तीस सौ लगेगा. हाँ sir, वो मॉडल available है", "answer": {"in_stock": true, "price_inr": 3575, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "लाइट पड़ी है। छः सौ सत्तर में देंगे। अं घर पर भेज देंगे, पचास अलग से", "answer": {"in_stock": true, "price_inr": 670, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "ख़त्म नहीं हुआ, अभी भी है, रेट सवा पाँच सौ है.", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "नहीं है... रुको रुको, है, एक पड़ा है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "रॉंग नंबर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "बाराशे रुपये कमी नाही होणार स्टॉकमध्ये आहे", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan maal pada hai aapke liye 480 godrej 40 mm padlock chahiye toh wo bhi hai 605 wala", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "stock madhe aahe pannas rupaye hotil delivery nahi dukanat yeun ghya kay aahe na kalva mag", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "stock khatam ho gaya hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "आज नही है कल आएगा डिलीवरी नही है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "लाईट संपली नाही, आहे, पूर्ण पॅकेट 515, सुटे पण आहेत 305 ला एक, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 515, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ जी, बताइए, रेट एक सौ साठ है, हाँ जी, मिल जाएगा", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "abhi stock finish ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "Yes, available hai rate 375 hai sir ghar pe bhej dunga, 50 lagega bhejne ka.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "haan, ready stock hai, GST extra lagega, rate chaar sau paintees", "answer": {"in_stock": true, "price_inr": 435, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "पौने बाईस सौ लगेगा है भाई", "answer": {"in_stock": true, "price_inr": 2175, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "एक मिनट... हाँ बोलिए। हाँ जी, मिल जाएगा। इक्कीस सौ पचास रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हैलो? हाँ, आवाज़ आ रही है. हाँ, है. डेढ़ सौ लगेगा. आपको आकर लेना पड़ेगा. और कुछ?", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "वो तो कल ही ख़त्म हो गया, हैवेल्स 450 एमएम फैन है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 450 एमएम फैन"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan tar haan ahe, ek pees char la milto, pyaket chaalis la", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "जी जी, उपलब्ध है, 5800 रुपये काहै, Usha 400 mm pedestal fan चाहिए तो वो भी है, 4960 वाला.", "answer": {"in_stock": true, "price_inr": 5800, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ सर अवेलेबल है सवा दो सौ लगादेंगे वैसे दो सौ पचपन का है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "वो मॉडल अवेलेबल नही है. Type R वाला है, चलेगा? वो. घर तक पहुंचा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Type R"}}
{"product": "Aegon angle grinder", "said": "हाँ भाई, बोलो, हाँ जी, स्टॉक में पड़ा है, प्राइस सवा चौदह सौ है.", "answer": {"in_stock": true, "price_inr": 1425, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "namaskar, bola, haan tar aatta sangta yenar nahi, sandhyakali phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "अं है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "DHRUV-PRO नहीं है, Mastech मल्टीमीटर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Mastech मल्टीमीटर"}}
{"product": "Bosch GWS angle grinder", "said": "स्टॉक में है सर, २१५० रुपीज़ है, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "आउट ऑफ स्टॉक नहीं है, अवेलेबल है, फ्री डिलीवरी है.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Ho sir, available aahe, rate 160 aahe sir, dukan 9 vajeparyant ughda aahe", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan, kaun? acha, bolo, ek customer khada hai, ek minute, ruko, dekh ke batata hun, aapko number kisne diya? acha thik hai, aur kuch chahiye?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "Inditrust वाला फिनिश है AmiciTools सोल्डरिंग आयरन वाला अवेलेबल है दुकान से ले जाना पडेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "AmiciTools सोल्डरिंग आयरन"}}
{"product": "Allure Auto car mat", "said": "यस सर,, पता नहीं भाई, बाद में कॉल करो, सीजन टाइम है ना, डिमांड ज्यादा है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "२४० में देदेंगे. जी, उपलब्ध है. अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "Havells Cista room heater", "said": "अभी नहीं है, ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हैलो? हाँ, आवाज़ आ रही है। त्योहार का सीज़न है ना। बाज़ार में माल की कमी चल रही है। अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना। अं ठीक है?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "तर शेठ बाहेर गेलेत, ते आले की सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "hai na, kitne chahiye? bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "हॅलो। पाठवायची सोय नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "जी बोलिए हां रखा है rate पूछ कर बताता हूं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Acha haan, ready stock hai, aapke liye do sau sath, home delivery available hai, assi charge", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर संपली नाही, आहे, सव्वापंधराशे रुपये होतील.", "answer": {"in_stock": true, "price_inr": 1525, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "Hai ji, mil jayega, company ka rate teen hajar panch sau hai, main paune untees sau lunga, ghar tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 2875, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "यस सर बोलिए है ना कितने चाहिए वो वो सौ फाइनल प्राइस है बेसिकली लड़का नहीं है भेजने को आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो बोलादादा हो पुट्टी आहे आता 960 आहे आधी 1180 होता घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 960, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "मिल जाएगा है अपने पास मतलब पाँच सौ पचानवे लगा देंगे वैसे ६८० का है", "answer": {"in_stock": true, "price_inr": 595, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan stock mein hai tees ka ek, chhah loge toh chaubis ke hisaab se laga dunga kya hai na dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Rate pone char sau hai, par abhi stock nahi hai, aaj nahin hai, kal aaega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ji boliye. haan sir, wo model available hai. saath plus GST. free delivery hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए वो हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "Haan, kaun? acha, bolo, bhaiya aap ruko do mint, phone pe hun, haan ji, cover available hai, chhah sau paintees rupaye ka hai, company ne is mahine phir rate badhaya, address bhej do, ladka de aayega", "answer": {"in_stock": true, "price_inr": 635, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "ho sir, available ahe, bagha rate deedashe ahe sir", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "एक मिनट... हाँ बोलिए, कल फ़ोन करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "हाँ, रखा है, तेरह सौ पचास रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ है पचपन लगा देंगे वैसे साठ का है वो क्या है होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 55, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "वो स्टॉक में है एक सौ बीस रुपये का है भेज देंगे 20 रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "वो क्या है आज नहीं है, कल आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हाँ है, कोई प्रॉब्लम नहीं। चार सौ नब्बे रुपीज़ है। देखिए और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 490, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हैलो? हाँ, आवाज़ आ रही है, देखिए नहीं है जी, ख़त्म, स्टड्स हेलमेट है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टड्स हेलमेट"}}
{"product": "Crasts LED torch", "said": "ek minute ha... haan bola. haan tar udya sakali maal yeil, aatta nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "है भाई, मिल जाएगा, तो, डिलीवरी नही होती, पिकअप करना पडेगा, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "अभी बिज़ी हूं, call बैक करता हूं, देखो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "tyre ka stock check karke batata hun. kal Sunday tha to dukan band thi.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "वो आइटम हम नहीं रखते, सीज़न टाइम है ना, डिमांड ज़्यादा है, अं बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, है. एक सौ बीस लगेगा, एक सौ तीस वाला रेट पुराना है, वो नहीं.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "जी उपलब्ध है भेज देता पर आज लड़का नहीं आया आ कर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan sir, bataiye, available hai, koi problem nahi, rate paune aath sau hai sir, bata dena.", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "रेट सवा दो सौ है सर, है भाई, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "मिल जाएगा, है अपने पास, रेट सवा बारह सौ है", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ बोलिए, हाँ जी, मिल जाएगा, आपके लिए 760, दुकान से ले जाना पड़ेगा, देखो बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 760, "delivery": false, "alternative": null}}
{"product": "Makita angle grinder", "said": "ho, thevto aamhi, ahe. bara kimmat nantar sangto, aatta list nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "हाँ रखते हैं, पर अभी ख़त्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "छब्बीस सौ का आता है, लेकिन अभी ख़त्म है, कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है भाई. 170... नहीं नहीं, पौने दो सौ है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan hai, arre 170 ka hai, isse kam nahi hoga, dus baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "haan ji, mil jayega, ek sau pachanve rupaye ka hai", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "वो रेट 6100 है हाँ हाँ मशीन है अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 6100, "delivery": true, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ जी, स्टॉक में पड़ा है. 160 में दे देंगे. एड्रेस भेजो, डिलीवर कर देंगे. दुकान दस बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "सध्या नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "जी, उपलब्ध है। लूज़ पीस 255 का, फुल पैकेट तीन सौ नब्बे।", "answer": {"in_stock": true, "price_inr": 390, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हाँ, माल पड़ा है. पाँच सौ पचानवे MRP लिखा है, पौने चार सौ लगा देंगे", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "म्हणजे आत्ता स्टॉक नाही, उद्या येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan sir bataiye khatam ho gaya big teflon roll hai same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Speedwav handle grip for Yamaha", "said": "आहे ना, छापील किंमत सहाशे पंचवीस आहे, आम्ही पावणेपाचशेला देतो.", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "अं नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "जी बोलिए, 290 का पड़ेगा, हाँ जी, स्टॉक में पड़ा है", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ sir, available है, डब्बा 1580 का है, दस piece वाला, एक का एक सौ पचासी", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी आहे का बघावं लागेल, नंतर सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "matlab HEWA wala khatam hai, Usha 230 mm exhaust fan hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha 230 mm exhaust fan"}}
{"product": "Makita drill machine", "said": "हो, मशीन शिल्लक आहे. सव्वा अठ्ठावीसशे रुपयांना.", "answer": {"in_stock": true, "price_inr": 2825, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "owner nahit, nantar call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "वीसला देतो, हां टेप संपली नाही, आहे.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हॅलो? आवाज येत नाहीये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "Haan ji, mil jayega, sava atthais sau laga denge, waise teen hajaar panch sau pachas ka hai", "answer": {"in_stock": true, "price_inr": 2825, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "Haan bola, te kalach sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ हाँ, है। 10 लोगे तो 160 के हिसाब से, एक का पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "rate ek so tees hai, available hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 130, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ तो अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "ऑयल है, मिल जाएगा, एमआरपी 355 है, पर हम तीन सौ पैंतीस में देते हैं, घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 335, "delivery": true, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "stock में है पैंतीस सौ पचास का है भेजदेंगे 50 रुपये delivery चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 3550, "delivery": true, "alternative": null}}
{"product": "JCB plier", "said": "तर हो पक्कड शिल्लक आहे ३८० रुपये कमी नाही होणार घरी पाठवतो ५० वेगळे", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "acha kal Sunday tha toh dukaan band thi, 10W-30 nahi hai, Shell 10W-30 engine oil 900 ml hai, home delivery hai, arre kya bataun, subah se bahut rush hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Shell 10W-30 engine oil 900 ml"}}
{"product": "10W-30 engine oil 1 L", "said": "ऑयल है, मिल जाएगा, पौने चार सौ रुपीज़ है.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Sadhya nahiye, order dilay, don divsat yeil, ekshe pannas hotil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "Bola daada, ho dada, ahe, kay aahe na rate savvaaththaveesshe aahe.", "answer": {"in_stock": true, "price_inr": 2825, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "Atomberg वाला फिनिश है, Bajaj सीलिंग फैन available है, अपना आदमी दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Bajaj सीलिंग फैन"}}
{"product": "Stanley bolt cutter", "said": "हैलो? हाँ, आवाज़ आ रही है। Stanley वाला ख़त्म है, गुडईयर बोल्ट कटर है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गुडईयर बोल्ट कटर"}}
{"product": "Lifelong 5 L geyser", "said": "haan bolo dukaan se le jana padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब का स्टॉक चेक करके बताता हूँ तो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Nahi milega bhai, 12 mm bolt hai, wo le lo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 mm bolt"}}
{"product": "Ceat Milaze tyre", "said": "Bagha ho, tyre milel, bill pahije asel tar GST vegla, rate naushe panchyannav, manus pathavto ghari.", "answer": {"in_stock": true, "price_inr": 995, "delivery": true, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "एक ग्राहक खड़ा है, एक मिनट, आवाज़ कट रही है, ज़ोर से बोलिए, पंखा है, मिल जाएगा, कंपनी ने फिर रेट बढ़ा दिया, 4680 लिखा है, छूट देकर 3900, त्योहार का सीज़न है ना", "answer": {"in_stock": true, "price_inr": 3900, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "775 ला पडेल। हो, मिळेल। सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "Pankha samplay, udya sakali maal yeyil, aatta nahiye, chaudashe pannas la padel", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हाँ सर, अवेलेबल है। 900, सॉरी सॉरी, 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "है है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "नॉट available sir हम्म उसकी जगह Speedwav horn ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav horn"}}
{"product": "Savsol 20W-40 engine oil", "said": "हाँ जी, मिल जाएगा। बेस्ट प्राइस सवा दो सौ।", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "अं टायर आहे मिळेल येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "Savsol nahi hai, Liqui Moly 10W oil hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Liqui Moly 10W oil"}}
{"product": "wall putty 20 kg", "said": "savva aathshe rupaye, kami nahi honar, putty sampli nahi, ahe, delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": true, "price_inr": 825, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "हाँ हाँ जी. वो क्या है हाँ, माल पड़ा है. दो सौ पचानवे... नहीं नहीं, ढाई सौ है", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "हां बोला साहेब आत्ता नाहीये परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Shillak nahi, big teflon roll deu shakto, pandhra padel", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Roots horn", "said": "अं जी बोलिए हां माल पडा है साढे सात सौ लगेगा", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape padi hai price tis hai free delivery hai", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "बघा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan, hai apne paas, acha rate 200 hai sir, thik hai?", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "एक मिनट... हाँ बोलिए. है भाई. पौने दो सौ रुपये लगेंगे. पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "जी बोलिए बीस पीस हैं स्टॉक में रेट अस्सी है घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "namaste ji, boliye, matlab chaar sau sattar lagega, han ha, hai", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, रखा है। देखिए सौ रुपये का है। भेजने का इंतज़ाम नहीं है।", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Mastech multimeter", "said": "ho sir, bola, sahashe panchahattar final, ho sir, available aahe", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ हाँ माल पड़ा है शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "हो दादा, आहे. दोनशे दहा रुपये, कमी नाही होणार. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर संपली नाही, आहे। आता 1850 आहे, आधी 1610 होता", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हाँ जी सर, बताइए, आज नहीं है, कल आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "motor aahe milel porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "haan ji, mil jaega, kya hai na paune do so hai, GST alag, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "pichle hafte bhi kisi ne yahi poocha tha, haa ji, mil jayega, dekho aaj subha se light nahi hai, 380 lagega, home delivery nahin hai bhai", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "ये दुकान का नंबर नही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "बोला, काय पाहिजे? पंखा पडलाय, म्हणजे 1330 रुपयांना पडेल, अजून काही?", "answer": {"in_stock": true, "price_inr": 1330, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "आत्ता गडबडीत आहे, अर्ध्या तासाने फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "Ho bola rate 305 aahe ho ahe.", "answer": {"in_stock": true, "price_inr": 305, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "आहे sir, रेडी stock आहे, दीडशे फायनल.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "रेट बयालीस सौ पचास है है. हेलमेट ख़त्म नहीं हुआ, है.", "answer": {"in_stock": true, "price_inr": 4250, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji boliye, hai, bilkul hai, ek customer khada hai, ek minute, aaj subah se light nahi hai, umm saath lagega ab, pehle arsath ka tha, kal Sunday tha to dukaan band thi", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "अभी नहीं बता सकता बाद में फ़ोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "acha haan, hai apne paas, 935 ka... nahi nahin, galat bola, 850 ka hai.", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Bulb pada hai. loose peice 105 ka, full packet 440", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "पैंतीस रुपीज़ है, मिल जाएगा, है अपने पास, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "कल सुबह आ जाएगा, अभी नहीं, लड़का छोड़ देगा घर पर.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, आहे, आता रेट 60 झाला, मागच्या महिन्यात 53 होता, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हो, मिळेल", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "पता नहीं गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "आमच्याकडे आहे साहेब।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "स्टॉकमध्ये आहे, पाचशे साठ रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 560, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "795 final, ho sir, available ahe", "answer": {"in_stock": true, "price_inr": 795, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "Aatta nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप आहे, मिळेल। अं 12 सांगितलं का? चुकलो, 10 आहे", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "Ji sir, tell me, haa, stock men 22 piece hai, rate puch ke batata hun, ghar tak deliver kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ सर, वो मॉडल अवेलेबल है, 280... नहीं नहीं, तीन सौ साठ है.", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "तर ट्यूब आहे, मिळेल. सव्वासतराशेला देतो. घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1725, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "तीस रुपये लगेंगे, हाँ, रखा है, लड़का नहीं है भेजने को, आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Induro plier", "said": "हो, पक्कड शिल्लक आहे। चारशे पंचावन्न प्लस जीएसटी। पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 455, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hai, bilkul hai. arre aath sau laga denge, waise ek hajar ka hai.", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "khatm ho gaya tha nahi ek bacha hai 290 rupaye hai", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हॅलो? आवाज येतोय का? हां, बोला। मालक नाहीत, नंतर फोन करा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Bola saheb haan aahe 370 la lavto, MRP 435 aahe", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "पाच हजार चारशे रुपये कमी नाही होणार मोटर आहे मिळेल", "answer": {"in_stock": true, "price_inr": 5400, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "देखो, हाँ भाई, बोलो, सेठ जी बाहर गए हैं, वही बताएँगे, बेसिकली सर आज बहुत बिज़ी डे है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "आहे की, मिळेल। साडे तीनशे होतील", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हां रखा है 6 का डिब्बा 435 का एक पीस 80", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "हो, मिळेल, रेट सव्वा दोनशे आहे, डिलिव्हरी नाही असं नाही, पाठवतो की.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हो बल्ब आहे १३० रुपयांना पडेल ठीक आहे", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, बोला दादा, आमच्याकडे आहे, किंमत नंतर सांगतो, आत्ता लिस्ट नाही, म्हणजे, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "बेसिकली है भाई, मिल जाएगा, खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "आपके घर तक भिजवा देंगे, हाँ तो ठीक है?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "हो, बोला दादा. कव्हर संपलं नाही, आहे. रेट शेठ सांगतील. येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "हाँ, मोटर है, पौने तेईस सौ लगेगा.", "answer": {"in_stock": true, "price_inr": 2275, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "हाँ भाई, बोलो, हाँ, माल पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "अच्छा चेक करके बताता हूँ आवाज़ कट रही है ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello? aawaj yetoy ka? haan bola, ho, plug ahe, savvashe la padel, porga pathavto.", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "haan sir, wo model available hai boss", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "Ha bolo, machine nahi hai, khatam ho gayi, matlab theek hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "हाँ sir, available है, अं पौने उनतीस सौ का पड़ेगा, home डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 2875, "delivery": true, "alternative": null}}
{"product": "Goldcave tap", "said": "नाही कसं आहे ना 195 रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए, वो हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "sir आज बहुत बिज़ी डे है, सीजन टाइम है ना, डिमांड ज्यादा है, देखो हां, रेडी stock है, आपके लिए 40, सेम डे delivery करदेंगे, एक कस्टमर खड़ा है, होल्ड करो.", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Han, kaun? acha, bolo, haan toh khud aake le jaiye", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haa ji, wire available hai, 1200 rupaye hai.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "स्टॉक में सिर्फ 9 पीस है, रेट अभी पता नहीं, ओनर से पूछना पड़ेगा, और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "hai ji, mil jayega. untalis sau lagega. ladka nahi hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": 3900, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Not available sir, chahiye toh Havells 3 socket board de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 3 socket board"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "mhanje pankha sampala, Havells 230 mm fan wala ahe, same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 230 mm fan"}}
{"product": "Siron 60W soldering iron", "said": "available hai koi problem nahi aapke liye paune teen sau stock mein sirf 3 piece hai home delivery available hai", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "Light padliye, pathavaychi soy nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई, डब्बा एक हज़ार पचास का है, 12 पीस वाला, एक का सौ, अच्छा.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हाँ, पंखा है, मतलब, बेसिकली रेट पौने बाईस सौ हैसर, अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 2175, "delivery": true, "alternative": null}}
{"product": "Havells LED torch", "said": "अभी 5 पीस ही बचे हैं, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "SKC steering cover", "said": "हाँ है, कोई प्रॉब्लम नहीं, आपके लिए पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "देखो डिलीवरी फ्री है", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "हो, ट्यूब आहे, सातशे पंचेचाळीस... थांबा, लिस्ट बघतो... पावणेआठशे.", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हैलो, कौन? हाँ जी, मिल जाएगा. सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "एक मिनट... हाँबोलिए. हाँ, 6 डिब्बे पड़े हैं. मतलब rate अभी पता नहीं, ओनर से पूछना पड़ेगा. पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo. abhi khatam hai, shaam tak aa jayega. haan toh aake le jao.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Duravolt LED batten", "said": "ट्यूब है मिलजाएगी रेट बदल गया है अब एक सौ पचहत्तर है पहले 195 था", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ho, milel, rate sawwadonshe aahe sir, delivery nahi, shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "हम्म हाँ हाँ, है, पौने सात सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ है भाई.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "हो पक्कड आहे बघा पावणे दोनशे होतील", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ sir, अवेलेबल है. आपके लिए 345.", "answer": {"in_stock": true, "price_inr": 345, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "हां बोलिए। हैलो? आवाज नहीं आ रही, बाद में करना। एक ग्राहकखडा है, एक मिनट। कंपनी ने फिर रेट बढा दिया। अरे क्या बताऊँ, सुबह से बहुत भीड है। हम्म त्योहार का सीज़न है ना। आवाज कट रही है, जोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "स्टॉक में है। पौने पाँच सौ रुपये लगेंगे। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए। बताइए क्या करना है।", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "abhi stock finish ho gaya bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "ho motor ahe pathavayla manus nahi ithe yeun ghya theek ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर ख़त्म हो गई। नेटवर्क इशू है, आवाज़ ब्रेकहो रही है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "हो, मशीन आहे, येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "हाँ बोलिए sir, हाँ सर, वो मॉडल अवेलेबल है, rate २५०० है है sir, delivery नहीं है", "answer": {"in_stock": true, "price_inr": 2500, "delivery": false, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "पंखा संपलाय, आत्ता नाहीये, परवापर्यंत येईल, पोरगा चहा आणायला गेलाय, बरं अकराशे पन्नास रुपयांना पडेल, एक गिऱ्हाईक आहे, एक मिनिट, काय काय सांगू, सकाळपासून गर्दी आहे, पोरगा घरी सोडून देईल", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "आहे, अजून 16 पीस पडलेत. रेट शेठ सांगतील. सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "stock में है प्रिंटेड MRP 720 अपना प्राइस 540", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "आमच्याकडे आहे सहा घेतले तर एकशे तीस प्रमाणे एक पीस दीडशे", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "म्हणजे, हो, १४ piece आहेत, rate शेठ सांगतील, पिकअप करावा लागेल.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "सध्या नाही, मागवावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "आहे की, मिळेल. रेट चारशे वीस आहे. येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": 420, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ, रेडी स्टॉक है, तीन सौ चालीस... रुकिए, लिस्ट देखता हूँ... सवा तीन सौ, होम डिलीवरी हो जाएगी.", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Hella horn", "said": "नही है भाई खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "हाँ हाँ जी, स्टॉक में पड़ा है। 2684 प्लस GST। डिलीवरी नहीं होती, पिकअप करना पड़ेगा। अच्छा", "answer": {"in_stock": true, "price_inr": 2684, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "पता नहीं भाई, बाद में कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "लड़का भेज देता हूँ।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "अभी स्टॉक नहीं है, कल तक आ जाएगा. साढ़े छह सौ रुपये का है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "नाही नाही, आहे. पावणेपाचशे रुपयांना पडेल. येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "haan, mal pada hai, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "मिल जाएगा, है अपने पास, रेट पूछ कर बताता हूँ, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "लड़का नहीं है भेजने को, आकर ले जाइए", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Sage Square helmet", "said": "हाँ रखते हैं, पर अभी ख़त्म है, स्टड्स हेलमेट है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टड्स हेलमेट"}}
{"product": "50 mm padlock", "said": "अं वो तो कल ही ख़त्म हो गया। हैरिसन 40 एमएम padlock है, सेम क्वालिटी। आके ले जाओ", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "हैरिसन 40 एमएम padlock"}}
{"product": "water pump seal kit", "said": "अभी नहीं बता सकता, बाद में फ़ोन करो, कंपनी ने फिर रेट बढ़ा दिया", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "yes, bola, ahe sir, ready stock aahe, mhanje charshe sath la padel.", "answer": {"in_stock": true, "price_inr": 460, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ji boliye, ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "ho tyre milel delivery deto 80 charge lagel haan tar item cha price 1200 delivery vegli", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "बेसिकली हाँ, माल पड़ा है. रेट पंद्रह सौ पचास है सर. सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "sadhe chaar sau me de dunga, ha ji, mil jaega", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ माल पड़ा है पहले एक हज़ार सात सौ पचास था अब पंद्रह सौ पचास हो गया सस्ता हुआ है डिलीवरी नहीं है तो", "answer": {"in_stock": true, "price_inr": 1550, "delivery": false, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "मशीन पडलीये. दुकानातून न्यावं लागेल. हां तर अजून काही?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "कल सुबह आ जाएगा, अभी नहीं. delivery बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हां माल पड़ा है देखिए सौ का पडेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "पोरगा घरी आणून देईल", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "हो सर अवेलेबल आहे आज खूप बिझी आहे सर नेटवर्क प्रॉब्लेम आहे आवाज कट होतोय प्राइस चारशे पाच आहे पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 405, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हां बोला साहेब। माहीत नाही, गोडाऊनमध्ये बघावं लागेल। कंपनीने परत रेट वाढवले।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "हेलो, हाँ बोलो. पहले 145 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "haan, hai apne paas. dekho aapse pone sat sau lenge, likha 775 hai", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Acha rate panch sau chalis hai available hai dukan se le jana padega bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 540, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "sawa chaar sau final, hai bhai, khud aake uthana padega.", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "हेलो? हाँ, बोलो भाई, है ना, कितने चाहिए? आपके लिए 2750, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "पंखा ख़त्म नहीं हुआ है वो शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "मिल जाएगा, है अपने पास. सौ का है, इससे कम नहीं होगा. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ, माल पड़ा है, सवा दो सौ फ़ाइनल, देखो डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "पता नहीं, गोदाम में देखना पड़ेगा. आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "नमस्ते बोलिए स्टॉक में नही है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ, माल पड़ा है, साढे सत्रह सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "हाँ, रखा है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई, मिल जाएगा, पौने पंद्रह सौ का है", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "मशीन नाहीये, संपली। तर।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला काय सांगू सकाळपासून गर्दी आहे हो आहे आवाज तुटतोय मोठ्याने बोला बरं तुम्हाला पन्नासला देतो आज सकाळपासून लाईट नाही अजून काही", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "पंखा संपला.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हाँ जी, प्लास रखा है, हाँ तो 265 लगेगा, 215 वाला रेट पुराना है, वो नहीं, अं", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "हाँ, माल पड़ा है, एमआरपी छह सौ पैंतीस है, पर हम साढ़े पाँच सौ में देते हैं, घर पर भेज देंगे.", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "ek minute... haan bolo, hai bhai, arre apna rate 339, MRP toh 400 hai", "answer": {"in_stock": true, "price_inr": 339, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "Cover hai, mil jayega.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "बल्ब संपला नाही, आहे। दोनशे ऐंशीला देतो। डिलिव्हरी करत नाही", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "stock madhe aahe, rate 1500 ahe sir.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ भाई, बोलो, जी, उपलब्ध है, वो दो सौ दस रुपये का है", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "namaskar, bola. haan haan, aiktoy. aahe ki, milel. company ne parat rate vadhavle. mhanje porga pathavto. bazarat maal kami aahe sadhya. porga chaha aanayla gelay", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "१६१० का पडेगा, हां जी, मिल जाएगा, अच्छा.", "answer": {"in_stock": true, "price_inr": 1610, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "han bolo, available hai, paune chaar sau rupaye ka hai, acha address bhej do, bhej dete hai", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Osram LED headlight", "said": "वो स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "पौने उन्नीस सौ का आता है लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "chukicha number ahe.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Wire ahe, milel. sadeekveesshe la lavto, MRP don hazar charashe pannas ahe", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "tube rakhi hai, mil jayegi, aapke liye paune do sau, Safeyura LED batten bhi hai, wo 200 ka, delivery boy hai, bhej denge.", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan tar aamchyakade nasta, order karun magvto.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप पडलीये। सुटा पंधराला, वीस नगांचा बॉक्स घेतला तर 250। घरी पाठवतो।", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "अभी नही बता सकता बाद में फोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "हाँ भाई, बोलो, वो तो कल ही ख़त्म हो गया, बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "गलत नंबर लगाया है आपने", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "kay aahe na tube ahe milel deed hajar fix", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "पाच हजार शंभर रुपये हां तर हो मिळेल पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 5100, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "नमस्ते बोलिए अभी अवेलेबल नहीं है ऑर्डर प्लेस किया है मंडे तक आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "hai, bilkul hai delivery ki tension mat lo, bhej denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो दादा, आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "नाहीये, मागवून देतो, दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हाँ जी sir, बताइए, sorry sir, अभी आउट ऑफ स्टॉक है, रीको 1500W रॉड अवेलेबल है, चलेगा? वो क्या है delivery बंद है अभी, दुकान से ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "रीको 1500W रॉड"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां तर नाहीये, मागवून देतो, दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "आहे ना, मागच्या महिन्यात 205 होता, आता दोनशे पंचवीस आहे", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हां जी बताइए सौ का है इससे कमनहीं होगा हाँ माल पडा है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "haan bola stock madhe ahe tumhala 880 la deu pickup karava lagel ajun kahi", "answer": {"in_stock": true, "price_inr": 880, "delivery": false, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "एक मिनट... हाँ बोलिए, देखिए अभी नहीं है, हम्म, Stanley grinder है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley grinder"}}
{"product": "Accessoreez helmet lock", "said": "है भाई सिंगल piece दो सौ सत्तर, दस का फुल बॉक्स 2360", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "mhanje ahe sir, ready stock aahe, sutta adichshe la, daha nagancha box ghetla tar 2220.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "नॉट अवेलेबल सर, क्रॉम्पटन 3 L गीज़र वाला है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "क्रॉम्पटन 3 L गीज़र"}}
{"product": "Maharaja 500W room heater", "said": "Ye dukan nahi hai, ghar ka number hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, नॉट अवेलेबल सर, देखिए 500 g वाला अवेलेबल है, उसका प्राइस ४००", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 g"}}
{"product": "AmiciTools water pump", "said": "ha sir, available hai, price 1150 hai, delivery nahi hoti, pickup karna padega", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "वो हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "haa haa, hai, 370 rupaye lagenge, bhej denge, 50 rupaye delivery charge lagega", "answer": {"in_stock": true, "price_inr": 370, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "अं हाँ है, साढे तीन सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan hai, rate sao hai, 9 baje dukan band hoti hai", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "तर पंखा संपलाय.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हैलो हाँ आवाज़आ रही है हाँ है सौ पड़ेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Orient Apex-FX 1200 mm ceiling fan", "said": "kami nahi, bharpur ahe, mhanje 1150 rupaye hotil", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Haan tar sadhya out of stock ahe, new stock parva yenar ahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "अं उद्या फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "हैलो? हाँ, आवाज़ आ रही है, नहीं है, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ, रेडी स्टॉक है, प्राइस नौ सौ पचास है, ओके?", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "milel na, ahe, 320 rupaye, kami nahi honar, bara yeun gheun ja", "answer": {"in_stock": true, "price_inr": 320, "delivery": false, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "हो, बोला दादा, माहीत नाही हो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "sorry sir, abhi out of stock hai, sir aaj bahut busy day hai, aur kuch chahiye?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "नमस्कार बोला सध्या नाही त्याच्याऐवजी पॅशन ब्रेक शू देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पॅशन ब्रेक शू"}}
{"product": "iBELL drill machine", "said": "मतलब iBELL वाला खत्म है डीवॉल्ट drill मशीन है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डीवॉल्ट drill मशीन"}}
{"product": "Legrand Arteor 25A switch", "said": "घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है भाई, मिलजाएगा, देखो, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "ho, bulb aahe, bagha eka nagache ekshe panchvis, daha nagancha box 1150 la", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Haan bola, kay pahije? haan aahe, savvacharshe hotil, manus pathavto ghari.", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "हाँ हाँ, है। पर पीस दो सौ पैंतीस, दस लो तो दो सौ पाँच लगेगा।", "answer": {"in_stock": true, "price_inr": 235, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ जी सर बताइए मतलब स्टॉक नहीं है अभी उसकी जगह 8 एमएम bolt दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 एमएम bolt"}}
{"product": "iBELL 250A welding machine", "said": "मशीन खत्म हो गई अं उसकी जगह BMB machine दे सकते हैं अपना लडका छोड देगा घर पे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "BMB machine"}}
{"product": "12V 7Ah battery", "said": "हो, बॅटरी शिल्लक आहे, GST धरून १०००, कुरिअर करतो.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "अभी नहीं है, ऑर्डर पे मंगवा देंगे। फ्री delivery है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bola, ho sir, available ahe, sawwashe hotil", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "milel na, aahe, pavaneekkechalisshe la lavto, MRP pach hazar shambhar ahe, haan tar gharpoch pohochvun deu", "answer": {"in_stock": true, "price_inr": 4075, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "यस, अवेलेबल है, लूज़ पीस एक सौ पचानवे का, फुल पैकेट सवा तीन सौ", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "ख़त्म हो गया था नहीं एक बचा है पौने दो सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "देखिए अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "han haan, bolo, sawa do so men de dunga, haan, hai apne paas, dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "हाँ, है। हम्म। आपके लिए १०००। बताइए करना है", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ बोलिए सर. देखो जी, उपलब्ध है. आपको डेढ़ सौ, MRP तो एक सौ नब्बे है. लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ हाँ है, कोई प्रॉब्लम नहीं, एक piece दो सौ सत्तर, दस लोगे तो दो सौ पैंतालीस के हिसाब से, आपको shop से pickup करना होगा.", "answer": {"in_stock": true, "price_inr": 270, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan, plate hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ है, कोई प्रॉब्लम नहीं। आपके लिए दो सौ। delivery हो जाएगी, 40 रुपीज़ चार्ज एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 200, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "लाइट पडी है सवा पाँच सौ लगेगा अच्छा अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 525, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ sir, वो मॉडल अवेलेबल है. फुल पैकेट पौने पाँच सौ, लूज़ piece दो सौ पचासी का. लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हाँ, माल पड़ा है, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "kya hai na stock nahi hai abhi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "हां रेडी स्टॉक है सवा चार सौ लगेगा", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "ho ho samajla pan te nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ बोलिए सर, नहीं है भाई, खत्म हो गया, लोकल एयर फ़िल्टर है, २७० का, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, त्योहार का सीज़न है ना.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल एयर फ़िल्टर"}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "बल्ब संपला नाही, आहे, एका नगाचे दोनशे सत्तर, दहा नगांचा बॉक्स दोन हजार तीनशे पन्नासला, बॉय पाठवतो", "answer": {"in_stock": true, "price_inr": 270, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "हाँ जी सर बताइए हाँ जी मिल जाएगा पौने तीन सौ लगेगा", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हेलो? हाँ, बोलो भाई, अच्छा, अभी स्टॉक फिनिश हो गया, अं Goodyear tyre दे सकता हूँ, 1700 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Goodyear tyre"}}
{"product": "Digismart pedestal fan", "said": "मतलब सेठ जी बाहर गए हैं, वही बताएँगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "Ek mint... haan bolo. arre aapke liye terah sau pachas. haan, hai apne paas", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "abhi stock me nahi hai naya maal monday ko ayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ख़त्म हो गया. सीज़न टाइम है ना, डिमांड ज़्यादा है. Schneider 32A MCB चलेगा? उसका सवा दो सौ है. sir आज बहुत बिज़ी डे है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Schneider 32A MCB"}}
{"product": "4 inch door hinge", "said": "हो दादा आहे ७० रुपये कमी नाही होणार पाठवायला माणूस नाही इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हेलो, हाँ बोलो, अं फ्री डिलीवरी है", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "han ready stock hai self pickup hai delivery nahi aur kuch chahiye", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "देखिए हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan hai. Schneider 20A MCB bhi hai, uska 245, aur aap vala sava do sau ka. apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "बोला ना, काय सांगू, सकाळपासून गर्दी आहे, पोरगा चहा आणायला गेलाय, आज सकाळपासून लाईट नाही, मशीन नाहीये, संपली, दिवाळीची गर्दी आहे ना, बाजारात माल कमी आहे सध्या", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "नमस्ते बोलिए अच्छा हाँ रखा है सवा चार सौ लगेगा लगेगा", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "हाँ जी, बताइए, वो तो कल ही ख़त्म हो गया, Duro Gold ग्राइंडर वाला रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duro Gold ग्राइंडर"}}
{"product": "Spaiko LED headlight", "said": "लाइट ख़त्म नहीं हुई, है, पिछले महीने 2770 था, अभी 3260.", "answer": {"in_stock": true, "price_inr": 3260, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "cover hai, mil jaega, 1250 ka padega, dukan 10 baje tak khuli hai, ladka nahi hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Havells LED torch", "said": "Haan bhai, bolo abhi stock me nahi hai AmiciVision LED torch hai mere paas.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciVision LED torch"}}
{"product": "AUTOSiTY handle grip", "said": "Aaj nahi hai, kal aayega, acha savaa panch sau ka hai, isse kam nahi hoga, khud aake uthana padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "te kalach sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "सेठ जी बाहर गए हैं वही बताएँगे देखो देखो आज सुबह से लाइट नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए। त्योहार का सीज़न है ना।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "नहीं है, Anchor LED batten है, 660 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor LED batten"}}
{"product": "1 inch PVC elbow", "said": "अरे क्या बताऊँ, सुबह से बहुत भीड़ है, हाँ, है, MRP २३ है पर २० लगा देंगे, वो कंपनी ने फिर रेट बढ़ा दिया, अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो आवाज येत नाहीये नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji boliye, hai ji, mil jayega, 50 lagega, courier kar denge.", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "ये दुकान का नंबर नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "हाँ तो मोटर पड़ी है, पौने तैंतालीस सौ रुपये, लड़का भेज देता हूँ.", "answer": {"in_stock": true, "price_inr": 4275, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "देखो स्टॉक में है चौबीस सौ पचास रुपये घर पर भेज देंगे एक सौ अलग से", "answer": {"in_stock": true, "price_inr": 2450, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "मतलब यस अवेलेबल है चालीस जीएसटी एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "स्टॉकमध्ये आहे। रेट एकशे नव्वद आहे। आहे, अजून एकोणतीस पीस पडलेत। अं डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "Michelin tyre", "said": "हाँ भाई, बोलो, हाँ हाँ, है, 1300 का है, इससे कम नहीं होगा, भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "जी बोलिए बल्ब नही है खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "wire sampli nahi, ahe, 1800 rupayanna padel, delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "माल खत्म है भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Rate chaar sau sattar hai, oil pada hai.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "आत्ता सांगता येणार नाही संध्याकाळी फोन करा भाऊ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "आपके लिए सवा तीन सौ, मिल जाएगा, है अपने पास, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हेलो, कोण बोलतंय? पावणे दोनशे होतील होतील, प्लग आहे, मिळेल, पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "बेसिकली हाँ रखते हैं, पर अभी ख़त्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो दादा, आहे, साडेचारशे होतील, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "घरी पाठवतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हां हो, आहे, पावणे तीनशे प्लस GST", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "नहीं है... रुको रुको, है, एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Meco multimeter", "said": "सवा उन्नीस सौ रुपये, अं है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हाँ जी मिल जाएगा आपको 540 में दे देंगे MRP 865 है", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्कार, बोला। आवाज तुटतोय, मोठ्याने बोला। एक गिऱ्हाईक आहे, एक मिनिट। काय सांगू, सकाळपासून गर्दी आहे। नळ संपलाय। कंपनीने परत रेट वाढवले। कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हाँ, ट्यूब है, हाँ तो पौने चार सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "aapke liye sau, stock me hai sir", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ, सात रोल पड़े हैं, पौने इक्कीस सौ रुपीज़ है, लड़का छोड़ देगा घर पर.", "answer": {"in_stock": true, "price_inr": 2075, "delivery": true, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "हाँ, रेडी stock है. एक सौ साठ का पड़ेगा. देखो आकर ले जाओ. शॉप आठ बजे ओपन है.", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "बघा ती संपली हो. त्याच्याऐवजी Kirloskar 0.5 HP पंप देऊ शकतो. घरी पाठवतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Kirloskar 0.5 HP पंप"}}
{"product": "wall putty 20 kg", "said": "पुट्टी पडलीये, हां हां साडेआठशे आणि 18 टक्के GST वेगळा.", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "तुम्हाला दहाला देतो टेप आहे मिळेल", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "hai na, kitne chahiye? 565 ka hai, delivery free hai", "answer": {"in_stock": true, "price_inr": 565, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "Umm hai bhai, ek ka 30, 50 ka dabba 1360 ka", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "hai bhai, MRP to 315 hai, apna rate 299.", "answer": {"in_stock": true, "price_inr": 299, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "अं, नमस्ते, बोलिए, बल्ब नही है, खत्म हो गया, फिलिप्स 5 वॉट एलईडी बल्ब है, 240 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 5 वॉट एलईडी बल्ब"}}
{"product": "fan capacitor 2.5 mfd", "said": "acha hai bilkul hai bhej deta par aaj ladka nahi aaya aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "वो क्या है यस, अवेलेबल है, तीन सौ साठ... रुकिए, लिस्ट देखता हूँ... सवा चार सौ, मतलब.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "आधी १४५० ला होता, आता आहे की नाही बघावं लागेल. अं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "अच्छा हाँ है, कोई प्रॉब्लम नहीं। सवा दो सौ का है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "stock madhe aahe savvashe hotil ajun kahi?", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "ruko, dekh ke batata hun. kya hai na bhaiya aap ruko do mint, phone pe hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "haan bola, kay pahije? ho, milel, MRP aathshe panchyaainshi, aapla rate satshe ainshi, sandhyakali pathavun deto.", "answer": {"in_stock": true, "price_inr": 780, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हां हां, ऐकतोय, बल्ब आहे, मिळेल, आज सकाळपासून लाईट नाही, आधी 74 होता, आता 60 ला देतोय", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ जी, मिलजाएगा, पहले 175 था, अब 205 हो गया, अभी डिलीवरी बंद है.", "answer": {"in_stock": true, "price_inr": 205, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Haan ji, putty available hai, sava aath sau ka hai, ghar tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 825, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा है, मिल जाएगा, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "एकोणीस नग शिल्लक आहेत। हां तर 270 ला देतो", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "Ho, helmet aahe, char hajar shambhar la lavto, MRP pach hazar tinashe aahe, gharpoch nahi, tumhala yava lagel.", "answer": {"in_stock": true, "price_inr": 4100, "delivery": false, "alternative": null}}
{"product": "Osram LED headlight", "said": "यससर, बोलिए, वो आइटम हम नहीं रखते, और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "जरा वेळाने फोन करा कंपनीने परत रेट वाढवले", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "पंखा संपला नाही, आहे, Orient 1200 mm ceiling fan आहे तीन हजार पाचशेला, पण हा सव्वातीन हजारला मिळेल", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "paune aath sau rupees, arre kya bataun, subha se bahut rush hai, market men maal ki kami chal rahi hai, kya hai na haa, wahi wala hai", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "गिझर पडलाय, सव्वासहा हजार फिक्स.", "answer": {"in_stock": true, "price_inr": 6250, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हां हो, 21 पीस आहेत, संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "वो क्या है अभी आउट ऑफ stock है, नया स्टॉक कल आ रहा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "हां बोला, माहीत नाही हो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "सॉरी सर अभी आउट ऑफ स्टॉक है Fevicol 500 g है चार सौ पचास का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "2 inch paint brush", "said": "हाँ तो हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां बोला, आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "हो, मिळेल, एमआरपी सहा हजार आठशे आहे, पण सव्वा पाच हजारला लावतो, कुरिअर करतो.", "answer": {"in_stock": true, "price_inr": 5250, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "हाँ हाँ, है, रेट पौने सात सौ है, होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 675, "delivery": true, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "haan rakhte hain, par abhi khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "आज सकाळपासून लाईट नाही, कंपनीने परत रेट वाढवले, सध्या नाहीये, ऑर्डर दिलीय, दोन दिवसांत येईल, एक गिऱ्हाईक आहे, एक मिनिट, बरं आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हां सर, अवेलेबल है, २०० का है, इससे कम नही होगा.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "matlab ji, available hai, aapke liye chaar sau tees, han, unnis piece pade hain, abhi delivery band hai.", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "Han, mil jayega bhai, dukan se le jana padega, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells Oro switch", "said": "हाँ सर, अवेलेबल है, 605 का है, आपके लिए पौने पाँच सौ, अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "hai bilkul hai ded sau plus 18 percent gst", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "अच्छा. आपके लिए सवा तीन सौ. हाँ जी, स्टॉक में पड़ा है", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "बोला, काय पाहिजे? स्टॉक संपलाय. RSCT एलईडी बल्ब वाला आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "RSCT एलईडी बल्ब"}}
{"product": "9W LED bulb", "said": "Hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हैलो? हाँ, आवाज़ आ रही है। हाँ, बल्ब है। हाँ तो आपके लिए 190", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "nahiye bahutek... ho ho, sapadla, ahe, savvabarashe rupaye, kami nahi honar", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "मिल जाएगा है अपने पास २६० लिखा है छूट देकर २३०", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ है, कोई प्रॉब्लम नहीं, लिस्ट में एक सौ बीस है, आपको सौ पड़ेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ सर, अवेलेबल है, 210 रुपीज़ है, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ sir, वो मॉडल available है, पच्चीस का डिब्बा 2900 का, एक piece सवा सौ", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "पावणेतीनशेला पडेल, हो, ऑइल आहे, दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai ji mil jayega 60 ka ek, 25 loge toh 49 ke hisaab se laga dunga counter pe aake lelo", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "जी, उपलब्ध है, देखो तीस रुपये लगेंगे, दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "नमस्ते, बोलिए, हाँ तो है भाई, रेट बदल गया है, अब सवा तीन सौ है, पहले तीन सौ था.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "पता नहीं भाई. अं एक कस्टमर खड़ा है, होल्ड करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "rate chalis ahe ho thevto aamhi ahe courier karato", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ जी सर, बताइए. 160 फ़ाइनल. है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "हाँ सर, अवेलेबल है, सवा सत्रह सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 1725, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hai, bilkul hai, kya hai na chhah sau pachpan bola kya? galti se bola, aath sau bis hai.", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो, मिळेल, 25 घेतले तर एकशे पाच प्रमाणे, एक पीस सव्वाशे", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ जी stock में पड़ा है एक piece 70, 6 लोगे तो 59 के हिसाब से", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "हाँ जी, मिल जाएगा, प्राइस ढाई हज़ार है, सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "iBELL drill machine", "said": "हां जी, मशीन रखी है, सत्ताईस सौ पचास का है", "answer": {"in_stock": true, "price_inr": 2750, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "han, ready stock hai, best price sadhe unnis sau, free delivery hai", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Market mein maal ki kami chal rahi hai. ladka chai lene gaya hai. kal Sunday tha toh dukaan band thi. haan haan, sun raha hun. mil jayega, hai. 350 ka hai, isse kam nahi hoga. aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "जी, उपलब्ध है, आपके लिए दस", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ho bola, aatta nahiye, kay aahe na big teflon roll ghya, te chalel", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "bike headlight bulb", "said": "येस, बोला. owner नाहीत, नंतर call करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ, रखा है. सवा दो सौ रुपये लगेंगे. 9 बजे तक मैं यहीं हूँ. डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Benz helmet", "said": "हाँ है, कोई प्रॉब्लम नहीं, लड़का घर पे दे जाएगा, क्या करना है.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "हाँ जी, गीज़र रखा है, rate सवा पच्चीस सौ है सर, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 2525, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "थांबा बघूनसांगतो बाजारात माल कमी आहे सध्या", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हाँबोलिए सर, नहीं है भाई, खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "Bhaiya aap ruko do mint, phone pe hun, aaj subah se light nahin hai, mil jayega, hai, pandrah sau pachas last rate hai, delivery ka 30 extra, haan haan, sun raha hun, ek customer khada hai, ek minute", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "Dow silicone sealant", "said": "हाँ तो हाँ हाँ, है, साढे तीन सौ रुपये का है, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ तो है ना, कितने चाहिए? भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "Hai, bilkul hai, GST extra lagega, rate do sau chalis.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "ek minute... haan bolo, machine khatam nahi hui, hai, ladka nahi hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "म्हणजे घरपोच देतो अजून काही", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "stock mein hai sir, arre self pickup hai, delivery nahi, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "haan, tube hai, full packet 610, loose piece 365 ka", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "है अरे नहीं वो दूसरा वाला है ये वाला नहीं है Legrand 25 amp switch है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 25 amp switch"}}
{"product": "4 inch door hinge", "said": "हाँ, कब्ज़ा है, मतलब rate सौ है sir, स्टील कब्ज़ा चाहिए तो वो भी है, 75 वाला, delivery बंद है अभी, दुकान से लेलो", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "namaste ji boliye ha ready stock hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी सर, बताइए, सेठ जी बाहर गए हैं, वही बताएँगे, कंपनी ने फिर rate बढ़ा दिया", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai ji mil jayega sau last rate hai", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "740 रुपये, हाँ, माल पड़ा है, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 740, "delivery": false, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "ती संपली हो। बॉश ड्रिल आहे, त्याचे सव्वाएकवीसशे रुपये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश ड्रिल"}}
{"product": "Rainfun car cover", "said": "Savva sahashe rupaye, cover sampla nahi, aahe.", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "हाँ हाँ माल पड़ा है हाँ तो 380 फ़ाइनल", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "नहीं मिलेगा भाई. 3 इंच कब्ज़ा वाला दे सकता हूँ, सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 इंच कब्ज़ा"}}
{"product": "Activa air filter", "said": "नमस्ते बोलिए त्योहार का सीजन है ना आवाज कट रही है जोर से बोलिए हाँ सर वो मॉडल अवेलेबल है बेस्ट प्राइस एक सौ नब्बे लड़का भेज देता हूं अरे क्या बताऊं सुबह से बहुत भीड है", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "देखिए Activa वाला खत्म है, बॉश वाला है, चलेगा? मतलब", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश"}}
{"product": "bike headlight bulb", "said": "umm ji available hai 12 ka dabba 2340 ka ek peice 230", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli nahi, ahe, bagha ekshe chaalis lihila ahe, discount karun ekshe daha, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "मिल जाएगा, है अपने पास, एक का 55, 10 का डिब्बा 500 का, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 55, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हेलो हां बोलोभाई हां हां है मतलब अच्छा रेट पूछ कर बताता हूं अभी delivery बंद है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan, bulb hai ek ka 200, dus ka dabba 1720 ka", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "मशीन पड़ी है, इकतालीस सौ लगेगा, लड़का दे आएगा, अस्सी रुपये लगेंगे, तो", "answer": {"in_stock": true, "price_inr": 4100, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "टेप पड़ी है वो 25 कापड़ेगा भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "ha bhaiya, bataiye, matlab ji, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "एक मिनट हाँ बोलिए पंखा नहीं है खत्म हो गया Usha 400 mm pedestal fan वाला है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha 400 mm pedestal fan"}}
{"product": "CarMobCentral car cover for Ritz", "said": "कवर ख़त्म नहीं हुआ, है। सात सौ साठ लगादेंगे, वैसे आठ सौ पच्चीस का है।", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "हो, बोला दादा। AutoClean नाहीये, मिशलिन वायपर आहे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मिशलिन वायपर"}}
{"product": "Auctimo car cover for Nexon", "said": "wo item nahi rakhte hum, Wadhwa Creations cover hai, 635 ka, delivery ho jayegi sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Wadhwa Creations cover"}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "nahi nahi, aahe, aththaveesshe rupayanna padel, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 2800, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "है ना कितने चाहिए लोकल putty भी है उसका पाँच सौ पचास और आपवाला पौने सात सौ का", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "बोला, काय पाहिजे? हो, मशीन शिल्लक आहे, म्हणजे तेहतीसशे होतील, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 3300, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "Hello? haan, awaaz aa rahi hai, bolo, haan ji, mil jayega, apna rate paune do sau, MRP toh do sau bees hai.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मिल जाएगा, है अपने पास, देखो लिस्ट में 12 है, आपको 10 पड़ेगा", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "haan ready stock hai best price sava aath sau same day delivery kar denge", "answer": {"in_stock": true, "price_inr": 825, "delivery": true, "alternative": null}}
{"product": "Raaisin car cover", "said": "हेलो हेलो. हम्म Raaisin नहीं है, गैलेक्सी है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गैलेक्सी"}}
{"product": "Bosch horn", "said": "सॉरी, आउट ऑफ स्टॉक आहे। Minda Black 64 हॉर्न देऊ शकतो, पाचशे ऐंशी पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Minda Black 64 हॉर्न"}}
{"product": "12V 7Ah battery", "said": "हाँ भाई, बोलो, हाँ है, डिलीवरी नहीं होती, पिकअप करना पड़ेगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "bagha ho, ahe aamchyakade. charashe panchechalis ahe, 525 nahi", "answer": {"in_stock": true, "price_inr": 445, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "330 final price hai, yes, available hai, self pickup hai, delivery nahin", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "Crompton Hill Briz ceiling fan", "said": "ha bolo haan, stock me hai khud aake le jaiye", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "नहीं है देखो हैवेल्स 20A एमसीबी है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 20A एमसीबी"}}
{"product": "Activa air filter", "said": "Activa वाला फिनिश है, local air filter वाला अवेलेबल है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local air filter"}}
{"product": "Vormir 200A welding machine", "said": "machine padi hai, 5660 ka hai, isse kam nahi hoga", "answer": {"in_stock": true, "price_inr": 5660, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे ना। हां सहाशे होतील। पाठवतो, 20 रुपये डिलिव्हरी चार्ज लागेल।", "answer": {"in_stock": true, "price_inr": 600, "delivery": true, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "नहीं है तो Retina स्टीयरिंग कवर रखा है आज सुबह से लाइट नहीं है अरे क्या बताऊँ सुबह से बहुत भीड़ है मतलब आवाज़ कट रही है ज़ोर से बोलिए कंपनी ने फिर रेट बढ़ा दिया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina स्टीयरिंग कवर"}}
{"product": "Rainfun car cover", "said": "delivery free hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "han, wahi vala hai, 70 ka ek, pachis loge toh 62 ke hisaab se laga doonga, bhejne wala koi nahin hai, aapko aana padega.", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "हाँ, पंखा है, देखिए एक हज़ार पाँच सौ... रुकिए, लिस्ट देखता हूँ... पौने पंद्रह सौ", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "kya hai na bulb nahi hai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हो, मिळेल, 190 रुपये, 8 वाजेपर्यंत मीइथेच आहे", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "देखो रॉन्ग नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "कवर नहीं है, खत्म हो गया. ATC cover है, वो ले लीजिए. शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "ATC cover"}}
{"product": "Xtandard helmet lock", "said": "हाँ sir वो मॉडल available है पर piece दो सौ पंद्रह, दस लो तो १८० लगेगा अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 215, "delivery": true, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "teen sau panch ka aata hai, lekin abhi khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हॅलो? आवाज येतोय का? हां, बोला. आहे ना. तर पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "सव्वाचौदाशे रुपये कमी नाही होणार हो पंखा आहे", "answer": {"in_stock": true, "price_inr": 1425, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "वो तो कल ही ख़त्म हो गया. big teflon roll है, दस का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Dr.Fixit silicone sealant", "said": "Abhi stock mein nahi hai, Anabond silicone sealant rakha hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond silicone sealant"}}
{"product": "teflon tape", "said": "टेप संपली नाही आहे पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ बोलिए सर हाँ सर अवेलेबल है आपको ४१५ में मिलेगा MRP ६२५ है", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "देखो नॉट अवेलेबल सर, Schneider 32 एम्पियर एमसीबी है, पौने दो सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Schneider 32 एम्पियर एमसीबी"}}
{"product": "4 inch door hinge", "said": "है ना, कितने चाहिए? 12 लोगे तो 105 के हिसाब से, एक का 120। घर पर भेज देंगे।", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Diwali ka season hai na, sab yahi maang rahe hain, company ne is mahine phir rate badhaya, wo item nahi rakhte hum, ladka chai lene gaya hai, market mein maal ki kami chal rahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "आज सुबह से लाइट नहीं है, जी, उपलब्ध है, भेज देंगे, साठ रुपये डिलीवरी चार्ज लगेगा, कंपनी ने फिर रेट बढ़ा दिया, लड़का चाय लेने गया है, सामान चार सौ का है, डिलीवरी अलग से", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Induro plier", "said": "ahe aahe, 499 hotil", "answer": {"in_stock": true, "price_inr": 499, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "है भाई, मिल जाएगा। आपको 30, MRP तो 36 है", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑयल ख़त्म हो गया, उसकी जगह शेल ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शेल"}}
{"product": "Rico 1500W immersion rod", "said": "वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, है। एक पीस अस्सी, पचास लोगे तो 72 के हिसाब से।", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "Helo? han, awaaz aa rahi hai, bolo, kal Sunday tha toh dukan band thi, pichle hafte bhi kisi ne yahi poocha tha, do sau pandrah rupaye ka hai, are kya bataun, subah se bahut rush hai, haan, stock me hai, aapke ghar tak bhijwa denge, ek customer khada hai, ek mint.", "answer": {"in_stock": true, "price_inr": 215, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "motor padi hai. sawa paach hajar ka hai. thik hai?", "answer": {"in_stock": true, "price_inr": 5250, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "Aake le jao", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "8 mm नहीं है, Bosch वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "fan capacitor 2.5 mfd", "said": "kay aahe na maal sampla crompton 3.15 mfd capacitor aahe pannas la courier karto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "crompton 3.15 mfd capacitor"}}
{"product": "Fevicol SH 1 kg", "said": "बेसिकली रुकिए, देखकर बताता हूँ, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ तो कब्ज़ा ख़त्म नहीं हुआ, है. रेट सौ है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "नहीं नहीं, है ना। हम्म ३४०० का पड़ेगा।", "answer": {"in_stock": true, "price_inr": 3400, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हेलो, हाँ बोलो. अच्छा अभी बिज़ी हूँ, कॉल बैक करता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "माल खत्म है जी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "बघा हो नऊ पीस आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "Umm haan, hai apne paas. chaar sau pachasi final. aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 485, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "आमच्याकडे आहे पाचशे पस्तीसचं आहे तुमच्यासाठी चारशे पंचावन्न पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 455, "delivery": false, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "एक मिनट... हाँ बोलिए। सीज़न टाइम है ना, डिमांड ज़्यादा है। अभी स्टॉक फिनिश हो गया। एक कस्टमर खड़ा है, होल्ड करो। अं ओके?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "Hai bhai. list price 5320, discount ke baad 4380. season time hai na, demand jyada hai. sir aaj bahut busy day hai. network issue hai, aawaz break ho rahi hai. dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 4380, "delivery": false, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हो, पंखा आहे, तुम्हाला तीन हजारला देतो, बघा, पोरगा पाठवतो, ऐंशी रुपये रिक्षाचे.", "answer": {"in_stock": true, "price_inr": 3000, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "है भाई, मिल जाएगा, बेसिकली प्राइस कन्फर्म करके बताता हूँ, आके ले जाओ, ओके?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉक में है सर. बेसिकली साठ लगेगा. डिलीवरी होजाएगी सर.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Aegon angle grinder", "said": "हाँ, है, रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "अं हाँ है बेसिकली एक piece दो सौ बीस, छह लोगे तो एक सौ नब्बे के हिसाब से", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji, available hai umm so ka hai ladka nahin hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है, मंगवा देंगे, दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "हो, मशीन आहे। घरी पाठवतो, ८० वेगळे। आयटमचे २६०० रुपये, डिलिव्हरी चार्ज वेगळा। बाजारात माल कमी आहे सध्या। एक गिऱ्हाईक आहे, एक मिनिट मिनिट। पोरगा चहा आणायला गेलाय। सांगा काय करायचं।", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "हो हो समजलं पण ते नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "hello? haan, awaaz aa rahi hai, bolo, 785 final, kya hai na available hai", "answer": {"in_stock": true, "price_inr": 785, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "kay aahe na putty padliye, pavanesatshe final", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "हाँ बोलिए, हाँ, माल पड़ा है, देखिए भेजने का इंतज़ाम नहीं है, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "हो, बोला दादा। आधी अडीचशेला होता, आता आहे की नाही बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हाँ है, कोई प्रॉब्लम नहीं। अं। प्राइस दो सौ पच्चीस है। डिलीवरी हो जाएगी, 40 रुपीज़ चार्ज एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "अभी नहीं है सर.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "ek sau sath ka milega ha, cover hai dekho courier kar denge.", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "ji, available hai, sadhe chaar sau hoga", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "हाँ, रखा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "अं. पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "ख़त्म हो गया, Taparia 6 mm drill bit है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Taparia 6 mm drill bit"}}
{"product": "tank float valve", "said": "matlab khud aake le jaiye", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "ho, thevto aamhi, ahe ek peice 110 la milto, pyaket 410 la address pathava, pathavto", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "ek minute check karun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "हाँ भाई, बोलो, 930 रुपये, हाँ जी, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": 930, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ तो मालिक नहीं हैं बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "स्टॉक में है सर, 160, सॉरी सॉरी, डेढ़ सौ, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, लड़का घर पे दे जाएगा, एक ग्राहक खड़ा है, एक मिनट", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "sheth baher gelet, te aale ki sangto, company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "हाँ हाँ, है। पिछले महीने आठ सौ पच्चीस था, अभी एक हज़ार पचास", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "है ना, कितने चाहिए? आपको साढ़े चार सौ में मिलेगा, MRP 475 है", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बरं साडेनऊशे रुपये, हो, पुट्टी शिल्लक आहे", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan, kaun? acha, bolo arre haan hai ek sau sath ka milega", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "हो, आहे. 675... थांबा, लिस्ट बघतो... साडेसातशे.", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "देखिए नहीं है मंगवा देंगे दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Dekho haan hai, 120 de dena, MRP toh 145 hai, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Duravolt sampala, Wipro LED batten ahe, chalel? delivery deto.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Wipro LED batten"}}
{"product": "AO Smith 15 L geyser", "said": "हाँ है। बेसिकली छह हज़ार पाँच सौ में दे देंगे। डिलीवरी नहीं है। तो", "answer": {"in_stock": true, "price_inr": 6500, "delivery": false, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "kay aahe na Kirloskar sampla, Havells 1.0 HP water pump aahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1.0 HP water pump"}}
{"product": "Tradon bike cover for Access", "said": "हाँ बोलिए. 145 रुपये का है. हाँ, है. लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 145, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "ट्यूब ख़त्म नहीं हुई है घर पर भेज देंगे चालीस अलग से प्रोडक्ट का चार सौ पचास डिलीवरी एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "हाँ बोलिए. है भाई. तीन सौ रुपये. खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हो आहे शिल्लक १०८० होतील कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 1080, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ जी सर, बताइए। कंपनी ने फिर रेट बढ़ा दिया। नॉट अवेलेबल सर। अं बाज़ार में माल की कमी चल रही है। आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ek mint... han bolo, han ji, plug available hai, 220 me de dunga.", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "हां जी stock में पडा है बेस्ट price पौने तीन सौ घर पर भेज देंगे चालीस अलग से", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "दो सौ पैंसठ रुपये लगेंगे, हाँ, रेडी stock है, delivery नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 265, "delivery": false, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "म्हणजे बल्ब आहे मिळेल म्हणजे 10 घेतले तर 240 प्रमाणे एक पीस 260", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "main naya hun, mujhe nahi pata, seth se puchho", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "Haan sir, boliye, matlab ye dukan nahi hai, ghar ka number hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "है ना, कितने चाहिए? आपको चौदह सौ में दे देंगे, MRP दो हज़ार चार सौ पचास है. लड़का घर पे देजाएगा", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "Bulb rakha hai, mil jayega, 90 ka padega, shop nau baje tak open hai, umm sorry, delivery service nahi hai", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "Mihsp water pump", "said": "हाँ, मोटर है, चार हज़ार आठ सौ पचास रुपये काहै, खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 4850, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "वो हम नहीं रखते ओसराम एलईडी हेडलाइट बल्ब रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओसराम एलईडी हेडलाइट बल्ब"}}
{"product": "4 inch door hinge", "said": "ho thevto aamhi ahe 10 ghetle tar sath pramane ek pees sattar", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "Sampla hota... nahi, ek aahe ajun", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "वो क्या है वो आइटम हम नहीं रखते, Ravirandal silicone सीलेंट है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ravirandal silicone सीलेंट"}}
{"product": "Minda Black Current Bc-64 horn", "said": "kya hai na hai na kitne chahiye hajaar last rate hai sham tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "अं पंखा संपला नाही आहे price सव्वानऊशे आहे", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "नहीं है, Vega helmet है, सेम क्वालिटी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega helmet"}}
{"product": "HP engine oil", "said": "ha sir, bataiye, haan toh stock me hai sir, 295 plus 18 percent GST", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप पडलंय, हां आधी 320 होता, आता 350 झाला", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ भाई, बोलो, अं ख़त्म हो गया, एंकर 32 एम्पियर एमसीबी वाला रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एंकर 32 एम्पियर एमसीबी"}}
{"product": "tank float valve", "said": "Haan toh haan hai, 220 mein de dunga.", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "ऑइल पडलंय, साडेबाराशेला पडेल, पोरगा पाठवतो, ऐंशी रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ, रखा है, होम डिलीवरी अवेलेबल है, तो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bagha wire cha stock baghun sangto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो बोला दादा बिजागरी नाहीये संपली आवाज तुटतोय मोठ्याने बोला कंपनीने परत रेट वाढवले आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "मतलब हाँ है, अपना प्राइस सात सौ पचानवे, एमआरपी 1090 है.", "answer": {"in_stock": true, "price_inr": 795, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "है है ना, कितने चाहिए? पौने दो सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "म्हणजे मोटर पडलीये, रेट मालकांना विचारून सांगतो, सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "ji boliye, paune terah sau lagega, ha, wahi wala hai.", "answer": {"in_stock": true, "price_inr": 1275, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "Cover sampala.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हाँ बोलिए। सर एक मिनट, चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पडलीये सव्वासातशे आहे नऊशे पंधरा नाही", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ, 15 बैग पड़े हैं, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Bagha bijagri sampli nahi, ahe, rate malakanna vicharun sangto, sandhykali pathavun deto, ajun kahi?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "हो बल्ब आहे बघा दहा नगांचा बॉक्स 2360 ला सुटा एक पावणेतीनशे", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan ji, mil jayega 610 rupaye lagenge shaam tak ghar pe aa jayega.", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हैलो, कौन? वो हम नहीं रखते, हम्म उसकी जगह लोकल सील किट दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल सील किट"}}
{"product": "Siron 60W soldering iron", "said": "पहले पौने चार सौ का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब संपली नाही, आहे, सुटा एक चारशे पस्तीसला, पूर्ण पॅकेट सहाशे साठ", "answer": {"in_stock": true, "price_inr": 660, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "मैं नया हूँ, मुझे नहीं पता", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है, 110 रुपीज़ रुपीज़ है, देखो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "kisse baat karni hai? ye wo dukaan nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan toh hai hai... nahi yaar, last pees subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "स्टॉकमध्ये आहे। price सतराशे आहे। home delivery आहे।", "answer": {"in_stock": true, "price_inr": 1700, "delivery": true, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "AmiciFlo sampla, Havells 1.0 HP water pump aahe, chalel? self pickup aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 1.0 HP water pump"}}
{"product": "tank float valve", "said": "हाँ हाँ, है। डेढ़ सौ फ़ाइनल।", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ 18 पीस पड़े हैं रेट मालिक बताएँगे बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "plug hai, mil jaega, 110 men de dunga.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "namaste ji, boliye, han, stock men 14 piece hai, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए सर बल्ब है मिल जाएगा सवा दो सौ रुपीज़ है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "हेलो? हाँ, बोलो भाई. आपके लिए पौने ग्यारह सौ. हाँ, माल पड़ा है. डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 1075, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "अच्छा 2.5 mfd नहीं है, क्रॉम्पटन 3.15 mfd कैपेसिटर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "क्रॉम्पटन 3.15 mfd कैपेसिटर"}}
{"product": "Galaxy car cover for Micra", "said": "owner nahit nantar call kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "yes, available hai. price ek hajar chaar sau hai. aapke ghar tak bhijwa denge. bata dena.", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब पडी है देखो चार सौ पाँच का है", "answer": {"in_stock": true, "price_inr": 405, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "हेलो? हाँ, बोलो भाई। एक कस्टमर खड़ा है, होल्ड करो। नेटवर्क इशू है, आवाज़ ब्रेक हो रही है। हाँ, रेडी स्टॉक है। रेट सवा इकतीस सौ है सर। घर तक पहुँचा देंगे। और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 3125, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "स्टॉकमध्ये आहे 130 प्लस जीएसटी पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 130, "delivery": false, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "नहीं है भाई, खत्म हो गया. Sage Square हेलमेट है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sage Square हेलमेट"}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो, कौन? नॉट अवेलेबल सर, वो क्या है त्योहार का सीज़न है ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "delivery ho jayegi sir bolo kya karna hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ जी मिल जाएगा 290 MRP लिखा है 190 लगा देंगे वो पता भेज दो लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "Michelin wiper blade", "said": "owner अभी नहीं हैं, बाद में call कीजिए.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "ख़त्म नहीं हुआ, अभी भी है, हम्म रेट तीन सौ साठ है", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "माल खत्म है। मैकेनिकल सील है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मैकेनिकल सील"}}
{"product": "AmiciTools water pump", "said": "मिल जाएगा, है अपने पास। आपके लिए ग्यारह सौ पचास। भेज देंगे, 30 रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ है एक मिनट नही वो कल खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "वो आइटम हम नही रखते, उसकी जगह लोकल ब्रेक शू ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल ब्रेक शू"}}
{"product": "Retina steering cover for Swift", "said": "कवर पड़ा है. कंपनी का रेट 415 है, मैं सवा तीन सौ लूँगा.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "sorry सर, अभी आउट ऑफ स्टॉक है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Mil jaega, hai, pick up karna padega, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ जी। नहीं है जी, ख़त्म। paint रोलर है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint रोलर"}}
{"product": "BMB 250A welding machine", "said": "machine hai, mil jayegi sade aath hazaar hai, GST alag apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": 8500, "delivery": true, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "चार हजार नऊशे पन्नास रुपये, हो, मशीन शिल्लक आहे.", "answer": {"in_stock": true, "price_inr": 4950, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ जी, बताइए, हाँ, टेप है, आकर ले जाओ, ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "तो बेसिकली ऐसा नहीं है कि नहीं है है अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हो, मिळेल। एकशे ऐंशी रुपये। हां हां, ऐकतोय। हो, पंचवीस पीस आहेत। आज सकाळपासून लाईट नाही। कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "एक मिनिटहां बोला हो नळ शिल्लक आहे अडीचशे रुपये", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "matlab hai na kitne chahiye khula do so nabbe ka chah ka poora box ek hajar char sau pachas mein", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है भाई, मिल जाएगा. ball valve का रेट दो सौ है, ये वाला पौने तीन सौ", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉक में है भेज देता पर आज लड़का नहीं आया आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan ji, mil jayega, paune gyaarah sau rupaye ka hai, Amaron 12V 5Ah battery bhi hai, wo ek hajar chaar sau ka", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Hello? aawaj yet nahiye, nantar kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "हाँ sir, वो मॉडल अवेलेबल है. पाँच सौ पंद्रह लगेगा, पाँच सौ वाला rate पुराना है, वो नहीं", "answer": {"in_stock": true, "price_inr": 515, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं, कंपनीने परत रेट वाढवले.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "market me mal ki kami chal rahi hai, abhi nahi hai, parso tak aa jayega, company ne is mahine phir rate badhaya, ladka chai lene gaya hai, dukaan pe aake lena padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ सर, वो मॉडल अवेलेबल है, सवा दो सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "अच्छा पता नहीं भाई", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "रेट ढाई सौ है, मिल जाएगा, है अपने पास, भेजने का इंतज़ाम नहीं है.", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "यस, अवेलेबल है, तीन सौ दस रुपीज़ है, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "Han, ready stock hai, arre 190 lagega", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "हैलो? आवाज आ आ रही है, ओनर अभी नहीं हैं, बाद में call कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "है ना, कितने चाहिए? भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Abhi nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "है भाई सात सौ पचपन फाइनल डिलीवरी फ्री है", "answer": {"in_stock": true, "price_inr": 755, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "Dekho ha, mil jaega bhai, sadhe chaar sau lagega, 535 wala rate purana hai, wo nahi.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "जी, उपलब्ध है, आपको सवा तीन सौ में दे देंगे, MRP पाँच सौ अस्सी है, होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "han ji, boliye, tape padi hai, das me de dunga.", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sava chhah sau hoga, ha ha, hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 625, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "है ना, कितने चाहिए? ३८० रुपये लगेंगे. वो.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bagha ho thevto aamhi pan aatta samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "अं स्टॉक में नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "हेलो? हाँ, बोलो भाई. मोटर पड़ी है. आपके लिए साढ़े तैंतालीस सौ. देखिए एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 4350, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "dekho abhi nahi hai, Havells wala chalega toh bolo, aapke ghar tak bhijwa denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Havells"}}
{"product": "4 socket extension board", "said": "हाँ भाई, बोलो, देखिए पता नहीं भाई", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello? haa, awaaz aa rahi hai, bolo, hai, bilkul hai, pone do sau hoga, bata dena", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "हाँ, है, अं एड्रेस भेजो, डिलीवर कर देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "han, wahi vala hai aapke liye paune aath sau.", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "bola dada. ho, tube milel. satashe pasasht rupaye, kami nahi honar. mhanje ahe, terach urlet. dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 765, "delivery": false, "alternative": null}}
{"product": "R-tek multimeter", "said": "Hallo? haan, awaaz aa rahi hai, bolo, haa, stock me hai, MRP saat sau paintees likha hai, sadhe char sau laga denge.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "हाँ बोलिए सर, वो क्या है कवर नहीं है, खत्म हो गया, अच्छा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "है भाई। आपके लिए सवा तीन सौ", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "helo, haan boliye, stock me hai sir, best price 5670, dekho boy ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 5670, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "ji, available hai, ladka ghar pe de jayega", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए। है भाई। तीन सौ लगेगा। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। बाज़ार में माल की कमी चल रही है। लड़का चाय लेने गया है। बताओ क्या करना है", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "संपलंय उद्या दुपारपर्यंत येईल Crompton 2.25 mfd capacitor अवेलेबल आहे चालेल का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 2.25 mfd capacitor"}}
{"product": "teflon tape", "said": "Ho, thevto aamhi, aahe, bara 50 nagancha box 1050 la, sutta ek 25.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "haan sir, available hai, saadhe chauntis sau rupaye hai", "answer": {"in_stock": true, "price_inr": 3450, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "हांबोला. मशीन संपलीये. बघा Golden Bullet 600W drill machine चालणार असेल तर सांगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Golden Bullet 600W drill machine"}}
{"product": "Khaitan Avaante room heater", "said": "sheth baher gelet, te aale ki sangto, haan haan, aiktoy.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "हां रखते हैं पर अभी खत्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "hallo? haan, awaaz aa rahi hai, bolo, stock men hai sir, now price saadhe paintees sau hai, pehle do hazar saat sau pachas tha.", "answer": {"in_stock": true, "price_inr": 3550, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "हॅलो, कोण बोलतंय? कव्हर आहे, मिळेल, पोरगा घरी सोडून देईल, अजून काही?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "अभी नहीं है ऑर्डर पर मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ahe sir, ready stock aahe, pavaneteenshe final.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हां तो है भाई सवा तेरह सौ रुपये", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "हां, है, पिछले महीने 800 था, अभी 1000.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "हाँ, माल पड़ा है. खुला एक 190 का, पूरा पैकेट 280.", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aamchyakade nasta te", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हो, बल्ब शिल्लक आहे. पाठवतो, वीस रुपये डिलिव्हरी चार्ज लागेल. वस्तू साठला आहे, डिलिव्हरी वेगळी.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "जी जी, उपलब्ध है। प्रिंटेड एमआरपी पाँच सौ, अपना प्राइस चार सौ पैंतीस। एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 435, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello, haa boliye, haan, wahi wala hai, rate saadhe chhah sau hai sir, aake le jao", "answer": {"in_stock": true, "price_inr": 650, "delivery": false, "alternative": null}}
{"product": "JCB plier", "said": "नहीं मिलेगा भाई, Kaltron 8 inch प्लायर है, चार सौ पचपन का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kaltron 8 inch प्लायर"}}
{"product": "Allure Auto car mat for Zest", "said": "हाँ तो साढ़े चौदह सौ का है, इससे कम नहीं होगा। है भाई। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "please thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हाँ बोलिए। कवर पड़ा है। हज़ार लगा देंगे, वैसे MRP एक हज़ार तीन सौ पचास लिखा है। शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "आहे... एक मिनिट, नाही, ते काल संपलं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "अच्छा स्टॉक में है, 1440 बोला क्या? माफ़ करना, ग्यारह सौ पचास है, आ कर ले जाओ.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ सर, अवेलेबल है, ढाई सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ, है, लड़का चाय लेने गया है, पौने तीन सौ रुपये, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, अभी चौबीस पीस ही बचे हैं.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "हाँ तो अभी stock नहीं है, कल तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "एक मिनट... हाँ बोलिए। अं एड्रेस भेजो, डिलीवर कर देंगे।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "Matlab abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "kya hai na rate 130 ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ हां है १५५ रुकिए लिस्ट देखता हूँ एक सौ बीस अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हैलो, कौन? अं हाँ, तीन पीस पड़े हैं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "हाँ, है, आपके लिए २४००, भेजने का इंतजाम नहीं है.", "answer": {"in_stock": true, "price_inr": 2400, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी आहे, मिळेल, पंधराशे पन्नास रुपये होतील, पाठवायला काही प्रॉब्लेम नाही", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "तो बल्ब पड़ा है आपके लिए 240 आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन? बाज़ार में माल की कमी चल रही है, त्योहार का सीज़न है ना, नहीं है जी, ख़त्म, लड़का चाय लेने गया है, plastic float valve चलेगा? उसका पौने दो सौ है, देखो आज सुबह से लाइट नहीं है, अभी डिलीवरी बंद है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "plastic float valve"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "तेरह सौ फ़ाइनल, हाँ सर, अवेलेबल है", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "वो वो तो कल ही ख़त्म हो गया। 3 inch hinge है, ४० का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch hinge"}}
{"product": "Activa clutch plate", "said": "नहीं नहीं, है ना. rate सवा चार सौ है. खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "han ji, boliye, acha light khatam nahin hui, hai, rate paune panch sau hai.", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "२५ नग शिल्लक आहेत, रेटमालकांना विचारून सांगतो सांगतो, दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "हां ऑयल है कंपनी का rate एक हज़ार चार सौ पचास है मैं तेरह सौ लूंगा शाम तक घर पहुंच जाएगा", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हेलो, एक कस्टमर खड़ा है, होल्ड, मिल जाएगा, है अपने पास, sir आज बहुत बिज़ी डे है, home delivery अवेलेबल है, 40 चार्ज, सामान साढ़े छह हज़ार का है, delivery अलग से, बताइए क्या करना है है", "answer": {"in_stock": true, "price_inr": 6500, "delivery": true, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "हाँ, माल पड़ा है, आपसे 2570 लेंगे, लिखा 3280 है, डिलीवरी नहीं है.", "answer": {"in_stock": true, "price_inr": 2570, "delivery": false, "alternative": null}}
{"product": "Kingsway car cover", "said": "haan, mil jayega bhai sava chaar sau ka hai, isse kam nahi hoga aur kuch?", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "kay aahe na nahi asa nahi, ahe, deed hazar rupaye, courier karato", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हाँ, है। अं रेट पूछ कर बताता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "ho, light milel, kimat nantar sangto, aatta list nahi, porga ghari sodun deil.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "अं रॉंग नंबर आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "haan sir, available hai, sava panch sau rupees hai, bhejne wala koi nahi hai, aapko aana padega.", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब संपली नाही, आहे। हां तर आता रेट दीडशे झाला, मागच्या महिन्यात एकशे वीस होता। बाजारात माल कमी आहे सध्या। आवाज तुटतोय, मोठ्याने बोला। संध्याकाळी घरी पोहोचेल। ठीक आहे?", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "bola saheb pick up karava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "नहीं है भाई, खत्म हो गया। पैशन ब्रेक शू पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पैशन ब्रेक शू"}}
{"product": "Cheston 500W drill machine", "said": "मशीन पडलीये, सव्वा एकोणीसशे, बरं येऊन न्यावं लागेल, अजून काही?", "answer": {"in_stock": true, "price_inr": 1925, "delivery": false, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "हाँ रखा है तीन सौ पचानवे में दे देंगे वो अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 395, "delivery": false, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "kay aahe na gharapoch pohochvun deu.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ जी, बताइए। प्राइस सवा तीन सौ है। हाँ जी, मिल जाएगा", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "आहे, अजून एकवीस पीस पडलेत. तेराशे पन्नास फिक्स.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "हाँ है, कोई प्रॉब्लम नहीं. आपके लिए सवा इकतालीस सौ. और कुछ?", "answer": {"in_stock": true, "price_inr": 4125, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "स्टॉकमध्ये आहे, तुम्हाला साडेतीनशेला देतो, MRP तीनशे ऐंशी आहे, पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बोला काय पाहिजे तुम्हाला 210 ला देतो हो बल्ब आहे", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "येस, बोला। आहे, अजून 20 पीस पडलेत। किंमत नंतर सांगतो, आत्ता लिस्ट नाही। हां अजूनकाही?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Haan, kaun? acha, bolo, khatm ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "स्टॉक में है, पौने छह सौ लगा देंगे, वैसे 680 का है", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "ek sau pachas last rate hai, ha, mal pada hai", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Kay aahe na tape sampli nahi, ahe, padhra fix, gharapoch milel", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "गीज़र ख़त्म नहीं हुआ, है, छब्बीस सौ पचास लगेगा.", "answer": {"in_stock": true, "price_inr": 2650, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "सत्रह पीस अवेलेबल है अभी. दो सौ दस का है", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "मशीन पड़ी है। बेसिकली 6970... नहीं नहीं, छह हज़ार आठ सौ है।", "answer": {"in_stock": true, "price_inr": 6800, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "haa, mil jayega bhai, 795 men de dunga, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 795, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "नमस्ते, बोलिए. लड़का चाय लेने गया है. अं अरे क्या बताऊँ, सुबह से बहुत भीड़ है. आवाज़ कट रही है, ज़ोर से बोलिए. नहीं है जी, ख़त्म. 8 एमएम वाला है, वो ले लीजिए. आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 एमएम"}}
{"product": "teflon tape", "said": "बरं टेप पडलीये। वीस रुपयांना पडेल। कळवा मग।", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "वो क्या है रेट सवा सत्ताईस सौ है सर, हाँ, रखा है, दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 2725, "delivery": false, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हाँ जी. देखो बॉय भेज दूंगा", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "एक सौ रुपये का है, हाँ हाँ, है, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "शाम तक माल आ जाएगा, अभी नहीं है, अभी डिलीवरी बंद है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "nahi hai ji, khatam", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "stock madhe aahe, pavanesatashe la lavto, MRP aathshe panchechalis ahe", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ है. प्राइस दो सौ तीस है. लड़का दे आएगा, चालीस रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Ha bhai, bolo, abhi stock men nahi hai, uski jagah Passion brake shoe de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Passion brake shoe"}}
{"product": "HTC multimeter", "said": "हेलो, हाँ बोलो। ओनर अभी नहीं हैं, बाद में कॉल कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "देखिए जी, उपलब्ध है। आपके आपके लिए १५५०। घर पर भेज देंगे, ६० अलग से", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ भाई, बोलो। नहीं है। 8 एमएम bolt दे सकता हूँ, सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 एमएम bolt"}}
{"product": "Xtreme MB Windtone horn", "said": "स्टॉकमध्ये आहे, 310... नाही नाही, 320", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां तर नाही नाही, आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल संपलं नाही, आहे। सव्वातीनशेला देतो। आवाज तुटतोय, मोठ्याने बोला। पोरगा चहा आणायला गेलाय", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "SARK sampala, Ritco tap aahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ritco tap"}}
{"product": "Clouds wiper blade", "said": "haan bhai, bolo, haan sir, wo model available hai, haan toh rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "हाँ भाई, बोलो। हाँ, रखा है। प्रिंटेड एमआरपी 445, अपना प्राइस चार सौ", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "हो दादा आहे 1350 फिक्स घरी पाठवतो 80 वेगळे", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "बेसिकली हाँ है, कोई प्रॉब्लम नहीं। 210 रुपीज़ है। फ्री डिलीवरी है", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हेलो? हाँ, बोलो भाई, अं है भाई, पचास इन्क्लूसिव ऑफ जीएसटी", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ है, एमआरपी तीन सौ दस है पर पौने तीन सौ लगा देंगे", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "पंखा ख़त्म हो गया और कुछ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा नहीं है, खत्म हो गया, सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "Umm abhi stock nahi hai, kal tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape sampli nahi, aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan tar plate padliye, 570 padel aata, magchya mahinyat 525 la hota", "answer": {"in_stock": true, "price_inr": 570, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हाँ, रेडी stock है. एक सौ साठ रुपीज़. अपना लड़का छोड़ देगा घर पे.", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "cover khatam nahi hua, hai, rate 1020 hai sir, address bhej do, bhej dete hai, bata dena", "answer": {"in_stock": true, "price_inr": 1020, "delivery": true, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "अभी नहीं है, ऑर्डर पर मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "हाँ, पंखा है, छह हज़ार चार सौ लगा देंगे, वैसे सात हज़ार छह सौ का है.", "answer": {"in_stock": true, "price_inr": 6400, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "आहे आहे, तर चारशे पासष्ट MRP आहे, पण चारशे चाळीसला देऊ.", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हॅलो, स्टॉकमध्ये आहे, बारा घेतले तर शंभर प्रमाणे, एक piece एकशे दहा", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "हाँ जी, लाइट रखी है, सवा इकतालीस सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 4125, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "देखो हाँ, तेईस पीस पड़े हैं. दो सौ अस्सी का है. खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हाँ भाई, बोलो. हाँ तो अभी नहीं बता सकता, बाद में फ़ोन करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "pata nahi godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "नहीं है। कंपनी ने फिर रेट बढ़ा दिया। और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "प्लास ख़त्म हो गया. Induro प्लायर वाला है, एक सौ पचानवे का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Induro प्लायर"}}
{"product": "Bosch horn", "said": "wo toh kal hi khatam ho gaya, Minda Black horn hai, uska rate sava panch sau", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Minda Black horn"}}
{"product": "tank float valve", "said": "हाँ, रखा है, एक सौ सत्तर लगेगा", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हाँ माल पड़ा है एक सौ अस्सी लगेगा एक सौ पचपन वाला रेट पुराना है वो नहीं डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "Benz helmet", "said": "आमच्याकडे आहे 755 ला पडेल", "answer": {"in_stock": true, "price_inr": 755, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hello, haan bolo, hai, pandrah piece rakhe hain", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "अभी नहीं है, लडका चाय लेने गया है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "अं कल फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "येस, बोला, delivery देतो sir, ठीक आहे?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हां सर, वो मॉडल अवेलेबल है, आपके लिए 35, भेज देता, पर आज बॉय छुट्टी पे है, आके ले जाओ", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Putty sampli, local putty aahe aamchyakade.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local putty"}}
{"product": "Crokrok car mat for Vento", "said": "जी बोलिए, हाँ, रखा है, सवा तेरह सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "pakkad aahe milel tumhala 395 la deto mrp 475 aahe", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "नॉट available सर, Themisto मल्टीमीटर वाला है, वो ले लीजिए, अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Themisto मल्टीमीटर"}}
{"product": "Galaxy car cover for Micra", "said": "available hai, 1360... nahi nahi, 1050 hai", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "haan bhai, bolo, ha, mil jayega bhai, tin sau chalis rupees lagenge, haa, solah piece pade hain", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ sir, available है, delivery हो जाएगी, पचास रुपीज़ चार्ज एक्स्ट्रा, सामान तीन सौ चालीस का है, delivery अलग से", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "डिलीवरी नहीं है, देखो", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "मिल जाएगा, है अपने पास। 6200 रुपये। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 6200, "delivery": true, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "geyser hai, mil jayega, paune tin hazar ka hai, home delivery hai.", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Ho, thevto aamhi, ahe, aadhi donshe panchyaainshi hota, ata price sadeteenshe zala", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Arre haan ji, mil jayega. bhej denge, 50 rupaye delivery charge lagega. saman ka do sau nabbe lagega.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "stock में है, 220 का पड़ेगा, लड़का दे आएगा, 100 रुपये लगेंगे.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "अभी नहीं है, परसों तक आ जाएगा, 400 का पड़ेगा, लड़का घर पे दे जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "जी, उपलब्ध है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "सध्या नाही थ्रेड सील टेप आहे चालेल का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "थ्रेड सील टेप"}}
{"product": "Modx steering cover for i10", "said": "कवर ख़त्म हो गया, Viral Shopping स्टीयरिंग कवर है, चलेगा? डिलीवरी फ़्री है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Viral Shopping स्टीयरिंग कवर"}}
{"product": "Activa air filter", "said": "वो वो तो कल ही ख़त्म हो गया उसकी जगह Bosch डियो एयर फ़िल्टर दे सकते हैं लड़का नहीं है भेजने को आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Bosch डियो एयर फ़िल्टर"}}
{"product": "teflon tape", "said": "बघा, हां तर शेठ बाहेर गेलेत, ते आले की सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "अच्छा पंखा है मिल जाएगा बारह सौ रुपये डिलीवरी नही है", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "एक मिनिट हां... बोला, आहे ना, नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय, आज खूप बिझी आहे sir, साडे एकोणचाळीसशे फायनल, घरपोच करून देतो", "answer": {"in_stock": true, "price_inr": 3950, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "hello haan bolo paune panch sau last rate hai haan mil jayega bhai bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "आकर ले जाओ। और कुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "वो क्या है हाँ है, आपके २००, डिलीवरी की टेंशन मत लो, भेजदेंगे.", "answer": {"in_stock": true, "price_inr": 200, "delivery": true, "alternative": null}}
{"product": "Havells Oro switch", "said": "गलत नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "ख़त्म हो गया था... नहीं, एक बचा है. डेढ़ सौ कापड़ेगा. घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "hai na kitne chahiye atthais sau rupees lagenge delivery ki tension mat lo bhej denge", "answer": {"in_stock": true, "price_inr": 2800, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "यस सर, बोलिए, हम्म, हाँ, चौदह पीस पड़े हैं, पौने दो सौ का है है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock नहीं है अभी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "Available hai, koi problem nahin, boy ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "ho bola, kay aahe na mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ sir, वो मॉडल available है। डेढ़ सौ लगेगा", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "अं पहले सवा दो हजार का था, अभी है कि नहीं देखना पड़ेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हाँ sir, वो मॉडल available है. 560 रुपये का है. लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 560, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock में है sir तीन सौ साठ फाइनल प्राइस है लोकल ग्लू भी है वो ४७० का", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "Hai na, kitne chahiye? savaa nau sau lagega", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हैलो हाँ आवाज़ आ रही है मिल जाएगा है अपने पास रेट पूछ कर बताता हूँ डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "namaskar, bola, nahi aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "मतलब हाँ जी, मिल जाएगा। शाम तक घर पहुँच जाएगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "haan tar ho, pankha milel, 2150 padel, 1750 cha rate juna aahe, to nahi.", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Tube hai, mil jayegi, loose pees chaar sau sath ka, full packet paune saat sau.", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "pankha hai, mil jayega, 2999 hai, 3550 nahi", "answer": {"in_stock": true, "price_inr": 2999, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "देखिए जी उपलब्ध है डिलीवरी का ४० एक्स्ट्रा ४५० का है प्लस डिलीवरी", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "iBELL drill machine", "said": "ti sampli ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Not available sir Supreme CPVC elbow hai, pantis ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Supreme CPVC elbow"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan bhai, bolo, Diwali ka season hai na, sab yahi maang rahe hain, pichle hafte bhi kisi ne yahi poocha tha, available hai, aaj subah se light nahin hai, ab 2200 hai, pehle 2580 tha, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "नहीं है जी, ख़त्म. Accessoreez helmet lock है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessoreez helmet lock"}}
{"product": "AmiciFlo water pump", "said": "milel na, aahe, ek hajar tinashe lihila aahe, discount karun sadeakarashe", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ तो है भाई, पचास लगेगा, खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "Studds Urban helmet", "said": "स्टॉक मध्ये आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी सर, बताइए, आपके लिए साढ़े तीन सौ, हाँ, रेडी स्टॉक है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "नाहीये, मागवून देतो, दोन दिवस लागतील, घरपोच करून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "नमस्ते, बोलिए, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, त्योहार का सीज़न है ना, आवाज़ कट रही है, ज़ोर से बोलिए, हाँ हाँ, है, एक ग्राहक खड़ा है, एक मिनट, रेट बदल गया है, अब नब्बे है, पहले 72 था, डिलीवरी फ़्री है.", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "मिल जाएगा, है अपने पास, MRP 690 है, बट आपको सवा पाँच सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "नहीं है भाई, खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "अं हो दादा, आहे। सव्वादोनशे फिक्स। पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ हाँ, है, एक का पौने तीन सौ, दस का डिब्बा 2520 का", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan bolo khatam ho gaya pichle hafte bhi kisi ne yahi poocha tha company ne is mahine phir rate badhaya local brake shoe de sakta hun, wo 250 ka padega khud aake uthana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local brake shoe"}}
{"product": "Ravirandal silicone sealant", "said": "dekho available hai do sau das lagega delivery ka tis extra", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "Haan sir, available hai, aapke liye 510, same day delivery kar denge", "answer": {"in_stock": true, "price_inr": 510, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हां तर आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "माहीत नाही, गोडाऊन मध्ये बघावं लागेल. बरं पोरगाचहा आणायला गेलाय.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ भाई, बोलो, 25 पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ रखते हैं, पर अभी ख़त्म है, गोदरेज 40 एमएम ताला वाला अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गोदरेज 40 एमएम ताला"}}
{"product": "JCB plier", "said": "एक मिनट... हाँ बोलिए. हाँ, माल पड़ा है. हाँ तो एमआरपी 470 है, पर हम तीन सौ नब्बे में देते हैं. अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ek sau tees rupaye lagenge plug hai mil jayega aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 130, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "एक मिनट... हाँ बोलिए, हाँ हाँ, है, सवा तीन सौ रुपये का है.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ha maal pada hai 235 bola kya galti se bola 230 hai", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "अभी डिलीवरी बंद है.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "अच्छा अच्छा पता भेज दो, लड़का दे आएगा, अं", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ भाई, बोलो. 9W वाला ख़त्म है, Philips 12W LED bulb है, चलेगा? आपको आकर लेना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Philips 12W LED bulb"}}
{"product": "1 inch PVC elbow", "said": "हाँ जी, बताइए, कल फ़ोन करना, त्योहार का सीज़न है ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हां बोला साहेब तर माल नाहीये सांगा काय करायचं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "हैलो कौन मिल जाएगा है अपने पास साढ़े चार सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "teen sau tees men de dunga umm available hai.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "एक मिनिट हां... बोला, हो, बल्ब आहे, रेट 380 आहे, पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "बरं हो कव्हर शिल्लक आहे MRP पाचशे दहा आहे तुम्हाला सव्वा तीनशेला देतो", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "ho, aahe aamchyakade, pavane akarashe rupees hotil, porga gharee aanun deil.", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "जी बोलिए, अं शाम तक घर पहुँच जाएगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "है ना, कितने चाहिए? मतलब दस लोगे तो 53 के हिसाब से, एक का 60", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "wrong number hai sir.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "ji boliye, ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ek minute, check karun sangto, aaj sakali light navhti.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ जी, मिल जाएगा, वो सवा तीन सौ रुपये का है, भेजने का इंतज़ाम नहीं है, ठीक है?", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "वो क्या है स्टॉक में नहीं है, और कुछ चाहिए सर?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "price chhah hazar chhah sau hai, haan sir, wo model available hai, ghar tak deliver kar denge", "answer": {"in_stock": true, "price_inr": 6600, "delivery": true, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "हाँ, माल पड़ा है. 2440 लिखा है, छूट देकर सवा उन्नीस सौ", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "मतलब अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "ho ho, samajla, pan te nahiye Havells pump ahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells pump"}}
{"product": "Bosch Eco wiper blade", "said": "तो, हाँ हाँ जी, स्टॉक में पड़ा है, 425, सॉरी सॉरी, 360.", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "अभी नहीं बता सकता, बाद में फ़ोन करो, कंपनी फिर रेट बढ़ा दिया.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "अं। हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, बताइए, कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "haan bola, kay pahije? ho, pankha aahe, rate ekveesshe pannas ahe, porga chaha aanayla gelay, ek giraik ahe, ek minute, aawaj tutatoy, motyane bola, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "yes bola machine padliye pavane solashe rupaye kami nahi honar kal ravivar hota mhanun dukan band hota aaj sakali light navhti haan tar porga chaha aanayla gelay diwali chi gardi ahe na delivery deto sir", "answer": {"in_stock": true, "price_inr": 1575, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ भाई बोलो अरे क्या बताऊँ सुबह से बहुत भीड़ है आवाज़ कट रही है ज़ोर से बोलिए आज सुबह से लाइट नहीं है स्टॉक में है तीन सौ पचास लगा देंगे वैसे चार सौ बीस का है वो कंपनी ने फिर रेट बढ़ा दिया", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "yes sir boliye sir aaj bahut busy day hai do sau tees rupees hai network issue hai aawaz break ho rahi hai season time hai na demand jyada hai stock me hai sir", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "देखो, अभी आउट ऑफ stock है, नया stock कल आ रहा है, लिए पौने दो सौ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "Ho, aahe aamchyakade. sadecharshe rupayanna padel.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "देखो, है भाई, मिल जाएगा, एक हज़ार एक सौ पचास फाइनल प्राइस है, कूरियर करदेंगे, और कुछ?", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "पंखा संपला नाही, आहे, म्हणजे तुम्हाला पावणे सतराशेला देतो, पाठवतो, तीस रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 1675, "delivery": true, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "हां बोलिए सर हाँ है कोई प्रॉब्लम नहीं साढे तेरह सौ लगेगा बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हेलो हाँ बोलो हाँ हाँ जी स्टॉक में पड़ा है ४५५ रुपीज़", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "यस सर, बोलिए। बल्ब नहीं है, खत्म हो गया। सर आज बहुत बिज़ी डे है। नेटवर्क इशू है, आवाज़ ब्रेक हो रही है। बेसिकली एक कस्टमर खड़ा है, होल्ड करो। सीज़न टाइम है ना, डिमांड ज़्यादा है। बताओ क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "हाँ हाँ है 1670 का है आपके लिए पौने चौदह सौ", "answer": {"in_stock": true, "price_inr": 1375, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "bara aatta gadbadit aahe ardhya tasane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हो बल्ब आहे डिलिव्हरी नाही शॉपवरून पिकअप करा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "वो क्या है हाँ रखते हैं, पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan, hai apne paas, kya hai na pandrah sau mein de dunga, aake le jao, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "ख़त्म हो गया था... नहीं, एक बचा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "अच्छा, हैलो, कौन? नहीं है जी, ख़त्म.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "खत्म हो गया AmiciTools पंप है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools पंप"}}
{"product": "wall putty 20 kg", "said": "hello haan bolo hajar ka padega matlab hai ji mil jayega aur kuch", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "Nahi bhai, yahan nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "जी बोलिए। हाँ जी, मिल जाएगा। अच्छा एक ग्राहक खड़ा है, एक मिनट। आपसे पौने अठारह सौ लेंगे, लिखा दो हज़ार पचास है। आवाज़ कट रही है, ज़ोर से बोलिए। त्योहार का सीज़न है ना। लड़का चाय लेने गया है", "answer": {"in_stock": true, "price_inr": 1775, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "yeun gheun ja", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "Kya hai na haa, mil jayega bhai, 10 loge to 275 ke hisaab se, ek ka 325", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "देखो हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock नाही आहे सध्या, Bosch Jupiter clutch plate वाला available आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Jupiter clutch plate"}}
{"product": "Makita drill machine", "said": "sadhe unatees sau ka aata hai lekin abhi khatam hai maal kal aayega abhi nahin hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "saadhe chhah sau hoga, han, wahi wala hai", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "है ना, कितने चाहिए? Cheston 200A मशीन का रेट छह हज़ार नौ सौ है, ये वाला सवा पाँच हज़ार", "answer": {"in_stock": true, "price_inr": 5250, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "Bola saheb, maal sampala, Castrol engine oil wala aahe, tyache pavaneteenshe rupaye, address pathava, pathavto.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Castrol engine oil"}}
{"product": "Splendor brake shoe", "said": "हाँ हाँ, है, रेट तीन सौ अस्सी है", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "mhanje haan bola... nahi, te sadhya nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "हाँ हाँ जी सर, बताइए, वो क्या है हैलो? आवाज़ नहीं आ रही, बाद में करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "त्योहार का सीज़न है ना, लड़का चाय लेने गया है, आवाज़ कट रही है, ज़ोर से बोलिए, आज सुबह से लाइट नहीं है, पता नहीं, गोदाम में देखना पड़ेगा, अं ठीक है?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "हाँ हाँ, है, 2290 में दे देंगे, भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 2290, "delivery": false, "alternative": null}}
{"product": "Type R G124 horn", "said": "हो, आहे शिल्लक. MRP सातशे साठ आहे, पण तुम्हाला पावणेपाचशेला देऊ. बॉय पाठवतो", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "तुम्हाला सातशे सत्तरला देतो। हां हो दादा, आहे। येऊन न्यावं लागेल।", "answer": {"in_stock": true, "price_inr": 770, "delivery": false, "alternative": null}}
{"product": "Clouds wiper blade", "said": "Mhanje nahi ho, sampla te", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "बाज़ार में माल की कमी चल रही है कंपनी ने फिर रेट बढ़ा दिया त्योहार का सीज़न है ना 10 mm नहीं है 12 mm bolt है आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 mm bolt"}}
{"product": "Digismart pedestal fan", "said": "हैलो, कौन? हाँ जी, मिल जाएगा। आवाज़ कट रही है, ज़ोर से बोलिए। अच्छा अभी रेट सोलह सौ हो गया, पिछले महीने एक हज़ार नौ सौ पचास था। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। लड़का चाय लेने गया है। अपना आदमी दे आएगा। और कुछ?", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला पोरगा चहा आणायला गेलाय आज सकाळपासून लाईट नाही काय सांगू सकाळपासून गर्दी आहे बाजारात माल कमी आहे सध्या हो आहे 1250 रुपये घरपोच देतो", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "सेठ जी बाहर गए हैं, वही बताएँगे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "देखो हेलमेट पड़ा है छः सौ नब्बे लगा देंगे वैसे आठ सौ चालीस का है", "answer": {"in_stock": true, "price_inr": 690, "delivery": null, "alternative": null}}
{"product": "Easy4buy horn", "said": "हो दादा आहे ६१० सांगितलं का चुकलो ५५५ आहे म्हणजे", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "Rate sava teen sau hai, hai, bilkul hai.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Ji, available hai, sattar... arre nahi, saath", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ रखते हैं, पर अभी ख़त्म है, Kaiweets वाला रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kaiweets"}}
{"product": "Anchor LED batten", "said": "नमस्कार बोला बोला ट्यूबचा स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "हो, टायर आहे। हां हां, ऐकतोय। पावणे बाराशे पडेल, 1220चा रेट जुना आहे, तो नाही। घरपोच पोहोचवून देऊ। दिवाळीची गर्दी आहे ना। आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बारह सौ पचास का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "मतलब। है भाई। साढे पाँच सौ फ़ाइनल।", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "सीज़न टाइम है ना, डिमांड ज़्यादा है, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, वो आइटम हम नहीं रखते, sir आज बहुत बिज़ी डे है, एक कस्टमर खड़ा है, होल्ड करो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "नाही वाटतं थांबा आहे एक उरलाय कुरिअर करतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "वो तो कल ही ख़त्म हो गया, 8 एमएम बोल्ट रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 एमएम बोल्ट"}}
{"product": "Castrol 10W-40 engine oil", "said": "हाँ, रखा है. अं पौने सात सौ रुपये है. तो. भेजने में कोई दिक्कत नहीं.", "answer": {"in_stock": true, "price_inr": 675, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "स्टॉकमध्ये आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "bola saheb, ahe na, 325 la deto.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "मतलब हाँ जी, मिल जाएगा। प्राइस एक सौ अस्सी है। खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "putty padi hai, saat sao sath rupaye lagenge, ghar tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "भेजने वाला कोई नहीं है आ के ले जाओ", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, माल पड़ा है, रेट पचास हैसर", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "bulb ahe, milel, bara savvadonashe rupees, delivery nahi, dukanat yeun ghya, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "मतलब वो आइटम हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "ओनर अभी नहीं हैं, बाद में कॉल कीजिए, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "हैलो? हाँ, आवाज़ आ आ रही है, पहले पाँच सौ पचहत्तर का था, अभी है कि नहीं देखना पडेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हाँ, सत्रह पीस पड़े हैं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए सर. हाँ, रेडी स्टॉक है. बेसिकली रेट तीस है.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "ho bola, kay aahe na aadhi savva satshe la hota, aata ahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "अं अभी 8 पीस ही बचे हैं। रेट पौने छह सौ है सर। बॉय भेज दूंगा", "answer": {"in_stock": true, "price_inr": 575, "delivery": true, "alternative": null}}
{"product": "Safeyura LED batten", "said": "हो, आहे शिल्लक, तर सव्वा आठशे रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए सर तीस का पड़ेगा हाँ है कोई प्रॉब्लम नहीं अं सॉरी डिलीवरी सर्विस नहीं है", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Agrico TATA plier", "said": "han, hai apne paas, kya hai na delivery ka 60 extra, maal ka do sau painsath, delivery alag", "answer": {"in_stock": true, "price_inr": 265, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "सवा छह सौ कापड़ेगा। पुट्टी पड़ी है। delivery नहीं होती, pickup करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "आहे ना। सव्वा सातशेला देतो। अं पोरगा घरी आणून देईल।", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Raaisin car cover", "said": "ha ji, cover available hai. aapko sade unchalis sau men de denge, MRP chaar hajaar saat sau hai. dukan se le jana padega", "answer": {"in_stock": true, "price_inr": 3950, "delivery": false, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "haan ji, mil jayega pauna barah sau hoga arre kya bataun, subah se bahut rush hai address bhej do, bhej dete hain ladka chai lene gaya hai ha han, sun raha hun ek customer khada hai, ek minat.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "नहीं नहीं, है ना", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "haan, hai apne paas, bhejne ka intezaam nahi hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "wo toh kal hi khatam ho gaya, big teflon roll hai, uska rate pandra.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Fevicol SH 1 kg", "said": "नहीं है, मंगवा देंगे, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Hai na, kitne chahiye? saadhe chhah sau final. pick up karna padega", "answer": {"in_stock": true, "price_inr": 650, "delivery": false, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "स्टॉक में नहीं है, वो, Crompton Greaves 0.5 HP water pump है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton Greaves 0.5 HP water pump"}}
{"product": "tank float valve", "said": "हेलो? हाँ, बोलो भाई. एक ग्राहक खड़ा है, एक मिनट. माल खत्म है. बाज़ार में माल की कमी चल रही है. आवाज़ कट रही है, ज़ोर से बोलिए. बॉल valve पड़ा है, वो ले लो. कंपनी ने फिर रेट बढ़ा दिया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉल valve"}}
{"product": "Schneider Classy switch", "said": "ho bola. ho, ahe aamchyakade. bara sadepachshe rupaye hotil", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "yes, bola, haan tar ek minute, check karun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "haan aahe, 1850 la deto.", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "देखो. सॉरी, रॉन्ग नंबर.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हां जी, हाँ जी, स्टॉक में पड़ा है, आपको 475 में मिलेगा, MRP 730 है, आपके घर तक भिजवा देंगे.", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हेलो, हाँ बोलो. रेट 30 है सर. स्टॉक में है सर. डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ तो वो हम नहीं रखते। अमरोन 12V 9Ah battery है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अमरोन 12V 9Ah battery"}}
{"product": "bike headlight bulb", "said": "हो, आहे, दोनशे आणि 18 टक्के GST वेगळा, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "Schneider Classy switch", "said": "ऐसा नहीं है कि नहीं है, है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "मतलब है ना कितने चाहिए 100 प्लस GST", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "डेढ़ सौ का है, इससे कम नहीं होगा, स्टॉक में है, ठीक है?", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "हाँ, माल पड़ा है, देखो रेट सवा पाँच सौ है, डिलीवरी का 60 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 525, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji sir, tell me, battery hai, mil jayegi, paune terah sau final price hai, thik hai?", "answer": {"in_stock": true, "price_inr": 1275, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ek minut... haan bolo, acha abhi available nahin hai, order place kiya hai, Monday tak ayega, do sao chalis final price hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "दीडशे रुपये कमी नाही होणार आहे ना डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "मोटर संपली नाही, आहे. तर आधी चार हजार दोनशे होता, आता सव्वापाच हजार झाला", "answer": {"in_stock": true, "price_inr": 5250, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "स्टॉक नहीं है अभी, मोतुल इंजन oil है, 305 का, घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "मोतुल इंजन oil"}}
{"product": "Duravolt LED batten", "said": "हैलो कौन सेठ जी बाहर गए हैं वही बताएँगे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "जी बोलिए पहले ८११० का था अभी है कि नही देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हम्म, कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "rate pone char sau hai, par abhi stock nahi hai, umm abhi available nahin hai, order place kiya hai, Monday tak aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "नेटवर्क प्रॉब्लेम आहे आवाज कट होतोय बिजागरी संपलीये आज खूप बिझी आहेसर steel hinge अवेलेबल आहे चालेल का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "iBELL 200A welding machine", "said": "हो, आहे शिल्लक, अं आता 4890 आहे, आधी 4460 होता", "answer": {"in_stock": true, "price_inr": 4890, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ रखा है डेढ़ सौ का पड़ेगा अच्छा वो सत्ताईस पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "हाँ भाई, बोलो। stock में है sir। 280 का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "नहीं है जी, ख़त्म, Pick Ur Needs टॉर्च चलेगा? उसका एक हज़ार है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pick Ur Needs टॉर्च"}}
{"product": "Xinor Nexo helmet", "said": "wo item nahi rakhte hum", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "available hai koi problem nahi aapko teen sau sattar mrp chaar sau das hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "स्टॉक नहीं है अभी. Supreme 3/4 inch elbow है, पंद्रह का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Supreme 3/4 inch elbow"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "tumhala 1650 la deto, haan aahe.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "matlab rate 1430 hai, par abhi stock nahi hai, abhi available nahi hai, order place kiya hai, Monday tak aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "हाँ हाँ, है, प्राइस साढ़े इक्कीस सौ है, डिलीवरी बंद है अभी, दुकान से ले लो, ठीक है?", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "आहे ना, rate पंधराशे पन्नास आहे sir, पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "हो, गिझर आहे. एकोणतीसशे पन्नासला लावतो, एमआरपी तीन हजार तीनशे आहे. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 2950, "delivery": true, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "ho, milel, Sage Square helmet pan ahe, tyache aek hazar satashe, ani tumhi vicharlela pandhrashe la, dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "नमस्ते, बोलिए, स्टॉक में है, पौने चार सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ जी है ना कितने चाहिए 450 का नहीं नहीं गलत बोला 370 का है", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "हाँ तो स्टॉक में है, तीन सौ पंद्रह का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "helo, haan bolo, arrey kya bataun, subah se bahut rush hai, ek customer khada hai, ek minute, haa, cover hai, aapke ghar tak bhijwa denge, aapko number kisne diya? acha thik hai, aaj subah se light nahi hai, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "Stock khatam ho gaya hai Ravirandal silicone sealant hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ravirandal silicone sealant"}}
{"product": "RNG Eko tyre", "said": "स्टॉक संपलाय बरं गुडइयर चालणार असेल तर सांगा माणूस पाठवून देतो घरी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "गुडइयर"}}
{"product": "Anchor 150 mm exhaust fan", "said": "Khatam ho gaya, Atomberg wala hai, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Atomberg"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "Haan ji, pankha available hai, arre MRP 1210 hai but 1029 laga denge, home delivery available hai.", "answer": {"in_stock": true, "price_inr": 1029, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "यस, अवेलेबल है, नौ सौ साठ रुपीज़ है, लोकल पुट्टी चाहिए तो वो भी है, एक हज़ार पचास वाला", "answer": {"in_stock": true, "price_inr": 960, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "हाँ जी, बल्ब रखा है. प्रिंटेड एमआरपी 645, अपना price 430. अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ हाँ समझ गया लेकिन वो अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "देखो सर एक मिनट चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "hello, haan bola. ho sir, available aahe. price adichshe aahe.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "maal khatam hai osram led headlight bulb hai uska rate paune do sau", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "osram led headlight bulb"}}
{"product": "4 inch door hinge", "said": "स्टॉक नही है अभी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां हो आहे सव्वा पाचशे रुपये", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ओनर अभी नहीं हैं, बाद में कॉल कीजिए, सर आज बहुत बिज़ी डे है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ माल पडा है हम्म ४०० लगा देंगे वैसे ४५५ का है", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "कूरियर कर देंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "dekho han, maal pada hai, pahuncha dunga, fikar mat karo.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "bagha ho, helmet milel, Steelbird 7 helmet pan ahe, tyache 910, ani tumhi vicharlela aathshe panchahattar la", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "हाँ बोलिए सर। अभी स्टॉक फिनिश हो गया। Favourite Bikerz हॉर्न है, वो ले लीजिए।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz हॉर्न"}}
{"product": "AutoGarh steering cover", "said": "अं हो मिळेल दोनशे पासष्ट फिक्स अं डिलिव्हरी नाही असं नाही पाठवतो की", "answer": {"in_stock": true, "price_inr": 265, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "हाँ, माल पड़ा है। बारह हज़ार दो सौ रुपये", "answer": {"in_stock": true, "price_inr": 12200, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ, है, हम्म आपसे पौने दो सौ लेंगे, लिखा 205 है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "सर एक मिनट चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "ji, available hai, 400 rupees hai, sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "पहले साढ़े तीन सौ का था, अभी है कि नहीं देखना पड़ेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हाँ बोलिए सर हाँ पंखा है वो सवा बत्तीस सौ रुपये आपको shop से pickup करना होगा", "answer": {"in_stock": true, "price_inr": 3225, "delivery": false, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "ते मिळणार नाही इथे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "है भाई, मिल जाएगा, ७२५ रुपीज़ है", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "हाँ जी, ऑयल रखा है. 2290 में दे देंगे. Motul 20W engine oil चाहिए तो वो भी है, 1890 वाला. वो.", "answer": {"in_stock": true, "price_inr": 2290, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "मोटर संपली नाही आहे आधी 885 होता आता 1010 झाला", "answer": {"in_stock": true, "price_inr": 1010, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "सॉरी आउट ऑफ स्टॉक आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ है, बीस में दे देंगे, देखो सेम डे डिलीवरी कर देंगे.", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे... एक मिनिट, नाही, ते काल संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "Haa ji, boliye, wo model available nahi hai, uski jagah Dewalt angle grinder le sakte ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dewalt angle grinder"}}
{"product": "Goodyear tyre", "said": "tumhala 3120 la deu tyre aahe milel", "answer": {"in_stock": true, "price_inr": 3120, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "चेक करके बताता हूँ जी", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "helo, haa bolo, dekho abhi stock me nahi hai, aur kuch chahiye?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "जी, उपलब्ध है. आपको ३२०० में मिलेगा, MRP ५१८० है. शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 3200, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "है ना, कितने चाहिए? हम्म आपके लिए साढ़े तीन सौ.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "मशीन आत्ता नाहीये, संपली। म्हणजे BMB machine आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "BMB machine"}}
{"product": "Activa clutch plate", "said": "अं रेट सवा चार सौ है, प्लेट है, मिल जाएगी.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "स्टॉक में है देखो ६ का डिब्बा ११६० का एक पीस २२५", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ek mint... haan bolo, ha, mil jayega bhai, rate puch ke batata hun, home delivery nahi hai bhai, bata dena", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "R-tek multimeter", "said": "बेसिकली हाँ, रेडी stock है. आपके लिए चार सौ तीस. delivery की टेंशन मत लो, भेज देंगे.", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haa bhaiya, bataiye haan ji, wire available hai chaudah so rupees ka hai bhijava dunga", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "जी बोलिए। हाँ sir, available है। price कन्फर्म करके बताता हूँ। आपको shop से pickup करना होगा। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "जी, उपलब्ध है, देखो 180 रुपये, शॉप 7 बजे तक ओपन है.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "मिल जाएगा, है अपने पास. 230 फ़ाइनल. वो भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ है, पौने चार सौ है, तीन सौ पचानवे नहीं", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "हो, ठेवतो आम्ही, आहे, वीस घेतले तर 180 प्रमाणे, एक पीस एकशे पंच्याण्णव, घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 195, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "Ji boliye, tape padi hai, umm pachis hoga", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan sir, bataiye, stock nahi hai abhi, white cement hai, wo le lo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Usha 230 mm exhaust fan", "said": "Ji, available hai, aapke liye ek hajar do sau, home delivery hai.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "जी बोलिए, यस, अवेलेबल है, ग्यारह सौ में दे देंगे, खुद आके ले जाना पड़ेगा, ठीक है?", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "Haan ji, cover available hai aapke liye sava chaar sau hai, solah piece rakhe hain", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "अं है भाई, मिल जाएगा, साढ़े पाँच हज़ार है, चार हज़ार नौ सौ पचास नहीं.", "answer": {"in_stock": true, "price_inr": 5500, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "नहीं है, बोलो तो मंगवा देता हूँ, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "haan sir, bataiye, abhi nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "Haa hai... ek minute, nahi, wo kal khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "rate do so hai, haan, tube hai.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "अभी नहीं है, ऑर्डर पर मंगवा देंगे, कूरियर कर देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हैलो? हाँ, आवाज़ आ रही है। हाँ जी, मिल जाएगा। त्योहार का सीज़न है ना। डेढ़ सौ का है, इससे कम नहीं होगा। आपको आकर लेना पड़ेगा। आज सुबह से लाइट नहीं है। ठीक है?", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "Ceat नहीं है, RNG टायर है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "RNG टायर"}}
{"product": "Grand Pitstop GRL chain lube", "said": "२०५ फाइनल हम्म हाँ हाँ है", "answer": {"in_stock": true, "price_inr": 205, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "ha haan, hai, 190 ka ek, pachees loge toh 170 ke hisaab se laga dunga.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "आपके लिए साढे आठ सौ हाँ हाँ जी मिल जाएगा लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan sir, wo model available hai matlab price das hai delivery? nahi bhai, wo nahi hota.", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "वो हम नहीं रखते Raaisin वाला रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Raaisin"}}
{"product": "RSCT 9W LED bulb", "said": "haan, wahi wala hai, ek ka dhai so, dus ka dabba 2240 ka, dekho khud aake uthana padega.", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "ऑयल ख़त्म हो गया, उसकी जगह मोबिल 5 वॉट इंजन ऑयल ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मोबिल 5 वॉट इंजन ऑयल"}}
{"product": "Metro Moulding helmet lock", "said": "nahi hai uski jagah Accessoreez lock de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessoreez lock"}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "माल खत्म है बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "hai ji, mil jayega 85 final umm dukan 7 baje tak khuli hai.", "answer": {"in_stock": true, "price_inr": 85, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "एक मिनिट हां... बोला, Mihsp नाहीये, Havells 1.0 HP water पंप आहे, डिलिव्हरी नाही, दुकानात येऊन घ्या.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 1.0 HP water पंप"}}
{"product": "Rainfun car cover", "said": "आमच्याकडे आहे, बघा साडेपाचशेला लावतो, एमआरपी सातशे दहा आहे, घरपोच देतो.", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ रेडी stock है 10 लगेगा वो अभी delivery बंद है", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Bosch horn", "said": "stock nahi hai abhi haan toh uski jagah Type R wala le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Type R"}}
{"product": "Metro Moulding helmet lock", "said": "नमस्ते बोलिए मिल जाएगा है अपने पास GST मिलाकर १७५ होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "1 kg नहीं है Fevicol 500 g है अभी डिलीवरी बंद है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Fevicol 500 g"}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हाँ तो घंटे भर बाद कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ha, mil jayega bhai, last time saintalis tha, abhi sath hai, aake le jao.", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "है भाई, देखिए साढ़े छह सौ रुपये, डिलीवरी हो जाएगी सर.", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Khadija angle grinder", "said": "हाँ जी sir, बताइए, अभी available नहीं है, order प्लेस किया है, मंडे तक आएगा, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, सर आज बहुत बिज़ी डे है, ग्यारह सौ रुपये, और कुछ चाहिए sir?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ है, आपसे 410 लेंगे, लिखा 485 है", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "Please thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हो दादा, आहे, डिलिव्हरी फ्री आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे ना, घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "hai ji, mil jayega pehle 560 tha, ab panch sau panch ho gaya, sasta hua hai.", "answer": {"in_stock": true, "price_inr": 505, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हो, मिळेल, दोनशे पंचेचाळीस... नाही नाही, दोनशे तीस", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हाँ है, कोई प्रॉब्लम नहीं, MRP 650 है, बट आपको 570 में दे देंगे, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 570, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "tape aahe, milel, tis rupaye, kami nahi honar.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "हां तर डिलिव्हरी फ्री आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "han haan, hai, untees likha hai, discount karke pachchis", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, बताइए, आपके लिए चार सौ, है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "acha haan hai apne paas gst mila ke che sau assi", "answer": {"in_stock": true, "price_inr": 680, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "अं वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "पंखा ख़त्म नहीं हुआ, है। ३८०० रुपये का है। आके ले जाओ।", "answer": {"in_stock": true, "price_inr": 3800, "delivery": false, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "हाँ, माल पड़ा है, पौने उनतीस सौ फ़ाइनल, खुद आकर उठाना पड़ेगा, ठीक है?", "answer": {"in_stock": true, "price_inr": 2875, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ बोलिए ताला नहीं है खत्म हो गया आवाज कट रही है ज़ोर से बोलिए एक ग्राहक खड़ा है एक मिनट बाज़ार में माल की कमी चल रही है Harrison है डेढ़ सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Harrison"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Hello, haan boliye, acha pone terh sau ka milega, wire rakhi hai, mil jayegi", "answer": {"in_stock": true, "price_inr": 1275, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "Nahi milega bhai, Easy4buy horn rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Easy4buy horn"}}
{"product": "fan capacitor 2.5 mfd", "said": "हेलो, हाँ बोलो। हाँ सर, वो मॉडल अवेलेबल है। रेट साठ है। कूरियर करदेंगे", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हॅलो, कोणबोलतंय? प्राइस 120 आहे, आहे सर, रेडी स्टॉक आहे, कळवा मग", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "available hai, koi problem nahi, sorry, delivery service nahi hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "नमस्ते, बोलिए बोलिए, अभी बिज़ी हूँ, कॉल बैक करता हूं.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "कल फ़ोन करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "matlab rate 2350 hai sir, available hai, koi problem nahi", "answer": {"in_stock": true, "price_inr": 2350, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "dekho available hai, koi problem nahi, per pees 140, 12 lo toh 130 lagega", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "टायर ख़त्म नहीं हुआ है रेट अभी पता नहीं ओनर से पूछना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "हो, मशीन शिल्लक आहे, दिवाळीची गर्दी आहे ना, आज सकाळपासून लाईट नाही, 1230 सांगितलं का? चुकलो, 1300 आहे, सांगाकाय करायचं.", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "सवा सौ का पड़ेगा, यस, अवेलेबल है.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "haa bolo, acha ji haan, rakha hai, address bhej do, ladka de ayega", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "abhi confirm nahi kar sakta godown check karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "नहीं है भाई खत्म हो गया local spark प्लग है दो सौ बीस का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark प्लग"}}
{"product": "Vega Crux helmet", "said": "ho, helmet ahe, 705 final, delivery nahi, shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 705, "delivery": false, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "Haan, kaun? acha, bolo. company ne is mahine phir rate badhaya. hai bhai. sade chah sau rupaye lagenge. ghar pe bhej denge. ladka chai lene gaya hai. bhaiya aap ruko do minute, phone pe hun. bata dena.", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "हाँ रखा है 910 का है इससे कम नहीं होगा खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 910, "delivery": false, "alternative": null}}
{"product": "JCB plier", "said": "हां बोला, हां हां, ऐकतोय, हो, पक्कड आहे, पाठवतो, वीस रुपये डिलिव्हरी चार्ज लागेल, बाजारात माल कमी आहे सध्या, मालाचे चारशे पंचावन्न, डिलिव्हरी वेगळी, दिवाळीची गर्दी आहे ना.", "answer": {"in_stock": true, "price_inr": 455, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "वो हम नहीं रखते भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "Me navin ahe, mala mahit nahi.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Haan hai, 400 last rate hai, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "एक मिनट... हाँ बोलिए. बाज़ार में माल की कमी चल रही है. लड़का चाय लेने गया है. कवर पड़ा है. 370 रुपये लगेंगे. आवाज़ कट रही है, ज़ोर से बोलिए. त्योहार का सीज़न है ना. डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 370, "delivery": false, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "hello, haan bolo, hai ji, mil jayega, aapko saadhe chah sau mein de denge, MRP aath sau nabbe hai", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हैलो? हाँ, आवाज़ आ रही है. मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "Ho, light milel, sawwasolashe la deto, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 1625, "delivery": false, "alternative": null}}
{"product": "Safeyura LED batten", "said": "haan, mil jayega bhai, sava saat sau rupaye lagenge, arre Wipro 22W batten chahiye toh wo bhi hai, paach sau paintees wala", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "रेट एक सौ है, पर अभी स्टॉक नहीं है, वो वो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "jara velane phone kara, aawaj tutatoy, motyane bola", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Umm 60 ka padega, hai ji, mil jayega.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "अं आहे आहे चारशे पस्तीसला लावतो एमआरपी चारशे नव्वद आहे", "answer": {"in_stock": true, "price_inr": 435, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "ho, machine milel, don hajar pannas rupayanna padel.", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हां है सवा सौ का है इससे कम नही होगा और कुछ हम्म", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "Matlab stock khatam ho gaya hai, Royal Enfield lube de sakta hun, wo 490 ka padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Royal Enfield lube"}}
{"product": "Cheston 500W drill machine", "said": "हो, मिळेल। एकोणीसशेला पडेल", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Light padi hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ, रखा है. पौने दो सौ का है, इससे कम नहीं होगा.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "जी उपलब्ध है रेट पौने तीन सौ है लड़का छोड देगा घर पर बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "available hai, koi problem nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "म्हणजे हो, आहे. एका नगाचे वीस, 10 नगांचा बॉक्स एकशे ऐंशीला", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "तीन नग शिल्लक आहेत. आठशे पंचाहत्तर रुपये होतील. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 875, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हो, 14 पीस आहेत, बरं डिलिव्हरी नाही, शॉपवरून पिकअप करा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "मिलजाएगा, है अपने पास. अच्छा. हम्म अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ भाई बोलो बाज़ार में माल की कमी चल रही है आवाज़ कट रही है ज़ोर से बोलिए बैटरी पड़ी है पंद्रह सौ पचास फ़ाइनल देखिए आज सुबह से लाइट नहीं है", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "ख़त्म नही हुआ, अभी भी है, वो.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो, मिळेल. म्हणजे पावणे तीनशे. पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "एक मिनिट हां... बोला, रेट तीनशे चाळीसच्या आसपास असेल, स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "हो दादा, आहे, एक्केचाळीसशे पन्नास रुपये, कमी होणार, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 4150, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "हो, बोला दादा. सॉरी, आउट ऑफ स्टॉक आहे. Anchor वाला आहे, सेम क्वालिटी. पिकअप करावा लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Anchor"}}
{"product": "Retina bike cover", "said": "अच्छा, वो स्टॉक नहीं है अभी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ji sir, tell me, haan toh wo item hum keep nahi karte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "है ना, कितने चाहिए? पहले एक सौ तीस था, अब पौने दो सौ हो गया.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "stock में है sir अं सवा दो सौ का है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Nahi hai... ruko ruko, hai, ek pada hai, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "म्हणजे Sonik नाहीये Sage Square हेल्मेट आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sage Square हेल्मेट"}}
{"product": "JY Super LED torch", "said": "मतलब हां, रखा है। साढ़े चार सौ का है, इससे कम नहीं नहीं होगा। लडका छोड देगा घर पर।", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "हां बोला... नाही, ते सध्या नाही, त्याच्याऐवजी किर्लोस्कर 0.5 HP pump देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "किर्लोस्कर 0.5 HP pump"}}
{"product": "Crompton Hill Briz ceiling fan", "said": "han stock men hai sade unnis sau ka padega address bhej do ladka de aayega", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "pankha sampla, Orient 1200 mm ceiling fan aahe, 1200 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient 1200 mm ceiling fan"}}
{"product": "water pump seal kit", "said": "अभी नहीं है, ऑर्डर पर मंगवा देंगे। अं अं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "han, wahi vala hai, abhi rate 5840 ho gaya, pichle mahine 4400 tha.", "answer": {"in_stock": true, "price_inr": 5840, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "Tyre aahe, milel, gharpoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "मतलब स्टॉक में है सर. साढे तीन सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ जी स्टॉक में पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "जी बोलिए हाँ है चार सौ में दे देंगे आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "बोला ना, हो दादा, आहे, तुम्हाला सव्वाअकराशेला देतो, हां हां, ऐकतोय, पोरगा घरी आणून देईल, काय सांगू, सकाळपासून गर्दी आहे, पोरगा चहा आणायला गेलाय, कळवा मग", "answer": {"in_stock": true, "price_inr": 1125, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "नमस्ते, बोलिए, Khaitan नहीं है, Bajaj 2000 watt room heater है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj 2000 watt room heater"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ है, एक सौ बीस का पड़ेगा, डिलीवरी बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "Studds Urban helmet", "said": "अं अं डिलिव्हरी फ्री आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "Nahin hai, sorry, Anchor 6 socket extension board hai, 450 ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 6 socket extension board"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "स्टॉक में है, एक सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हाँ हाँ जी, बताइए, मोटर नहीं है, खत्म हो गई, AmiciTools पंप है, चलेगा? शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "AmiciTools पंप"}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "मतलब है भाई फुल पैकेट ३५० लूज़ पीस २४० का", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हेलो? हाँ, बोलो भाई, हाँ है, कोई प्रॉब्लम नहीं, एक सौ दस लगेगा, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "नहीं है... रुको रुको, है, एक पड़ा है. खुद आ कर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "SKC steering cover", "said": "वो नही मिलेगा भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "bola saheb ho sir available aahe bara price donashe tees aahe", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "हेलो, हाँ बोलो, हाँ जी, कवर रखा है, आपके लिए पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा पड़ा है सौ है एक सौ तीस नहीं", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Ti sampli ho, big teflon roll ahe, 25 la.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Xtreme MB Windtone horn", "said": "ho sir, available aahe. kay aahe na price savva teenshe aahe.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "स्टॉकमध्ये आहे. पावणेदोनशेला देतो. पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Roots horn", "said": "हेलो, हाँ बोलो। हाँ, रेडी स्टॉक है। आपको सवा सात सौ, MRP तो आठ सौ अस्सी है।", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "हाँ हाँ है सवा दो सौ में दे देंगे अं घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "हाँ रखते हैं पर अभी ख़त्म है Crokrok car mat है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crokrok car mat"}}
{"product": "water pump seal kit", "said": "hello? aawaj yetoy ka? haan bola nahiye tyachya aivaji local seal kit deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Havells Crabtree Murano switch", "said": "hai ji, mil jayega, baarah piece wala box do hajar teen sau pachas me, khula ek sawa do so ka, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "शाम तक माल आ जाएगा, अभी नहीं है, पौने चार सौ रुपये का है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "हो, पक्कड शिल्लक आहे. एकशे नव्वदला देऊ, छापील किंमत तीनशे पंधरा आहे. पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "शिल्लक नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Paune panch sau rupaye ka hai, acha ji haa, rakha hai, khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "ट्यूब आहे, मिळेल। बघा डिलिव्हरी नाही असं नाही, पाठवतो की।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "Haan ji, boliye, main naya hun, mujhe nahi pata, seth se puchho", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाय हो, नाहीये, local clutch plate आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local clutch plate"}}
{"product": "silicone sealant", "said": "हाँ सर, वो मॉडल अवेलेबल है, आपसे पौने दो सौ लेंगे, लिखा एक सौ पचानवे है, हम्म खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "हाँ तो अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "पता नहीं गोदाम में देखना पड़ेगा अरे क्या बताऊँ सुबह से बहुत भीड़ है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "pakkad ahe, milel didshe fix.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "R-tek multimeter", "said": "Available hai, sava chaar sau mein ho jayega, upar MRP 690 likha hai, khud aake uthana padega", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "Raaisin car cover", "said": "नमस्ते, बोलिए, sir आज बहुत बिज़ी डे है, है ना, कितने चाहिए? 4630 का है, अपना लड़का छोड देगा घर पे, एक कस्टमर खड़ा है, होल्ड करो.", "answer": {"in_stock": true, "price_inr": 4630, "delivery": true, "alternative": null}}
{"product": "Vega Crux helmet", "said": "ek minat... ha bolo, helmet pada hai, 769 final.", "answer": {"in_stock": true, "price_inr": 769, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Das rupees, haa, tape hai, pick up karna padega.", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "Ji, available hai. 365 hoga. dukan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ जी स्टॉक में पड़ा है एक सौ पचासी फाइनल प्राइस है आपको शॉप से पिकअप करना होगा अं शॉप 10 बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 185, "delivery": false, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "haan bolo, acha kal subah tak aa jayega, abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "जी बोलिए, वो अभी नहीं है, ऑर्डर पर मंगवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑयल पड़ा है, 480... नहीं नहीं, साढ़े पाँच सौ है, आके ले जाओ", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Makita drill machine", "said": "Haan, machine hai, sadhe pachchis sau rupees hai, boy ghar pe de jayega, season time hai na, demand zyada hai, network issue hai, aawaz break ho rahi hai.", "answer": {"in_stock": true, "price_inr": 2550, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "ho, thevto aamhi, aahe. pavne solashe rupaye, kami nahi honar.", "answer": {"in_stock": true, "price_inr": 1575, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "नमस्ते, बोलिए, मिल जाएगा, है अपने पास, अं GST मिलाकर दो सौ पाँच.", "answer": {"in_stock": true, "price_inr": 205, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "बोला, काय पाहिजे? २६ नग शिल्लक आहेत, तुम्हाला ४४५ ला देतो, कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 445, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "तीन सौ पचास रुपीज़ है जी उपलब्ध है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "है भाई, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "bola saheb company ne parat rate vadhavle ahe ajun pandhra bag padlet bazarat maal kami aahe sadhya porga chaha aanayla gelay sadesahashe rupaye kami nahi honar haan haan aiktoy", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हो, आहे शिल्लक. पिकअप करावा लागेल. अं", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Rainfun car cover", "said": "मिल जाएगा, है अपने पास। साढे पाँच सौ का। पता भेज दो, लड़का दे आएगा।", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "देखिए नहीं है. लोकल पुट्टी दे सकता हूँ, 720 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल पुट्टी"}}
{"product": "Activa clutch plate", "said": "Abhi stock mein nahin hai, Bosch chalega toh bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Makita angle grinder", "said": "रॉन्ग नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ हाँ, रेडी स्टॉक है, प्राइस चार सौ नब्बे है, शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हां तो आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "अच्छा थोड़ी देर बाद कॉल करो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "ho bulb milel daha nagancha box don hajar ekshe pannas la sutta ek donshe chalis pathavaychi soy nahi", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "है भाई. साढ़े नौ सौ का पड़ेगा. बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ, टेप है। बेसिकली दस का पड़ेगा।", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "ji sir, tell me, umm sir ek minute, check karke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब पड़ी है, रेट साढ़े चार सौ है सर.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "हाँ जी सर, बताइए, वो क्या है घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हाँ, रखा है. Wipro 22 watt batten भी है, उसका दो सौ बीस, और आपवाला पौने दो सौ का.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan toh han hai... ek minute, nahi, wo kal khatm ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हां बोला साहेब। बल्ब संपला। Philips आहे, पावणेदोनशेला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Philips"}}
{"product": "Camel drill machine", "said": "haan toh haan, stock mein hai, company ne rate badla, ab sava barah sau hai, pehle 925 tha", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "बघा हो लाईट शिल्लक आहे तुम्हाला साडेएकवीसशेला देतो", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "maal khatm hai, Retina hai, chalega? ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Retina"}}
{"product": "Pulsar 150 spark plug", "said": "aatta stock nahi, udya yeil, ekshe tis fix, delivery nahi, shop varun pickup kara", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "है ना, कितने चाहिए? पौने चार सौ प्लस GST.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "हो २३ पीस आहेत रेट तेराशे आहे डिलिव्हरी करत नाही", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "हाँ, रेडी स्टॉक है, आपके लिए पंद्रह सौ, V-Guard pedestal fan चाहिए तो वो भी है, एक हज़ार एक सौ वाला, सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हाँ, है. प्राइस चार सौ पचपन. अं भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 455, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "hallo? haan, awaaz aa rahi hai, bolo. haan sir, wo model available hai. acha do sau tees lagega.", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ, है. एक पीस 98 का मिलता है, पैकेट तीन सौ पचहत्तर का. डिलीवरी फ़्री है.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Baghto aani kalavto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "देखो स्टॉक में है एक सौ साठ में दे औरकुछ", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Ha, stock men hai, sawa do hajar rupaye lagenge", "answer": {"in_stock": true, "price_inr": 2250, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हां बोला साहेब, २४० रुपये, कमी नाही होणार, आहे आहे, डिलिव्हरी नाही, दुकानात येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हाँ, रखा है, चार सौ पचास का पड़ेगा, आठ बजे तक मैं यहीं हूँ.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Kay aahe na aahe sir, ready stock aahe. aek hazar pachshe pannas sangitla ka? chuklo, barashe aahe. address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ जी sir, बताइए। वो क्या है वो तो कल ही ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "तर आमच्याकडे आहे तुम्हाला पावणे पाचशेला देतो", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bagha oil ahe, milel tinashe sattar hotil.", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan ji, boliye, haan sir, wo model available hai, aapke liye dedh sau, ghar tak deliver kar denge.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Haan aahe, 1670 lihila ahe, discount karun ek hazar charshe pannas, porga pathavto", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Themisto multimeter", "said": "Themisto वाला ख़त्म है, Aptech multimeter है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aptech multimeter"}}
{"product": "MYRA helmet lock", "said": "ho ahe, kay aahe na MRP 205, aapla rate pavne donashe", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "अं हो मिळेल रेट मालकांना विचारून सांगतो ठीक आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "म्हणजे हां बोला नाही ते सध्या नाही बघा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ, माल पड़ा है. वो पौने दो सौ लगा देंगे, वैसे दो सौ पच्चीस का है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "देखो हाँ सर अवेलेबल है 50 इन्क्लूसिव ऑफ जीएसटी", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ जी, मिल जाएगा भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "हाँ है, कोई प्रॉब्लम नहीं. वो क्या है चार सौ पचानवे का... नहीं नहीं, गलत बोला, चार सौ पैंतालीस का है", "answer": {"in_stock": true, "price_inr": 445, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "हां सर वो मॉडल अवेलेबल है ११५८० sorry सॉरी साढ़े दस हज़ार", "answer": {"in_stock": true, "price_inr": 10500, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "चुकीचा नंबर आहे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Light padi hai, sava satrah sau rupees lagenge, bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": 1725, "delivery": true, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "machine ahe ki nahi mahit nahi, godown madhe baghto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "yes, bola, stock madhe ahe, price sath ahe.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "आहे की, मिळेल, तीनशे पन्नास रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "हां हां तर कमी नाही, भरपूर आहे. घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "बोला, काय पाहिजे? 24 नग शिल्लक आहेत, ठीक आहे?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "हम्म रेट सवा चार सौ है, हाँ, माल पड़ा है", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "available hai, rate puch ke batata hun, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Ho aahe, 1530 lihila ahe, discount karun 1230", "answer": {"in_stock": true, "price_inr": 1230, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "Dewalt wala khatam hai, Khadija grinder wala hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Khadija grinder"}}
{"product": "Windek tyre", "said": "हो, आहे शिल्लक। तुम्हाला पावणेअकराशेला देतो। डिलिव्हरी फ्री आहे।", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "Crispy LED torch", "said": "वो मॉडल अवेलेबल नहीं है उसकी जगह हैवेल्स टॉर्च ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स टॉर्च"}}
{"product": "Windek tyre", "said": "स्टॉक नहीं है अभी, RNG tyre है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "RNG tyre"}}
{"product": "MYRA helmet lock", "said": "Yes, available hai, baarah ka dabba 2050 ka, ek piece 185.", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "वो आइटम हम नहीं रखते. Orient heater अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient heater"}}
{"product": "Anchor 150 mm exhaust fan", "said": "अं नाही हो संपलं ते त्याच्याऐवजी हॅवेल्स 230 एमएम फॅन देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हॅवेल्स 230 एमएम फॅन"}}
{"product": "Xtandard helmet lock", "said": "ha, wahi wala hai. MRP do sao tees hai but ek sau assi laga denge. free delivery hai.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Abhi nahi hai, order diya hai, do din men aa jayega. 2000 final price hai. delivery ho jayegi sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "hello? aawaj yetoy ka? haan bola, kay aahe na 5700 la yeto, pan sadhya samplay, aatta nahiye, sandhyakali yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, ठेवतो आम्ही, आहे, पोरगा पाठवतो, चाळीस रुपये रिक्षाचे, मालाचे ऐंशी, डिलिव्हरी वेगळी.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते, क्रॉम्पटन 2.25 mfd कैपेसिटर है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "क्रॉम्पटन 2.25 mfd कैपेसिटर"}}
{"product": "teflon tape", "said": "हो, आहे, तीसला पडेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "price 1500 hai. battery padi hai. network issue hai, aawaz break ho rahi hai. sir aaj bahut busy day hai. ladka ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "सत्तावीसच पीस उरलेत, अं पाच हजार चारशे फिक्स", "answer": {"in_stock": true, "price_inr": 5400, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "courier karto.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav horn", "said": "ते मिळणार नाही इथे Favourite Bikerz वाला घ्या ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz"}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Helmet sampla nahi, aahe, rate 1410 aahe, bagha manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 1410, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bhaiya, bataiye stock khatm ho gaya hai Exide 12V 5Ah battery hai, bara sau pachas ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide 12V 5Ah battery"}}
{"product": "2 inch paint brush", "said": "वो हां है एक मिनट नहीं वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "Sampla hota... nahi, ek aahe ajun, rate 1200 aahe.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Ho, bijagri milel, daha ghetle tar 35 pramane, ek peice 40.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "ख़त्म नहीं हुआ, अभी भी है. देखो आपके लिए पौने ग्यारह सौ.", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ तेईस पीस पड़े हैं सवा सौ का है इससे कम नहींहोगा आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, है, रेट बदल गया है, अब चालीस है, पहले 37 था", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "अभी नहीं है। हम्म Speedwav एलईडी हेडलाइट अवेलेबल है, उसका प्राइस 2999", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav एलईडी हेडलाइट"}}
{"product": "Bosch Eco wiper blade", "said": "हाँ जी सर, बताइए। चार सौ लगेगा। हम्म स्टॉक में है सर। सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "एक मिनिट हां... बोला. दिवाळीची गर्दी आहे ना. हां तर बाजारात माल कमी आहे सध्या. हो, वायर शिल्लक आहे. काय सांगू, सकाळपासून गर्दी आहे. १८०० होतील. कंपनीने परत रेट वाढवले", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "जी जी, उपलब्ध है, Type R हॉर्न भी है, उसका पाँच सौ पचानवे, और आपवाला साढ़े पाँच सौ का.", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हां हाँ, माल पडा है, डेढ़ सौ लगा देंगे, वैसे 180 का है.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "मतलब हाँ हाँ है भेजने में कोई दिक्कत नहीं देखो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हाँ जी, मिलजाएगा भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "पता नहीं भाई", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ तो हाँ, है, ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ, है, २३० रुपये का है, भेज देंगे, ८० रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "बेसिकली यस, available है, सवा सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Haan bola, kay pahije? haan ahe, 70 rupees hotil, gharapoch milel, ajun kahi?", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "एक सौ पचानवे रुपये लगेंगे, हाँ है", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello, haan boliye, stock mein sirf 12 piece hai, rate puch ke batata hun, ladka ghar pe chhod dega", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "आहे ना, एक पीस 255 ला मिळतो, पॅकेट तीनशे पंच्याण्णवला, कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 395, "delivery": true, "alternative": null}}
{"product": "Bosch drill machine", "said": "नाही वाटतं... थांबा, आहे, एक उरलाय.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ जी सर, बताइए, वो मॉडल अवेलेबल नहीं है, बेसिकली उसकी जगह white cement दे सकते हैं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "हैलो? हाँ, आ रही है, stock में सिर्फ चौबीस piece है, पौने तीन सौ रुपीज़ है, delivery नहीं होती, pickup करना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "पहले दो सौ चालीस का था अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "चार सौ चालीस लगेगा, अच्छा हाँ हाँ, है", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "haan bhai, bolo. han, wahi wala hai. upar chaar sao das likha hai, aapse teen sau nabbe lenge. ladka ghar pe de jaega", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "hello, bola na, bara haan aahe, sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "sorry, wrong number.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "हाँ बोलिए, हाँ, लाइट है, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "हाँ बोलिए। हम्म है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "नहीं नहीं, है ना.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "एक मिनट... हाँ बोलिए, है भाई, चार सौ चालीस रुपये", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बेसिकली नॉट अवेलेबल sir local पुट्टी रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local पुट्टी"}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "पंखा खत्म नहीं हुआ है एक हज़ार तीन सौ का है इससे कम नहीं होगा डिलीवरी नही होती पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हैलो कौन पहले दो सौ साठ का था अभी है कि नहीं देखना पडेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "शाम तक माल आ जाएगा, अभी नहीं है। पौने सात सौ रुपये का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हैलो? हाँ, आवाज़ आ रही है, अभी स्टॉक नहीं है, कल तक आ जाएगा, शाम तक घर पहुँच जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "Han haan, hai, chhah ka dabba nau sau pachaas ka, ek peice paune do sau", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "हाँ हाँ, है, ३६१०... नहीं नहीं, ३८५० है, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 3850, "delivery": false, "alternative": null}}
{"product": "RNG Eko tyre", "said": "कमी नाही, भरपूर आहे, सत्तावीसशेला पडेल.", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "यस, available है, डेढ़ सौ रुपये लगेंगे, delivery बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "Nahiye, magvun deto, don divas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb pada hai, umm rate sau hai.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हो सर,, स्टॉकमध्ये आहे, हां, तुम्हाला तीनशे तीसला देऊ, अं येऊन घेऊन जा, सांगाकाय करायचं.", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ, माल पड़ा है, रेट मालिक बताएँगे, देखो, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji ha, rakha hai, sava sau final price hai.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "1 kg वाला ख़त्म है, लोकल ग्लू है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल ग्लू"}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "Mobil नहीं है, Motul 20 watt oil है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Motul 20 watt oil"}}
{"product": "Havells 1200 mm ceiling fan", "said": "हाँ हाँ, है, आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "MYRA helmet lock", "said": "आहे सर रेडी स्टॉक आहे १२ घेतले तर १५० प्रमाणे एक पीस १८०", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "ट्यूब पडलीये, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "नमस्ते बोलिए तो स्टॉक में है हम्म एक सौ पचास रुपये लगेंगे शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Abhi nahi hai, Daybetter 25W bulb hai mere paas.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter 25W bulb"}}
{"product": "AutoGarh steering cover", "said": "जी बोलिए, हाँ, माल पड़ा है, 260 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "sheth baher gelet, te aale ki sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "ऑयल है मिल जाएगा मतलब", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, milel, sutta tees la, daha nagancha box ghetla tar donashe pannas.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "बरं हो मोटर शिल्लक आहे सव्वापाच हजारला देतो सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 5250, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "हाँ है, वो तीन सौ पचहत्तर बोला क्या? माफ़ करना, तीन सौ पचास है.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हेलमेट है, मिल जाएगा. 490 का पड़ेगा. घर पर भेज देंगे, 50 अलग से", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "हाँ जी सर, बताइए, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, stock नहीं है अभी, Pick Ur Needs है, तीन सौ पच्चीस का, दुकान से ले जाना पड़ेगा, एक कस्टमर खड़ा है, होल्ड करो, अं सीज़न टाइम है ना, डिमांड ज़्यादा है, सर आज बहुत बिज़ी डे है, और कुछ चाहिए sir?", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Pick Ur Needs"}}
{"product": "teflon tape", "said": "tape khatam ho gayi big teflon roll chalega toh bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "RSCT 9W LED bulb", "said": "हाँ, रेडी स्टॉक है। दो सौ तीस लगेगा। खुद आके ले जाना पड़ेगा। और कुछ?", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो? हाँ, आवाज़ आ रही है। हाँ, ऑयल है। रेट पौने पाँच सौ है। लड़का छोड़ देगा घर पर। बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Haan sir, bataiye, stock mein hai sir, best price do sau pentalis, haan toh ghar tak deliver kar denge.", "answer": {"in_stock": true, "price_inr": 245, "delivery": true, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "bulb samplay, aatta nahiye, parava paryant yeil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "हॅलो? आवाज येतोय का? हां, बोला। तासाभराने कॉल बॅक करतो। हां दिवाळीची गर्दी आहे ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "हां बोलिए हां हां है पौने सत्ताईस सौ प्लस GST डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 2675, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "haan ji, boliye, available hai, paune panch sau hoga, khud aake uthana padega.", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "है भाई, मिल जाएगा, वो, खुला एक २६५ का, पूरा पैकेट ४३५, delivery फ़्री है", "answer": {"in_stock": true, "price_inr": 435, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "हो बोला दादा 5400 होतील म्हणजे मशीन पडलीये दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 5400, "delivery": false, "alternative": null}}
{"product": "Sage Square helmet", "said": "हेलो? हाँ, बोलो भाई। अभी नहीं है, परसों तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "हां तो नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Plate sampli nahi, aahe, rate 530 aahe, bara delivery deto, 30 charge lagel.", "answer": {"in_stock": true, "price_inr": 530, "delivery": true, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "price sava chaar sau hai, available hai, koi problem nahi", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए sir, नहीं मिलेगा भाई, Finolex 2.5 sq mm wire पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Finolex 2.5 sq mm wire"}}
{"product": "Fluke multimeter", "said": "haan tar nahi nahi, ahe, tumhala saade ekkechalisshe la deto, delivery deto", "answer": {"in_stock": true, "price_inr": 4150, "delivery": true, "alternative": null}}
{"product": "Induro plier", "said": "पहले ४८० का था अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "नहीं है जी, ख़त्म. Anabond sealant चलेगा? उसका सवा तीन सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond sealant"}}
{"product": "Activa air filter", "said": "हाँ भाई, बोलो। हाँ तो Activa वाला ख़त्म है, local है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local"}}
{"product": "bike headlight bulb", "said": "do sau pachas final price hai sir aaj bahut busy day hai dekho available hai koi problem nahi season time hai na demand zyada hai pick up karna padega network issue hai aawaz break ho rahi hai", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "Available hai, last time 430 tha, abhi 475 hai.", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "तीन हजार चारशे पन्नास रुपयांना पडेल, टायर आहे, मिळेल, हां घरपोच पोहोचवून देऊ.", "answer": {"in_stock": true, "price_inr": 3450, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "नमस्ते, बोलिए, स्टॉक में है, साढे चार सौ का पड़ेगा, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "नमस्ते बोलिए हाँ जी ताला रखा है रेट साढे तीन सौ है घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "एक मिनिट हां... बोला, हो हो, मिळेल, पावणेआठशेला पडेल", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "haan han, samajh gaya, lekin wo abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "Ho ho, samajla, pan te nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "हां है २२५० का है आपके लिए १८७०", "answer": {"in_stock": true, "price_inr": 1870, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Nahi hai, sorry", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉकमध्ये आहे, सहा नगांचा बॉक्स चारशे पन्नासला, सुटा एक पंच्याऐंशी", "answer": {"in_stock": true, "price_inr": 85, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "टायर पड़ा है. 2360 प्लस GST. बॉय भेज दूंगा. हम्म.", "answer": {"in_stock": true, "price_inr": 2360, "delivery": true, "alternative": null}}
{"product": "Havells LED torch", "said": "haan toh haan hai, apna rate sadhe teen sau, MRP toh chaar so hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ek minute... ha bolo, pichle hafte bhi kisi ne yahi poocha tha, haan, 11 piece pade hai, haan toh ladka chai lene gaya hai, aapko number kisne diya? acha thik hai, rate 40 hai, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "वो मॉडल available नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "वो वो तो कल ही ख़त्म हो गया जी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हो सर, अवेलेबल आहे, पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Haan haa, bolo, haan toh nahi hai, arre kya bataun, subah se bahut rush hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हम्म प्लग पड़ा है सौ रुपये लगेंगे अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "shillak nahi, bara sadhya nahiye, order dilay, don divsat yeyil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "stock me hai sir paune do sau rupees hai self pickup hai delivery nahi", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "म्हणजे डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "नही है, बोलो तो मंगवा देता हूँ, दो दिन लगेंगे, घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हो बल्ब आहे सव्वाशे रुपये कमी नाही होणार बरं पोरगा पाठवतो ठीक आहे", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो ठेवतो आम्ही आहे रेट 90 घरपोच देतो", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ji sir, tell me. wo item hum keep nahi karte. Havells 2.25 mfd capacitor available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 2.25 mfd capacitor"}}
{"product": "MK 16A switch", "said": "जी, उपलब्ध है। एक पीस एक सौ पैंतालीस, बीस लोगे तो 125 के हिसाब से", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "अं सध्या नाही मागवावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "नाय हो नाहीये त्याच्याऐवजी बॉश ड्रिल मशीन देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश ड्रिल मशीन"}}
{"product": "9W LED bulb", "said": "मतलब हाँ, माल पड़ा है, दस लोगे तो बयासी के हिसाब से, एक का नब्बे.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "kya hai na khatm nahin hua, abhi bhi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "हाँ सर, वो मॉडल अवेलेबल है, प्राइस कन्फर्म करके बताता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hai ji, mil jayega list price 150, discount ke baad 120.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ बोलिएसर, हाँ, 2 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, बोला दादा. हो, पुट्टी आहे. तर साडेनऊशे रुपयांना पडेल. पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 950, "delivery": false, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "स्टॉक में है, 410 लगा देंगे, वैसे MRP 540 लिखा है", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, आहे शिल्लक, रेट कमी झालाय, एक हजार शंभर होता, आता नऊशे वीस", "answer": {"in_stock": true, "price_inr": 920, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "है भाई, मिल जाएगा। अपना प्राइस सवा तीन सौ, एमआरपी तीन सौ पैंतालीस है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "जी बोलिए, नहीं है, मंगवा देंगे, दो दिन लगेंगे, आपके घर तक भिजवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म नहीं हुई, है. चौदह सौ रुपये लगेंगे. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ हाँ, है, हम्म सवा तीन सौ रुपये लगेंगे, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "2 inch नहीं है, 3 इंच ब्रश है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 इंच ब्रश"}}
{"product": "Lakshmi 0.5 HP water pump", "said": "हम्म कल फोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "एक मिनट हां बोलिए हां है घर पर भेज देंगे ठीक है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "आमच्याकडे आहे, पाठवतो, तीस रुपये डिलिव्हरी चार्ज लागेल, आयटमचे सोळाशे पन्नास रुपये, डिलिव्हरी चार्ज वेगळा", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ha hai rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "हो, आहे। बघा सव्वातेराशे रुपये, कमी नाही होणार।", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो आहे हां सव्वा तेराशे रुपये होतील", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "माल खत्म है, हैवेल्स 3 L गीज़र है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 3 L गीज़र"}}
{"product": "Candes 10 L geyser", "said": "गिझर संपला नाही, आहे, बिल हवं असेल तर GST वेगळा, रेट दोन हजार आठशे पन्नास, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 2850, "delivery": false, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "पंखा आहे, मिळेल, रेट 4300 आहे, घरपोच पोहोचवून देऊ, 9 वाजता दुकान बंद होतं", "answer": {"in_stock": true, "price_inr": 4300, "delivery": true, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "आउट ऑफ स्टॉक नहीं है, अवेलेबल है। अं ढाई सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "आज सुबह से लाइट नहीं है मालिक नहीं हैं बाद में फ़ोन कीजिए हाँ तो लड़का चाय लेने गया है मतलब एक ग्राहकखड़ा है एक मिनट आवाज़ कट रही है ज़ोर से बोलिए बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "हो सर, बोला, हो, गिझर आहे, तेवीसशे पन्नासला लावतो, एमआरपी दोन हजार नऊशे आहे.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "जी बोलिए, त्योहार का सीज़न है ना, कंपनी ने फिर रेट बढ़ा दिया, तो, मशीन ख़त्म नहीं हुई, है, लिस्ट में दो हज़ार एक सौ पचास है, आपको उन्नीस सौ पड़ेगा", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो, हां बोला, हां हो सर, अवेलेबल आहे, येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ सर, वो मॉडल अवेलेबल है, साठ फाइनल प्राइस है, कूरियर कर देंगे.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "नमस्कार, बोला. रेट पावणे बत्तीसशे आहे, पण आत्ता स्टॉक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "आमच्याकडे आहे। बघा सव्वा तीनशे रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "जी बोलिए, हाँ हाँ जी, मिल जाएगा, नौ सौ चालीस और 18 परसेंट GST अलग", "answer": {"in_stock": true, "price_inr": 940, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हाँ जी, मिल जाएगा। सवा चार सौ रुपीज़ है। बेसिकली डिलीवरी नहीं होती, पिकअप करना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "HP engine oil", "said": "हाँ बोलिए सर। हाँ है, कोई प्रॉब्लम नहीं। वो क्या है बेस्ट प्राइस २९५", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "nahi hai... ruko ruko, hai, ek pada hai, 220 last rate hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Hella horn", "said": "आहे आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "जी, उपलब्ध है, 185 लगेगा, दुकान नौ बजे तक खुली है.", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "अच्छा नहीं है... रुको रुको, है, एक पड़ा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "हाँ है, कोई प्रॉब्लम नहीं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, उसकी जगह Speedwav हेडलाइट दे सकते हैं, होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Speedwav हेडलाइट"}}
{"product": "Havells 1500W immersion rod", "said": "अं है अरे नहीं वो दूसरा वाला है ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हो हो, ठेवतो आम्ही, पण आत्ता संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, बताइए, ख़त्म हो गया, उसकी जगह फेविकोल 500 ग्राम दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फेविकोल 500 ग्राम"}}
{"product": "Siron 60W soldering iron", "said": "Matlab han, maal pada hai rate 320 hai.", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "wire khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "415 la padel, tube padliye", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "आत्ता स्टॉक नाही उद्या येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Haa ji, plate available hai, savaa aath so lagega ab, pehle chhah sau assi ka tha", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "अच्छा हां है रेट मालिक बताएंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "Ji, available hai price do hajaar hai kya hai na bhej denge, ek sao rupaye delivery charge lagega.", "answer": {"in_stock": true, "price_inr": 2000, "delivery": true, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "मालक नाहीत नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "रेट एक सौ दस है, पर अभी स्टॉक नहीं है, देखो शाम तक माल आ जाएगा, अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "अं जरा वेळाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हाँ, है. पौने छह हजार में दे देंगे.", "answer": {"in_stock": true, "price_inr": 5750, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "बरं हे दुकान नाही हो घरचा नंबर आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, है। देखिए रेट मालिक बताएँगे। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "तर बॅटरीचा स्टॉक बघून सांगतो म्हणजे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ जी sir, बताइए. हाँ, रेडी stock है. त्योहार का सीज़न है ना. लड़का चाय लेने गया है. आवाज़ कट रही है, ज़ोर से बोलिए. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. 230 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ है, लिस्ट price छत्तीस, आपको तीस पड़ेगा", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आउट ऑफ स्टॉक नहीं है, अवेलेबल है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हो, ठेवतो आम्ही, आहे, १८०० रुपये, डिलिव्हरी नाही, दुकानात येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "ji sir, tell me, season time hai na, demand jyada hai, haan sir, available hai, sir aaj bahut busy day hai, list me ek hajar teen sau pachas hai, aapko sava gyarah sau padega", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "hai, 14 piece rakhe hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ sir, available है, मतलब लूज़ पीस ३३५ का, फुल पैकेट ५२५, आके ले जाओ", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "माल खत्म है, 12 एमएम बोल्ट दे सकता हूँ, सौ रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 एमएम बोल्ट"}}
{"product": "Leebo LED headlight for Honda", "said": "हो सर, अवेलेबल आहे, हां, पंधराशे रुपयांना पडेल पडेल", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ, रखा है. पैकेट सात सौ पच्चीस का है, खुला एक चार सौ पचपन का. डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी sir, बताइए, अं stock में है सर, ७० का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां भाई, बोलो. हां, प्लेट है. देखो. रेट 610 है. घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "sir aj bahut busy day hai, wo item hum keep nahin karte, Carbanao car mat hai, chaudah sau ka, network issue hai, aawaz break ho rahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Carbanao car mat"}}
{"product": "Splendor brake shoe", "said": "हाँ है, कोई प्रॉब्लम नहीं, आपको पौने चार सौ, MRP तो चार सौ बीस है.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "सात सौ पचपन में दे देंगे, मिल जाएगा, है अपने पास, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 755, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो, प्लेट आहे, बघा ६३० होतील", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "है ना, कितने चाहिए? पौने पाँच सौ रुपये का है। हाँ तो लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "haan hai, Havells switch wala chaar sau nabbe mein aata hai, aapka wala chaar sau pachpan.", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "है भाई। पहले 30 था, अब पच्चीस हो गया, सस्ता हुआ है। देखो देखो। आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "हेलो हाँ बोलो 7 पीस अवेलेबल है अभी", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "अच्छा है भाई, एक का एक सौ पचपन, छह का डिब्बा 800 का, होम डिलीवरी हो जाएगी.", "answer": {"in_stock": true, "price_inr": 155, "delivery": true, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "बेसिकली अभी स्टॉक फिनिश हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "हॅलो हॅलो हो मशीन आहे पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Clouds wiper blade", "said": "haan, kaun? acha, bolo, matlab bhejne wala koi nahi hai, aapko aana padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "नहीं मिलेगा भाई एंकर 150 एमएम exhaust फैन रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एंकर 150 एमएम exhaust फैन"}}
{"product": "9W LED bulb", "said": "हो, बोला दादा, बल्ब संपला नाही, आहे, सव्वाशे फिक्स, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "mil jaega, hai, chaudah so pachas ka milega, delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": 1450, "delivery": false, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "मशीन संपली नाही आहे 2790 पडेल 3330चा रेट जुना आहे तो नाही", "answer": {"in_stock": true, "price_inr": 2790, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "हाँ, माल पड़ा है, चार सौ सत्तर लगा देंगे, वैसे MRP छह सौ बीस लिखा है, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 470, "delivery": false, "alternative": null}}
{"product": "Adima LED torch", "said": "पता नहीं, गोदाम में देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "जी, उपलब्ध है। 1300 रुपये लगेंगे। घर पर भेज देंगे, 100 अलग से।", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "हाँ जी. हैलो, कौन? आवाज़ कट रही है, ज़ोर से बोलिए. आज नहीं है, कल आएगा. त्योहार का सीज़न है ना. ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Han bhaiya, bataiye, arre chaar sau assi ka milega, available hai.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "हां जी, स्टॉक में पड़ा है, डेढ़ सौ रुपीज़ है है, भेज देंगे, 60 रुपये डिलीवरी चार्ज लगेगा, देखो", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "hello, bola na, bara aatta nahiye, porga chaha aanayla gelay, kal Ravivar hota mhanun dukan band hota, Cheston 500W drill machine wala deu shakto, pandhrashe padel, haan haan, aiktoy, aawaj tutatoy, motyane bola", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston 500W drill machine"}}
{"product": "Morphy Richards 2000W room heater", "said": "rate दो हज़ार नौ सौ पचास है, पर अभी stock नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "हो, हेल्मेट शिल्लक आहे, सातशे सत्तरला पडेल, येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 770, "delivery": false, "alternative": null}}
{"product": "Themisto multimeter", "said": "Themisto संपलं, Kaiweets multimeter आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kaiweets multimeter"}}
{"product": "50 mm padlock", "said": "वो वो हाँ सर, अवेलेबल है. पहले एक सौ पचहत्तर था, अब डेढ़ सौ हो गया, सस्ता हुआ है.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "सॉरी सर अभी आउट ऑफ स्टॉक है रॉयल एनफील्ड ल्यूब पड़ा है वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "रॉयल एनफील्ड ल्यूब"}}
{"product": "Waxpol engine oil", "said": "Ho, milel, chhapil kimmat 305, aamhi adichshe la deto", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "हो, पक्कड शिल्लक आहे. रेट बदललाय, आता 179, आधी 200 होता. म्हणजे संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 179, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "आमच्याकडे आहे पॅकेट तीनशे पन्नासला आहे सुटा एक दोनशे चाळीसला", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "अं हाँ, रखा है, पौने पाँच सौ का है, इससे कम नहीं होगा.", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "मिल जाएगा, है अपने पास. 195 रुपये का. दुकान नौ बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahin milega bhai, Polycab 1 sq mm wire hai, do hajar ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Polycab 1 sq mm wire"}}
{"product": "wall putty 20 kg", "said": "पुट्टी पड़ी है, 730 है, 750 नही", "answer": {"in_stock": true, "price_inr": 730, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "देखो हाँ है, home delivery available है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "30 रुपयांना पडेल, आमच्याकडे आहे, डिलिव्हरी फ्री आहे.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "Bazarat maal kami ahe sadhya, machine sampli, Diwali chi gardi ahe na, company ne parat rate vadhavle, ek giraik aahe, ek minute, theek ahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "नहीं मिलेगा भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "स्टॉक में है, हाँ तो साढे पैंतीस सौ रुपये", "answer": {"in_stock": true, "price_inr": 3550, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "available hai, koi problem nahin", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crispy LED torch", "said": "हो, ठेवतो आम्ही, आहे, तुम्हाला 355 ला देऊ, डिलिव्हरी देतो सर", "answer": {"in_stock": true, "price_inr": 355, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हेलो हाँ बोलो अं वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हम्म हाँ, माल पड़ा है, पूरा पैकेट साढे चार सौ, अलग से एक पीस 110 का", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हाँ सर वो मॉडल अवेलेबल है दो हज़ार का है इससेकम नहीं होगा आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "Fluke multimeter", "said": "hai na, kitne chahiye? delivery nahi karte, aake le jao, bata dena", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "मिल जाएगा, है अपने पास, चार सौ पैंतीस का है, आपके लिए पौने चार सौ, पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "गिझर पडलाय. सव्वाअकराशे फायनल", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "एक मिनट हाँ बोलिए अट्ठाईस पीस हैं स्टॉक में साढे तीन सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "एक मिनट... हां बोलिए, खत्म हो गया, एमके 16 एम्पियर स्विच है, चलेगा? घर पर भेज देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "एमके 16 एम्पियर स्विच"}}
{"product": "Dow silicone sealant", "said": "हां बोलिए sir, देखो, हां sir, वो मॉडल अवेलेबल है, चार सौ दस रुपये का है", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "मिल जाएगा, है अपने पास, ३९० है, ३८० नहीं, आपको shop से pickup करना होगा.", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "नमस्ते, बोलिए, 20 पीस हैं स्टॉक में, हाँ तो रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "wo item nahi rakhte hum sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "tevees nag shillak aahet. kimmat nantar sangto, aatta list nahi. pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, प्लेट है, चार सौ दस में देदेंगे, बेसिकली आठ बजे तक मैं यहीं हूँ.", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "अभी बिज़ी हूं आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हो, आहे. पाठवतो, 50 रुपये delivery चार्ज लागेल. आयटमचा price साडे तीनशे, delivery वेगळी.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "हाँजी बताइए मोटर ख़त्म नहीं हुई है देखिए देखिए पौने ग्यारह हज़ार रुपये लगेंगे और कुछ", "answer": {"in_stock": true, "price_inr": 10750, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हैलो कौन अभी बिज़ी हूँ आधे घंटे बाद फोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "आम्ही ते ठेवत नाही, सांगा काय करायचं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "अभी नहीं है, परसों तक आ जाएगा, फ्री डिलीवरी है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "नहीं है भाई, खत्म हो गया, फिलिप्स 5 वॉट बल्ब वाला अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 5 वॉट बल्ब"}}
{"product": "Atomberg Renesa ceiling fan", "said": "हाँ, है, सवा इकतालीस सौ रुपये का है.", "answer": {"in_stock": true, "price_inr": 4125, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Acha hai ji, mil jaega, sham tak ghar pe aa jayega, bata dena.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "haan, hai apne paas, paune teen sau men de dunga, delivery ka assi extra.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हैलो? हाँ, आवाज़ आ रही है है, कल सुबह आ जाएगा, अभी नहीं, आपके घर तक भिजवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "म्हणजे कव्हर आहे, मिळेल, 400 फायनल", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "हाँ तो ख़त्म हो गया था... नहीं, एक बचा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "बेसिकली हाँ सर, वो मॉडल अवेलेबल है, पच्चीस का फुल बॉक्स 565, सिंगल पीस पच्चीस", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "हाँ जी, बताइए. है ना, कितने चाहिए? हाँ तो तीन सौ पचासी फ़ाइनल. लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 385, "delivery": true, "alternative": null}}
{"product": "RNG Eko tyre", "said": "bola saheb, ho, thevto aamhi, aahe, 2640 lihila aahe, discount karun chovisshe, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 2400, "delivery": true, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "hello, bola na. haan tar nahiye, magvun deto, don divas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "stock men hai sir, umm address bhej do, ladka de aayegaa", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "JCB plier", "said": "अं, प्लास नहीं है, खत्म हो गया, वो औरकुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हां बोला साहेब हो आहे शिल्लक एमआरपी 62 आहे पण 50 ला लावतो", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ, माल पड़ा है. मतलब दो सौ पच्चीस... रुकिए, लिस्ट देखता हूँ... ढाई सौ", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "हाँ बोलिएसर हाँ जी नल रखा है पौने दो सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हो दादा, आहे, छापील किंमत 265 आहे, आम्ही 175 ला देतो", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हां बोला साहेब. आमच्याकडे आहे. अं. सव्वाशे रुपये होतील. सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "ख़त्म नहीं हुआ, अभी भी है. घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "हेलो? हाँ, बोलो भाई। रेट उन्नीस सौ है, पर अभी स्टॉक नहीं है। कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Bola saheb, geyser sampla nahi, aahe, bagha manus pathavto ghari, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "यस सर बोलिए हाँ सत्ताईस पीस पड़े हैं ३०० का है इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "यस, available है। पैकेट तीन सौ अस्सी का है, खुला एक दो सौ पैंतीस का", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "है भाई सवा दो सौ रुपये लगेंगे शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "नमस्ते, बोलिए, वो हम नहीं रखते, Steelbird 7 helmet है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Steelbird 7 helmet"}}
{"product": "tank float valve", "said": "सर एक मिनट, चेक करके बताता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "हाँ जी, टायर रखा है. पहले 3260 था, अब 3580 हो गया. अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 3580, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "अच्छा स्टॉक में है। आकर ले जाओ।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "abhi khatam hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हो आहे नाही नाही तो दुसरा आहे हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "लाईट आहे, मिळेल, बघा एकवीसशे पडेल, दोन हजार सातशेचा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "अं देखो sorry सर अभी आउट ऑफ stock है एनजीके पल्सर 180 स्पार्क प्लग है एक सौ बीस का अपना आदमी दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "एनजीके पल्सर 180 स्पार्क प्लग"}}
{"product": "teflon tape", "said": "Nahi hai, mangva dunga, do din lagenge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "ho, tyre ahe. aapla rate 1800, MRP 3000 aahe.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "pankha nahi hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "मतलब वो मॉडल available नहीं है. लोकल एयर फ़िल्टर वाला है, सेम क्वालिटी. एड्रेस भेजो, डिलीवर कर देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "लोकल एयर फ़िल्टर"}}
{"product": "silicone sealant", "said": "अं कल सुबह आ जाएगा अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब पडलाय। 315... थांबा, लिस्ट बघतो... 250", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, haan, ready stock hai, season time hai na, demand jyada hai, courier kar denge.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "बोला, काय पाहिजे? बरं उद्या सकाळी माल येईल, आत्ता नाहीये.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "आमच्याकडे आहे, 275... नाही नाही, अडीचशे.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "putty padi hai acha aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब नहीं है, खत्म हो गया, वो ओसराम हैलोजन बल्ब दे सकता हूँ, डेढ़ सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओसराम हैलोजन बल्ब"}}
{"product": "Schneider Classic switch", "said": "मतलब हाँ है 495 इन्क्लूसिव ऑफ जीएसटी", "answer": {"in_stock": true, "price_inr": 495, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ho, thevto aamhi, ahe, shambhar rupayanna padel, yeun gheun ja", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Roots horn", "said": "देखिए जी, उपलब्ध है. भेज देंगे, पचास रुपये डिलीवरी चार्ज लगेगा. आठ सौ पचपन का है, प्लस डिलीवरी", "answer": {"in_stock": true, "price_inr": 855, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "है ना, कितने चाहिए? आपसे ढाई सौ लेंगे, लिखा 280 है", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "बोला, काय पाहिजे? बाजारात माल कमी आहे सध्या। आज सकाळपासून लाईट नाही। माल नाहीये। अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "जी, उपलब्ध है, एक का एक सौ, 12 का डिब्बा 1040 का", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan toh kisse baat karni hai? ye wo dukaan nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हेलो? हाँ, बोलो भाई. अभी बिज़ी हूँ, कॉल बैक करता हूँ. बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हेलो? हाँ, बोलो भाई। बेसिकली नॉट अवेलेबल सर।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "maal kal ayega, abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "माल नाहीये, Crompton 1500 watt rod आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 1500 watt rod"}}
{"product": "Orient ABS room heater", "said": "है भाई, मिल जाएगा, 1800... नहीं नहीं, 1490 है.", "answer": {"in_stock": true, "price_inr": 1490, "delivery": null, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "अभी available नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा, 590 रुपये लगेंगे, delivery बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ है, चालीस का है, इससे कम नहीं होगा, मतलब बॉय भेज दूंगा", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Ho, bijagri aahe. pannas final", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "अभी नहीं है, वो Pick Ur Needs टॉर्च वाला है, पौने आठ सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pick Ur Needs टॉर्च"}}
{"product": "Royal Enfield chain lube", "said": "हां तर हो आहे पावणेदोनशेला देतो देतो पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "hai hai... nahi yaar, last piece subah bik gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "नमस्कार, बोला. हो, पंखा शिल्लक आहे. किंमत नंतर सांगतो, आत्ता लिस्ट नाही.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "अभी नहीं है, ऑर्डर पर मंगवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "यससर बोलिए यस अवेलेबल है आपको 1860 MRP तो 2200 है", "answer": {"in_stock": true, "price_inr": 1860, "delivery": null, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "ha sir, boliye, haan toh nahi hai ji, khatam, Clouds wiper blade rakha hai, dukan pe aake lena padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Clouds wiper blade"}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "rate saat sau hai, par abhi stock nahi hai, abhi out of stock hai, new stock kal aa raha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Haa, ready stock hai, teen sao hai, 290 nahi.", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "हेलो? हाँ, बोलो भाई, स्टॉक नहीं है अभी, एक कस्टमर खड़ा है, होल्ड करो, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, देखो Retina cover है, साढ़े चार सौ का, सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina cover"}}
{"product": "Usha 1000W immersion rod", "said": "Ho sir, available aahe, price pavanepachshe aahe, mhanje boy ghari deliver karel.", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "chalis rupaye ka hai, han, mil jayega bhai", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Han, stock men hai. aapke liye sath. khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "हाँ जी, स्टॉक में पड़ा है, सवा अड़तीस सौ लगेगा, चार हज़ार एक सौ वाला rate पुराना है, वो नहीं", "answer": {"in_stock": true, "price_inr": 3825, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "hallo haan awaaz aa rahi hai bolo savaa saat sau last rate hai mil jayega hai", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "haan haan, bolo, arre rate saadhe unnis sau ke aas paas hoga, stock malik se puchna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "han bhai, bolo, hai na, kitne chahiye? 410 ka hai, isse kam nahin hoga, shaam tak ghar pe aa jayega", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "maal kal aayega, abhi nahi hai, courier kar denge sir", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "माल खत्म है. लड़का चाय लेने गया है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "aamchyakade nasta te, Easy4buy horn deu shakto, 1040 padel, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Easy4buy horn"}}
{"product": "Splendor brake shoe", "said": "teen sau nabbe ka padega, yes, available hai", "answer": {"in_stock": true, "price_inr": 390, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "हां बोला साहेब हां हां ऐकतोय आज सकाळपासून लाईट नाही मशीन आहे की नाही माहीत नाही गोडाऊनमध्ये बघतो आवाज तुटतोय मोठ्याने बोला", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मतलब हाँ रेडी stock है 20 है 24 नही", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "yes bola ho machine milel savvaekkechalisshe la lavto mrp char hazar naushe pannas ahe gharpoch milel", "answer": {"in_stock": true, "price_inr": 4125, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हाँ जी, मिलजाएगा, 815 रुपये, अभी डिलीवरी बंद है.", "answer": {"in_stock": true, "price_inr": 815, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Kya hai na hai na, kitne chahiye? ek hajar teen sau pachas bola kya? galti se bola, terah sau hai", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "Haan ji, boliye, season time hai na, demand jyada hai, haan toh do sau pachhattar final, haan, cover hai, network issue hai, aawaz break ho rahi hai", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "ha, mil jayega bhai, 370 last rate hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "मतलब यस, अवेलेबल है, सवा चार सौ का है, लोकल ग्लू चाहिए तो वो भी है, 545 वाला", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "Available hai, satrah sau rupaye, shaam tak pahuncha denge, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 1700, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "बेसिकली हाँ हाँ, है. MYRA लॉक का रेट एक सौ पैंतीस है, ये वाला पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ है। सिंगल पीस 185, छह का फुल बॉक्स 955। मतलब भेजने का इंतज़ाम नहीं है।", "answer": {"in_stock": true, "price_inr": 185, "delivery": false, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "२०५० लगेगा। हाँ, ऑयल है। और कुछ?", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "हां बोला म्हणजे हो मशीन शिल्लक आहे एक हजार नऊशे रुपये कमी नाही नाही होणार संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": 1900, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "आज नहीं है, कल आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "है भाई, रेट सौ है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "लाईट पडलीये. पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "अं ख़त्म हो गया था... नहीं, एक बचा है, डेढ़ हज़ार रुपये का है, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "स्टॉक में है, २३५० लगेगा, लड़का छोड़ देगा घर पर, औरकुछ?", "answer": {"in_stock": true, "price_inr": 2350, "delivery": true, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "हैलो हाँ आवाज़आ रही है मतलब हैलो आवाज़ नहीं आ रही बाद में करना लड़का चाय लेने गया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हो सर available आहे price सव्वा पाचशे आहे", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हम्म नही है जी, खत्म, local seal किट वाला पडा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal किट"}}
{"product": "Osram LED headlight", "said": "light sampli haan haan aiktoy aatta nahiye parava paryant yeil aaj sakali light navhti savvacharshe fix", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हेलो, हाँ बोलो, जी, उपलब्ध है, मतलब प्राइस पच्चीस है", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "घंटे भर बाद कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ, रखा है, आपके लिए 380.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "है भाई, मिल जाएगा, तीन सौ अस्सी लगा देंगे, वैसे 450 का है, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पडलीये. तुम्हाला सहाशे तीस, छापील किंमत सातशे पंधरा.", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "देखो अभी नहीं है, परसों तक आ जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "वो तो कल ही ख़त्म हो गया. iBELL 200A machine वाला है, सेम क्वालिटी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "iBELL 200A machine"}}
{"product": "Safeyura LED batten", "said": "हां माल पड़ा है सात सौ पैंसठ का नहीं नहीं गलत बोला सात सौ साठ का है", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "हैलो, कौन? यस, अवेलेबल है. अच्छा. अं चार सौ पंद्रह रुपयेरुपये. सॉरी, डिलीवरी सर्विस नहीं है.", "answer": {"in_stock": true, "price_inr": 415, "delivery": false, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "वो वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "पौने ग्यारह सौ रुपये का है है ना कितने चाहिए वो क्या है बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हैलो, कौन? हाँ, माल पड़ा है। रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "बिजागरी पडलीये, बारा घेतले तर बासष्ट प्रमाणे, एक पीस सत्तर", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "hai ji, mil jayega.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "नही है, बोलो तो मंगवा देता हूँ, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है ना, कितने चाहिए? 170 रुपये का है", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "एक मिनिट हां... बोला, हो, पंखा शिल्लक आहे, तुम्हाला १६०० ला देतो.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "एक गिऱ्हाईक आहे एक मिनिट पुट्टी संपली नाही आहे दिवाळीची गर्दी आहे ना आज सकाळपासून लाईट नाही 710 पडेल 730चा रेट जुना आहे तो नाही अजून काही", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Haan, hai apne paas, acha do sau pandrah MRP hai, ek sau pachasi men ho jayega.", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, रेडी स्टॉक है। प्राइस तीन सौ तीस है। अं घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए, वो हम नहीं रखते, वो, Havells 2.5 sq mm wire दे सकता हूँ, 2050 रुपये का, दुकान से ले जाना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 2.5 sq mm wire"}}
{"product": "Siron 60W soldering iron", "said": "हाँ है आपके घर तक भिजवा देंगे और कुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Orient ABS room heater", "said": "Ha sir, bataiye, available hai, koi problem nahin, savaa satrah sau ka hai, ghar tak deliver kar denge.", "answer": {"in_stock": true, "price_inr": 1725, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "अं डिलीवरी नहीं करते, दुकान पर आकर ले जाइए, और और कुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Mihsp water pump", "said": "हां हां हां, है. 4700 रुपये. दुकान से ले जाना पडेगा", "answer": {"in_stock": true, "price_inr": 4700, "delivery": false, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "नहीं नहीं, है ना। 8398 रुपये।", "answer": {"in_stock": true, "price_inr": 8398, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "Hai ji, mil jayega, aath sau das rupaye ka hai, aath baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 810, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "haan ji, boliye, aapko number kisne diya? acha thik hai, aapke liye 1600, aaj subah se light nahi hai, umm haan, hai apne paas, ladka ghar pe de jayega, thik hai?", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Bosch drill machine", "said": "मशीन खत्म नहीं हुई, है, नौ हज़ार तीन सौ... रुकिए, लिस्ट देखता हूँ... नौ हज़ार चार सौ", "answer": {"in_stock": true, "price_inr": 9400, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "डिलीवरी नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "नाय हो, नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "हाँ सर वो मॉडल अवेलेबल है सर", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "आहे की, मिळेल. साडे पाच हजारला पडेल", "answer": {"in_stock": true, "price_inr": 5500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Wire padi hai, aapse 2100 lenge, likha 2420 hai", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "हाँ, रखा है, पौने दो सौ में देदेंगे.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी का स्टॉक चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haa, mal pada hai, aapke liye pachasi, home delivery nahi hai bhai", "answer": {"in_stock": true, "price_inr": 85, "delivery": false, "alternative": null}}
{"product": "Khadija angle grinder", "said": "बाज़ार में माल की कमी चल रही है. कंपनी ने फिर रेट बढ़ा दिया. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. मशीन ख़त्म हो गई. लड़का चाय लेने गया है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "हैलो हाँ आवाज़ आ रही है मतलब सर एक मिनट चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हेलो? हाँ, बोलो भाई. सर आज बहुत बिज़ी डे है. सीज़न टाइम है ना, डिमांड ज़्यादा है. बेसिकली एक कस्टमर खड़ा है, होल्ड करो. अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हेलो? हाँ, बोलो भाई, हाँ सर, वो मॉडल अवेलेबल है, मतलब 375 MRP लिखा है, 320 लगा देंगे, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ बोलिए। साढ़े तीन सौ में दे देंगे। जी, उपलब्ध है। लड़का नहीं है भेजने को, आकर ले जाइए।", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बोला ना, पुट्टी संपलीये, बाजारात माल कमी आहे सध्या, लोकल पुट्टी घ्या, ते चालेल, काय सांगू, सकाळपासून गर्दी आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल पुट्टी"}}
{"product": "wall putty 20 kg", "said": "बोला ना पुट्टी नाहीये संपली बिर्ला वॉल putty 40 kg आहे त्याचा प्राइस सव्वासहाशे पिकअप करावा लागेल हां", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "बिर्ला वॉल putty 40 kg"}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "सॉरी सर, अभी आउट ऑफ स्टॉक है, बॉश वाइपर पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश वाइपर"}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "haan, maal pada hai, full packet sadhe chaar sau, loose piece do sau pachattar ka.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "अभी नहीं है, परसों तक आ जाएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "mhanje stock madhe aahe tumhala pavne donshe la deu sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "pavanecharshe final, ho, tube milel.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "आपको आकर लेना पड़ेगा. तो. औरकुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "कल सुबह आ जाएगा, अभी नहीं. आकर ले जाओ.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "तुम्हाला साडे अकराशेला देऊ, हो, आहे शिल्लक", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "यस, available है, आपको तीन सौ नब्बे, MRP तो पाँच सौ पाँच है.", "answer": {"in_stock": true, "price_inr": 390, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "ek minute... haan bolo, stock mein sirf 20 piece hai, rate puch ke batata hun", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ बोलिए सर। हाँ हाँ, है। आपको पचासी, MRP तो एक सौ पाँच है।", "answer": {"in_stock": true, "price_inr": 85, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हेलो? हाँ, बोलो भाई, अभी स्टॉक नहीं है, कल तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "हां हां. अं आमच्या कडे नसतं, ऑर्डर करून मागवतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "अं अं हो sir available आहे pickup करावा लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "Bara aahe sir, ready stock aahe, delivery deto sir", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हो, आहे शिल्लक, पंधरा रुपये होतील, हो, दोन पीस आहेत", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हाँ हाँ बोलिए सर. नहीं है, मंगवा देंगे, दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "बघा ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "रेट सवा पाँच सौ है, मतलब हाँ, माल पड़ा है, ठीक है?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Crispy LED torch", "said": "405 में दे देंगे. हाँ हाँ, है. अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 405, "delivery": false, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "Abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "बेसिकली पहले पचास का था अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "kya hai na abhi khatam hai, shaam tak aa jayega, sadhe unnis sau last rate hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "२० kg नहीं है local putty है दुकान से ले जाना पडेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local putty"}}
{"product": "bike headlight bulb", "said": "bulb ahe ka baghava lagel, nantar sangto, ek giraik ahe, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "यस सर, बोलिए, जी, उपलब्ध है, तो, सौ रुपये.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "acha ji available hai 10 peice wala box 2310 mein khula ek 265 ka", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "गिझर संपला नाही, आहे, 2850चं आहे, तुमच्यासाठी पावणेसव्वीसशे, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 2575, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हो, १३ पीस आहेत, १५ रुपये होतील.", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो, बिजागरी आहे, सव्वाशे होतील, बघा, पाठवलं असतं, पण आज पोरगा नाही, येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "मिल जाएगा है अपने पास अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हो sir, अवेलेबल आहे, पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "हाँ है कोई प्रॉब्लम नहीं भेज देता पर आज बॉय छुट्टी पे है आके ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "हाँ जी, स्टॉक में पड़ा है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "आज नहीं है, कल आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "है भाई, साढ़े बारह सौ लगा देंगे, वैसे एक हज़ार छह सौ का है", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "haa hai 10 loge to 180 ke hisaab se, ek ka sava do sau apna aadmi de aaega", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "है भाई, एक पीस पौने तीन सौ, 6 लोगे तो दो सौ पचास के हिसाब से", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bagha bulb sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ, रखा है. अस्सी लगेगा, पचहत्तर वाला rate पुराना है, वो नहीं", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "कव्हर संपलं नाही, आहे, साडे तीनशे रुपये होतील, पोरगा पाठवतो, पन्नास रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Tata Agrico plier", "said": "प्लास पड़ा है 239 लगा देंगे वैसे 290 का है शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 239, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ख़त्म हो गया, Exide 12V 9Ah बैटरी है, सेम क्वालिटी, सॉरी, डिलीवरी सर्विस नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Exide 12V 9Ah बैटरी"}}
{"product": "4 socket extension board", "said": "हाँ हाँ रेडी stock है बॉस", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "namaste ji, boliye, rate ikkis sau hai, haa haan, hai", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "तर हो, ठेवतो आम्ही, आहे. Carbanao कार मॅट पण आहे, त्याचे चार हजार सातशे पन्नास, आणि तुम्ही विचारलेला साडेछत्तीसशेला", "answer": {"in_stock": true, "price_inr": 3650, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, माल पड़ा है, ढाई सौ लगेगा, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "तर बल्ब पडलाय। पाठवतो, 50 रुपये डिलिव्हरी चार्ज लागेल। मालाचे 150, डिलिव्हरी वेगळी", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Raaisin car cover", "said": "कव्हर आहे मिळेल बघा अडतीसशे होतील अं माणूस पाठवून देतो घरी", "answer": {"in_stock": true, "price_inr": 3800, "delivery": true, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "तर हो, आहे शिल्लक। 490चं आहे, तुमच्यासाठी साडे चारशे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "यस sir, बोलिए. एक कस्टमर खड़ा है, होल्ड करो. हाँ, ऑयल है. सीज़न टाइम है ना, डिमांड ज़्यादा है. दो सौ पचासी रुपीज़ है. नेटवर्क इशू है, आवाज़ ब्रेक हो रही है. sir आज बहुत बिज़ी डे है. घर पर भेज देंगे, 30 अलग से", "answer": {"in_stock": true, "price_inr": 285, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "खत्म नही हुआ अभी भी है पौने छह सौ का का है इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "है भाई 60 का है कम नहीं होगा शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "ho bola manus pathavto gharee", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "hai ji, mil jayega, umm 60 rupaye ka hai, delivery ho jayegi.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ते आमच्याकडे नसतं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan, tape hai, 20 ka hai, thread seal tape bhi hai, wo 18 ka.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "haan, hai apne paas, 255 laga denge, waise 330 ka hai", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "सौ रुपये, मिल जाएगा, है अपने पास", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो, प्लेट शिल्लक आहे। तुम्हाला ५७० ला। पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 570, "delivery": false, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "acha kal phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "Hello, haan bolo, wo toh kal hi khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "Umm atharah piece available hai abhi, delivery free hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "ha sir, bataiye, stock men hai sir, sadhe gyarah sau rupees, stock mein sirf unnis piece hai.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ बोलिए सर। वो मॉडल अवेलेबल नहीं है। Exide 12V 5Ah battery वाला अवेलेबल है, उसका प्राइस पंद्रह सौ पचास।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide 12V 5Ah battery"}}
{"product": "Savsol 20W-40 engine oil", "said": "हाँ हां माल पडा है ढाई सौ रुपये का है भेजने वाला कोई नही है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "haa, hai apne paas, acha 5100 rupees lagenge, address bhej do, ladka de ayega, kal Sunday tha to dukaan band thi, awaaz kat rahi hai, zor se bolo.", "answer": {"in_stock": true, "price_inr": 5100, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Kya hai na khatam ho gaya tha... nahi, ek bacha hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "है भाई। हज़ार लगेगा। लड़का भेज देता हूँ।", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "पहले 310 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "हो sir, बोला, हो sir, अवेलेबल आहे, 3650 फायनल", "answer": {"in_stock": true, "price_inr": 3650, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "हां बोला... नाही, ते सध्या नाही। त्याच्याऐवजी Stanley 550 watt drill देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley 550 watt drill"}}
{"product": "teflon tape", "said": "है भाई डिलीवरी बंद है अभी दुकान से ले लो", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ताला नहीं है, खत्म हो गया। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। Godrej 40 mm padlock है, वो ले लीजिए। एक ग्राहक खड़ा है, एक मिनट। शाम तक घर पहुँच जाएगा। त्योहार का सीज़न है ना", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Godrej 40 mm padlock"}}
{"product": "50 mm padlock", "said": "Tala nahin hai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "हेलो, हाँबोलो, मोटर ख़त्म नहीं हुई, है, छब्बीस सौ लगेगा लगेगा, अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "आहे की, मिळेल, रेट मालकांना सांगतो, कुरिअर करतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "haan ji boliye check karke batata hun market me maal ki kami chal rahi hai company ne is mahine phir rate badhaya aapko number kisne diya acha thik hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हैलो, कौन? हाँ, माल पड़ा है। देखो साढ़े तीन सौ लगेगा। घर पर भेज देंगे।", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Havells LED torch", "said": "maal kal aayega, abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "आत्ता नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "मिल जाएगा, है अपने पास। रेट ३०० है।", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "हाँ, रखा है. रेट साढ़े तेईस सौ है. डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": 2350, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ha, ready stock hai, price 1450 hai, same day delivery kar denge.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "udya sakali maal yeil, aatta nahiye, tinashe ainshi rupaye, kami nahi honar.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हो, आहे। आता पावणेपाचशे आहे, आधी तीनशे पासष्ट होता", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "हो, टायर आहे. तुम्हाला पावणे तेहतीसशेला देतो", "answer": {"in_stock": true, "price_inr": 3275, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "हाँ जी, स्टॉक में पड़ा है. बाज़ार में माल की कमी चल रही है. ३८५० रुपीज़ है. मतलब आज सुबह से लाइट नहीं है", "answer": {"in_stock": true, "price_inr": 3850, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "stock नाही आहे सध्या अं Ritco tap घ्या ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ritco tap"}}
{"product": "Dewalt angle grinder", "said": "मशीन नही है, खत्म हो गई, वो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "स्टॉक नहीं है अभी, Usha ceiling fan है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha ceiling fan"}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "है ना, कितने चाहिए? पौने छह सौ रुपये का है.", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "जी, उपलब्ध है, ढाई सौ का पड़ेगा, घर पर भेज देंगे, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "han ji, boliye, abhi khatam hai bhai, bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हाँ, रेडी स्टॉक है. रेट अभी पता नहीं, ओनर से पूछना पड़ेगा. बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "mana kisne kiya hai na", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "मशीन नहीं है, खत्म हो गई, और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "बोला, काय पाहिजे? ट्यूब संपली, त्याच्याऐवजी Duravolt batten वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duravolt batten"}}
{"product": "Morphy Richards 2000W room heater", "said": "पौने सत्ताईस सौ का है, इससे कम नहीं होगा, स्टॉक में है, लड़का घर पे दे जाएगा, और कुछ?", "answer": {"in_stock": true, "price_inr": 2675, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "बॉय भेज दूंगा, ठीक है?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां ऐंशी रुपये, कमी नाही होणार, हो दादा, आहे", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "स्टॉक में है। एक सौ नब्बे रुपये का है। अं। भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हाँ भाई बोलो सवा हजार का पड़ेगा मिल जाएगा है अपने पास आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "han, kaun? acha, bolo, nahi hai, sorry, ek customer khada hai, ek minute.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ जी, ताला रखा है. दो सौ बीस final price है. delivery की टेंशन मत लो, भेज देंगे.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "हां जी, नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "नहीं है सर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "नाही कसं, आहे ना", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "आहे, अजून पाच पीस पडलेत, बघा रेट मालकांना विचारून सांगतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब नही है खत्म हो गया एक ग्राहक खडा है एक मिनट", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "साढे पंद्रह सौ लगेगा। वो क्या है लाइट पड़ी है", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "रेट 650 है, पर अभी stock नहीं है. आज नहीं है, कल आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "डेढ़ सौ का आता है लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "अं गलत नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हां जी मिलजाएगा सवा तीन सौ रुपये का है लड़का भेज देता हूं हम्म", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायरचा स्टॉक बघून सांगतो. म्हणजे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "aatta nahiye, parva paryant yeyil. tumhala deedashe la deto. address pathva, pathavto.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "अच्छा हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "अं हाँ रखते हैं पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "यस अवेलेबल है 695 इन्क्लूसिव ऑफ जीएसटी", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हेलो, हां बोलो, अभी स्टॉक नहीं है, कल तक आजाएगा, अपना लडका छोड देगा घर पे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "Umm han, stock mein hai, aapke liye paune do so.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Ek sau pachpann final ji, available hai.", "answer": {"in_stock": true, "price_inr": 155, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Sir ek minute, check karke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "हाँ है, कोई प्रॉब्लम नही, पौने ग्यारह सौ का पडेगा", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "हेलो हाँ बोलो भाई जी उपलब्ध है पौने तीन सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "1 kg नहीं है, local glue है, शाम तक घर पहुँच जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local glue"}}
{"product": "bike headlight bulb", "said": "हो, ठेवतो आम्ही, आहे। तुम्हाला पावणेदोनशेला देतो।", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "वो हाँ, है, दस लोगे तो एक सौ पचास के हिसाब से, एक का पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "नाही म्हणालो का आहे हो", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हाँ, है. 500 रुपये. लड़का छोड़ देगा घर पर.", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "20 पीस अवेलेबल है अभी औरऔर कुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "नमस्ते, बोलिए. वो आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "नॉट अवेलेबल सर. गैलेक्सी कार कवर अवेलेबल है, उसका प्राइस 1300.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गैलेक्सी कार कवर"}}
{"product": "Benz helmet", "said": "Benz नहीं है, Sonik 7 हेलमेट है, अच्छा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik 7 हेलमेट"}}
{"product": "4 inch door hinge", "said": "हेलो? हाँ, बोलो भाई, price सौ है, कब्ज़ा है, मिल जाएगा, courier कर देंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "han haan bolo nahi milega bhai birla wala hai chhah so sath ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "birla"}}
{"product": "4 socket extension board", "said": "Hai bhai rate pauna chaar sau hai shaam tak pahunchaa denge.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "हां भाई बोलो हाँ है सवा बारह सौ का है इससे कम नहीं नहीं होगा हां तो पता भेज दो लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 1225, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी, वायर रखी है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "hai bhai, paune aath sau final, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हेलो, हाँ बोलो, मतलब अभी स्टॉक नहीं है, कल तक आ जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "आहे की, मिळेल, सहाशे तीस फिक्स, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 630, "delivery": false, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हो, मिळेल। बघा 6 घेतले तर 255 प्रमाणे, एक पीस 300", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "नेटवर्क इशू है आवाज़ ब्रेक हो रही है सीज़न टाइम है ना डिमांड ज़्यादा है वो आइटम हम नहीं रखते Avita switch है सेम क्वालिटी तो एक कस्टमर खड़ा है होल्ड करो बताओ क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Avita switch"}}
{"product": "Activa clutch plate", "said": "हाँ, है, साढ़े चार सौ का पड़ेगा, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "mil jayega, hai, baais sau pachas me de dunga, ladka de ayega, 20 rupaye auto ka", "answer": {"in_stock": true, "price_inr": 2250, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "रेट सातशे पंचाण्णवच्या आसपास असेल स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "हेलो, हेलो, हाँ बोलो, गीज़र है, मिल जाएगा, 2950 प्लस GST", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ सर, वो मॉडल अवेलेबल है, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "साडेचारशे रुपये। हो, ठेवतो आम्ही, आहे। हां येऊन न्यावं लागेल।", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "JCB plier", "said": "hai na, kitne chahiye? 440 mein de dunga, address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 440, "delivery": true, "alternative": null}}
{"product": "Havells Oro switch", "said": "हम्म है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "हाँ भाई, बोलो, सेठ जी बाहर गए हैं, वही बताएँगे, एक ग्राहक खड़ा है, एक मिनट.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "haan sir, boliye, haan, maal pada hai, teen sau pachattar final.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "ho bola, ho, milel, 152 la lavto, MRP 180 ahe, pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": 152, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "aamchyakade nasta, order karun magvto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "नहीं है... रुको रुको, है, एक पड़ा है. अं एक सौ सत्तर रुपीज़ है. कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "stock में है sir, प्रिंटेड एमआरपी दो सौ सत्तर, अपना price डेढ़ सौ, delivery बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "है ना, कितने चाहिए? अं, पहले 1050 था, अब 870 हो गया, सस्ता हुआ है, भेजने वाला कोई नही है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 870, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "50 mm संपलं, हॅरिसन 60 एमएम कुलूप आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हॅरिसन 60 एमएम कुलूप"}}
{"product": "silicone sealant", "said": "हां, है, सवा तीन सौ फाइनल प्राइस है, हम्म, डिलीवरी हो जाएगी सर.", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan haan, hai bhaiya.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "aahe ki, milel, 100 rupaye, kami nahi honar, gharapoch pohochvun deu.", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हां जी मतलब हाँ 8 piece पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे आहे. बघा तुम्हाला साडेनऊशेला देऊ", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "एक गिऱ्हाईक आहे, एक मिनिट. आज सकाळपासून लाईट नाही. हां हां, ऐकतोय. आवाज तुटतोय, मोठ्याने बोला. स्टॉकमध्ये आहे. बरं आधी 975 होता, आता 1249 झाला. अजून काही?", "answer": {"in_stock": true, "price_inr": 1249, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Matlab nahi bhai, yahan nahi milega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "हैलो हां आवाज आ रही है रेट १०५० है sir stock में है sir", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "मशीन ख़त्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "एक कस्टमर खड़ा है, होल्ड करो, जी, उपलब्ध है, साढ़े आठ हज़ार लगेगा, सर आज बहुत बिज़ी डे है, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 8500, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "umm haa, stock me chaar piece hai, ek sau sath final price hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "हाँ जी sir, बताइए, हाँ, रखा है, सवा पाँच सौ रुपये का है, आपको shop से pickup करना होगा.", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हॅलो आवाज येतोय का हां बोला हां तर रेट ऐंशीच्या आसपास असेल स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हाँ भाई, बोलो, यस, available है, हाँ तो आपको एक सौ पचहत्तर में मिलेगा, MRP 265 है.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "hai bhai. ek ka assi, dus ka dabba saat sau panch ka", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "नमस्कार, बोला। मोटर आत्ता नाहीये, संपली। त्याच्याऐवजी किर्लोस्कर 0.5 HP वॉटर पंप देऊ शकतो। अं कुरिअर करतो।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "किर्लोस्कर 0.5 HP वॉटर पंप"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हां बोला साहेब, हो, आहे, बघा सव्वाशे रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "नमस्ते, बोलिए. स्टॉक में है. तीन सौ बीस में दे देंगे. सोलह पीस हैं स्टॉक में. अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "dekho haan sir, available hai chah ka full box 1380, single piece do sau sath", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "है ना कितने चाहिए मतलब आपको डेढ़ सौ MRP तो एक सौ पचासी है", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "haan sir, bataiye, sir ek minute, check karke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "stock में है सर फुल पैकेट 280 लूज piece 170 का delivery नहीं होती pickup करना पडेगा", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "haan toh han, wahi wala hai. ek piece sava teen sau, chhah wala box loge to 1680. delivery nahi hai", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip", "said": "हाँ, रखा है, चार सौ पचासी final प्राइस है, डिलीवरी नहीं होती, pickup करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 485, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "आहे सर, रेडी स्टॉक आहे, दोनशे पन्नास आणि 18 टक्के GST वेगळा, पोरगा घरी सोडून देईल", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "हो कव्हर आहे बघा रेट बदललाय आता ३६० आधी ३३० होता", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अं। बॅटरी संपली नाही, आहे। 1500 रुपयेहोतील। घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा, आकर ले जाओ.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Meco multimeter", "said": "हो हो, आहे। 2200 पडेल, 2660चा रेट जुना आहे, तो नाही। पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 2200, "delivery": true, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ सर, अवेलेबल है, सवा दो सौ का एक, छह का डब्बा 1180 का.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Galat number hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Kabza nahi hai, khatam ho gaya, steel hinge hai, sau ka, delivery free hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "steel hinge"}}
{"product": "4 inch door hinge", "said": "बिजागरी संपली नाही आहे बरंरेट तीस आहे येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "है भाई, मिल जाएगा. 2980... नहीं नहीं, पच्चीस सौ है", "answer": {"in_stock": true, "price_inr": 2500, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "आहे आहे की, मिळेल। अं। एमआरपी 305, आपला रेट पावणेतीनशे।", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Han ji, putty available hai, aath so assi lagega, ladka de aayega, 50 rupaye auto ka.", "answer": {"in_stock": true, "price_inr": 880, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "hai, bilkul hai, MRP teen sao assi likha hai, sadhe teen sau laga denge, bhijwa dunga.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "है भाई, मिल जाएगा. आपके लिए साढे आठ सौ", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "जीबोलिए. नहीं है. वो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हैलो कौन हम्म हाँ जी मिल जाएगा तो एमआरपी 375 है पर हम 320 में देते हैं", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "सर आज बहुत बिज़ी डे है. एक कस्टमर खड़ा है, होल्ड करो. नेटवर्क इशू है, आवाज़ ब्रेक हो रही है. है भाई, मिल जाएगा. सवा पाँच सौ लगेगा, 590 वाला रेट पुराना है, वो नहीं. घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 525, "delivery": true, "alternative": null}}
{"product": "RNG Eko tyre", "said": "Haan haan, hai. shaam tak pahuncha denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "हैलो, कौन? अच्छा कवर पड़ा है। घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हैलो? हाँ, आवाज़ आ रही है, कंपनी ने फिर रेट बढ़ा दिया, आवाज़ कट रही है, ज़ोर से बोलिए, हाँ जी, मिल जाएगा, रेट बदल गया है, अब एक सौ नब्बे है, पहले एक सौ पचपन था, लड़का चाय लेने गया है, लड़का घर पे दे जाएगा, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, आहे। आठशे आहे, एक हजार नाही", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "maharaja nahin hai havells heater hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "havells heater"}}
{"product": "16A MCB", "said": "मतलब अभी नहीं बता सकता, बाद में फ़ोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "है भाई, तीन हज़ार बोला क्या? माफ़ करना, पौने छब्बीस सौ है.", "answer": {"in_stock": true, "price_inr": 2575, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "अभी स्टॉक नहीं है, कल तक आ जाएगा, कूरियर कर देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "नमस्कार बोला बोला मोटर आहे की नाही माहीत नाही गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan haan bolo haan plug hai 210 lagega", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हो सर, अवेलेबल आहे, अडीचशेला पडेल", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "हाँ जी sir, बताइए, पंखा नहीं है, खत्म हो गया, उसकी जगह Crompton fan ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton fan"}}
{"product": "Starvin pedestal fan", "said": "देखो, नमस्ते, बोलिए, पंखा है, मिल जाएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "Hello, ha boliye, main naya hun, mujhe nahin pata, seth se puchho.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "अं हाँ सर, अवेलेबल है. MRP ५९० है, आपको ३७५ में दे देंगे.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "म्हणजे हो, आहे, रेट दोनशे सत्तर आहे सर", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "ha, teen piece pade hain, arre 140 ka hai, apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1.5 sq mm संपलं, Havells 2.5 sq mm wire आहे, चालेल? पाठवायची सोय नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 2.5 sq mm wire"}}
{"product": "iBELL 200A welding machine", "said": "Nahin hai. BMB welding machine hai, uska rate 5600.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "BMB welding machine"}}
{"product": "Dow silicone sealant", "said": "देखो माल खत्म है Anabond sealant वाला है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond sealant"}}
{"product": "Splendor brake shoe", "said": "Namaste ji, boliye, wo toh kal hi khatam ho gaya, awaaz kat rahi hai, zor se bolo, local brake shoe hai, 170 ka, Diwali ka season hai na, sab yahi maang rahe hain, aapko number kisne diya? acha thik hai, ladka bhej deta hun, haa han, sun raha hun, bolo kya karna hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "Duro Gold angle grinder", "said": "अच्छा हाँ हाँ, है। सवा हज़ार का पड़ेगा।", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "Tyre ahe, milel, 1830 rupaye, kami nahi honar, sandhyakalee pathavun deto", "answer": {"in_stock": true, "price_inr": 1830, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "यस, अवेलेबल है, आपको सवा दो सौ, MRP तो दो सौ पचासी है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Arre ha ji, mil jayega. sadhe chaar sau ka padega.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "अं हां माल पड़ा है पहले उनहत्तर था अब साठ हो गया सस्ता हुआ है", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "यस सर बोलिए वो क्या है पता नहीं भाई बाद में call करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Oro switch", "said": "हाँ भाई, बोलो। माल खत्म है। Schneider switch है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Schneider switch"}}
{"product": "4 socket extension board", "said": "हाँ जी सर, बताइए. स्टॉक में है सर. पाँच सौ नब्बे का है, इससे कम नहीं होगा. मतलब बॉय भेज दूंगा.", "answer": {"in_stock": true, "price_inr": 590, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ हाँ है देखो दो सौ अस्सी का नहीं नहीं गलत बोला दो सौ चालीस का है लड़का घर पे देजाएगा", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "haan rakhte hain, par abhi khatam hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "acha mana kisne kiya? hai na", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Haa sir, wo model available hai, bhej deta, par aaj ladka nahi aya, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Retina bike cover", "said": "हो, आहे, दोनशे पंधरा सांगितलं का? चुकलो, दोनशे साठ आहे", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "मिलजाएगा, है अपने पास। अं साढ़े सात सौ रुपये लगेंगे। भेजने का इंतज़ाम नहीं है। हम्म।", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हां जी हाँ सर वो मॉडल अवेलेबल है डिलीवरी की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "nahi kasa, ahe na. pavanebarashe la deto", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ha sir, available hai, ek sao chalis rupaye, umm khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "हो बोला दादा मशीन संपली", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "मोटर नाहीये संपली हां तर AmiciTools water pump देऊ शकतो 2700 पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools water pump"}}
{"product": "Activa air filter", "said": "अच्छा अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "ghante bhar baad call karo haan haa sun raha hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "hai, bilkul hai, list price 710, discount ke baad 655", "answer": {"in_stock": true, "price_inr": 655, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "Pavanebara hazar fix, stock madhe aahe, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 11750, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "मशीन पड़ी है, Golden Bullet 600W drill machine भी है, उसका 1380, और आपवाला साढे तेरह सौ का", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ, है, 10 का डिब्बा 1740 का, एक पीस 199", "answer": {"in_stock": true, "price_inr": 199, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "हाँ जी, बताइए, कवर पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँबोलिए, यस, अवेलेबल है, प्राइस सवा चार सौ.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan, hai apne paas, dedh sau hoga, aake le jao", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "ji, available hai, terah sau pachas lagega ab, pehle ek hajaar ka tha", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "bulb pada hai, bara ka dabba 2780 ka, ek peice do sau painsath", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "बघा संपलं होतं... नाही, एक आहे अजून.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Available hai, koi problem nahi, nabbe lagega, 98 wala rate purana hai, wo nahi", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "है ना, कितने चाहिए? पौने तीन सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "हाँ, है, सवा तेरह सौ का पड़ेगा, लड़का नहीं है भेजने को, आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 1325, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "nahi bhai, yahan nahi milega, umm Shell 10W-30 engine oil 900 ml chalega toh bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Shell 10W-30 engine oil 900 ml"}}
{"product": "water pump seal kit", "said": "हाँ सर, वो मॉडल अवेलेबल है, अं पौने दो सौ, जीएसटी एक्स्ट्रा, अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "आत्ता नाहीये, परवापर्यंत येईल, सव्वादोनशेला पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "chukicha number aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "Ha sir, boliye, abhi nahi hai, parso tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "वो, हाँ, है, 1160 लगेगा, बेसिकली delivery फ्री है", "answer": {"in_stock": true, "price_inr": 1160, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "म्हणजे हेल्मेट संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हैलो, कौन? स्टॉक में है। सौ लगेगा लगेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "देखो. अं अभी बिज़ी हूँ, कॉल बैक करता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "सवा नौ सौ का है, इस से कम नहीं होगा, हाँ हाँ है, कोई प्रॉब्लम नहीं", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "Haan ji, boliye, umm haan ji, mil jayega, address bhej do, ladka de aayega", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "पौने बारह सौ का है, हम्म, स्टॉक में है सर.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ है, 20 रुपये का है, बॉय भेज दूंगा, बताओ क्या करनाहै", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "Wo toh kal hi khatam ho gaya, umm ladka chai lene gaya hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "उद्या फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हाँ हाँ, है, रेट साढ़े तीन सौ है, अभी डिलीवरी बंद है, और कुछ?", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "लाईट आत्ता नाहीये, संपली, ओस्राम एलईडी हेडलाईट आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओस्राम एलईडी हेडलाईट"}}
{"product": "wall putty 20 kg", "said": "अं हो, 24 बॅगा आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "aahe na. bara charashe sattar la lavto, MRP 515 ahe. manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 470, "delivery": true, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "बोला ना. हो, लाईट शिल्लक आहे. रेट मालकांना विचारून सांगतो. पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Safeyura LED batten", "said": "मैं नया हूँ, मुझे नहीं पता, आज सुबह से लाइट नहीं है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "पता नहीं भाई, बाद में कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "अं बल्ब पडलाय। रेट एकशे चाळीस आहे। डिलिव्हरी करत नाही", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "Havells Cista room heater", "said": "haan, hai apne paas, do hajar bola kya? galti se bola, chaubis sau pachas hai", "answer": {"in_stock": true, "price_inr": 2450, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ, रखा है, रेट मालिक बताएँगे, हम्म अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हैलो? आवाज़ नहीं आ रही, बाद में करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "stock me hai sir do so pachas final price hai ghar tak deliver kar denge aur kuch", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "Ha haa, bolo, haa ji, bulb available hai, 230 rupaye, thik hai?", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ हाँ है तीन सौ पंद्रह नहीं नहीं पौने तीन सौ है हाँ तो खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "ho ahe, eka nagache pavanedonshe, panchvis nagancha box tina hazar aathshe pannas la, haan tar dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "sampla, aaj nahi, udya yeil, delivery nahi karat.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ जी, मिल जाएगा. एक पीस 98 का मिलता है, पैकेट पौने चार सौ का.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "डिलीवरी नहीं होती, पिकअप करना पड़ेगा, बताओ क्या करना है.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "Modx नहीं है SKC स्टीयरिंग कवर वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "SKC स्टीयरिंग कवर"}}
{"product": "Daybetter 25W LED bulb", "said": "हाँ रखा है अच्छा दो सौ पचानवे नही नही पौने तीन सौ है", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "हाँ जी, stock में पड़ा है। सिंगल piece पौने दो सौ, दस का फुल बॉक्स एक हज़ार पाँच सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "वो हम नहीं रखते Type R हॉर्न दे सकता हूँ साढ़े पाँच सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Type R हॉर्न"}}
{"product": "Prithvi 6.5 HP water pump", "said": "मोटर है, मिलजाएगी, Havells 1.0 HP water pump का rate 13240 है, ये वाला 10100.", "answer": {"in_stock": true, "price_inr": 10100, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "ho bola, ho dada, ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "maal khatam hai, Wipro 15 watt LED bulb hai, wo le lo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Wipro 15 watt LED bulb"}}
{"product": "2 inch paint brush", "said": "ji sir, tell me, kya hai na 40 ka hai, ji, available hai", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हॅलो, कोण बोलतंय? पिकअप करावा लागेल.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ जी,, स्टॉक में नहीं है, हम्म कंपनी ने फिर रेट बढ़ा दिया, आज सुबह से लाइट नहीं है, व्हाइट सीमेंट चलेगा? उसका एक सौ पचास है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "व्हाइट सीमेंट"}}
{"product": "AdroitZ bike cover", "said": "Yes, bola, ho, cover aahe, rate 415 aahe sir, delivery nahi, shop varun pickup kara.", "answer": {"in_stock": true, "price_inr": 415, "delivery": false, "alternative": null}}
{"product": "HTC multimeter", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ हाँ, है, साढ़े छह सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "acha maal khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "नमस्कार, बोला. ट्यूब पडलीये. एमआरपी 470, आपला रेट 410. दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 410, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "ख़त्म नहीं हुआ, अभी भी है. तीन सौ बीस लगेगा. आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 320, "delivery": false, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "अच्छा हाँ भाई वो आइटम हम नहीं रखते Goodyear वाला है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Goodyear"}}
{"product": "Safeyura LED batten", "said": "ट्यूब पडलीये भाऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "haa, mil jayega bhai, teen so chalis rupees lagenge", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "पक्कड पडलीये, आमचा रेट 170, MRP 285, येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 170, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Haan bhaiya, bataiye, bulb khatam nahi hua, hai, MRP 550 hai, discount ke bad teen sau painsath padega.", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हां जी पाँच पीस हैं स्टॉक में दो सौ बीस रुपये शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "वो स्टॉक में है सर, डिलीवरी हो जाएगी सर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "माहीत नाही, गोडाऊनमध्ये बघावं लागेल भाऊ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "वो मॉडल अवेलेबल नहीं है, उसकी जगह डीवॉल्ट एंगल ग्राइंडर ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डीवॉल्ट एंगल ग्राइंडर"}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "Nahi nahi, hai na, haan toh unnis sau pachas ka padega.", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ तो स्टॉक में है। एक सौ बीस बोला क्या? माफ़ करना, डेढ़ सौ है।", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "हाँ, माल पड़ा है, GST मिलाकर सात सौ पाँच", "answer": {"in_stock": true, "price_inr": 705, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Milel na, ahe. 1290 rupaye. gharpoch nahi, tumhala yava lagel. kalva mag", "answer": {"in_stock": true, "price_inr": 1290, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "अच्छा. मतलब अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा। सवा सौ लगेगा, एक सौ चालीस वाला रेट पुराना है, वो नहीं", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली नाही, आहे। 800 रुपये", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "कवर नहीं है, खत्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "हेलो? हाँ, बोलो भाई, स्टॉक में है sir, चार सौ पचासी लगा देंगे, वैसे 530 का है, घर पर भेज देंगे.", "answer": {"in_stock": true, "price_inr": 485, "delivery": true, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "बघा हो मिळेल साडे अठराशे होतील", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हो, ट्यूब शिल्लक आहे. साडेसहाशेला देतो.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "हां बोला, 14च पीस उरलेत, किंमत नंतर सांगतो, आत्ता लिस्ट नाही, डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "JY Super LED torch", "said": "हाँजी बताइए बताइए अभी नहीं है ऑर्डर पर मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हाँ रेडी स्टॉक है प्राइस कन्फर्म करके बताता हूँ और और कुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "स्टॉक में नही है iBELL 200A welding machine चलेगा उसका पौने दस हजार है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "iBELL 200A welding machine"}}
{"product": "wall putty 20 kg", "said": "पुट्टी आहे, मिळेल, नऊशे चाळीस रुपये, कमी नाही होणार, पोरगा घरी सोडून देईल", "answer": {"in_stock": true, "price_inr": 940, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "अच्छा चेक करके बताता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ हाँ, है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "namaste ji, boliye, haan toh 90 last rate hai, hai ji, mil jaega, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "बोला बोला, काय पाहिजे? आत्ता नाहीये, पोरगा चहा आणायला गेलाय, आवाज तुटतोय, मोठ्याने बोला, एक गिऱ्हाईक आहे, एक मिनिट, Crompton 3.15 mfd capacitor वाला देऊ शकतो, पन्नास पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 3.15 mfd capacitor"}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "हाँ हाँ भाई, बोलो. हम्म पंखा ख़त्म हो गया. एटमबर्ग fan है, सवा हज़ार का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एटमबर्ग fan"}}
{"product": "Havells 1500W immersion rod", "said": "बरं रेट सातशे पंचवीस आहे, हो, ठेवतो आम्ही, आहे, माणूस पाठवून देतो घरी.", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हेलो, हाँबोलो। अभी नहीं है। Ridroid एलईडी हेडलाइट अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ridroid एलईडी हेडलाइट"}}
{"product": "tank float valve", "said": "Haan sir, available hai, umm 320 last rate hai, sir aaj bahut busy day hai, network issue hai, aawaz break ho rahi hai, home delivery available hai, 20 charge.", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "हाँ बोलिए सर, पंखा पड़ा है, होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "नमस्ते, बोलिए, मतलब प्लेट पड़ी है, पौने पाँच सौ लगेगा, ठीक है?", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "मतलब हाँ, रखा है, तीन सौ पचास का है.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "तुम्हाला पस्तीसशेला देतो टायर पडलाय", "answer": {"in_stock": true, "price_inr": 3500, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप पडलंय 265चं आहे तुमच्यासाठी दोनशे चाळीस", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो, बल्ब आहे, सुटा दीडशेला, 12 नगांचा बॉक्स घेतला तर एक हजार सहाशे पन्नास", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "हो sir बोला नळ आहे मिळेल नेटवर्क प्रॉब्लेम आहे आवाज कट होतोय आज खूप बिझी आहे sir दोनशे पंचवीसला देऊ छापील किंमत तीनशे साठ आहे delivery नाही शॉपवरून pickup करा", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए चेक करके बताता हूँ एक ग्राहक खड़ा है एक मिनट", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "haan haan, hai, ek sau pachanve... are nahi, pauna do so.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "kya hai na available hai pentis rupaye bata dena", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "nahi vatta... thamba, aahe, ek urlay.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "उद्या फोन करा हांहां ऐकतोय", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "वो तो कल ही ख़त्म हो गया, मैकेनिकल सील वाला है, वो ले लीजिए, अं होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "मैकेनिकल सील"}}
{"product": "2 inch paint brush", "said": "है भाई. वो 65 में दे देंगे.", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "है ना, कितने चाहिए? अच्छा एक हज़ार दो सौ पचास बोला क्या? माफ़ करना, बारह सौ है, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "Anchor LED batten", "said": "Ho, tube ahe, MRP ekshe ainshi aahe, pan tumhala didshe la deu.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "बोला, काय पाहिजे? संपलंय। Orient हीटर आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient हीटर"}}
{"product": "tank float valve", "said": "ho dada, ahe, pavanedonshe fix, yeun nyava lagel.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "han ji, kabza available hai. pichle mahine 67 tha, ab assi. address bhej do, ladka de aayega", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "नहीं मिलेगा भाई जी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "है भाई, मिल जाएगा, रेट दो सौ साठ है सर, होम डिलीवरी अवेलेबल है, तीस चार्ज", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग पड़ा है, डेढ़ सौ में दे देंगे.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "हाँ है, कोई प्रॉब्लम नहीं। घर पर भेज देंगे, ४० अलग से। आइटम प्राइस ४२००, डिलीवरी चार्ज अलग", "answer": {"in_stock": true, "price_inr": 4200, "delivery": true, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "अभी नहीं है, लक्ष्मी पंप है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लक्ष्मी पंप"}}
{"product": "teflon tape", "said": "humare paas nahi hota, mangwana padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "देखिए कम नहीं है बहुत पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "ho bola, tumhala 185 la deto, ho, pakkad milel, sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "है भाई. वो ५७५ का पड़ेगा. भेज देंगे, ५० रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 575, "delivery": true, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "हाँ हाँ बोलिए तो नहीं है मतलब Raaisin cover चलेगा उसका सवा ग्यारह सौ है शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Raaisin cover"}}
{"product": "MK 16A switch", "said": "hai, bilkul hai, haan toh 140 laga denge, waise MRP 175 likha hai", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "देखिए प्लास पड़ा है, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "yes sir, boliye, 150 rupees ka hai, haan, ready stock hai, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "नॉट अवेलेबल सर, Sonik helmet है, चार सौ नब्बे का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik helmet"}}
{"product": "ESN 2000W immersion rod", "said": "haan ji, boliye, han, char piece pade hai, apna aadmi de aaega, theek hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Stock madhe aahe, rate donshe chalis ahe sir, haan tar delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "वो, हाँ बोलिए सर, अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "देखो, खत्म नहीं हुआ, अभी भी है, रेट चार सौ नब्बे है", "answer": {"in_stock": true, "price_inr": 490, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "देखिए पहले सवा दो सौ का था अभी है कि नहीं देखना पडेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "हां जी, स्टॉक में पडा है. साढे तेरह सौ का पडेगा. अं आपको shop से पिकअप करना होगा.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "हाँ, माल पड़ा है. सोलह सौ का है, इससे कम नहीं होगा. अं. डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ, है. हाँ तो 320 रुपये लगेंगे. अभी 23 पीस ही बचे हैं. अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 320, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "kya hai na haan ji, mil jayega, 30 rupaye ka hai, paint roller bhi hai, wo 23 ka", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "rate tis ahe sir, tape padliye, boy ghari deliver karel.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ बारह पीस पड़े हैं एक सौ सत्तर फ़ाइनल", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "wire sampli nahi, aahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "हम्म वो तो कल ही ख़त्म हो गया, Retina कवर है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina कवर"}}
{"product": "Minda Black Current Bc-64 horn", "said": "Arre haa ha, hai, aapko paune gyarah sau men milega, MRP ek hajar chhah sau hai", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "तर संपले सगळे, AutoClean wiper blade आहे आमच्याकडे, घरपोच देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "AutoClean wiper blade"}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "माल संपला, हॅवेल्स 1.0 HP वॉटर पंप वाला देऊ शकतो, साडेपाच हजार पडेल, पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "हॅवेल्स 1.0 HP वॉटर पंप"}}
{"product": "DHRUV-PRO multimeter", "said": "हाँ है, कोई प्रॉब्लम नहीं, एक सौ पचपन लगा देंगे, वैसे 200 का है", "answer": {"in_stock": true, "price_inr": 155, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello? aawaj yetoy ka? haan bola, porga gharee sodun deil", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "स्टॉकमध्ये आहे। बघा। म्हणजे म्हणजे सव्वा तीनशेला देतो", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Haan toh nahi hai... ruko ruko, hai, ek pada hai, 40 rupaye hai", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है, सौ फाइनल प्राइस है, भेज देता, पर आज बॉय छुट्टी पे है, आके ले जाओ.", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "iBELL drill machine", "said": "rate sava atthais sau ke aas paas hoga stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "अभी नहीं है। देखो देखो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "hai na kitne chahiye 275 ka padega delivery ka 20 extra", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "matlab han mil jayega bhai pauna nau sau rupaye", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "dekho cover hai, mil jayega do so pachas rupaye lagenge ladka de aayega, pachaas rupaye auto ka", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "यस, अवेलेबल है, प्राइस 20 है, देखो लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हाँ जी सर, बताइए। हाँ है। वो क्या है प्रिंटेड एमआरपी सात सौ, अपना प्राइस चार सौ पचपन", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan ji wire available hai 2000 ka milega 9 baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 2000, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "हाँ, माल पड़ा है, देखिए आपके लिए एक सौ पचासी, डिलीवरी का 30 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ है, कोई प्रॉब्लम नहीं. पौने अठारह सौ रुपये लगेंगे. लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 1775, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "18च पीस उरलेत, रेट मालकांना विचारून सांगतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "यस, अवेलेबल है, ११५ का है, आपके लिए ९०.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "cover khatam ho gaya, Cross Fingers bike cover wala de sakta hun, wo chaar sau ka padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers bike cover"}}
{"product": "Speedwav handle grip for Apache", "said": "rate charashe aahe, ho aahe, delivery deto.", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ती संपली हो, त्याच्याऐवजी बिर्ला wall पुट्टी 40 kg देऊ शकतो, पाठवायची सोय नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "बिर्ला wall पुट्टी 40 kg"}}
{"product": "Duravolt LED batten", "said": "अच्छा हाँ, माल पड़ा है. १८० लगेगा, २३० वाला रेट पुराना है, वो नहीं. अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "आत्ता नाहीये, परवापर्यंत येईल. पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "हॅलो, कोण बोलतंय? शेठ बाहेर गेलेत, ते आले की सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "मशीनचा स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "हाँ, है. रेट बदल गया है, अब छह सौ बीस है, पहले पाँच सौ पैंसठ था. अच्छा डिलीवरी फ़्री है.", "answer": {"in_stock": true, "price_inr": 620, "delivery": true, "alternative": null}}
{"product": "Type R G124 horn", "said": "यस सर, बोलिए. त्योहार का सीज़न है ना. आवाज़ कट रही है, ज़ोर से बोलिए. नहीं है जी, ख़त्म. Favourite Bikerz horn है, पाँच सौ बीस का. वो क्या है एक ग्राहक खड़ा है, एक मिनट. अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz horn"}}
{"product": "DHRUV-PRO multimeter", "said": "एक सौ अस्सी का है, इससे कम नहीं होगा, मिल जाएगा, है अपने पास, बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ हाँ है, रेट दो सौ चालीस है सर.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "स्टॉकमध्ये आहे, एमआरपी 395 आहे, पण 335 ला लावतो", "answer": {"in_stock": true, "price_inr": 335, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Abhi busy hun, aadhe ghante baad phone karna bhai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Aaj nahi hai, kal aaega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "Hai na, kitne chahiye? sawa gyaarah sau hai, 1280 nahi.", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "यस sir बोलिए साढ़े तीन सौ रुपीज है हां जी stock में पडा है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "बेसिकली हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "ऑयल नही है, खत्म हो गया, Castrol 10W engine oil है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Castrol 10W engine oil"}}
{"product": "Vista 500 ml chain lube", "said": "हां बोला। ते कालच संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा पड़ा है आपके लिए ७० मतलब लड़का नहीं है भेजने को आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Makita drill machine", "said": "haan bola ek giraik aahe, ek minute aatta gadbadit aahe, ardhya tasane phone kara haan haan, aiktoy theek ahe?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ जी, मिल जाएगा, 330 लगा देंगे, वैसे 425 का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "stock me hai sir, GST mila ke 220", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "हैलो, कौन? है ना, कितने चाहिए? सवा सैंतीस सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 3725, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "जी बोलिए. देखो हाँ हाँ, है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "ख़त्म नहीं हुआ, अभी भी है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "हाँ तो मिल जाएगा है अपने पास एक हज़ार एक सौ का है आपके लिए आठ सौ पचासी", "answer": {"in_stock": true, "price_inr": 885, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "pavanesatrashe hotil mhanje light ahe, milel pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": 1675, "delivery": false, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "ऑयल ख़त्म हो गया, Shell 10 watt engine oil है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Shell 10 watt engine oil"}}
{"product": "Carbanao car mat for Swift", "said": "देखो अभी स्टॉक नहीं है, कल तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ बोलिए, अभी नहीं बता सकता, बाद में फ़ोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हां आहे ना, अडीचशे आहे, 205 नाही", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप ख़त्म नहीं हुई, है, एक पीस 15, 25 लोगे तो 13 के हिसाब से", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "पंखा संपला उषा 400 एमएम फॅन वाला आहे साडे सोळाशेला पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "उषा 400 एमएम फॅन"}}
{"product": "Grand Pitstop GRL chain lube", "said": "हां, बरं हो, आहे शिल्लक, 195 होतील होतील, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 195, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, रखा है, बेस्ट price सवा तीन सौ, लड़का दे आएगा, पचास रुपये लगेंगे, अच्छा", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "हेलो हाँ बोलो नहीं है भाई खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी पडलीये एमआरपी 1830 आहे पण 1450 ला लावतो तर पाठवायला माणूस नाही इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 1450, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "पाँच सौ दस रुपये का है, है भाई, हम्म, अभी डिलीवरी बंद है.", "answer": {"in_stock": true, "price_inr": 510, "delivery": false, "alternative": null}}
{"product": "Benz helmet", "said": "han sir, available hai, GST mila ke 745", "answer": {"in_stock": true, "price_inr": 745, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "मशीन आहे, मिळेल. तुम्हाला 1520 ला देतो. घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 1520, "delivery": true, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "haan sir, bataiye, arre abhi stock finish ho gaya, uski jagah Accessory Mart de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessory Mart"}}
{"product": "8 mm drill bit", "said": "हां तर आहे की, मिळेल. एक पीस साठ, दहा घेतले तर एक्कावन्न प्रमाणे लावतो.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "है है... अरे नही, वो दूसरा वाला है, ये वाला नहीं है। Tradon bike cover पड़ा है, वो ले लो। delivery नहीं होती, pickup करना पडेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Tradon bike cover"}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "aahe, solach urlet, ajun kahi?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हम्म बल्ब का पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "hai ji, mil jaega, arre aapko number kisne diya? acha thik hai, Diwali ka season hai na, sab yahi maang rahe hain, ladka chai lene gaya hai, company ne is mahine phir rate badhaya, 2120 laga denge, waise 2700 ka hai.", "answer": {"in_stock": true, "price_inr": 2120, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Wo item hum keep nahin karte Bosch Dio air filter hai, ek sau sath ka delivery nahi hoti, pickup karna padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Bosch Dio air filter"}}
{"product": "Havells 1500W immersion rod", "said": "हाँ सर वो मॉडल अवेलेबल है छह सौ सत्तर लगेगा", "answer": {"in_stock": true, "price_inr": 670, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Aapke liye saadhe atarah sau, arre available hai, koi problem nahi, same day delivery kar denge", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "अच्छा शाम तक माल आ जाएगा, अभी नहीं है, एक हज़ार आठ सौ पचास का पड़ेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "stock men hai sir ab atthais sau pachas hai, last month do hajar panch sau tha", "answer": {"in_stock": true, "price_inr": 2850, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Haan, hai apne paas, aath sau das hoga, ghar pe bhej dunga, assi lagega bhejne ka", "answer": {"in_stock": true, "price_inr": 810, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "हो, ठेवतो आम्ही, आहे। तुम्हाला चारशे पंच्याण्णवला देतो। घरपोच देतो", "answer": {"in_stock": true, "price_inr": 495, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है, सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "नमस्ते, बोलिए. सवा पाँच सौ रुपये. हाँ जी, कवर रखा है.", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हेलो? हाँ, बोलो भाई, ओनर अभी नहीं हैं, बाद में कॉल कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "hai ji, mil jayega, acha bhijava dunga.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब पड़ा है डिलीवरी की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मतलब आज नहीं है, कल आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Acha ha bolo... nahi, wo nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ सर वो मॉडल अवेलेबल है आके ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "Hello, haan bolo, saadhe chaar sau last rate hai, ha ji, bulb available hai", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "यस सर, बोलिए, रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "bagha plate sampli.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "dekho haan, ready stock hai, season time hai na, demand zyada hai, last month teen sau pachasi tha, ab sava chaar sau hai, network issue hai, aawaz break ho rahi hai", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "tyre ahe, milel, teen hazar shambhar MRP ahe, pan pavane ekonisshe la deu", "answer": {"in_stock": true, "price_inr": 1875, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Haan, stock men hai, ek ka sau, barah ka dabba ek hajaar pachas ka.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "ट्यूब नहीं है, खत्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "है भाई, प्राइस चौदह सौ है, सेम डे डिलीवरी कर देंगे.", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "मिल जाएगा, है अपने पास. पंद्रह का पड़ेगा", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "आहे की, मिळेल, 2920 रुपये होतील.", "answer": {"in_stock": true, "price_inr": 2920, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "mhanje haan bola nahi te sadhya nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "हैलो? हाँ, आवाज़ आ रही है, अभी नहीं है, ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "हां तर आत्ता नाहीये, परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ है सर.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "हेलो स्टॉक में है हज़ार लगेगा देखो घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Kya hai na ha, wahi vala hai, last time 36 tha, abhi 40 hai.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "नमस्ते, बोलिए, जी, उपलब्ध है, दो हज़ार लगेगा, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 2000, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "हो, ठेवतो आम्ही, आहे. दोनशे पाच, GST वेगळा.", "answer": {"in_stock": true, "price_inr": 205, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "हाँ जी, मिल जाएगा, पाँच सौ पचपन का है, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "sampala hota... nahi, ek aahe ajun, 190 rupees", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, बोला दादा, सव्वापंधराशे फिक्स, अं आमच्याकडे आहे, माणूस पाठवून देतो घरी", "answer": {"in_stock": true, "price_inr": 1525, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हां जी, आज नहीं है, कल आएगा, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्ते बोलिए घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "हो बोला दादा कव्हर पडलंय 650 रुपये होतील पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "है भाई. पर पीस पचपन, बीस लो तो पचास लगेगा. डिलीवरी हो जाएगी सर.", "answer": {"in_stock": true, "price_inr": 55, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "अरे क्या बताऊँ, सुबह से बहुत भीड़ है. जी, उपलब्ध है. लड़का चाय लेने गया है. आवाज़ कट रही है, ज़ोर से बोलिए. भेज देंगे, 30 रुपये डिलीवरी चार्ज लगेगा. एक ग्राहक खड़ा है, एक मिनट. माल का पौने चार सौ, डिलीवरी अलग", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "हाँ, रेडी स्टॉक है. लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब ख़त्म नहीं हुई, है. अच्छा साढे छह सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "हाँ, पच्चीस पीस पड़े हैं. 466 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 466, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हां बोलिए, जी जी, उपलब्ध है, पौने बारह सौ का है, इससे कम नहीं होगा, लडका छोड देगा घर पर", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "हां जी मिल जाएगा डिलीवरी नही है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "अच्छा हाँ रखते हैं, पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "आमच्याकडे नसतं, ऑर्डर करून मागवतो, डिलिव्हरी नाही, शॉपवरून पिकअप करा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "वो मॉडल अवेलेबल नहीं है मतलब RNG tyre वाला है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "RNG tyre"}}
{"product": "50 mm padlock", "said": "तीन सौ साठ रुपीज़है, ताला ख़त्म नहीं हुआ, है.", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "nal ahe, milel, bara savvasahashe la padel, manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 625, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हेलो? हाँ, बोलो भाई, हाँ सर, अवेलेबल है, इक्कीस सौ का है, डिलीवरी नहीं होती, पिकअप करना पड़ेगा, देखो बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "हो सर बोला तुम्हाला एक हजारला देतो बघा हो सर अवेलेबल आहे", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है. बॉय भेज दूंगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Namaste ji, boliye, wo item nahi rakhte hum", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ, रखा है, रेट मालिक बताएँगे, आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "देखिए हाँ हाँ है शाम तक पहुँचा देंगे ठीक है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "अभी दो पीस ही बचे हैं। देखिए आपके लिए सवा चार सौ", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji sir, tell me, matlab sorry sir, abhi out of stock hai, Amaron available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron"}}
{"product": "Metro Conti 140/70-17 tyre", "said": "30 पीस हैं स्टॉक में। रेट तीन हज़ार चार सौ है। देखो भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 3400, "delivery": false, "alternative": null}}
{"product": "JCB plier", "said": "JCB वाला ख़त्म है, Agrico plier है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Agrico plier"}}
{"product": "Digismart pedestal fan", "said": "हॅलो? आवाज येतोय का? हां, बोला, तेराशे पन्नासला येतो, पण सध्या संपलाय, हां आत्ता नाहीये, संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हाँ, २१ piece पड़े हैं. देखो देखो. आपके लिए ३४०.", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हैलो, कौन? ढाई सौ फ़ाइनल, है ना, कितने चाहिए? डिलीवरी फ़्री है, देखिए बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "एक मिनट... हाँ बोलिए, अभी नहीं है, परसों तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "हॅलो स्टॉकमध्ये आहे रेट बदललाय आता एक हजार आधी 825 होता", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "हेलमेट नहीं है खत्म हो गया Sonik 7 helmet अवेलेबल है उसका प्राइस आठ सौ पंद्रह", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik 7 helmet"}}
{"product": "10W-30 engine oil 1 L", "said": "haan tar tinashe daha fix, aahe ki, milel, theek aahe?", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Tape sampli nahi, ahe, sutta 30 la, 50 nagancha box ghetla tar 1330", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ जी, मिल जाएगा. सात सौ चालीस फ़ाइनल. भेजने वाला कोई नहीं है, आ के ले जाओ. दुकान 7 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 740, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Arre abhi stock men nahin hai. Supreme wala hai, pachees ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Supreme"}}
{"product": "Crispy LED torch", "said": "hello? aawaj yetoy ka? haan bola, ho, thevto aamhi, aahe, 380 padel aata, magchya mahinyat 310 la hota", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "जी बोलिए, हाँ है, अं रेट ढाई सौ है सर.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "ek min... haan bolo. boy ghar pe de jayega.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "हैलो, कौन? हेलमेट खत्म हो गया। स्टड्स हेलमेट वाला है, चलेगा? पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "स्टड्स हेलमेट"}}
{"product": "Duravolt LED batten", "said": "Haan aahe, bagha pavanedonshe fix, sandhyakali pathavun deto.", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Havells LED torch", "said": "मतलब नहीं है, मंगवा देंगे, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "bagha te model available nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "एक मिनट हाँ बोलिए मतलब लडका चाय लेने गया है एक ग्राहक खडा है एक मिनट सेठ जी बाहर गए हैं वही बताएंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Owner abhi available nahi hain, baad mein call kijiye bhaiya.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "अं ते आमच्याकडे नसतं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, maal pada hai, khud aake le jaiye, thik hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नाही ना संपलंय भाऊ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "Haan, maal pada hai, 220 ka... nahi nahi, galat bola, 195 ka hai", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Easy4buy horn", "said": "हां है, कोई प्रॉब्लम नहीं, सवा छह सौ लगेगा, मतलब बताइए क्या है", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हम्म हाँ जी, मिल जाएगा. दस लोगे तो एक सौ दस के हिसाब से, एक का सवा सौ. कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ बोलिए, मैं नया हूँ, मुझे नहीं पता", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "हाँ बोलिए सर। सीज़न टाइम है ना, डिमांड ज़्यादा है। स्टॉक में सिर्फ बीस पीस है। आपके लिए चार हज़ार। सर आज बहुत बिज़ी डे है। नेटवर्क इशू है, आवाज़ ब्रेक हो रही है", "answer": {"in_stock": true, "price_inr": 4000, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "Yes sir, boliye wo modal available nahin hai uski jagah Vega helmet le sakte ho ghar tak deliver kar denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Vega helmet"}}
{"product": "tank float valve", "said": "मतलब हां जी स्टॉक में पडा है अब प्राइस पौने चार सौ है पहले चार सौ पैंसठ था पता भेज दो लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "stock में है sir, rate नौ हज़ार दो सौ है, एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 9200, "delivery": true, "alternative": null}}
{"product": "Dow silicone sealant", "said": "हैलो, कौन? आज सुबह से लाइट नहीं है, एक ग्राहक खड़ा है, एक मिनट, अभी नहीं बता सकता, बाद में फ़ोन करो, कंपनी ने फिर रेट बढ़ा दिया", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "हाँ बोलिए, मतलब थोड़ी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "नहीं है भाई, खत्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "kyun nahi milegaa, hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "रेट 1100 है पर अभी स्टॉक नहीं है अभी नहीं है ऑर्डर दिया है दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "आपके लिए 420। जी, उपलब्ध है। देखो लड़का भेज देता हूँ।", "answer": {"in_stock": true, "price_inr": 420, "delivery": true, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "बेसिकली हाँ sir, वो मॉडल अवेलेबल है, आपके लिए ढाई सौ, एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ जी, मिल जाएगा, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "हाँ हाँ, है, वो आपसे 315 लेंगे, लिखा 340 है, घर पर भेज देंगे.", "answer": {"in_stock": true, "price_inr": 315, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ek minute ha... haan bola nahiye tyachya aivaji Schneider 32A MCB wala deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Schneider 32A MCB"}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "पहले तेरह सौ पचास का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Easy4buy horn", "said": "हाँ सर अवेलेबल है ६०० लगा देंगे वैसे MRP ९०० लिखा है मतलब खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "हाँ है 190 रुकिए लिस्ट देखता हूँ 200", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "हाँ जी सर ATC नहीं है Retina बाइक कवर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina बाइक कवर"}}
{"product": "bike headlight bulb", "said": "स्टॉकमध्ये आहे, म्हणजे, दोनशे दहाला लावतो, MRP दोनशे पस्तीस आहे.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ जी, स्टॉक में पड़ा है, चार सौ सत्तर फाइनल प्राइस है, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 470, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ji sir, tell me, matlab han, ready stock hai, sava do sau final price hai.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Bagha plate aahe ki nahi mahit nahi, godown madhe baghto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "stock me hai sir. pauna do so ka hai. delivery nahi hoti, pickup karna padega", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Haan sir, boliye. matlab ji haan, rakha hai. pichle mahine 91 tha, ab 100.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "बघा हो, हेल्मेट आहे। 490 रुपये होतील। डिलिव्हरी फ्री आहे। सात वाजेपर्यंत वाजेपर्यंत मी इथेच आहे", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "बरं Activa संपलं, लोकल क्लच प्लेट आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल क्लच प्लेट"}}
{"product": "Stanley angle grinder", "said": "स्टॉक में है सर. पौने छब्बीस सौ प्लस GST. खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 2575, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हां जी, ऑयल रखा है. तीन सौ रुपये का है. आपको आकर लेना पडेगा", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "हो, आहे। 320... नाही नाही, पावणे चारशे।", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "म्हणजे हो हो, समजलं, पण ते नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "वो तो कल ही ख़त्म हो गया, Autofurnish रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autofurnish"}}
{"product": "Halonix 10W LED bulb", "said": "हां तर बल्ब संपला नाही, आहे, 370 MRP आहे, पण 315 ला देऊ, डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 315, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "kulup sampala nahi, ahe, bara tinashe tees, GST vegla.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "हाँ, माल पड़ा है, डिलीवरी नहीं होती, पिकअप करना पड़ेगा, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "हो आहे आधी एक हजार होता आता एक हजार एकशे पन्नास झाला", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "हॅलो, कोण बोलतंय? लाईट संपली नाही, आहे, म्हणजे पावणेचारशेला देतो, येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "Ha, mal pada hai, ek peice 180, 10 wala box loge toh 1600, delivery ho jayegi.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "नॉट अवेलेबल सर, उसकी जगह Clouds wiper blade ले सकते हो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Clouds wiper blade"}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ, माल पड़ा है. पूरा पैकेट छह सौ नब्बे, अलग से एक पीस तीन सौ अस्सी का", "answer": {"in_stock": true, "price_inr": 690, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "मिल जाएगा, है अपने पास, दस का फुल बॉक्स दो हज़ार दो सौ पचास, सिंगल पीस पौने तीन सौ, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "Ha, maal pada hai. delivery nahin hoti aisa nahi hai, bhej dunga", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, माल पड़ा है, सिंगल piece सौ, 25 का फुल box दो हज़ार दो सौ पचास, वो.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "haan tar kami nahi, bharpur ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Nahin hai, haa han, sun raha hun, local seal kit wala hai, 130 ka, ladka chai lene gaya hai, market me mal ki kami chal rahi hai, matlab are kya bataun, subah se bahut rush hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Speedwav LED headlight", "said": "namaskar, bola. ho, light milel. kay aahe na 1540 fix. yeun gheun ja", "answer": {"in_stock": true, "price_inr": 1540, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "बेसिकली अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "नल ख़त्म हो गया, SARK tap चलेगा? उसका ढाई सौ है, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "SARK tap"}}
{"product": "Cheston 500W drill machine", "said": "हां बोला साहेब, मशीन संपली नाही, आहे, सव्वा दोन हजार होतील, घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 2250, "delivery": true, "alternative": null}}
{"product": "BMB welding machine", "said": "बोला, काय पाहिजे? हो, मिळेल, सव्वाएक्केचाळीसशेला देतो, सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 4125, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "ji boliye hai na kitne chahiye arre 330 last rate hai", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "bulb hai mil jaega ek sau pachis laga denge waise mrp 155 likha hai", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "आहे, अजून चौदा पीस पडलेत, पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "नमस्कार बोला हो दादा आहे एमआरपी 2080 आहे पण सव्वाएकोणीसशेला लावतो येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 1925, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "देखो, मिल जाएगा, है अपने पास, पच्चीस लोगे तो छप्पन के हिसाब से, एक का सत्तर.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "Tyre sampla bhau", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "अं है भाई, पैकेट ६८५ का है, खुला एक ४२५ का.", "answer": {"in_stock": true, "price_inr": 685, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ जी, मिल जाएगा. वो 10 लोगे तो एक सौ दस के हिसाब से, एक का सवा सौ", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Orient Apex-FX 1200 mm ceiling fan", "said": "ख़त्म नहीं हुआ, अभी भी है. आपके लिए सवा ग्यारह सौ. अं अभी डिलीवरी बंद है.", "answer": {"in_stock": true, "price_inr": 1125, "delivery": false, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "namaste ji, boliye, haan, mal pada hai, chaar sau laga denge, waise MRP 600 likha hai", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "ते मिळणार नाही इथे हां हां तर Bosch 10 एमएम ड्रिल बिट चालणार असेल तर सांगा पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Bosch 10 एमएम ड्रिल बिट"}}
{"product": "Ravirandal silicone sealant", "said": "हाँ बोलिए. मिल जाएगा, है अपने पास. एक ग्राहक खड़ा है, एक मिनट. पिछले महीने 185 था, अभी 200. भेजने का इंतज़ाम नहीं है. आज सुबह से लाइट नहीं है. ठीक है?", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji boliye, sorry sir, abhi out of stock hai, arre uski jagah Bosch de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Duravolt LED batten", "said": "नमस्ते बोलिए ट्यूब खत्म नही हुई है पौने दो सौ फाइनल पता भेज दो लडका दे आएगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "वो, हम्म हाँ हां, समझ गया, लेकिन वो अभी नही है, SARK tap है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "SARK tap"}}
{"product": "Havells 1500W immersion rod", "said": "देखिए सेठ जी बाहर गए हैं, वही बताएँगे।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Mana kisne kiya? hai na", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "han bhaiya, bataiye, humare paas nahi hota, mangwana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "जी बोलिए 4 पीस हैं स्टॉक में अं ठीक है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "घर तक पहुँचा देंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ruko, dekh ke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "stock khatam ho gaya hai, Exide wala hai, same quality, sorry, delivery service nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Exide"}}
{"product": "8 mm drill bit", "said": "Malak nahit, nantar phone kara, kay sangu, sakal pasun gardi ahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "जी, उपलब्ध है, रेट बदल गया है, अब 200 है, पहले 180 था", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "Han, wahi wala hai, umm saadhe bais sau final, apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": 2250, "delivery": true, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "एक गिऱ्हाईक आहे, एक मिनिट, आवाज तुटतोय, मोठ्याने बोला, हां हां, ऐकतोय, हो, पंखा आहे, पावणे तीन हजार रुपये, कमी नाही होणार, पोरगा पाठवतो, पन्नास रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "Mastech multimeter", "said": "यस sir बोलिए वो मॉडल available नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "हाँ सर, अवेलेबल है, एक कस्टमर खड़ा है, होल्ड करो, डिलीवरी का 60 एक्स्ट्रा, सीज़न टाइम है ना, डिमांड ज़्यादा है, सर आज बहुत बिज़ी डे है, 6499 का है, प्लस डिलीवरी.", "answer": {"in_stock": true, "price_inr": 6499, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bhaiya, bataiye, nahi hai ji, khatam, Taparia 6 mm drill bit hai, wo le lo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Taparia 6 mm drill bit"}}
{"product": "teflon tape", "said": "Rate 25 aahe, pan aatta stock nahi, aatta nahiye, parva paryant yeil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "hello, haan boliye, acha haan, maal pada hai, rate seth ji batayenge, khud aake le jaiye", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "अच्छा बल्ब पड़ा है, एक का 265, 10 का डिब्बा 2200 का, दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 265, "delivery": false, "alternative": null}}
{"product": "Anchor LED batten", "said": "Haan tar stock madhe ahe, delivery deto, vees charge lagel, item cha price ekshe panchyaainshi, delivery vegli.", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "सवा तीन सौ final प्राइस है, हाँ, रेडी stock है.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "Stock me hai sir, 4720 hai, sorry sorry, 5600.", "answer": {"in_stock": true, "price_inr": 5600, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "वो हम नहीं रखते जी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "कम नहीं है, बहुत पड़ा है, 2150 लगेगा, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हाँ जी मिल जाएगा दो सौ पचास रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "माल खत्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "अं रॉन्ग नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "जी बोलिए. कल फ़ोन करना. वो अरे क्या बताऊँ, सुबह से बहुत भीड़ है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "स्टॉक में है सर, चौदह सौ पचास है, एक हज़ार चार सौ नहीं", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ रेडी stock है आपके लिए पौने नौ सौ", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "हाँ, रखा है। एक हज़ार पचास लगा देंगे, वैसे एक हज़ार दो सौ का है", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "हां जी मशीन रखी है अपना लड़का छोड देगा घर पे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "तर शेठ बाहेर गेलेत, ते आले की सांगतो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "एक कस्टमर खड़ा है, होल्ड करो, सर आज बहुत बिज़ी डे है, है भाई, मिल जाएगा, सीज़न टाइम है ना, डिमांड ज़्यादा है, चार सौ तीस लगेगा, डिलीवरी बंद है अभी, दुकान से ले लो, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, मतलब ओके?", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Adima LED torch", "said": "स्टॉकमध्ये नाहीये, Crasts टॉर्च आहे, चालेल का? पिकअप करावा लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Crasts टॉर्च"}}
{"product": "AutoPowerz LED headlight", "said": "बोला बोला काय पाहिजे ते मिळणार नाही इथे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "ha bolo, galat number hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "तर हो, मिळेल। 195... नाही नाही, पावणे दोनशे।", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "लड़का छोड देगा घर पर, देखो.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Battery sampli nahi, aahe. rate pavne barashe ahe", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "Moonstruck 2000W immersion rod", "said": "अं बरं पाठवायची सोय नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "han sir boliye ji available hai paanch sau chalis plus 18 percent gst ladka nahin hai bhejne ko aake le jao", "answer": {"in_stock": true, "price_inr": 540, "delivery": false, "alternative": null}}
{"product": "iBELL drill machine", "said": "हैलो कौन ख़त्म हो गया Dewalt drill machine है वो ले लीजिए हाँ तो डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Dewalt drill machine"}}
{"product": "Kingsway car cover", "said": "हैलो, कौन? थोड़ी देर बाद कॉल करो, हाँ तो अरे क्या बताऊँ, सुबह से बहुत भीड़ है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ माल पड़ा है रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "R-tek multimeter", "said": "स्टॉक में है, ४०० रुपये लगेंगे, Aptech multimeter चाहिए तो वो भी है, ४४५ वाला, डिलीवरी फ़्री है.", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Michelin tyre", "said": "एक मिनिट हां... बोला। हां हां, ऐकतोय। अं काय सांगू, सकाळपासून गर्दी आहे। आहे की, मिळेल। दिवाळीची गर्दी आहे ना। १३०० फिक्स।", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "abhi nahin, order pe mangwa denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हेलो, हाँ बोलो, है ना, कितने चाहिए? तो, अब आठ सौ चालीस है, लास्ट मंथ 725 था.", "answer": {"in_stock": true, "price_inr": 840, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "milel na, aahe, rate sadeterashe ahe, boy ghari deliver karel", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "ख़त्म हो गया था नहीं एक बचा है पौने सात सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "अभी नहीं है जी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "Mhanje daha nag shillak aahet, rate char hazar pannas ahe", "answer": {"in_stock": true, "price_inr": 4050, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "हां जी बताइए पंखा खत्म हो गया उसकी जगह Bajaj fan वाला दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj fan"}}
{"product": "4 socket extension board", "said": "माल खत्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "हेलो हाँ बोलो सीज़न टाइम है ना डिमांड ज़्यादा है सोलह piece हैं stock में दो हज़ार पचास फाइनल प्राइस है नेटवर्क इशू है आवाज़ ब्रेक हो रही है courier कर देंगे", "answer": {"in_stock": true, "price_inr": 2050, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "मशीन पडलीये MRP 3250 आहे पण पावणे एकोणतीसशेला लावतो घरपोच करून देतो", "answer": {"in_stock": true, "price_inr": 2875, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ रखा है तीन सौ दस में दे देंगे", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "yes available hai dekho home delivery available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "जी, उपलब्ध है। सात सौ पैंतीस का पड़ेगा", "answer": {"in_stock": true, "price_inr": 735, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "वो क्या है हाँ हाँ समझ गया लेकिन वो अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर नाहीये, संपली", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "हेलो? हाँ, बोलो भाई, स्टॉक में है सर, आपके लिए चार सौ बीस", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "मतलब हाँ जी, मिल जाएगा। ६९५... नहीं नहीं, सात सौ दस है।", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "हैलो? हाँ, आवाज़ आ रही है। बेसिकली स्टॉक नहीं है अभी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं नाही, आहे, हां तर आधी 330 होता, आता पावणे चारशे झाला", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "आहे सर, रेडी स्टॉक आहे साहेब.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हेलो हाँ बोलो भाई हाँ जी stock में पड़ा है साढ़े बारह सौ फ़ाइनल आके ले जाओ", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "acha haan ji, wire available hai, 1650 rupees lagenge, Polycab 2.5 sq mm wire chahiye to wo bhi hai, 2080 vala, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bola dada tinashe vees la deto oil ahe milel", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नाही ना, संपलंय, बघा Amaron 12V 5Ah battery आहे, त्याचे तेराशे रुपये, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Amaron 12V 5Ah battery"}}
{"product": "bike headlight bulb", "said": "बेस्ट प्राइस दो सौ दस, यस, अवेलेबल है", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "यस, available है, सीज़न टाइम है ना, डिमांड ज़्यादा है, रेट पौने चार सौ है sir, एक कस्टमर खड़ा है, होल्ड करो, बताओ क्या करना है.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब नहीं है, खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "देखो नहीं नहीं, है ना. पौने चौदह सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 1375, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "यस सर, बोलिए. हाँ, है. दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "येऊन घेऊन जा.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crispy LED torch", "said": "Available hai, Havells LED torch ka rate teen sau sattar hai, ye wala savaa chaar sau", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "वो क्या है आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "नहीं है जी, ख़त्म, तपारिया 10 एमएम ड्रिल बिट है, 90 का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "तपारिया 10 एमएम ड्रिल बिट"}}
{"product": "Retina bike cover", "said": "cover sampala nahi, ahe, kay aahe na donshe sath padel ata, magchya mahinyat donshe panchvis la hota, pathavaychi soy nahi", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "नमस्कार, बोला. बघा. रेट तीस. टेप पडलीये. सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "hai bhai delivery nahi bhai wo nahi hota", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "नहीं नहीं है ना रेट तेरह सौ है", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, मिळेल, 80 रुपये, कमी नाही होणार, येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ हाँ, है, साढ़े तीन सौ रुपये, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, माल पड़ा है। डिलीवरी का 50 एक्स्ट्रा। देखो सामान 150 का है, डिलीवरी अलग से", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "stock me hai sir, our price 465, MRP 760 hai.", "answer": {"in_stock": true, "price_inr": 465, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "8 piece available hai abhi rate seth ji batayenge khud aake le jaiye", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "ती संपली हो, AmiciTools वॉटर पंप घ्या, ते चालेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools वॉटर पंप"}}
{"product": "V-Guard Clado exhaust fan", "said": "pankha sampla nahi aahe price pavaneakarashe aahe kay aahe na delivery deto 20 charge lagel", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "जी, उपलब्ध है. घर पर भेज देंगे, बीस अलग से. सामान पचास का है, डिलीवरी अलग से", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "helmet khatam nahin hua, hai, aath so das last rate hai, dukan 8 baje tak khuli hai, pick up karna padega.", "answer": {"in_stock": true, "price_inr": 810, "delivery": false, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "हाँ, माल पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "hello haan awaaz aa rahi hai bolo matlab halonix wala khatam hai wipro 15 watt bulb hai chalega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "wipro 15 watt bulb"}}
{"product": "Allure Auto car mat for Zest", "said": "Allure Auto nahi hai, Packy Poda car mat hai network issue hai, aawaz break ho rahi hai sir aaj bahut busy day hai season time hai na, demand zyada hai bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Packy Poda car mat"}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "हो, आहे शिल्लक. पावणेपाचशेला देतो. घरी पाठवतो, पन्नास वेगळे", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हेलो हाँ बोलो भाई स्टॉक नहीं है अभी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "udya phone kara. company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "hello haan awaaz aa rahi hai bolo hai bilkul hai aur kuch chahiye", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "Maal khatm hai, Castrol 10 watt engine oil de sakta hun, wo 460 ka padega, dekho home delivery nahi hai bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Castrol 10 watt engine oil"}}
{"product": "Leebo LED headlight for Honda", "said": "light hai mil jayegi savaa pandrah sau ka hai isse kam nahi hoga delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 1525, "delivery": true, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "Helo, han bolo, acha sava teen sau rupees, hai bhai, ghar tak pahunchaa denge", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "ho, thevto aamhi, aahe, aapla rate tinashe pannas, MRP teenshe ainshi aahe", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "available hai, koi problem nahi, matlab sawa chah sau final price hai, pick up karna padega", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "तर नाही म्हणालो का? आहे हो। पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ सर, वो मॉडल अवेलेबल है, रेट पाँच सौ पचपन है सर, अं डिलीवरी हो जाएगी सर", "answer": {"in_stock": true, "price_inr": 555, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हाँ जी, मिल जाएगा, साढ़े चार सौ लगेगा, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "बेसिकली हाँ सर, अवेलेबल है, आपके लिए 330, लड़का दे आएगा, 100 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "आहे आहे की, मिळेल. पावणे चारशेला देऊ, छापील किंमत सहाशे दहा आहे.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "अभी नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Meco multimeter", "said": "हाँ जी, stock में पड़ा है, अं बेस्ट प्राइस सवा इक्कीस सौ", "answer": {"in_stock": true, "price_inr": 2125, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "एक ग्राहक खड़ा, एक मिनट. स्टॉक में है. आज सुबह से लाइट नहीं है. 1350 लिखा है, छूट देकर बारह सौ पचास. त्योहार का सीज़न है ना.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "आहे की, मिळेल. एक हजार सातशे... नाही नाही, साडे एकोणीसशे", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप आत्ता नाहीये, संपली, अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "अच्छा हाँ है, वो वो, आके ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "हो कव्हर आहे रेट सातशे आहे डिलिव्हरी नाही असं नाही पाठवतो की", "answer": {"in_stock": true, "price_inr": 700, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Abhi stock me nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "आज नहीं है, कल आएगा। रेट छह सौ तीस है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हां हां बोला, सध्या नाही, मागवावं लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ सर, अवेलेबल है। वो क्या है 70 फाइनल प्राइस है। डिलीवरी नहीं होती, पिकअप करना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Bosch wiper blade", "said": "अं माहीत नाही हो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "हो, मशीन शिल्लक आहे। सव्वाबावीसशेला पडेल। म्हणजे संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": 2225, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बेसिकली हाँ है, कोई प्रॉब्लम नहीं, चौदह सौ पचास लगेगा.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "है भाई आपको 495 में मिलेगा MRP 770 है delivery हो जाएगी सर", "answer": {"in_stock": true, "price_inr": 495, "delivery": true, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "हाँ है, कोई प्रॉब्लम नहीं, पौने पाँच सौ लगा देंगे, वैसे पाँच सौ पंचानबे का है, होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "हॅलो? आवाज येतोय का? हां, बोला. आमच्याकडे आहे. हां पावणेदोनशे होतील", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ho aahe... nahi nahi, to dusara ahe, ha nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "है ना, कितने चाहिए? अच्छा साढे तीन सौ कापडेगा. बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "aaj nahi, udya yeyil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "स्टॉक नहीं है अभी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "namaste ji, boliye, motor rakhi hai, mil jayegi, paach hajar ek sau ka hai, aapke liye chaar hazar chaar sau, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 4400, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "mana kisne kiya? hai na.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "कव्हर पडलंय तुम्हाला 600 ला देतो MRP 900 आहे हां तर संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 600, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, मतलब हैवेल्स 2.25 mfd कैपेसिटर है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 2.25 mfd कैपेसिटर"}}
{"product": "Daybetter 25W LED bulb", "said": "बल्ब संपला नाही आहे म्हणजे १० नगांचा बॉक्स दोन हजार एकशे पन्नासला सुटा एक अडीचशे", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "hello, bola na. ho, bulb aahe. bara savvashe la padel. pathavayla manus nahi, ithe yeun ghya.", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "नमस्ते बोलिए हाँ है पिछले महीने २३ था अभी ३० वो कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हां तर स्टॉकमध्ये आहे छापील 23 डिस्काउंट करून 20 कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "देखो ख़त्म हो गया था नहीं एक बचा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan hai, pone do so last rate hai, arre bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "Price pavaneteenshe ahe ho sir, available aahe delivery deto sir.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "आमच्याकडे आहे, आधी 140 होता, आता दीडशे झाला.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ हाँ, है. बेस्ट price एक सौ पचास. delivery नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "HTC multimeter", "said": "हाँ बोलिए। हैलो? आवाज़ नहीं आ रही, बाद में करना। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। लड़का चाय लेने गया है। और कुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "आहे की, मिळेल, रेट दीडशे आहे", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट संपली नाही, आहे, पावणेआठशे होतील.", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "हो, कव्हर शिल्लक आहे, रेट तीनशे पंधरा आहे, पोरगा घरी आणून देईल, सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 315, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ti sampli ho amaron 12v 5ah battery ahe tyache 1050 rupaye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "amaron 12v 5ah battery"}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "aatta stock nahi, udya yeil, pickup karava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "देखो है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "1380 final price hai haa sir, wo model available hai delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 1380, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हो, आहे शिल्लक, आता वीस आहे, आधी 23 होता", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "4 socket वाला खत्म है एंकर वाला है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एंकर"}}
{"product": "Atomberg Renesa ceiling fan", "said": "ho, pankha milel, haan haan, aiktoy, ata chautisshe pannas aahe, aadhi teen hazar naushe hota, gharapoch milel, aawaj tutatoy, motyane bola.", "answer": {"in_stock": true, "price_inr": 3450, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan toh aapke liye chhah sau, ha ha, hai, counter pe aake lelo", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "है भाई आपके पौने पाँच हज़ार खुद आकर उठाना पडेगा", "answer": {"in_stock": true, "price_inr": 4750, "delivery": false, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "मिल जाएगा, है अपने पास, आपको २७५ में दे देंगे, MRP २९० है.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बोला ना, हो दादा, आहे, पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Studds Urban helmet", "said": "Out of stock nahi hai, available hai, rate 815 hai sir", "answer": {"in_stock": true, "price_inr": 815, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "Haa bolo, woh hum nahi rakhte, Mastech multimeter hai, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Mastech multimeter"}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "हाँभाई बोलो रुकिए देख कर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ रेडी stock है 1700 फ़ाइनल फ़ाइनल", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "देखिए ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "kal Sunday tha toh dukaan band thi, pichle hafte bhi kisi ne yahi poocha tha, ek customer khada hai, ek minute, haan, maal pada hai, aapke liye gyarah sau", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ऐसा नहीं है कि नहीं है, है, 220 रुपये का है.", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "नहीं मिलेगा भाई। अं। Fevicol 500 g है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "Havells 1500W immersion rod", "said": "जी, उपलब्ध है, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "वो आइटम हम नहीं रखते। उसकी जगह local ब्रेक शू ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local ब्रेक शू"}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "स्टॉक में है. रेट उन्नीस सौ है.", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "nahi ahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "Tata नहीं है, JCB plier वाला है। डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "JCB plier"}}
{"product": "Candes 10 L geyser", "said": "नाहीये बहुतेक हो हो सापडलं आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "sorry आउट ऑफ stock आहे Crasts torch देऊ शकतो ३३० पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crasts torch"}}
{"product": "Fevicol SH 1 kg", "said": "Haan, maal pada hai, delivery ka chalis extra, saman ka teen sau das lagega", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Haan, kaun? acha, bolo, aaj nahin hai, kal aayega, aapke liye ek hajar pachas, are kya bataun, subah se bahut rush hai, han haan, sun raha hun, pichle hafte bhi kisi ne yahi poocha tha, delivery ho jayegi, ladka chai lene gaya hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "सव्वासातशेला येतो पण सध्या संपलाय बघा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "अं माल खत्म है उषा 230 एमएम फैन चलेगा उसका तेईस सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "उषा 230 एमएम फैन"}}
{"product": "12V 7Ah battery", "said": "हेलो? हाँ, बोलो भाई, बैटरी पड़ी है, एक हज़ार छह सौ पचास का है, आपके लिए पंद्रह सौ", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हम्म जी, उपलब्ध है, भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "स्टॉक में सिर्फ 17 पीस है। रेट अभी पता नहीं, ओनर से पूछना पड़ेगा। पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "पक्कड पडलीये. संध्याकाळी घरी पोहोचेल. कळवा मग.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Dewalt drill machine", "said": "mahit nahi ho, bara bazarat mal kami ahe sadhya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "kya hai na kaun yahan koi dukan nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हॅलो? आवाज येतोय का? हां, बोला, टायर संपला नाही, आहे, पावणे दोन हजार फिक्स, घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "Haa sir, wo model available hai, ladka ghar pe chhod dega.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "Wipro sampala, Anchor ahe, chaalel? porga ghari aanun deil.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Anchor"}}
{"product": "CarMobCentral car cover for Ritz", "said": "आहे सर, रेडी स्टॉक आहे। सव्वा सातशे फायनल। बघा डिलिव्हरी देतो सर।", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हाँ, माल पड़ा है, साढ़े छह सौ का है, इससे कम नहीं होगा, मतलब", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "stock khatam ho gaya hai harrison 60 mm padlock hai wo lelo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "harrison 60 mm padlock"}}
{"product": "Havells 450 mm pedestal fan", "said": "जी, उपलब्ध है, रेट ४३०० है, शाम तक पहुँचा देंगे, तो.", "answer": {"in_stock": true, "price_inr": 4300, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "namaste ji, boliye, aapke liye sau, available hai, bhejne wala koi nahi hai, aapko aana padega.", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "Kingsway नहीं है, Crokrok मैट है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crokrok मैट"}}
{"product": "Crompton Hill Briz ceiling fan", "said": "हेलो, हाँ बोलो। हाँ जी, पंखा रखा है। पौने इक्कीस सौ रुपीज़ है। एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 2075, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "aahe ki, milel, donshe pannas la lavto, MRP tinashe pandhara aahe.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "Haan, kaun? acha, bolo, hai, atharah piece rakhe hain, acha paanch sau pachanve ka hai, isse kam nahi hoga.", "answer": {"in_stock": true, "price_inr": 595, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ भाई, बोलो, हाँ, है, सौ है, GST अलग से, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "म्हणजे हो आहे नाही नाही तो दुसरा आहे हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब पडलीये। रेट शेठ सांगतील।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "हाँ, सात पीस पड़े हैं. इकतीस सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 3100, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "ho sir, bola, kay sangu, sakal pasun gardi ahe, haan ahe, 270... nahi nahi, 310, bazarat maal kami ahe sadhya.", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "हाँ, है, आपको चार सौ में दे देंगे, MRP छह सौ तीस है.", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "stock me hai sir, bhej deta, par aaj ladka nahi aaya, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Retina bike cover", "said": "हाँ रखा है आपके आपके २५० दुकान ७ बजे तक खुली है शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "स्टॉक में है सर", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "ताला पड़ा है, पौने चार सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "जी बोलिए, मतलब अभी नहीं है, order पे मंगवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "मिल जाएगा है अपने पास कंपनी का रेट दो सौ बीस है मैं पौने दो सौ लूंगा पता भेज दो लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "umm sava sau ka aata hai, lekin abhi khatm hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "available hai. ek so pachas ka ek, chah loge toh ek so chalis ke hisaab se laga dunga.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "यस सर, बोलिए। अं शाम तक माल आ जाएगा, अभी नहीं है। सर आज बहुत बिज़ी डे है। एक कस्टमर खड़ा है, होल्ड होल्ड करो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "सवा पाँच सौ रुपये लगेंगे मिल जाएगा है अपने पास अं भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "Makita drill machine", "said": "हो, मशीन आहे, 2950, GST वेगळा, बघा", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हेलो? हाँ, बोलो भाई. देखो हाँ है, कोई प्रॉब्लम नहीं. 230 है, GST अलग से. डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "स्टॉक में है। 200 है, 185 नहीं।", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ सर, वो मॉडल अवेलेबल है, दो सौ अस्सी रुपीज़ है, भेजने का इंतज़ाम नहीं है, मतलब शॉप 7 बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हाँ है। बेसिकली छह का फुल बॉक्स 51, सिंगल पीस दस। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "Porga chaha aanayla gelay, Prithvi sampla, Havells 1.0 HP water pump aahe, chalel? aawaj tutatoy, motyane bola, Diwali chi gardi aahe na, pickup karava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 1.0 HP water pump"}}
{"product": "water pump seal kit", "said": "है भाई। रेट मालिक बताएंगे बताएँगे। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हाँ है, कोई प्रॉब्लम नहीं, तीन सौ चालीस रुपीज़ है, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "हैलो? हाँ, आवाज़ आ रही है। अच्छा पंखा नहीं है, खत्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ तो हाँ हाँ, है, एक सौ पचास का पड़ेगा, भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, बताइए. हाँ हाँ, है. सवा चार सौ लगेगा.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "हो ठेवतो आम्ही आहे एक पीस 270, 10 घेतले तर 225 प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां हां बोला प्लेट संपली नाही आहे अं 820 रुपयांना पडेल संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 820, "delivery": true, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ जी स्टॉक में पड़ा है एक पीस दो सौ बीस, बीस लोगे तो एक सौ अस्सी के हिसाब से कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ जी, stock में पड़ा है। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ जी, मिल जाएगा, तीस में दे देंगे, लड़का छोड़ देगा घर पर, और कुछ?", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "साडेचौदाशेला देतो, बॅटरी संपली नाही, आहे, संध्याकाळी घरी पोहोचेल.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "है भाई. दस का डिब्बा एक हज़ार तीन सौ पचास का, एक पीस डेढ़ सौ", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "माहीत नाही गोडाऊन मध्ये बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "नमस्ते, बोलिए, हाँ हाँ हां, है, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "वो मैं नया हूँ, मुझे नहीं पता, आज सुबह से लाइट नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "हो, कव्हर आहे। आता पावणे तीनशे आहे, आधी दोनशे चाळीस होता", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "abhi dukaan band kar raha hun kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "पंखा पडा है 2300 है 1800 नही मतलब बॉय भेज दूंगा", "answer": {"in_stock": true, "price_inr": 2300, "delivery": true, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "वो मॉडल अवेलेबल नहीं है Inditrust soldering आयरन है साढ़े पाँच सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Inditrust soldering आयरन"}}
{"product": "wall putty 20 kg", "said": "हाँ जी, stock में पड़ा है। सवा सात सौ का है, इससे कम नहीं होगा। देखो डिलीवरी हो जाएगी सर।", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "20 kg नहीं है लोकल वाला है शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "लोकल"}}
{"product": "iBELL drill machine", "said": "हो, मशीन शिल्लक आहे, डिलिव्हरी नाही असं नाही, पाठवतो की.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हैलो, कौन? ट्यूब नहीं है, खत्म हो गई, और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "Kya hai na helmet nahi hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हैलो आवाज नहीं आ रही बाद में करना एक ग्राहक खडा है एक मिनट कंपनी ने फिर rate बढा दिया", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "हैलो? हाँ, आवाज़ आ रही है, त्योहार का सीज़न है ना, कंपनी ने फिर रेट बढ़ा दिया, मिल जाएगा, है अपने पास, डेढ़ सौ रुपये, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "nal sampla nahi, aahe. 230 la deu, chhapil kimmat 365 aahe", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "matlab available hai chaar sau pachis mein ho jayega upar mrp 475 likha hai", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "umm haan geyser hai delivery nahi hoti aisa nahi hai bhej dunga", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हाँ, माल पड़ा है, साढ़े पाँच हज़ार का है, इससे कम नहीं होगा, घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "नहीं है। ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हो, बल्ब शिल्लक आहे। सुटा एक 96 ला, पूर्ण पॅकेट 399। अं", "answer": {"in_stock": true, "price_inr": 399, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan tar 8 nag shillak ahet, 510 la deto.", "answer": {"in_stock": true, "price_inr": 510, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "हाँ भाई, बोलो, वो हम नहीं रखते, अच्छा उसकी जगह Packy Poda मैट दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Packy Poda मैट"}}
{"product": "Vormir 200A welding machine", "said": "हाँ, माल पड़ा है, देखिए आपके लिए 5400, लड़का नहीं है भेजने को, आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 5400, "delivery": false, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्ते, बोलिए. हम्म थोड़ी देर बाद कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ, ताला है, वो क्या है ४८० रुपीज़ है.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "ha bolo, ji, available hai, paune chaar hajar rupaye ka hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 3750, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "रेट पौने सोलह सौ हां पंखा है देखो घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 1575, "delivery": true, "alternative": null}}
{"product": "Goodyear tyre", "said": "हाँ बोलिए sir। है ना, कितने चाहिए? ३६५० रुपीज़ है। अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 3650, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "हो, मिळेल, तुम्हाला 360 ला देतो", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "haa bhaiya, bataiye haan, maal pada hai MRP chah sau painsath likha hai, panch so laga denge.", "answer": {"in_stock": true, "price_inr": 500, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ जी, बताइए। सर आज बहुत बिज़ी डे है। दस फाइनल प्राइस है। वो। एक कस्टमर खड़ा है, होल्ड करो। सीज़न टाइम है ना, डिमांड ज़्यादा है। स्टॉक में है। बेसिकली डिलीवरी बंद है अभी, दुकान से लेलो। और कुछ?", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब ख़त्म नहीं हुई, है। पैकेट सवा सात सौ का है, खुला एक चार सौ अस्सी का", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "aamchyakade nasta te local seal kit aahe, donshe vees la mhanje porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local seal kit"}}
{"product": "Usha 230 mm exhaust fan", "said": "हां तर मी नवीन आहे, मला माहीत नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "नहीं है जी, ख़त्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "नमस्ते, बोलिए. मिल जाएगा, है अपने पास. आज सुबह से लाइट नहीं है. बाज़ार में माल की कमी चल रही है. तीन सौ तीस रुपये. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. त्योहार का सीज़न है ना. ठीक है?", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "han mil jayega bhai sava pachis sau me de dunga", "answer": {"in_stock": true, "price_inr": 2525, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "arre hai, bilkul hai, savaa aath so rupees, han, stock men bara piece hai", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "kya hai na kam nahi hai, bahut pada hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan, hai apne paas, 15 ka hai, umm courier kar denge", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb hai, mil jayega. abhi rate assi ho gaya, pichle mahine 64 tha.", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हेलो, बेसिकली सवा सौ रुपये का है, है भाई, मिल जाएगा, आके ले जाओ.", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "70 ka padega. han ji, bulb available hai", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "हाँ रखा है हम्म एक का ढाई सौ, दस का डिब्बा दो हज़ार दो सौ का", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ सर, अवेलेबल है। तेरह सौ रुपये का है। अमरोन 12V 5Ah बैटरी भी है, वो 1100 का। आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hai, bilkul hai do sau pachas, 270 nahin bola maine dekho aapke ghar tak bhijwa denge.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "umm malik abhi nahi hain, baad mein phone karo. ek customer khada hai, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "स्टॉक में है। पच्चीस रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हाँ, माल पड़ा है. रेट साढ़े ग्यारह सौ है.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "आहे ना। सव्वा चारशे फिक्स। ठीक आहे?", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "बल्ब संपला नाही आहे दहा घेतले तर 245 प्रमाणे एक piece दोनशे साठ pickup करावा लागेल", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "येस, बोला, तुम्हाला 15 ला देऊ, म्हणजे आहे सर, रेडी स्टॉक आहे", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "जीबोलिए जीबोलिए, अभी नहीं है, ऑर्डर पर मंगवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हेलो? हाँ, बोलो भाई, सर आज बहुत बिज़ी डे है, एक कस्टमर खड़ा है, होल्ड करो, हाँ, है, एक सौ बीस रुपीज़ है, और कुछ?", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "सवा सोलह सौ का है, इससे कम नहीं होगा, वायर ख़त्म नहीं हुई, है.", "answer": {"in_stock": true, "price_inr": 1625, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "वो तो कल ही ख़त्म हो गया जी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "अच्छा, अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है, बेस्ट प्राइस १८०", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "Motor hai, mil jayegi, 12730 likha hai, discount karke sava gyarah hazar, abhi delivery band hai", "answer": {"in_stock": true, "price_inr": 11250, "delivery": false, "alternative": null}}
{"product": "Schneider Classy switch", "said": "hello, haan bola, bara stock nahi aahe sadhya, Legrand 25 amp switch wala aahe, tyacha price pachshe pannas, haan haan, aiktoy, aaj sakali light navhti, kay sangu, sakal pasun gardi aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 25 amp switch"}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हाँ जी बताइए हम्म मिल जाएगा है अपने पास तीन सौ बीस फ़ाइनल", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए सर रेट बारह सौ पचास है पर अभी स्टॉक नहीं है अभी नहीं है परसों तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "नहीं मिलेगा भाई जी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "यस, अवेलेबल है, आपके लिए ४५०, सेम डे डिलीवरी कर देंगे.", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हां जी, मिल जाएगा. भेज देता, पर आज लडका नहीं आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Haan bola, ahe ahe, ekshe vees la lavto, MRP ekshe panchechalis ahe, gharapoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ जी sir, बताइए, सीज़न टाइम है ना, डिमांड ज़्यादा है, हाँ जी, stock में पड़ा है, बेस्ट प्राइस पौने दो सौ, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "ख़त्म हो गया था... नहीं, एक बचा है। रेट पौने सात सौ है। दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 675, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब पडलाय, बघा, अं तुम्हाला सव्वा दोनशेला, घरी पाठवतो, 30 वेगळे.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "nahi hai... ruko ruko, hai, ek pada hai, pone che hajar rupaye ka hai.", "answer": {"in_stock": true, "price_inr": 5750, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "Available hai, chaar sau paintees ka hai, aapke liye pauna chaar sau, shaam tak ghar pe aa jayega.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "वो आपके लिए 320 हाँ रखा है", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब पड़ा है एक पीस 140, 10 लोगे तो 110 के हिसाब से", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "हाँ जी ऑयल रखा है हमारा रेट 265 MRP तो 320 है खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 265, "delivery": false, "alternative": null}}
{"product": "Crispy LED torch", "said": "नहीं है बोलो तो मंगवा देता हूँ दो दिन लगेंगे खुद आकर उठाना पडेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ जी, मिल जाएगा. एक का पैंतीस, बीस का डिब्बा 640 का", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "देखिए नहीं नहीं, है ना, 110 रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "hello bola na light sampli ajun kahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "hello, han boliye are kya bataun, subah se bahut rush hai stock me hai sir market me maal ki kami chal rahi hai ladka de aayega, 40 rupaye auto ka company ne is mahine phir rate badhaya item price paune nau sau, delivery charge alag aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 875, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe ahe, 610 rupayanna padel, porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ भाई, बोलो। रेट 250 है, पर अभी स्टॉक नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हैलो, कौन? पौने छह हज़ार फ़ाइनल। हाँ जी, मशीन रखी है।", "answer": {"in_stock": true, "price_inr": 5750, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "है भाई, मिल जाएगा, सिंगल piece ३०५, १२ का फुल box ३१८०, एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 305, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ जी, वो हम नहीं रखते भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "पंखा संपलाय। Havells 450 mm pedestal fan घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 450 mm pedestal fan"}}
{"product": "Metro Moulding helmet lock", "said": "हाँ हाँ sir, available है. बेसिकली डेढ़ सौ का एक, 25 का डब्बा तीन हज़ार तीन सौ पचास का. तो.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "है ना, कितने चाहिए? लास्ट मंथ दो हज़ार तीन सौ पचास था, अब छब्बीस सौ पचास है", "answer": {"in_stock": true, "price_inr": 2650, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "मतलब नहीं है भाई, खत्म हो गया, Usha 1000W rod है, पौने नौ सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha 1000W rod"}}
{"product": "V-GRIP steering cover for Ritz", "said": "हाँ है, प्रिंटेड MRP 940, अपना price 625, आपको shop से पिकअप करना होगा.", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "है भाई, मिल जाएगा. देखिए लिस्ट में एक हज़ार पचास है, आपको पौने नौ सौ पड़ेगा.", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Duravolt नाहीये फिलिप्स 20W बॅटन आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 20W बॅटन"}}
{"product": "Aegon angle grinder", "said": "मशीन आहे, मिळेल। रेट तेराशे आहे। हां। पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "namaskar bola battery aahe milel savvabarashe la lavto mrp ek hazar sahashe ahe", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "aahe ki, milel 425 plus GST gharpoch pohochvun deu.", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Induro plier", "said": "abhi khatm hai, shaam tak aa jayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bola, kay pahije? kal Ravivar hota mhanun dukan band hota, ahe, satrach urlet, kay sangu, sakal pasun gardi ahe, 220 rupees, kalva mag.", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "Ahe na, tumhala sade charashe la deto, delivery? nahi ho, te nahi karat.", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "एक मिनट... हाँ बोलिए, रेट 110 है सर, हाँ सर, वो मॉडल अवेलेबल है", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "हेलो, हाँ बोलो. वो क्या है वो हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "Han bhai, bolo, machine padi hai, pachis sau pachas ka milega", "answer": {"in_stock": true, "price_inr": 2550, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "हो गिझर शिल्लक आहे रेट पावणे सव्वीसशे आहे सर", "answer": {"in_stock": true, "price_inr": 2575, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो? हाँ, आवाज़ आ रही है. जी, उपलब्ध है. एक सौ अस्सी का पड़ेगा. पंद्रह पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हम्म है ना, कितने चाहिए? आपको पाँच सौ पच्चीस में दे देंगे, MRP पाँच सौ पचपन है।", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "हैलो, कौन? चार सौ पैंतीस लगेगा. है ना, कितने चाहिए? आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "Avita switch", "said": "hello? aawaj yetoy ka? haan bola, company ne parat rate vadhavle, ek giraik aahe, ek minute, jara velane phone kara, aawaj tutatoy, motyane bola, kay sangu, sakal pasun gardi aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "Dekho sava hazar hoga, han, mil jayega bhai, thik hai?", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हो, बल्ब शिल्लक आहे, तुम्हाला ऐंशीला देतो", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "अच्छा टायर है मिल जाएगा बारह सौ पचास में देंगे फ्री डिलीवरी है", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "जी बोलिए। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। हाँ तो पुट्टी ख़त्म हो गई। त्योहार का सीज़न है ना। Birla wall putty 40 kg दे सकता हूँ, साढ़े छह सौ रुपये का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Birla wall putty 40 kg"}}
{"product": "Philips 5W LED bulb", "said": "सौ रुपये का है. है ना, कितने चाहिए?", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "आहे ना। आधी एकशे सत्तर होता, आता प्राइस एकशे पंच्याण्णव झाला।", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "बरं रेट 378च्या आसपास असेल, स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "stock में है सर, अं पौने दो सौ का एक, छह का डब्बा नौ सौ पैंतीस का.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "स्टॉक में है सर। पौने पंद्रह सौ रुपीज़ है। 7 बजे तक मैं यहीं हूँ।", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "सॉरी सर, अभी आउट ऑफ स्टॉक है. और कुछचाहिए सर?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, बताइए। देखो सवा तीन सौ में दे देंगे। हाँ, रखा है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन आहे, मिळेल. 2190 रुपये होतील. पोरगा घरी आणून देईल.", "answer": {"in_stock": true, "price_inr": 2190, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "हाँ तो है भाई, रेट बदल गया है, अब पौने दो सौ है, पहले एक सौ साठ था, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "Haan, mil jayega bhai pandrah final sham tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "हो दादा, आहे. तुम्हाला साडेसहाशेला देतो. आठ वाजता दुकान बंद होतं.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "है है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, अच्छा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ जी, स्टॉक में पड़ा है, बीस फ़ाइनल", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Woh hum nahi rakhte.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "जी, उपलब्ध है, तीन सौ MRP लिखा है, एक सौ अस्सी लगा देंगे, होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "haan, kaun? acha, bolo, available hai, courier kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए सर, डेढ़ सौ का पड़ेगा, हाँ सर, वो मॉडल अवेलेबल है.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "हेलो हाँ बोलो हाँ हाँ रेडी स्टॉक है MRP 270 है आपको दो सौ पंद्रह में दे देंगे", "answer": {"in_stock": true, "price_inr": 215, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "हां बोला साहेब सव्वासहाशे रुपये हो कव्हर शिल्लक आहे", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "मशीन आहे, मिळेल, दोन हजार आठशे रुपये, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 2800, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट आहे मिळेल ८३० रुपये होतील दुकानातून न्यावं लागेल बघा सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "Ritco tap", "said": "स्टॉक में है एमआरपी चार सौ है पर हम ढाई सौ में देते हैं", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "अं त्योहार का सीज़न है ना। देखो। पता नहीं भाई। आवाज़ कट रही है, ज़ोर से बोलिए। आज सुबह से लाइट नहीं है। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। औरकुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "अं कव्हर संपलं नाही, आहे. रेट 635 आहे. घरी पाठवतो, 20 वेगळे.", "answer": {"in_stock": true, "price_inr": 635, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "हाँ जी, बताइए, जी जी, उपलब्ध है, मतलब तीन सौ", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "हो, आहे शिल्लक, म्हणजे, हां एका नगाचे सव्वा दोनशे, 6 नगांचा बॉक्स 1120 ला, पाठवायची सोय नाही.", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "आहे आहे. दीडशेला पडेल. म्हणजे पोरगा पाठवतो, चाळीस रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "haan tar helmet padlay. pathavayla kahi problem nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "तीन सौ साठ रुपीज़ है हाँ जी स्टॉक में पड़ा है", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "हेलमेट नहीं है, खत्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "stock नहीं है अभी. देखो गुडईयर बोल्ट कटर available है, उसका प्राइस साढे नौ सौ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गुडईयर बोल्ट कटर"}}
{"product": "Ritco tap", "said": "हम्म stock में है, MRP 390 है, discount के बाद दो सौ पैंतालीस पड़ेगा", "answer": {"in_stock": true, "price_inr": 245, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "namaste ji, boliye, bulb khatam nahi hua, hai, arre teen sau painsath me ho jayega, upar MRP panch sau pachas likha hai", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हम्म जी, उपलब्ध है, 310 का... नही नही, गलत बोला, पौने चार सौ का है", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हम्म हाँ, रखा है, भेज देता, पर आज लडका नही आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Easy4buy horn", "said": "है भाई, छह सौ चालीस लगा देंगे, वैसे MRP 960 लिखा है.", "answer": {"in_stock": true, "price_inr": 640, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "मशीन ख़त्म हो गई, Cheston 500 वॉट ड्रिल मशीन है, सेम क्वालिटी, आपको shop से पिकअप करना होगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Cheston 500 वॉट ड्रिल मशीन"}}
{"product": "silicone sealant", "said": "हाँ रेडी stock है तीन सौ पचास GST एक्स्ट्रा मतलब आपको shop से pickup करना होगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "मी नवीन आहे मला माहीत नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "एक मिनट... बोलिए, हां, रेडी स्टॉक है, सत्तर रुपीज है, वो शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "नमस्ते, बोलिए. मतलब हाँ, माल पड़ा है. बारह सौ पचास का पड़ेगा", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हॅलो। नाही वाटतं... थांबा, आहे, एक उरलाय", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हां जी, स्टॉक में पडा है। पौने छह सौ लगेगा। देखो", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "मतलब सॉरी रॉन्ग नंबर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "haan, hai apne paas, sava hazar ka hai, isse kam nahi hoga, haan toh bata dena", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "है भाई, होम डिलीवरी हो जाएगी, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "तो हां बोलिए सर अभी अवेलेबल नही है ऑर्डर प्लेस किया है मंडे तक आएगा अं बेस्ट प्राइस 120", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "नमस्ते, बोलिए। अच्छा नहीं है, मंगवा देंगे, दो दिन लगेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "अं हो, सात पीस आहेत, तुम्हाला 215 ला देऊ", "answer": {"in_stock": true, "price_inr": 215, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ जी, बल्ब रखा है, पैकेट 390 का है, खुला एक 260 का", "answer": {"in_stock": true, "price_inr": 390, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो। हो, मिळेल काका।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "म्हणजे संपलं नाही अजून, आहे, 1600 रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 1600, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "है भाई, चार हज़ार आठ सौ पचास का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 4850, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "हेलो? हाँ, बोलो भाई, हाँ, रेडी स्टॉक है, 700 MRP लिखा है, साढ़े चार सौ लगा देंगे.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हॅलो? आवाज येतोय का? हां, बोला, अं, आत्ता नाहीये, परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब पडा है, सौ लगेगा, बॉय भेज दूंगा, बेसिकली ओके?", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "ढाई सौ का है, इससे कम नहीं होगा, हाँ हाँ, है.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "हाँ बोलिए सर, तेईस पीस हैं स्टॉक में, एक कस्टमर खड़ा है, होल्ड करो, सवा बारह सौ रुपीज़ है, सर आज बहुत बिज़ी डे है, सॉरी, डिलीवरी सर्विस नहीं है, वो क्या है सीज़न टाइम है ना, डिमांड ज़्यादा है", "answer": {"in_stock": true, "price_inr": 1225, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "है भाई एक सौ पचास का नहीं नहीं गलत बोला दो सौ का है अं दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "मतलब कब्ज़ा ख़त्म हो गया, steel कब्ज़ा पड़ा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel कब्ज़ा"}}
{"product": "50 mm padlock", "said": "येस बोला तर हो कुलूप शिल्लक आहे 280 फायनल", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "नहीं है जी, ख़त्म, Aptech multimeter वाला है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aptech multimeter"}}
{"product": "10W-30 engine oil 1 L", "said": "umm abhi busy hun, aadhe ghante baad phone karna", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हेलो, हाँ बोलो, अभी बिज़ी हूँ, call बैक करता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "हैलो, कौन? शाम तक माल आ जाएगा, अभी नहीं है। शाम तक घर पहुँच जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "सॉरी सर, अभी आउट ऑफ स्टॉक है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ख़त्म हो गया था... नहीं, एक बचा है, देखो रेट नौ सौ है.", "answer": {"in_stock": true, "price_inr": 900, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "Sorry sir, abhi out of stock hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "हाँ जी मिल जाएगा लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "Ji boliye, paune panch sau ka hai, haan hai, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "शिल्लक नाही, कळवा मग.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "han, maal pada hai, haan toh courier kar denge sir, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Haan hai, 235 rupaye lagenge, bata dena.", "answer": {"in_stock": true, "price_inr": 235, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "savaa paanch hazar rupaye. ha, mil jaega bhai", "answer": {"in_stock": true, "price_inr": 5250, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "stock mein hai sir. free delivery hai. sir aaj bahut busy day hai. season time hai na, demand jyada hai. kya hai na network issue hai, aawaz break ho rahi hai. aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हेलो? हाँ, बोलो भाई, अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Aaj nahi hai, kal ayega, sadhe barah sau rupaye ka hai, bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Vega Crux helmet", "said": "Vega वाला ख़त्म है, Sage Square helmet है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sage Square helmet"}}
{"product": "50 mm padlock", "said": "हाँ बोलिए सर। हाँ है, कोई प्रॉब्लम नहीं। सवा तीन सौ लगेगा। घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "tumhala 3600 la deto, ho, light aahe", "answer": {"in_stock": true, "price_inr": 3600, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "मोटर संपली नाही, आहे. बिल हवं असेल तर GST वेगळा, रेट साडे अकराशे. घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "Benz helmet", "said": "हेल्मेट संपलं, हां वेगा हेल्मेट वाला देऊ शकतो, आठशे तीस पडेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वेगा हेल्मेट"}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "देखो अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पडेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan, maal pada hai, sava teen sau mein de dunga", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अं संपले सगळे, हॅवेल्स 1 sq एमएम वायर वाला आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हॅवेल्स 1 sq एमएम वायर"}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "seth ji bahar gaye hain wahi batayenge", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हाँ बोलिए। वो हम नहीं रखते। Packy Poda car mat अवेलेबल है, उसका प्राइस 1400। लड़का भेज देता हूँ", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Packy Poda car mat"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "साठ का है, इससे कम नही होगा, मिल जाएगा, है अपने पास, लड़का छोड देगा घर पर", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "देखो स्टॉक में है ५८० MRP लिखा है ३६५ लगा देंगे", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "हम्म पौने दस हज़ार फ़ाइनल. जी, उपलब्ध है.", "answer": {"in_stock": true, "price_inr": 9750, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "aahe sir, ready stock ahe. satshe tees la lavto, MRP naushe tis ahe. manus pathavto ghari.", "answer": {"in_stock": true, "price_inr": 730, "delivery": true, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "स्टॉकमध्ये आहे, चारशे पंचेचाळीसचं आहे, तुमच्यासाठी साडेतीनशे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "हां मशीन आहे मिळेल पाच हजार दोनशे रुपये होतील घरपोच नाही तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 5200, "delivery": false, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "अच्छा अच्छा है भाई। सवा सैंतीस सौ का पडेगा। डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": 3725, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "वो अभी नहीं है स्टील कब्ज़ा है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टील कब्ज़ा"}}
{"product": "4 inch door hinge", "said": "हाँ जी कब्ज़ा रखा है रेट 50 है अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "nahi nahi, hai na.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "ho, bulb ahe, sade charashe la deto, RSCT 9 watt LED bulb pan aahe, te pachshe sattar la, manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ जी, मिल जाएगा, आपके लिए तीन सौ अस्सी", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Aahe ki, milel. teenshe ainshi rupaye hotil.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "स्टॉक में नहीं है AutoClean वाइपर ब्लेड है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AutoClean वाइपर ब्लेड"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर संपली नाही, आहे, तुम्हालातुम्हाला सव्वा सोळाशेला देतो, घरपोच पोहोचवून देऊ.", "answer": {"in_stock": true, "price_inr": 1625, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "यस सर, बोलिए. वो मॉडल अवेलेबल नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "हाँ जी, मिल जाएगा। २००० का है, इससे कम नहीं होगा। शाम तक पहुँचा देंगे।", "answer": {"in_stock": true, "price_inr": 2000, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "उद्या फोन करा. हां बाजारात माल कमी आहे सध्या", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "सर आज बहुत बहुत बिज़ी डे है, नेटवर्क इशू है, आवाज़ ब्रेकहो रही है, हाँ, माल पड़ा है, सीज़न टाइम है ना, डिमांड ज़्यादा है, पौने तीन सौ लगेगा, 320 वाला रेट पुराना है, वो नहीं, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "9W संपलं सिस्का 12W एलईडी बल्ब आहे चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिस्का 12W एलईडी बल्ब"}}
{"product": "water pump seal kit", "said": "मतलब हाँ जी मिल जाएगा MRP दो सौ सत्तर है पर दो सौ दस लगा देंगे", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ha kaun acha bolo sava do so rupees haan hai apne paas haan ha sun raha hun ek customer khada hai ek minute home delivery hai", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "haan haan, bolo, haan, pankha hai, sawa iktis sau ka hai, isse kam nahi hoga, kya hai na bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 3125, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "है भाई। डेढ़ सौ का पड़ेगा।", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "अं। आज नाही, उद्या येईल। सव्वा एकतीसशे रुपये होतील।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "han bolo, main naya hun, mujhe nahi pata, seth se puchho", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "स्टॉक में है sir, 370, sorry sorry, 300, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, सर आज बहुत बिज़ी डे है", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "Cover khatam nahin hua, hai, sadhe chhah sau laga denge, waise saat so sattar ka hai.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "मतलब हाँ, माल पड़ा है, 1000 लगेगा.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "हाँ भाई, बोलो, पौने चौदह सौ रुपये लगेंगे, जी, उपलब्ध है, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 1375, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "चारशे पस्तीस रुपये। हो, कव्हर आहे।", "answer": {"in_stock": true, "price_inr": 435, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां बोला साहेब. प्लेट पडलीये. कुरिअर करतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "stock madhe aahe. saha nagancha box 1800 la, sutta ek savvateenshe. self pickup aahe", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "नमस्ते, बोलिए, हाँ, रखा है, २१० रुपये, शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "अं आमच्याकडे आहे. तुम्हाला आठशे पाचला देतो, MRP 910 आहे", "answer": {"in_stock": true, "price_inr": 805, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "जी बोलिए, गीज़र है, मिल जाएगा, आपके लिए साढे सत्ताईस सौ, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ सर, वो मॉडल अवेलेबल है। डेढ़ सौ रुपीज़ है। हाँ, तेरह पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "बीस पीस अवेलेबल है अभी, तीन सौ पचपन रुपये का है, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 355, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Stock samplay, bara white cement deu shakto, savvadonashe padel.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Sage Square helmet", "said": "आउट ऑफ स्टॉक नहीं है, अवेलेबल है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "हाँ जी, मिल जाएगा, हाँ तो रेट पौने ग्यारह सौ है, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "है ना कितने चाहिए २० का डिब्बा चार हज़ार छह सौ पचास का एक पीस ढाई सौ", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "आधी पावणेचारशेला होता, आता आहे की नाही बघावं लागेल.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "Hai... are nahi, wo dusra wala hai, ye vala nahi hai, Ravirandal sealant rakha hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ravirandal sealant"}}
{"product": "10W-30 engine oil 1 L", "said": "अभी नहीं है, Shell 20W-40 engine oil है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Shell 20W-40 engine oil"}}
{"product": "AmiciFlo water pump", "said": "सवा ग्यारह सौ रुपीज़ है, स्टॉक में है सर, बताओक्या करना है.", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "bhaiya aap ruko do minute, phone pe hun, company ne is mahine phir rate badhaya, abhi nahi hai, parso tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "मशीन संपली नाही, आहे. सहा हजार पाचशे सांगितलं का? चुकलो, साडेपाच हजार आहे", "answer": {"in_stock": true, "price_inr": 5500, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "आज नहीं है कल आएगा आपको आकर लेना पडेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हां नाहीये मागवून देतो दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Haa hai bhijava dunga.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "120 रुपये का है, जी, उपलब्ध है.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "matlab abhi nahi, order pe mangva denge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "हो दादा, आहे, बिल हवं असेल तर GST वेगळा, रेट चारशे पंचावन्न", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "nahi kasa, ahe na. satashe panch rupees.", "answer": {"in_stock": true, "price_inr": 705, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "बघा हो, मिळेल. ८७... थांबा, लिस्ट बघतो... नव्वद.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ है कोई प्रॉब्लम नहीं आपकेलिए सवा बारह सौ भेज देता पर आज लड़का नहीं आया आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1225, "delivery": false, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "एक मिनिट हां बोला मशीनचा स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हां 25 नग शिल्लक आहेत रेट शेठ सांगतील कुरिअर करतो अजून काही", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "हेलमेट पड़ा है। ८४० रुपये रुपये लगेंगे। शाम तक पहुँचा देंगे।", "answer": {"in_stock": true, "price_inr": 840, "delivery": true, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "हाँ जी, ऑयल रखा है, अं मोबिल 5 वॉट ऑयल भी है, उसका नौ सौ साठ, और आपवाला सवा बारह सौ का", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "yes sir, boliye, han hai, sava satrah sau final price hai, dukan se le jana padega", "answer": {"in_stock": true, "price_inr": 1725, "delivery": false, "alternative": null}}
{"product": "MK 16A switch", "said": "मिल जाएगा, है अपने पास। एक पीस 145, 10 लोगे तो 120 के हिसाब से", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "स्टॉक में नहीं है. कैस्ट्रॉल 20W-40 इंजन oil दे सकता हूँ, साढे पाँच सौ रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कैस्ट्रॉल 20W-40 इंजन oil"}}
{"product": "Vega Crux helmet", "said": "हाँ बोलिए सर पता नहीं भाई बाद में कॉल करो सर आज बहुत बिज़ी डे है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "स्टॉक में है, अं 10 लोगे तो एक सौ पचासी के हिसाब से, एक का सवा दो सौ, लड़का नहीं है भेजने को, आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Hai bhai 420 me de dunga arre haan, stock me 2 dabbe hai khud aake uthana padega.", "answer": {"in_stock": true, "price_inr": 420, "delivery": false, "alternative": null}}
{"product": "Aptech multimeter", "said": "है भाई मिलजाएगा भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "टायर ख़त्म हो गया. CEAT 3.00 tyre चलेगा? उसका ग्यारह सौ है. वो वो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "CEAT 3.00 tyre"}}
{"product": "Candes 10 L geyser", "said": "अभी नहीं है, परसों तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हेलो। कंपनी ने फिर रेट बढ़ा दिया। में माल की कमी चल रही है। आज सुबह से लाइट नहीं है। माल खत्म है। Accessory Mart car mat है, 1350 का। घर पर भेज देंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Accessory Mart car mat"}}
{"product": "Dewalt angle grinder", "said": "है भाई वो देखो डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "स्टॉक में है, ढाई सौ रुपीज़ रुपीज़ है, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "बघा चारच पीस उरलेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "यस, अवेलेबल है, रेट बदल गया है, अब 550 है, पहले 430 था", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग आहे, मिळेल, दोनशे चाळीसला देतो देतो, घरी पाठवतो, कळवा मग", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "येऊन घेऊन जा, सांगा काय करायचं.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ जी, प्लेट रखी है. आठ सौ चालीस है, छह सौ सत्तर नहीं.", "answer": {"in_stock": true, "price_inr": 840, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "नल नही है खत्म हो गया जी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "wo item hum keep nahi karte, Michelin wiper blade rakha hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin wiper blade"}}
{"product": "Prithvi 6.5 HP water pump", "said": "है भाई, 10800 में दे देंगे, अच्छा Kirloskar 0.5 HP pump भी है, वो 11260 का, अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": 10800, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ सर, वो मॉडल अवेलेबल है. आठ सौ बीस लगा देंगे, वैसे एक हज़ार पचास का है", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "है ना, कितने चाहिए? सवा दो सौ लगेगा, अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "haan, hai apne paas ji.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "ji haan, rakha hai, aapke liye 2499", "answer": {"in_stock": true, "price_inr": 2499, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "Ho bola, ho, milel, MRP 585 ahe, pan charshe sattar la lavto, porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 470, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Hai na, kitne chahiye? pachis sau pachas rupaye lagenge, nau baje dukaan band hoti hai, ladka ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": 2550, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "nahi milanar, tyachya aivaji Type R horn deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Type R horn"}}
{"product": "silicone sealant", "said": "rate 160 आहे, पण आत्ता stock नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "आमच्याकडे आहे, बाराशे रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "stock madhe ahe, pavneteenshe rupees, porga pathavto.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ जी, बताइए, दुकान से ले जाना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो? हाँ, आवाज़ आ रही है, वो तो कल ही ख़त्म हो गया, अच्छा white cement वाला चलेगा? उसका 280 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Vormir 200A welding machine", "said": "आहे आहे ना, साडे पंचेचाळीसशे पडेल, 5770चा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 4550, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hai na, kitne chahiye? 240 rupees ka hai, kya hai na NGK Pulsar 180 spark plug bhi hai, wo 210 ka.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "हाँ हाँ, है। Anchor 150 mm exhaust fan भी है, उसका एक हज़ार सात सौ, और आपवाला सवा उन्नीस सौ का", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "नहीं है... रुको रुको, है, एक पड़ा है। बेस्ट प्राइस सवा सात सौ", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "आपके लिए 660, यस, अवेलेबल है, डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 660, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "देखिए बल्ब है मिल जाएगा एक सौ साठ में दे देंगे घर पर भेजदेंगे 60 अलग से", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "अं है ना, कितने चाहिए? बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "मतलब अरे क्या बताऊँ, सुबह से बहुत भीड़ है। शाम तक माल आजाएगा, अभी नहीं है। त्योहार का सीज़न है ना। और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear bolt cutter", "said": "हाँ जी, मिल जाएगा, मतलब अब साढे सात सौ है, पहले आठ सौ बीस था", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "ट्यूब आहे, मिळेल, 830 MRP आहे, पण 465 ला देऊ, दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 465, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हाँ भाई, बोलो, हम्म हैलो? आवाज़ नहीं आ रही, बाद में करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "बोलाना. हो हो, मशीन आहे. साडेसतराशेला पडेल. पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "१० mm नहीं है 12 एमएम बोल्ट है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 एमएम बोल्ट"}}
{"product": "8 mm drill bit", "said": "हो, मिळेल। तुम्हाला सव्वाशेला देतो।", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "नाही म्हणालो का? आहे हो, म्हणजे, बघा डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "जी बोलिए। नहीं है। mechanical सील है, वो ले लीजिए।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "mechanical सील"}}
{"product": "Ritco tap", "said": "haa ji, nal available hai, matlab MRP chaar sau hai, aapko dhaee sau mein de denge", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Goodyear bolt cutter", "said": "895 rupees hai, hai bhai.", "answer": {"in_stock": true, "price_inr": 895, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "sheth baher gelet te aale ki sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "हाँ बोलिए सर, कल फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "नमस्कार, बोला। बिजागरी संपलीये। उद्या सकाळी माल येईल, आत्ता नाहीये। शंभर फिक्स। घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "ते मिळणार नाही इथे, Anchor 6 socket extension board चालणार असेल तर सांगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 6 socket extension board"}}
{"product": "Benz helmet", "said": "हाँ, माल पड़ा है. 735 का... नहीं नहीं, गलत बोला, 800 का है.", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "kabza hai, mil jaega, aapke liye 100, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan, hai apne paas, saath lagega ab, pehle 45 ka tha", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ रखा है बेसिकली लोकल सील kit भी है उसका दो सौ पचहत्तर और आपवाला तीन सौ सत्तर का", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "है ना, कितने चाहिए? एक सौ नब्बे लगेगा", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "sir आज बहुत बिज़ी डे है। हाँ है, कोई प्रॉब्लम नहीं। सीज़न टाइम है ना, डिमांड ज़्यादा है। एक कस्टमर खड़ा है, होल्ड करो। नेटवर्क इशू है, आवाज़ ब्रेक हो रही है। आपसे दो सौ चालीस लेंगे, लिखा दो सौ पैंसठ है। और कुछ?", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "हां जी हां बोलिए sir अभी बिजी हूं call बैक करता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "मशीन ख़त्म नहीं हुई, है, पैंतीस सौ पचास लगेगा, 3200 वाला रेट पुराना है, वो नहीं", "answer": {"in_stock": true, "price_inr": 3550, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "हाँ है. 2430 का... नहीं नहीं, गलत बोला, 2900 का है", "answer": {"in_stock": true, "price_inr": 2900, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "ek minute... haan bolo, pankha khatam ho gaya, uski jagah Anchor 150 mm exhaust fan le sakte ho, same day delivery kar denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Anchor 150 mm exhaust fan"}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "अच्छा अच्छा, बेसिकली हाँ रखते हैं, पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "अभी नहीं है, परसों तक आ जाएगा. होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "helo, haan boliye acha price 1500 hai available hai, koi problem nahi", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "आमच्याकडे नसतं, ऑर्डर करून मागवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "अभी नहीं है sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "stock madhe aahe, rate saadetinashe ahe sir", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "जी जी, उपलब्ध है, बीस फ़ाइनल, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "मतलब रेट तीन सौ चालीस है, पर अभी स्टॉक नही है, शाम तक माल आ जाएगा, अभी नही है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "अं पिकअप करावा लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ho bola. aahe ki, milel. charshe daha fix.", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "वो, मोटर पड़ी है, एमआरपी 1430 है पर बारह सौ पचास लगादेंगे", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan ji, boliye, hai na, kitne chahiye? do sau panch ka hai, aapke liye ek sau painsath", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "Usha नहीं है, हैवेल्स 450 एमएम पेडस्टल fan है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 450 एमएम पेडस्टल fan"}}
{"product": "Accessory Mart car mat for Duster", "said": "देखो गलत नंबर लगाया है आपने, वो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "है ना कितने चाहिए साठ रुपये लगेंगे क्रॉम्पटन 2.25 mfd कैपेसिटर भी है वो पचास का", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "hai, bilkul hai sir", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "Cheston नहीं है Stanley 550 watt drill machine है खुद आके ले जाना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Stanley 550 watt drill machine"}}
{"product": "8 mm drill bit", "said": "ho, milel, ainshi rupees, kami nahi honar", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "Kay aahe na machine padliye, char hazar satashe pannas final, 7 vajeparyant mi ithech aahe.", "answer": {"in_stock": true, "price_inr": 4750, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "म्हणजे हो मशीन आहे साडेअठराशेला पडेल कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "हाँ, है. पौने छह हजार का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 5750, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ हाँ जी सर, बताइए, मतलब अभी नहीं है, ऑर्डर पे मंगवा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "है भाई मिल जाएगा साढ़े पाँच सौ का है आके ले जाओ", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हैलो? हां, आवाज आ रही है. बाज़ार में माल की कमीचल रही है. त्योहार का सीजन है ना. लडका चाय लेने गया है. स्टॉक में है. 1100 रुपये. शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हो हो दादा, आहे। ५४१०... नाही नाही, ५२७७।", "answer": {"in_stock": true, "price_inr": 5277, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello, ha boliye, price pauna terah sau hai, stock me hai sir, delivery ho jayegi sir, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 1275, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, बॅटरी शिल्लक आहे, तर पावणेतेराशे पडेल, 1330चा रेट जुना आहे, तो नाही, पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 1275, "delivery": false, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "ऑइल पडलंय, आधी 325 होता, आता चारशे झाला.", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हां तर हो, आहे शिल्लक, सव्वा दोनशे होतील, डिलिव्हरी नाही, दुकानात येऊन घ्या, तर", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "mhanje sampala hota... nahi, ek ahe ajun, tumhala 30 la deu.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा पड़ा है. दस का डिब्बा 915 का, एक पीस 110.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हां बोला, ट्यूब आत्ता नाहीये, संपली, अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "नहीं नहीं, है ना, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "चौदह सौ का पड़ेगा, हाँ है, कोई प्रॉब्लम नहीं", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ जी बताइए माल खत्म है हम्म Supreme दे सकता हूँ ३० रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Supreme"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी वायर रखी है साढ़े सोलह सौ रुपये लगेंगे होम डिलीवरी हो जाएगी बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "Ji boliye haan ji, mil jaega delivery nahi hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हैलो कौन अभी कन्फर्म नहीं कर सकता गोडाउन चेक करना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Han bhai, bolo, sir aaj bahut busy day hai, network issue hai, aawaz break ho rahi hai, haan toh not available sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "Rate saadhe gyarah sau hai, par abhi stock nahi hai abhi stock me nahi hai, naya maal Monday ko aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "अच्छा, बेसिकली मशीन ख़त्म हो गई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "nahiye, magvun deto, don divas lagtil, home delivery aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ, 28 बैग पड़े हैं, साढ़े आठ सौ रुपये का है, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "देखो stock में है। 430 का पड़ेगा", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "सॉरी, आउट ऑफ स्टॉक आहे, Themisto multimeter वाला आहे, सेम क्वालिटी, बघा डिलिव्हरी देतो सर.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Themisto multimeter"}}
{"product": "teflon tape", "said": "वो क्या है नहीं है, बोलो तो मंगवा देता हूँ, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "acha stock nahin hai abhi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "मतलब हाँ बल्ब है एक piece 280 का मिलता है पैकेट 425 का कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "अभी अवेलेबल नही है ऑर्डर प्लेस किया है मंडे तक आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "बोला काय पाहिजे स्टॉकमध्ये आहे हजार रुपये कमी नाही होणार पत्ता पाठवा पाठवूनदेतो", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "Acha stock mein hai sir, rate ek sau nabbe hai sir, home delivery available hai, sath charge", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "21 रोल हैं स्टॉक में. प्राइस प्राइस कन्फर्म करके बताता हूँ. एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो? आवाज येतोय का? हां, बोला, स्टॉक संपलाय, बघा local spark plug वाला आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "Galaxy car cover for Micra", "said": "पोरगा घरी सोडून देईल.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Osram LED headlight", "said": "Acha bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "येस, बोला, आवाज तुटतोय, मोठ्याने बोला, कंपनीने परत rate वाढवले, एक गिऱ्हाईक आहे, एक मिनिट, सध्या आउट ऑफ स्टॉक आहे, नवीन stock परवा येणार आहे, काय सांगू, सकाळ पासून गर्दी आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aahe ahe, eka nagache chalis, 25 nagancha box 895 la, pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": 40, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "पैंतीस रुपये लगेंगे जी उपलब्ध है", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "नमस्ते, बोलिए, वो हाँ जी, मिल जाएगा, पौने पाँच हज़ार फ़ाइनल, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 4750, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हाँ रखते हैं, पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "बोला, काय पाहिजे? हो, ठेवतो आम्ही, आहे। सव्वातीनशे रुपये, कमी नाही होणार। संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ हां, है, वो क्या है दस का फुल box 2010, सिंगल पीस सवा दो सौ", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "han sir, boliye, kya hai na motor nahi hai, khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "तर रेट सव्वा पंधराशे आहे, पण आत्ता स्टॉक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "हैलो? हाँ, आवाज़ आ रही है। हाँ जी, मिल जाएगा। पहले 790 था, अब 1040 हो गया", "answer": {"in_stock": true, "price_inr": 1040, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "हाँ बोलिए सर, हाँ, रेडी स्टॉक है, चार सौ पंद्रह लगा देंगे, वैसे 475 का है", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब संपलीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "यस सर, बोलिए. हाँ है. हमारा रेट 810, MRP तो 920 है. खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 810, "delivery": false, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "एक मिनिट मिनिट हां... बोला. तेरा नग शिल्लक आहेत. रेट शेठ सांगतील", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "kya hai na haan hai apne paas aapko char hajar do sau pachas mrp 5390 hai sorry delivery service nahi hai", "answer": {"in_stock": true, "price_inr": 4250, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "प्लीज़ थोड़ी देर बाद कॉल करो नेटवर्क इशू है आवाज़ ब्रेक हो रही है सर आज बहुत बिज़ी डे है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ बोलिए सर, नहीं है, उसकी जगह Havells 3 socket board ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 3 socket board"}}
{"product": "Orient 400 mm pedestal fan", "said": "पंखा नहीं है, खत्म हो गया, उसकी जगह V-Guard fan दे सकते हैं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-Guard fan"}}
{"product": "Shakti 220A welding machine", "said": "hello? aawaj yetoy ka? haan bola, kay aahe na Shakti sampala, Cheston 200 amp welding machine aahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston 200 amp welding machine"}}
{"product": "Candes 1000W immersion rod", "said": "Nahi hai, Crompton 1500W immersion rod wala hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 1500W immersion rod"}}
{"product": "water pump seal kit", "said": "नहीं है जी, ख़त्म। मैकेनिकल seal वाला है, पौने तीन सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मैकेनिकल seal"}}
{"product": "AmiciVision LED torch", "said": "है ना, कितने चाहिए? 1200 लगेगा, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, लड़का चाय लेने गया है, त्योहार का सीज़न है ना, भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "अं सर एक मिनट, चेक करके बताता हूँ। एक ग्राहक खड़ा है, एक मिनट।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "बघा हो दादा, आहे, साडेपाचशे रुपये होतील", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "haan toh maal khatam hai, Michelin wiper hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin wiper"}}
{"product": "Crompton 3 L geyser", "said": "Bola dada. Crompton nahiye, AO Smith 15 L geyser ahe. Diwali chi gardi ahe na. gharpoch pohochvun deu. aaj sakali light navhti", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "AO Smith 15 L geyser"}}
{"product": "Carbanao car mat for Swift", "said": "स्टॉकमध्ये आहे पावणे चौदाशे फायनल", "answer": {"in_stock": true, "price_inr": 1375, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "hai bilkul hai rate puch ke batata hun bhejne ka intezaam nahi hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, पुट्टी शिल्लक आहे. पावणेनऊशे फिक्स. पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 875, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हो, आहे, पंधराला देतो", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "नाहीये Havells 1.0 HP water pump आहे चार हजार पाचशे पन्नासला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1.0 HP water pump"}}
{"product": "Savsol 20W-40 engine oil", "said": "यस, अवेलेबल है, सवा दो सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "रेट दहा आहे पण आत्ता स्टॉक नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Ji, available hai, do sau tees last rate hai, courier kar denge", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "हो आहे... नाही नाही, तो दुसरा आहे, हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Ho, light ahe, pavane atharashe rupayanna padel, ekonis nag shillak aahet", "answer": {"in_stock": true, "price_inr": 1775, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "machine sampli. bara Cheston 500 watt drill ahe, tyacha price 9400", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston 500 watt drill"}}
{"product": "Activa clutch plate", "said": "प्लेट संपली नाही, आहे दादा.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "हो, बोला दादा, आहे ना, साडेपाचशे होतील, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "अं स्टॉक नहीं है अभी जी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Light padliye, kalva mag", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "अभी नहीं है, परसों तक आ जाएगा. 609 का पड़ेगा. लड़का घर पे दे जाएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "है ना, कितने चाहिए? अभी रेट 40 हो गया, पिछले महीने 33 था। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। आज सुबह से लाइट नहीं है", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "Ho, tyre milel, 3560 cha ahe, tumchyasathi tin hazar.", "answer": {"in_stock": true, "price_inr": 3000, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "ji han rakha hai chhah loge toh ek sao painsath ke hisaab se ek ka ek sau pachasi", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "हम्म. देखिए वो हम नहीं रखते. हैवेल्स 1500 वॉट रॉड दे सकता हूँ, ३५५ रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 1500 वॉट रॉड"}}
{"product": "water pump seal kit", "said": "नमस्ते बोलिए नॉट available सर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो, बल्ब आहे, सहा घेतले तर 155 प्रमाणे, एक पीस पावणेदोनशे, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "मतलब, हैलो? हाँ, आवाज़ आ रही है, ऑयल नहीं है, खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "haa, kaun? acha, bolo, network issue hai, aawaz break ho rahi hai, season time hai na, demand jyada hai, stock men hai sir, best price gyarah so, pick up karna padega, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "है ना, कितने चाहिए? मतलब 8270 लगेगा. ठीक है?", "answer": {"in_stock": true, "price_inr": 8270, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हां रखा है रेट सवा पाँच सौ है", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aahe na mrp 445 ahe pan 350 la lavto", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "stock madhe ahe, 80 la lavto, MRP 88 ahe, delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "हॅलो कोण बोलतंय हां तर हो लाईट आहे रेट 1550 आहे कळवा मग", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "आहे की, मिळेल, दोनशे चाळीसला लावतो, एमआरपी तीनशे पाच आहे, बरं पोरगा घरी सोडून देईल.", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी सर, बताइए. वायर है, मिल जाएगी. आपको पौने इक्कीस सौ, MRP तो 2380 है", "answer": {"in_stock": true, "price_inr": 2075, "delivery": null, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "मिल जाएगा, है अपने पास, इक्कीस सौ पचास लगा देंगे, वैसे दो हज़ार तीन सौ पचास का है", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "एकोणतीस नग शिल्लक आहेत, आज खूप बिझी आहे sir, rate सातशे चाळीस आहे sir, नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय", "answer": {"in_stock": true, "price_inr": 740, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "अं अडीचशे रुपये आहे की मिळेल ठीक आहे", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "Stock madhe aahe, bagha GST dharun 485, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 485, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हेलो बल्ब ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "अं मतलब चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "आहे की, मिळेल, पावणेतेराशे रुपयेहोतील, घरपोच देतो", "answer": {"in_stock": true, "price_inr": 1275, "delivery": true, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "aaj sakali light navhti, machine padliye, 2050 rupayanna padel, porga chaha aanayla gelay", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ भाई, बोलो। हाँ हाँ, है। बीस का पड़ेगा।", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हां बोला साहेब, माल नाहीये, तपारिया 10 mm ड्रिल बिट देऊ शकतो, 120 पडेल, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "तपारिया 10 mm ड्रिल बिट"}}
{"product": "R-tek multimeter", "said": "हाँ हाँ जी सर, बताइए, पता नहीं भाई, बाद में कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हाँ तो है भाई सवा चौदह सौ फ़ाइनल डिलीवरी का तीस एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 1425, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "वो तो कल ही ख़त्म हो गया, कैंडेस 1200W room हीटर चलेगा? उसका पौने ग्यारह सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कैंडेस 1200W room हीटर"}}
{"product": "Bosch screwdriver bit set", "said": "साढ़े तीन सौ में दे देंगे. है भाई. होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "हाँ जी, स्टॉक में पड़ा है, Philips 20W एलईडी बैटन का रेट 245 है, ये वाला 195", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ जी, ट्यूब रखी है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "स्टॉक में है सर, 600 का है, आपके लिए चार सौ सत्तर.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "available hai. matlab free delivery hai. bata dena", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "एक मिनिट हां... बोला. ते कालच संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "hai, bilkul hai, ek hazar rupaye ka hai, ha, stock men 12 piece hai, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Pulsar 150 नहीं है NGK पल्सर 180 स्पार्क प्लग है दुकान से ले जाना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "NGK पल्सर 180 स्पार्क प्लग"}}
{"product": "1 inch PVC elbow", "said": "जी बोलिए है भाई बीस रुपये का है लड़का नहीं है भेजने को आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब आहे की नाही माहीत नाही गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ है home डिलीवरी available है 50 चार्ज 1050 का है प्लस delivery", "answer": {"in_stock": true, "price_inr": 1050, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "अभी अवेलेबल नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Plug hai, mil jaega, ek sao tees rupaye, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 130, "delivery": true, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "Porga chaha aanayla gelay, machine ahe, milel, rate 5600 aahe, yeun gheun ja, bazarat mal kami ahe sadhya", "answer": {"in_stock": true, "price_inr": 5600, "delivery": false, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "स्टॉक में है, मतलब आपके लिए 760, घर तक पहुँचा देंगे, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "JY Super LED torch", "said": "एक मिनिट हां... बोला, आत्ता नाहीये, म्हणजे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "हाँ जी, स्टॉक में पड़ा है. देखिए तीन सौ चालीस, सॉरी सॉरी, तीन सौ पचहत्तर.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "माल खत्म है, Philips LED headlight bulb पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Philips LED headlight bulb"}}
{"product": "12V 7Ah battery", "said": "भाई, बोलो, स्टॉक में है, 950 में देदेंगे", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "सेठ जी बाहर गए हैं, वही बताएँगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "अं म्हणजे उद्या फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Haan sir, boliye, haan, maal pada hai, 180 lagega ab, pehle 195 ka tha.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "माल खत्म है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "pata nahi bhai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "अं स्टॉक में नहीं है, उसकी जगह सुप्रीम 3/4 इंच एल्बो दे सकते हैं, आके ले जाओ.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "सुप्रीम 3/4 इंच एल्बो"}}
{"product": "50 mm padlock", "said": "tala khatm ho gaya dekho harrison 60 mm padlock available hai uska price sava teen sau", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "harrison 60 mm padlock"}}
{"product": "12V 7Ah battery", "said": "battery sampli nahi, aahe, price savvabarashe aahe, delivery deto, 20 charge lagel.", "answer": {"in_stock": true, "price_inr": 1225, "delivery": true, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "अं हो sir available आहे तुम्हाला साडे सातशेला देतो येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "है भाई. MRP 5830 है, आपको 3600 में दे देंगे.", "answer": {"in_stock": true, "price_inr": 3600, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी stock में पड़ा है डब्बा 315 का है छह piece वाला एक का साठ", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "पता नहीं भाई, बाद में कॉल करो, आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "ji available hai sava panch sau de dena mrp to che sau paintees hai aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "नमस्कार बोला चुकीचा नंबर आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "प्राइस पौने नौ सौ है, हाँ है, कोई प्रॉब्लम नहीं, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 875, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "सव्वातेराशे रुपये होतील, मशीन पडलीये", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "अभी स्टॉक नहीं है, कल तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "Panch sau panch rupaye lagenge, ha, cover hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 505, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "पता नहीं, गोदाम में देखना पड़ेगा, आज सुबह से लाइट नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "देखो स्टॉक में है सर, बीस का फुल बॉक्स 2510, सिंगल पीस 140", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ रखा है सौ रुपयेका है अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "हाँ, रेडी स्टॉक है, रेट ग्यारह हज़ार नौ सौ है सर", "answer": {"in_stock": true, "price_inr": 11900, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "हाँ जी, मिल जाएगा, दो सौ पाँच में दे देंगे, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 205, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी है, मिल जाएगी, वो पहले 580 था, अब 660 हो गया.", "answer": {"in_stock": true, "price_inr": 660, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर आहे, मिळेल। छापील 1700, डिस्काउंट करून 1400। आवाज तुटतोय, मोठ्याने बोला। कंपनीने परत रेट वाढवले", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "जी बोलिए. हाँ तो हाँ जी, मिल जाएगा. पैंतीस सौ पचास रुपये", "answer": {"in_stock": true, "price_inr": 3550, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हाँ है डेढ़ सौ लगेगा एंकर एलईडी बैटन चाहिए तो वो भी है एक सौ पच्चीस वाला", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "रेट सात हज़ार आठ सौ है, पर अभी स्टॉक नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "अं stock में है, पौने दो सौ है, 135 नहीं", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "आहे ना एक पीस २५०, ६ घेतले तर २१० प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब संपली नाही आहे MRP एकशे ऐंशी आहे तुम्हाला दीडशेला देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "stock madhe aahe, pick up karava lagel, ajun kahi?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Khadija angle grinder", "said": "सव्वाबाराशे फिक्स फिक्स। मशीन आहे, मिळेल। डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 1225, "delivery": false, "alternative": null}}
{"product": "Agrico TATA plier", "said": "म्हणजे रॉंग नंबर.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "वो क्या है नॉट available सर.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हैलो? हाँ, आवाज़ आ रही है, नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "मिल जाएगा, है अपने पास. अभी रेट साढ़े पाँच सौ हो गया, पिछले महीने चार सौ पैंसठ था. अं", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "sir आज बहुत बिज़ी डे है मैं नया हूँ मुझे नहीं पता एक कस्टमर खड़ा है होल्ड करो सीज़न टाइम है ना डिमांड ज़्यादा है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "पहले पचास का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ, तेरह पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "Liqui Moly संपलं, Mobil 5 वॉट इंजिन ऑइल आहे, चालेल? घरपोच पोहोचवून देऊ.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Mobil 5 वॉट इंजिन ऑइल"}}
{"product": "1 inch PVC elbow", "said": "Haan sir, available hai, pandrah rupaye", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हां आज खूप बिझी आहे sir, बल्ब संपला नाही, आहे, नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय, मागच्या महिन्यात तीनशे सत्तर होता, आता साडे चारशे आहे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "stock में है. मतलब डब्बा 3000 का है, 25 piece वाला, एक का 130.", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "स्टॉक में है, १४० फ़ाइनल, डिलीवरी नहीं है, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "Osram LED headlight", "said": "Mi navin aahe, mala mahit nahi saheb", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "हो ट्यूब आहे घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "यस sir, बोलिए। एक कस्टमर खड़ा है, होल्ड करो। sir आज बहुत बिज़ी डे है। हाँ sir, वो मॉडल available है। अं पौने दो सौ लगा देंगे, वैसे 210 का है। सीज़न टाइम है ना, डिमांड ज़्यादा है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "एक मिनिट हां... बोला, मशीन आहे, मिळेल, आता 2450 आहे, आधी 2910 होता, आज सकाळपासून लाईट नाही, एक गिऱ्हाईक आहे, एक मिनिट, कंपनीने परत रेट वाढवले, अजून काही?", "answer": {"in_stock": true, "price_inr": 2450, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "Tube aahe, milel, 195 la padel, delivery nahi karat, kalva mag", "answer": {"in_stock": true, "price_inr": 195, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट संपली नाही, आहे, पाठवतो, 20 रुपये डिलिव्हरी चार्ज लागेल, मालाचे सातशे चाळीस, डिलिव्हरी वेगळी", "answer": {"in_stock": true, "price_inr": 740, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "वो आइटम हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "है भाई, मिल जाएगा, आठ हज़ार चार सौ है, सात हज़ार एक सौ नहीं, एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 8400, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब संपला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Pulsar 150 sampla, NGK Pulsar 180 spark plug ahe, chaalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "NGK Pulsar 180 spark plug"}}
{"product": "Accessory Mart car mat for Duster", "said": "हाँ सर, वो मॉडल अवेलेबल है, बॉय भेज दूंगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आवाज़ कट रही है, ज़ोर से बोलिए, एक ग्राहक खड़ा है, एक मिनट, है भाई, 910 बोला क्या? माफ़ करना, 1000 है.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "ji sir, tell me, arre Retina wala khatam hai, Tradon bike cover hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Tradon bike cover"}}
{"product": "teflon tape", "said": "Stock madhe ahe, mhanje pannas nagancha box 635 la, sutta ek 15", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Ha sir, wo model available hai, rate painsath hai sir.", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Ji boliye price pachpann hai haan sir, available hai", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Hai, bilkul hai, sadhe teen sau ka hai.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "lagta hai nahi hai haan mil gaya hai assi final price hai", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "dekhna padega bhai, yaad nahi hai ji", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "नाही असं नाही, आहे। तुम्हाला 480 ला देतो।", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "हाँ बोलिए सर, हाँ जी, मिल जाएगा, लड़का घर पे दे जाएगा, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "haan bola, kay pahije? haan haan, aiktoy, aawaj tutatoy, motyane bola, kal Ravivar hota mhanun dukan band hota, ahe na, magchya mahinyat 3200 hota, ata ekonachalisshe ahe", "answer": {"in_stock": true, "price_inr": 3900, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "बघतो आणि कळवतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां, वायर है. पौने सत्रह सौ में दे देंगे. बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ha, mil jaega bhai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा, रेट बदल गया है, अब 60 है, पहले 71 था, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई, मिल जाएगा, एक सौ दस, sorry सॉरी, सौ, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "वो तो कल ही ख़त्म हो गया। हैवेल्स 20A एमसीबी है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 20A एमसीबी"}}
{"product": "MYRA helmet lock", "said": "हाँ जी, स्टॉक में पड़ा है। डब्बा नौ सौ चालीस का है, छह पीस वाला, एक का पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "Helmet pada hai, MRP to panch sau sath hai, apna rate chaar sau nabbe, delivery boy hai, bhej denge.", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "Makita angle grinder", "said": "हो हो, समजलं, पण ते नाहीये.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "हां रखा है कूरियर कर देंगे ठीक है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "हाँ, है, ५४० रुपये का है.", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "Stock khatm ho gaya hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Ek minute ha... haan bola, light sampli, udya sakali maal yeil, aatta nahiye, rate sadesahashe ahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "हैलो? हाँ, आवाज़ आ रही है, साढ़े सात हज़ार लगेगा, हाँ है, कोई प्रॉब्लम नहीं.", "answer": {"in_stock": true, "price_inr": 7500, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "1000 ka aata hai, lekin abhi khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "rate sadhe chaar sau hai, hai bhai, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "जीबोलिए। देखिए नहीं है जी, ख़त्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "बोला ना ती संपली हो Kirloskar 0.5 HP pump आहे आमच्याकडे अं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar 0.5 HP pump"}}
{"product": "Schneider Classy switch", "said": "सर एक मिनट, चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "देखो हाँ, 25 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "नमस्ते, बोलिए, हैलो? आवाज़ नही आ रही, बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हां बोला, तुम्हाला 310 ला देतो, हो, मिळेल, पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "milel na, ahe, pavanecharshe fix", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "out of stock nahi hai, available hai, paune char sau ka milega", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "जीबोलिए, हम्म, अभी नहीं बता सकता, बाद में फ़ोन करो, बाज़ार में माल की कमी चल रही है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "यस, available है, आपको शॉप से pickup करना होगा, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "नॉट अवेलेबल सर. सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "है ना कितने चाहिए 535 में दे देंगे 9 बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 535, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "हेल्मेट संपलं। Vega helmet वाला घ्या, ते चालेल।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega helmet"}}
{"product": "Camel drill machine", "said": "अं स्टॉक में सिर्फ 29 पीस है रेट पौने पंद्रह सौ है", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हो, बोला दादा, आहे ना, म्हणजे, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हां प्लग संपलाय, आत्ता नाहीये, परवापर्यंत येईल, पिकअप करावा लागेल, एक गिऱ्हाईक आहे, एक मिनिट, आज सकाळपासून लाईट नाही, कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Allure Auto car mat", "said": "आहे की, मिळेल। पावणे सतराशेला लावतो, एमआरपी दोन हजार एकशे पन्नास आहे।", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "nahiye bahutek... ho ho, sapadla, ahe, pandhrashe pannas rupees hotil, porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "है भाई प्राइस इनक्रीज़ हो गया पहले 330 था अब 435", "answer": {"in_stock": true, "price_inr": 435, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हेलो हां बोलो भाई खत्म हो गया वो क्या है डीवॉल्ट ग्राइंडर है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डीवॉल्ट ग्राइंडर"}}
{"product": "Activa air filter", "said": "हाँ, 30 पीस पड़े हैं. वो क्या है आपके लिए 300.", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो? आवाज येतोय का? हां, बोला। नाही हो, संपलं ते।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "बघा पंखा संपला नाही आहे ४६७० नाही नाही ५५०० पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "dekho ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "पंखा ख़त्म नहीं हुआ, है, रेट चौबीस सौ पचास है, घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 2450, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "मशीन पडलीये पावणे पंचवीसशे रुपये कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 2475, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "rate पौने दो सौ है sir. मतलब हाँ है, कोई प्रॉब्लम नहीं", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "Hello, bola na. bagha aahe aahe. pavaneekveesshe hotil. kalva mag.", "answer": {"in_stock": true, "price_inr": 2075, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "कल सुबह आ जाएगा, अभी नहीं। आपके लिए एक सौ सत्तर। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "arey kya bataun, subah se bahut rush hai, aaj subah se light nahin hai, available hai, koi problem nahi, awaaz kat rahi hai, zor se bolo, haan toh aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Windek tyre", "said": "अं, हां, रखा है बॉस", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "खत्म हो गया था नहीं एक बचा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "स्टॉक में है, पंद्रह सौ पचास रुपये, मतलब 8 पीस हैं स्टॉक में.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "अच्छा हाँ हाँ है हज़ार पड़ेगा", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँजी, बताइए, Activa नहीं है, Bosch Dio air filter वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Dio air filter"}}
{"product": "50 mm padlock", "said": "जी, उपलब्ध है. एक सौ नब्बे प्लस GST. हाँ तो भेजने का इंतज़ाम नहीं है.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "Waxpol engine oil", "said": "umm haan han, sun raha hun, abhi khatam hai, shaam tak aa jayega, kal Sunday tha toh dukan band thi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "तर रॉंग नंबर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ sir वो मॉडल available है सवा तीन सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "आत्ता गडबडीत आहे, अर्ध्या तासाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ, रेडी स्टॉक है, 80 लगेगा लगेगा, लड़का दे आएगा, 40 रुपये लगेंगे.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Themisto multimeter", "said": "हो आहे सहाशे पासष्ट फिक्स पोरगा पाठवतो साठ रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 665, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, प्लग रखा है, पर piece 130, 20 लो तो 115 लगेगा", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "हो, मिळेल, अं पिकअप करावा लागेल, ठीक आहे?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "पंखा ख़त्म हो गया. Orient 1200 mm सीलिंग fan है, 2940 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient 1200 mm सीलिंग fan"}}
{"product": "wall putty 20 kg", "said": "है ना कितने चाहिए अच्छा बेस्ट प्राइस 880 आपको शॉप से पिकअप करना होगा ओके", "answer": {"in_stock": true, "price_inr": 880, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "जी बोलिए. पता नहीं भाई, बाद में call करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "कव्हर पडलंय, रेट शेठ सांगतील, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "SARK tap", "said": "नळ संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "हैलो, कौन? हाँ, है, पौने पच्चीस सौ में दे देंगे, खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 2475, "delivery": false, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "मतलब पंखा ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "kay aahe na aamchyakade nasta, order karun magvto, manus pathavto ghari", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "हाँ, रखा है, रेट पूछ कर बताता हूँ, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "पता नहीं भाई.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "Han hai, dekho Havells 1500W rod wala 1090 men aata hai, aapka wala 815, dukaan se le jana padega", "answer": {"in_stock": true, "price_inr": 815, "delivery": false, "alternative": null}}
{"product": "AmiciTools water pump", "said": "मोटर आहे, मिळेल, 1111 रुपये, सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 1111, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "हाँ sir, वो मॉडल available है, rate सवा तीन सौ है, courier कर देंगे.", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हो दादा, आहे, हां एकशे नव्वद रुपये.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "सर एक मिनट, चेक करके बताता हूँ, एक कस्टमर खड़ा है, होल्ड करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "अं लाईट संपली नाही आहे दोन हजारला देतो दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "available hai rate do so nabbe hai aapke ghar tak bhijwa denge.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "नाहीये बहुतेक हो हो सापडलं आहे अकराशे पन्नासला देतो हां घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "Ha hai, 1350 ka milega", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Haa, hai apne paas, 80 laga denge, waise 100 ka hai, dekho ladka nahin hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khadija angle grinder", "said": "nahi vatta... thamba, ahe, ek urlay.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "लाइट खत्म हो गई Ridroid headlight वाला चलेगा उसका सवा तेरह सौ है अं घर तक पहुंचा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Ridroid headlight"}}
{"product": "BMB welding machine", "said": "आहे आहे, रेट तीन हजार सहाशे पन्नास आहे.", "answer": {"in_stock": true, "price_inr": 3650, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Haan bola, kay pahije? ho, thevto aamhi, aahe, haan tar shambhar fix, porga ghari aanun deil.", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "एक मिनट हांबोलिए मतलब 7Ah नही है Amaron 12V 9Ah battery है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "Maharaja 500W room heater", "said": "हाँ, है, पौने ग्यारह सौ का है, इससे कम नहीं होगा, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है, ball valve रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ball valve"}}
{"product": "Allure Auto car mat", "said": "माल खत्म है, देखिए Accessory Mart mat अवेलेबल है, चलेगा? भेजने का इंतज़ाम नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Accessory Mart mat"}}
{"product": "Crokrok car mat for Vento", "said": "हाँ, रखा है, आपके लिए पौने बारह सौ, हम्म.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "अं ऑयल ख़त्म नहीं हुआ, है, 385 रुपये.", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "तर ट्यूब पडलीये पावणे दोनशे होतील अं कळवा मग", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "rate savvacharshe aahe sir, ho sir, available aahe, self pickup aahe.", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "hello, haan bola, mhanje sorry, out of stock aahe, white cement aahe aamchyakade", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "bike headlight bulb", "said": "bulb pada hai ek sau pachaanve bola kya galti se bola paune do sau hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "स्टॉकमध्ये आहे, एक पीस 285 ला मिळतो, पॅकेट सव्वाचारशेला, कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "Bola dada, ho dada, ahe, pavanepanchvisshe rupees hotil.", "answer": {"in_stock": true, "price_inr": 2475, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "बोला काय पाहिजे काय सांगू सकाळपासून गर्दी आहे हां हां ऐकतोय आहे की मिळेल तुम्हाला पंधराशेला देतो कंपनीने परत रेट वाढवले संध्याकाळपर्यंत पाठवूनदेतो पोरगा चहा आणायला गेलाय ठीक आहे अं", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "hai, bilkul hai. free delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "बोला ना हां हो मोटर आहे 4700 रुपये पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 4700, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "अं अभी नहीं है, Crompton 2.25 mfd capacitor चलेगा? उसका सौ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 2.25 mfd capacitor"}}
{"product": "Activa clutch plate", "said": "हाँ है Bosch Jupiter clutch plate का रेट 860 है ये वाला सवा छह सौ", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan bhai, bolo, matlab han, wahi wala hai, sham tak ghar pe aa jayega, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "baghto ani kalavto, bara company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ बोलिए स्टॉक में है सर सौ लगेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "एक हां... बोला। आमच्याकडे नसतं, ऑर्डर करून मागवतो। बघा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "nahi hai... ruko ruko, hai, ek pada hai, arre same day delivery kar denge.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "खत्म नहीं हुआ अभी भी है अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "स्टॉक में है सर. हम्म घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "जी बोलिए है ना कितने चाहिए बाज़ार में माल की कमी चल रही है हमारा रेट दो सौ दस MRP तो दो सौ साठ है एक ग्राहक खड़ा है एक मिनट शाम तक घर पहुँच जाएगा कट रही है ज़ोर से बोलिए लड़का चाय लेने गया है और कुछ", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "ho bola, pankha sampla nahi, aahe, rate sawwa akarashe aahe sir, aaj sakali light navhti, delivery deto, ainshi charge lagel, haan haan, aiktoy, ajun kahi?", "answer": {"in_stock": true, "price_inr": 1125, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "बोला, काय पाहिजे? मालक नाहीत, नंतर फोन करा, बाजारात माल कमी आहे सध्या", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "2.5 mfd वाला ख़त्म है Havells 3.15 mfd कैपेसिटर है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 3.15 mfd कैपेसिटर"}}
{"product": "CEAT 3.00-18 tyre", "said": "अं अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "वो क्या है हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "ji, available hai, sawa sau mein ho jayega, upar MRP 155 likha hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "स्टॉक नहीं है अभी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "हाँ sir, available है, पौने पाँच सौ फाइनल price है, सेम डे delivery कर देंगे", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ जी, बताइए, अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "अं हां तर उद्या फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हां बोला, स्टॉकमध्ये आहे, नव्वद रुपये, दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "dekho haan, mil jayega bhai, apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "stock madhe ahe Vormir 200 amp machine cha rate 10410, ha pavanenau hazar la ghari pathavto", "answer": {"in_stock": true, "price_inr": 8750, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "लाईट पडलीये रेट रेट 1550 आहे सर", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हो, मिळेल, बघा, घरपोच देतो, अजून काही?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Aahe aahe, deed hazar rupayanna padel, pathavla asta, pan aaj porga nahi, yeun ghya", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हेलो? हाँ, बोलो भाई. सवा सौ रुपये का है. यस, अवेलेबल है.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "देखिए नहीं है, मंगवा देंगे, दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "एक कस्टमर खड़ा है, होल्ड करो. नेटवर्क इशू है, आवाज़ ब्रेक हो रही है. प्लेट पड़ी है. सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "hai na, kitne chahiye? our price paune chaar sau, MRP paach sau pachanve hai", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "hello, ha bolo acha ghante bhar baad call karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Oro switch", "said": "स्टॉक में है. अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "यस सर, बोलिए, सर आज बहुत बिज़ी डे है, हां, 24 पीस पडे हैं, बेस्ट प्राइस 225, वो, एक कस्टमर खडा है, होल्डकरो, नेटवर्क इशू है, आवाज ब्रेक हो रही है.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "आहे आहे, सातशे पंचवीसला लावतो, एमआरपी 860 आहे.", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "stock में नही है V-GRIP cover वाला है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-GRIP cover"}}
{"product": "Kingsway car cover", "said": "हां बोला साहेब, बघा ओनर नाहीत, नंतर कॉल करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हो हो, बोला, नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय, संपलंय, उद्या दुपारपर्यंत येईल, म्हणजे, आज खूप बिझी आहे सर, अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "पंखा संपला नाही, आहे। साडेपंधराशे रुपये, कमी नाही होणार होणार। डिलिव्हरी? नाही हो, ते नाही करत।", "answer": {"in_stock": true, "price_inr": 1550, "delivery": false, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "available hai, 2680 hai, sorry sorry, ikkis sau", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "ho sir, bola, atta stock nahi, udya yeil, self pickup ahe", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए, हाँ, है, रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "तीस नग शिल्लक आहेत. एक हजार तीनशे पन्नासला पडेल. घरपोच पोहोचवून देऊ.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ha, kaun? acha, bolo, abhi stock nahi hai, kal tak aa jayega, aapko number kisne diya? acha thik hai, aapko shop se pickup karna hoga, ladka chai lene gaya hai, company ne is mahine phir rate badhaya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "संपलं। local glue घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local glue"}}
{"product": "Vheelocityin car cover for Vento", "said": "हाँ जी, स्टॉक में पडा है, देखो पहले एक हज़ार तीन सौ था, अब बारह सौ हो गया, सस्ता हुआ है, आके ले जाओ", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "आमच्याकडे आहे. आमचा rate 292, MRP 370", "answer": {"in_stock": true, "price_inr": 292, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan, kaun? acha, bolo, ji, available hai, rate seth ji batayenge.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "वो क्या है सेठ जी बाहर गए हैं, वही बताएँगे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "संपलं होतं... नाही, एक आहे अजून", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Hello, haan boliye, haan sir, wo model available hai, 1600 final, boy ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "अं पोरगा घरी आणून देईल", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "हाँ जी, मिल जाएगा, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "ट्यूब संपली नाही, आहे. 1390... थांबा, लिस्ट बघतो... सोळाशे. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "स्टॉकमध्ये आहे, 5500 फिक्स, घरपोच देतो.", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "शाम तक माल आ जाएगा, अभी नहीं है, 250 में दे देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "एक मिनट... हाँ बोलिए। हाँ, रखा है। साढ़े छह सौ का पड़ेगा। भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 650, "delivery": false, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "ओनर अभी नहीं हैं, बाद में कॉल कीजिए.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "Ek minut... haan bolo, ha, mil jaega bhai, MRP do sao likha hai, ek sao sath laga denge", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी, बताइए। वो तो कल ही ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "चौंतीस सौ पचास का पड़ेगा, है ना, कितने चाहिए? आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 3450, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हां हो, आहे शिल्लक, २४० होतील.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ है, कोई प्रॉब्लम नहीं, सवा चार सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "plug khatam ho gaya, local spark plug de sakta hun, wo do so ka padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "water pump seal kit", "said": "स्टॉकमध्ये आहे. mechanical seal आहे 360 ला, पण हा 320 ला मिळेल.", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho, aahe aamchyakade.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "यस सर, बोलिए, हाँ सर, अवेलेबल है, रेट साढ़े चार सौ है सर, डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वो हम नहीं रखते उसकी जगह फिनोलेक्स 1 स्क्वेयर एमएम वायर दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिनोलेक्स 1 स्क्वेयर एमएम वायर"}}
{"product": "12V 7Ah battery", "said": "haan hai ji.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "namaste ji, boliye haan ji, helmet available hai acha upar 655 likha hai, aapse 575 lenge", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "हेलो ३८०० फाइनल प्राइस है स्टॉक में है सर", "answer": {"in_stock": true, "price_inr": 3800, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "नाही नाही, आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "आमच्याकडे आहे। एमआरपी एकशे पाच आहे, पण नव्वदला लावतो। घरपोच नाही, तुम्हाला यावं लागेल। म्हणजे", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "हेलो, हाँ बोलो, हाँ, रेडी स्टॉक है, पौने पंद्रह सौ लगा देंगे, वैसे एक हज़ार नौ सौ का है.", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Aahe aahe, mhanje rate panchvis aahe sir, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "abhi khatam hai bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "Haan, wahi wala hai, kya hai na aapse sava teen sau lenge, likha teen sau pachasi hai", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "मशीन संपली नाही, आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हां बोला एकशे पन्नास रुपये होतील हो कुलूप शिल्लक आहे पोरगा घरी सोडून देईल सांगा काय", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हो, मिळेल, बघा आधी एकशे चाळीस होता, आता प्राइस पावणेदोनशे झाला.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "हाँ, रखा है, डिलीवरी की टेंशन मत लो, भेज देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हाँ, है। आपसे पंद्रह सौ लेंगे, लिखा 1740 है।", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ है पौने पाँच सौ रुपये का है तो तो मतलब घर पर भेज देंगे 50 अलग से", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Osram LED headlight", "said": "ek min... haa bolo, light khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "maal sampala, V-GRIP steering cover aahe, tyache pavanedonshe rupees.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-GRIP steering cover"}}
{"product": "Crompton pedestal fan", "said": "हां बोला। म्हणजे हो, आहे। 2850 रुपयांना पडेल। कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 2850, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "सगळं संपलं, V-GRIP cover घ्या, ते चालेल, दुकानातून न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "V-GRIP cover"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "पहले 1050 का था अभी है कि नहीं देखना पड़ेगा अं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "tube padliye, ek piece charashe la milto, packet sahashe sath la, porga ghari sodun deil.", "answer": {"in_stock": true, "price_inr": 660, "delivery": true, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "पंखा संपला नाही, आहे, rate शेठ सांगतील", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "अडीचशे रुपयांनापडेल. तर. हो, कुलूप आहे", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "जी बोलिए है ना कितने चाहिए सौ रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "बघा हो सर, अवेलेबल आहे. तुम्हाला तेराशेला देऊ", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हो आहे शिल्लक सुटा 260 ला 6 नगांचा बॉक्स घेतला तर 1440", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है ना कितने चाहिए डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "यस, अवेलेबल है, रेट पौने ग्यारह सौ है सर, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "जी, उपलब्ध है. २४० रुपीज़ है. शॉप ९ बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "Haa sir, available hai, rate sava teen sau hai sir, self pickup hai, delivery nahi.", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "ji haan, rakha hai, paune do sau last rate hai, delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हो, मिळेल, 30 ला देतो, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "Wo item hum keep nahi karte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "बेसिकली अभी नहीं बता सकता, बाद में फ़ोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी मिल जाएगा बेसिकली एक सौ दस पड़ेगा लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "नहीं है भाई खत्म हो गया अं ball valve वाला अवेलेबल है चलेगा पता भेज दो लड़का दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "ball valve"}}
{"product": "12V 7Ah battery", "said": "namaskar, bola, deed hazar rupaye, kami nahi honar, ho, milel, ek giraik aahe, ek minute, porga chaha aanayla gelay", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो, कोण बोलतंय? नाहीये, मागवून देतो, दोन दिवस लागतील, पोरगा पाठवतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "पक्कड पडलीये। तुम्हाला दीडशेला देतो।", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "अं किससे बात करनी है? यहां कोई दुकान नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "hallo? haan, awaaz aa rahi hai, bolo, hai, bilkul hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "arre hai, bilkul hai, 6 loge toh 260 ke hisaab se, ek ka 320, khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 320, "delivery": false, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "नहीं है जी, ख़त्म। मोतुल 20W ऑयल दे सकता हूँ, सवा उन्नीस सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मोतुल 20W ऑयल"}}
{"product": "teflon tape", "said": "टेप संपली नाही, आहे, पंचवीस घेतले तर 13 प्रमाणे, एक पीस पंधरा, हां", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "यस, available है, price तीन सौ चालीस है, भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "JY Super LED torch", "said": "mil jayega, hai aapke liye chaar sau", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "Hai bhai acha 410 ka hai ladka bhej deta hun.", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "सवा चार सौ में दे देंगे। बाज़ार में माल की कमी चल रही है। आज सुबह से लाइट नहीं है। हाँ, माल पड़ा है। वो पता भेज दो, लड़का दे आएगा। त्योहार का सीज़न है ना।", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "हाँ भाई, बोलो, अं हैलो? आवाज़ नहीं आ रही, बाद में करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "ho, machine milel, Khadija grinder pan aahe, tyache 3950, ani tumhi vicharlela 3200 la.", "answer": {"in_stock": true, "price_inr": 3200, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan toh not available sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "है भाई, मिल जाएगा, बेस्ट प्राइस चौदह सौ, Havells 450 mm fan चाहिए तो वो भी है, 1570 वाला, डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ रखते हैं, पर अभी ख़त्म है, Bosch 10 mm drill bit है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch 10 mm drill bit"}}
{"product": "Havells Cista room heater", "said": "हाँ भाई, बोलो। अं है भाई, मिल जाएगा। साढे पच्चीस सौ रुपये", "answer": {"in_stock": true, "price_inr": 2550, "delivery": null, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "haan, hai apne paas, sava do sau last rate hai, ladka ghar pe chhod dega.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ जी मिल जाएगा आपके लिए सौ कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Abhi stock nahi hai, kal tak aa jayega, 125 final price hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Haan bhai, bolo 190 ka padega plug pada hai bhejne wala koi nahi hai, aapko aana padega kya hai na bata dena.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ है कोई प्रॉब्लम नहीं home डिलीवरी अवेलेबल है पचास चार्ज माल का एक सौ अस्सी डिलीवरी अलग", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "एक मिनट... हाँ बोलिए. बेसिकली वो तो कल ही ख़त्म हो गया. Cheston drill मशीन वाला है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston drill मशीन"}}
{"product": "Xinor Nexo helmet", "said": "हाँ जी स्टॉक में पड़ा है बेसिकली MRP 640 है बट आपको पाँच सौ साठ में दे देंगे", "answer": {"in_stock": true, "price_inr": 560, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरीचा स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "बेसिकली अभी अवेलेबल नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हैलो, कौन? तीन सौ चालीस का पड़ेगा. हाँ है, कोई प्रॉब्लम नहीं. खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "Hai bhai, 130 rupees lagenge, Taparia 6 mm drill bit bhi hai, wo 110 ka", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन पड़ी है, पौने उन्नीस सौ का है, इससे कम नहीं होगा, आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 1875, "delivery": false, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "आहे आहे, येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "सव्वापंधराशे रुपयांना पडेल, हो, आहे शिल्लक", "answer": {"in_stock": true, "price_inr": 1525, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "बोला, काय पाहिजे? ती संपली हो, स्टील बिजागरी आहे आमच्याकडे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टील बिजागरी"}}
{"product": "silicone sealant", "said": "yes, available hai price 330 hai aake le jao.", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो दादा, आहे काका।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ sir, वो मॉडल available है, एक सौ सत्तर price है.", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "सवा सत्रह सौ में दे देंगे हां जी मिल जाएगा डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 1725, "delivery": false, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "अड़तीस सौ पचास में दे देंगे। हेलमेट ख़त्म नहीं हुआ, है।", "answer": {"in_stock": true, "price_inr": 3850, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "han ji, mil jayega do sao rupaye ka hai", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हैलो, कौन? है भाई. रेट साढ़े तीन सौ है. देखिए कूरियर कर देंगे.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ जी सर, बताइए, देखो नॉट अवेलेबल सर.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बरं बॅटरी संपली, अमरॉन 12V 5Ah बॅटरी आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अमरॉन 12V 5Ah बॅटरी"}}
{"product": "Apex Vivid helmet", "said": "नहीं नहीं, है ना। देखो उनतालीस सौ फ़ाइनल", "answer": {"in_stock": true, "price_inr": 3900, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हो हो दादा आहे तर छापील 475 डिस्काउंट करून चारशे", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "cover nahi hai, khatam ho gaya, Cross Fingers wala hai, wo le lo, delivery nahi karte, aake le jao.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Cross Fingers"}}
{"product": "Shakti 250A welding machine", "said": "हां मशीन पडलीये 5900 होतील कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 5900, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ माल पड़ा है एक सौ दस रुपये का है Crompton 2.25 mfd capacitor चाहिए तो वो भी है एक सौ पैंतालीस वाला", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "प्लीज़ थोडी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "Machine ahe, milel. pavanedaha hazar fix. manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 9750, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ, रेडी स्टॉक है. बेस्ट प्राइस 310. होम डिलीवरी अवेलेबल है, 60 चार्ज.", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "स्टॉकमध्ये आहे. रेट पावणे चारशे आहे. येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "Khadija angle grinder", "said": "Haan ahe, kay aahe na self pickup aahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "हां जी मिल जाएगा ढाई सौ रुपये होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "देखो, अभी बिजी हूँ, call बैक करता हूं बॉस", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "है भाई, मिल जाएगा, 10 का डिब्बा 260 का, एक पीस तीस, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हां बोला। बरं आधी एकशे पन्नासला होता, आता आहे की नाही बघावं लागेल।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "नहीं है भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "हां जी बताइए आज सुबह से लाइट नही है लडका चाय लेने गया है साढ़े अड़तीस सौ लगेगा जी उपलब्ध है आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 3850, "delivery": true, "alternative": null}}
{"product": "Moonstruck 2000W immersion rod", "said": "हेलो, हाँ बोलो. स्टॉक में है सर. पौने पाँच सौ में दे देंगे. डिलीवरी हो जाएगी सर. ओके?", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "Helmet hai, mil jayega panch sau pachasi hai, 575 nahi delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 585, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho bola, sheth baher gelet, te aale ki sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "abhi nahi hai, parso tak aa jayega, courier kar denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "hai ji, mil jayega, single piece paune teen sau, chhah ka full box ek hajar chaar sau pachas", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां रेडी स्टॉक है देखो आपके लिए लिए 1950 कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "आपके घर तक भिजवा देंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो? आवाज येतोय का? हां, बोला, टेप पडलीये, 10 रुपये", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "acha haan, stock me hai sade teen sau hai, GST alag delivery boy hai, bhej denge.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Type R G124 horn", "said": "हाँ जी, बताइए। मिल जाएगा, है अपने पास। और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "आहे की मिळेल येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "वो आइटम हम नहीं रखते, फिलिप्स 5W LED बल्ब है, 260 का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 5W LED बल्ब"}}
{"product": "9W LED bulb", "said": "बोला ना, आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "है ना, कितने चाहिए? बारह लोगे तो दो सौ पन्द्रह के हिसाब से, एक का दो सौ पचास, अच्छा, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "BMB welding machine", "said": "हैलो, कौन? हैलो? आवाज़ नहीं आ रही, बाद में करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "नाहीये बहुतेक... हो हो, सापडलं, आहे, पावणेएक्केचाळीसशेला पडेल, अं घरी पाठवतो.", "answer": {"in_stock": true, "price_inr": 4075, "delivery": true, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "स्टॉक में है sir, अपना price साढे तैंतीस सौ, MRP 5420 है.", "answer": {"in_stock": true, "price_inr": 3350, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "अं बल्ब है, मिल जाएगा, रेट डेढ़ सौ है, भेज देंगे, 50 रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो, ऑइल शिल्लक आहे. पाचशे तीस, जीएसटी एक्स्ट्रा.", "answer": {"in_stock": true, "price_inr": 530, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हां, माल पड़ा है. साढ़े चार सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Haan aahe, pathavto, shambhar rupaye delivery charge lagel, barashe pannas la aahe, delivery charge vegla", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "आपके लिए 260। हाँ, माल पड़ा है।", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर पडलीये। आवाज तुटतोय, मोठ्याने बोला। बाजारात माल कमी आहे सध्या। आज सकाळपासून लाईट नाही। गर्दी आहे ना। 1880... थांबा, लिस्ट बघतो... पंधराशे पन्नास", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "गीज़र है, मिल जाएगा। पौने ग्यारह सौ लगा देंगे, वैसे 1380 का है।", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "खुद आकर उठाना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Windek tyre", "said": "Stock madhe ahe, rate naushe panchahattar aahe sir", "answer": {"in_stock": true, "price_inr": 975, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "Han, kaun? acha, bolo, acha dekhna padega bhai, yaad nahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "yes sir, boliye, plate rakhi hai, mil jayegi, rate puch ke batata hun, bhijava dunga, thik hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "यस सर, बोलिए. Speedwav वाला फिनिश है, AUTOSiTY हैंडल ग्रिप अवेलेबल है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AUTOSiTY हैंडल ग्रिप"}}
{"product": "CEAT 3.00-18 tyre", "said": "जी, उपलब्ध है. अच्छा. आपके लिए १५५०. भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "हो, मिळेल, पाठवतो, वीस रुपये डिलिव्हरी चार्ज लागेल, वस्तू सव्वाचारशेला आहे, डिलिव्हरी वेगळी", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "dekho sava chaar sau final hai ji mil jayega", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "kya hai na pehle 319 ka tha, abhi hai ki nahi dekhna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "हाँ, है। पच्चीस लोगे तो एक सौ पैंतालीस के हिसाब से, एक का पौने दो सौ। दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Makita angle grinder", "said": "आहे, अजून दहा पीस पडलेत। पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "जी बोलिए, देखो हाँ, चार पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "हाँ तो मिल जाएगा, है अपने पास, Fluke मल्टीमीटर का रेट 3250 है, ये वाला सत्ताईस सौ, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 2700, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "वो क्या है अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "नॉट अवेलेबल सर. किर्लोस्कर वॉटर pump वाला अवेलेबल है, उसका प्राइस चार हज़ार आठ सौ पचास", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "किर्लोस्कर वॉटर pump"}}
{"product": "Inalsa OFR 2000W room heater", "said": "वो हम नहीं रखते बजाज 2000 वॉट रूम heater वाला है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बजाज 2000 वॉट रूम heater"}}
{"product": "Lakshmi 0.5 HP water pump", "said": "देखो है अरे नहीं वो दूसरा वाला है ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "नमस्ते बोलिए हाँ माल पड़ा है उन्नीस सौ पचास में देदेंगे हां सोलह पीस पडे हैं", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "हां बोला... नाही, ते सध्या नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हैलो कौन नहीं है भाई खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "matlab hai hai... nahi yaar, last piece subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हम्म। हाँ बोलिए सर। डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "मालक नाहीत, नंतर फोन करा भाऊ.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हैलो, कौन? बेसिकली Splendor वाला ख़त्म है, पैशन है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पैशन"}}
{"product": "Windek tyre", "said": "हाँ जी sir, बताइए, है भाई, एक हज़ार दो सौ पचास का है, आपके लिए नौ सौ पचासी, अपना आदमी दे आएगा.", "answer": {"in_stock": true, "price_inr": 985, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हम्म हाँ जी, प्लेट रखी है, 700 रुपये", "answer": {"in_stock": true, "price_inr": 700, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ जी सर, बताइए, बेसिकली हाँ, है, ढाई सौ फाइनल प्राइस है.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए, वो हम नहीं रखते, मतलब Bosch है, सवा सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Ceat Milaze tyre", "said": "अं हो टायर आहे तर सव्वा बाराशे आहे sir", "answer": {"in_stock": true, "price_inr": 1225, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "cover khatam ho gaya, Raaisin wala hai, uska rate 635", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Raaisin"}}
{"product": "4 socket extension board", "said": "हाँ हाँ, है। 410 लगेगा। लड़का भेज देता हूँ।", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "बरं नाही आहे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "एक मिनिट हां... बोला, बरं हो, पक्कड शिल्लक आहे, सव्वा दोनशे रुपये होतील, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Hella horn", "said": "Hella वाला फिनिश है, Favourite Bikerz horn अवेलेबल है. बॉय भेज दूंगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Favourite Bikerz horn"}}
{"product": "Apex Vivid helmet", "said": "ek minute ha... haan bola, savvaekonachalisshe la deto, helmet ahe, milel, pathavaychi soy nahi, theek ahe?", "answer": {"in_stock": true, "price_inr": 3925, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Haan bhai, bolo, abhi nahin, order pe mangva denge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan aahe, 350 rupaye, porga ghari aanun deil, aath vajeparyant mi ithech aahe", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "पाँच डिब्बे अवेलेबल है अभी रेट सवा तीन सौ है सर", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बेसिकली बल्ब है, मिल जाएगा, delivery बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haa, mil jayega bhai, khula ek sau pachas ka, bis ka poora box do hajar char sau pachas mein, kya hai na ladka ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हाँ जी, टेप रखी है, बीस रुपये का है, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "बॉय भेज दूंगा", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "available hai, haan toh MRP 945 hai par 810 laga denge", "answer": {"in_stock": true, "price_inr": 810, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "आके ले जाओ", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai bolo haan wahi wala hai pehle pachattar tha ab ek sau ho gaya delivery boy hai bhej denge kal sunday tha toh dukaan band thi kya hai na arre kya bataun subah se bahut rush hai", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "बोला ना हो पंखा आहे 2150 रुपये कमी नाही होणार पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 2150, "delivery": true, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "hai bhai, aapke liye panch sau sattar, acha delivery ka assi extra", "answer": {"in_stock": true, "price_inr": 570, "delivery": true, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "तो। हाँ जी, बताइए। रुकिए, देखकर बताता हूँ। आज सुबह से लाइट नहीं है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "तर कव्हर आहे, मिळेल, तुम्हाला ९४५ ला देतो, MRP १३०० आहे.", "answer": {"in_stock": true, "price_inr": 945, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "हाँ जी, पंखा रखा है, चौदह सौ पचास लगेगा, हाँ, छह पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "hai bhai. paune do sau hoga. kya hai na delivery ki tension mat lo, bhej denge", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हेलो? हाँ, बोलो भाई. हाँ, रेडी stock है. पंद्रह लगेगा", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "नाही ना, संपलंय, आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ है. रेट साढ़े तीन सौ है सर.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "उद्या फोन करा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हाँ है, कोई प्रॉब्लम नहीं, एक का सवा तीन सौ, दस का डिब्बा दो हज़ार आठ सौ का, अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हाँ हाँ, है. एक का पौने तीन सौ, दस का डिब्बा दो हज़ार चार सौ का", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Moonstruck 2000W immersion rod", "said": "आहे की, मिळेल. पावणेसहाशे आणि 18 टक्के GST वेगळा.", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "बरं हो पक्कड शिल्लक आहे दोनशे पासष्ट रुपये होतील घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 265, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "बेसिकली हां रेडी स्टॉक है दस का फुल बॉक्स २०९० सिंगल पीस दो सौ पचपन", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "अच्छा स्टॉक में नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हां. बरं माल संपला. हॅरिसन 40 एमएम padlock चालणार असेल तर सांगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हॅरिसन 40 एमएम padlock"}}
{"product": "8 mm drill bit", "said": "haan bola, kay pahije? aamchyakade nasta te, company ne parat rate vadhavle, Diwali chi gardi ahe na, kay sangu, sakal pasun gardi ahe, haan haan, aiktoy.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "हैलो, कौन? सॉरी सर, अभी आउट ऑफ स्टॉक है, Pick Ur Needs torch वाला अवेलेबल है, उसका प्राइस दो सौ अस्सी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pick Ur Needs torch"}}
{"product": "Havells 1500W immersion rod", "said": "रेट ७६० है पर अभी स्टॉक नही है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "ट्यूब पड़ी है. फुल पैकेट एक हज़ार छह सौ पचास, लूज़ पीस 1100 का. होम डिलीवरी अवेलेबल है.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हेलो? हाँ, बोलो भाई, है भाई, सौ प्लस GST, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "स्टॉक में है, सौ में दे देंगे, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "Haan hai, teen sau pachattar last rate hai.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ सर, अवेलेबल है। रेट ३५० है सर। लड़का दे आएगा, २० रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "स्टॉकमध्ये आहे, तीस रुपये, कमी नाही होणार, delivery नाही, शॉपवरून pickup करा.", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ख़त्म हो गया था नहीं एक बचा है प्राइस इक्कीस सौ अच्छा", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "वो मॉडल available नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "haan ji mil jayega aapke liye dhai sau bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "बघा रॉंग नंबर आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हाँ हाँ, रेडी stock है, सवा दो सौ लगेगा, दो सौ अस्सी वाला rate पुराना है, वो नहीं.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ हाँ, है, एक सौ पच्चीस लगा देंगे, वैसे MRP 155 लिखा है.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "पहले पंद्रह का था अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Helo, haan bolo sir ek minute, check karke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "hai na, kitne chahiye? 1500 rupees ka hai, umm aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "देखो नहीं है भाई, खत्म हो गया। सर आज बहुत बिज़ी डे है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "हाँ है. दो सौ पचासी लगा देंगे, वैसे 320 का है.", "answer": {"in_stock": true, "price_inr": 285, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Aahe... ek minute, nahi, te kal sampala.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape hai mil jayegi tees lagega ghar tak pahunchaa denge", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "मशीन पड़ी है देखो ५७०० प्लस जीएसटी", "answer": {"in_stock": true, "price_inr": 5700, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "haan toh ha hai, 645 ka hai, isse kam nahin hoga.", "answer": {"in_stock": true, "price_inr": 645, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "हाँ रेडी स्टॉक है साढ़े अठारह सौ लगेगा शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "abhi confirm nahi kar sakta, godown chek karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, Bosch है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Anchor LED batten", "said": "sorry sir, abhi out of stock hai, arre uski jagah Duravolt LED batten le sakte ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duravolt LED batten"}}
{"product": "Activa clutch plate", "said": "नमस्ते, बोलिए। आज सुबह से लाइट नहीं है। हाँ, माल पड़ा है। 400 रुपये लगेंगे। अभी डिलीवरी बंद है। आवाज़ कट रही है, ज़ोर से बोलिए।", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Stanley angle grinder", "said": "haan toh machine khatm ho gayi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "नॉट अवेलेबल सर. Fluke मल्टीमीटर अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fluke मल्टीमीटर"}}
{"product": "MYRA helmet lock", "said": "आत्ता नाहीये परवापर्यंत येईल एकशे पंच्याण्णवला पडेल पोरगा घरी आणून देईल", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "ho ahe, sadeteenshe la deu, chhapil kimmat 560 aahe, gharapoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हाँ, रखा है, साढ़े तीन सौ रुपये लगेंगे, और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "यस, अवेलेबल है, प्राइस इनक्रीज़ हो गया, पहले 1660 था, अब 2200.", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "ho, pankha milel, price 1449 ahe, delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": 1449, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हां जी stock में पडा है डेढ़ सौ लगेगा आके ले जाओ", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Available hai, koi problem nahi, chaunsath... are nahi, sattar.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "hello, haa bolo, matlab ghar pe bhej denge", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब नहीं है, खत्म हो गया। Philips LED headlight bulb है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Philips LED headlight bulb"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Ruko, dekh ke batata hun. aaj subha se light nahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हां बोला साहेब हो ठेवतो आम्ही आहे तुम्हाला सव्वासातशेला देतो एक गिऱ्हाईक आहे एक मिनिट आवाज तुटतोय मोठ्याने बोला कुरिअर करतो दिवाळीची गर्दी आहे ना कळवा मग", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "आठशे पन्नासला येतो, पण सध्या संपलाय.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "नमस्ते, बोलिए, कूरियर कर देंगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "bola dada, ho, pakkad milel, ekshe sattar rupayanna padel, sandhykali pathavun deto, ajun kahi?", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "जी उपलब्ध है १२० रुपये का है", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "rate pauna ikkis sau hai light rakhi hai mil jayegi", "answer": {"in_stock": true, "price_inr": 2075, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हाँ रखा है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "शाम तक माल आ जाएगा, अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Mil jayega, hai, aath sau paanch bola kya? galti se bola, sava aath sau hai, awaaz kat rahi hai, zor se bolo, aapko number kisne diya? acha thik hai, aaj subah se light nahi hai", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ha bolo, ghar pe bhej denge", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "हाँ जी, मिल जाएगा. MRP 500 है, बट आपको 405 में दे देंगे.", "answer": {"in_stock": true, "price_inr": 405, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "अभी स्टॉक फिनिश हो गया. Technolube 400 ml chain lube वाला अवेलेबल है, उसका प्राइस साढ़े चार सौ. भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Technolube 400 ml chain lube"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर ख़त्म नहीं हुई, है, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "geyser nahin hai, khatm ho gaya, acha uski jagah AO Smith 15 L geyser de sakte hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AO Smith 15 L geyser"}}
{"product": "Mufasa 4W LED bulb", "said": "बल्ब नहीं है खत्म हो गया Daybetter एलईडी बल्ब अवेलेबल है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter एलईडी बल्ब"}}
{"product": "Roots horn", "said": "नमस्ते, बोलिए, वो पहले 860 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "haan, hai apne paas. rate puch ke batata hun. sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "आहे की मिळेल एक पीस 280, 12 घेतले तर 255 प्रमाणे लावतो बघा पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, बोला दादा, आत्ता मिळत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हो सर, बोला. अं हो दादा, आहे. एमआरपी ४३५ आहे पण ३५० ला लावतो", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो, ठेवतो आम्ही, आहे। बिल हवं असेल तर GST वेगळा, रेट 1550। येऊन घेऊन जा।", "answer": {"in_stock": true, "price_inr": 1550, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, 4 पीस पड़े हैं, रेट पूछ कर बताता हूँ, होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "haa, wahi wala hai. ek pees do so pachattar ka milta hai, packet char sau pachas ka", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "हाँ हाँ, पंखा है, एक हज़ार दो सौ बोला क्या? माफ़ करना, सवा ग्यारह सौ है", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "आहे ना, पोरगा पाठवतो, ५० रुपये रिक्षाचे, म्हणजे मालाचे १२०, डिलिव्हरी वेगळी.", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हो दादा, आहे। अकराशे पन्नास रुपयेहोतील", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ, पुट्टी है. सवा आठ सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "हां जी स्टॉक में पडा है MRP 810 है बट आपको पाँच सौ चालीस में दे देंगे", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "देखो मैं नया हूँ, मुझे नहीं पता।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "बोला ना, हो, टेप आहे, म्हणजे रेट पंचवीस आहे, डिलिव्हरी नाही, दुकानात येऊन घ्या, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए सर, हाँ सर, अवेलेबल है, पौने चार सौ का पड़ेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "वो आइटम हम नहीं रखते, देखो Havells 1.0 HP pump available है, उसका price बारह सौ पचास.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1.0 HP pump"}}
{"product": "Xtreme MB Windtone horn", "said": "आपके लिए तीन सौ सत्तर. हाँ हाँ, है. ठीक है?", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हां बोला साहेब, आहे ना, तीनशे दहा रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "लड़का चाय लेने गया है, हाँ, रखा है, 25 का... नहीं नहीं, गलत बोला, बीस का है, बाज़ार में माल की कमी चल रही है, अं अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "साढ़े चार सौ में देदेंगे. है है भाई, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "cover aahe, milel, bara 477 rupaye", "answer": {"in_stock": true, "price_inr": 477, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "हेलो? हाँ, बोलो भाई। स्टॉक में नहीं है। Retina कवर वाला है, दो सौ पचपन का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina कवर"}}
{"product": "1 inch PVC elbow", "said": "स्टॉक नहीं है अभी। अं एस्ट्रल सीपीवीसी एल्बो अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एस्ट्रल सीपीवीसी एल्बो"}}
{"product": "teflon tape", "said": "हॅलो, कोण बोलतंय? टेप संपली, म्हणजे big teflon roll चालणार असेल तर सांगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "yes sir, boliye, please thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "एक मिनिट हां... बोला, हां सध्या नाही, AutoClean वायपर घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AutoClean वायपर"}}
{"product": "Type R G124 horn", "said": "नाही असं नाही, आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "Hai na, kitne chahiye? gyarah sau final bhijwa dunga.", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "मिल जाएगा है अपने पास चार हज़ार छह सौ पचास का है आपके लिए चार हज़ार अं आके ले जाओ", "answer": {"in_stock": true, "price_inr": 4000, "delivery": false, "alternative": null}}
{"product": "Orient ABS room heater", "said": "pandrah sau lagega ji haan, rakha hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "stock में है, 1240 final प्राइस है.", "answer": {"in_stock": true, "price_inr": 1240, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी आत्ता नाहीये, संपली। Amaron 12V 9Ah बॅटरी आहे, त्याचे एक हजार पन्नास रुपये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 9Ah बॅटरी"}}
{"product": "Goldcave tap", "said": "जी बोलिए, जी, उपलब्ध है, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "यस सर, बोलिए। Steelbird वाला ख़त्म है, Sage Square हेलमेट है, चलेगा? लड़का भेज देता हूँ", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Sage Square हेलमेट"}}
{"product": "10W-30 engine oil 1 L", "said": "han sir, boliye, han, wahi wala hai, sadhe teen sau ka hai, isse kam nahi hoga", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "stock में है सर, वो क्या है पर piece एक सौ नब्बे, दस लो तो 170 लगेगा, वो, सॉरी, delivery सर्विस नहीं है", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "Ho, bulb milel MRP 180 aahe but 140 la deu.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "लड़का नहीं है भेजने को, आकर ले जाइए।", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "अं. हॅलो, कोण बोलतंय? रॉंग नंबर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "हां है, कोई प्रॉब्लम नही. आपके लिए ६४६०. खुद आ कर उठाना पडेगा", "answer": {"in_stock": true, "price_inr": 6460, "delivery": false, "alternative": null}}
{"product": "AmiciTools water pump", "said": "Han, motor hai, umm paune baarah sau ka hai.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "मिल जाएगा, है अपने पास, वो वो उनतीस सौ है, तीन हज़ार एक सौ नहीं, आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 2900, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "हाँ, रखा है. पैकेट साढ़े पंद्रह सौ का है, खुला एक एक हज़ार पचास का. पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan toh haan, ready stock hai 280 final bata dena", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "मतलब हां जी स्टॉक में पडा है २५२० लिखा है डिस्काउंट करके १९५०", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "Starvin नहीं है, उषा 400 एमएम पेडस्टल fan है, अच्छा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "उषा 400 एमएम पेडस्टल fan"}}
{"product": "Havells 1.0 HP water pump", "said": "एकोणीसच पीस उरलेत। रेटशेठ सांगतील। पिकअप करावा लागेल।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "वो क्या है सॉरी sir, अभी आउट ऑफ stock है, Accessory Mart car mat है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessory Mart car mat"}}
{"product": "9W LED bulb", "said": "सगळं संपलं, आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट ख़त्म हो गई, लोकल क्लच plate चलेगा? उसका सवा चार सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल क्लच plate"}}
{"product": "16A MCB", "said": "हाँ जी, बताइए. हम्म रेट २१० है, पर अभी स्टॉक नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हाँ सर, वो मॉडल अवेलेबल है, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ माल पड़ा है देखिए एक पीस दो सौ अस्सी का मिलता है पैकेट साढ़े चार सौ का", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, टेप आहे, दहा होतील, घरी पाठवतो, 9 वाजेपर्यंत मी इथेच आहे.", "answer": {"in_stock": true, "price_inr": 10, "delivery": true, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "oil aahe, milel, tumhala pavane charshe la deto.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "arre stock khatam ho gaya hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "हां आत्ता मिळत नाही, विप्रो 15W बल्ब आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "विप्रो 15W बल्ब"}}
{"product": "HTC multimeter", "said": "हम्म आपको आकर लेना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "ho sir, bola. Ravirandal nahiye, Dow sealant ahe. boy ghari deliver karel", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Dow sealant"}}
{"product": "MYRA helmet lock", "said": "अं नहीं मिलेगा भाई सर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हो ट्यूब शिल्लक आहे तुम्हाला साडेचारशेला देतो घरपोच नाही तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हां माल पडा है देखो आपके लिए तीस delivery बंद है अभी दुकान से लेलो", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "आहे आहे, एक हजार एकशे पन्नास रुपये, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "दिवाळीची गर्दी आहे ना। गिझर संपला। एओ स्मिथ 15 L गिझर आहे, अठ्ठावीसशेला। बाजारात माल कमी आहे सध्या। कंपनीने परत रेट वाढवले। हां हां, ऐकतोय। सांगा काय करायचं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एओ स्मिथ 15 L गिझर"}}
{"product": "Modx steering cover for i10", "said": "हां बोला साहेब. हां हां. बरं कव्हर संपलं नाही, आहे. पाठवायला माणूस नाही, इथे येऊन घ्या. कळवा मग.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "नमस्कार, बोला. हो, ट्यूब आहे. 1590 फिक्स.", "answer": {"in_stock": true, "price_inr": 1590, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "है है ना, कितने चाहिए? रेट 25 है, अच्छा डिलीवरी की टेंशन मत लो, भेज देंगे.", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "stock madhe aahe. bara pach hajar naushe la deto. Vormir 200 amp welding machine pahije asel tar te pan ahe, 6610", "answer": {"in_stock": true, "price_inr": 5900, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, प्लेट है, पाँच सौ चालीस लगेगा, डिलीवरी हो जाएगी, 50 रुपीज़ चार्ज एक्स्ट्रा.", "answer": {"in_stock": true, "price_inr": 540, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "tape aahe, milel tumhala vis la deu", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "हॅलो, कोण बोलतंय? कव्हर पडलंय, घरपोच देतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब पडलाय। तुम्हाला सत्तर, छापील किंमत सत्त्याऐंशी। बघा घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "Haan toh haan sir, wo model available hai, abhi rate 4050 ho gaya, pichle mahine 3640 tha", "answer": {"in_stock": true, "price_inr": 4050, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "मिल जाएगा, है अपने पास, ग्यारह सौ पचास लगेगा, तो, ठीक है?", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ek minute ha haan bola battery padliye rate badalla aata sawwa pandhrashe aadhi 1820 hota dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 1525, "delivery": false, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "हो sir बोला हो sir available आहे बघा rate पावणे दोनशे आहे sir", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हां जी। हाँ हाँ जीसर, बताइए। स्टॉक में है सर। सवा चार सौ, जीएसटी एक्स्ट्रा।", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "नमस्ते, बोलिए, हाँ, रेडी स्टॉक है, मतलब सवा दो सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "बल्ब संपला नाही आहे हां सुटा सव्वा तीनशेला सहा नगांचा बॉक्स घेतला तर एक हजार आठशे", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Nahi hai ji, khatam, kya hai na Bosch 10 mm drill bit chalega? sava sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch 10 mm drill bit"}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए सर, हाँ हाँ, है, रेट चार सौ बीस है सर, बताओ क्या करना है.", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "main naya hun, mujhe nahi pata, seth se puchho.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "Aegon संपलं बॉश ग्राइंडर आहे चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश ग्राइंडर"}}
{"product": "Philips 5W LED bulb", "said": "है ना, कितने चाहिए? एक पीस सवा सौ, 10 लोगे तो एक सौ दस के हिसाब से", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "है भाई मिल जाएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हां बोला साहेब, आत्ता स्टॉक नाही, उद्या येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Haa han, sun raha hun. haan, ikkis piece pade hain. ladka chai lene gaya hai. bhaiya aap ruko do minute, phone pe hun. rate pone panch sau hai. are kya bataun, subah se bahut rush hai. bata dena.", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "ताला है मिल जाएगा शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "डिलीवरी हो जाएगी सर", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अच्छा, हैलो? हाँ, आवाज़ आ रही है, देखो मिल जाएगा, है अपने पास, एक सौ बीस का पड़ेगा", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ भाई बोलो बल्ब ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "हाँ बोलिए, स्टॉक में है, लिस्ट price 770, आपको 710 पड़ेगा", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, 3 इंच ब्रश है, चलेगा? अपना आदमी दे आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "3 इंच ब्रश"}}
{"product": "CEAT 3.00-18 tyre", "said": "जीबोलिए, हम्म, आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ जी मिल जाएगा डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Ek minute ha... haan bola aahe ki, milel aata 210 aahe, aadhi 240 hota bara delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "हाँ है, बेस्ट प्राइस चार सौ नब्बे, फ्री डिलीवरी है, और कुछ?", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हॅलो, म्हणजे स्टॉकमध्ये आहे, 260... नाही नाही, 310", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "यस sir, बोलिए, मिल जाएगा, है अपने पास, एक कस्टमर खड़ा है, होल्ड करो, देखिए नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, delivery फ़्री है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Khatm ho gaya dekho Astral 3/4 inch elbow hai, uska rate pachis", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Astral 3/4 inch elbow"}}
{"product": "Anchor LED batten", "said": "अं हो ठेवतो आम्ही आहे डिलिव्हरीची काळजी करू नका पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ तो हाँ हाँ है सवा सौ का पड़ेगा घर पर भेजदेंगे पचास अलग से", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "ख़त्म हो गया था नहीं एक बचा है मतलब बेस्ट प्राइस 600 शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 600, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, प्लग रखा है. दस का डिब्बा एक हज़ार चार सौ पचास का, एक पीस एक सौ सत्तर", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ, बैटरी है, आपके लिए साढ़े चौदह सौ, भेजने का इंतज़ाम नहीं है, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1450, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ जी, मिल जाएगा. मतलब तीन सौ साठ रुपये. भेजने में कोई दिक्कत नहीं.", "answer": {"in_stock": true, "price_inr": 360, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हाँ, माल पड़ा है. पच्चीस का पड़ेगा. आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "स्टॉक में नहीं है Bosch 6 mm drill bit वाला है साठ का अं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch 6 mm drill bit"}}
{"product": "Duravolt LED batten", "said": "त्योहार का सीज़न है ना, अभी नहीं है, एंकर बैटन है, 180 का, एक ग्राहक खड़ा है, एक मिनट, आज सुबह से लाइट नहीं है, अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एंकर बैटन"}}
{"product": "Cross Fingers bike cover for Honda", "said": "हाँ है पहले 480 था अब 435 हो गया सस्ता हुआ है sorry delivery सर्विस नहीं है तो", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Aahe na, 440... thamba, list baghto... sadepachshe, porga ghari aanun deil", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑयल ख़त्म हो गया। कैस्ट्रॉल 10W-30 इंजन oil 900 एमएल है, तीन सौ का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कैस्ट्रॉल 10W-30 इंजन oil 900 एमएल"}}
{"product": "Bosch horn", "said": "नहीं है जी ख़त्म आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्ते बोलिए नही है जी खत्म उसकी जगह Ritco वाला दे सकते हैं डिलीवरी हो जाएगी सर", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Ritco"}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "ख़त्म हो गया था नहीं एक बचा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "नहीं है भाई खत्म हो गया उसकी जगह big टेफ्लॉन रोल वाला ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big टेफ्लॉन रोल"}}
{"product": "8 mm drill bit", "said": "जी, उपलब्ध है, डेढ़ सौ में दे देंगे.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "जी बोलिए. हाँ, ऑयल है. तेरह सौ पचास में दे देंगे. आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "haan, hai apne paas, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "ho, oil aahe, pavane paachashe rupayanna padel, delivery nahi, dukanat yeun ghya, theek aahe?", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "हो सर, बोला, शेठ बाहेर गेलेत, ते आले की सांगतो, नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हाँ बोलिए सर, बेसिकली हाँ सर, अवेलेबल है, पौने सोलह सौ है, GST अलग से", "answer": {"in_stock": true, "price_inr": 1575, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "बरं प्लेट पडलीये रेट 450 आहे", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "ख़त्म हो गया था... नहीं, एक बचा है. दुकान से ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हो, मिळेल, हां संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ बोलिए। वो हम नहीं रखते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "एक मिनिट हां... बोला, म्हणजे मोटर संपली.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "स्टॉक में है सर। देखो पर पीस एक सौ अस्सी, छह लो तो एक सौ साठ लगेगा", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "यस सर बोलिए अरे क्या बताऊँ सुबह से बहुत भीड़ है लड़का चाय लेने गया है मशीन का स्टॉक चेक करके बताता हूँ ओके", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "एक मिनट... हां बोलिए, स्टॉक में है, रेट सवा ग्यारह सौ है", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉक में है, 65 लिखा है, छूट देकर साठ, अभी डिलीवरी बंद है.", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate khatam ho gayi. Bosch chalega? chhah sau sattar ka hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "wall putty 20 kg", "said": "हाँ भाई, बोलो। हाँ जी, पुट्टी रखी है। साढे सात सौ लगेगा। लड़का घर पे दे जाएगा। वो", "answer": {"in_stock": true, "price_inr": 750, "delivery": true, "alternative": null}}
{"product": "Khadija angle grinder", "said": "हैलो? हाँ, आवाज़ आ रही है, सात पीस हैं स्टॉक में, वो डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "मतलब नहीं मिलेगा भाई. आज सुबह से लाइट नहीं है. Retina कवर है, सवा चार सौ का. लड़काचाय लेने गया है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina कवर"}}
{"product": "12V 7Ah battery", "said": "नाहीये बहुतेक... हो हो, सापडलं, आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "220 final, haan, stock mein hai", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ती संपली हो. फिनोलेक्स 1 sq एमएम वायर चालणार असेल तर सांगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिनोलेक्स 1 sq एमएम वायर"}}
{"product": "fan capacitor 2.5 mfd", "said": "हां हां माल पडा है सौ का है इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "है भाई पौने चार सौ फ़ाइनल घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Havells LED torch", "said": "हाँ, रखा है, हम्म डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "Motor ahe, milel kay aahe na 2100 rupayanna padel pick up karava lagel", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पडलीये GST धरून आठशे", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "ha mil jayega bhai price increase ho gaya pehle ek hajar teen sau pachas tha now paune satrah sau", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब संपली नाही आहे तुम्हाला एकशे सत्तरला देतो MRP दोनशे आहे येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 170, "delivery": false, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "हो कव्हर शिल्लक आहे ४३५ रुपये होतील कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 435, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ sir, available है. अं सत्तर रुपीज़ है", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हाँ भाई बोलो टायर ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "Hello, bola na, cover padlay, rate badalla, ata pavanedonshe, aadhi 145 hota, gharpoch milel", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "Namaste ji, boliye, nahi bhai, yaha nahin milegaa.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "हो, पंखा आहे, हजार होतील.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "है ना, कितने चाहिए? 6080... नहीं नहीं, 5183 है", "answer": {"in_stock": true, "price_inr": 5183, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "अं, नही मिलेगा भाई, एम-सील है, 310 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एम-सील"}}
{"product": "Orient Apex-FX 1200 mm ceiling fan", "said": "एक मिनट... हाँ हाँ, पंखा नहीं है, खत्म हो गया, हैवेल्स 1200 एमएम सीलिंग fan है, 1340 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 1200 एमएम सीलिंग fan"}}
{"product": "1 inch PVC elbow", "said": "आहे, अजून 13 पीस पडलेत.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "paune do sau lagega, han, hai apne paas, delivery ho jayegi, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Haan tar Bajaj nahiye, Racold 3 L geyser ahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Racold 3 L geyser"}}
{"product": "9W LED bulb", "said": "हाँ भाई, बोलो. हाँ, माल पड़ा है. सवा सौ रुपये. हम्म. होम डिलीवरी हो जाएगी.", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "hello, haan bolo, haan sir, available hai, 10 lagega, courier kar denge sir.", "answer": {"in_stock": true, "price_inr": 10, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ जी, स्टॉक में पड़ा है. मतलब एक का एक सौ, बारह का डिब्बा 985 का", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो सर, अवेलेबल आहे, तुम्हाला 310 ला देऊ, पाठवतो, 20 रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ सर, वो मॉडल अवेलेबल है, 12 mm bolt का रेट 125 है, ये वाला 120", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "हाँ सर वो मॉडल अवेलेबल है डिलीवरी की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "म्हणजे आहे ना, आमचा rate दीडशे, MRP एकशे पंचाहत्तर, येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "geyser rakha hai, mil jayega, rate pauna gyaarah sau hai, bhej denge, ek sau rupees delivery charge lagega", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "नहीं है बोलो तो मंगवा देता हूँ दो दिन लगेंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "pehle 1050 ka tha, abhi hai ki nahi dekhna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हो, ट्यूब आहे, पॅकेट पावणेसातशेला आहे, सुटा एक 425 ला, घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 675, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "Hai ji, mil jayega, aapke liye panch sau bees, ghar pe bhej dunga, 40 lagega bhejne ka.", "answer": {"in_stock": true, "price_inr": 520, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "माणूस पाठवून देतो घरी। अजून काही?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, Duro Gold grinder पड़ा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duro Gold grinder"}}
{"product": "Accessory Mart car mat for Duster", "said": "stock नहीं है अभी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "ho bola, stock madhe ahe, bagha charshe vees la deu, chhapil kimmat satashe pannas ahe, pick up karava lagel.", "answer": {"in_stock": true, "price_inr": 420, "delivery": false, "alternative": null}}
{"product": "Khadija angle grinder", "said": "हां बोला, मशीन पडलीये, अकराशे पन्नास रुपये, घरपोच पोहोचवून देऊ.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan toh 7ah nahi hai amaron 12v 5ah battery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "amaron 12v 5ah battery"}}
{"product": "9W LED bulb", "said": "haan ji, bulb available hai, ek sau bees lagega", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "namaskar, bola, aaj sakali light navhti, kal Ravivar hota mhanun dukan band hota, milel na, aahe, 380 cha aahe, tumchyasathi 300, kay sangu, sakal pasun gardi ahe, haan haan, aiktoy, yeun nyava lagel, kalva mag", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, haan bola, chukicha number aahe.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "hello, bola na. ho, machine ahe. mhanje pavanenau hazar rupaye, kami nahi honar", "answer": {"in_stock": true, "price_inr": 8750, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "नाहीये, मागवून देतो, दोन दिवस लागतील. बघा घरपोच पोहोचवून देऊ.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "RNG Eko tyre", "said": "स्टॉक में है सर। आपके लिए पौने अट्ठाईस सौ। शॉप सात बजे तक ओपन है।", "answer": {"in_stock": true, "price_inr": 2775, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "अच्छा जी, उपलब्ध है, 12 लोगे तो 235 के हिसाब से, एक का 249", "answer": {"in_stock": true, "price_inr": 249, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "namaste ji, boliye, cover khatm ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ माल पड़ा है आपको shop से pickup करना होगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "स्टॉक में है, सवा चौदह सौ में दे देंगे, वो क्या है और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 1425, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हां बोला... नाही, ते सध्या नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "बघा, Digismart संपलं, Havells 450 mm fan वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 450 mm fan"}}
{"product": "2 inch paint brush", "said": "नमस्ते, बोलिए। हाँ, है। आपके लिए 45। होम डिलीवरी होजाएगी। बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 45, "delivery": true, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "हो, आहे शिल्लक, रेट पावणे आठशे आहे, संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "स्टॉक में है, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "maal khatm hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "एक मिनट... हाँ बोलिए। अच्छा शाम तक माल आ जाएगा, अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "आहे की, मिळेल. अडीचशेला देतो. संध्याकाळी घरी पोहोचेल.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ है, कोई प्रॉब्लम नहीं, बेस्ट प्राइस साढ़े तेरह सौ", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "हेलो? हाँ, बोलो भाई. सर आज बहुत बिज़ी डे है. एक कस्टमर खड़ा है, होल्ड करो. ख़त्म हो गया. मिशलिन टायर है, सवा अठारह सौ का. सीज़न टाइम है ना, डिमांड ज़्यादा है. बताओ क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मिशलिन टायर"}}
{"product": "Activa clutch plate", "said": "750 का है, इससे कम नहीं होगा। स्टॉक में है", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल आहे मिळेल सहाशे साठ नाही नाही पाचशे चाळीस", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "नमस्ते, बोलिए. माल खत्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "हॅलो? आवाज येतोय का? हां, बोला। हो, मशीन आहे। पावणेतीन हजार फिक्स। माणूस पाठवून देतो घरी।", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो हो, वायर शिल्लक आहे। एक हजार आठशे पन्नास आणि १८ टक्के GST वेगळा", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "देखिए नहीं है भाई, खत्म हो गया। बॉल valve है, 300 का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉल valve"}}
{"product": "Savsol 20W-40 engine oil", "said": "namaskar, bola, oil sampala, Castrol 10 watt engine oil aahe aamchyakade.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Castrol 10 watt engine oil"}}
{"product": "4 inch door hinge", "said": "एक मिनिट हां... बोला. आत्ता सांगता येणार नाही, संध्याकाळी फोन करा. हां हां, ऐकतोय", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "म्हणजे संपलं होतं नाही एक आहे अजून", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan tar tape sampliye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "Wuerth वाला ख़त्म है शेल 10 वॉट ऑयल है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शेल 10 वॉट ऑयल"}}
{"product": "4 socket extension board", "said": "तीन सौ तीस रुपये. जी, उपलब्ध है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हो ठेवतो आम्ही आहे एका नगाचे 290, बारा नगांचा बॉक्स 3040 ला", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "Bulb aahe, milel, bagha ek piece donshe tees la milto, packet tinashe panchyannav la", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ, रेडी स्टॉक है, चालीस लगेगा, हाँ, 26 पीस पड़े हैं.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "हॅलो हां बोला हो आहे शिल्लक साडे पाचशे फायनल बॉय पाठवतो सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ जी, मिल जाएगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "नहीं है भाई, खत्म हो गया, देखो ओसराम एलईडी हेडलाइट बल्ब अवेलेबल है, चलेगा? अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "ओसराम एलईडी हेडलाइट बल्ब"}}
{"product": "Studds Urban helmet", "said": "715 रुपयांना पडेल. हां हेल्मेट आहे, मिळेल", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "हो दादा, आहे, तुम्हाला 205 ला देतो, आठ वाजता दुकान बंद होतं.", "answer": {"in_stock": true, "price_inr": 205, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "बोला ना, ऑइल संपलं नाही, आहे, दोनशे नव्वद पडेल, 230चा रेट जुना आहे, तो नाही, डिलिव्हरी फ्री आहे, दिवाळीची गर्दी आहे ना, आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Hello? aawaj yetoy ka? haan bola aamhi te thevat nahi Bosch Dio air filter aahe aamchyakade.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Dio air filter"}}
{"product": "tank float valve", "said": "ho aahe, savvateenshe rupaye, mhanje ghari pathavto, pannas vegle.", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Fluke multimeter", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "स्टॉक में है सर। अब चौबीस सौ पचास है, लास्ट मंथ दो हज़ार था। आके ले जाओ", "answer": {"in_stock": true, "price_inr": 2450, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "आहे आहे, पावणे तीनशे पडेल, दोनशे पंधराचा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "वो रॉन्ग नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "650 hotil. tube padliye.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "अभी stock फिनिश हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "हाँ है कोई प्रॉब्लम नहीं 410 बोला क्या माफ़ करना पाँच सौ है घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "अं देखिए घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ, रखा है, हम्म पैकेट ३९० का है, खुला एक २५० का.", "answer": {"in_stock": true, "price_inr": 390, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "Hai ji, mil jaega, rate 1150 hai, haan toh delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "हाँ जी सर, बताइए। नहीं मिलेगा भाई। Galaxy car cover पड़ा है, वो ले लो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Galaxy car cover"}}
{"product": "ESN 2000W immersion rod", "said": "haan hai, rate paune chaar sau hai.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Ek minut... ha bolo, matlab maal khatam hai, chahiye toh Lakshmi pump de dunga.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Lakshmi pump"}}
{"product": "4 socket extension board", "said": "नमस्ते, बोलिए. वो तो कल ही ख़त्म हो गया. Havells 3 socket board है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 3 socket board"}}
{"product": "Metro Moulding helmet lock", "said": "बोला ना, आमच्याकडे आहे, एमआरपी दोनशे पाच आहे, पण एकशे साठला लावतो.", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Mufasa 4W LED bulb", "said": "अं गलत नंबर लगाया है आपने", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "हाँ जी, बताइए। Anchor वाला ख़त्म है, Duravolt एलईडी बैटन है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duravolt एलईडी बैटन"}}
{"product": "silicone sealant", "said": "देखो है ना, कितने चाहिए? बेस्टप्राइस दो सौ नब्बे, हाँ, नौ पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "पहले २८०० का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan toh ji haa, rakha hai, packet 120 ka hai, khula ek 14 ka", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "प्लीज़ थोड़ी देर बाद कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "सध्या आउट ऑफ स्टॉक आहे नवीन स्टॉक परवा येणार आहे डिलिव्हरी देतो सर नेटवर्क प्रॉब्लेम आहे आवाज कट होतोय म्हणजे आज खूप बिझी आहे सर", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ सर, वो मॉडल अवेलेबल है, ढाई सौ लगेगा, डिलीवरी बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "haan, ready stock hai, panch sau tees last rate hai, khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 530, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Hai, bilkul hai, 340 lagega, haan toh bhej deta, par aj ladka nahin aaya, aake le jao", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "हो ठेवतो आम्ही आहे रेट पावणेदोनशे आहे सर येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "hai bhai rate puch ke batata hun home delivery nahi hai bhai bolo kya karna hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "SARK tap", "said": "hello haan boliye ha ji nal available hai upar saat sau pachpan likha hai aapse sadhe chhah sau lenge free delivery hai", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "आपने गलत नंबर मिलाया है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हां स्टॉकमध्ये आहे पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "Tube khatam nahi hui, hai, list price do hajar do sau, discount ke baad sava atharah sau.", "answer": {"in_stock": true, "price_inr": 1825, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "kya hai na abhi stock finish ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "देखो. है ना, कितने चाहिए? मतलब GST मिलाकर १०००", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Haan ahe, tumhala 70 la deto, delivery deto.", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हो, बोला दादा। पाठवायची सोय नाही।", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "Haan han, bolo, ha, maal pada hai, savaa gyaarah sau mein de dunga.", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "aahe ahe Allure Auto car mat pan ahe, tyache aek hazar, ani tumhi vicharlela savvaterashe la", "answer": {"in_stock": true, "price_inr": 1325, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "हाँ जी, कवर रखा है, एमआरपी 2380 है, डिस्काउंट के बाद साढ़े तेरह सौ पड़ेगा.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, सहा पीस आहेत। रेट शेठ सांगतील। येऊन न्यावं लागेल। अजून काही?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "pata nahi, godown mein dekhna padega bhai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "मतलब अभी नहीं है, ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "hai bhai 2000 mein de dunga delivery nahi bhai wo nahi hota", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "बावीस नग शिल्लक आहेत। तुम्हाला पावणे अठ्ठावीसशेला देतो। तर पाठवायला माणूस नाही, इथे येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 2775, "delivery": false, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "Namaskar, bola bagha owner nahit, nantar call kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "जी बोलिए अभी 2 पीस ही बचे हैं रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "hello? haan, awaaz aa rahi hai, bolo. nahi milega bhai. uski jagah Crompton 3 L geyser de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 3 L geyser"}}
{"product": "Viral Shopping steering cover for Sail", "said": "आम्ही ते ठेवत नाही Modx आहे त्याचे दोनशे पंधरा रुपये हां", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Modx"}}
{"product": "Tata Agrico plier", "said": "250 ला देतो. हो, मिळेल. पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "nahiye, magvun deto, don divas lagtil, home delivery aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan sir, wo model available hai, MRP toh tin sau pachis hai, apna rate ek sau assi", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "मशीन आहे का बघावं लागेल नंतर सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो सर अवेलेबल आहे अं 50 आहे 41 नाही", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ है कोई प्रॉब्लम नहीं दो सौ पचासी नहीं नहीं ढाई सौ है", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "matlab check karke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "नाहीये बहुतेक हो हो सापडलं आहे rate चारशे पन्नास आहे sir", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "नमस्ते, बोलिए, घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aahe, ajun 2 bag padlet, pavaneaathshe rupaye, delivery deto", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "humare paas nahi hota mangwana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "Maal sampla, mhanje Aryshaa LED torch wala aahe, tyacha price 175", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aryshaa LED torch"}}
{"product": "Modx steering cover for i10", "said": "abhi nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "ho bola, stock madhe aahe, 1300 fix, kay aahe na pathavayla manus nahi, ithe yeun ghya.", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "है है ना कितने चाहिए डिलीवरी होजाएगी 100 रुपीज़ चार्ज एक्स्ट्रा आइटम प्राइस 200 डिलीवरी चार्ज अलग", "answer": {"in_stock": true, "price_inr": 200, "delivery": true, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "बेसिकली वो आइटम हम नहीं रखते Allure Auto car mat है 1299 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Allure Auto car mat"}}
{"product": "CSI 1 L geyser", "said": "ho, thevto aamhi, aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर नहीं है, खत्म हो गई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "Goldcave संपलं, SARK tap वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "SARK tap"}}
{"product": "Camel drill machine", "said": "बघा हो, मशीन शिल्लक आहे, एक हजार सहाशे पन्नास... थांबा, लिस्ट बघतो... चौदाशे पन्नास", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Putty sampli nahi, aahe, mhanje hazar rupaye, porga ghari aanun deil", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Schneider Classic switch", "said": "nahi ho, sampla te.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Dekho oil nahi hai, khatam ho gaya, bhaiya aap ruko do minat, phone pe hun", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "हम्म नहीं मिलेगा भाई। क्रॉम्पटन fan दे सकता हूँ, 1420 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "क्रॉम्पटन fan"}}
{"product": "JCB plier", "said": "Han bolo... nahin, wo nahin hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "haan bolo. paune teen sau lagega. han, stock mein hai", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "नमस्ते, बोलिए, नहीं है, बताइए क्या करना है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ भाई, बोलो, आज नहीं है, कल आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "मशीन आहे मिळेल 6700चं आहे तुमच्यासाठी साडे पाच हजार कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery sampliye, Amaron 12V 5Ah battery ahe, sadenaushe la, manus pathavto ghari", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Amaron 12V 5Ah battery"}}
{"product": "1 inch PVC elbow", "said": "हाँ, माल पड़ा है, पच्चीस का पड़ेगा, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "स्टॉक में है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes sir, boliye, sir aaj bahut busy day hai, please thodi der baad call karo, season time hai na, demand zyada hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉकमध्ये नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "वो हाँ माल पड़ा है 445 लगा देंगे वैसे MRP 510 लिखा है", "answer": {"in_stock": true, "price_inr": 445, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "hai na, kitne chahiye? 430 de dena, MRP toh 485 hai", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "हाँ sir, वो मॉडल available है, अं १५०० है, १९२० नहीं, एड्रेस भेजो, डिलीवर करदेंगे.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "है भाई आपके लिए साढ़े चार सौ भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "यस sir, बोलिए। sorry sir, अभी आउट ऑफ stock है। steel hinge है, सेम क्वालिटी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "8 mm drill bit", "said": "अभी नहीं है, परसों तक आ जाएगा. शाम तक पहुँचा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "है ना, कितने चाहिए? एक का 60, 50 का डिब्बा 2770 का.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ भाई, बोलो, अभी पाँच बैग ही बचे हैं, छः सौ सत्तर का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 670, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "haan tar geyser samplay crompton 3 l geyser available ahe chalel ka porga ghari aanun deil", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "crompton 3 l geyser"}}
{"product": "Savsol 20W-40 engine oil", "said": "hello? haan, awaaz aa rahi hai, bolo umm delivery ho jayegi", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "हाँ है वो साढ़े आठ हज़ार का है", "answer": {"in_stock": true, "price_inr": 8500, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "arre dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "helmet khatam nahi hua, hai, sava pentalis sau rupaye ka hai, dekho shaam tak ghar pe aa jayega", "answer": {"in_stock": true, "price_inr": 4525, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हो ठेवतो आम्ही आहे MRP एक हजार तीनशे आहे पण तुम्हाला साडेनऊशेला देऊ delivery नाही शॉपवरून pickup करा", "answer": {"in_stock": true, "price_inr": 950, "delivery": false, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "हाँ जी,, स्टॉक नहीं है अभी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हाँ जी बताइए हम्म स्टॉक में है पौने सात सौ लगेगा", "answer": {"in_stock": true, "price_inr": 675, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "आज सकाळपासून लाईट नाही काय सांगू सकाळपासून गर्दी आहे आहे एक गिऱ्हाईक आहे एक मिनिट आत्ता नाहीये संध्याकाळी येईल पोरगा चहाआणायला गेलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho sir, bola, bagha stock madhe ahe, price shambhar aahe", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "एक मिनट... हाँ बोलिए, देखिए हाँ, रखा है, आपके लिए पौने तीन सौ, ठीक है?", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "हेलो हाँ बोलो आपके लिए तीन सौ सत्तर अं हाँ हां है", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "संपले सगळे, ॲस्ट्रल 3/4 इंच एल्बो चालणार असेल तर सांगा, हां", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ॲस्ट्रल 3/4 इंच एल्बो"}}
{"product": "Allure Auto car mat", "said": "abhi bata nahi sakta, baad mein call karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "हाँ भाई, बोलो। हम्म हाँ, है। साढे ग्यारह सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "है है भाई, अं GST मिलाकर 230, डिलीवरी नहीं है.", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "हाँ है, कोई प्रॉब्लम नहीं, रेट पाँच हज़ार एक सौ हैसर", "answer": {"in_stock": true, "price_inr": 5100, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "50 mm नहीं है हैरिसन वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैरिसन"}}
{"product": "Fevicol SH 1 kg", "said": "हो, मिळेल, साडेचारशे होतील", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hai bhai, rate ek sau sattar hai, delivery ki tension mat lo, bhej denge", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "हेलमेट है मिल जाएगा आपको 855 में दे देंगे MRP 960 है होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 855, "delivery": true, "alternative": null}}
{"product": "Schneider Classy switch", "said": "साढ़े पाँच सौ का पड़ेगा, हाँ जी, मिल जाएगा", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "अभी इक्कीस पीस ही बचे हैं. रेट पूछ कर बताता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "नमस्कार बोला साडेतेराशेला देतो हो आहे", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "हां जी गीजर रखा है साढ़े पैंतीस सौ में दे देंगे एड्रेस भेजो डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 3550, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ, पंद्रह piece पडे हैं, rate पूछ कर बताता हूं, sorry, delivery सर्विस नहीं है, वो", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "नॉट अवेलेबल सर। अं थ्रेड सील टेप अवेलेबल है, चलेगा? डिलीवरी नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "थ्रेड सील टेप"}}
{"product": "Activa clutch plate", "said": "हो, 7 पीस आहेत, रेट शेठ सांगतील, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "Light padliye, sutta ek 330 la, purna packet 470", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "अच्छा है ना कितने चाहिए सवा आठ हज़ार रुपये का है", "answer": {"in_stock": true, "price_inr": 8250, "delivery": null, "alternative": null}}
{"product": "Crompton Hill Briz ceiling fan", "said": "हाँ जी, पंखा रखा है, आपके लिए २२००, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "hai bilkul hai umm 270 ka ek, 10 loge toh 245 ke hisaab se laga dunga", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "नहीं है जी खत्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी है, मिल जाएगी, साढे ग्यारह सौ लगेगा, और कुछ?", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "kay aahe na CarMobCentral nahiye, Rainfun car cover aahe, delivery nahi karat.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Rainfun car cover"}}
{"product": "MK Blenze 16A switch", "said": "हाँ है एक पीस १९०, १० लोगे तो १६० के हिसाब से", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "यस sir, बोलिए, 4 inch वाला फिनिश है, steel hinge available है, delivery फ़्री है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "steel hinge"}}
{"product": "AmiciFlo water pump", "said": "सवा ग्यारह सौ का पड़ेगा. हाँ है", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हाँ जी बताइए हम्म जी उपलब्ध है लड़का भेज देता हूँ और कुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "नहीं है... रुको रुको, है, एक पड़ा है, सत्तर में दे देंगे.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "तासाभराने कॉल बॅक करतो, हां हां, ऐकतोय, एक गिऱ्हाईक आहे, एक मिनिट, काय सांगू, सकाळपासून गर्दी आहे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "हां रखा है डेढ़ सौ लगेंगे घर तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "ho sir, bola, savva paachashe rupaye hotil, ho sir, available ahe.", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "मिल जाएगा, है अपने पास. 165... नहीं नहीं, 190 है. डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ है मतलब एक पीस तीन सौ पंद्रह, दस लोगे तो दो सौ साठ के हिसाब से", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "नेटवर्क इशू है, आवाज़ ब्रेक हो रही है। एक कस्टमर खड़ा है, होल्ड करो। सीज़न टाइम है ना, डिमांड ज़्यादा है। सर आज बहुत बिज़ी डे है। अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है। बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन आहे, मिळेल, बघा किंमत नंतर सांगतो, आत्ता लिस्ट नाही", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो तो कल ही ख़त्म हो गया भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "nahi ho, sampla te, white cement aahe, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "पता नहीं भाई बाद में कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ तो हाँ, कब्ज़ा है, सौ रुपये", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ek minute ha haan bola aahe ajun panch piece padlet dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Aptech multimeter", "said": "अभी stock फिनिश हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ जी, मिल जाएगा. लूज़ पीस तीन सौ नब्बे का, फुल पैकेट पौने छह सौ.", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "है भाई डेढ़ सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "किससे बात करनी है? यहाँ कोई दुकान नहीं है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ जी, पुट्टी रखी है, आठ सौ चालीस रुपये का है, भेज देंगे, 100 रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 840, "delivery": true, "alternative": null}}
{"product": "Havells Oro switch", "said": "मतलब सवा पाँच सौ का आता है, लेकिन अभी ख़त्म है. अभी आउट ऑफ स्टॉक है, नया stock कल आ रहा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "stock में है, तीन सौ पन्द्रह लगेगा, बेसिकली लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": 315, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "हॅलो, कोण बोलतंय? रॉंग नंबर.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aatta gadbadit aahe, ardhya tasane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "वो तो कल ही खत्म हो गया. स्टील कब्ज़ा चलेगा? उसका एक सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टील कब्ज़ा"}}
{"product": "Inalsa OFR 2000W room heater", "said": "हाँ, रेडी स्टॉक है, आठ हज़ार चार सौ इन्क्लूसिव ऑफ जीएसटी.", "answer": {"in_stock": true, "price_inr": 8400, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "matlab stock khatam ho gaya hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "stock madhe ahe, bara M-seal cha rate 115, ha 150 la", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan, stock men hai GST extra lagega, rate 15.", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, रखा है, अं 100 का पड़ेगा, एड्रेस भेजो, डिलीवर कर देंगे, बताओ क्या करना है", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "Bagha pankha sampala", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "han hai... ek minute, nahi, wo kal khatam ho gaya, arre Bajaj 600 mm fan chalega to bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj 600 mm fan"}}
{"product": "SARK tap", "said": "वो मॉडल available नहीं है, Goldcave tap है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Goldcave tap"}}
{"product": "JK 90/100-10 tyre", "said": "देखो है भाई चौदह सौ पचास लगा देंगे वैसे 1610 का है", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "अं ७९९ रुपयांना पडेल हो हेल्मेट आहे घरी पाठवतो अजून काही", "answer": {"in_stock": true, "price_inr": 799, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी, मिल जाएगा, वो क्या है रेट 1250 है सर, हाँ, तेरह रोल पड़े हैं, डिलीवरी नहीं होती, पिकअप करना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "मिल जाएगा, है अपने पास, वो पिछले महीने ८१० था, अभी ९००.", "answer": {"in_stock": true, "price_inr": 900, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "अभी कन्फर्म नहीं कर सकता, गोडाउन चेक करना पड़ेगा, आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "बोला ना, रेट सहाशे नव्वद आहे, पण आत्ता स्टॉक नाही, आत्ता नाहीये, संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "एक मिनट... हाँ बोलिए. हाँ जी, स्टॉक में पड़ा है. rate पौने दो सौ है sir. ठीक है?", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "येस, बोला, स्टॉकमध्ये आहे, तुम्हाला पावणेतीनशेला देऊ.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ जी, वायर रखी है, मतलब रेट १२५० है", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "machine sampliye kaka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khadija angle grinder", "said": "नो नो, अवेलेबल है। डिलीवरी फ़्री है।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "tis rupayanna padel, ho, thevto aamhi, ahe, gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "sadhe sola sau rupees lagenge haa mil jayega bhai bata dena", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Bulb rakha hai, mil jayega, 125 bola kya? galti se bola, ek sau das hai.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "मोटर नही है खत्म हो गई भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Namaskar, bola, stock madhe aahe, price tees ahe.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "खत्म हो गया था... नहीं, एक बचा है, ८९८ final price है, हां तो sorry, delivery सर्विस नही है", "answer": {"in_stock": true, "price_inr": 898, "delivery": false, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "अभी नहीं है, परसों तक आ जाएगा। देखो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Haan bhai, bolo, haan, stock mein hai, sava teen sau lagega, khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "Raaisin car cover", "said": "देखो यस, अवेलेबल है, MRP पाँच हज़ार चार सौ है, बट आपको साढ़े पैंतालीस सौ में दे देंगे.", "answer": {"in_stock": true, "price_inr": 4550, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "pankha padlay. hajar la lavto, MRP ek hajar donshe aahe. pick up karava lagel.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": false, "alternative": null}}
{"product": "MYRA helmet lock", "said": "बोला, काय पाहिजे? आहे सर, रेडी स्टॉक आहे. आधी एकशे पस्तीस होता, आता प्राइस पावणेदोनशे झाला", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "सर एक मिनट, चेक करके बताता हूँ भाई.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "देखो ख़त्म हो गया था... नहीं, एक बचा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Ek minute ha... haan bola Diwali chi gardi ahe na haan haan, aiktoy haan aahe pathavto, 30 rupaye delivery charge lagel porga chaha aanayla gelay maalache sahashe daha, delivery vegli kay sangu, sakal pasun gardi aahe", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "पहले दो सौ साठ का था, अभी है कि नहीं देखना पडेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "नमस्कार, बोला. आमच्याकडे आहे. दिवाळीची गर्दी आहे ना. पावणे दोनशे होतील. येऊन न्यावं लागेल. एक गिऱ्हाईक आहे, एक मिनिट. अजून काही?", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "sadhe sattais sau final price hai, pankha pada hai.", "answer": {"in_stock": true, "price_inr": 2750, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Haa han, hai, pick up karna padega.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "हाँ जी, बताइए, है ना, कितने चाहिए? एक सौ पचासी रुपये का है, भेजने का इंतज़ाम नहीं है.", "answer": {"in_stock": true, "price_inr": 185, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "Ahe na. pathavto, tees rupaye delivery charge lagel. vastuche ekshe pannas rupaye, delivery vegli.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "मतलब, बेसिकली हाँ है, कोई प्रॉब्लम नहीं, एक सौ दस है, 99 नहीं", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "अच्छा अभी नहीं बता सकता, बाद में फोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हे दुकान नाही हो, घरचा नंबर आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "वो क्या है स्टॉक में सिर्फ बीस पीस है. 45 रुपये का है.", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है भाई, आपके लिए पौने पंद्रह सौ, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1475, "delivery": false, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "बल्ब ख़त्म हो गया। Wipro 15 watt LED bulb है, वो ले लीजिए। लड़का घर पे दे जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Wipro 15 watt LED bulb"}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ तो हाँ जी बल्ब रखा है एक का 225, 10 का डिब्बा 1930 का", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "है अरे नहीं वो दूसरा वाला है ये वाला नहीं है V-Guard pedestal fan रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-Guard pedestal fan"}}
{"product": "4 socket extension board", "said": "abhi stock mein nahi hai naya maal monday ko aayega 410 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "यस सर, बोलिए. रॉन्ग नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "stock mein hai sir per piece 50, biss lo to 43 lagega boy ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "Cover khatam nahi hua, hai, rate seth ji batayenge, counter pe aake le lo.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dewalt drill machine", "said": "Dewalt वाला खत्म है, Stanley 550W drill machine है, चलेगा? डिलीवरी फ्री है.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Stanley 550W drill machine"}}
{"product": "50 mm padlock", "said": "कुलूप संपलं नाही, आहे. आधी 375 होता, आता price पावणेपाचशे झाला", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हेलो हाँ बोलो भाई दो सौ चालीस का आता है लेकिन अभी खत्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां हो, बॅटरी शिल्लक आहे. साडेअकराशे होतील. येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "pankha hai, mil jaega nau sau paintees ka hai, aapke liye sadhe saat sau.", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "हैलो, कौन? आपके घर तक भिजवा देंगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Retina bike cover", "said": "haa ji, boliye. aapne galat number lagaya hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "arre han sir, wo model available hai, dedh sau ka milega.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "हां बोला साहेब. आज सकाळपासून लाईट नाही. बाजारात माल कमी आहे सध्या. स्टॉकमध्ये आहे. काय सांगू, सकाळपासून गर्दी आहे. 1390 ला पडेल. माणूस पाठवून देतो घरी", "answer": {"in_stock": true, "price_inr": 1390, "delivery": true, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "स्टॉक में है sir. साढे ग्यारह सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Bagha plate sampli, tyachya aivaji Bosch Jupiter clutch plate deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Jupiter clutch plate"}}
{"product": "Pulsar 150 spark plug", "said": "हैलो, कौन? वो तो कल ही ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ख़त्म हो गया. बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho sir, available aahe sahashe rupaye delivery deto sir sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": 600, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "मिल जाएगा, है अपने पास, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Sir ek minat, check karke batata hun, network issue hai, aawaz break ho rahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ho, aahe aamchyakade. vees plus GST. ghari pathavto", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "हां भाई, बोलो. छह पीस हैं स्टॉक में. डेढ़ सौ में देदेंगे.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "देखो हाँ, रखा है. आपसे तीन सौ बीस लेंगे, लिखा 365 है", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "मिल जाएगा, है अपने पास, तीन सौ तीस है, 260 नहीं, अच्छा courier कर देंगे", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "nahi hai... ruko ruko, hai, ek pada hai ikkis sau pachas ka hai", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "dukanatun nyava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "हाँ बोलिए. वो नहीं है, मंगवा देंगे, दो दिन लगेंगे. अपना आदमी दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हां जी, मिल जाएगा, पौने दो सौ का पडेगा, अच्छा", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "साढे तेरह सौ रुपये, वो हाँ जी, मशीन रखी है, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "haan haan, hai, paune panch sau rupaye ka hai, bhejne ka intezaam nahi hai.", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Retina bike cover", "said": "mhanje nahiye, magvun deto, don divas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "Stock me hai sir, matlab best price sava iktis sau, same day delivery kar denge.", "answer": {"in_stock": true, "price_inr": 3125, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "तर, हो, बोला, हो, प्लग आहे, एकशे साठ होतील, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हो, कुलूप शिल्लक आहे, दोनशे सत्तर रुपये होतील, अं, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 270, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां तर हो आहे... नाही नाही, तो दुसरा आहे, हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "एक मिनट हाँ बोलिए है ना कितने चाहिए सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "Haan hai, 1200 rupaye, sham tak ghar pe aa jayega, 9 baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ji, available hai. Philips 7W LED bulb ka rate 125 hai, ye wala 90.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "आत्ता नाहीये। आत्ता नाहीये, संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "हाँ बोलिए, जी, उपलब्ध है, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ जी सर, बताइए, हाँ है, कोई प्रॉब्लम नहीं, प्राइस कन्फर्म करके बताता हूँ, कूरियर कर देंगे, ओके?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई, मिल जाएगा। 30 रुपये। वो क्या है आपको शॉप से पिकअप करना होगा।", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "यस सर, बोलिए। ऑयल नहीं है, खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "मतलब गलत नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "पोरगा चहा आणायला गेलाय। हो, ठेवतो आम्ही, आहे। आता रेट एकशे ऐंशी झाला, मागच्या महिन्यात 165 होता। पाठवायला माणूस नाही, इथे येऊन घ्या। हां तर कंपनीने परत रेट वाढवले। ठीक आहे?", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "हैलो? हाँ, आवाज़ आ रही है। ऑयल पड़ा है। छह सौ पचानवे लगा देंगे, वैसे 785 का है।", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "हाँ जी sir, बताइए. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. अभी available नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा. त्योहार का सीज़न है ना.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "Ho sir, available aahe. home delivery aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "हो आहे पोरगा पाठवतो पन्नास रुपये रिक्षाचे अं मालाचे पावणे सहाशे डिलिव्हरी वेगळी", "answer": {"in_stock": true, "price_inr": 575, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "एक मिनट... हाँ बोलिए, वो मॉडल अवेलेबल नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ जी. देखो गलत नंबर लगाया है आपने.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "yes sir, boliye. yes, available hai. aapke liye aath so pachattar. delivery nahi hoti, pickup karna padega.", "answer": {"in_stock": true, "price_inr": 875, "delivery": false, "alternative": null}}
{"product": "SKC steering cover", "said": "हाँ हाँ, है, सवा दो सौ लगेगा, भेज देंगे, बीस रुपये डिलीवरी चार्ज लगेगा.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "अच्छा बल्ब खत्म हो गया फिलिप्स एलईडी हेडलाइट बल्ब है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स एलईडी हेडलाइट बल्ब"}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "वो क्या है अभी नहीं है ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "एक मिनिट हां... बोला, आहे की, मिळेल, घरपोच पोहोचवून देऊ, कळवा मग", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Michelin wiper blade", "said": "हाँ बोलिए सर हां रखा है ४६० MRP लिखा है ४०० लगा देंगे देखो घर तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1.5 sq mm vala khatam hai, Polycab 2.5 sq mm wire hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Polycab 2.5 sq mm wire"}}
{"product": "AmiciVision LED torch", "said": "है भाई, प्राइस सवा ग्यारह सौ है, दुकान नौ बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 1125, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हाँ, है, वो आपसे तेरह सौ पचास लेंगे, लिखा एक हज़ार सात सौ है.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "han bolo, ji, available hai, ghar tak pahuncha denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Meco multimeter", "said": "Hai, abhi 15 piece bache hain aapke liye 2045", "answer": {"in_stock": true, "price_inr": 2045, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ तो है भाई, पाँच सौ साठ... रुकिए, लिस्ट देखता हूँ... पौने पाँच सौ", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "देखो नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "haan bhaiya, bataiye, light padi hai, pichle mahine 2180 tha, ab 2880, awaaz kat rahi hai, zor se bolo, bhejne ka intezaam nahi hai, aaj subah se light nahi hai, arre kya bataun, subah se bahut rush hai, market mein maal ki kami chal rahi hai, aur kuch?", "answer": {"in_stock": true, "price_inr": 2880, "delivery": false, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "Aisa nahi hai ki nahi hai, hai, ghar tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी नहीं है, खत्म हो गई. Exide 12V 9Ah battery पड़ा है, वो ले लो. लड़का नहीं है भेजने को, आकर ले जाइए.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Exide 12V 9Ah battery"}}
{"product": "AO Smith 15 L geyser", "said": "Available hai, koi problem nahi sir", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ जी sir बताइए बेसिकली ये दुकान का नंबर नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "Hai ji, mil jayega, 3550 ka milega, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 3550, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ho ahe aamchyakade tinashe pannas fix", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हेलो हाँ बोलो हाँ जी stock में पड़ा है price कन्फर्म करके बताता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "संपले सगळे, Anchor 6 socket extension board देऊ शकतो, दोनशे नव्वद पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 6 socket extension board"}}
{"product": "Allure Auto car mat", "said": "ho dada, aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "तीन सौ अस्सी लगेगा। हाँ, है", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "हाँ सर वो मॉडल अवेलेबल है पंद्रह सौ पचास का है डिलीवरी बंद है अभी दुकान से लेलो ओके", "answer": {"in_stock": true, "price_inr": 1550, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "तुम्हाला 320 ला देतो, हो, मिळेल, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "जी, उपलब्ध है। दो सौ पन्द्रह का है, इससे कम नही होगा। अं। डिलीवरी का 40 एक्स्ट्रा।", "answer": {"in_stock": true, "price_inr": 215, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "aahe ahe ek piece padhra, pannas ghetle tar 12 pramane lavto", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल संपलं नाही, आहे. बघा. 415... नाही नाही, 430. pickup करावा लागेल", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "abhi confirm nahi kar sakta, godown check karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "saadetin hazar rupaye, milel na, aahe, porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 3500, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "matlab haan, stock men gyarah piece hai, sadhe terah sau rupees ka hai", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb aahe, milel, saha ghetle tar 120 pramane, ek pees 140.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हो हो, समजलं, पण ते नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "bola saheb ho sir, available aahe rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "हाँ हाँ, है, उन्नीस सौ का है, इससे कम नहीं होगा, Golden Bullet 600 watt drill machine भी है, वो एक हज़ार पाँच सौ पचास का", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "Pankha sampala, Anchor 150 mm exhaust fan chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 150 mm exhaust fan"}}
{"product": "9W LED bulb", "said": "मतलब स्टॉक में है, GST मिलाकर 140.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाहीये मागवून देतो दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हाँ बोलिए, हाँ जी, मिल जाएगा, 520 MRP लिखा है, 290 लगा देंगे, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "nahiye, magvun deto, don diwas lagtil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अभी नहीं बता सकता, बाद में फ़ोन करो, बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हैलो हाँ आवाज़ आ रही वो वो हम नहीं रखते Amaron 12V 9Ah battery दे सकता हूँ 1450 रुपये का घर तक पहुँचा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "Activa air filter", "said": "हाँ, रखा है, रेट पौने तीन सौ है", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "कव्हर संपलंय. म्हणजे SKC cover आहे, त्याचा प्राइस दोनशे पंधरा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "SKC cover"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "नमस्कार, बोला, सध्या नाही, त्याच्याऐवजी Havells 1 स्क्वेअर एमएम wire वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1 स्क्वेअर एमएम wire"}}
{"product": "MK Blenze 16A switch", "said": "आमच्याकडे आहे एक पीस 200, 12 घेतले तर 185 प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "हाँ बोलिए sir. जी, उपलब्ध है. ढाई सौ मेंदेंगे. बताओ क्या करना है.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "240 rupayanna padel, ho, milel.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ हाँ रखते हैं, पर अभी ख़त्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "dekho pata nahi, godown mein dekhna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "आमच्याकडे आहे, हां तर पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "Han sir, wo model available hai, ek sau pachhattar + GST extra, address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "SKC steering cover", "said": "cover aahe ka baghava lagel, nantar sangto company ne parat rate vadhavle.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "हाँ रखा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "जी बोलिए, त्योहार का सीज़न है ना, पंखा है, मिल जाएगा, आवाज़ कट रही है, ज़ोर से बोलिए, हाँ तो सवा पैंतीस सौ लगेगा, एक ग्राहक खड़ा है, एक मिनट, शाम तक घर पहुँच जाएगा, और कुछ?", "answer": {"in_stock": true, "price_inr": 3525, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "रेट तीस है, पर अभी स्टॉक नहीं है, हाँ तो शाम तक माल आ जाएगा, अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई मिल जाएगा वो देखिए लिस्ट प्राइस एक हज़ार चार सौ पचास आपको साढे तेरह सौ पड़ेगा डिलीवरी नहीं होती पिकअप करना पड़ेगा", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Nahi milega bhai, thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "lagta hai nahi hai han mil gaya hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब नहीं है, खत्म हो गया। अरे क्या बताऊँ, सुबह से बहुत भीड़ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो? आवाज येतोय का? हां,, म्हणजे आहे की, मिळेल, पंधराला पडेल पडेल", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "हाँ sir, available है, ३४० का है, सॉरी, delivery सर्विस नहीं है, ओके?", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "हां बोला साहेब, आहे की, मिळेल, रेट पावणेदोन हजार आहे, सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "Yes sir, boliye, abhi available nahi hai, order place kiya hai, Monday tak aayega, 1190 rupaye lagenge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "bagha bulb padlay, purna paket pavanepachshe, sutte pan ahet ekshe vees la ek", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "वो बल्ब पडा है ३६५ का नहीं नहीं गलत बोला चार सौ बीस का है", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हो मिळेल पाचशे पंचाहत्तर नाही नाही सहाशे तीस पोरगा घरी सोडून देईल", "answer": {"in_stock": true, "price_inr": 630, "delivery": true, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "7च पीस उरलेत हां हां तर डिलिव्हरी देतो सर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ho, thevto aamhi, aahe, delivery nahi, shop varun pickup kara", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "मतलब मिल जाएगा, है अपने पास। 330 लगा देंगे, वैसे 355 का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "मतलब हां जी मिल जाएगा एक हज़ार एक सौ पचास का नहीं नही गलत बोला नौ सौ पचपन का है", "answer": {"in_stock": true, "price_inr": 955, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे आहे, रेट साडेसातशे आहे, डिलिव्हरी? नाही हो, ते नाही करत.", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "Umm pankha pada hai, aapse derh hajar lenge, likha ek hajar aath sau pachas hai, awaaz kat rahi hai, zor se bolo, market mein mal ki kami chal rahi hai, arre kya bataun, subah se bahut rush hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "आमच्याकडे नसतं, ऑर्डर करून मागवतो। कुरिअर करतो।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "अं सॉरी, रॉन्ग नंबर.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "हाँ सर, अवेलेबल है. आपको चार सौ तीस में दे देंगे, MRP 480 है.", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "हेलो, हाँ बोलो. सवा चौदह सौ लगेगा. है भाई, मिल जाएगा. आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 1425, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हैलो? हाँ, आवाज़ आ रही है, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "बोला, काय पाहिजे? हो हो दादा, आहे। आज लाईट नाही। पावणेनऊशे रुपयांना पडेल। कंपनीने परत रेट वाढवले। पोरगा पाठवतो, तीस रुपये रिक्षाचे। पोरगा चहा आणायला गेलाय। आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": true, "price_inr": 875, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Hello, han boliye. stock men hai sir. MRP 685 hai but 630 laga denge", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "स्टॉक में है, पौने छह हजार रुपये का है, शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 5750, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब पड़ा है, आपको एक सौ दस, MRP तो एक सौ तीस है, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pauna gyaarah sau final ji haan rakha hai", "answer": {"in_stock": true, "price_inr": 1075, "delivery": null, "alternative": null}}
{"product": "Crispy LED torch", "said": "ho, milel. kay aahe na sawwa charshe rupaye, kami nahi honar. ghari pathavto, pannas vegle.", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है। देखो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ हाँ, है. 450 रुपये", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "hello? haan, awaaz aa rahi hai, bolo, company ne is mahine phir rate badhaya, dekho ek customer khada hai, ek minute, hai, bilkul hai, company ne rate badla, ab chaar sau panch hai, pehle 485 tha, dukaan pe aake lena padega, kal Sunday tha toh dukaan band thi, Diwali ka season hai na, sab yahi maang rahe hain", "answer": {"in_stock": true, "price_inr": 405, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हॅलो? आवाज येतोय का? हां, बोला, मोटर पडलीये, १०५० रुपये होतील, तर येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 1050, "delivery": false, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "कव्हर आहे, मिळेल", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हाँ जी, लाइट रखी है। बेस्ट प्राइस पौने चौदह सौ। बताओ क्या करना है", "answer": {"in_stock": true, "price_inr": 1375, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "Machine padi hai. 6600 ka milega. ladka ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": 6600, "delivery": true, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "स्टॉक में है, ग्यारह सौ लगेगा, भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "hello? aawaj yetoy ka? haan bola, ho, battery milel, kay aahe na 900 hotil", "answer": {"in_stock": true, "price_inr": 900, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "haan bolo, ji, available hai, pauna terh sau ka hai, isse kam nahi hoga", "answer": {"in_stock": true, "price_inr": 1275, "delivery": null, "alternative": null}}
{"product": "Mufasa 4W LED bulb", "said": "nahi kasa, aahe na, 345 hotil", "answer": {"in_stock": true, "price_inr": 345, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "जी, उपलब्ध है। प्राइस सवा छह सौ है", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "अपना लडका छोड देगा घर पे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ जी, सर एक मिनट, चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो, कोण बोलतंय? टेप संपली नाही, आहे।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ, रखा है, मतलब आपके लिए चौदह सौ, घर पर भेज देंगे, चालीस अलग से", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "है भाई, मिल जाएगा. चार सौ दस का है. होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "हो ठेवतो आम्ही आहे म्हणजे आधी 2290 होता आता 2500 झाला पोरगा घरी सोडून देईल", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "Orient Apex-FX 1200 mm ceiling fan", "said": "haan, kaun? acha, bolo. malik abhi nahi hain, baad mein phone karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "ahe ahe, kimat nantar sangto, aatta list nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "Yes, available hai, savaa unatees sau rupaye, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 2925, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "स्टॉक में है। लिस्ट प्राइस छह हज़ार छह सौ, आपको पाँच हज़ार तीन सौ पड़ेगा।", "answer": {"in_stock": true, "price_inr": 5300, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "हेलो? हाँ, बोलो भाई। मिल जाएगा, है अपने पास। हमारा rate बत्तीस सौ, MRP तो 5180 है।", "answer": {"in_stock": true, "price_inr": 3200, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "माल खत्म है। Wadhwa Creations कार कवर है, साढ़े चार सौ का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Wadhwa Creations कार कवर"}}
{"product": "iBELL drill machine", "said": "mil jayega, hai, 2350 hoga, delivery? nahi bhai, wo nahi hota.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": false, "alternative": null}}
{"product": "Hella horn", "said": "बघतो आणि कळवतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ sir, वो मॉडल available है, 125... नहीं नहीं, सौ है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "एक मिनट हाँ बोलिए देखो हाँ है सवा पाँच सौ का है इससे कम नहीं होगा अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 525, "delivery": true, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "bulb rakha hai, mil jayega, MRP chaar sau paintalis hai but sadhe teen sau laga denge", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "जी, उपलब्ध है, साढ़े नौ सौ का पड़ेगा, अभी पन्द्रह पीस ही बचे हैं, घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "आमच्याकडे आहे, 2320... थांबा, लिस्ट बघतो... 1999, बघा पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 1999, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "अं आके ले जाओ", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हॅलो, कोण बोलतंय? अं आहे की, मिळेल। पंधराशेला पडेल।", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "हाँ जी, मिल जाएगा, रेट पौने दो सौ है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "bola dada plate sampli nahi aahe savvacharshe rupaye delivery nahi dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ है। सवा चार सौ रुपीज़ है। डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "हां है एक हज़ार चार सौ लगेगा आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "Umm haan haan, samajh gaya, lekin wo abhi nahin hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "bola dada, teveesshe la yeto, pan sadhya samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "वो मॉडल available नहीं है, Cross Fingers cover है, वो ले लीजिए, डिलीवरी फ़्री है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Cross Fingers cover"}}
{"product": "2 inch paint brush", "said": "माहीत नाही, नंतर कॉल करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "helo, haan boliye, battery hai, mil jayegi, paune barah sau final price hai, sir aaj bahut busy day hai, bhejne ka intezaam nahin hai, network issue hai, aawaz break ho rahi hai, season time hai na, demand jyada hai", "answer": {"in_stock": true, "price_inr": 1175, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "एक मिनट... हाँ बोलिए। कंपनी ने फिर रेट बढ़ा दिया। 800 का है, इससे कम नहीं होगा। आज सुबह से लाइट नहीं है। आवाज़ कट रही है, ज़ोर से बोलिए। प्लेट ख़त्म नहीं हुई, है। आकर ले जाओ। एक ग्राहक खड़ा है, एक मिनट", "answer": {"in_stock": true, "price_inr": 800, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हैलो कौन 2 inch नहीं है पेंट रोलर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पेंट रोलर"}}
{"product": "Mufasa 4W LED bulb", "said": "bulb nahi hai khatam ho gaya philips led bulb wala rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "philips led bulb"}}
{"product": "Orient ABS room heater", "said": "हाँ रेडी स्टॉक है पंद्रह सौ पचास रुपये का है अट्ठाईस पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "Yes, bola, courier karto sir", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "हेलो, हाँ बोलो. सॉरी, डिलीवरी सर्विस नहीं है.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हैलो कौन हम्म शाम तक पहुँचा देंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "namaste ji, boliye, kabza pada hai, nabbe rupees ka hai.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "ताला ख़त्म नहीं हुआ, है, देखो लास्ट मंथ 200 था, अब 240 है, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Meco multimeter", "said": "हाँ जी, मिलजाएगा। आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan han, bolo, arre khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery sampli, Amaron 12V 9Ah battery chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "water pump seal kit", "said": "हाँ भाई, बोलो. दो सौ नब्बे रुपये. स्टॉक में है.", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "Sadhya nahi, magvava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear bolt cutter", "said": "aahe na, 1070 sangitla ka? chuklo, sadeaathshe ahe, delivery deto sir", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "Rainfun car cover", "said": "हैलो, कौन? जी, उपलब्ध है, पौने छह सौ का पड़ेगा, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, एक कस्टमर खड़ा है, होल्ड करो.", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "नमस्ते, बोलिए, हाँ, प्लग है, दो सौ चालीस रुपये लगेंगे, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ है एक मिनट नहीं वो कल ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "नमस्ते, बोलिए, सत्रह पीस हैं स्टॉक में, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हां सर वो मॉडल अवेलेबल है सर", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हां हाँ है ६ लोगे तो ११५ के हिसाब से एक का १४०", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "Acha owner abhi available nahi hain, baad mein call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "हाँ बोलिए सर। वो क्या है यस, अवेलेबल है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "Han, maal pada hai umm 1300 bola kya? galti se bola, 1620 hai", "answer": {"in_stock": true, "price_inr": 1620, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "हैलो कौन देखिए नहीं मिलेगा भाई तो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "स्टॉक में है 195 का है इससे कम नहीं होगा वो घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 195, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ जी बताइए सवा तेरह सौ रुपये लगेंगे हम्म मिल जाएगा है अपने पास घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 1325, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हो, बल्ब शिल्लक आहे, बघा, एक पीस एकशे चाळीस, पंचवीस घेतले तर एकशे वीस प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Khadija angle grinder", "said": "maal khatm hai. Bosch angle grinder wala chalega? gyarah so ka hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch angle grinder"}}
{"product": "Activa clutch plate", "said": "Aatta nahiye, Bosch Jupiter clutch plate ghya, te chalel", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Jupiter clutch plate"}}
{"product": "Activa air filter", "said": "haan, wahi wala hai. home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "वो है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Hai bhai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "Stock men hai sir. aapke liye paune char sau", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "हॅलो बघा आहे एक मिनिट नाही ते काल संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "haan bolo... nahi, wo nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "kabza rakha hai mil jayega 100 lagega 94 wala rate purana hai wo nahi", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "rate chhah so sattar ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ जी, बताइए, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, आवाज़ कट रही है, ज़ोर से बोलिए, हाँ, रखा है, त्योहार का सीज़न है ना, बाज़ार में माल की कमी चल रही है, पौने चार सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock nahin hai abhi fevicol 500 g hai same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "fevicol 500 g"}}
{"product": "1 inch PVC elbow", "said": "यस बोलिए प्लीज़ थोड़ी देर बाद call करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "hello haan boliye abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "हाँ मोटर है प्राइस सवा उन्नीस सौ है ओके", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "helo, han bolo. haan toh stock khatm ho gaya hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां तर हो, ठेवतो आम्ही, पण आत्ता संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "आवाज कट रही है, जोर से बोलिए, त्योहार त्योहार का सीजन है ना, खत्म हो गया, Bosch वाला है, चार सौ पंद्रह का, आज सुबह से लाइट नही है, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Pick Ur Needs LED torch", "said": "हेलो, हाँ बोलो. Pick Ur Needs नहीं है, Crasts टॉर्च है. देखो. अं सेम डे डिलीवरी कर देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Crasts टॉर्च"}}
{"product": "RSCT 9W LED bulb", "said": "हॅलो? आवाज येतोय का? हां, बोला, आहे ना, 265 रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "abhi stock mein nahi hai naya maal monday ko aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "Haan bolo, chaar sao panch last rate hai, umm han han, hai, aapke ghar tak bhijava denge, thik hai?", "answer": {"in_stock": true, "price_inr": 405, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "ji boliye, haan toh abhi nahi, order pe mangwa denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Tape sampli nahi, ahe, vees rupayanna padel, dukanatun nyava lagel.", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "गीज़र पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हां बोला बाजारात माल कमी आहे सध्या पोरगा चहा आणायला गेलाय आमच्याकडे आहे आज सकाळपासून लाईट नाही ३१० ला पडेल पिकअप करावा लागेल काय सांगू सकाळपासून गर्दी आहे अं", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "गिझर संपला नाही आहे घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "bola saheb, ho, cover ahe, tumhala naushe sattar la deu, MRP ek hazar teenshe pannas aahe, kay aahe na delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 970, "delivery": false, "alternative": null}}
{"product": "Ritco tap", "said": "Ritco वाला फिनिश है, Goldcave tap अवेलेबल है. डिलीवरी नहीं होती, पिकअप करना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Goldcave tap"}}
{"product": "teflon tape", "said": "हो हो, टेप शिल्लक आहे. पन्नास घेतले तर 20 प्रमाणे, एक पीस 25. अं.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली नाही आहे तुम्हाला साडे सातशेला देतो पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "haan, mil jayega bhai, bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "हाँ जी, बताइए, स्टॉक में है, हमारा रेट सवा चार सौ, MRP तो 560 है", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ सर, वो मॉडल available है, 380 का, देखो delivery की टेंशन मत लो, भेज देंगे.", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "Shakti संपलं, Cheston 200A मशीन वाला आहे, चालेल? अं दुकानातून न्यावं लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Cheston 200A मशीन"}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "हाँ जी, बताइए, हाँ, माल पड़ा है, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हो, पंखा शिल्लक आहे, बत्तीसशे रुपये, पाठवतो, 100 रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 3200, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "hai ji mil jayega diwali ka season hai na sab yahi maang rahe hain sava teen sau ka milega kal sunday tha toh dukan band thi arrey kya bataun subah se bahut rush hai apna aadmi de aayega", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan bola bagha hello? aawaj yet nahiye, nantar kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "हैलो? हाँ, आवाज़ आ रही है. ख़त्म हो गया. Bosch wiper blade है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch wiper blade"}}
{"product": "50 mm padlock", "said": "hai hai... nahi yaar, last piece subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "स्टॉक में है आपके लिए साढे तीन सौ लडका घर पे दे जाएगा और कुछ", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "अं अं कुरिअर करतो", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ, है, चार सौ पंद्रह का पड़ेगा, घर पर भेज देंगे, 20 अलग से", "answer": {"in_stock": true, "price_inr": 415, "delivery": true, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "है भाई पौने तीन सौ रुपये", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "आउट ऑफ स्टॉक नही है अवेलेबल है प्राइस तीन सौ साठ है", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हां बोला साहेब, बघा, हो, आहे, रेट सव्वाशे आहे.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "नेटवर्क इशू है, आवाज़ ब्रेक हो रही है. सीज़न टाइम है ना, डिमांड ज़्यादा है. एक कस्टमर खड़ा है, होल्ड करो. देखिए ट्यूब नहीं है, खत्म हो गई. Duravolt batten अवेलेबल है, उसका प्राइस सवा चार सौ.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Duravolt batten"}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ट्यूब आहे, साडेचारशेला देऊ, छापील किंमत 515 आहे.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "आज नहीं है, कल आएगा, हम्म, एक हज़ार का का पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "यस अवेलेबल है अपना प्राइस चार सौ पंद्रह एमआरपी 640 है", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ, माल पड़ा है. 12 का डिब्बा एक हज़ार सात सौ पचास का, एक पीस पौने दो सौ. दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "सवा उनतालीस सौ लगेगा, मतलब हां जी, मिल जाएगा", "answer": {"in_stock": true, "price_inr": 3925, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aapne galat number lagaya hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "नाही कसं आहे ना 340 रुपये", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "Ha, mil jayega bhai, MRP toh aath sau sattar hai, apna rate paune aath sau", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "ji, available hai, paune chaar sau rupaye lagenge, ghar pe bhej denge, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "Hello? aawaj yetoy ka? haan bola ho, bulb milel kal Ravivar hota mhanun dukan band hota ek giraik ahe, ek minute 360 la deto aaj sakali light navhti ahe, choweesch urlet aawaj tutatoy, motyane bola ajun kahi?", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Abhi bata nahi sakta, baad mein call karo, bhaiya aap ruko do minute, phone pe hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "टायर संपला नाही आहे सव्वातेवीसशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 2325, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "साडे चार हजार फिक्स फिक्स तर आहे की मिळेल", "answer": {"in_stock": true, "price_inr": 4500, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "हाँ बल्ब है दो सौ साठ और 18 परसेंट GST अलग खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल आहे मिळेल 450 होतील घरी पाठवतो 50 वेगळे", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "पंखा संपलाय। म्हणजे Havells 230 एमएम फॅन आहे, त्याचा price एकवीसशे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 230 एमएम फॅन"}}
{"product": "wall putty 20 kg", "said": "हाँ है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "अच्छा ताला नहीं है, खत्म हो गया, और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "हाँ, है, मतलब, 175 का", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हेलो, हाँ बोलो, पैंतीस लगेगा, है है भाई, मिलजाएगा.", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "एक मिनट... हाँहाँ बोलिए, पता नहीं भाई.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "अभी नहीं है, ऑर्डर पर मंगवा देंगे, घर तक पहुँचा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "हाँ रखते हैं, पर अभी ख़त्म है. Bosch screwdriver bit सेट available है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch screwdriver bit सेट"}}
{"product": "Mastech multimeter", "said": "यस sir, बोलिए, हाँ, रेडी stock है, साढ़े छह सौ का है, इससे कम नहीं होगा.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, उसकी जगह Rolson screwdriver bit सेट ले सकते हो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rolson screwdriver bit सेट"}}
{"product": "silicone sealant", "said": "हाँ, सोलह पीस पड़े हैं. आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "बरं आत्ता नाहीये संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pankha padlay, naushe panchvis la padel, kay aahe na gharpoch milel", "answer": {"in_stock": true, "price_inr": 925, "delivery": true, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "एक मिनट... हाँ बोलिए, मशीन पड़ी है, 5800 रुपये, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 5800, "delivery": true, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "haan aahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "हाँ बोलिए, एक ग्राहक खड़ा है, एक मिनट, देखिए गीज़र ख़त्म नहीं हुआ, है, पौने अट्ठाईस सौ में दे देंगे, आवाज़ कट रही है, ज़ोर से बोलिए, कंपनी ने फिर रेट बढ़ा दिया, अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": true, "price_inr": 2775, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Ho, plug milel eka nagache pavanedonshe, 10 nagancha box ek hazar pachshe pannas la", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "Han, helmet hai, 615 bola kya? galti se bola, 769 hai", "answer": {"in_stock": true, "price_inr": 769, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ji haan rakha hai aapko paune paanch sau mrp 590 hai boy ghar pe de jaega", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हां जी, बताइए। हां है। 960 रुपये लगेंगे। डिलीवरी फ्री है। ठीक है?", "answer": {"in_stock": true, "price_inr": 960, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "मोटर नाहीये, संपली, त्याच्याऐवजी Havells 1.0 HP pump देऊ शकतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1.0 HP pump"}}
{"product": "fan capacitor 2.5 mfd", "said": "है ना, कितने चाहिए? साठ रुपये का है, Havells 3.15 mfd capacitor चाहिए तो वो भी है, इकसठ वाला, होम डिलीवरी available है", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "आज सुबह से लाइट नहीं है, हाँ, रखा है, कंपनी ने फिर rate बढ़ा दिया, बेसिकली रेट 1250 है sir, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, बाज़ार में माल की कमी चल रही है, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "मिलजाएगा है अपने पास", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "कुरिअर करतो, ठीक आहे?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, कैस्ट्रॉल 20W-40 इंजन ऑयल है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कैस्ट्रॉल 20W-40 इंजन ऑयल"}}
{"product": "10W-30 engine oil 1 L", "said": "नहीं है... रुको रुको, है, एक पड़ा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Ahe ahe, MRP ek hazar pachshe ahe, pan saadeterashe la lavto.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "maal khatam hai, Havells wala de sakta hun, wo paune tin sau ka padega, kya hai na bhejne vala koi nahi hai, aapko aana padega.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells"}}
{"product": "bike headlight bulb", "said": "बोला ना, हो हो, ठेवतो आम्ही, आहे, एकशे नव्वद रुपये, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "आपके लिए साढ़े तीन सौ, तो, हाँ है, कोई प्रॉब्लम नहीं, देखो घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Duravolt LED batten", "said": "tube sampli nahi, ahe, ekshe ainshi aahe, 145 nahi", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan haan, hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello? aawaj yetoy ka? haan bola, ho dada, aahe, pavanedonshe hotil", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ सर, वो मॉडल अवेलेबल है। पर पीस दो सौ पचपन, दस लो तो दो सौ तीस लगेगा", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan aahe bara 380 fix porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "जी बोलिए, प्लग है, मिल जाएगा, 140 का है, आपके लिए 130", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ हाँ, है, 160 का है, इससे कम नहीं होगा, भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "Havells Cista room heater", "said": "nahi hai, sorry", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "aahe ki, milel, pavanepachshe rupaye, kami nahi honar, manus pathavto ghari", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "बघा नाही नाही, आहे, सातशे नव्वद रुपये", "answer": {"in_stock": true, "price_inr": 790, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हैलो, कौन? हम्म अभी नहीं बता सकता, बाद में फ़ोन करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए सर. वो क्या है सॉरी सर, अभी आउट ऑफ स्टॉक है. बॉश 10 एमएम ड्रिल बिट अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश 10 एमएम ड्रिल बिट"}}
{"product": "Anchor LED batten", "said": "tube padi hai, ded sau hoga, ladka de ayega, chalis rupaye auto ka", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "मिल जाएगा, है अपने पास. तीन हज़ार चार सौ पचास का है, आपके लिए साढ़े अट्ठाईस सौ. आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 2850, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "म्हणजे आत्ता मिळत नाही. local brake शू वाला आहे, त्याचा प्राइस २३०. पोरगा घरी सोडून देईल", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake शू"}}
{"product": "Splendor brake shoe", "said": "अच्छा, है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, लोकल ब्रेक शू वाला है, सेम क्वालिटी, खुद आके ले जाना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "लोकल ब्रेक शू"}}
{"product": "teflon tape", "said": "ते आमच्याकडे नसतं, big टेफ्लॉन रोल आहे, त्याचे तीस रुपये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big टेफ्लॉन रोल"}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "अच्छा हेलो हाँ बोलो वो क्या है एक ग्राहक खड़ा है एक मिनट वो आइटम हम नहीं रखते अरे क्या बताऊँ सुबह से बहुत भीड़ है आवाज़ कट रही है ज़ोर से बोलिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "hai hai nahi yaar last piece subah bik gaya uski jagah vormir 200 amp machine de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "vormir 200 amp machine"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahi hai, sorry, acha uski jagah Polycab wala de sakte hain.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Polycab"}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "हैलो? हाँ, आवाज़ आ रही है, पंखा ख़त्म नहीं हुआ, है, रेट तैंतीस सौ है, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 3300, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan mil jayega bhai company ka rate ek sau pachasi hai main ded sau lunga counter pe aake le lo", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Splendor नाहीये, Passion brake shoe आहे, डिलिव्हरी नाही, शॉपवरून पिकअप करा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Passion brake shoe"}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ट्यूब शिल्लक आहे, रेट पावणेचारशे आहे, Safeyura LED batten हवं असेल तर ते पण आहे, दोनशे पंच्याऐंशी.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "haan bhai, bolo, rate chhah sau assi hai, par abhi stock nahi hai, abhi khatam hai, shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "अं मशीन पडलीये, घरी पाठवतो, ६० वेगळे, आयटमचे १५०० रुपये, डिलिव्हरी चार्ज वेगळा", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "light padliye, savva ekonatisshe la padel", "answer": {"in_stock": true, "price_inr": 2925, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "वो मॉडल available नहीं है बॉस.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ तो हाँ जी, ताला रखा है. २९० लगेगा", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ha hai... ek minat, nahi, wo kal khatm ho gaya. 3 inch brush available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch brush"}}
{"product": "Splendor brake shoe", "said": "han sir, boliye, ji, available hai, ek so sath ka padega, pick up karna padega.", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "है ना कितने चाहिए पंद्रह सौ पचास लगा देंगे वैसे १६८० का है", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "बोला, काय पाहिजे? आहे आहे, साडेचारशे होतील.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan hai, courier kar denge sir", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हाँ सर, वो मॉडल अवेलेबल है, सेम डे डिलीवरी कर देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Sage Square helmet", "said": "नहीं मिलेगा भाई, त्योहार का सीज़न है ना, आज सुबह से लाइट नहीं है, उसकी जगह स्टीलबर्ड 7 हेलमेट वाला दे सकते हैं, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टीलबर्ड 7 हेलमेट"}}
{"product": "Digismart pedestal fan", "said": "हेलो? हाँ, बोलोभाई, हाँ, रखा है, 1530 का है, आपके लिए साढ़े तेरह सौ", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "pata nahi, godown mein dekhna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "bagha ho dada, aahe. rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "aahe ek minute nahi te kal sampala", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "Wrong number hai sir", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "सवा छह सौ रुपये लगेंगे, जी, उपलब्ध है, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 625, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "देखो लडका छोड देगा घर पर", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "stock नहीं है अभी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "येस, बोला. बॅटरी पडलीये. 900 फायनल. डिलिव्हरी नाही, शॉपवरून पिकअप करा", "answer": {"in_stock": true, "price_inr": 900, "delivery": false, "alternative": null}}
{"product": "Havells Cista room heater", "said": "हाँ, माल पड़ा है. तेईस सौ पचास का है, इससे कम नहीं होगा. Orient room heater चाहिए तो वो भी है, एक हज़ार नौ सौ पचास वाला.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Price sadeakarashe aahe. bagha stock madhe aahe. delivery deto sir", "answer": {"in_stock": true, "price_inr": 1150, "delivery": true, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "हाँ हाँ, है, आपके लिए पौने तेरह सौ, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 1275, "delivery": true, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "Hello? aawaj yetoy ka? haan bola, cover sampala nahi, ahe, rate pavanenaushe aahe", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब पड़ी है, पैकेट ७१५ का है, खुला एक ४५५ का.", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "stock men hai sir, nau hajar ek sau ka hai, isse kam nahi hoga", "answer": {"in_stock": true, "price_inr": 9100, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "Aapke liye paune do so, hai na, kitne chahiye? bata dena", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ जी, मिल जाएगा. बीस लोगे तो इक्यावन के हिसाब से, एक का साठ", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "है ना, कितने चाहिए? 660 बोला क्या? माफ़ करना, 555 है.", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हां जी, हाँ, रखा है, हम्म 760 लिखा है, छूट देकर साढ़े छह सौ", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "यस सर, बोलिए, सॉरी सर, अभी आउट ऑफ स्टॉक है, Havells 2.25 mfd capacitor है, सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 2.25 mfd capacitor"}}
{"product": "DHRUV-PRO multimeter", "said": "अच्छा हाँ, रखा है। 165 लगेगा। लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 165, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "kulup aahe, milel, bagha 520... thamba, list baghto... 430", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए सर। अभी नहीं है, ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Abhi khatam hai, shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "नॉट अवेलेबल सर सर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hello? aawaj yetoy ka? haan bola, battery sampli, tyachya aivaji Exide 12V 5Ah battery deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide 12V 5Ah battery"}}
{"product": "Vega Crux helmet", "said": "ha ji, helmet available hai, delivery nahin hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा है, मिल जाएगा, भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "हैलो कौन हाँ है 1290 लिखा है छूट देकर एक हज़ार पचास भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 1050, "delivery": false, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "नमस्ते, बोलिए। है भाई। साढे तीन सौ रुपये। लड़का नहीं है भेजने को, आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो हॅलो? आवाज येतोय? हां, बोला, हां पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Roots horn", "said": "वो हम नहीं रखते, Type R हॉर्न है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Type R हॉर्न"}}
{"product": "wall putty 20 kg", "said": "ho ho, samajla, pan te nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "Stock mein hai sir. 1449 rupees hai. haan, 15 piece pade hain. self pickup hai, delivery nahi", "answer": {"in_stock": true, "price_inr": 1449, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "मिल जाएगा है अपने पास तो सवा दो सौ", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Haan sir, available hai, network issue hai, aawaz break ho rahi hai, pick up karna padega, dekho sir aaj bahut busy day hai, season time hai na, demand zyada hai, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "अभी नहीं है, ऑर्डर पे मंगवा देंगे, आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "मतलब. हाँ तो स्टॉक में नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "हो मशीन शिल्लक आहे साडे पाच हजार रुपये घरपोच देतो", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ जी, स्टॉक में पड़ा है, रेट चालीस है सर, स्टॉक में सिर्फ 8 पीस है", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "Nahi hai, bolo toh mangwa deta hun, apna aadmi de aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Ho ahe, bagha adichshe rupees", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "हो, बोला दादा। हो, कव्हर आहे। MRP पाचशे सत्तर आहे, तुम्हाला तीनशे साठला देतो।", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "hai, bilkul hai, bhejne me koi problem nahi.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Ho, battery aahe, pathavto, 100 rupaye delivery charge lagel, mhanje vastuche 1450 rupaye, delivery vegli", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "हो, मशीन शिल्लक आहे, तुम्हाला तेराशेला देतो, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "हो, आहे शिल्लक, बाराशे पन्नासला पडेल", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Khadija angle grinder", "said": "abhi confirm nahi kar sakta, godown check karna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "अं मोटरचा स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "आहे... एक मिनिट, नाही, ते काल संपलं, Orient 1200 एमएम सीलिंग फॅन आहे, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient 1200 एमएम सीलिंग फॅन"}}
{"product": "10W-30 engine oil 1 L", "said": "पौने चार सौ फ़ाइनल ऑयल ख़त्म नहीं हुआ है आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "nahi vatta... thamba, aahe, ek urlay, 80 la deto, bagha address pathva, pathavto", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "हैलो हाँ आवाज आ रही है हां 22 पीस पडे हैं रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "कवर ख़त्म हो गया Autofurnish cover वाला दे सकता हूँ 4050 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autofurnish cover"}}
{"product": "Splendor brake shoe", "said": "अच्छा वो आइटम हम नहीं रखते, उसकी जगह Passion brake shoe वाला ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Passion brake shoe"}}
{"product": "Bosch screwdriver bit set", "said": "आमच्याकडे आहे, पावणेचारशेला देतो.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "Haan sir, available hai, apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "nahi bhai, yahan nahi milega, chahiye to Michelin wala de dunga.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बल्ब संपलाय, म्हणजे Philips 5W bulb आहे, चालेल का? संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Philips 5W bulb"}}
{"product": "50 mm padlock", "said": "abhi nahi hai, order diya hai, do din me aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello? haan, awaaz aa rahi hai, bolo available hai aapke liye so apna aadmi de ayega", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Aegon angle grinder", "said": "हां बोला साहेब, स्टॉकमध्ये आहे, आता चौदाशे पन्नास आहे, आधी 1220 होता, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 1450, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "arre abhi confirm nahi kar sakta, godown chek karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "हाँ जी, मिल जाएगा। तेरह सौ है, एक हज़ार एक सौ नहीं", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "आहे ना पावणेदोनशे फिक्स", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "haan haan hai bhej denge sath rupaye delivery charge lagega item teen sau pachas ka hai delivery charge alag", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ तो हाँ, माल पड़ा है। खुला एक दो सौ तीस का, पूरा पैकेट तीन सौ साठ", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "पावणे नऊशेला येतो पण सध्या संपलाय तर", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "milel na, aahe aathshe ainshi hotil bagha kalva mag", "answer": {"in_stock": true, "price_inr": 880, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Haan sir, wo model available hai, aapke liye pandrah, acha free delivery hai, aur kuch?", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "कव्हर पडलंय चारशे पंचेचाळीस होतील होतील", "answer": {"in_stock": true, "price_inr": 445, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "स्टॉक में है सर आपसे १४० लेंगे लिखा १७० है आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "संपले सगळे दिवाळीचीगर्दी आहे ना हां हां ऐकतोय काय सांगू सकाळपासून गर्दी आहे आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हो, कुलूप शिल्लक आहे, पावणेतीनशेला देऊ, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "ho, bulb milel, charshe pastees la deu, chhapil kimmat sahashe panchavann aahe", "answer": {"in_stock": true, "price_inr": 435, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "available hai. teen sao sattar rupaye lagenge. bhej denge, bis rupaye delivery charge lagega", "answer": {"in_stock": true, "price_inr": 370, "delivery": true, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "हाँ जी, मिल जाएगा। पिछले महीने 2390 था, अभी 3050। हम्म लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 3050, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "म्हणजे कव्हर संपलं नाही आहे एमआरपी 195 आहे पण 160 ला लावतो बघा", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "matlab haan, wahi wala hai. market mein maal ki kami chal rahi hai. han haa, sun raha hun. 120... arre nahin, ek sau das.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "हैलो, कौन? आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला, बिजागरी आहे, मिळेल, तीस रुपये होतील, येऊन न्यावं लागेल, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "स्टॉकमध्ये आहे, सव्वा नऊशे होतील होतील, अंसांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "बेस्ट प्राइस साढ़े तीन सौ, हाँ सर, वो मॉडल अवेलेबल है, डिलीवरी हो जाएगी सर", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "देखो हाँ, माल पड़ा है। पंद्रह लगा देंगे, वैसे सोलह का है। आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 15, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "4 inch वाला ख़त्म है, स्टील कब्ज़ा है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टील कब्ज़ा"}}
{"product": "Candes 10 L geyser", "said": "एक मिनट... हाँ बोलिए, हाँ जी, गीज़र रखा है, आपके लिए 2850, एक ग्राहक खड़ा है, एक मिनट, कंपनी ने फिर रेट बढ़ा दिया, आवाज़ कट रही है, ज़ोर से बोलिए, लड़का घर पे दे जाएगा, त्योहार का सीज़न है ना.", "answer": {"in_stock": true, "price_inr": 2850, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "जी, उपलब्ध है, अच्छा भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग ख़त्म हो गया, local spark plug पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "10W-30 engine oil 1 L", "said": "आहे ना, रेट तीनशे आहे, पाठवायची सोय नाही, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "han haan, hai, chaar hajar panch sau pachas hai, sorry sorry, panch hajar ek sau, bhejne ka intezaam nahi hai.", "answer": {"in_stock": true, "price_inr": 5100, "delivery": false, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "जी, उपलब्ध है। आपके लिए 1300। डिलीवरी हो जाएगी सर।", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हां जी मिल जाएगा २५ का डिब्बा ३७६० का एक पीस १६५", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "हां तर तुम्हाला पावणे दोनशेला देतो। हो, मिळेल। पाठवायची सोय नाही। कळवा मग।", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Bosch drill machine", "said": "ho, ahe aamchyakade, tumhala 9100 la deto.", "answer": {"in_stock": true, "price_inr": 9100, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1400 rupees, kami nahi honar, ho, wire ahe", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "ख़त्म नहीं हुआ, अभी भी है, प्राइस साढ़े सात सौ है", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "संपले सगळे, 8 mm bolt आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 mm bolt"}}
{"product": "Cheston 200A welding machine", "said": "हाँ हाँ जी, मिल जाएगा। मतलब आपके लिए 4050। iBELL 200 amp welding machine भी है, वो 3290 का", "answer": {"in_stock": true, "price_inr": 4050, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हाँ जी sir, बताइए, हाँ sir, available है, एक ग्राहक खड़ा है, एक मिनट, एक हज़ार लगा देंगे, वैसे 1120 का है, कंपनी ने फिर rate बढ़ा दिया, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, एड्रेस भेजो, डिलीवर कर देंगे, और कुछ?", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "हां बोला हेल्मेट संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "स्टॉकमध्ये आहे। तुम्हाला १५०० ला देतो।", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "वो स्टॉक में है. 2380 का पड़ेगा. डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 2380, "delivery": false, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "haan bhaiya, bataiye, arre ruko, dekh ke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "है भाई, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो बिजागरी आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "han, hai apne paas, 462 last rate hai, bhej denge, 20 rupaye delivery charge lagega.", "answer": {"in_stock": true, "price_inr": 462, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "स्टॉक में है सर, आपको नौ सौ पचानवे में दे देंगे, MRP एक हज़ार तीन सौ पचास है, होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 995, "delivery": true, "alternative": null}}
{"product": "Schneider Classy switch", "said": "हाँ बोलिए, हम्म पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "सध्या नाही मागवावं लागेल कुरिअर करतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "Arre abhi khatam hai, shaam tak aa jayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "Han, kaun? acha, bolo hai bhai MRP 600 likha hai, 399 laga denge", "answer": {"in_stock": true, "price_inr": 399, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "अं जी उपलब्ध है सॉरी डिलीवरी सर्विस नहीं है देखो औरकुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "एक मिनट... हाँ बोलिए। माल खत्म है। उसकी जगह Polycab 2.5 sq mm wire दे सकते हैं। कूरियर कर देंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Polycab 2.5 sq mm wire"}}
{"product": "9W LED bulb", "said": "pickup करावा लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "नहीं है भाई खत्म हो गया उसकी जगह Sage Square हेलमेट दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sage Square हेलमेट"}}
{"product": "fan capacitor 2.5 mfd", "said": "हां जी बताइए मिल जाएगा है अपने पास आपसे सौ लेंगे लिखा एक सौ बीस है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हैलो? हाँ, आवाज़ आ रही है. है भाई. पचास में दे देंगे. लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हो दादा, आहे, 290 होतील, डिलिव्हरी फ्री आहे, अजून काही?", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "आमच्याकडे आहे. 555 फायनल. delivery नाही, शॉपवरून pickup करा.", "answer": {"in_stock": true, "price_inr": 555, "delivery": false, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "हाँजी. पंखा ख़त्म हो गया. Havells 1200 mm fan वाला है, चलेगा? खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Havells 1200 mm fan"}}
{"product": "Kaltron 8 inch plier", "said": "plas khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "हां है तीन हज़ार तीन सौ सॉरी सॉरी साढे पच्चीस सौ", "answer": {"in_stock": true, "price_inr": 2550, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "थोड़ी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हाँ सर, वो मॉडल अवेलेबल है, फुल पैकेट तीन सौ पचासी, लूज़ पीस दो सौ पंद्रह का, डिलीवरी नहीं होती, पिकअप करना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 385, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "है ना, कितने चाहिए? साढ़े चार सौ में दे देंगे, अं भेजने का इंतज़ाम नही है.", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Windek tyre", "said": "हो दादा, आहे, आता रेट 960 झाला, मागच्या महिन्यात 1180 होता, संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": 960, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "aahe ahe, solashe rupees hotil, delivery deto.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "ladka dekh ke batayega, ruko zara, ladka chai lene gaya hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "होम डिलीवरी हो जाएगी", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Sage Square helmet", "said": "han haan, hai, matlab chaar sau sattar laga denge, waise MRP chhah sau pachis likha hai", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Ho bola, motor ahe, milel, 1200 rupayanna padel.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "haa, ready stock hai, haan toh aapko shop se pickup karna hoga.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "नमस्ते, बोलिए, घर पर भेज देंगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "हाँ, पंखा है, पौने बारह सौ लगेगा, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "milel na, aahe, price 180 ahe", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है Syska 7W LED bulb भी है उसका निन्यानवे और आपवाला सवा सौ का", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "है भाई मिलजाएगा हम्म एक का पचास, बारह का डिब्बा पाँच सौ पच्चीस का अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "hai ji, mil jayega, chaar so pachasi last rate hai, Sonik 7 helmet chahiye toh wo bhi hai, chaar so das wala, delivery boy hai, bhej denge", "answer": {"in_stock": true, "price_inr": 485, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "आपके लिए साढ़े तीन सौ, है भाई, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "तर AllExtreme नाहीये, Speedwav हेडलाईट आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav हेडलाईट"}}
{"product": "Crompton 3 L geyser", "said": "हाँ हाँ, रखा है. वो एओ स्मिथ 15 L geyser का रेट 2820 है, ये वाला बाईस सौ पचास", "answer": {"in_stock": true, "price_inr": 2250, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पड़ी है, सात सौ अस्सी का है, इससे कम नहीं होगा, अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 780, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bola kay pahije rate tinashe daha ahe pan aatta stock nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ बोलिए मिल जाएगा है अपने पास सौ रुपये लगेंगे कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "बल्ब पड़ा है, MRP 395 है पर तीन सौ पच्चीस लगा देंगे", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Haan haan, bolo, mil jayega, hai, aapse sadhe barah sau lenge, likha ek hajar panch sau pachas hai, sham tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "जी, उपलब्ध है, एक पीस एक सौ साठ, दस लोगे तो 130 के हिसाब से, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "मिल जाएगा है अपने पास लिए तीन सौ अस्सी डिलीवरी नही है", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "उद्या सकाळी माल येईल, आत्ता नाहीये। घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "ha ji, boliye, woh hum nahi rakhte, awaaz kat rahi hai, zor se bolo, haa haan, sun raha hun", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप संपली नाही आहे अं सुटा वीसला पन्नास नगांचा बॉक्स घेतला तर ८६०", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sorry, out of stock ahe. mhanje Birla wall putty 40 kg aahe, tyacha price pavanenaushe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Birla wall putty 40 kg"}}
{"product": "Usha 1000W immersion rod", "said": "वो मॉडल अवेलेबल नहीं है Crompton 1500W immersion rod वाला अवेलेबल है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 1500W immersion rod"}}
{"product": "Raaisin car cover", "said": "प्लीज़ थोड़ी देर बाद कॉल करो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हैलो कौन कंपनी ने फिर रेट बढ़ा दिया अभी नहीं है लड़का चाय लेने गया है एक ग्राहक खड़ा है एक मिनट त्योहार का सीज़न है ना और कुछ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "tube padi hai, paune do sau ka milega, arre dukan 9 baje tak khuli hai, home delivery nahi hai bhai.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "Raaisin car cover", "said": "यस सर बोलिए हाँ कवर है अं MRP 5290 है बट आपको चौवालीस सौ पचास में दे देंगे मतलब", "answer": {"in_stock": true, "price_inr": 4450, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "है भाई, मिल जाएगा। आपसे पौने चार सौ लेंगे, लिखा चार सौ पचहत्तर है। डिलीवरी बंद है अभी, दुकान से ले लो।", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "yes, available hai. price sava terah sau hai. delivery ka pachas extra", "answer": {"in_stock": true, "price_inr": 1325, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "Udya phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "सव्वासातशे रुपये आहे आहे येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 725, "delivery": false, "alternative": null}}
{"product": "SARK tap", "said": "hai, bilkul hai, aapse panch so pachanve lenge, likha saat sao pandrah hai, delivery nahi hai.", "answer": {"in_stock": true, "price_inr": 595, "delivery": false, "alternative": null}}
{"product": "SARK tap", "said": "सहाशे पंचवीसला देतो हो आहे शिल्लक", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Crispy LED torch", "said": "है ना कितने चाहिए 355 है 285 नहीं", "answer": {"in_stock": true, "price_inr": 355, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "हेलो? हाँ, बोलो भाई, अभी स्टॉक नहीं है, कल तक आ जाएगा, अं सर आज बहुत बिज़ी डे है, सीज़न टाइम है ना, डिमांड ज़्यादा है, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, एक कस्टमर खड़ा है, होल्ड करो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "हां बोला... नाही, ते सध्या नाही, Racold 3 L गिझर घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Racold 3 L गिझर"}}
{"product": "wall putty 20 kg", "said": "हाँ भाई बोलो गलत नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "हो आहे बघा तर रेट पावणे बाराशे आहे कळवा मग", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "han, wahi vala hai, sattais sau last rate hai.", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "हाँ, माल पड़ा है. ५४० MRP लिखा है, ३३० लगा देंगे. हम्म शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "जी बोलिए। हाँ जी, प्लेट रखी है। आपके लिए पौने नौ सौ", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "एक मिनट हां बोलिए जी उपलब्ध है पौने दो सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "Tyre rakha hai, mil jayega, 3000 last rate hai, counter pe aake le lo, theek hai?", "answer": {"in_stock": true, "price_inr": 3000, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "यस सर बोलिए हाँ जी स्टॉक में पड़ा है देखो साढ़े चार सौ रुपीज़ है सॉरी डिलीवरी सर्विस नहीं है", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "एक कस्टमर खड़ा है, होल्ड करो, सर आज बहुत बिज़ी डे है, हाँ सर, अवेलेबल है, डिलीवरी हो जाएगी, 100 रुपीज़ चार्ज एक्स्ट्रा, आइटम प्राइस 1220, डिलीवरी चार्ज अलग, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, अच्छा.", "answer": {"in_stock": true, "price_inr": 1220, "delivery": true, "alternative": null}}
{"product": "HTC multimeter", "said": "हाँ, है, देखो छह सौ पचास रुपये, घर पर भेज देंगे, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हो, ठेवतो आम्ही, आहे। चारशे रुपयांना पडेल। घरपोच नाही, तुम्हाला यावं लागेल।", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "है ना, कितने चाहिए? भेज देता, पर आज बॉय छुट्टी पे है, आके ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "7Ah वाला फिनिश है, Exide 12V 5Ah battery वाला available है, होम delivery हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Exide 12V 5Ah battery"}}
{"product": "Duro Gold angle grinder", "said": "पता भेज दो लडका दे आएगा", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "paanch piece available hai abhi acha ek sau pachas rupees lagenge", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "बघा हो सर, अवेलेबल आहे, एक हजार होतील, घरपोच करून देतो", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हाँ तो हाँ जी कवर रखा है 1150 लगा देंगे वैसे 1270 का है", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "६० रुपीज़ है है भाई मिल जाएगा", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "abhi nahin hai, parso tak aa jayega. ek sau sattar hoga. apna aadmi de ayega", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "डिलीवरी हो जाएगी सर, देखो और कुछ?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "geyser rakha hai, mil jayega, ghar pe bhej dunga, 80 lagega bhejne ka, item pauna baais sau ka hai, delivery charge alag", "answer": {"in_stock": true, "price_inr": 2175, "delivery": true, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "बल्ब ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ, पुट्टी है. नौ सौ पचास लगेगा. होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Stanley angle grinder", "said": "hello, bola na, manus pathavto ghari.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "हो टेप आहे बरं 15 फिक्स डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ भाई, बोलो, मिल जाएगा, है अपने पास, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "नमस्ते, बोलिए, हाँ हाँ तो हाँ, है, छः हज़ार में दे देंगे, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 6000, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "pata nahi, godown me dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bhaiya, bataiye, haan ji, tala available hai, dekho sava do sau rupaye, ghar tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "स्टॉक में है, GST मिलाकर दो हज़ार, दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "नमस्ते, बोलिए। सात सौ रुपये का है। मतलब हाँ, हेलमेट है", "answer": {"in_stock": true, "price_inr": 700, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "आज नहीं है, कल आएगा. तीन सौ नब्बे रुपये का है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "स्टॉक में है। अपना प्राइस 110, एमआरपी 140 है। अं सॉरी, डिलीवरी सर्विस नहीं है।", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "हाँ जी, बताइए, Modx नहीं है, V-GRIP स्टीयरिंग कवर है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-GRIP स्टीयरिंग कवर"}}
{"product": "Motul 20W-40 engine oil", "said": "oil sampala nahi, aahe, aapla rate saade teenshe, MRP teenshe sattar aahe", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "हां तर हो दादा, आहे, साडे सहा हजारला लावतो, एमआरपी 7270 आहे", "answer": {"in_stock": true, "price_inr": 6500, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हेलो हाँ बोलो भाई पौने दो सौ हाँ है होम डिलीवरी अवेलेबल है", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "हाँ, रखा है, लास्ट मंथ 695 था, अब आठ सौ पैंसठ है", "answer": {"in_stock": true, "price_inr": 865, "delivery": null, "alternative": null}}
{"product": "iBELL drill machine", "said": "हो, मशीन आहे, तुम्हाला 2350 ला देतो, माणूस पाठवून देतो घरी.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हाँ जी, हेलमेट रखा है, छः सौ पचहत्तर का... नहीं नहीं, गलत बोला, पौने छह सौ का है, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 575, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ha, kaun? acha, bolo, haa, plug hai, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "लाईट आहे, मिळेल. हां तर सव्वासहाशे रुपये. Leebo एलईडी headlight हवं असेल तर ते पण आहे, पाचशे पंचाण्णव. घरपोच पोहोचवून देऊ.", "answer": {"in_stock": true, "price_inr": 625, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "हम्म, delivery बंद है अभी, दुकान से ले लो", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells Oro switch", "said": "नहीं मिलेगा भाई।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है। मतलब AUTOSiTY हैंडल ग्रिप वाला available है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AUTOSiTY हैंडल ग्रिप"}}
{"product": "50 mm padlock", "said": "हाँ भाई बोलो हाँ सोलह पीस पड़े हैं एक सौ पचास रुपये", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "यस, अवेलेबल है, अं 520 रुपये.", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर नहीं है, खत्म हो गई, Finolex 1 sq mm wire वाला पड़ा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Finolex 1 sq mm wire"}}
{"product": "Lakshmi 0.5 HP water pump", "said": "motor sampli, udya sakali maal yeil, aatta nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "ahe na, pavne tinashe rupees hotil", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "नहीं है रुको रुको है एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "हॅलो? आवाज येतोय का? हां, बोला, तुम्हाला चारशेला देऊ, म्हणजे हो सर, अवेलेबल आहे", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "ho bola, atta nahiye, Avita switch wala ahe, tyache donshe pandhra rupaye, pathavaychi soy nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Avita switch"}}
{"product": "Bajaj immersion rod", "said": "है भाई. आपसे 455 लेंगे, लिखा 585 है. अच्छा आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 455, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "ho, aahe aamchyakade, rate panchavann ahe, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "arre haan, wahi wala hai, rate seth ji batayenge.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "aahe ahe, ek piece donashe sath, bara ghetle tar donshe pandhara pramane lavto", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "nahi nahi, aahe, shambhar rupayanna padel.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "Hello, haa bolo, wo item nahi rakhte hum, Vega helmet wala available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega helmet"}}
{"product": "12V 7Ah battery", "said": "Hai, bilkul hai, 1210... arey nahi, 1300", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "रेट सवा सत्रह सौ है है ना कितने चाहिए", "answer": {"in_stock": true, "price_inr": 1725, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "Haan bhai, bolo, nahi bhai, yahan nahi milega, uski jagah Autoplus lock wala de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autoplus lock"}}
{"product": "VVM bike cover for Honda", "said": "arre hai bhai, teen sau assi ka hai, isse kam nahi hoga.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "sorry sir abhi out of stock hai arre company ne is mahine phir rate badhaya dr.fixit silicone sealant available hai chalega are kya bataun subah se bahut rush hai aaj subah se light nahi hai pichle hafte bhi kisi ne yahi poocha tha aur kuch", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "dr.fixit silicone sealant"}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए sir हाँ है price कन्फर्म करके बताता हूँ डिलीवरी होजाएगी sir अं", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "aatta nahiye, sandhyakali yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो, बोला दादा, हां तर ऑइलचा स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "available hai, paune teen sau final", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "haan, maal pada hai, rate badh gaya, 925 se barah sau ho gaya", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "है ना कितने चाहिए वो 10 लोगे तो 190 के हिसाब से एक का 235", "answer": {"in_stock": true, "price_inr": 235, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "haan sir, wo model available hai, home delivery available hai, 50 charge, product ka 355, delivery extra.", "answer": {"in_stock": true, "price_inr": 355, "delivery": true, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "हैलो, कौन? हाँ, माल पड़ा है, एमआरपी 550 है, डिस्काउंट के बाद साढ़े तीन सौ पड़ेगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "nahi hai, sorry, Havells switch hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells switch"}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "जी उपलब्ध है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb hai mil jayega 140 last rate hai ghar pe bhej dunga 20 lagega bhejne ka", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "पंधरा रुपयांना पडेल. स्टॉकमध्ये आहे.", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "अभी 5 पीस ही बचे हैं, बारह सौ में दे देंगे देंगे.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "हाँ जी, बल्ब रखा है। डब्बा चार हज़ार का है, बीस पीस वाला, एक का दो सौ चालीस।", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए सर। हाँ, बल्ब है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "बघा, आहे आहे की, मिळेल, 50 घेतले तर चव्वेचाळीस प्रमाणे, एक पीस पन्नास", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "हाँ है, कोई प्रॉब्लम नहीं, ढाई हज़ार रुपये लगेंगे, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ सर, वो मॉडल अवेलेबल है, सवा सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर संपली। हॅवेल्स 1 स्क्वेअर mm वायर आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हॅवेल्स 1 स्क्वेअर mm वायर"}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हो दादा, आहे, सव्वा चारशे फायनल", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "अं हाँ जी, मिल जाएगा, साढे उनतीस सौ में दे देंगे, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 2950, "delivery": false, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "aahe, ajun daha piece padlet, pickup karava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए। हाँ, है। अं खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Carzex LED headlight", "said": "जी, उपलब्ध है, 1800 रुपये का है, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Dewalt drill machine", "said": "मशीन खत्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे सर रेडी स्टॉक आहे बघा प्राइस पंचाहत्तर आहे सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "मतलब हां रखते हैं पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "बोला ना। पावणेदोनशे रुपये। आहे की, मिळेल। पोरगा घरी सोडून देईल। बरं कळवा मग।", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "एक मिनट... हाँ बोलिए. स्टॉक में है. 475 रुपये लगेंगे. खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "वो क्या है हाँ सर, वो मॉडल अवेलेबल है, पैंतीस का है है, कूरियर कर देंगे.", "answer": {"in_stock": true, "price_inr": 35, "delivery": true, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "pankha sampala nahi, ahe. delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "हाँ है बेस्ट प्राइस ४९० शॉप ७ बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 490, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हेलो? हाँ, बोलो भाई, किससे बात करनी है? यहाँ कोई दुकान नहीं है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हाँ, बल्ब है. दस का डिब्बा दो हज़ार दो सौ का, एक पीस दो सौ चालीस.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "umm machine khatam nahi hui, hai, 1250 rupees hai, delivery nahi hoti aisa nahi hai, bhej dunga.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "arre abhi nahi, order pe mangwa denge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "Acha ha, stock me hai company ka rate ek hajar teen sau hai, main gyara sau lunga", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "rate sava chaar sau hai. haan, wahi wala hai. bhijwa dunga. aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "हाँ जी, स्टॉक में पड़ा है, साढ़े पैंतालीस सौ फाइनल प्राइस है, वो क्या है हाँ, इक्कीस पीस पड़े हैं, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 4550, "delivery": true, "alternative": null}}
{"product": "Schneider Classy switch", "said": "देखिए जी, उपलब्ध है, एमआरपी ८७० है, पर हम ५६० में देते हैं, भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 560, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "hai, bilkul hai dekho sava teen sau plus GST.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "haan bola, kay pahije? milel na, aahe, gharpoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "सध्या नाहीये, ऑर्डर दिलीय, दोन दिवसांत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नमस्ते, बोलिए, हाँ हाँ, है, सौ लगेगा, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "ATC bike cover", "said": "बोलिए बोलिए। स्टॉक में है। 152 लगेगा", "answer": {"in_stock": true, "price_inr": 152, "delivery": null, "alternative": null}}
{"product": "R-tek multimeter", "said": "स्टॉक में है ४८५ का नहीं नहीं गलत बोला पौने चार सौ का है शाम तक घर पहुँच जाएगा", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "हाँ रखा है एक पीस दो सौ नब्बे, दस लोगे तो दो सौ पचास के हिसाब से", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "है भाई मिल जाएगा एक पीस दो सौ पच्चीस, दस लोगे तो एक सौ नब्बे के हिसाब से", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "bara stock nahi aahe sadhya paint roller aahe aamchyakade", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "Legrand Mylinc 6A switch", "said": "Umm ji ha, rakha hai, best price 150.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब ख़त्म हो गई Duravolt LED बैटन है चलेगा पता भेज दो लड़का दे आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Duravolt LED बैटन"}}
{"product": "Daybetter 25W LED bulb", "said": "hai, bilkul hai, acha das ka dabba 2510 ka, ek piece paune teen sau.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "हाँ जी, स्टॉक में पड़ा है, चार सौ पाँच... नहीं नहीं, चार सौ है, वो delivery हो जाएगी sir", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "देखो हाँ, रेडी स्टॉक है. रेट दो सौ पचानवे हैसर", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हैलो आवाज़ नहीं आ रही बाद में करना त्योहार का सीज़न है ना बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "Pankha ka stock check karke batata hun kya hai na market mein maal ki kami chal rahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हां बोला साहेब. म्हणजे पिकअप करावा लागेल.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan, hai apne paas, aapke liye sau.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "हॅलो, कोण बोलतंय? हो, आहे शिल्लक। एक गिऱ्हाईक आहे, एक मिनिट। काय सांगू, सकाळपासून गर्दी आहे। आता रेट साडे पंचेचाळीसशे झाला, मागच्या महिन्यात तीन हजार सातशे होता", "answer": {"in_stock": true, "price_inr": 4550, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हो, पक्कड आहे. बरं तुम्हाला अडीचशेला देतो", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "Diwali chi gardi aahe na, tumhala sadecharshe la deto, haan ahe, kal Ravivar hota mhanun dukan band hota, ek giraik aahe, ek minute, haan haan, aiktoy, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai ji, mil jayega, price paune do sau hai, address bhej do, ladka de aayega", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Ek minute... ha bolo aapko number kisne diya? acha thik hai haan hai sao ka milegaa market me maal ki kami chal rahi hai ladka chai lene gaya hai bata dena", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "स्टॉक में है पर पीस दो सौ बीस, 10 लो तो 200 लगेगा एड्रेस भेजो डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "umm ha, motor hai, delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "1 inch संपलं, सुप्रीम 3/4 इंच एल्बो वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सुप्रीम 3/4 इंच एल्बो"}}
{"product": "Aryshaa LED torch", "said": "हाँ बोलिए. मिल जाएगा, है अपने पास. एक ग्राहक खड़ा है, एक मिनट. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. लड़का चाय लेने गया है. 300 का पड़ेगा. 9 पीस हैं स्टॉक में. और कुछ?", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Easy4buy horn", "said": "हाँ भाई, बोलो. रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हम्म नही नही, है ना, दो सौ का है", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब पडलाय, एका नगाचे पावणे दोनशे, बारा नगांचा बॉक्स 1750 ला, डिलिव्हरी नाही, दुकानात येऊन घ्या, बघा.", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "ATC bike cover", "said": "अभी 17 पीस ही बचे हैं, लड़का घर पे दे जाएगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ है, डेढ़ सौ रुपीज़ है, सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "available hai, ek sau assi hai, sorry sorry, pone do so", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अं सर एक मिनट, चेक करके बताता हूँ, सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हां बोला साहेब, आत्ता मिळत नाही, विप्रो 15 वॉट बल्ब आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "विप्रो 15 वॉट बल्ब"}}
{"product": "Activa air filter", "said": "हेलो, हाँ बोलो। 5 पीस हैं स्टॉक में। रेट पूछ कर बताता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "स्टॉक में है, प्रिंटेड MRP ४५५, अपना प्राइस २५५", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "नळ आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ हाँ, है, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, कंपनी ने फिर रेट बढ़ा दिया, एक सौ दस फ़ाइनल, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं, तर त्याच्याऐवजी Godrej 60 mm padlock देऊ शकतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Godrej 60 mm padlock"}}
{"product": "CarMobCentral car cover for Ritz", "said": "हेलो? हाँ, बोलो भाई। वो आइटम हम नहीं रखते।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "हो, कव्हर आहे, तर हजार फिक्स, दुकान आठ वाजेपर्यंत उघडं आहे.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "नहीं है जी ख़त्म भाई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "Namaskar, bola, aahe ki, milel, bara kimmat nantar sangto, aatta list nahi, dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "यस available है ३१० का एक, ६ का डब्बा १६८० का डिलीवरी हो जाएगी sir", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "हां भाई, बोलो, अभी बिजी हूं, कॉल बैक करता हूं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "tyre ka stock check karke batata hun, awaaz kat rahi hai, zor se bolo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "ऑइल संपलं नाही, आहे. 1650 ला पडेल. Castrol 10W engine oil पण आहे, ते 1370 ला.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "nahi asa nahi, ahe, pathavaychi soy nahi.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "wo item nahi rakhte hum.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "बेसिकली अभी stock नहीं है, कल तक आजाएगा, delivery बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "sampala bajaj 600 mm ceiling fan chalnar asel tar sanga gharpoch nahi tumhala yava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "bajaj 600 mm ceiling fan"}}
{"product": "Apex Vivid helmet", "said": "हेलमेट नहीं है, खत्म हो गया, स्टीलबर्ड 7 हेलमेट पड़ा है, वो ले लो, शाम तक पहुँचा देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "स्टीलबर्ड 7 हेलमेट"}}
{"product": "Atomberg exhaust fan", "said": "bara ho ho, samajla, pan te nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "वो मॉडल available नहीं है, Amaron 12V 9Ah battery वाला चलेगा? उसका नौ सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "12V 7Ah battery", "said": "ha ha samajh gaya lekin wo abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "Motor sampli nahi, aahe, hajar hotil, delivery? nahi ho, te nahi karat.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": false, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "हाँ जी, बताइए, हाँ हाँ, है, पहले दो सौ नब्बे था, अब सवा तीन सौ हो गया", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "हेलो, हाँ बोलो, पंखा ख़त्म नहीं हुआ, है, आपको साढ़े अठारह सौ, MRP तो दो हज़ार तीन सौ पचास है.", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "Haan tar ho, milel, sadecharashe rupaye hotil, porga pathavto, dukan saat vajeparyant ughda ahe.", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Wo model available nahi hai, Havells 2.25 mfd capacitor available hai, uska price assi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 2.25 mfd capacitor"}}
{"product": "Atomberg exhaust fan", "said": "पंखा पड़ा है. एक हज़ार नौ सौ का... नहीं नहीं, गलत बोला, सवा तेईस सौ का है. लड़का भेज देता हूँ.", "answer": {"in_stock": true, "price_inr": 2325, "delivery": true, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "कवर का स्टॉक चेककरके बताता हूँ हम्म", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Ji, available hai, aapke liye dedh sau, aapke ghar tak bhijava denge, aur kuch?", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "जी बोलिए रुकिए देख कर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "माल खत्म है, लड़का चाय लेने गया है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "wrong number ahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "बघा वायर पडलीये २१०० होतील", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन पड़ी है, 1950 का है है, फ्री डिलीवरी है, और कुछ?", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हां जी, मिल जाएगा, रेट मालिक बताएँगे, मतलब, लड़का घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "हाँ हाँ, है, अं लिस्ट में एक हज़ार तीन सौ पचास है, आपको पौने बारह सौ पड़ेगा, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "है ना, कितने चाहिए? ठीक है? तो", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan ahe, 395... nahi nahi, 470, home delivery aahe", "answer": {"in_stock": true, "price_inr": 470, "delivery": true, "alternative": null}}
{"product": "Speedwav handle grip", "said": "एक मिनट... हाँ बोलिए. लड़का भेज देता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan ahe, Havells 32A MCB cha rate 225, ha 230 la, delivery nahi karat.", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "बोला, काय पाहिजे? मोटर संपली नाही, आहे, अं साडे बारा हजारला पडेल, घरपोच देतो.", "answer": {"in_stock": true, "price_inr": 12500, "delivery": true, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "बघा रेट दोनशे वीस आहे। हो, आहे। घरपोच पोहोचवून देऊ।", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हेलो? हाँ, बोलो भाई, नेटवर्क इशू है, आवाज़ ब्रेक हो रही है, एक कस्टमर खड़ा है, होल्ड करो, अभी available नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा, सर आज बहुत बिज़ी डे है, सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "हां हां, ऐकतोय। V-Guard नाहीये, उषा 230 एमएम एक्झॉस्ट फॅन आहे। तर आवाज तुटतोय, मोठ्याने बोला। काय सांगू, सकाळपासून गर्दी आहे। कंपनीने परत रेट वाढवले", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "उषा 230 एमएम एक्झॉस्ट फॅन"}}
{"product": "iBELL drill machine", "said": "हो, ठेवतो आम्ही, आहे. चोवीसशे पन्नासला लावतो, एमआरपी तीन हजार एकशे पन्नास आहे.", "answer": {"in_stock": true, "price_inr": 2450, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan tar nahiye bahutek... ho ho, sapadla, ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "हाँ हाँ सर, अवेलेबल है, बेसिकली लिस्ट प्राइस 1480, आपको 1200 पड़ेगा", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "नमस्ते, बोलिए. ऑयल है, मिल जाएगा. कंपनी का रेट 550 है, मैं सवा चार सौ लूँगा", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "matlab 7Ah nahi hai, Amaron 12V 5Ah battery hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 5Ah battery"}}
{"product": "teflon tape", "said": "नाही ना, संपलंय, big teflon roll चालणार असेल तर सांगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Osram LED headlight", "said": "हाँ भाई, बोलो, देखो माल खत्म है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "है ना कितने चाहिए दो सौ सत्तर रुपये पता भेज दो लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": 270, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Owner abhi available nahin hain, bad mein call kijiye, acha aapko number kisne diya? acha thik hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Easy4buy horn", "said": "यस sir, बोलिए, है ना, कितने चाहिए? 599 का है, इससे कम नहीं होगा, sorry, delivery सर्विस नहीं है.", "answer": {"in_stock": true, "price_inr": 599, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "स्टॉक में है अस्सी और १८ परसेंट GST अलग दुकान से ले जाना पडेगा", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "haa bhaiya, bataiye, aapko number kisne diya? acha thik hai, bulb khatam ho gaya, bhaiya aap ruko do minute, phone pe hun, Diwali ka season hai na, sab yahi maang rahe hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, आहे। बरं पंधरा रुपये होतील। कुरिअर करतो। सांगा काय करायचं।", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "एक सौ पचास लगेगा। मिल जाएगा, है अपने पास। बताओ क्या करना है", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "हाँ जी, स्टॉक में पड़ा है, सवा इकतालीस सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 4125, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Plate khatam nahi hui, hai sadhe saat sau mein de dunga self pickup hai, delivery nahi.", "answer": {"in_stock": true, "price_inr": 750, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Wire rakhi hai, mil jayegi, soleh sau ka padega, 8 baje dukaan band hoti hai, aake le jao", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ, माल पड़ा है। ढाई सौ लगा देंगे, वैसे दो सौ नब्बे का है।", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "यस अवेलेबल है खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "हाँ, रखा है. सात सौ तीस इन्क्लूसिव ऑफ GST. एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 730, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "यस, अवेलेबल है। 10 का फुल बॉक्स 2340, सिंगल पीस 275। मतलब आके ले जाओ", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "मिल जाएगा, है अपने पास, अब 230 है, पहले 270 था.", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai bhai paune tin sau ka hai", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "स्टॉक में सिर्फ नौ पीस है रेट अभी पता नहीं ओनर से पूछना पड़ेगा लड़का घर पे दे जाएगा और कुछ चाहिए सर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हाँ जी, बताइए. लड़का चाय लेने गया है. एक ग्राहक खड़ा है, एक मिनट. कंपनी ने फिर रेट बढ़ा दिया. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. हाँ, रखा है. आपसे 1250 लेंगे, लिखा 1540 है.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "नमस्ते, बोलिए, छह सौ पैंसठ का है, इससे कम नहीं होगा, हाँ, माल पड़ा है", "answer": {"in_stock": true, "price_inr": 665, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "है ना, कितने चाहिए? 400 और 18 परसेंट GST अलग. हम्म दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "haan ji, boliye, sadhe panch sau lagega, haan haan, hai, abhi delivery band hai.", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हो, बोला दादा. हो, ट्यूब शिल्लक आहे. तुम्हाला पावणेचारशेला देतो, MRP 670 आहे", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "हाँ हाँ, है, चार हज़ार पाँच सौ पचास फाइनल प्राइस है, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 4550, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "जी, उपलब्ध है। देखो 805... नहीं नहीं, 800 है", "answer": {"in_stock": true, "price_inr": 800, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "अच्छा, है भाई, आपके लिए पौने बारह सौ.", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "आउट ऑफ stock नहीं है, अवेलेबल है, मतलब तीन सौ पैंतीस लगेगा", "answer": {"in_stock": true, "price_inr": 335, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अं अभी बिज़ी हूँ, call बैक करता हूं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हाँ जी मशीन रखी है आपके लिए पौने छह हज़ार घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 5750, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "अच्छा ऑयल पड़ा है आपको 490 MRP तो 530 है", "answer": {"in_stock": true, "price_inr": 490, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "Arre han haan, samajh gaya, lekin wo abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "अं वो क्या है हाँ जी मोटर रखी है भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Bola dada, battery sampliye, Amaron 12V 9Ah battery aahe, same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "Dow silicone sealant", "said": "हाँ भाई, बोलो। अभी नहीं है। त्योहार का सीज़न है ना।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "हाँ जी, मिल जाएगा, चार सौ पचहत्तर लगा देंगे, वैसे MRP 760 लिखा है, आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "han sir, wo model available hai panch so bees final price hai apna aadmi de ayega", "answer": {"in_stock": true, "price_inr": 520, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "लाईट आहे, मिळेल। पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "जी, उपलब्ध है। सवा नौ हजार प्लस GST", "answer": {"in_stock": true, "price_inr": 9250, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "अच्छा हाँ रखा है एक सौ तीस का है आपके लिए एक सौ बीस", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "यस सर बोलिए हाँ है बेस्ट प्राइस पच्चीस", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "मशीन नहीं है, खत्म हो गई। Khadija angle grinder अवेलेबल है, उसका प्राइस पौने बारह सौ। लडका छोड देगा घर पर", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Khadija angle grinder"}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हाँ भाई, बोलो, है भाई, 2520 का है, आपके लिए 2050, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 2050, "delivery": true, "alternative": null}}
{"product": "Goldcave tap", "said": "हाँ जी, मिल जाएगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "प्राइस पौने पाँच सौ है, हाँ जी, कवर रखा है", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "हाँ है प्राइस 270 डिलीवरी की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": 270, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "यस सर, बोलिए, हाँ सर, अवेलेबल है, सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Plate sampli. Bosch Jupiter clutch plate ahe, chaalel ka? dukanatun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Bosch Jupiter clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "तीन सौ दस लगेगा, स्टॉक में है, घर पर भेज देंगे, और कुछ?", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हो sir बोला बघा स्टॉकमध्ये आहे तुम्हाला पावणे चारशेला देऊ कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर है, मिल जाएगी, सोलह सौ रुपये, खुद आकर उठाना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "माल खत्म है सर.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "हो, टायर आहे, 1200 ला लावतो, एमआरपी 1400 आहे", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "namaskar, bola, wire padliye, bagha delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "हेलो, हां बोलो। तो। बेस्ट प्राइस सवा पाँच सौ। अं जी, उपलब्ध है। और कुछ?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "motor rakhi hai mil jayegi dekho mrp 12550 hai par sadhe gyarah hajar laga denge", "answer": {"in_stock": true, "price_inr": 11500, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "हां। संपलं होतं... नाही, एक आहे अजून", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म नहीं हुई, है. पौने चौदह सौ का है, इससे कम नहीं होगा. और कुछ?", "answer": {"in_stock": true, "price_inr": 1375, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हां हो, आहे. तीनशे वीसला देऊ, छापील किंमत चारशे दहा आहे. कुरिअर करतो.", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "सव्वाबावीसशेला देतो, हां आहे ना", "answer": {"in_stock": true, "price_inr": 2225, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "available hai rate do so sattar hai delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 270, "delivery": false, "alternative": null}}
{"product": "Retina bike cover", "said": "नमस्कार, बोला, बघा आत्ता सांगता येणार नाही, संध्याकाळी फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "है ना, कितने चाहिए? आपके लिए पौने चार सौ।", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "हो, ठेवतो आम्ही, आहे, एकशे पंचेचाळीस आहे, 170 नाही", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "लड़का भेज देता हूँ.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Stock madhe aahe, 240 la padel, dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हां तो आपने गलत नंबर मिलाया है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "Oil padlay, kay aahe na 695 plus GST.", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हाँ तो आपके लिए 30, हाँ sir, वो मॉडल available है, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "Diwali chi gardi aahe na, haan tar ho, thevto aamhi, ahe, kal Ravivar hota mhanun dukan band hota, rate sadechaudashe ahe sir, self pickup aahe, aawaj tutatoy, motyane bola", "answer": {"in_stock": true, "price_inr": 1450, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Dekho haan, maal pada hai. home delivery available hai, 40 charge. product ka 960, delivery extra.", "answer": {"in_stock": true, "price_inr": 960, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "है भाई. 12 का डिब्बा 3230 का, एक पीस 290", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "sadhe chhah hajar ka padega, ha, maal pada hai", "answer": {"in_stock": true, "price_inr": 6500, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "देखो नहीं है भाई खत्म हो गया Castrol 20W-40 engine oil पड़ा है वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Castrol 20W-40 engine oil"}}
{"product": "Dr.Fixit silicone sealant", "said": "हाँ बोलिए सर। अट्ठाईस पीस हैं स्टॉक में। 210 रुपये", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ है. बारह सौ लगेगा. शॉप दस बजे तक ओपन है. पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो हो, बल्ब शिल्लक आहे, वीस घेतले तर एकशे पाच प्रमाणे, एक पीस सव्वाशे, संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "हो दादा, आहे, एक पीस 165, 10 घेतले तर 140 प्रमाणे लावतो, घरपोच नाही, तुम्हाला यावं लागेल, हां", "answer": {"in_stock": true, "price_inr": 165, "delivery": false, "alternative": null}}
{"product": "Waxpol engine oil", "said": "available hai, koi problem nahi. paune teen sau plus GST", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "थोड़ी देर बाद कॉल करो जी.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "हाँ, माल पड़ा है. एमआरपी 230 है, पर हम एक सौ पचानवे में देते हैं.", "answer": {"in_stock": true, "price_inr": 195, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "तर। हां बोला साहेब। साडे चारशे रुपयांनापडेल। हां तर हो, हेल्मेट आहे। पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ऐंशीला पडेल। हो, आहे। कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "अं मिल जाएगा, है अपने पास। रेट पूछ कर बताता हूँ।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "mil jayega, hai ji", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "देखिए खत्म हो गया Vega वाला दे सकता हूं 790 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega"}}
{"product": "Crompton 3 L geyser", "said": "Hai na, kitne chahiye? 2250 hai, 1920 nahi", "answer": {"in_stock": true, "price_inr": 2250, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "अं हो दादा, आहे, संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "bagha tumhala 1500 la deto, battery aahe, milel, ajun kahi?", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "ji sir, tell me. cover rakha hai, mil jayega. savaa teen sau final price hai.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हैलो? आवाज़ आ रही है, पंखा पड़ा है, हाँ तो कंपनी का रेट दो हज़ार एक सौ पचास है, मैं एक हज़ार नौ सौ पचास लूँगा.", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "स्टॉक में है sir सौ रुपीज़ है है पता भेज दो लड़का दे आएगा shop दस बजे तक ओपन है", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हो, आहे शिल्लक, 25 रुपये, आहे, अजून 2 पीस पडलेत, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "हो, ठेवतो आम्ही, आहे, तीनशे पंचावन्न रुपये, कमी नाही होणार.", "answer": {"in_stock": true, "price_inr": 355, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बल्ब पडलाय. किंमत नंतर सांगतो, आत्ता लिस्ट नाही", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ है, कोई प्रॉब्लम नहीं। एक सौ साठ में दे देंगे। लड़का दे आएगा, चालीस रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "आमच्याकडे आहे, रेट बदललाय, आता 90, आधी 77 होता", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "आमच्याकडे आहे, रेट दीडशे आहे, घरपोच देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Goldcave tap", "said": "nal hai, mil jayega. 195 rupaye. ghar tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 195, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "बोलिए सर है भाई मिल जाएगा आपके लिए १७५ शाम तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ पाँच डिब्बे पड़े हैं चार सौ चालीस का पड़ेगा", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "Yes, bola. wrong number aahe sir", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "udya phone kara. kal Ravivar hota mhanun dukan band hota", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "हम्म. हाँ जी, बताइए. देखो सॉरी सर, अभी आउट ऑफ स्टॉक है. AdroitZ bike cover अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AdroitZ bike cover"}}
{"product": "Crompton 1500W immersion rod", "said": "हेलो? हाँ, बोलो भाई, मतलब, अं रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "haan, stock mein hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "मतलब नहीं है, उसकी जगह steel hinge दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "Bajaj immersion rod", "said": "हाँ सर, अवेलेबल है। 545 रुपये लगेंगे। बॉय भेज दूंगा।", "answer": {"in_stock": true, "price_inr": 545, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "तीन सौ पचास रुपये हाँ रखा है अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai, abhi ikkis piece bache hain, rate puch ke batata hun, bhijwa dunga", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "आहे की, मिळेल, एमआरपी 6110 आहे, पण 5168 ला लावतो", "answer": {"in_stock": true, "price_inr": 5168, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हाँ हाँ जी, मिल जाएगा. एक हज़ार पाँच सौ MRP लिखा है, हज़ार लगा देंगे", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "ट्यूब आत्ता नाहीये, संपली। फिलिप्स 20 वॉट batten आहे, एकशे नव्वदला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 20 वॉट batten"}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ बोलिए। है ना, कितने चाहिए? पौने पाँच सौ का है, इससे कम नहीं होगा।", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हॅलो? आवाज येतोय का? हां, बोला। ऑइल संपलंय। Motul 20W-40 engine oil आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Motul 20W-40 engine oil"}}
{"product": "Philips Astra Line 20W LED batten", "said": "स्टॉकमध्ये आहे। सव्वासातशेला पडेल। दुकान दहा वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "Han hai. abhi rate sava sau ho gaya, pichle mahine chhiyanve tha. pick up karna padega.", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "mana kisne kiya? hai na, 185 rupaye lagenge", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "मिल जाएगा, है अपने पास भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "हाँ हाँ, है, तीन सौ दस फ़ाइनल, डिलीवरी का 40 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 310, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "पौने आठ सौ रुपये लगेंगे. हाँ, है. लड़का नहीं है भेजने को, आकर ले जाइए. और कुछ?", "answer": {"in_stock": true, "price_inr": 775, "delivery": false, "alternative": null}}
{"product": "Goldcave tap", "said": "हो, नळ आहे. हां. एकशे पंचाऐंशीला देऊ, छापील किंमत 285 आहे", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello, haan bolo, haan ji, kabza available hai, umm price chalis hai.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "aadhi savva sadatisshe la hota, aata aahe ki nahi baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "देखो, नहीं है जी, ख़त्म, बजाज 3 L गीज़र वाला दे सकता हूँ, अड़तीस सौ रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बजाज 3 L गीज़र"}}
{"product": "Carbanao car mat for Swift", "said": "हाँ जी, बताइए, हाँ, है, रेट पौने पंद्रह सौ है है, अच्छा लड़का घर पे देजाएगा, ठीक है?", "answer": {"in_stock": true, "price_inr": 1475, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "देखो नहीं है, मंगवा देंगे, दो दिन लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग है, मिल जाएगा, हम्म 250 में दे देंगे.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "स्टॉक नहीं है अभी। तो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "आमच्याकडे आहे. सुटा एक 9 ला, पूर्ण पॅकेट सत्तर. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली. लोकल पुट्टी देऊ शकतो, 940 पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल पुट्टी"}}
{"product": "10W-30 engine oil 1 L", "said": "म्हणजे नाही असं नाही, आहे, 530 रुपयांनापडेल", "answer": {"in_stock": true, "price_inr": 530, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "हाँ बोलिए sir माल खत्म है कैंडेस 1200 वॉट रूम हीटर चलेगा उसका साढ़े पच्चीस सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कैंडेस 1200 वॉट रूम हीटर"}}
{"product": "50 mm padlock", "said": "Namaskar, bola, kulup nahiye, sampla, Harrison 40 mm padlock ghya, te chalel, haan tar yeun nyava lagel.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Harrison 40 mm padlock"}}
{"product": "Waxpol engine oil", "said": "ha ji, oil available hai, MRP 280 hai but dhai sau laga denge.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप संपली नाही आहे पोरगा पाठवतो अजून काही", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "स्टॉक में है सर, MRP पाँच सौ दस है, बट आपको सवा तीन सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हां बल्ब है पूरा पैकेट चार सौ चालीस अलग से एक पीस दो सौ पचहत्तर का", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए, लड़का चाय लेने गया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हो, आहे, बघा पावणेसात हजार प्लस GST.", "answer": {"in_stock": true, "price_inr": 6750, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "यस, अवेलेबल है, पर पीस एक सौ पैंसठ, छह लो तो एक सौ चालीस लगेगा.", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ रखा है एक पीस 150, 12 लोगे तो 125 के हिसाब से", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "देखो पंखा पड़ा है लास्ट मंथ 1630 था अब सवा उन्नीस सौ है", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हम्म है भाई, तीस रुपये लगेंगे, बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "वो वो हाँ, है। 10 लोगे तो 35 के हिसाब से, एक का 40", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "आहे ना। तुम्हाला १२५, छापील किंमत १३५।", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "2.5 mfd वाला ख़त्म है, Crompton 2.25 mfd capacitor है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 2.25 mfd capacitor"}}
{"product": "Activa air filter", "said": "नहीं है भाई, खत्म हो गया. local air filter available है, उसका price 260", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local air filter"}}
{"product": "tank float valve", "said": "Shaam tak ghar pe aa jayega", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Tata Agrico plier", "said": "pakkad sampli", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "सेठ जी बाहर गए हैं, वही बताएंगे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient Apex-FX 1200 mm ceiling fan", "said": "यस sir, बोलिए। नेटवर्क इशू है, आवाज़ ब्रेक हो रही है। सर आज बहुत बिज़ी डे है। साढ़े तेरह सौ रुपये लगेंगे। है भाई। एक कस्टमर खड़ा है, होल्ड करो। एड्रेस भेजो, डिलीवर कर देंगे। सीज़न टाइम है ना, डिमांड ज़्यादा है। ओके?", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "पोरगा चहा आणायला गेलाय. दिवाळीची गर्दी आहे ना. एक गिऱ्हाईक आहे, एक मिनिट. स्टॉकमध्ये आहे. कंपनीने परत रेट वाढवले. साडेचारशे रुपये होतील. म्हणजे. सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "Khaitan नहीं है, Orient room heater वाला है। वो क्या है खुद आकर उठाना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Orient room heater"}}
{"product": "12V 7Ah battery", "said": "haa, kaun? acha, bolo kya hai na stock khatam ho gaya hai Exide 12V 5Ah battery chalega? sade barah sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide 12V 5Ah battery"}}
{"product": "9W LED bulb", "said": "नहीं है। Philips चलेगा? उसका डेढ़ सौ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Philips"}}
{"product": "12V 7Ah battery", "said": "hello, bola na, battery padliye, sadebarashe hotil, ajun kahi?", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "बरं नाही म्हणालो का? आहे हो", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "हाँ बोलिए हाँ माल पड़ा है 1000 में दे देंगे", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "सवा पाँच सौ लगेगा, हां, रखा है, ठीक है?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "हॅलो? आवाज येतोय का? हां, बोला। स्टॉकमध्ये आहे। साडेपंधराशे रुपये होतील।", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "हेलो? हाँ, बोलो भाई। अभी नहीं है, order पर मंगवा देंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Ji, available hai, dedh sau ka padega", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Rainfun car cover", "said": "है भाई, मिल जाएगा, आपके लिए सवा छह सौ, आके ले जाओ", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "sava teen sau hoga. ji haan, rakha hai. home delivery hai", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "है भाई, आपको 440 में मिलेगा, MRP 490 है.", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "हाँ पंखा है मतलब चार हज़ार पचास है 4480 नहीं", "answer": {"in_stock": true, "price_inr": 4050, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ, रखा है, 265 का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 265, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "अभी बिजी हूं, आधे घंटे बाद फ़ोन करना.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "नहीं मिलेगा भाई स्टीलबर्ड 7 हेलमेट चलेगा उसका 490 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "स्टीलबर्ड 7 हेलमेट"}}
{"product": "MK 16A switch", "said": "है भाई, मिल जाएगा. बीस का फुल बॉक्स दो हज़ार चार सौ, सिंगल पीस एक सौ पैंतालीस.", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हाँ जी, बताइए। 440 फ़ाइनल। ऑयल है, मिल जाएगा। एड्रेस भेजो, डिलीवर कर देंगे।", "answer": {"in_stock": true, "price_inr": 440, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "aamchyakade nasta te.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hello haan bola bagha pickup karava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी आहे, मिळेल, सातशे पन्नास फिक्स, पोरगा पाठवतो, तीस रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 750, "delivery": true, "alternative": null}}
{"product": "Themisto multimeter", "said": "haan toh stock mein hai sir sawa chhah sau rupees hai.", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "अभी नहीं है, बजाज रॉड है, चलेगा? होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "बजाज रॉड"}}
{"product": "Xtreme MB Windtone horn", "said": "हाँ जी, बताइए, कंपनी ने फिर रेट बढ़ा दिया, है ना, कितने चाहिए? त्योहार का सीज़न है ना, आज सुबह से लाइट नहीं है, तीन सौ साठ लगा देंगे, वैसे 440 का है", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "बेसिकली हाँ जी, स्टॉक में पड़ा है, पूरा पैकेट सौ, अलग से एक पीस 12 का.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर ख़त्म नहीं हुई, है. मतलब पौने तेरह सौ लगेगा. लड़का दे आएगा, तीस रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 1275, "delivery": true, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "Ho aahe, tinashe pastis hotil, ek giraik ahe, ek minute, bara kay sangu, sakal pasun gardi aahe, pathavto, tis rupees delivery charge lagel.", "answer": {"in_stock": true, "price_inr": 335, "delivery": true, "alternative": null}}
{"product": "Safeyura LED batten", "said": "हां बोला काय सांगू सकाळपासून गर्दी आहे हो ठेवतो आम्ही आहे म्हणजे कंपनीने परत रेट वाढवले आठशे सत्तर रुपये कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 870, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "वो क्या है बेस्ट price साढ़े उन्नीस सौ। stock में है। हम्म", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "हाँ जी, पंखा रखा है, 1750 लिखा है, छूट देकर चौदह सौ पचास, डिलीवरी हो जाएगी सर.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "है भाई मिल जाएगा आपको साढे इकतीस सौ MRP तो 3450 है एड्रेस भेजो डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 3150, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "बरं हो, पक्कड आहे, दोनशे वीस... नाही नाही, एकशे ऐंशी", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ऑइल संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "जी, उपलब्ध है, कंपनी का रेट २८५ है, मैं २४० लूँगा.", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "स्टॉक नाही आहे सध्या भाऊ म्हणजे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan ji, tala available hai, chaar sao lagega, bhejne ka intezaam nahi hai.", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "check karke batata hun, haan haan, sun raha hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "हाँ बोलिए, १८५० का पड़ेगा, हम्म हाँ हाँ, है.", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "है भाई, 10 का फुल बॉक्स 1620, सिंगल पीस 185", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "हॅलो, कोण बोलतंय? स्टॉकमध्ये आहे. म्हणजे ८८० रुपये होतील", "answer": {"in_stock": true, "price_inr": 880, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "हाँ, मोटर है, चौवालीस सौ, जीएसटी एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 4400, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हैलो, कौन? स्टॉक में है, २८० रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हां, है, पैशन ब्रेक शू का रेट 300 है, ये वाला 380", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ जी, stock में पड़ा है. ८० फाइनल price है. delivery बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Ho, kulup milel, mhanje savvacharshe fix, theek ahe?", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock में नहीं है. NGK Pulsar 180 spark plug है, २०० का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "NGK Pulsar 180 spark plug"}}
{"product": "Candes 10 L geyser", "said": "ho sir, available aahe rate 3000 aahe sir pathavto, 100 rupaye delivery charge lagel.", "answer": {"in_stock": true, "price_inr": 3000, "delivery": true, "alternative": null}}
{"product": "Camel drill machine", "said": "Han, stock me hai, GST mila ke 1300, pahuncha doonga, fikar mat karo", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "अभी नहीं है, Motul 20W-40 engine oil है, 490 का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Motul 20W-40 engine oil"}}
{"product": "Retina bike cover", "said": "स्टॉक में है सर. मतलब दो सौ पचहत्तर का है, इससे कम नहीं होगा. भेज देंगे, 30 रुपये डिलीवरी चार्ज लगेगा", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "रेट 350 आहे, हो, कव्हर शिल्लक आहे.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "हाँ बोलिए सर स्टॉक में सिर्फ 11 पीस है प्राइस कन्फर्म करके बताता हूँ खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "आपके लिए साढे तीन सौ. स्टॉक में है. आपके घर तक भिजवा देंगे. और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट है, मिल जाएगी। पौने छह सौ। घर पर भेज देंगे, 20 अलग से। हम्म", "answer": {"in_stock": true, "price_inr": 575, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हो सर बोला आहे सर रेडी स्टॉक आहे आता पंधरा आहे आधी तेरा होता", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "kya hai na machine padi hai, delivery ho jayegi sir", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "नहीं है जी, ख़त्म, लोकल एयर फ़िल्टर पड़ा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल एयर फ़िल्टर"}}
{"product": "Speedwav horn", "said": "haan maal pada hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "है भाई, सौ है, तिरानवे नहीं.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "yes, bola. baghto aani kalavto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "सॉरी, आउट ऑफ स्टॉक आहे. अँकर 3 सॉकेट बोर्ड आहे, त्याचा प्राइस साडे पाचशे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अँकर 3 सॉकेट बोर्ड"}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी सर, बताइए, अच्छा, हाँ, रेडी स्टॉक है, 100 का है", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "बेसिकली वो आइटम हम नहीं रखते, Osram headlight वाला है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Osram headlight"}}
{"product": "Havells LED torch", "said": "५ पीस हैं स्टॉक में. ३३० रुपयेका. अं आपको आ कर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन? साढ़े तीन सौ रुपये, हाँ, माल पड़ा है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "आहे ना एका नगाचे पंचवीस, पंचवीस नगांचा बॉक्स 570 ला", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "हाँ भाई, बोलो, हम्म हाँ, रखा है, लड़का नहीं है भेजने को, आकर ले जाइए, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, एक ग्राहक खड़ा है, एक मिनट, ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "bola dada, ho, thevto aamhi, ahe, pavanesat hazar la padel, dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": 6750, "delivery": false, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "एक हज़ार का है यस available है औरकुछ", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "पौने उनतालीस सौ रुपये लगेंगे। हाँ, रखा है। त्योहार का सीज़न है ना। एक ग्राहक खड़ा है, एक मिनट। अरे क्या बताऊँ, सुबह से बहुत भीड़ है। आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 3875, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ बोलिए सर. पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "Kya hai na haan bolo... nahi, wo nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "बोला, काय पाहिजे? हो, ठेवतो आम्ही, आहे। रेट शेठ सांगतील। येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Michelin wiper blade", "said": "अं वो क्या है गलत नंबर लगाया है आपने", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "please thodi der baad call karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "सव्वादोनशेला पडेल, हां तर हो दादा, आहे, पाठवायची सोय नाही.", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "ऐसा नहीं है कि नहीं है, है, rate 570 है", "answer": {"in_stock": true, "price_inr": 570, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "ha sir, boliye, ladka ghar pe chhod dega, aur kuch chahiye?", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HTC multimeter", "said": "woh hum nahi rakhte uski jagah kaiweets multimeter de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "kaiweets multimeter"}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "अं sorry रॉन्ग नंबर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "हां भाई बोलो है भाई देखो हमारा रेट चार सौ बीस MRP तो पाँच सौ पचपन है आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 420, "delivery": true, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "यस sir अं प्लीज़ थोड़ी देर बाद call करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ जी, बताइए, त्योहार का सीज़न है ना, लड़का चाय लेने गया है, आज सुबह से लाइट नहीं है, आवाज़ कट रही है, ज़ोर से बोलिए, देखो नहीं मिलेगा भाई, बाज़ार में माल की कमी चल रही है, और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हां बोला, एकशे वीस रुपये, कमी नाही होणार, प्लग पडलाय", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "cover pada hai, barah sau pachas, 1300 nahi bola maine, delivery nahin karte, aake le jao", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "जी उपलब्ध है तो", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "yes sir, boliye, yes, available hai, umm sadhe panch sau rupees hai, thik hai?", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "सगळं संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "अभी available नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा। छत्तीस सौ पचास रुपीज़ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हैलो? हाँ, आवाज़ आ रही है. rate साढ़े तीन सौ है सर. नेटवर्क इशू है, आवाज़ ब्रेक हो रही है. sir आज बहुत बिज़ी डे है. सीज़न टाइम है ना, डिमांड ज़्यादा है. एक कस्टमर खड़ा है, होल्ड करो. हाँ, रेडी stock है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "नहीं है रुको रुको है एक पड़ा है लिए अट्ठाईस सौ पचास लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 2850, "delivery": true, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "यस, अवेलेबल है. एक हज़ार पाँच सौ लगेगा. सीज़न टाइम है ना, डिमांड ज़्यादा है. सर आज बहुत बिज़ी डे है. लड़का दे आएगा, बीस रुपये लगेंगे. नेटवर्क इशू है, आवाज़ ब्रेक हो रही है.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "कब्ज़ा पड़ा है, 60 है, 51 नहीं.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ, रखा है, MRP 315 है, आपको 270 में दे देंगे", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "available hai list me 490 hai, aapko 400 padega", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "umm ha, stock me hai aapke liye do hajar pachas ladka nahi hai bhejne ko, aake le jao", "answer": {"in_stock": true, "price_inr": 2050, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "han sir, bataiye, available hai, sir aaj bahut busy day hai, season time hai na, demand zyada hai, rate savaa do so hai.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "मतलब वो तो कल ही ख़त्म हो गया MYRA helmet लॉक है वो ले लीजिए हम्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "MYRA helmet लॉक"}}
{"product": "AmiciTools water pump", "said": "मोटर संपली नाही आहे एक हजार पन्नास रुपयांना पडेल डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 1050, "delivery": true, "alternative": null}}
{"product": "Havells Cista room heater", "said": "बेसिकली बेस्ट प्राइस 2670, यस, अवेलेबल है", "answer": {"in_stock": true, "price_inr": 2670, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "ट्यूब पडलीये, चारशे तीसला देऊ, छापील किंमत 765 आहे, कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "ahe sir, ready stock ahe, 25 la padel, Astral 3/4 inch elbow pan aahe, te 31 la.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "hallo, han bolo, kya hai na Autofy wala khatm hai, Steelbird helmet wala hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Steelbird helmet"}}
{"product": "12V 7Ah battery", "said": "Haan sir, boliye dekho 7Ah nahi hai, Amaron 12V 9Ah battery hai address bhej do, ladka de aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Amaron 12V 9Ah battery"}}
{"product": "Philips 5W LED bulb", "said": "स्टॉकमध्ये आहे, 130 ला लावतो, एमआरपी 165 आहे", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "आहे की, मिळेल. तीनशे पासष्ट प्लस GST. कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 365, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "मिल जाएगा है अपने पास देखो आपसे पंद्रह लेंगे लिखा उन्नीस है भेजने वाला कोई नहीं है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 15, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "एक मिनिट हां... बोला, म्हणजे स्टॉक संपलाय, काय सांगू, सकाळपासून गर्दी आहे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "कमी नाही, भरपूर आहे, 210 होतील, संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "हाँ है। १५५ रुपीज़ है", "answer": {"in_stock": true, "price_inr": 155, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "नमस्कार बोला अं कव्हर आहे मिळेल पावणेपाचशेला देतो", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "नमस्कार, बोला, बल्ब संपला नाही, आहे, सव्वा तीनशे रुपये, कमी नाही होणार, येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 325, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ भाई, बोलो, जी, उपलब्ध है, 120 रुपये.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "स्टॉक में है, अच्छा सवा चार सौ लगा देंगे, वैसे MRP पाँच सौ साठ लिखा है", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "जी बोलिए, कवर है, मिल जाएगा, सवा पाँच सौ रुपये लगेंगे, ठीक है?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "नमस्ते बोलिए रुकिए देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "नहीं है जी, ख़त्म भाई.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ हाँ है MRP 1560 है पर तेरह सौ लगा देंगे मतलब भेजने वाला कोई नहीं है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 1300, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हाँ, माल पड़ा है, खुला एक 225 का, पूरा पैकेट 405", "answer": {"in_stock": true, "price_inr": 405, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "मशीन आहे, मिळेल, म्हणजे price 9500 आहे, पोरगा घरी आणून देईल, 60 रुपये लागतील", "answer": {"in_stock": true, "price_inr": 9500, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी प्लग रखा है 98 सॉरी सॉरी 110", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "संपलं होतं... नाही, एक आहे अजून, बघा, तुम्हाला चारशे साठला देतो.", "answer": {"in_stock": true, "price_inr": 460, "delivery": null, "alternative": null}}
{"product": "Type R G124 horn", "said": "आम्ही ते ठेवत नाही, हां Easy4buy हॉर्न वाला आहे, त्याचा प्राइस पाचशे चाळीस", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Easy4buy हॉर्न"}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "पंखा पडलाय. कळवा मग.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ जी सर, बताइए, अं अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है, सर आज बहुत बिज़ी डे है, तीन सौ फाइनल प्राइस है, सीज़न टाइम है ना, डिमांड ज़्यादा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हां, हां तर बल्ब संपलाय, त्याच्याऐवजी Philips 12 watt LED bulb वाला देऊशकतो, डिलिव्हरी करत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Philips 12 watt LED bulb"}}
{"product": "Benz helmet", "said": "हेलो? हां, बोलो भाई. स्टॉक में है सर. 740 का है, इससे कम नही होगा.", "answer": {"in_stock": true, "price_inr": 740, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "tinashe rupayanna padel ho dada, ahe porga ghari sodun deil", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "sham tak ghar pe aa jayega.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "एक मिनिट हां... बोला, अं आहे आहे, तुम्हाला चारशे पासष्ट, छापील किंमत पाचशे तीस", "answer": {"in_stock": true, "price_inr": 465, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "Hello? han, awaaz aa rahi hai, bolo, bas 21 piece bache hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नमस्ते, बोलिए. बैटरी ख़त्म हो गई. Exide 12V 9Ah battery है, 1550 का. अं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide 12V 9Ah battery"}}
{"product": "1 inch PVC elbow", "said": "हाँ, रखा है। पच्चीस का डिब्बा पाँच सौ पैंतालीस का, एक पीस पच्चीस", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Lagta hai nahi hai... haa mil gaya, hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "Ho, oil ahe, aapla rate donshe navvad, MRP 350 ahe, mhanje address pathava, pathavto.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हैलो? हाँ, आवाज़ आ रही है, ताला ख़त्म हो गया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "एक मिनट... हाँ बोलिए। वो घंटे भर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "यस sir, बोलिए। हाँ जी, stock में पड़ा है। मतलब तैंतीस लिखा है, छूट देकर तीस। अपना लड़का छोड़ देगा घर पे। एक कस्टमर खड़ा है, होल्ड करो। सीज़न टाइम है ना, डिमांड ज़्यादा है। sir आज बहुत बिज़ी डे है", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "cover sampala nahi, aahe, sadechaudashe la lavto, MRP ek hazar sahashe pannas aahe.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "यस, अवेलेबल है. 190, सॉरी सॉरी, 165.", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हाँ रखते हैं, पर अभी ख़त्म है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "aapke liye sava chhabbis sau ji available hai address bhej do bhej dete hain", "answer": {"in_stock": true, "price_inr": 2625, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Haan, hai apne paas bhai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, 5 पीस पड़े हैं. आपके लिए एक सौ अस्सी. भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "हां बोला साहेब. हां. पावणेसत्तावीसशे रुपयांना पडेल. पंखा आहे, मिळेल.", "answer": {"in_stock": true, "price_inr": 2675, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "अभी 21 पीस ही बचे हैं, ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "वो क्या है हाँ जी, स्टॉक में पड़ा है, आपसे सवा तीन सौ लेंगे, लिखा 380 है.", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "आत्ता नाहीये संध्याकाळी येईल पोरगा पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Maal khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "टायर नहीं है खत्म हो गया JK 90 tyre है सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "JK 90 tyre"}}
{"product": "fan capacitor 2.5 mfd", "said": "अच्छा कंपनी ने फिर रेट बढ़ा दिया, नहीं मिलेगा भाई, एक ग्राहक खड़ा है, एक मिनट", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा, पता भेज दो, लड़का दे आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "स्टॉक में नहीं है. V-Guard pedestal fan है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-Guard pedestal fan"}}
{"product": "water pump seal kit", "said": "अं हाँ जी, स्टॉक में पड़ा है, पौने दो सौ रुपीज़ है.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "बघा बल्ब आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो, बघा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ है, कोई प्रॉब्लम नहीं, आपको आकर लेना पड़ेगा, और कुछ?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है, अं 150 का है, आपके लिए 120, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "स्टॉक में है सर. मतलब रेट 260 है सर. होम डिलीवरी अवेलेबल है, 80 चार्ज. मतलब", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "हैलो कौन हाँ जी मिल जाएगा सतासी का है आपके लिए अस्सी", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "रेट 225 है। देखिए मिल जाएगा, है अपने पास", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "nahi milega bhai mechanical seal hai uska rate ek sau nabbe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "mechanical seal"}}
{"product": "Speedwav handle grip for Yamaha", "said": "ha, hai apne paas, chaar sau nabbe lagega, home delivery available hai, 50 charge", "answer": {"in_stock": true, "price_inr": 490, "delivery": true, "alternative": null}}
{"product": "Tata Agrico plier", "said": "देखो Tata वाला ख़त्म है, JCB है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "JCB"}}
{"product": "water pump seal kit", "said": "बेसिकली अभी स्टॉक फिनिश हो गया देखो लोकल सील किट अवेलेबल है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल सील किट"}}
{"product": "Orient 400 mm pedestal fan", "said": "यस अवेलेबल है देखो डिलीवरी हो जाएगी 50 रुपीज़ चार्ज एक्स्ट्रा आइटम प्राइस 3910 डिलीवरी चार्ज अलग", "answer": {"in_stock": true, "price_inr": 3910, "delivery": true, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "hello? aawaj yetoy ka? haan bola, rate adichshe ahe, ho, cover milel", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "ho, machine milel, bagha terashe pannas la deto.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "milel na, aahe, 155 sangitla ka? chuklo, 210 aahe, porga pathavto", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Dow silicone sealant", "said": "आत्ता गडबडीत आहे अर्ध्या तासाने फोन करा बघा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "हैलो, कौन? आपके लिए साढ़े चार सौ, यस, available है, अपना लड़का छोड़ देगा घर पे", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हां तर हो, मिळेल, साडे चारशे रुपयांना पडेल, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "HP engine oil", "said": "हां हां है अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Mhanje pavneekonatisshe hotil, geyser aahe, milel, gharpoch pohochvun deu.", "answer": {"in_stock": true, "price_inr": 2875, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ahe ahe, haan tar chaalis la padel.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अं हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, अमरोन 12V 9Ah बैटरी अवेलेबल है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अमरोन 12V 9Ah बैटरी"}}
{"product": "AutoClean wiper blade", "said": "यस, available है। प्राइस तीन सौ पचानवे है है। हां, पच्चीस piece पडे हैं", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "haan ji, machine available hai. pauna ekkis sau ka padega. courier kar denge", "answer": {"in_stock": true, "price_inr": 2075, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "बाज़ार में माल की कमी चल रही है, हाँ है, कोई प्रॉब्लम नहीं, एक सौ सत्तर लगेगा, आज सुबह से लाइट नहीं है, त्योहार का सीज़न है ना, स्टॉक में सिर्फ 29 पीस है, एक ग्राहक खड़ा है, एक मिनट.", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "ahe aahe delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "नेटवर्क इशू है आवाज़ ब्रेक हो रही है हम्म सीज़न टाइम है ना डिमांड ज़्यादा है स्टॉक नहीं है अभी एंकर बैटन अवेलेबल है उसका प्राइस 780", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एंकर बैटन"}}
{"product": "1 inch PVC elbow", "said": "नहीं है... रुको रुको, है, एक पड़ा है, आपके लिए पंद्रह, डिलीवरी बंद है अभी, दुकान से ले लो.", "answer": {"in_stock": true, "price_inr": 15, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "आधी ऐंशीला होता, आता आहे की नाही बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "हॅलो, कोण बोलतंय? हां तर मशीन आहे, मिळेल, पावणेछत्तीसशे रुपये होतील, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 3575, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "यस sir बोलिए हाँ जी मिल जाएगा पौने पंद्रह सौ रुपीज़ है", "answer": {"in_stock": true, "price_inr": 1475, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "एक मिनट... हाँ बोलिए। पहले इकतालीस सौ पचास का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हाँ बोलिए. स्टॉक में है. ६५५ लगेगा. खुद आकर उठाना पड़ेगा. बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 655, "delivery": false, "alternative": null}}
{"product": "AmiciTools water pump", "said": "स्टॉकमध्ये आहे, तर आधी 930 होता, आता एक हजार पन्नास झाला.", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "जी बोलिए, साढ़े तीन सौ का पड़ेगा, हाँ, है.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "अच्छा हाँ हाँ है डिलीवरी नहीं करते दुकान पर आकर ले जाइए और कुछ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "नॉट अवेलेबल sir, अं Havells 230 mm exhaust fan वाला पड़ा है, वो ले लो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 230 mm exhaust fan"}}
{"product": "MK 16A switch", "said": "नमस्ते, बोलिए। अभी अवेलेबल नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "नहीं है भाई, खत्म हो गया, Shell 10W-30 इंजन ऑयल 900 एमएल अवेलेबल है, उसका प्राइस पौने पाँच सौ, डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Shell 10W-30 इंजन ऑयल 900 एमएल"}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "Haan ji, geyser available hai, arre 2270... are nahi, 3000, bhejne wala koi nahi hai, aapko ana padega", "answer": {"in_stock": true, "price_inr": 3000, "delivery": false, "alternative": null}}
{"product": "Autofurnish car cover for Santro Xing", "said": "haan, ready stock hai, aapke liye chaar sau panch", "answer": {"in_stock": true, "price_inr": 405, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "रॉन्ग नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "hai ji, mil jayega, kya hai na rate chhah sau paintees hai", "answer": {"in_stock": true, "price_inr": 635, "delivery": null, "alternative": null}}
{"product": "Havells Oro switch", "said": "हां हो आहे पावणेपाचशेला देतो पाठवायला माणूस नाही इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "कवर ख़त्म हो गया, AdroitZ बाइक कवर वाला रखा है, डिलीवरी फ़्री है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "AdroitZ बाइक कवर"}}
{"product": "wall putty 20 kg", "said": "हाँ बोलिए। मैं नया हूँ, मुझे नहीं पता।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "कम नहीं है, बहुत पड़ा है। दुकान से ले जाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "पंखा ख़त्म हो गया. Anchor 150 mm exhaust fan वाला है, 1350 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 150 mm exhaust fan"}}
{"product": "Wipro 22W LED batten", "said": "हाँ बोलिए सर हाँ सर वो मॉडल अवेलेबल है प्राइस साढ़े चार सौ है", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "hai ji, mil jayega, ab teen sau nabbe hai, pehle chaar sau sattar tha, khud aake uthana padega.", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "han bolo hai bhai dedh sau final", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हैलो हां आवाज आ रहीहै बेसिकली हां है बेस्ट price पाँच सौ पैंतीस खुद आके ले जाना पडेगा", "answer": {"in_stock": true, "price_inr": 535, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "हाँ बोलिए सर। वो स्टॉक में है सर। कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हैलो, कौन? अं बाज़ार में माल की कमी चल रही है, त्योहार का सीज़न है ना, एक ग्राहक खड़ा है, एक मिनट, वो हम नहीं रखते, कंपनी ने फिर रेट बढ़ा दिया, आवाज़ कट रही है, ज़ोर से बोलिए, बताइए क्या करना है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हो दादा, आहे, तर तुम्हाला 635, छापील किंमत 685", "answer": {"in_stock": true, "price_inr": 635, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "stock khatam ho gaya hai, awaaz kat rahi hai, zor se bolo, aapko number kisne diya? acha thik hai, local brake shoe chalega? 240 ka hai, Diwali ka season hai na, sab yahi maang rahe hain, are kya bataun, subah se bahut rush hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "teflon tape", "said": "Hello? haan, awaaz aa rahi hai, bolo, nahi hai, pichle hafte bhi kisi ne yahi poocha tha, matlab thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "आत्ता मिळत नाही, सांगा काय करायचं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb khatm nahi hua, hai, 89 bola kya? galti se bola, 90 hai.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "ho dada, aahe sadesahashe ahe, satshe ainshi nahi.", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "ha ji, mil jayega, upar 5290 likha hai, aapse chauwalis sau pachas lenge", "answer": {"in_stock": true, "price_inr": 4450, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "Yes, available hai haan toh Speedwav horn bhi hai, uska teen sau sattar, aur aap wala teen sao pachpan ka abhi delivery band hai", "answer": {"in_stock": true, "price_inr": 355, "delivery": false, "alternative": null}}
{"product": "Hella horn", "said": "नमस्ते, बोलिए। पता नहीं, गोदाम में देखना पड़ेगा। आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "bulb pada hai single peice do sau chalis, chhah ka full box ek hajar teen sau", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "SARK tap", "said": "हेलो हां बोलो अच्छा हाँ जी स्टॉक में पडा है सवा छह सौ रुपीज है देखो भेजने वाला कोई नही है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "available hai koi problem nahi bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "हां बोला, हो, मिळेल, पावणे तीनशे होतील", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "ji boliye, stock khatam ho gaya hai, kal Sunday tha toh dukaan band thi, thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Yamaha", "said": "पोरगा पाठवतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "नल है, मिल जाएगा. आपको पौने तीन सौ में दे देंगे, MRP 440 है", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "एक मिनट... हां बोलिए, शाम तक माल आ जाएगा, अभी नही है, चौबीस सौ पचास में दे देंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "है भाई छह सौ लगेगा", "answer": {"in_stock": true, "price_inr": 600, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Haan aahe, bara MRP 365 ahe, pan tinashe la lavto.", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "हाँ भाई, बोलो. है भाई. एक हज़ार एक सौ रुपये का है. डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "यस, अवेलेबल है, अभी रेट 665 हो गया, पिछले महीने 780 था", "answer": {"in_stock": true, "price_inr": 665, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "nahi aahe havells 6 socket extension board ahe aamchyakade porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "havells 6 socket extension board"}}
{"product": "tank float valve", "said": "हैलो? हाँ, आवाज़ आ रही है। बारह पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "अच्छा हाँ, 9 पीस पड़े हैं, रेट 400 है सर, आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं मिलेगा भाई paint roller है वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "Activa air filter", "said": "स्टॉक में है सर। आपके लिए सवा तीन सौ। डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँभाई, बोलो. सवा चार सौ लगेगा. हाँ, ताला है", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "मशीन संपली Cheston 500W drill चालणार असेल तर सांगा संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Cheston 500W drill"}}
{"product": "Speedwav handle grip for Apache", "said": "अभी नहीं बता सकता, बाद में फ़ोन करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हो सर, अवेलेबल आहे. प्राइस चारशे ऐंशी आहे.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "हो आहे नाही नाही तो दुसरा आहे हा नाहीये Avita switch वाला घ्या ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Avita switch"}}
{"product": "Retina bike cover", "said": "yes, bola. Retina nahiye, Cross Fingers bike cover wala ahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers bike cover"}}
{"product": "4 inch door hinge", "said": "hai na, kitne chahiye? sau lagega.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "kulup ahe, milel, rate savvateenshe aahe, delivery deto, ajun kahi?", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "मिल जाएगा, है अपने पास. 2699 लगेगा. घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 2699, "delivery": true, "alternative": null}}
{"product": "ATC bike cover", "said": "कव्हर संपलं नाही, आहे। तुम्हाला १५०, छापील किंमत १७०", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "वो मॉडल अवेलेबल नहीं है। local वाला पड़ा है, वो ले लो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local"}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "बेस्ट प्राइस साढे इकतीस सौ हाँ जी गीज़र रखा है", "answer": {"in_stock": true, "price_inr": 3150, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "है भाई. डब्बा दो हज़ार नौ सौ का है, बीस पीस वाला, एक का पौने दो सौ", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "जी उपलब्ध है एक ग्राहक खड़ा है एक मिनट आपके लिए 280 आवाज़ कट रही है ज़ोर से बोलिए हाँ 28 पीस पड़े हैं लड़का चाय लेने गया है बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "जी बोलिए, नहीं मिलेगा भाई, हैवेल्स room हीटर है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स room हीटर"}}
{"product": "Almonard pedestal fan", "said": "पंखा नहीं है खत्म हो गया Crompton pedestal fan है वो ले लीजिए देखो खुद आके ले जाना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Crompton pedestal fan"}}
{"product": "bike headlight bulb", "said": "नहीं है। उसकी जगह फिलिप्स halogen बल्ब वाला दे सकते हैं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स halogen बल्ब"}}
{"product": "water pump seal kit", "said": "हाँ है, कोई प्रॉब्लम नहीं। सवा तीन सौ का है", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "sorry sir अभी आउट ऑफ stock है उसकी जगह हैवेल्स 1500 वॉट rod वाला दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 1500 वॉट rod"}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "नमस्ते, बोलिए. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. कंपनी ने फिर रेट बढ़ा दिया. हैलो? आवाज़ नहीं आ रही, बाद में करना. आवाज़ कट रही है, ज़ोर से बोलिए. एक ग्राहक खड़ा है, एक मिनट", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "helmet padlay 725 la lavto mrp 850 ahe", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "mhanje te model available nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "नहीं मिलेगा भाई, Cheston 200A वेल्डिंग मशीन वाला है, वो ले लीजिए.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston 200A वेल्डिंग मशीन"}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "अं वो आइटम हम नहीं रखते, ओसराम एलईडी हेडलाइट है, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओसराम एलईडी हेडलाइट"}}
{"product": "2 inch paint brush", "said": "अस्सी का पड़ेगा। हाँ जी, मिल जाएगा", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "acha hai bhai, 990 mein de dunga, aur kuch?", "answer": {"in_stock": true, "price_inr": 990, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "वो हम नही रखते Usha 400 mm pedestal fan चलेगा उसका पौने सोलह सौ है दुकान से ले जाना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Usha 400 mm pedestal fan"}}
{"product": "Raaisin car cover", "said": "han ji, cover available hai, MRP 5270 hai, discount ke baad 4430 padega, boy ghar pe de jayega", "answer": {"in_stock": true, "price_inr": 4430, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां माणूस पाठवून देतो घरी", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "जरा वेळाने फोन करा अं काय सकाळपासून गर्दी आहे तर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "8 mm संपलं, Taparia 6 mm drill bit वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Taparia 6 mm drill bit"}}
{"product": "bike headlight bulb", "said": "यस सर, बोलिए, अच्छा है भाई, मिल जाएगा, देखो, 210 लगेगा.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ बोलिए मतलब अभी आउट ऑफ stock है नया stock कल आ रहा है अभी delivery बंद है", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हम्म, स्टॉक में है सर, २७० लिखा है, डिस्काउंट करके २१०", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho ahe satashe chalis hotil kalva mag", "answer": {"in_stock": true, "price_inr": 740, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "haan, mil jaega bhai, chaar sau sath hai, sorry sorry, panch sau pandrah.", "answer": {"in_stock": true, "price_inr": 515, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "मतलब प्लीज़ थोड़ी देर बाद कॉल करो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Mil jayega, hai, aapke liye pone char sau, sham tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "वो क्या है हैलो? आवाज़ नहीं आ रही, बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ, है, घर तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ जी. हैलो, कौन? हाँ, रखा है. डेढ़ सौ का पड़ेगा", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "हेलो। रेट पौने दो सौ है, पर अभी स्टॉक नहीं है। अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "hello, haan bolo, Diwali ka season hai na, sab yahi maang rahe hain, maal kal aayega, abhi nahi hai, ek customer khada hai, ek minute, so mein de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "ते मॉडेल अवेलेबल नाही, Passion ब्रेक शू आहे, सेम क्वालिटी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Passion ब्रेक शू"}}
{"product": "Havells Cista room heater", "said": "हां सर वो मॉडल अवेलेबल है डिलीवरी हो जाएगी पचास रुपीज चार्ज एक्स्ट्रा आइटम प्राइस पौने छब्बीस सौ डिलीवरी चार्ज अलग", "answer": {"in_stock": true, "price_inr": 2575, "delivery": true, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "स्टॉक में है २१५ रुकिए लिस्ट देखता हूं २२०", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "मतलब हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "वो वो अं 8 mm नहीं है तपारिया 6 एमएम drill बिट है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "तपारिया 6 एमएम drill बिट"}}
{"product": "Activa air filter", "said": "4 पीस हैं स्टॉक में, रेट पूछ कर बताता हूँ, अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Rainfun car cover", "said": "ji, available hai, 480... ek minute, rate list dekhta hun... haan sawa chhah sau, counter pe aake le lo", "answer": {"in_stock": true, "price_inr": 625, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नहीं है जी ख़त्म Crompton रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton"}}
{"product": "Spaiko LED headlight", "said": "stock में सिर्फ अट्ठाईस piece है। सवा तीन हजार रुपये लगेंगे", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "ho sir, bola, ahe sir, ready stock ahe, tumhala pavanecharshe la deu, MRP charashe pannas ahe", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए हाँ जी बल्ब रखा है देखो अब प्राइस पौने दो सौ है पहले एक सौ पचपन था", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "mil jayega hai ek sau sattar lagega", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "आठशे चाळीस होतील. ट्यूब पडलीये. कुरिअर करतो. ठीक ठीक आहे?", "answer": {"in_stock": true, "price_inr": 840, "delivery": true, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "स्टॉक में है। तीन सौ साठ का है, इससे कम नहीं होगा। डिलीवरी की टेंशन मत लो, भेज देंगे।", "answer": {"in_stock": true, "price_inr": 360, "delivery": true, "alternative": null}}
{"product": "Waxpol engine oil", "said": "Haan haan, bolo, hai bhai, dhaee sau laga denge, waise do so sattar ka hai.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "है ना, कितने चाहिए? हम्म GST मिलाकर पच्चीस. भेजने का इंतज़ाम नहीं है.", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "रेट तीन सौ पचासी है. हाँ सर, वो मॉडल अवेलेबल है", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "han, stock me satrah piece hai, rate puch ke batata hun, sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "RNG Eko tyre", "said": "कल सुबह आ जाएगा, अभी नहीं, 2500 रुपये, आकर ले जाओ.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "BMB welding machine", "said": "मालिक नहीं हैं बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "एक मिनट हाँबोलिए तो Allure Auto नहीं है Accessory Mart कार मैट है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessory Mart कार मैट"}}
{"product": "Speedwav handle grip for Apache", "said": "अच्छा नही है जी, खत्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "मशीन संपली.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haa ji mil jayega rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी पड़ी है. पता भेज दो, लड़का दे आएगा", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "वो क्या है हाँ, 11 पीस पड़े हैं. डिलीवरी हो जाएगी सर", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "हाँ, है, अब सवा सैंतीस सौ है, पहले 3230 था", "answer": {"in_stock": true, "price_inr": 3725, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "मशीन संपली नाही, आहे। रेट मालकांना विचारून सांगतो", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "stock madhe ahe, bagha sadesaha hajar rupees hotil.", "answer": {"in_stock": true, "price_inr": 6500, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "है भाई रेट ढाई सौ है", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "हो, ठेवतो आम्ही, पण आत्ता संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "Aapko number kisne diya? acha thik hai, are kya bataun, subah se bahut rush hai, bhaiya aap ruko do min, phone pe hun, wo toh kal hi khatm ho gaya, Allure Auto car mat hai, uska rate pandrah sau, sham tak ghar pe aa jaega, ladka chai lene gaya hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Allure Auto car mat"}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "हाँ जी, मिल जाएगा सर।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "Ho dada, aahe. sadeterashe la lavto, MRP ek hajar sahashe ahe", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ, माल पड़ा है। दो सौ अस्सी रुपये लगेंगे। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 280, "delivery": false, "alternative": null}}
{"product": "RNG Eko tyre", "said": "han, kaun? acha, bolo, khatm ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है कोई प्रॉब्लम नहीं प्राइस 140 है Syska 12 watt LED bulb भी है वो 155 का आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 140, "delivery": false, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "स्टॉक में है, अभी रेट 1330 हो गया, पिछले महीने 1020 था, डिलीवरी फ़्री है.", "answer": {"in_stock": true, "price_inr": 1330, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "देखिए लाइट पडी है Speedwav LED हेडलाइट का रेट ११७० है ये वाला १६५०", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "Tube padliye, paket savvapandhrashe la ahe, sutta ek ek hazar pannas la, address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 1525, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "तर आत्ता नाहीये, परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "हाँ है। तो। वो वो क्या है अपना प्राइस साढ़े चार सौ, एमआरपी 590 है। लड़का घर पे दे जाएगा।", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "बिजागरी आहे, मिळेल, सुटा चाळीसला, पन्नास नगांचा बॉक्स घेतला तर एक हजार सातशे.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "abhi nahin, order pe mangwa denge. delivery ho jayegi", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fluke multimeter", "said": "हेलो। मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "हो, ठेवतो आम्ही, आहे। एक पीस 165, 10 घेतले तर 150 प्रमाणे लावतो।", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Check karke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "हाँ, रखा है. रेट दो सौ पैंसठ है. अच्छा घर पर भेजदेंगे.", "answer": {"in_stock": true, "price_inr": 265, "delivery": true, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "हो सर, अवेलेबल आहे, एक पीस 185, 10 घेतले तर 155 प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "pavnepaachashe la yeto, pan sadhya samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हो आहे चारशे चाळीसला पडेल कळवा मग", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बल्ब आहे, मिळेल, छापील पाचशे, डिस्काउंट करून सव्वाचारशे", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "अं अं आत्ता मिळत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "तुम्हाला सहाशे दहाला देऊ। हो सर, available आहे।", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "हॅलो, कोण बोलतंय? हो, लाईट शिल्लक आहे, अं साडेतेराशे रुपयांना पडेल, दुकानातून न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ek minute... ha bolo, ha ha, hai, ek sau das rupaye lagenge.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "चार सौ फ़ाइनल हाँ रखा है होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Fluke multimeter", "said": "तो। अं Fluke नहीं है, Aptech मल्टीमीटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aptech मल्टीमीटर"}}
{"product": "iBELL drill machine", "said": "हॅलो कोण बोलतंय मशीन आहे का बघावं लागेल नंतर सांगतो कंपनीने परत रेट वाढवले", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "मिल जाएगा, है अपने पास, बेस्ट प्राइस 205, होम डिलीवरी अवेलेबल है, 20 चार्ज.", "answer": {"in_stock": true, "price_inr": 205, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Ti sampli ho, company ne parat rate vadhavle", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "Dow sampla, Anabond ahe, chalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anabond"}}
{"product": "water pump seal kit", "said": "मतलब हाँ माल पड़ा है लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बोला, काय पाहिजे? बघा हो, पुट्टी आहे. पावणे सातशे रुपये. माणूस पाठवूनदेतो घरी", "answer": {"in_stock": true, "price_inr": 675, "delivery": true, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "sorry sir, abhi out of stock hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "पाँच सौ पचपन फाइनल हां सर वो मॉडल अवेलेबल है", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ सर, वो मॉडल अवेलेबल है. अपना लड़का छोड़ देगा घर पे. देखो बताओ क्या करना है. वो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "हो लाईट आहे GST धरून पाचशे पंचाहत्तर पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 575, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "स्टॉक में नहीं है. हाँ तो एस्ट्रल सीपीवीसी एल्बो दे सकता हूँ, 15 रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एस्ट्रल सीपीवीसी एल्बो"}}
{"product": "Easy4buy horn", "said": "हाँ, माल पड़ा है। एमआरपी नौ सौ पचासी है, पर हम छह सौ पचपन में देते हैं", "answer": {"in_stock": true, "price_inr": 655, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "Hello? kuch sunai nahi de raha, baad mein karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हाँ हां रखते हैं, पर अभी ख़त्म है, वो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "stock mein hai sir, aaj subah se light nahi hai, haan haan, sun raha hun, chaar so pachis... ek minute, rate list dekhta hun... haan chaar sao assi", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "यस, available है, खुला एक सत्तासी का, पूरा पैकेट तीन सौ साठ", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "मोटर नहीं है खत्म हो गई देखिए किर्लोस्कर 0.5 HP वॉटर पंप दे सकता हूँ 1250 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "किर्लोस्कर 0.5 HP वॉटर पंप"}}
{"product": "1 inch PVC elbow", "said": "wo model available nahi hai, Astral wala hai, pandra ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Astral"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हेलो, हाँ बोलो, हाँ जी, मिल जाएगा, सौ का है, इससे कम नहीं होगा.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "वो क्या है V-Guard नहीं है, Crompton 3 L geyser है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 3 L geyser"}}
{"product": "8 mm drill bit", "said": "हाँ सर, available है. बारह लोगे तो 120 के हिसाब से, एक का डेढ़ सौ. घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "एक गिऱ्हाईक आहे एक मिनिट माल संपला आज सकाळपासून लाईट नाही आवाज तुटतोय मोठ्याने बोला तर कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "बोला, काय पाहिजे? हो दादा, आहे, अडीचशे होतील, घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "हाँ भाई भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "Abhi stock nahin hai, kal tak aa jayega, aapke liye sadhe chhah hazar, umm delivery ho jayegi sir", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hai hai... nahi yaar, last peice subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "अं हाँ जी, मिल जाएगा. 61 लिखा है, छूट देकर पचास. दुकान से ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ जी सर, बताइए, हाँ हाँ, रखा है, ३८० रुपये लगेंगे, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "bola saheb, te kalach sampla, bagha white cement ghya, te chalel.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Apex Vivid helmet", "said": "आमच्याकडे आहे, पाच हजार शंभरचं आहे, तुमच्यासाठी सव्वाएकोणचाळीसशे, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 3925, "delivery": false, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "हाँ, कवर है. मतलब आपके लिए ४००. डिलीवरी का ५० एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "हम्म है भाई. दस का डिब्बा 1300 का, एक पीस 145", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हाँ जी, बल्ब रखा है, तीन सौ का है, इससे कम नहीं होगा, आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "hallo haa boliye kabza pada hai 70 lagega aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "है ना, कितने चाहिए? सोलह सौ पचास का है, इससे कम नहीं होगा. मतलब डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": false, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "देखो स्टॉक नहीं है अभी, JK 90 tyre है, साढे ग्यारह सौ का, लड़का भेज देता हूँ.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "JK 90 tyre"}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉक में है सर, रेट पौने चार सौ है सर, घर तक पहुँचा देंगे", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, रखा है. पौने तीन सौ फ़ाइनल. लड़का नहीं है भेजने को, आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "है ना, कितने चाहिए? 1140 रुपये लगेंगे.", "answer": {"in_stock": true, "price_inr": 1140, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "मिल जाएगा, है अपने पास। पौने चार सौ का है, इससे कम नहीं होगा।", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "वो हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "wrong number ahe.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Datsun Go", "said": "हाँ, रखा है। साढे तेरह सौ फाइनल प्राइस है", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हम्म हाँ जी, प्लास रखा है, दो सौ पचपन लगेगा, Kaltron 8 इंच प्लायर चाहिए तो वो भी है, तीन सौ पचास वाला", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "दोनशे साठला पडेल। आमच्याकडे आहे", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "kay aahe na udya sakali maal yeil, atta nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "yes sir, boliye, stock mein hai sir, last month 205 tha, ab 270 hai, boy ghar pe de jayega.", "answer": {"in_stock": true, "price_inr": 270, "delivery": true, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "है भाई, आपसे 350 लेंगे, लिखा 440 है", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "वो सेठ जी बाहर गए हैं, वही बताएँगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "हाँ, ट्यूब है. सवा आठ सौ रुपये.", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां हां हां, ऐकतोय. ती संपली हो. बाजारात माल कमी आहे सध्या. Exide आहे, सव्वा अकराशेला. हां आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Exide"}}
{"product": "bike headlight bulb", "said": "Stock samplay, sadhya nahiye, order dilay, don divsat yeil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "putty sampli nahi, ahe aathshe vees la lavto, MRP 1030 aahe delivery nahi karat", "answer": {"in_stock": true, "price_inr": 820, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "अभी available नहीं है order प्लेस किया है मंडे तक आएगा सेम डे डिलीवरी कर देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "Hai... arre nahin, wo dusra wala hai, ye wala nahin hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, रखा है, एक सौ पैंतालीस बोला क्या? माफ़ करना, डेढ़ सौ है", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Haan aahe, pavanetinashe plus GST, bara yeun gheun ja.", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "acha kam nahi hai bahut pada hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "हॅलो? आवाज येतोय का? हां, बोला. हो, मशीन आहे. पावणेसत्तावीसशेला पडेल. पाठवायची सोय नाही.", "answer": {"in_stock": true, "price_inr": 2675, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ भाई, बोलो, हाँ जी, स्टॉक में पड़ा है, रेट सौ है सर", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Han sir, boliye, ji, available hai, rate assi hai, arre ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "अच्छा स्टॉक में है। MRP 665 है, आपको 499 में दे देंगे। भेजने का इंतज़ाम नहीं है", "answer": {"in_stock": true, "price_inr": 499, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ हाँ रखते हैं पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "pickup karava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "नहीं मिलेगा भाई, Havells 1500 वॉट इमर्शन रॉड चलेगा? उसका सवा चार सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 1500 वॉट इमर्शन रॉड"}}
{"product": "Fluke multimeter", "said": "हो सर अवेलेबल आहे ४८३० थांबा लिस्ट बघतो सव्वासदतीसशे", "answer": {"in_stock": true, "price_inr": 3725, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "देखो नहीं है... रुको रुको, है, एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "हां बोला। उद्या फोन करा। एक गिऱ्हाईक आहे, एक मिनिट", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "हाँ सर, अवेलेबल है। 6 का फुल बॉक्स छह सौ पैंतीस, सिंगल पीस सवा सौ। एड्रेस भेजो, डिलीवर कर देंगे", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "Haan bolo, umm abhi nahi, order pe mangva denge, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "yes, available hai, 505 bola kya? galti se bola, paune chhah sau hai", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, रखा है, साढ़े तीन सौ फ़ाइनल.", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "जी बोलिए, कल सुबह आ जाएगा, अभी नहीं, पौने आठ सौ रुपये लगेंगे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "network issue hai aawaz break ho rahi hai abhi available nahi hai order place kiya hai monday tak aayega season time hai na demand zyada hai sir aaj bahut busy day hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "haa, wahi wala hai, rate 680 hai, bata dena.", "answer": {"in_stock": true, "price_inr": 680, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "जी, उपलब्ध है, हम्म लूज पीस 115 का, फुल पैकेट चार सौ तीस, आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "Bulb nahi hai, khatam ho gaya, chahiye toh RSCT 9W LED bulb de dunga.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "RSCT 9W LED bulb"}}
{"product": "tank float valve", "said": "umm abhi dukan band kar raha hun, kal subah phone karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "हाँ, है, साढ़े पाँच हज़ार रुपये काहै, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 5500, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "माल खत्म है. mechanical seal है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "mechanical seal"}}
{"product": "Mastech multimeter", "said": "हाँ हाँ हाँ है खुद आकर उठाना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "यस, अवेलेबल है, डिलीवरी की टेंशन मत लो, भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "हाँ तो Candes नही है Crompton 3 L geyser है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 3 L geyser"}}
{"product": "Vista 500 ml chain lube", "said": "हाँ जी सर, बताइए. अभी स्टॉक फिनिश हो गया. मतलब उसकी जगह Royal Enfield lube दे सकते हैं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Royal Enfield lube"}}
{"product": "4 socket extension board", "said": "है भाई, मिल जाएगा, आपके लिए पौने पाँच सौ, कूरियर कर देंगे", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "tube rakhi hai mil jayegi kya hai na aapse chaar sau sattar lenge likha panch sau chalis hai", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "umm haan, wahi wala hai, ghar pe bhej dunga, 50 lagega bhejne ka, uska rate sava teen sau, bhejne ka alag.", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "जीबोलिए, ख़त्म हो गया, Accessory Mart कार मैट चलेगा? उसका डेढ़ हज़ार है, आपके घर तक भिजवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Accessory Mart कार मैट"}}
{"product": "Xinor Nexo helmet", "said": "हां रखते हैं पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "हां तर संपलं होतं नाही एक आहे अजून तर पावणेएकवीसशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 2075, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो हो, मिळेल, पन्नास नगांचा बॉक्स तीन हजार पाचशेला, सुटा एक ऐंशी", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "पहले 1600 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "कल सुबह आ जाएगा, अभी नहीं, पौने पाँच हज़ार रुपये लगेंगे, आकर ले जाओ", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Aptech multimeter", "said": "maal khatam hai, matlab Themisto multimeter hai, uska rate savaa do sau", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Themisto multimeter"}}
{"product": "Xtandard helmet lock", "said": "ये दुकान नहीं है, घर का नंबर है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "arre nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "haan sir bataiye abhi available nahi hai order place kiya hai monday tak aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "स्टॉक नाही आहे सध्या", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "हां है प्राइस साढ़े पाँच सौ है होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 550, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो ठेवतो आम्ही आहे सव्वा एकवीसशे रुपयांना पडेल घरपोच पोहोचवून देऊ", "answer": {"in_stock": true, "price_inr": 2125, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "नमस्कार बोला ते मॉडेल अवेलेबल नाही 8 mm bolt अवेलेबल आहे चालेल का संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "8 mm bolt"}}
{"product": "Splendor brake shoe", "said": "हाँ, रखा है, आपसे 150 लेंगे, लिखा 165 है, कूरियर कर देंगे.", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "आहे की, मिळेल, तुम्हाला तेराशेला देतो, वीसच पीस उरलेत, एक गिऱ्हाईक आहे, एक मिनिट, हां हां, ऐकतोय, सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "बेसिकली अभी अवेलेबल नहीं है, ऑर्डर प्लेस किया है, मंडे तक आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ हाँ है अस्सी रुपये लगेंगे आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "हाँ रखते हैं पर अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Haan sir, bataiye. pankha nahi hai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "कव्हर आहे, मिळेल, प्राइस ४५५० आहे, संध्याकाळी घरी पोहोचेल", "answer": {"in_stock": true, "price_inr": 4550, "delivery": true, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "हां हां तर सगळं संपलं, तर.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "R-tek multimeter", "said": "हाँ, है। MRP 740 है, आपको 455 में दे देंगे।", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "यस, अवेलेबल है, पौने तीन सौ रुपये का है, शाम तक घर पहुँच जाएगा.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो आहे शिल्लक २३० रुपये होतील माणूस पाठवून देतो घरी", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ है। एक सौ बीस का।", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "देखिए वो हम नहीं रखते, Supreme 3/4 inch elbow रखा है, लड़का घर पे दे जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Supreme 3/4 inch elbow"}}
{"product": "1 inch PVC elbow", "said": "अभी बिजी हूं, आधे घंटे बाद फोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "नमस्ते, बोलिए, हाँ हाँ, है, पौने आठ सौ रुपये लगेंगे, ठीक है?", "answer": {"in_stock": true, "price_inr": 775, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "देखो हाँ है। लूज़ पीस दो सौ दस का, फुल पैकेट पौने चार सौ", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "ji sir, tell me, dekho nahi hai, sorry, SARK tap chalega? 205 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "SARK tap"}}
{"product": "Dewalt drill machine", "said": "हां जी सर बताइए छः हज़ार पाँच सौ का है हाँ जी स्टॉक में पडा है आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 6500, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "haan bolo, yes, available hai, chaar so tees laga denge, waise MRP chaar sau pachanve likha hai", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "acha Windek vala khatam hai, Michelin tyre hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin tyre"}}
{"product": "50 mm padlock", "said": "वो हम नहीं रखते। हाँ तो उसकी जगह Godrej दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Godrej"}}
{"product": "Aptech multimeter", "said": "स्टॉकमध्ये आहे. सव्वादोनशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हो, मशीन आहे, 5700 आहेसर, होम डिलिव्हरी आहे.", "answer": {"in_stock": true, "price_inr": 5700, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "आउट ऑफ stock नहीं है अवेलेबल है प्राइस ४० है", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "है भाई अच्छा रेट ११०० है डिलीवरी का १०० एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "स्टॉक में है सर। तो तो। 8 का... नहीं नहीं, गलत बोला, 10 का है।", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "अं है भाई, अं, सवा तीन सौ रुपयेका है", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, टेप शिल्लक आहे. तुम्हाला तीसला देऊ. माणूस पाठवून देतो घरी. सांगा काय करायचं. अं", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "यस available है छः सौ अस्सी रुपीज़ है आके ले जाओ ठीक है", "answer": {"in_stock": true, "price_inr": 680, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "हाँ जी ट्यूब रखी है मतलब देखो 1440 बोला क्या माफ़ करना 1740 है", "answer": {"in_stock": true, "price_inr": 1740, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "हाँ बोलिए, शाम तक माल आ जाएगा, अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "aahe aahe, ghari pathavto, pannas vegle, savvadonshe la aahe, delivery charge vegla.", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "haan hai apne paas 1640 ka padega home delivery nahin hai bhai bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 1640, "delivery": false, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "स्टॉकमध्ये आहे. 150 रुपये होतील. अं सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "मतलब बल्ब पड़ा है delivery की टेंशन मत लो भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "म्हणजे माहीत नाही हो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "haan sir, available hai, 20 peice vala box 3560 mein, khula ek 198 ka.", "answer": {"in_stock": true, "price_inr": 198, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "जी उपलब्ध है 675 रुकिए लिस्ट देखता हूँ छह सौ अस्सी", "answer": {"in_stock": true, "price_inr": 680, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "जी बोलिए, हाँ सर, अवेलेबल है, सौ का पड़ेगा.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हैलो कौन देखो नॉट अवेलेबल सर हैवेल्स 32A एमसीबी वाला पड़ा है वो ले लो त्योहार का सीज़न है बाज़ार में माल की कमी चल रही है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "हैवेल्स 32A एमसीबी"}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "khatam nahi hua, abhi bhi hai. sava unnis sau ka milega", "answer": {"in_stock": true, "price_inr": 1925, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "वो तो कल ही ख़त्म हो गया, फिलिप्स 20 वॉट एलईडी बैटन वाला चलेगा? उसका चार सौ सत्तर है, अं.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स 20 वॉट एलईडी बैटन"}}
{"product": "tank float valve", "said": "hai ji mil jaega ek sau nabbey rupaye delivery free hai bata dena", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "Arre thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "haan hai dekho das loge toh 275 ke hisaab se ek ka 299", "answer": {"in_stock": true, "price_inr": 299, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "हाँ हाँ, है, हम्म एमआरपी 940 है, पर हम 625 में देते हैं", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "हो ट्यूब आहे सुटा एक २६५ ला पूर्ण पेकेट ४००", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ जी, मिल जाएगा। बारह लोगे तो 170 के हिसाब से, एक का दो सौ।", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ हाँ है ठीक है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "एक मिनट... हाँ बोलिए. हाँ, 4 पीस पड़े हैं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "haan ahe, pathavla asta, pan aaj porga nahi, yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "yes sir, boliye, stock me hai sir, saadhe teis sau plus GST.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "Ho bola, wrong number ahe.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "रेट तीन हजारच्या आसपास असेल, स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "है भाई मिल जाएगा आपके लिए 725 आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": true, "price_inr": 725, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Maal kal aayega, abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat", "said": "hai, bilkul hai, 1450 lagega ab, pehle 1780 ka tha.", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "हाँ बोलिए सर देखो है भाई सवा दो सौ रुपये का है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "हॅलो, हां बोला, आहे ना, म्हणजे 165, GST वेगळा.", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हैलो, कौन? है भाई, मिल जाएगा, ढाई सौ का है, इससे कम नहीं होगा", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बल्ब पडलाय, एक पीस 120 ला मिळतो, पॅकेट सव्वाचारशेला, येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "Havells LED torch", "said": "मिल जाएगा है अपने पास पौने तीन सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "हो, आहे शिल्लक, तर, 3000 लापडेल, सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 3000, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, रेडी stock है, आके ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Agrico TATA plier", "said": "haan, mil jayega bhai, paune tin sau ka milega, bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": 275, "delivery": false, "alternative": null}}
{"product": "AutoClean Maruti Baleno wiper blade", "said": "माल खत्म है जी.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "bagha stock madhe aahe, vis nagancha box naushe panchvis la, sutta ek panchavann", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "aapke liye sau han sir available hai free delivery hai", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Mihsp water pump", "said": "pick up karava lagel, mhanje ajun kahi?", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Vega Crux helmet", "said": "जी, उपलब्ध है। सात सौ सत्तर, जीएसटी एक्स्ट्रा।", "answer": {"in_stock": true, "price_inr": 770, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "kay aahe na bulb aahe, milel. ek piece 270, 10 ghetle tar 225 pramane lavto.", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "नॉट अवेलेबल sir ball valve है चलेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ball valve"}}
{"product": "Splendor brake shoe", "said": "हो सर, बोला। नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय। एक मिनिट, चेक करून सांगतो। आज खूप बिझी आहे सर", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो? आवाज येतोय का? बोला. आमच्याकडे आहे. 80 ला पडेल. पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ बोलिए सर. देखो हाँ है. सौ लगेगा", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "हो, कव्हर शिल्लक आहे। बघा छापील किंमत ५७५ आहे, आम्ही ५०० ला देतो। येऊन न्यावं लागेल।", "answer": {"in_stock": true, "price_inr": 500, "delivery": false, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "अं नॉट available सर, Havells पंप वाला available है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells पंप"}}
{"product": "water pump seal kit", "said": "नहीं है भाई, खत्म हो गया, local seal kit है, सेम क्वालिटी, delivery नहीं होती, पिकअप करना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local seal kit"}}
{"product": "8 mm drill bit", "said": "येस, बोला. आत्ता मिळत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "stock madhe ahe, bara 1500 la lavto, MRP 1760 ahe, courier karto sir", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "बेस्ट प्राइस ढाई सौ हां है कोई प्रॉब्लम नही अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "delivery ho jayegi", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Makita drill machine", "said": "हां बोला. मशीन संपली नाही, आहे. सव्वाअठ्ठावीसशे रुपये होतील.", "answer": {"in_stock": true, "price_inr": 2825, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "hallo han awaaz aa rahi hai bolo matlab dekhna padega bhai yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "हाँ तो सॉरी सर, अभी आउट ऑफ स्टॉक है, उसकी जगह Anchor 6 सॉकेट एक्सटेंशन board ले सकते हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Anchor 6 सॉकेट एक्सटेंशन board"}}
{"product": "Rolson screwdriver bit set", "said": "हाँ जी, मिल जाएगा, देखो डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "bulb padlay kay aahe na daha ghetle tar donshe vees pramane ek piece donshe panchechalis", "answer": {"in_stock": true, "price_inr": 245, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "बल्ब आहे मिळेल 250 रुपयांना पडेल पाठवायची सोय नाही सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Osram LED headlight", "said": "dekho abhi khatam hai, shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हेलो? हाँ, बोलो भाई, सीज़न टाइम है ना, डिमांड ज़्यादा है, सर आज बहुत बिज़ी डे है, एक कस्टमर खड़ा है, होल्ड करो, हाँ सर, वो मॉडल अवेलेबल है, पहले 165 था, अब डेढ़ सौ हो गया, सस्ता हुआ है", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape ka stock check karke batata hun, acha aaj subah se light nahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर संपलीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "देखिए हाँ, माल पड़ा है, दस रुपये का है.", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "हैलो? हाँ, आवाज़ आ रही है. पता नहीं भाई, बाद में कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ बोलिए सर, पहले सवा तीन सौ का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji, available hai, paintalis rupaye ka hai.", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "हां बोलिए, गीज़र खत्म नहीं हुआ, है, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "stock madhe ahe, ekonisshe fix, theek aahe?", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "संपलं होतं... नाही, एक आहे अजून, दुकानातून न्यावं लागेल", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "आहे, अजून ७ पीस पडलेत, रेट ४० आहे सर.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ जी मिल जाएगा एक पीस 280, दस लोगे तो 245 के हिसाब से", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "abhi busy hun, aadhe ghante baad phone karna market men maal ki kami chal rahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ho, battery aahe, 1200 hotil, courier karto", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "अभी नहीं है, order पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हॅलो, कोण बोलतंय? हां हां, ऐकतोय, मोटर आहे, मिळेल, तुम्हाला पावणेअकराशेला देतो, बाजारात माल कमी आहे सध्या, अं 10 नग शिल्लक आहेत, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 1075, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "ट्यूब आहे मिळेल एक पीस 405 ला मिळतो पॅकेट 620 ला कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 620, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "स्टॉक में है, रेट मालिक बताएँगे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "जी, उपलब्ध है, आपको पंद्रह, MRP तो अठारह है, होम delivery available है.", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ बोलिए sir, शाम तक पहुँचा देंगे.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "ha sir, bataiye, kal subah tak aa jayega, abhi nahi hai, ladka nahin hai bhejne ko, aake le jao.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells Oro switch", "said": "हो दादा, आहे। बघा सव्वा पाचशे रुपये, कमी नाही होणार। पिकअप करावा लागेल", "answer": {"in_stock": true, "price_inr": 525, "delivery": false, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "haan sir, available hai, 1120... nahi nahi, 865 hai", "answer": {"in_stock": true, "price_inr": 865, "delivery": null, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "oil khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "देखो वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ बोलिए। कंपनी ने फिर रेट बढ़ा दिया। मिल जाएगा, है अपने पास। त्योहार का सीज़न है ना। लड़का चाय लेने गया है। 40 का पड़ेगा। आवाज़ कट रही है, ज़ोर से बोलिए। डिलीवरी का 20 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "Wuerth HC 5W-40 4 L engine oil", "said": "Wuerth नहीं है, शेल 10 वॉट ऑयल है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शेल 10 वॉट ऑयल"}}
{"product": "8 mm drill bit", "said": "है है ना, कितने चाहिए? एक सौ पैंतीस का... नहीं नहीं, गलत बोला, एक सौ बीस का है, लड़का छोड़ देगा घर पर", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "बोला ना कव्हर पडलंय सव्वातीनशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "अं हां रेडी स्टॉक है रेट साढे चौंतीस सौ है ओके", "answer": {"in_stock": true, "price_inr": 3450, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "अभी स्टॉक फिनिश हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हैलो, कौन? आवाज़ कट रही है, ज़ोर से बोलिए. एक ग्राहक खड़ा है, एक मिनट. नहीं है जी, ख़त्म. big टेफ्लॉन रोल दे सकता हूँ, बीस रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big टेफ्लॉन रोल"}}
{"product": "1 inch PVC elbow", "said": "Wo item hum keep nahin karte.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "aekonees nag shillak aahet kay aahe na pavne terashe hotil", "answer": {"in_stock": true, "price_inr": 1275, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "बघा लाईट पडलीये, घरपोच नाही, तुम्हाला यावं लागेल.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Easy4buy horn", "said": "wo to kal hi khatm ho gaya Favourite Bikerz horn chalega? 600 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz horn"}}
{"product": "Fevicol SH 1 kg", "said": "मिल जाएगा है अपने पास भेज देंगे 30 रुपये डिलीवरी चार्ज लगेगा माल का 320 डिलीवरी अलग", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Bola dada, bazarat maal kami aahe sadhya, aahe aahe, kal Ravivar hota mhanun dukan band hota, bagha satashe pannas rupaye, kami nahi honar, manus pathavto ghari, kay sangu, sakal pasun gardi aahe", "answer": {"in_stock": true, "price_inr": 750, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Ha ji, boliye dekho woh hum nahi rakhte.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "हाँ सर, वो मॉडल अवेलेबल है, रेट मालिक बताएँगे, अच्छा डिलीवरी बंद है अभी, दुकान से ले लो", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "Hello? aawaj yetoy ka? haan bola, bulb padlay, rate 80 ahe sir, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "येस, बोला, हो, ऑइल शिल्लक आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हम्म आपने गलत नंबर मिलाया है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Euro Care Hyundai Creta wiper blade", "said": "हेलो, हाँ बोलो। अच्छा लड़का छोड़ देगा घर पर।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HP engine oil", "said": "स्टॉक में है. २९५ लगा देंगे, वैसे MRP ३६० लिखा है", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "Helo, ha bolo rate paune sattais sau ke aas paas hoga, stock malik se puchna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crispy LED torch", "said": "mahit nahi, godown madhe baghava lagel kaka.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "hello, haan bola, te model available nahi, aawaj tutatoy, motyane bola, haan haan, aiktoy, Bosch angle grinder ghya, te chalel, kal Ravivar hota mhanun dukan band hota, aaj sakali light navhti, courier karto sir", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Bosch angle grinder"}}
{"product": "Anchor 150 mm exhaust fan", "said": "अभी नहीं है ऑर्डर पे मंगवा देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "haan ha, bolo, abhi khatam hai bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "स्टॉक में है, डेढ़ सौ रुपये, अं घर पर भेज देंगे, बीस अलग से", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हेलो, हाँ बोलो, मतलब प्लीज़ थोड़ी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goodyear tyre", "said": "संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "kya hai na haan, machine hai, delivery ho jayegi sir.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हो आहे डिलिव्हरी नाही दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "नॉट available sir. Candes 1200 watt room heater है, सेम क्वालिटी. फ्री डिलीवरी है.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Candes 1200 watt room heater"}}
{"product": "Activa clutch plate", "said": "हाँ sir, वो मॉडल available है, आपके लिए सात सौ साठ, खुद आके ले जाना पड़ेगा", "answer": {"in_stock": true, "price_inr": 760, "delivery": false, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "नमस्ते, बोलिए, हाँ हाँ, है, आपसे उन्नीस सौ पचास लेंगे, लिखा दो हज़ार तीन सौ है", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "सॉरी, आउट ऑफ स्टॉक आहे, तर बजाज रॉड अवेलेबल आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बजाज रॉड"}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "आहे ना। 1900 आहे, 1700 नाही", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "Haan bolo, market mein maal ki kami chal rahi hai, woh hum nahi rakhte, kal Sunday tha toh dukaan band thi, Diwali ka season hai na, sab yahi maang rahe hain, bhaiya aap ruko do minute, phone pe hun, aur kuch chahiye?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हाँ तो अभी आउट ऑफ स्टॉक है, नया स्टॉक कल आ रहा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock nahi hai abhi, kya hai na Bosch Pulsar 180 spark plug wala hai, same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch Pulsar 180 spark plug"}}
{"product": "8 mm drill bit", "said": "हेलो? हाँ, बोलो भाई। है है भाई, मिल जाएगा। सवा सौ का पड़ेगा। अं।", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "pankha ahe, milel, terashe pannas rupayanna padel", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "अठरा नग शिल्लक आहेत, पावणेतीनशे रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "मतलब खुद आके ले जाना पडेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciTools water pump", "said": "बोला ना। पावणेबाराशे फिक्स। हो, मिळेल", "answer": {"in_stock": true, "price_inr": 1175, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "हां जी मिल जाएगा पिछले महीने 1120 था अभी 1500 देखो शाम तक पहुंचा देंगे", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ho, milel. sattar rupaye, kami nahi honar. haan tar Crompton 2.25 mfd capacitor pan aahe, te shahaainshi la.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "कव्हर संपलंय. नेटवर्क प्रॉब्लेम आहे, आवाज कट होतोय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "mana kisne kiya? hai na, sadhe chaar sau ka hai", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब संपला सध्या नाहीये ऑर्डर दिलीय दोन दिवसांत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan sir, bataiye, umm wo item hum keep nahi karte, bolo kya karna hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "हाँ रखते हैं, पर अभी ख़त्म है, Retina cover है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina cover"}}
{"product": "Activa air filter", "said": "haan, stock mein hai, ek sau assi ka hai, umm pahuncha dunga, fikar mat karo", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Crompton Hill Briz ceiling fan", "said": "हाँ रखा है 2600 नहीं नहीं इक्कीस सौ है", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "स्टॉक में है, MRP एक हज़ार दो सौ पचास है पर ग्यारह सौ लगा देंगे", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो, ऑइल आहे. साडेचारशे रुपयांना पडेल. पोरगा घरी सोडून देईल", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "नहीं नहीं, है ना. साढे उनतीस सौ में दे देंगे.", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "जी, उपलब्ध है. दस का डिब्बा दो हज़ार दो सौ पचास का, एक पीस ढाई सौ. शाम तक घर पहुँच जाएगा.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "पंखा संपला, बघा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ बोलिए सर, यस, अवेलेबल है, अब प्राइस एक सौ बीस है, पहले 110 था", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप खत्म नहीं हुई, है, हम्म 10 फाइनल, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "अभी नहीं बता सकता, बाद में फ़ोन करो, एक ग्राहक खड़ा है, एक मिनट.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho, thevto aamhi, ahe purna packet ainshi, sutte pan ahet nau la ek.", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "साढे सात सौ लगेगा वो क्या है प्लेट है मिल जाएगी", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "aahe aahe, pavanesatashe la deu, chhapil kimmat 920 aahe, delivery deto", "answer": {"in_stock": true, "price_inr": 675, "delivery": true, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "हाँ जी मिल जाएगा पर piece पौने दो सौ, 6 लो तो एक सौ साठ लगेगा आपके घर तक भिजवा देंगे", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "एक मिनिट हां... बोला, हो, ऑइल शिल्लक आहे, चारशे साठ रुपये होतील, कुरिअर करतो", "answer": {"in_stock": true, "price_inr": 460, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "बरं पंखा आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "kya hai na haan, maal pada hai, list price 540, discount ke baad 425.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हैलो हाँ आवाज़ आ रही है हाँ रखा है वो साढ़े तेरह सौ फ़ाइनल तो अपना आदमी दे आएगा", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Sage Square helmet", "said": "haan, kaun? acha, bolo. haan toh kal phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "यस सर, बोलिए, बेसिकली पंद्रह का है, इससे कम नहीं होगा, हाँ जी, मिल जाएगा, बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan ji, putty available hai, company ne rate badla, ab 980 hai, pehle 775 tha.", "answer": {"in_stock": true, "price_inr": 980, "delivery": null, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "stock nahi hai abhi, Accessory Mart car mat hai, same quality, ladka nahin hai bhejne ko, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Accessory Mart car mat"}}
{"product": "Retina steering cover for Alto", "said": "Cover aahe, milel, price pavaneteenshe aahe", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "haan bola haan aahe sahashe panchyannav fix", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ladka dekh ke batayega, ruko zara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "है ना, कितने चाहिए? आपसे पाँच सौ चालीस लेंगे, लिखा छह सौ पच्चीस है, लड़का घर पे देजाएगा.", "answer": {"in_stock": true, "price_inr": 540, "delivery": true, "alternative": null}}
{"product": "Adima LED torch", "said": "yes sir, boliye, season time hai na, demand zyada hai, sir aaj bahut busy day hai, yes, available hai, matlab best price ek sao pachasi, free delivery hai.", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "अं शिल्लक नाही दादा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "देखिए प्लेट नहीं है, खत्म हो गई", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "टेप आहे, मिळेल, 18... थांबा, लिस्ट बघतो... 20, पाठवायची सोय नाही", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "हाँ जी, मिल जाएगा, 810 MRP लिखा है, 540 लगा देंगे.", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "आपको शॉप से पिकअप करना होगा", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "Hallo? ha, awaaz aa rahi hai, bolo, woh hum nahi rakhte, arey kya bataun, subah se bahut rush hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "965 होतील हो ठेवतो आम्ही आहे", "answer": {"in_stock": true, "price_inr": 965, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "देखो हाँ रखते हैं पर अभी ख़त्म है Rainfun car cover वाला रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rainfun car cover"}}
{"product": "Legrand Arteor 25A switch", "said": "हाँ सर, वो मॉडल available है, पर piece 275, 10 लो तो 260 लगेगा.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "लाईट आहे, मिळेल, घरी पाठवतो, 30 वेगळे, आयटमचे अठराशे पन्नास रुपये, डिलिव्हरी चार्ज वेगळा.", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "सौ लगेगा, यस, अवेलेबल है, सर आज बहुत बिज़ी डे है, सीज़न टाइम है ना, डिमांड ज़्यादा है, घर पर भेज देंगे, और कुछ चाहिए सर?", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "मोटर नही है खत्म हो गई Kirloskar 0.5 HP वॉटर पंप चलेगा उसका साढ़े उन्नीस सौ है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar 0.5 HP वॉटर पंप"}}
{"product": "Allure Auto car mat for Zest", "said": "haan hai, terah sau pachas rupaye, apna aadmi de aayega.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "सव्वा तीनशे रुपयांना पडेल, हो, मिळेल, पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "mhanje sadebechalisshe la deto, aahe ki, milel, pathavayla manus nahi, ithe yeun ghya.", "answer": {"in_stock": true, "price_inr": 4250, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "यस, अवेलेबल है, price कन्फर्म करके बताता हूँ, आपको shop से पिकअप करना होगा.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Dewalt drill machine", "said": "Kya hai na machine rakhi hai, mil jayegi company ka rate aath hajar hai, main sava saat hazar lunga", "answer": {"in_stock": true, "price_inr": 7250, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "hello, han boliye, yes, available hai, rate seth ji batayenge.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली Birla wall putty 40 kg आहे सहाशे नव्वदला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Birla wall putty 40 kg"}}
{"product": "Apex Vivid helmet", "said": "हो, ठेवतो आम्ही, पण आत्ता संपलंय, Sonik helmet चालणार असेल तर सांगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik helmet"}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "Light aahe, milel, aata 475 aahe, aadhi 360 hota", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "hai bhai boss.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "Pehle pone panch sau ka tha, abhi hai ki nahin dekhna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "mil jaega hai sava chauntis sau lagega bhej denge 40 rupees delivery charge lagega", "answer": {"in_stock": true, "price_inr": 3425, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "not available sir, arre Legrand 25A switch hai, 319 ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 25A switch"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "अच्छा, पंखा नहीं है, खत्म हो गया, Usha fan वाला चलेगा? उसका पौने ग्यारह सौ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha fan"}}
{"product": "Bosch Eco wiper blade", "said": "माहीत नाही हो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "हम्म। सर एक मिनट, चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "यस सर, बोलिए, ओनर अभी नहीं हैं, बाद में कॉल कीजिए, आज सुबह से लाइट नहीं है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, आहे. पंचवीस रुपये होतील. संध्याकाळपर्यंत पाठवून देतो. कळवा मग.", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "यस sir, बोलिए। अभी अवेलेबल नहीं है, order प्लेस किया है, मंडे तक आएगा। फ्री delivery है।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crasts LED torch", "said": "हेलो बोलो भाई देखो अभी आउट ऑफ स्टॉक है नया स्टॉक कल आ रहा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip for Apache", "said": "हाँ जी, बताइए. वो गलत नंबर है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "खत्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "aahe ki, milel, 10 ghetle tar 125 pramane, ek piece 150, bagha delivery nahi karat", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Orient ABS room heater", "said": "ते कालच संपलं, हां तर कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Stock madhe ahe, bara pathavayla kahi problem nahi.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हॅलो हॅलो, कोण बोलतंय? वायर पडलीये, आता एक हजार तीनशे पन्नास आहे, आधी 1030 होता", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "हैलो, कौन? हेलमेट नहीं है, खत्म हो गया। Sage Square helmet चलेगा? उसका आठ सौ पच्चीस है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sage Square helmet"}}
{"product": "CSI 1 L geyser", "said": "नमस्ते बोलिए वो हां है आपके लिए ११०० बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "पौने तीन हज़ार का आता है लेकिन अभी ख़त्म है अभी stock नहीं है कल तक आ जाएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "haan bhaiya, bataiye, hai bhai, do sau assi mein ho jayega, upar MRP 500 likha hai.", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "बल्ब आहे मिळेल बारा नगांचा बॉक्स 2600 ला सुटा एक 240", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "ji haan, rakha hai, aapko ek hajaar mein de denge, MRP ek hajar teen sau pachas hai", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "हैलो? हाँ, आवाज़ आ रही है। हाँ तो प्लास ख़त्म हो गया। JCB रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "JCB"}}
{"product": "tank float valve", "said": "हां, माल पड़ा है। पौने तीन सौ रुपये का है। घर पर भेज देंगे, 60 अलग से", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Tata Agrico plier", "said": "हो, पक्कड शिल्लक आहे, रेट रेट सव्वा दोनशे आहे.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हो दादा आहे 80 फिक्स सांगा काय करायचं", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "हाँ तो अभी नहीं है, Legrand 25A स्विच है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 25A स्विच"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "नाही हो, संपलं ते", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "abhi khatam hai bhai daybetter 25w led bulb hai paune teen sau ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "daybetter 25w led bulb"}}
{"product": "Pulsar 150 spark plug", "said": "haan toh hai... arrey nahin, wo dusra wala hai, ye vala nahin hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "एक मिनट... हां बोलिए. आवाज कट रही है, जोर से बोलिए. वो कंपनी ने फिर रेट बढा दिया. ट्यूब है कि नही, देखकर बताता हूं. अरे क्या बताऊं, सुबह से बहुत भीड है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, उषा 1250 एमएम सीलिंग फैन है, वो ले लीजिए, फ्री delivery है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "उषा 1250 एमएम सीलिंग फैन"}}
{"product": "Bajaj immersion rod", "said": "हैलो हाँ आवाज़ आ रही है हाँ रखा है साढ़े चार सौ लगा देंगे वैसे MRP छह सौ लिखा है", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola. nahi milnar. haan tar 3 inch hinge aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch hinge"}}
{"product": "fan capacitor 2.5 mfd", "said": "हां हां माल पडा है एक सौ बीस है १०५ नहीं", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "हाँ सर, वो मॉडल अवेलेबल है, 1100 लगेगा, डिलीवरी नहीं होती, पिकअप करना पड़ेगा, बताइए क्या करना है", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Wire sampli nahi, aahe, 1250 rupees hotil, Finolex 2.5 sq mm wire pan ahe, te 1480 la", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "helmet rakha hai, mil jayega, ladka de aayega, pachas rupaye auto ka, uska rate savaa saat sau, bhejne ka alag", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "नहीं है रुको रुको है एक पड़ा है 299 लगेगा आपको आकर लेना पड़ेगा देखो", "answer": {"in_stock": true, "price_inr": 299, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "15 hotil, ho, thevto aamhi, ahe.", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ho tape milel daha ghetle tar 8 pramane ek pees daha", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan aahe, daha ghetle tar athara pramane, ek piece vees.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हो, मशीन आहे, म्हणजे 1210 होतील.", "answer": {"in_stock": true, "price_inr": 1210, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "ho aahe, donshe sangitla ka? chuklo, pavanedonshe aahe, boy ghari deliver karel", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Rate tis hai, par abhi stock nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Pehle 260 ka tha, abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Helo, haa boliye, umm sir ek minute, chek karke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "rate sava chah sau hai sir. yes, available hai.", "answer": {"in_stock": true, "price_inr": 625, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "हां बोला साहेब. पंखा संपला नाही, आहे. तुम्हाला 1440 ला देतो. डिलिव्हरी फ्री आहे. सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": 1440, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "नमस्ते, बोलिए, स्टॉक में है, आठ सौ तीस रुपये, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए.", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "available hai, paune atthais sau ka padega, kya hai na aur kuch?", "answer": {"in_stock": true, "price_inr": 2775, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "नमस्ते, बोलिए, ऑयल नहीं है, खत्म हो गया, अं शेल 20W-40 इंजन ऑयल वाला है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शेल 20W-40 इंजन ऑयल"}}
{"product": "16A MCB", "said": "Arre wo toh kal hi khatam ho gaya. Havells 20A MCB hai, wo le lo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 20A MCB"}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, मिळेल, किंमत नंतर सांगतो, आत्ता लिस्ट नाही", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "थोड़ी देर बाद कॉल करो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "not available sir bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "कंपनी ने फिर रेट बढ़ा दिया. अरे क्या बताऊँ, सुबह से बहुत भीड़ है. नहीं है. Xtandard वाला चलेगा? उसका 185 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Xtandard"}}
{"product": "AmiciTools water pump", "said": "Ho, motor milel, bara dukanatun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ji boliye haan toh abhi dukan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "है भाई, पैंतीस है, चौंतीस नहीं", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "ऐसा नही है कि नही है है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो, ठेवतो आम्ही, आहे, 460 रुपये, येऊन घेऊन जा.", "answer": {"in_stock": true, "price_inr": 460, "delivery": false, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "pick up karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "Ji, available hai saat sau pandrah rupaye.", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Ji haan, rakha hai. rate sau hai. aake le jao.", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "नमस्ते, बोलिए, ताला नहीं है, खत्म हो गया, अं गोदरेज 60 एमएम ताला वाला चलेगा? उसका चार सौ पचास है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "गोदरेज 60 एमएम ताला"}}
{"product": "Camel drill machine", "said": "हाँ है. आपके लिए 1510. हाँ, 14 piece पड़े हैं.", "answer": {"in_stock": true, "price_inr": 1510, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "hello, bola na sadeektisshe fix ho, milel.", "answer": {"in_stock": true, "price_inr": 3150, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "हैलो, कौन? है ना, कितने चाहिए? पिछले महीने दो हज़ार दो सौ पचास था, अभी ढाई हज़ार. अभी डिलीवरी बंद है", "answer": {"in_stock": true, "price_inr": 2500, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "सौ प्राइस है। अच्छा। बेसिकली हाँ, माल पड़ा है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
'''.strip().splitlines()]
TRAIN_V2 = [json.loads(line) for line in r'''
{"product": "CEAT 3.00-18 tyre", "said": "No no, it's there. We'll deliver by evening", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch drill machine", "said": "kami nahi, bharpur ahe, rate sadenau hazar ahe", "answer": {"in_stock": true, "price_inr": 9500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "संपलं काका", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not available right now.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "हो, बल्ब शिल्लक आहे। बरं सुटा एक २९० ला, पूर्ण पॅकेट ४३०", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "tyre sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "aj nahin hai, kal aayegaa.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "tube padliye sutta ek 940 la purna packet 1600 porga pathavto", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "ख़त्म हो गया। Daybetter वाला चलेगा? उसका दो सौ सत्तर है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter"}}
{"product": "Bosch screwdriver bit set", "said": "arre haa ready stock hai tin sau pachattar final price hai bhejne wala koi nahin hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 375, "delivery": false, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "Yes, available, our price is 1,402, MRP is 2,000, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 1402, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ठेवतो आम्ही, पण आत्ता संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "That we don't have. I can give Golden Bullet drill instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Golden Bullet drill"}}
{"product": "Havells 1200 mm ceiling fan", "said": "arre kya bataun, subah se bahut rush hai, haan ji, mil jayega, aapko number kisne diya? acha thik hai, 2510... ek minute, rate list dekhta hun... haan sattais sau pachas, market mein maal ki kami chal rahi hai, ghar pe bhej denge.", "answer": {"in_stock": true, "price_inr": 2750, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "wo model available nahin hai, Michelin wiper available hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin wiper"}}
{"product": "BMB welding machine", "said": "taintalis sau lagega, acha haan, machine hai", "answer": {"in_stock": true, "price_inr": 4300, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "हो, आहे शिल्लक। १८०० रुपये, कमी नाही होणार। पोरगा पाठवतो, ६० रुपये रिक्षाचे।", "answer": {"in_stock": true, "price_inr": 1800, "delivery": true, "alternative": null}}
{"product": "Agrico TATA plier", "said": "पक्कड संपली नाही, आहे, 255 रुपये", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "aahe na, mhanje sutta ek ekshe sattar la, purna pyaket pavaneteenshe.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "नल है मिल जाएगा 320 MRP लिखा है 210 लगा देंगे", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan bolo, available hai, 260 final.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes, we keep that, I would send it, but my boy is on leave today, come and take it", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "haan sir, bataiye, 90 rupaye hai, han, maal pada hai, network issue hai, aawaz break ho rahi hai, delivery nahi hoti, pickup karna padega, dekho season time hai na, demand jyada hai", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1.5 sq mm nahi hai, 1 sq mm wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm"}}
{"product": "Shakti 250A welding machine", "said": "मशीन नाहीये, संपली, Vormir वाला आहे, 5130 ला, पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Vormir"}}
{"product": "Philips 5W LED bulb", "said": "Haan ahe, kya hai na matlab panchvis ghetle tar ekshe pandhara pramane, ek piece savvashe", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "Abhi nahin, order pe mangva denge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "haan, wahi wala hai bhaiya", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "एक मिनिट हां... बोला, पंधराशे पन्नास फिक्स, हो, आहे", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो आवाज येतोय का हां बोला बारा नग शिल्लक आहेत अजून काही", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, oil available hai, 410 rupaye, matlab delivery nahin karte, aake le jao", "answer": {"in_stock": true, "price_inr": 410, "delivery": false, "alternative": null}}
{"product": "Cheston angle grinder", "said": "हो, बोला दादा, ते आमच्याकडे नसतं, बरं Khadija grinder आहे, त्याचे साडेबाराशे रुपये.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Khadija grinder"}}
{"product": "teflon tape", "said": "टेप आहे, मिळेल, दहा घेतले तर बावीस प्रमाणे, एक पीस पंचवीस, डिलिव्हरी करत नाही.", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Yes, it's there. If you take 12 it's 115 each, one piece is 130", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "It's not finished, we have plenty", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "Yes sir, we have. Basically, it was 235, now reduced to 199.", "answer": {"in_stock": true, "price_inr": 199, "delivery": null, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "aahe, donch urlet, price charashe tees aahe.", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Moonstruck 2000W immersion rod", "said": "आमच्याकडे आहे, रेट कमी झालाय, पाचशे पंच्याऐंशी होता, आता सव्वा पाचशे", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "Michelin tyre", "said": "हाँ बोलिए. वो हम नहीं रखते. बताइए क्या करना है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "milel na, ahe, sutta pannas la, 10 nagancha box ghetla tar charshe sath, porga pathavto, dekho aur kuch?", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "Roots horn", "said": "नमस्कार, बोला. शिल्लक नाही. Minda Black वाला घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Minda Black"}}
{"product": "10W-30 engine oil 1 L", "said": "ho sir, bola, kay aahe na sorry, out of stock aahe, Diwali chi gardi aahe na", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crasts LED torch", "said": "हैलो, कौन? हाँ, रखा है. आठ सौ पैंसठ रुपये. भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 865, "delivery": false, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "haan, kaun? acha, bolo, abhi khatam hai bhai, chahiye toh Orient heater de dunga, home delivery nahi hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Orient heater"}}
{"product": "9W LED bulb", "said": "han, bees piece pade hain, ek sau tees rupaye, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 130, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "सातशे साठ फिक्स. पुट्टी संपली नाही, आहे. हां तर ठीक आहे?", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "7Ah nahi hai, 5Ah wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "5Ah"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ रखा है एक पीस 92 का मिलता है पैकेट 415 का", "answer": {"in_stock": true, "price_inr": 415, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "aaj nahi, udya yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "बरं BMB संपलं, शक्ती मशीन आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "शक्ती मशीन"}}
{"product": "Carzex LED headlight", "said": "लाईट पडलीये, अठराशे होतील.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "yes available hai ek sau chalis final", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "haan haan, bolo, market mein maal ki kami chal rahi hai, umm ji, available hai, aaj subah se light nahi hai, kal Sunday tha toh dukaan band thi, bhaiya aap ruko do minute, phone pe hun, 2499 ka hai, isse kam nahi hoga, delivery ho jayegi, bata dena.", "answer": {"in_stock": true, "price_inr": 2499, "delivery": true, "alternative": null}}
{"product": "AmiciTools water pump", "said": "अभी नहीं है, ऑर्डर दिया है, दो दिन में आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल। 40 ला देतो। दुकान दहा वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "ho bola, sade saha hajar la deto, haan ahe", "answer": {"in_stock": true, "price_inr": 6500, "delivery": null, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Yes... one minute, no, that got finished yesterday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "मशीन पडलीये. रेट ५२०० आहे.", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "helmet sampla nahi aahe savva aathshe hotil pathavla asta pan aaj porga nahi yeun ghya", "answer": {"in_stock": true, "price_inr": 825, "delivery": false, "alternative": null}}
{"product": "Hella horn", "said": "Haan ha, hai, acha nau sau das, aath sau nahin bola maine.", "answer": {"in_stock": true, "price_inr": 910, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "३५० रुपये कमी नाही होणार स्टॉकमध्ये आहे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "हॅलो आवाज येतोय का हां बोला ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "३३० रुपयांना पडेल। हो, ऑइल शिल्लक आहे।", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "hello? haan, awaaz aa rahi hai, bolo, haan haan, hai, umm chaar sau assi rupaye ka hai, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 480, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हां बोला साहेब, हो दादा, आहे, दिवाळीची गर्दी आहे ना, रेट पंचवीस आहे, डिलिव्हरी नाही, दुकानात येऊन घ्या, कंपनीने परत रेट वाढवले, पोरगा चहा आणायला गेलाय, बाजारात माल कमी आहे सध्या", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "मतलब ५२० का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "आज नाही, उद्या येईल। बरं पंधरा होतील।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "आहे ना, 355 प्लस GST", "answer": {"in_stock": true, "price_inr": 355, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai aur kuch?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "milel na aahe 12 ghetle tar 50 pramane ek pees 60 gharpoch milel", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Modx steering cover for i10", "said": "Cover nahi hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Motul 20W-40 engine oil", "said": "मिल जाएगा, है अपने पास, मतलब हमारा रेट 305, MRP तो 325 है.", "answer": {"in_stock": true, "price_inr": 305, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो? आवाज येतोय का? हां, बोला, हो, आहे, रेट शेठ सांगतील", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate sampli nahi, aahe, aadhi 625 hota, ata 780 zala, gharpoch nahi, tumhala yava lagel.", "answer": {"in_stock": true, "price_inr": 780, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "No, we don't have it. We have white cement, same thing", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "white cement"}}
{"product": "Havells 1500W immersion rod", "said": "हॅलो आवाज येतोय का हां बोला थांबा बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RiderShine bike cover for Activa", "said": "Available hai, koi problem nahi, teen sao bees mein de dunga, dukaan nau baje tak khuli hai, free delivery hai.", "answer": {"in_stock": true, "price_inr": 320, "delivery": true, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "haan, motor hai, season time hai na, demand jyada hai, chhabees sau ka hai, sir aj bahut busy day hai, network issue hai, aawaz break ho rahi hai, ladka de aayega, 30 rupaye auto ka", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Haan, kaun? acha, bolo aaj subah se light nahi hai haan, maal pada hai aapko number kisne diya? acha thik hai acha haan haan, sun raha hun market mein maal ki kami chal rahi hai 90 ka padega shaam tak pahuncha denge", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ek minute ha... haan bola, samplay, udya dupari yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RNG Eko tyre", "said": "हो, टायर शिल्लक आहे। सव्वीसशेला देतो। डिलिव्हरी? नाही हो, ते नाही करत।", "answer": {"in_stock": true, "price_inr": 2600, "delivery": false, "alternative": null}}
{"product": "Spaiko LED headlight", "said": "कंपनीने परत रेट वाढवले। मालक नाहीत, नंतर फोन करा। काय सांगू, सकाळपासून गर्दी आहे। बाजारात माल कमी आहे सध्या। दिवाळीची गर्दी आहे ना। अजून काही?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "एक मिनिट हां... बोला. आहे ना. सव्वा चार हजार आणि 18 टक्के GST वेगळा. डिलिव्हरी करत नाही.", "answer": {"in_stock": true, "price_inr": 4250, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हो हो समजलं पण ते नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बरं थांबा बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन? नहीं मिलेगा भाई। और कुछ?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "haan haan, bolo, matlab hai na, kitne chahiye? rate 170 hai", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Aegon angle grinder", "said": "dekho khatam ho gaya tha... nahi, ek bacha hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "नाही ना, संपलंय। आज नाही, उद्या येईल। बघा डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट संपलीये। Jupiter clutch plate घ्या, ते चालेल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "1 inch PVC elbow", "said": "अभी नहीं है। CPVC elbow है, 10 का।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "CPVC elbow"}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "बरं आमच्याकडे नसतं, ऑर्डर करून मागवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "yes, bola, kay aahe na ahe na, theek ahe?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "15 नग शिल्लक आहेत, घरी पाठवतो.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Ek minute ha... haan bola, wire sampli nahi, ahe, gharapoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "नाहीये, मागवून देतो, दोन दिवस लागतील", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हॅलो? आवाज येतोय का? हां, बोला, स्टॉकमध्ये आहे, तुम्हाला चारशे पन्नासला देतो, MRP सातशे वीस आहे, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "Hello, haan boliye. haan sir, wo model available hai. chhah so pachanve rupaye lagenge", "answer": {"in_stock": true, "price_inr": 695, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "sava teen sau last rate hai, ji haan, rakha hai", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब आहे, मिळेल, तुम्हाला ४७५ ला देतो, MRP ५४० आहे, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 475, "delivery": true, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "available hai barah ka full box 2680 single piece do sau pachaas matlab sorry delivery service nahi hai", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Bagha ho, kulup milel, teenshe navvad rupayanna padel, delivery nahi, dukanat yeun ghya.", "answer": {"in_stock": true, "price_inr": 390, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "बरं तुम्हाला 1400 ला देतो। हो, मशीन शिल्लक आहे। अजून काही?", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "नमस्ते, बोलिए, वो तो कल ही ख़त्म हो गया, Favourite Bikerz horn दे सकता हूँ, 865 रुपये का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz horn"}}
{"product": "Bosch drill machine", "said": "sampla nahi ajun, aahe, 9970 la deto", "answer": {"in_stock": true, "price_inr": 9970, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "जी बोलिए, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, कल सुबह आ जाएगा, अभी नहीं, आज सुबह से लाइट नहीं है, डिलीवरी फ़्री है, आवाज़ कट रही है, ज़ोर से बोलिए, एक ग्राहक खड़ा है, एक मिनट, मतलब ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Easy4buy horn", "said": "Basically, yes, we have it. 575 for you, printed price is 865. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 575, "delivery": false, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "Hello, yes, I can hear you. Available, sir. 530 rupees. OK?", "answer": {"in_stock": true, "price_inr": 530, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "Bola saheb, mahit nahi, godown madhe baghava lagel, Diwali chi gardi aahe na, aawaj tutatoy, motyane bola, ek giraik ahe, ek minute, company ne parat rate vadhavle.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "Haan bola, kay pahije? ho, oil ahe, porga chaha aanayla gelay, kay sangu, sakal pasun gardi aahe, 1499 hotil, ho, sola piece aahet, aawaj tutatoy, motyane bola", "answer": {"in_stock": true, "price_inr": 1499, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "बोला ना, कंपनीने परत रेट वाढवले, हो, ऑइल आहे, हां हां, ऐकतोय, सव्वापाचशेला देतो, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ho battery ahe 1450 rupayanna padel", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Nahi bhai, yahan nahi milega, local clutch plate chalega? 770 ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local clutch plate"}}
{"product": "4 inch door hinge", "said": "नमस्कार, बोला, बिजागरी नाहीये, संपली, बरं आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat", "said": "haan bola kay pahije ho dada ahe mhanje aata sadesolashe aahe aadhi 1280 hota", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "Stock madhe ahe, bara purna packet 1750, sutte pan ahet 1040 la ek.", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, mangva dunga, do din lagenge", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SKC steering cover", "said": "नमस्ते, बोलिए. वो तो कल ही ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we keep that. The mechanical seal costs 120, the one you asked for is 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "Not available right now. See, we have JK tyre, that's 1150", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "JK tyre"}}
{"product": "12V 7Ah battery", "said": "battery aahe, milel saheb", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, hai ji, mil jayega, 60 lagega", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Normally 70, but it's finished.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "हैलो, कौन? देखिए हाँ, कवर है. आपके लिए दो सौ पैंसठ. डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 265, "delivery": false, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "namaskar, bola, abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "sampla nahi ajun, aahe, pandhrashe hotil.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "नहीं है रुको रुको है एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "Yes sir, tell me. Basically, yes, it's there. 3,940 rupees.", "answer": {"in_stock": true, "price_inr": 3940, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "है भाई, अच्छा एक हज़ार पचास रुपये", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Aadhi sattar la hota, ata ahe ki nahi baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya tha... nahi, ek bacha hai, 170 rupaye lagenge", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Mil jayega, hai, dekho MRP toh chaar sau hai, apna rate dhai sau", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "Yes sir, tell me. Yes sir, we have. Market is short of stock these days. Delivery is 30 extra. The item is 775, delivery separate. My boy has gone for tea. OK?", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "सध्या नाही, मागवावं लागेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बघा आहे... एक मिनिट, नाही, ते काल संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "namaskar, bola, bara aawaj tutatoy, motyane bola, owner nahit, nantar call kara, kay sangu, sakal pasun gardi aahe, company ne parat rate vadhavle, porga chaha aanayla gelay", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "हो, ट्यूब आहे। चारशे सत्तर होतील", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "आहे की, मिळेल, सोळाशे पन्नास रुपये", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "No V-GRIP, only Retina cover.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Retina cover"}}
{"product": "Cheston 200A welding machine", "said": "Ek minute... ha bolo, available hai, paune chaar hajar last rate hai", "answer": {"in_stock": true, "price_inr": 3750, "delivery": null, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "helo, haan bolo nahin bhai, yaha nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices LED bulb", "said": "Diwali chi gardi ahe na, bulb sampla nahi, ahe, company ne parat rate vadhavle, savvacharshe padel, 440 cha rate juna aahe, to nahi, ek giraik ahe, ek minute.", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "motor sampli nahi ahe arre rate badalla ata 1140 aadhi 1330 hota thik hai", "answer": {"in_stock": true, "price_inr": 1140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "dekho ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "aaj nahi hai kal aayega 530 ka milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Hello? It comes to 160. Basically, yes, we keep that", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "Ji, available hai, 395 lagega", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "है भाई आपके लिए सवा दो सौ दुकान दस बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "Matlab ho, aahe aamchyakade, aapla rate tinashe pach, MRP tinashe pannas ahe", "answer": {"in_stock": true, "price_inr": 305, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "cover hai, mil jaega, 150 lagega, Retina cover chahiye to wo bhi hai, 145 wala, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "han ji, mil jayega, pone chaar sau men de dunga.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "haan toh rate 525 ke aas paas hoga, stock malik se puchna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "हाँ जी, मिल जाएगा, 470 लगा देंगे, वैसे MRP 620 लिखा है.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "Hello? Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Han, kaun? acha, bolo. rate teen sao sath hai, par abhi stock nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Hello? kuch sunai nahi de raha, baad mein karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "tape aahe, milel, padhra rupayanna padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 15, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Bijagri aahe, milel, kay aahe na sutta 30 la, 12 nagancha box ghetla tar 295, gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "हो दादा, आहे, चौदाशे रुपये होतील, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Accessoreez helmet lock", "said": "Ho sir, available aahe, daha ghetle tar 250 pramane, ek piece 270", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "ek customer khada hai, ek mint, awaaz kat rahi hai, zor se bolo, ha haan, sun raha hun, pankha khatam nahi hua, hai, gyarah sau pachas lagega", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "एक मिनिट हां... बोला। ती संपली हो।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "tube sampli nahi ahe purna packet 1550 sutte pan ahet 1060 la ek", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "han sir, boliye, awaaz kat rahi hai, zor se bolo, haan ji, tape available hai, acha aapko number kisne diya? acha theek hai, rate 25 hai, arre kya bataun, subah se bahut rush hai, bas 22 piece bache hain", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Plug samplay, Pulsar 180 wala aahe, 210 la.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180"}}
{"product": "JY Super LED torch", "said": "available hai, teen sao pachattar rupaye ka hai", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1250 la padel, wire padliye.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "haan bola, kay pahije? baghto aani kalavto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हां बोला साहेब हां तर ऑइल आहे की नाही माहीत नाही गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "han ji, boliye, abhi busy hun, aadhe ghante bad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Actually, yes, available. 1250, final price. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1250, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "aahe ki, milel teenshe panchvis padel, donashe navvad cha rate juna aahe, to nahi address pathva, pathavto thik hai?", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "मिल जाएगा है अपने पास ७७०० लगेगा और कुछ", "answer": {"in_stock": true, "price_inr": 7700, "delivery": null, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "नाही नाही, आहे, 750 रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Crompton Hill Briz ceiling fan", "said": "ji, available hai, 2250 hai, 2240 nahi, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 2250, "delivery": false, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "aamhi te thevat nahi. Speedwav headlight ahe, 1450 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav headlight"}}
{"product": "Windek tyre", "said": "tyre pada hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "company ne parat rate vadhavle, Diwali chi gardi aahe na, ho, thevto aamhi, aahe, haan haan, aiktoy, aaj sakali light navhti, tumhala 723 la deto, theek ahe?", "answer": {"in_stock": true, "price_inr": 723, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "aamhi te thevat nahi, tyachya aivaji 32A MCB deu shakto.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "32A MCB"}}
{"product": "V-GRIP steering cover for Ritz", "said": "म्हणजे स्टॉकमध्ये आहे, आमचा रेट पाचशे पंचेचाळीस, MRP 820", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "बरं हो, हेल्मेट शिल्लक आहे, पावणेनऊशे आहे, आठशे पंचावन्न नाही", "answer": {"in_stock": true, "price_inr": 875, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Bola dada, aahe, 10ch urlet, rate sheth sangtil, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch Eco wiper blade", "said": "mhanje ho, savvees piece aahet, tumhala 365 la deu", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Network issue hai, aawaz break ho rahi hai, please thodi der baad call karo, sir aaj bahut busy day hai, season time hai na, demand jyada hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ESN 2000W immersion rod", "said": "Han, maal pada hai, sadhe teen sau hoga, ladka de aayega, chalis rupaye auto ka.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Company ne is mahine phir rate badhaya, kal Sunday tha toh dukaan band thi, available hai, abhi rate do sau ho gaya, pichle mahine ek so sattar tha.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. See, it's 180 rupees.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan tar hai, 2 piece rakhe hai, delivery nahin hai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "अच्छा बल्ब नहीं है, खत्म हो गया, LED headlight bulb रखा है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "Tata Agrico plier", "said": "पक्कड संपली नाही, आहे, दोनशे पंचावन्न रुपये होतील.", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "nahiye, aaj nahi, udya yeil, 1250 la deto, bagha delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Sorry sir, abhi out of stock hai, uski jagah 2.25 mfd capacitor le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.25 mfd capacitor"}}
{"product": "9W LED bulb", "said": "haan sir, available hai 90 + GST extra", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Haan bola. ji haa, rakha hai. 320 hoga", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर आहे, मिळेल। १३५० होतील। 2.5 sq mm wire पण आहे, ते १७३० ला", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "हो, मशीन शिल्लक आहे, बघा साडेनऊ हजार आणि 18 टक्के GST वेगळा", "answer": {"in_stock": true, "price_inr": 9500, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "aahe... ek minute, nahi, te kal sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan rakhte hain, par abhi khatam hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Milel na, aahe, aadhi 76 hota, aata 100 zala", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Malak nahit, nantar phone kara bhau", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ, छह पीस पड़े हैं, एक सौ चालीस में दे देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, bola, ho, plug milel, rate donashe daha ahe sir", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म नहीं हुआ, अभी भी है। घर पर भेज देंगे।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "V-Guard Clado exhaust fan", "said": "pankha sampala, sadhya nahiye, order dilay, don divsat yeil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Shakti संपलं, iBELL machine आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "iBELL machine"}}
{"product": "Modx steering cover for i10", "said": "कव्हर पडलंय, चारशे पन्नास रुपये होतील, पोरगा पाठवतो", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "acha haan, mil jayega bhai, 320 ka ek, 6 loge toh 285 ke hisaab se laga dunga", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "Aaj sakali light navhti, pankha sampla nahi, aahe, aata akarashe aahe, aadhi 850 hota, ek giraik aahe, ek minute, theek aahe?", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "Ji boliye, haan, ready stock hai, price 2500 hai, free delivery hai", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "mhanje wo toh kal hi khatam ho gaya ek customer khada hai ek minute", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "ho bola, machine sampli nahi, aahe, 10250 rupaye hotil, gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 10250, "delivery": false, "alternative": null}}
{"product": "Goodyear tyre", "said": "३४५० रुपये होतील। टायर संपला नाही, आहे। घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 3450, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "तुम्हाला साडे पाच हजारला देतो। हां तर मोटर पडलीये", "answer": {"in_stock": true, "price_inr": 5500, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "हो मशीन आहे तुम्हाला पावणेपाच हजारला देतो डिलिव्हरी नाही दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 4750, "delivery": false, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Abhi nahi hai, order diya hai, do din me aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "bagha bulb sampala nahi, aahe, donashe chalis la padel, delivery deto", "answer": {"in_stock": true, "price_inr": 240, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb samplay kay aahe na 7w led bulb aahe tyache 100 rupees", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7w led bulb"}}
{"product": "50 mm padlock", "said": "haan sir, bataiye, matlab owner abhi available nahi hain, baad mein call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "Available, sir. It was 205, now reduced to 175.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "dekho hai bhai, aapke liye 3590, bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": 3590, "delivery": false, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो बल्ब शिल्लक आहे एक पीस 315, 6 घेतले तर 285 प्रमाणे लावतो", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ek minute ha... haan bola, oil aahe, milel, pavanecharshe fix, aahe, ajun 17 dabe padlet, address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Sold out, sir, it's very busy since morning", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "Haan bhaiya, bataiye, ha, mil jayega bhai, awaaz kat rahi hai, zor se bolo, market me maal ki kami chal rahi hai, company ne rate badla, ab atharah sau pachas hai, pehle 2110 tha, Diwali ka season hai na, sab yahi maang rahe hain, ek customer khada hai, ek minute", "answer": {"in_stock": true, "price_inr": 1850, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "haan aahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ti sampli ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "pakkad padliye, kay aahe na tumhala 400 la deto, MRP 480 aahe, porga pathavto.", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "We have it, yes. Actually, that will be Rs 250. You'll have to collect it from the shop", "answer": {"in_stock": true, "price_inr": 250, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Haan tar ho ho, samajla, pan te nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "aahe, bavisch urlet delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बरं आत्ता सांगता येणार नाही, संध्याकाळी फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "ho dada, aahe, sadeaathshe la padel", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "म्हणजे नाही कसं, आहे ना, २७०० ला पडेल", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "बोला ना, पंधरा नग शिल्लक आहेत, पाच हजार दोनशे रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "haan, mil jayega bhai, pehle 2350 tha, ab 3050 ho gaya.", "answer": {"in_stock": true, "price_inr": 3050, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "सेठ जी बाहर गए हैं, वही बताएँगे। त्योहार का सीज़न है ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho, plate aahe, rate sheth sangtil", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 250A welding machine", "said": "हैलो, कौन? पहले ८६०० का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "hai bhai, khula ek 170 ka, poora packet do sau pachasi.", "answer": {"in_stock": true, "price_inr": 285, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "ho, milel paket 1750 la aahe, sutta ek 1080 la", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "han ji, boliye check karke batata hun network issue hai, aawaz break ho rahi hai season time hai na, demand jyada hai sir aaj bahut busy day hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "हाँ भाई, बोलो, हाँ, है, ढाई हज़ार का पड़ेगा, शाम तक पहुँचा देंगे.", "answer": {"in_stock": true, "price_inr": 2500, "delivery": true, "alternative": null}}
{"product": "Vega Crux helmet", "said": "hello? aawaj yet nahiye, nantar kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Arre oil khatam nahi hua, hai, ab chaar sau chalis hai, pehle 330 tha", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "haan, kaun? acha, bolo, dekho ha hai, ek hajar chhah sau plus 18 percent GST, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan ahe bara pathavto 20 rupaye delivery charge lagel 430 la aahe delivery charge vegla", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "Windek tyre", "said": "hallo? haan, awaaz aa rahi hai, bolo, tyre nahin hai, khatam ho gaya, acha Michelin wala de sakta hun, wo 1100 ka padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin"}}
{"product": "water pump seal kit", "said": "pehle 320 ka tha, abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "ji boliye, ha, stock mein 22 piece hai, khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "एक मिनिट हां... बोला, हो, आहे, म्हणजे तुम्हाला दीडशेला देतो", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "स्टॉकमध्ये आहे, सातशे साठ... थांबा, लिस्ट बघतो... सहाशे ऐंशी, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 680, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "नमस्कार, बोला, हो, आहे शिल्लक, पावणेतीनशे होतील.", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Pick Ur Needs LED torch", "said": "hai ji, mil jaega, aapke liye 650, shop 8 baje tak open hai", "answer": {"in_stock": true, "price_inr": 650, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "गीज़र पड़ा है, २८०० रुपये का है", "answer": {"in_stock": true, "price_inr": 2800, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes yes, I understood, but we don't have it.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji haan, rakha hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "No sir, not there.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Only 23 pieces left, anything else?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "माल खत्म है. अरे क्या बताऊँ, सुबह से बहुत भीड़ है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "Yes, go ahead. Yes, it's there. Price is 705. We can deliver.", "answer": {"in_stock": true, "price_inr": 705, "delivery": true, "alternative": null}}
{"product": "Mastech multimeter", "said": "हाँ, माल पड़ा है, 790 प्लस GST", "answer": {"in_stock": true, "price_inr": 790, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "Ha, stock men nau piece hai. rate seth ji batayenge. delivery free hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "हां तर हो, ठेवतो आम्ही, आहे। एक पीस 165, 20 घेतले तर 135 प्रमाणे लावतो।", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho aahe, pavaneteenshe plus GST", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan, stock me hai, 400 ka hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 400, "delivery": true, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "hello? aawaj yetoy ka? haan bola, aamhi te thevat nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "एक ग्राहक खड़ा है, एक मिनट, मिल जाएगा, है अपने पास, बाज़ार में माल की कमी चल रही है, 400... नहीं नहीं, 440 है, त्योहार का सीज़न है ना", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "bas bara piece bache hain, kal Sunday tha toh dukaan band thi, awaaz kat rahi hai, zor se bolo, 840 last rate hai, Diwali ka season hai na, sab yahi maang rahe hain", "answer": {"in_stock": true, "price_inr": 840, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haa, ready stock hai, nabbe ka padega, bhej deta, par aj ladka nahi aya, aake le jao", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है, सत्तर में दे देंगे, ठीक है?", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला, बॅटरी पडलीये, तुम्हाला 1200 ला देतो, बघा पोरगा पाठवतो, अजून काही?", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे की, मिळेल। ५६० रुपये होतील। कळवा मग।", "answer": {"in_stock": true, "price_inr": 560, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "Haan toh haan rakhte hain, par abhi khatam hai Usha fan hai, same quality", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha fan"}}
{"product": "Crompton 1500W immersion rod", "said": "Haan, hai apne paas, 489 rupaye", "answer": {"in_stock": true, "price_inr": 489, "delivery": null, "alternative": null}}
{"product": "MK Blenze 16A switch", "said": "haan, stock mein hai, khula ek sao pachasi ka, 6 ka poora box ek hajaar mein, mhanje theek aahe?", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "Haan, wahi wala hai awaaz kat rahi hai, zor se bolo chhabbis sau ka hai, isse kam nahin hoga ek customer khada hai, ek minute are kya bataun, subah se bahut rush hai bata dena", "answer": {"in_stock": true, "price_inr": 2600, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Han, ready stock hai, do sau paintalis hai, do sau sath nahi, sorry, delivery service nahin hai.", "answer": {"in_stock": true, "price_inr": 245, "delivery": false, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "hai abhi chhah piece bache hain 690 hoga haan toh sir aaj bahut busy day hai network issue hai aawaz break ho rahi hai aur kuch chahiye", "answer": {"in_stock": true, "price_inr": 690, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "oil rakha hai, mil jayega. sadhe panch sau hai, GST alag", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "CSI 1 L geyser", "said": "देखिए हाँ जी, गीज़र रखा है। बताइए क्या करना है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Oil rakha hai, mil jayega, rate 480 hai sir, delivery ho jayegi sir", "answer": {"in_stock": true, "price_inr": 480, "delivery": true, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "Stock khatm ho gaya hai, acha Bajaj heater hai, uska rate 8800", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj heater"}}
{"product": "Leebo LED headlight for Pulsar 150", "said": "आहे आहे, 1380 होतील, घरी पाठवतो, 60 वेगळे", "answer": {"in_stock": true, "price_inr": 1380, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "हॅलो? आवाज येतोय का? हां, बोला, स्टॉक संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "मशीन संपली.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, we have 12 pieces. 120 rupees", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "आहे आहे। रेट बदललाय, आता 1100, आधी 910 होता।", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "geyser aahe, milel, tehatisshe pannas padel aata, magchya mahinyat 2840 la hota", "answer": {"in_stock": true, "price_inr": 3350, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "maal sampala Bosch wala ahe, tyache 2890 rupaye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Retina bike cover", "said": "Ho, thevto aamhi, pan atta samplay. mhanje tyachya aivaji Tradon wala deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Tradon"}}
{"product": "12V 7Ah battery", "said": "बैटरी ख़त्म नहीं हुई, है, एक हज़ार एक सौ रुपये का है.", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "stock madhe ahe packet 435 la ahe, sutta ek 255 la pickup karava lagel", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampala nahi ajun, ahe, 40 hotil.", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "हाँ जी, मिल जाएगा। ५०० रुपये का है", "answer": {"in_stock": true, "price_inr": 500, "delivery": null, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "aahe aahe, kay aahe na 1150 la deto", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "tumhala navvad la deto, bijagri padliye", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "हो, मिळेल। घरपोच देतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "ho bola, ho, milel, 1380 rupaye hotil, bagha gharapoch milel", "answer": {"in_stock": true, "price_inr": 1380, "delivery": true, "alternative": null}}
{"product": "Ritco tap", "said": "hai ji, mil jayega teen sao sath MRP hai, do sau pachis mein ho jayega", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "han ji, boliye, ek customer khada hai, ek min, machine khatam ho gayi, Stanley drill chalega to bolo, pichle hafte bhi kisi ne yahi poocha tha, ha haan, sun raha hun, company ne is mahine phir rate badhaya, dukan pe aake lena padega, bata dena.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Stanley drill"}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "mil jayega, hai panch hazar chhah sau last rate hai.", "answer": {"in_stock": true, "price_inr": 5600, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "ho dada, aahe, teen hajar tinashe pannas... thamba, list baghto... sadatisshe pannas", "answer": {"in_stock": true, "price_inr": 3750, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ho, oil milel.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, है। भेजने वाला कोई नहीं है, आ के ले जाओ। ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "20 kg नाहीये, वॉल पुट्टी 40 किलो आहे. कंपनीने परत रेट वाढवले. हां हां, ऐकतोय. आज सकाळपासून लाईट नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वॉल पुट्टी 40 किलो"}}
{"product": "Metro Moulding helmet lock", "said": "Haan haan, hai, 165 ka milega, MYRA lock bhi hai, wo 230 ka, dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 165, "delivery": false, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "helmet nahiye, sampla kaka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "१९०० रुपये लगेंगे वायर ख़त्म नहीं हुई है घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 1900, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Kya hai na jara velane phone kara, thik hai?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Bola saheb. ahe na. rate 160 aahe.", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, bola bagha ti sampli ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "ho ahe, bara donshe panchyaainshi MRP ahe, pan ekshe sattar la deu, gharee pathavto", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "SARK tap", "said": "Yes... one minute, no, that got finished yesterday.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ, माल पड़ा है, देखिए तीन सौ चालीस में दे देंगे.", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं 40 mm वाला आहे 500 ला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm"}}
{"product": "teflon tape", "said": "Arre hai ji, mil jaega, terah bola kya? galti se bola, pandrah hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "तीनशे सत्तर रुपये, कमी नाही होणार, हो, आहे शिल्लक.", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "bagha ho, battery milel. 950 rupayanna padel", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Waxpol engine oil", "said": "kya hai na hai, bilkul hai, MRP toh 360 hai, apna rate 295.", "answer": {"in_stock": true, "price_inr": 295, "delivery": null, "alternative": null}}
{"product": "AmiciTools water pump", "said": "हां बोला साहेब, मोटर पडलीये, अकराशे पन्नासला देतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो दादा, आहे, म्हणजे एक पीस 70, दहा घेतले तर 61 प्रमाणे लावतो, घरी पाठवतो.", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "haan, ready stock hai, loose piece 460 ka, full packet 725", "answer": {"in_stock": true, "price_inr": 725, "delivery": null, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "abhi stock mein nahi hai ibell wala chalega toh bolo dekho delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "ibell"}}
{"product": "Raaisin car cover", "said": "ha, ready stock hai, printed MRP 4990, our price 4200", "answer": {"in_stock": true, "price_inr": 4200, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ हाँ, है. अस्सी में दे देंगे", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "hello, bola na. aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes sir, we have, earlier it was 35, now it's 40", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "nahi hai 40 kg wala rakha hai delivery nahi karte aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "40 kg"}}
{"product": "8 mm drill bit", "said": "हो, आहे शिल्लक, तुम्हाला एकशे पन्नासला देतो, डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi dukaan band kar raha hun kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "लाईट संपली Ridroid headlight आहे त्याचे सतराशे रुपये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ridroid headlight"}}
{"product": "Rainfun car cover", "said": "Mil jayega, hai, 645 lagega, hai, abhi 25 piece bache hain", "answer": {"in_stock": true, "price_inr": 645, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "machine rakhi hai, mil jayegi, sadhe barah sau plus GST, delivery nahin karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Nahiye, matlab 3 socket board aahe, tyache 390 rupees", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 socket board"}}
{"product": "Shakti 220A welding machine", "said": "ho machine milel bara sade saha hajar ani 18 takke gst vegla porga pathavto", "answer": {"in_stock": true, "price_inr": 6500, "delivery": true, "alternative": null}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "hai bhai ji.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "हो, ठेवतो आम्ही, आहे, 110 होतील, पोरगा पाठवतो.", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bulb aahe milel tumhala 140 la deto", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "ho bola, light sampli nahi, aahe, bagha aaj sakaadi light navhti, kay sangu, sakal pasun gardi ahe, 1960 la deto", "answer": {"in_stock": true, "price_inr": 1960, "delivery": null, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "१७५० रुपये होतील। मशीन संपली नाही, आहे। पत्ता पाठवा, पाठवून देतो", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "ahe... ek minute, nahi, te kal sampla Fevicol 500 g ahe, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "Philips 5W LED bulb", "said": "स्टॉक में है, एक सौ बीस में दे देंगे, RSCT bulb भी है, वो एक सौ पचपन का.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "matlab pichle hafte bhi kisi ne yahi poocha tha, bhaiya aap ruko do minute, phone pe hun, abhi nahi hai, Diwali ka season hai na, sab yahi maang rahe hain, ladka chai lene gaya hai, bolo kya karna hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "पता नहीं, गोदाम में देखना पड़ेगा. बाज़ार में माल की कमी चल रही है.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "abhi available nahi hai, order place kiya hai, Monday tak aayegaa, 5000 ka milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "हाँ बोलिए हाँ माल पड़ा है 2000 है GST अलग से आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 2000, "delivery": false, "alternative": null}}
{"product": "Bosch horn", "said": "hai, abhi 3 piece bache hain, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Not out of stock, we have it. Price is 2310.", "answer": {"in_stock": true, "price_inr": 2310, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Haan toh khatam ho gaya tha... nahin, ek bacha hai, 25 ka hai, isse kam nahin hoga.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "Abhi stock me nahi hai, naya mal Monday ko aayega, ek hazar hoga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan bola, aadhi pastis la hota, aata aahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ, रखा है. 110 का पड़ेगा. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "पहले 1590 का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "स्टॉक संपलाय.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "बोला, काय पाहिजे? 2999 रुपयांना पडेल। बरं आहे ना। डिलिव्हरी नाही, दुकानात येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 2999, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Why not, it's available. Price is 290. Pickup only.", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "Nahi asa nahi, aahe rate 30 aahe ghari pathavto.", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "It's in stock. Price is 100.", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this? I'm busy right now, call in half an hour, there was no power since morning", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan sir, boliye, kal subah tak aa jayega, abhi nahi hai, savaa sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "Yes sir, tell me. Yes, we have 17 pieces. One customer is here, one minute. Market is short of stock these days. Rs 4670 only. It's festival season, everyone is asking. See, tell me what to do", "answer": {"in_stock": true, "price_inr": 4670, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "Maal khatam hai. Usha fan chalega toh bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Usha fan"}}
{"product": "16A MCB", "said": "stock madhe aahe dada", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मतलब हाँ हाँ है सात सौ चालीस है GST अलग से", "answer": {"in_stock": true, "price_inr": 740, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Kay aahe na nahi milega bhai, Fevicol 500 g hai, uska rate 390", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol 500 g"}}
{"product": "9W LED bulb", "said": "bola saheb, bulb samplay, bagha aaj sakali light navhti.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी नाहीये, संपली. अजून काही?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan rakha hai pehle 1680 tha ab 1400 ho gaya sasta hua hai", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हो, कव्हर आहे, 4620... नाही नाही, 4200.", "answer": {"in_stock": true, "price_inr": 4200, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "sorry sir, abhi out of stock hai, bhaiya aap ruko do minute, phone pe hun.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "नमस्कार, बोला, AmiciTools संपलं, Techtest iron आहे, चालेल? कंपनीने परत रेट वाढवले, बघा बाजारात माल कमी आहे सध्या", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Techtest iron"}}
{"product": "12V 7Ah battery", "said": "स्टॉक में है, पहले 1170 था, अब चौदह सौ पचास हो गया", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "ho ahe chhapil kimmat 145, aamhi 115 la deto.", "answer": {"in_stock": true, "price_inr": 115, "delivery": null, "alternative": null}}
{"product": "Havells Cista room heater", "said": "It's 2260 rupees, but right now no stock.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ek minute... haan bolo rate pachaas ke aas paas hoga, stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "Hello, yes, I can hear you. We have 8 pieces in stock right now. Rate I'll confirm and tell you", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हो दादा, आहे, आधी सहाशे पंचाण्णव होता, आता पाचशे ऐंशीला देतोय", "answer": {"in_stock": true, "price_inr": 580, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bagha 4 inch sampla, steel hinge aahe, chaalel?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "steel hinge"}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ हाँ, है। एक सौ पन्द्रह... नहीं नहीं, एक सौ है। होम डिलीवरी हो जाएगी।", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "हैलो, कौन? रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "एक मिनिट हां बोला स्टॉकमध्ये नाहीये त्याच्याऐवजी Dio air filter देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dio air filter"}}
{"product": "wall putty 20 kg", "said": "पुट्टी पडलीये, पावणे सहाशे, GST वेगळा", "answer": {"in_stock": true, "price_inr": 575, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "माल संपला.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हो मिळेल म्हणजे ४५५ रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "थोड़ी देर बाद कॉल करो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "That we don't have. I can give Dow sealant instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dow sealant"}}
{"product": "CarMobCentral car cover for Ritz", "said": "दिवाळीची गर्दी आहे ना, हो, कव्हर आहे, सहाशे पंचेचाळीस रुपये, कमी नाही होणार, पोरगा चहा आणायला गेलाय, हां तर बाजारात माल कमी आहे सध्या, आवाज तुटतोय, मोठ्याने बोला", "answer": {"in_stock": true, "price_inr": 645, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "Yes, we have 21 pieces. 119/-. We can deliver.", "answer": {"in_stock": true, "price_inr": 119, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "That we don't have. I can give local putty instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local putty"}}
{"product": "Bajaj 3 L geyser", "said": "geyser rakha hai, mil jayega, 2820 rupees, home delivery available hai, 100 charge", "answer": {"in_stock": true, "price_inr": 2820, "delivery": true, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "ek minute ha... haan bola, aahe, ajun 9 piece padlet, gharpoch milel", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "नमस्कार, बोला, पावणेतीनशेला देतो, स्टॉकमध्ये आहे, संध्याकाळपर्यंत पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 275, "delivery": true, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "मशीन पडलीये, म्हणजे 5300 रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 5300, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Hai ji, mil jayega, 165 rupaye lagenge, stock me sirf 6 piece hai.", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant (pack of 2)", "said": "Haa, ready stock hai khula ek ek so pachpan ka, poora packet do sau pachpan shaam tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 255, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "kya hai na kya hai na bijagri aahe ki nahi mahit nahi, godown madhe baghto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ माल पड़ा है दस लगेगा अच्छा दो पीस हैं स्टॉक में", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "nahi hai, Vega helmet hai, uska rate 675, ajun kahi?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega helmet"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "bulb pada hai aapko sava chaar sau mein de denge, MRP chhah sau chalis hai bhijwa dunga theek aahe?", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "cover aahe, milel, chhapil kimmat 605, aamhi 385 la deto", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "Carzex LED headlight", "said": "Hai bhai, 2020 mein de dunga, arre 9 baje tak main yahin hun, kalva mag", "answer": {"in_stock": true, "price_inr": 2020, "delivery": null, "alternative": null}}
{"product": "Camel drill machine", "said": "hai, abhi athara piece bache hain, 1450 ka milega", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "haan haan, hai, 480 hai, 380 nahi", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "आहे ना. १५५० रुपये, कमी नाही होणार. पोरगा पाठवतो, ५० रुपये रिक्षाचे.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "han, tala hai, 230 lagega, 215 wala rate purana hai, wo nahi, arre free delivery hai.", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Sorry, finished. Ball valve is available, 170 rupees. Actually, home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "ball valve"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर पड़ी है. १९०० है, GST अलग से", "answer": {"in_stock": true, "price_inr": 1900, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "नमस्कार बोला 385 ला देतो लाईट पडलीये हां तर पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 385, "delivery": true, "alternative": null}}
{"product": "Golden Bullet HI93 600W drill machine", "said": "तुम्हाला पंधराशे पन्नासला देतो, हो, मशीन शिल्लक आहे", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "putty sampli Diwali chi gardi ahe na ek giraik aahe, ek minute 40 kg wala ahe, tyache 940 rupaye thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 kg"}}
{"product": "water pump seal kit", "said": "जी, उपलब्ध है, मतलब local seal kit का रेट २६५ है, ये वाला ३१०", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "haan bhai, bolo, mil jayega, hai, 1400 mein ho jayega, upar MRP 2000 likha hai, delivery free hai", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "स्टॉक में है. तीन सौ दस और 18 परसेंट GST अलग. मतलब आपको आकर लेना पड़ेगा.", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हो, पुट्टी आहे. 720 रुपयांना पडेल. घरी पाठवतो, 100 वेगळे.", "answer": {"in_stock": true, "price_inr": 720, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Kami nahi, bharpur aahe, address pathva, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "rate 810 ahe. kay aahe na ho, thevto aamhi, aahe. address pathava, pathavto. theek aahe?", "answer": {"in_stock": true, "price_inr": 810, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "260 ka padega haan, ready stock hai sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "Ho aahe, eka nagache 210, 12 nagancha box 2260 la.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "Nahiye, magvun deto, don diwas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Ek minute ha... haan bola, ti sampli ho, porga chaha aanayla gelay, company ne parat rate vadhavle, kal Ravivar hota mhanun dukan band hota, thread seal tape aahe, 15 la, theek aahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "thread seal tape"}}
{"product": "Legrand Mylinc 6A switch", "said": "है भाई, एक पीस एक सौ पचास, 12 लोगे तो 135 के हिसाब से.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "milel na ahe eka nagache pannas, 50 nagancha box 2190 la", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "haan toh hai, bilkul hai, dus loge toh 210 ke hisaab se, ek ka do sau pachas", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "ha, maal pada hai rate do hajar pachaas hai bara ghar pe bhej dunga, pachaas lagega bhejne ka", "answer": {"in_stock": true, "price_inr": 2050, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Haan, stock mein hai, home delivery nahi hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हो, मशीन आहे, किंमत नंतर सांगतो, आत्ता लिस्ट नाही, ठीक आहे?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ho aahe, 280 aahe, 235 nahi, aur kuch?", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "umm motor nahi hai, khatam ho gayi, AmiciTools wala hai, uska rate 1200", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "Bosch horn", "said": "ek minute ha haan bola bagha nahiye magvun deto don divas lagtil", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Windek tyre", "said": "कल सुबह आ जाएगा, अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "Woh hum nahin rakhte, Rico wala chalega? 350 ka hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rico"}}
{"product": "teflon tape", "said": "Bagha ho dada, ahe, 12 ghetle tar 13 pramane, ek peice 15", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "We have it, yes, Themisto multimeter is also there at 820, and this one is 795, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 795, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan haan, samajh gaya, lekin wo abhi nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "It's in stock. Rs 10 only. Let me know.", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "शिल्लक नाही, Bajaj वाला आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj"}}
{"product": "Modx steering cover for Xcent", "said": "आहे आहे, साडे चारशेला देऊ, छापील किंमत 520 आहे, डिलिव्हरी फ्री आहे.", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Khadija angle grinder", "said": "wo item nahi rakhte hum sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shell Advance AX7 10W-40 engine oil", "said": "ji sir, tell me, network issue hai, aawaz break ho rahi hai, haan toh sir aaj bahut busy day hai, haa, ready stock hai, MRP to 545 hai, apna rate 440, season time hai na, demand jyada hai.", "answer": {"in_stock": true, "price_inr": 440, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat", "said": "Rs 3,170 only. Basically, yes, we have it. Delivery is free.", "answer": {"in_stock": true, "price_inr": 3170, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ahe... ek minute, nahi, te kal sampala, Pulsar 180 wala ahe, chaalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180"}}
{"product": "JY Super LED torch", "said": "No, we don't have it. See, we have Aryshaa torch, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Aryshaa torch"}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "namaskar, bola, cover ahe, milel, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli, sanga kay karaycha", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "जी बोलिए। हाँ तो हैलो? आवाज़ नहीं आ रही, बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "haan bolo, dekho helmet nahin hai, khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "Yes, we keep that. Delivery is not a problem, we'll send it", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Ho ho, samajla, pan te nahiye Jupiter clutch plate chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "Crispy LED torch", "said": "stock madhe aahe 435 nahi nahi 395", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "Rate ektisshe pannas ahe, ahe na, delivery deto.", "answer": {"in_stock": true, "price_inr": 3150, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "मशीन संपलीये, आत्ता नाहीये, संध्याकाळी येईल, घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Haan haan, aiktoy, Diwali chi gardi ahe na, mahit nahi, godown madhe baghava lagel, aaj sakali light navhti, company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ventil 230 mm exhaust fan", "said": "haan, stock mein hai, terah sau pachas mein de dunga, delivery ka assi extra", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes yes, available, see, it comes to 110.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "नमस्ते, बोलिए, पहले 1590 का था, अभी है कि नहीं देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, maal pada hai, haan toh 12 ka dabba 1190 ka, ek piece ek so bees, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "Induro plier", "said": "plas pada hai Agrico plier wala 460 mein aata hai, aapka wala 500 delivery free hai", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "Hello, who is this? Not in stock now, it will come tomorrow evening. Market is short of stock these days. It's festival season, everyone is asking. Okay so, there was no power since morning. OK?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "आमच्याकडे आहे, एकशे नव्वद रुपये, कमी नाही होणार, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "देखिए मिल जाएगा, है अपने पास, सवा नौ सौ प्लस GST.", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "tube sampli nahi, ahe, MRP 220 ahe, tumhala 185 la deto", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan ha, hai 10 ka hai", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "ji sir, tell me, sorry sir, abhi out of stock hai, ladka chai lene gaya hai, kal Sunday tha toh dukaan band thi, aaj subah se light nahi hai, bolo kya karna hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Han ji, boliye, abhi nahi hai, order diya hai, do din me aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "ek minute ha... haan bola, ho, pakkad milel, kay aahe na aadhi 515 hota, aata 420 la detoy", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Aahe... ek minute, nahi, te kal sampla.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "Hello, yes, I can hear you. Sold out, sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "ji boliye, arre bulb nahi hai, khatam ho gaya, Daybetter bulb chalega? do sau tees ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Daybetter bulb"}}
{"product": "50 mm padlock", "said": "हाँ तो ताला ख़त्म हो गया", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "2.5 mfd wala khatam hai, 2.25 mfd wala hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.25 mfd"}}
{"product": "10W-30 engine oil 1 L", "said": "hai, bilkul hai, bagha 410 ka padega, mhanje shaam tak pahuncha denge, bata dena", "answer": {"in_stock": true, "price_inr": 410, "delivery": true, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "milel na, aahe, rate sadesahashe aahe, mhanje ghari pathavto, pannas vegle", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "70 rupaye hotil ho, ahe aamchyakade delivery nahi karat bata dena.", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "aahe ki, milel, savvashe la padel.", "answer": {"in_stock": true, "price_inr": 125, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "mahit nahi godown madhe baghava lagel haan haan aiktoy", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "hello, haan boliye, umm available hai, koi problem nahi.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "stock madhe aahe kaka", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Racold Pronto Pro 3 L geyser", "said": "Haan bola, kal Ravivar hota mhanun dukan band hota, aaj sakali light navhti, geyser aahe, milel, 2580 hotil, ek giraik aahe, ek minute, sandhyakali pathavun deto, Diwali chi gardi aahe na", "answer": {"in_stock": true, "price_inr": 2580, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "म्हणजे बॅटरीचा स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ, रखा है, एक पीस 90, 10 लोगे तो 83 के हिसाब से", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Haan ji, boliye khatam ho gaya CPVC wala hai, uska rate 15", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "CPVC"}}
{"product": "Autofurnish car cover for Santro Xing", "said": "हो, आहे शिल्लक, चारशे पस्तीस रुपये, पाठवायला माणूस नाही, इथे येऊन घ्या.", "answer": {"in_stock": true, "price_inr": 435, "delivery": false, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "Haan, ready stock hai, MRP 400 hai, discount ke baad 339 padega, same day delivery kar denge", "answer": {"in_stock": true, "price_inr": 339, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "abhi nahi hai, order diya hai, do din mein aa jaega, aapke liye 1500", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "We don't have it, but I can order it, takes two days", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "हाँ हाँ, है, आपके लिए 2949, ठीक है?", "answer": {"in_stock": true, "price_inr": 2949, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "Ji sir, tell me, khatam ho gaya, umm network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, season time hai na, demand jyada hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Mil jayega, hai, single piece paune do sau, dus ka full box ek hajaar panch sau pachas, same day delivery kar denge.", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "pankha rakha hai, mil jayega, arre ek hajaar chhah sau hoga, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "Sulfar Lutian water pump", "said": "मोटर पडलीये रेट ३९०० आहे पाठवतो ५० रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 3900, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Pankha padlay, don hazar rupayanna padel", "answer": {"in_stock": true, "price_inr": 2000, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "hello haan bola tumhala pachshe pandhara la deu ho cover aahe", "answer": {"in_stock": true, "price_inr": 515, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Uh, we have it, yes. 90 rupees. Tell me what to do.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Owner abhi available nahi hain, baad mein call kijiye", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Yes, we have 8 pieces", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Aahe sir, ready stock aahe, 12 nagancha box don hajar tinashe la, sutta ek savvadonshe", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Ji haan, rakha hai delivery nahin hai", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Haan, wahi wala hai, dekho thik hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "संपलं नाही अजून, आहे, तुम्हाला २९५० ला देतो", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ho, plug milel, 120 rupayanna padel, local spark plug pahije asel tar te pan aahe, 145, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ji, available hai rate pandrah hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan tar ahe ki, milel, rate 50 aahe", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "जी, उपलब्ध है। हमारा रेट ५८०, MRP तो ७०० है", "answer": {"in_stock": true, "price_inr": 580, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "बल्ब संपला नाही आहे 230 रुपये घरपोच नाही तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "HP engine oil", "said": "aahe na, chhapil kimmat 395, aamhi 325 la deto", "answer": {"in_stock": true, "price_inr": 325, "delivery": null, "alternative": null}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "mil jayega, hai, rate saat sau pandrah hai.", "answer": {"in_stock": true, "price_inr": 715, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "bagha haan bola... nahi, te sadhya nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Moulding helmet lock", "said": "बाजारात माल कमी आहे सध्या। हो, आहे। दिवाळीची गर्दी आहे ना। तुम्हाला 190 ला देतो, MRP 290 आहे। कंपनीने परत रेट वाढवले।", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "देखिए बल्ब ख़त्म हो गया LED वाला दे सकता हूँ 120 रुपये का आपको आकर लेना पड़ेगा", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED"}}
{"product": "Sulfar Lutian water pump", "said": "Nahi hai, mangwa dunga, do din lagenge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "nahi kasa, ahe na, 350 rupayanna padel", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "haan bhai, bolo, abhi stock mein nahi hai, Orient heater hai, barah sau pachas ka, delivery ho jayegi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Orient heater"}}
{"product": "Ceat Milaze tyre", "said": "kal subah tak aa jayega, abhi nahin hai, khud aake le jaiye", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "जी, उपलब्ध है। रेट मालिक बताएँगे।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 3 L geyser", "said": "Yes, tell me, okay so, it's in stock, 3800, final price, anything else?", "answer": {"in_stock": true, "price_inr": 3800, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "499 lagega. han haan, hai.", "answer": {"in_stock": true, "price_inr": 499, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "ahe sir, ready stock ahe. terashe fix. sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "Yes, we keep that. 3,750, final price. We don't deliver", "answer": {"in_stock": true, "price_inr": 3750, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes, we have it, boss.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Hello, good morning. Yes, available. For you, 90. Home delivery is available", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "Clouds wiper blade", "said": "kya hai na nahi milega bhai bosch wiper hai 630 ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "bosch wiper"}}
{"product": "Activa air filter", "said": "Yes, available. Price changed, it's 290 now, was 260 last month. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "आहे ना पावणेपाचशेला देऊ छापील किंमत आठशे पंचेचाळीस आहे", "answer": {"in_stock": true, "price_inr": 475, "delivery": null, "alternative": null}}
{"product": "Bosch wiper blade", "said": "आहे की, मिळेल. छापील किंमत 390 आहे, आम्ही 360 ला देतो. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 360, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "paach sau pachas rupaye hai, haan toh available hai, koi problem nahi, aur kuch?", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hello? I can't hear you, call again later.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "पंखा संपला नाही, आहे. तीन हजार आठशे रुपये", "answer": {"in_stock": true, "price_inr": 3800, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "Haan rakhte hain, par abhi khatm hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Yes sir, tell me, uh, no 10 mm, only the 8 mm one", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 mm"}}
{"product": "Cheston 500W drill machine", "said": "नहीं है Bosch drill चलेगा उसका दो हज़ार पचास है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch drill"}}
{"product": "AutoPowerz LED headlight", "said": "nahiye bahutek ho ho sapadla ahe sahashe tees rupayanna padel", "answer": {"in_stock": true, "price_inr": 630, "delivery": null, "alternative": null}}
{"product": "Retina bike cover", "said": "cover aahe ka baghava lagel, nantar sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mufasa 4W LED bulb", "said": "Han, wahi wala hai haan toh 340 ka hai bhijava dunga.", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "Yes sir, we have, ₹705, Vega helmet is also there, that one is 770, I'll send it with our boy.", "answer": {"in_stock": true, "price_inr": 705, "delivery": true, "alternative": null}}
{"product": "Autofy Trouper helmet", "said": "nahi aahe, Vega wala chalnar asel tar sanga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vega"}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai. assi ka hai, isse kam nahi hoga. haan toh khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ते आमच्याकडे नसतं 40 किलो वाला आहे आमच्याकडे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 किलो"}}
{"product": "9W LED bulb", "said": "Ha sir, wo model available hai, last month 79 tha, ab 90 hai.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Carbanao car mat for Swift", "said": "हाँ, माल पड़ा है, आपके लिए तेरह सौ पचास, भेज देंगे, 50 रुपये डिलीवरी चार्ज लगेगा.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "nahi vatta... thamba, ahe, ek urlay. 860 hotil. delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 860, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "25 nag shillak ahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "price 610 ahe, ho, aahe aamchyakade, porga pathavto", "answer": {"in_stock": true, "price_inr": 610, "delivery": true, "alternative": null}}
{"product": "Avita switch", "said": "haan, mil jayega bhai, teen sau paintalis ka padega", "answer": {"in_stock": true, "price_inr": 345, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes sir, tell me, we have 30 pieces in stock right now, company increased rates again, it's festival season, everyone is asking, for you, 70, it's very busy since morning, we don't deliver", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "बोला काय पाहिजे रेट पावणेनऊशे आहे पण आत्ता स्टॉक नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JCB plier", "said": "Yes yes, available. Printed price is 475, I'll give it at 395", "answer": {"in_stock": true, "price_inr": 395, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "Bulb khatam nahi hua, hai, 249 ka padega, delivery nahi karte, aake le jao, dukaan 10 baje tak khuli hai, ajun kahi?", "answer": {"in_stock": true, "price_inr": 249, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, 26 piece aahet, pachshe sath la deto, delivery deto", "answer": {"in_stock": true, "price_inr": 560, "delivery": true, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "वो तो कल ही ख़त्म हो गया, क्रॉम्पटन फैन है, वो ले लीजिए, लड़का भेज देता हूँ", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "क्रॉम्पटन फैन"}}
{"product": "Siron 60W soldering iron", "said": "haan, hai apne paas bhaiya", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "stock samplay, udya sakali maal yeil, aatta nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "abhi dukaan band kar raha hun, kal subah phone karo sir.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Agrico TATA plier", "said": "aadhi donashe sattar la hota, aata aahe ki nahi baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "Available, sir. I don't know the rate, the owner will tell you.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "हॅलो? आवाज येतोय का? हां, बोला, आवाज तुटतोय, मोठ्याने बोला, काय सांगू, सकाळपासून गर्दी आहे, आहे आहे, हां तर आज सकाळपासून लाईट नाही, 6200 फिक्स, पोरगा पाठवतो, 40 रुपये रिक्षाचे, कळवा मग", "answer": {"in_stock": true, "price_inr": 6200, "delivery": true, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "mhanje ho, bulb ahe. ekshe panchavis plus GST. delivery nahi karat", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "han, wahi vala hai, best price das", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, boliye, matlab ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "ho aahe... nahi nahi, to dusra aahe, ha nahiye, AUTOSiTY grip chalnar asel tar sanga.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AUTOSiTY grip"}}
{"product": "Speedwav handle grip", "said": "Hai bhai, 455 lagega", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Roots horn", "said": "hai, abhi pachees piece bache hain, dekho aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "हां बोला साहेब हां तर आत्ता नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "haan, stock mein hai, ek hajar panch sau hai, sorry sorry, sawa hajar, bhejne wala koi nahi hai, aapko aana padega.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haa ji plug available hai rate puch ke batata hun ghar pe bhej denge bolo kya karna hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Prithvi 6.5 HP water pump", "said": "ji haan, rakha hai, gyarah hazar nau sau final price hai, dekho delivery? nahi bhai, wo nahi hota", "answer": {"in_stock": true, "price_inr": 11900, "delivery": false, "alternative": null}}
{"product": "Crasts LED torch", "said": "Bara stock madhe ahe, 825 rupaye, kami nahi honar", "answer": {"in_stock": true, "price_inr": 825, "delivery": null, "alternative": null}}
{"product": "HTC multimeter", "said": "नाही कसं, आहे ना. म्हणजे 750 रुपये होतील", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ahe 17ch urlet yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "Bazarat maal kami aahe sadhya, haan haan, aiktoy, ek giraik aahe, ek minute, ti sampli ho, kal Ravivar hota mhanun dukan band hota, kay aahe na company ne parat rate vadhavle, theek aahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "han bulb hai full packet 380 loose peice 86 ka", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "machine khatm ho gayi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VVM bike cover for Honda", "said": "Yes, available. 380 rupees. Shop closes at 7", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Hai na, kitne chahiye? 40 hai, GST alag, bhijwa dunga.", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "JCB plier", "said": "Pakkad sampli nahi, ahe, rate 420 aahe sir", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Tradon bike cover for Access", "said": "hello? aawaj yetoy ka? haan bola, ho, 12 piece aahet, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "वो हम नहीं रखते आज सुबह से लाइट नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "It's in stock. 80, GST extra", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Speedwav handle grip", "said": "han, kaun? acha, bolo, dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Cover cha stock baghun sangto. haan tar ek giraik aahe, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lakshmi 0.5 HP water pump", "said": "ahe na, don hajar pachshe... thamba, list baghto... don hajar shambhar.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "Khatam nahi hua, abhi bhi hai, dekho paune chaar sau rupaye.", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "3000 ka aata hai, lekin abhi khatam hai, abhi stock nahi hai, kal tak aa jayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch drill machine", "said": "ho, thevto aamhi, pan aatta samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "हाँ, माल पड़ा है. खुला एक 330 का, पूरा पैकेट 600.", "answer": {"in_stock": true, "price_inr": 600, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "Hello, good morning. We have it, yes. 370, final price. Let me know", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Yes sir, we have. Price is 190. See, we can deliver, 80 rupees delivery charge", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "Techtest 60W soldering iron", "said": "अभी सत्ताईस पीस ही बचे हैं। साढ़े पाँच सौ का पड़ेगा। आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 550, "delivery": false, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "hello, bola na, ho aahe, 410 hotil", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "acha mahit nahi godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "kay aahe na ahe na, rate badalla, aata sahashe pannas, aadhi 580 hota, address pathva, pathavto.", "answer": {"in_stock": true, "price_inr": 650, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "Abhi nahi hai boss.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "Yes, we have it. 1200, the 1370 rate is old, not that.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "1950 का है, इससे कम नहीं होगा, हाँ हाँ, है", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "rate 460 aahe sir, ho sir, available aahe, delivery nahi, shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 460, "delivery": false, "alternative": null}}
{"product": "AO Smith 15 L geyser", "said": "haa ji, boliye. arre nahi hai ji, khatm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "jara velane phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello, bola na, ahe, 21ch urlet, ek hazar charshe fix", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "kya hai na aj nahi hai, kal aayega, unnees sau last rate hai, bhejne vala koi nahin hai, aapko aana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "Bola saheb, ahe, 22ch urlet, company ne parat rate vadhavle, 45 hotil, porga chaha aanayla gelay, bazarat maal kami ahe sadhya, aawaj tutatoy, motyane bola, sanga kay karaycha", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "Bajaj 3 L geyser", "said": "ख़त्म हो गया, Havells geyser चलेगा? उसका 2820 है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells geyser"}}
{"product": "Autoplus helmet lock", "said": "हाँ भाई बोलो है ना कितने चाहिए साढ़े तीन सौ में दे देंगे", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan ji boliye rate 790 ke aas paas hoga stock malik se puchna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बरं हॅलो? आवाज येत नाहीये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan ji, boliye, abhi stock mein nahi hai, chahiye toh ball valve de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ball valve"}}
{"product": "50 mm padlock", "said": "420 rupaye, tala pada hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 420, "delivery": true, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "जी बोलिए. अभी नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ तो अभी नहीं है, ऑर्डर पर मंगवा देंगे, होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Aptech multimeter", "said": "hello? han, awaaz aa rahi hai, bolo. abhi stock men nahi hai, naya maal Monday ko aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "mhanje ti sampli ho, company ne parat rate vadhavle, aaj sakaadi light navhti, kay sangu, sakal pasun gardi ahe, 3 inch wala ahe, 70 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch"}}
{"product": "50 mm padlock", "said": "ताला पड़ा है, 320 का पड़ेगा, मतलब हाँ, 27 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate khatam nahin hui, hai, 420 ka padega.", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "Yes sir, tell me. Yes, we keep that. I'll give it for 230, the MRP is 365. Actually, we don't deliver.", "answer": {"in_stock": true, "price_inr": 230, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो, ठेवतो आम्ही, आहे भाऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "कल सुबह आ जाएगा अभी नहीं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Rate 60 hai, par abhi stock nahi hai, sanga kay karaycha.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "abhi stock men nahi hai. awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Haan bhaiya, bataiye, ha, wahi wala hai, company ne is mahine phir rate badhaya, aapko number kisne diya? acha theek hai, pehle 32 tha, ab 40 ho gaya", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Rolson screwdriver bit set", "said": "देखिए नहीं है, बॉश बिट सेट रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बॉश बिट सेट"}}
{"product": "2 inch paint brush", "said": "haan sir bataiye han ha hai rate abhi pata nahi malik se puchna padega delivery ho jayegi sir theek hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Maal khatam hai, Havells pump hai, wo lelo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells pump"}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "ji boliye. pankha nahi hai, khatam ho gaya. Bajaj wala chalega to bolo. dekho ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Bajaj"}}
{"product": "4 inch door hinge", "said": "bijagri sampli, sadhya nahiye, order dilay, don divsat yeil, sandhyakali pathavun deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "We have it, yes. For you, 400.", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "ho, nal milel, tumhala 225 la deto, MRP 360 aahe, gharpoch milel", "answer": {"in_stock": true, "price_inr": 225, "delivery": true, "alternative": null}}
{"product": "Wipro Garnet 20W LED batten (pack of 2)", "said": "अच्छा मिल जाएगा है अपने पास खुला एक 950 का पूरा पैकेट 1600", "answer": {"in_stock": true, "price_inr": 1600, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "light aahe, milel, ek piece tinashe pannas la milto, packet savva paachashe la, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ha, kabza hai 30 lagega ab, pehle 34 ka tha.", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "oil padlay delivery nahi ho te nahi karat", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "बोला ना, पुट्टी आहे की नाही माहीत नाही, गोडाऊनमध्ये बघतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "हो बोला दादा आमच्याकडे आहे आमचा रेट 555 MRP 635", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Themisto multimeter", "said": "ho dada, aahe. sahashe vees rupaye. address pathva, pathavto. bata dena.", "answer": {"in_stock": true, "price_inr": 620, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ho, ahe aamchyakade, ajun kahi?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "aahe ki, milel, 775 la deto, ho, aekonees piece aahet, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 775, "delivery": true, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Basically, we have 23 pieces in stock right now", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया, plastic float valve रखा है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "plastic float valve"}}
{"product": "4 inch door hinge", "said": "हो, आहे शिल्लक, एकशे वीस रुपयांना पडेल.", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "है भाई, चार सौ पचास लगेगा, भेजने वाला कोई नहीं है, आ के ले जाओ.", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "haan tar baghto ani kalavto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "Mil jayega, hai, MRP paach sau nabbe hai, but aapko saadhe chaar sau me de denge.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "Abhi confirm nahi kar sakta, godown chek karna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "tube ahe, milel, bagha sadeaathshe rupaye hotil", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Cheston angle grinder", "said": "ho ahe porga chaha aanayla gelay gharee pathavto 80 vegle haan haan aiktoy maalache 1329 delivery vegli", "answer": {"in_stock": true, "price_inr": 1329, "delivery": true, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हैलो, कौन? त्योहार का सीज़न है ना, लड़का चाय लेने गया है, आज सुबह से लाइट नहीं है, बाज़ार में माल की कमी चल रही है, पंखा ख़त्म नहीं हुआ, है, अभी रेट उन्नीस सौ पचास हो गया, पिछले महीने 1810 था, ठीक है?", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "It used to be 230, I'll have to check if it's there", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हॅलो? आवाज येतोय का? हां, बोला. 1.5 sq mm नाहीये, 2.5 sq mm wire आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "4 inch door hinge", "said": "Available, sir. 90/-.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Inditrust soldering iron", "said": "hai na kitne chahiye 594 laga denge waise mrp 900 likha hai ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 594, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Ek minute ha... haan bola, aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells LED torch", "said": "Hello, good morning. Basically, yes, it's there. 330 for you, printed price is 375.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Aahe, ajun teraa piece padlet, 580 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "ek minute... haan bolo, do hajar panch sau mein de dunga, available hai", "answer": {"in_stock": true, "price_inr": 2500, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, kaun? acha, bolo, malik abhi nahi hain, baad mein phone karo, ek customer khada hai, ek minute, kal Sunday tha toh dukaan band thi, aapko number kisne diya? acha thik hai, market mein maal ki kami chal rahi hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro 15W LED bulb (pack of 2)", "said": "yes sir, boliye, bulb pada hai, last month tin sau assi tha, ab char sau pachis hai", "answer": {"in_stock": true, "price_inr": 425, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "Ji, available hai. ek ka 199, barah ka dabba 2040 ka. kay aahe na khud aake le jaiye.", "answer": {"in_stock": true, "price_inr": 199, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो? आवाज येत नाहीये, नंतर करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हो मिळेल घरपोच देतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Abhi nahi, order pe mangwa denge.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "namaskar, bola, ho, milel, 60 fix", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "haan hai... ek minute, nahi, wo kal khatam ho gaya, Osram headlight hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Osram headlight"}}
{"product": "iBELL IBL drill machine", "said": "bola saheb ho machine aahe 4790 rupaye kami nahi honar", "answer": {"in_stock": true, "price_inr": 4790, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Crompton Greaves संपलं, AmiciTools pump आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools pump"}}
{"product": "1 inch PVC elbow", "said": "Actually, yes, we have it. It comes to 40. Pickup only.", "answer": {"in_stock": true, "price_inr": 40, "delivery": false, "alternative": null}}
{"product": "Allure Auto car mat for Zest", "said": "हां तर नाय हो, नाहीये. त्याच्याऐवजी Crokrok mat देऊ शकतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crokrok mat"}}
{"product": "fan capacitor 2.5 mfd", "said": "Thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "nahi asa nahi, aahe, 20 fix", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bola, kay pahije? te model available nahi, M-seal available ahe, chaalel ka? bara home delivery aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "M-seal"}}
{"product": "Bajaj 3 L geyser", "said": "Available, sir, tell me what to do", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Arteor 25A switch", "said": "बघा हो मिळेल सुटा सव्वा तीनशेला सहा नगांचा बॉक्स घेतला तर एक हजार सातशे संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 325, "delivery": true, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "hello, haa bolo, haan, maal pada hai, rate seth ji batayenge", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JY Super LED torch", "said": "हाँ बोलिए बाज़ार में माल की कमी चल रही है सेठ जी बाहर गए हैं वही बताएँगे लड़का चाय लेने गया है हाँ तो त्योहार का सीज़न है ना कंपनी ने फिर रेट बढ़ा दिया आवाज़ कट रही है ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "Ha, kaun? acha, bolo, abhi busy hun, aadhe ghante baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "HP engine oil", "said": "bagha haan aahe, tumhala 280 la deto, MRP 340 aahe", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Anchor LED batten", "said": "Yes, tell me. We have 23 pieces in stock right now", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 5W LED bulb", "said": "हाँ हाँ, है। 12 लोगे तो 110 के हिसाब से, एक का 119", "answer": {"in_stock": true, "price_inr": 119, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "ek minute... ha bolo, malik abhi nahi hain, bad mein phone karo, ek customer khada hai, ek minut, haa ha, sun raha hun, umm Diwali ka season hai na, sab yahi maang rahe hain.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "haan toh wo item hum keep nahi karte. MK switch hai, same quality.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "MK switch"}}
{"product": "R-tek multimeter", "said": "We have 16 pieces in stock right now, it comes to 455.", "answer": {"in_stock": true, "price_inr": 455, "delivery": null, "alternative": null}}
{"product": "Havells Crabtree Murano switch", "said": "Yes, we keep that. One piece 220, if you take 10 I'll do 185 each", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "yes available hai price untalis sau pachas hai ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 3950, "delivery": true, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "नहीं है... रुको रुको, है, एक पड़ा है, तीन सौ पचानवे फ़ाइनल, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 395, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "Not available right now. We have the Modx one, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Modx"}}
{"product": "AO Smith 15 L geyser", "said": "गिझर पडलाय. 6000 रुपयांना पडेल. दुकान 10 वाजेपर्यंत उघडं आहे.", "answer": {"in_stock": true, "price_inr": 6000, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "aahe ki, milel ata pavaneteenshe ahe, aadhi 340 hota aur kuch?", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "Savsol 20W-40 engine oil", "said": "म्हणजे ऑइल संपलं नाही, आहे। 325... थांबा, लिस्ट बघतो... 270", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "संपलंय साहेब।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "haan ha, hai 1600 rupees ka hai kya hai na ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "stock madhe aahe, sutta 165 la, vees nagancha box ghetla tar 2720, porga pathavto, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 165, "delivery": true, "alternative": null}}
{"product": "Kingsway car cover", "said": "Your voice is breaking, speak loudly. Sold out, sir. There was no power since morning. We have the Autofurnish one, that's 460. My boy has gone for tea. Tell me what to do", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Autofurnish"}}
{"product": "9W LED bulb", "said": "Yes yes, available. ₹90", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "Lagta hai nahi hai... ha mil gaya, hai, paanch sau chalis hoga, bhijwa dunga", "answer": {"in_stock": true, "price_inr": 540, "delivery": true, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "मशीन आहे, मिळेल, रेट कमी झालाय, 2110 होता, आता 1950", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "hello, haan bola, ho, oil milel, bill pahije asel tar GST vegla, rate 430, pickup karava lagel", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "yes, bola, light sampliye, bazarat mal kami ahe sadhya, theek ahe?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "I thought it was finished, but there's one left. 330/-.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Schneider Classic switch", "said": "ho, milel, MRP 660 aahe, tumhala 430 la deto", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Nahi vatta... thamba, ahe, ek urlay", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello haan awaaz aa rahi hai bolo kay aahe na haan maal pada hai pichasi plus 18 percent gst", "answer": {"in_stock": true, "price_inr": 85, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, bijagri ahe, 10 nagancha box 745 la, sutta ek 90, bara delivery nahi karat", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "हो, बोला दादा, ट्यूब संपली नाही, आहे, 730 MRP आहे, पण 410 ला देऊ", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "आमच्याकडे आहे डिलिव्हरीची काळजी करू नका पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan toh hai ji, mil jayega, 90 rupaye.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "Machine sampli. Cheston wala aahe, 4700 la", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cheston"}}
{"product": "1 inch PVC elbow", "said": "See, it's not there right now, it's ordered, should come by Monday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हैलो आवाज़ नहीं आ रही बाद में करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "hai, bilkul hai 5400 plus GST", "answer": {"in_stock": true, "price_inr": 5400, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "220 lagega, han, hai apne paas", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "haan bola ho milel sadesatrashe rupayanna padel", "answer": {"in_stock": true, "price_inr": 1750, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "hello, bola na, ho daada, ahe, aek hazar rupaye.", "answer": {"in_stock": true, "price_inr": 1000, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "हैलो? हाँ, आवाज़ आ रही है, हाँ हाँ, है, 820 MRP लिखा है, 545 लगा देंगे.", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "जरा वेळाने फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AUTOSiTY handle grip", "said": "Out of stock. My boy has gone for tea. We were closed yesterday, Sunday. OK?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Xtandard helmet lock", "said": "नमस्कार, बोला, नाहीये, Accessoreez lock चालणार असेल तर सांगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Accessoreez lock"}}
{"product": "Tradon bike cover for Access", "said": "Normally 150, but it's finished. Not in stock now, it will come tomorrow evening", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप आहे मिळेल डिलिव्हरी नाही दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Bosch screwdriver bit set", "said": "आहे की, मिळेल, तीनशे पंचाहत्तर होतील, पोरगा पाठवतो, ऐंशी रुपये रिक्षाचे.", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Xtreme MB Windtone horn", "said": "namaste ji, boliye, abhi dukaan band kar raha hun, kal subah phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "Out of stock, madam", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "हॅलो? आवाज येतोय का? हां, बोला. हो, मशीन आहे. पाच हजार दोनशे आणि 18 टक्के GST वेगळा", "answer": {"in_stock": true, "price_inr": 5200, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Sadhya out of stock aahe, new stock parva yenar aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "hello, bola na, aahe aahe, aapla rate 1420, MRP 2510 aahe", "answer": {"in_stock": true, "price_inr": 1420, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "Yes, go ahead, yes sir, we have, 260 rupees", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bhaiya bataiye nahi hai 9ah wala chalega pandrah sau ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "9ah"}}
{"product": "Ritco tap", "said": "nal hai, mil jaega, address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "See, it's finished, sorry", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "han sir wo model available hai 1400 lagega bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "namaskar, bola, ho, thevto aamhi, ahe, aata daha ahe, aadhi 8 hota", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "बोला, काय पाहिजे? आहे ना. रेट 260 आहे", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Inalsa OFR 2000W room heater", "said": "hello aawaj yetoy ka haan bola bagha ahe ahe saat hazar naushe rupayanna padel aur kuch", "answer": {"in_stock": true, "price_inr": 7900, "delivery": null, "alternative": null}}
{"product": "Aryshaa LED torch", "said": "abhi available nahi hai, order place kiya hai, Monday tak aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "haan wahi vala hai 330 men ho jayega upar mrp 365 likha hai khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 330, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "Price 438 ahe, ho, milel.", "answer": {"in_stock": true, "price_inr": 438, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "पुट्टीचा स्टॉक बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iBELL 200A welding machine", "said": "Yes, it's there. Rate I'll confirm and tell you", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita angle grinder", "said": "नाही नाही, आहे, बरं चौतीसशेला पडेल.", "answer": {"in_stock": true, "price_inr": 3400, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Battery khatam nahi hui, hai. pandreh sau ka padega. 12V 9Ah battery chahiye toh wo bhi hai, ek hajar saat sau pachas wala. dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Duravolt LED batten", "said": "Ek minute ha... haan bola, tube sampliye, atta nahiye, parva paryant yeyil, ekshe sattar fix", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हो, हेल्मेट शिल्लक आहे, सव्वा सातशे होतील, पोरगा पाठवतो, तीस रुपये रिक्षाचे", "answer": {"in_stock": true, "price_inr": 725, "delivery": true, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "मिल जाएगा, है अपने पास, एक सौ चालीस लगेगा", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "हां बोला, आज सकाळपासून लाईट नाही, हां हां, ऐकतोय, काय सांगू, सकाळपासून गर्दी आहे, एक गिऱ्हाईक आहे, एक मिनिट, शेठ बाहेर गेलेत, ते आले की सांगतो, ठीक आहे?", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AdroitZ bike cover", "said": "ji boliye, ji han, rakha hai, MRP 660 likha hai, 420 laga denge.", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "दोनशे सत्तर फिक्स। आहे आहे", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "७० ला येतो, पण सध्या संपलाय।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "cover ahe, milel, 615 la deto", "answer": {"in_stock": true, "price_inr": 615, "delivery": null, "alternative": null}}
{"product": "Candes 10 L geyser", "said": "Aamhi te thevat nahi, kal Ravivar hota mhanun dukan band hota.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "ha sir, bataiye. not available sir. uski jagah AmiciTools wala le sakte ho", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "Makita angle grinder", "said": "मशीन आत्ता नाहीये संपली", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "मशीन है, मिल जाएगी. हाँ तो 5700 लगेगा. डिलीवरी का 50 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 5700, "delivery": true, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "बोला ना। अठ्ठावीसशे पन्नासला देतो। म्हणजे स्टॉकमध्ये आहे।", "answer": {"in_stock": true, "price_inr": 2850, "delivery": null, "alternative": null}}
{"product": "Speedwav horn", "said": "हाँ भाई, बोलो, अच्छा चेक करके बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "ji sir, tell me, yes, available hai, arre price 1230 hai, bata dena", "answer": {"in_stock": true, "price_inr": 1230, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "है ना कितने चाहिए तीस रुपये लगेंगे भेजने वाला कोई नहीं है आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ji, available hai, sadhe teen sau ka hai, isse kam nahi hoga, ghar pe bhej dunga, 100 lagega bhejne ka.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "बोला, काय पाहिजे? शेठ बाहेर गेलेत, ते आले की सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो? आवाज येत नाहीये, नंतर करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "Hello, yes, I can hear you. Yes, available. That will be Rs 615. Pickup only. OK?", "answer": {"in_stock": true, "price_inr": 615, "delivery": false, "alternative": null}}
{"product": "Goldcave tap", "said": "नमस्ते, बोलिए, पता नहीं, गोदाम में देखना पड़ेगा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "Metro संपलं, Michelin वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Michelin"}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। है ना, कितने चाहिए? आपके लिए एक सौ पचास। शाम तक पहुँचा देंगे।", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "बोला, काय पाहिजे? आवाज तुटतोय, मोठ्याने बोला, ती संपली हो, उद्या सकाळी माल येईल, आत्ता नाहीये, हां तर साडेअठराशे रुपये, पोरगा चहा आणायला गेलाय, घरपोच देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ भाई बोलो आज सुबह से लाइट नहीं है चौदह पीस हैं स्टॉक में एक ग्राहक खड़ा है एक मिनट अरे क्या बताऊँ सुबह से बहुत भीड़ है पचास का पड़ेगा", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "है भाई, अब १२५० है, पहले १०८० था, डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 1250, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "pata nahi, godown mein dekhna padega bhaiya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बाजारात माल कमी आहे सध्या बॅटरी आत्ता नाहीये संपली कंपनीने परत रेट वाढवले 9Ah वाला आहे चालेल का कळवा मग", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "9Ah"}}
{"product": "16A MCB", "said": "ते कालच संपलं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल। म्हणजे पंचावन्नला पडेल।", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "Nahi hai... ruko ruko, hai, ek pada hai, 235 ka hai.", "answer": {"in_stock": true, "price_inr": 235, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "एक मिनिट हां... बोला। प्लग संपलाय। म्हणजे लोकल वाला घ्या, ते चालेल। घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "लोकल"}}
{"product": "Avita switch", "said": "Han ji, boliye, aapko number kisne diya? acha theek hai, helo? kuch sunai nahi de raha, baad me karo, ek customer khada hai, ek minut, market me maal ki kami chal rahi hai, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pankha padlay aata naushe panchavann ahe aadhi aathshe tees hota delivery nahi shop varun pickup kara", "answer": {"in_stock": true, "price_inr": 955, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "haan sir, boliye acha haan hai 25 rupaye lagenge.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "ji, available hai, 200 laga denge, waise MRP 305 likha hai.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Ek minute ha... haan bola, bagha ho, machine ahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bhaiya, bataiye, dekho dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "kya hai na tera nag shillak aahet sandhyakali pathavun deto sanga kay karaycha.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "आज सकाळपासून लाईट नाही. आवाज तुटतोय, मोठ्याने बोला. कंपनीने परत रेट वाढवले. बघतो आणि कळवतो. काय सांगू, सकाळपासून गर्दी आहे.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी पडलीये, 1150 आहे, 1450 नाही, म्हणजे येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 1150, "delivery": false, "alternative": null}}
{"product": "Usha 230 mm exhaust fan", "said": "pankha ahe, milel. rate savva hazar ahe.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ha sir, boliye, thodi der baad call karo.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "tala khatam nahi hua, hai, 60 mm padlock ka rate do sau hai, ye wala do sao assi", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "haan bhai, bolo, pichle hafte bhi kisi ne yahi poocha tha, mal khatm hai, company ne is mahine phir rate badhaya, aur kuch chahiye?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "एक मिनट... हाँ बोलिए, सोलह सौ पचास का पड़ेगा, पंखा पड़ा है", "answer": {"in_stock": true, "price_inr": 1650, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आहे ना, आठशे सत्तर सांगितलं का? चुकलो, सातशे आहे, घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 700, "delivery": false, "alternative": null}}
{"product": "Wipro Garnet Plus 22W LED batten", "said": "ट्यूब संपली नाही, आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wipro Garnet 10W LED batten (pack of 2)", "said": "hai hai... nahin yaar, last piece subah bik gaya, Anchor batten hai, wo le lo, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Anchor batten"}}
{"product": "Sonik G-7 helmet", "said": "Hello, bola na, kay aahe na stock madhe aahe, tumhala sadesatshe la deto, MRP aathshe pannas aahe", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Michelin wiper blade", "said": "एक ग्राहक खड़ा है, एक मिनट, स्टॉक में है, देखिए चार सौ पंद्रह लगा देंगे, वैसे MRP 480 लिखा है, कंपनी ने फिर रेट बढ़ा दिया, आवाज़ कट रही है, ज़ोर से बोलिए, भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 415, "delivery": false, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "maal sampla, mhanje tyachya aivaji Ravirandal sealant deu shakto, bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Ravirandal sealant"}}
{"product": "Bosch drill machine", "said": "Ha ji, boliye kya hai na haan, stock mein hai 8300 rupaye lagenge dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 8300, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "We have it, yes, Rs 200 only", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Haan sir, bataiye. haan sir, available hai. 65 rupaye. delivery free hai", "answer": {"in_stock": true, "price_inr": 65, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "kay aahe na nahiye, magvun deto, don divas lagtil, thik hai?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "haan tar nahi kasa, aahe na, pathavayla manus nahi, ithe yeun ghya", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "mahit nahi, godown madhe baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Avita switch", "said": "hello bola na sadhya nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "pankha padlay, mhanje delivery deto, 20 charge lagel, maalache 3200, delivery vegli.", "answer": {"in_stock": true, "price_inr": 3200, "delivery": true, "alternative": null}}
{"product": "Havells 1500W immersion rod", "said": "हाँ जी मिल जाएगा सात सौ पचपन रुपये का है घर पर भेज देंगे चालीस अलग से", "answer": {"in_stock": true, "price_inr": 755, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Yes, available. I'll give it for 1250, the MRP is 2210.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "V-GRIP steering cover for Ritz", "said": "Bagha milel na, ahe, delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "hello, ha bolo, haan toh haa ji, mil jaega, chalis hoga, bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "ho pankha milel chovisshe rupees kami nahi honar delivery deto daha vajeparyant me ithech ahe", "answer": {"in_stock": true, "price_inr": 2400, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला साहेब। वायर संपली नाही, आहे। रेट शेठ सांगतील। डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "I thought it was finished, but there's one left. Pickup only", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "matlab nau sau pachpan rupaye cover pada hai.", "answer": {"in_stock": true, "price_inr": 955, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "Available, sir. 1,390 rupees. We can deliver, 60 rupees delivery charge", "answer": {"in_stock": true, "price_inr": 1390, "delivery": true, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "ek customer khada hai ek minute woh hum nahi rakhte pichle hafte bhi kisi ne yahi poocha tha market mein maal ki kami chal rahi hai aapko number kisne diya acha thik hai agrico plier chalega ek sau nabbe ka hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "agrico plier"}}
{"product": "Ravirandal silicone sealant", "said": "hello, ha bolo, available hai, 210 ka hai, isse kam nahin hoga, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "संपलंय, त्याच्याऐवजी 12V 5Ah battery देऊ शकतो, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "12V 5Ah battery"}}
{"product": "Philips Astra Line 20W LED batten", "said": "आहे ना, ७६० पडेल, ७८०चा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 760, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "एक मिनिट हां बोला आज सकाळपासून लाईट नाही बाजारात माल कमी आहे सध्या पोरगा चहा आणायला गेलाय टायर आहे का बघावं लागेल नंतर सांगतो एक गिऱ्हाईक आहे एक मिनिट", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हैलो? हाँ, आवाज़ आ रही है, देखिए अभी बिज़ी हूँ, आधे घंटे बाद फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley angle grinder", "said": "बोला, काय पाहिजे? ती संपली हो. त्याच्याऐवजी Bosch वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch"}}
{"product": "Anchor LED batten", "said": "ट्यूब आहे, मिळेल, पाठवतो, 60 रुपये डिलिव्हरी चार्ज लागेल, वस्तू 195 ला आहे, डिलिव्हरी वेगळी.", "answer": {"in_stock": true, "price_inr": 195, "delivery": true, "alternative": null}}
{"product": "MYRA helmet lock", "said": "ji boliye ho aahe aamchyakade aapla rate 170 mrp 200 ahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "Hello, who is this? My boy has gone for tea. It's 2700 rupees. Yes sir, we have. See, one customer is here, one minute. Market is short of stock these days", "answer": {"in_stock": true, "price_inr": 2700, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हो दादा, आहे, बरं 40 होतील, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "Haan sir, wo model available hai. matlab price baarah sau hai.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "Shillak nahi saheb", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield chain lube", "said": "ek minute, check karun sangto, ek giraik aahe, ek minute", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "maal kal aayega abhi nahi hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Yes, tell me. Yes, we have it. Price changed, it's 3510 now, was 2810 last month. We'll deliver by evening", "answer": {"in_stock": true, "price_inr": 3510, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "बरं बल्ब संपलाय। LED वाला आहे, चालेल का? डिलिव्हरी करत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED"}}
{"product": "Sulfar Lutian water pump", "said": "acha abhi busy hun aadhe ghante baad phone karna", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "udya sakali maal yeil, aatta nahiye, acha bata dena", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "nahi nahi, aahe, tumhala 1500 la deto.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "प्लेट पडलीये साहेब.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हो, टेप शिल्लक आहे, बिल हवं असेल तर GST वेगळा, रेट वीस", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "हेल्मेट संपलं नाही, आहे। ४३० ला पडेल।", "answer": {"in_stock": true, "price_inr": 430, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "Yes, go ahead. It's in stock. Now 2,260, before it was 1870.", "answer": {"in_stock": true, "price_inr": 2260, "delivery": null, "alternative": null}}
{"product": "Apex Vivid helmet", "said": "नाही ना, संपलंय साहेब", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "ha han, bolo, ha, hai apne paas, rate pachaas hai, dekho theek hai?", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "एक मिनिट हां... बोला. कव्हर संपलं नाही, आहे. 745 ला देतो", "answer": {"in_stock": true, "price_inr": 745, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ahe na kimat nantar sangto aatta list nahi delivery nahi shop varun pickup kara sanga kay karaycha", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Company increased rates again. It's in stock. My boy has gone for tea. We were closed yesterday, Sunday. 1150, final price.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "aatta nahiye, kalva mag.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "abhi nahi hai, order diya hai, do din me aa jaega, delivery free hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho, ahe aamchyakade, sutta ek 8 la, purna paket 60", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "maal kal aayega, abhi nahi hai. umm home delivery nahi hai bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "Haan bola, kay pahije? bara aahe, ajun 7 piece padlet, 140 fix", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "ek customer khada hai, ek minut, Diwali ka season hai na, sab yahi maang rahe hain, haan ji, mil jayega, are kya bataun, subah se bahut rush hai, 745 bola kya? galti se bola, 590 hai, pichle hafte bhi kisi ne yahi poocha tha, thik hai?", "answer": {"in_stock": true, "price_inr": 590, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "It's finished, sorry.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "Yes, we keep that, the box of 10 is 2270, single piece 259, home delivery is available.", "answer": {"in_stock": true, "price_inr": 259, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "हाँ, है। मतलब 625 MRP लिखा है, 415 लगा देंगे। डिलीवरी फ़्री है।", "answer": {"in_stock": true, "price_inr": 415, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "रेट 30च्या आसपास असेल, स्टॉक बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "150 ला पडेल, म्हणजे आहे की, मिळेल, येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "bara tube padliye. tumhala charshe tees la deto, MRP 765 aahe. matlab address pathava, pathavto", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Kam nahin hai, bahut pada hai, aapke liye derh sau", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "Bara hai na, kitne chahiye? 3340 hai, 3270 nahi bagha dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 3340, "delivery": false, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "Yes, tell me. Uh, we have 15 pieces in stock right now. That will be Rs 1160", "answer": {"in_stock": true, "price_inr": 1160, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "wo model available nahi hai, kal Sunday tha toh dukaan band thi, chahiye toh big teflon roll de dunga, arre kya bataun, subah se bahut rush hai, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Kingsway car mat for Ecosport", "said": "Not in stock now, it will come tomorrow evening. It comes to 2010", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Accessory Mart car mat for Duster", "said": "kay aahe na ahe ahe, aapla rate battisshe pannas, MRP pach hazar tinashe ahe.", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "haan sir boliye nahi milega bhai acha chahiye toh local seal kit de dunga", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "silicone sealant", "said": "हाँ जी, मिल जाएगा. अभी रेट पौने तीन सौ हो गया, पिछले महीने दो सौ पंद्रह था", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बल्ब संपलाय, 12W LED bulb आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12W LED bulb"}}
{"product": "MK 16A switch", "said": "140 rupaye ka hai haan toh haa, mil jaega bhai", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "Yes, go ahead, yes, it's there, 515 plus GST, we don't deliver.", "answer": {"in_stock": true, "price_inr": 515, "delivery": false, "alternative": null}}
{"product": "Bikeblazer bike cover", "said": "Cover nahi hai, khatam ho gaya, matlab Cross Fingers cover hai, wo le lo", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers cover"}}
{"product": "teflon tape", "said": "Mhanje ho, tape milel, panchavis rupayanna padel, pathavayla manus nahi, ithe yeun ghya.", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "ATC bike cover", "said": "pehle ek sao paintalis ka tha, abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "Hai, bilkul hai, kya hai na 635 ka packet hai, khule bhi hain 410 wale.", "answer": {"in_stock": true, "price_inr": 635, "delivery": null, "alternative": null}}
{"product": "Anabond silicone sealant", "said": "मतलब आज सुबह से लाइट नहीं है, हाँ, माल पड़ा है, 310... नहीं नहीं, 275 है, त्योहार का सीज़न है ना, और कुछ?", "answer": {"in_stock": true, "price_inr": 275, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ख़त्म हो गया था... नहीं, एक बचा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "70 la padel, haan tar ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Speedwav handle grip", "said": "bola saheb, company ne parat rate vadhavle, aaj sakali light navhti, hai, bilkul hai, rate badh gaya, 395 se 495 ho gaya, bhijava dunga, aur kuch?", "answer": {"in_stock": true, "price_inr": 495, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो कोण बोलतंय ती संपली हो दिवाळीची गर्दी आहे ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BMB welding machine", "said": "hello, bola na, bagha aatta gadbadit aahe, ardhya tasane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "हो, कव्हर आहे. MRP १५१० आहे, तुम्हाला ११०० ला देतो. घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Retina संपलं, V-GRIP वाला आहे, चालेल?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "V-GRIP"}}
{"product": "Activa clutch plate", "said": "म्हणजे संपलंय. लोकल वाला देऊ शकतो, 610 पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "लोकल"}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "हाँ बोलिए, आवाज़ कट रही है, ज़ोर से बोलिए, माल खत्म है, कंपनी ने फिर रेट बढ़ा दिया, त्योहार का सीज़न है ना", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो? आवाज येतोय का? हां, बोला। काय सांगू, सकाळपासून गर्दी आहे। बावीसच पीस उरलेत। हां हां, ऐकतोय। आवाज तुटतोय, मोठ्याने बोला। आज सकाळपासून लाईट नाही। तुम्हाला पंचवीसला देतो। संध्याकाळपर्यंत पाठवून देतो।", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Bajaj immersion rod", "said": "We have it, yes. Actually, I'll give it for 470, the MRP is 625.", "answer": {"in_stock": true, "price_inr": 470, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "नहीं है... रुको रुको, है, एक पड़ा है.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceat Milaze tyre", "said": "udya sakaali mal yeil, aatta nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, it's there, it's 320, not 335", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "हो, मिळेल. 170 सांगितलं का? चुकलो, 190 आहे", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Viral Shopping steering cover for Mahindra", "said": "cover aahe, milel, ek customer khada hai, ek minute, rate badalla, aata 220, aadhi 250 hota, market mein maal ki kami chal rahi hai, arre kya bataun, subah se bahut rush hai, gharpoch milel.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Vista 500 ml chain lube", "said": "हाँ भाई, बोलो। देखिए नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "See, owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Blow Hot 2000W room heater", "said": "मतलब नहीं है... रुको रुको, है, एक पड़ा है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 250A welding machine", "said": "हो, आहे शिल्लक", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello, bola na, ek hazar pachshe la yeto, pan sadhya samplay", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan bhaiya bataiye tape khatam ho gayi sir aaj bahut busy day hai network issue hai aawaz break ho rahi hai season time hai na demand zyada hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crokrok car mat for Vento", "said": "ji ha, rakha hai. dekho Packy Poda mat bhi hai, uska 1300, aur aap wala 1299 ka", "answer": {"in_stock": true, "price_inr": 1299, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, go ahead. It's festival season, everyone is asking. Out of stock. We were closed yesterday, Sunday.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "Hello, who is this? It's in stock. That will be Rs 165", "answer": {"in_stock": true, "price_inr": 165, "delivery": null, "alternative": null}}
{"product": "AmiciVision LED torch", "said": "Ji boliye. gyaarah sau pachas ka padega. ladka chai lene gaya hai. company ne is mahine phir rate badhaya. han, wahi vala hai. ek customer khada hai, ek mint. pichle hafte bhi kisi ne yahi poocha tha. bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "नमस्ते, बोलिए. हाँ, रखा है. आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "हाँ, माल पड़ा है, रेट बदल गया है, अब १०५० है, पहले ९१५ था", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "Hai, bilkul hai. khula ek do sau saath ka, poora packet chaar sau painsath. delivery nahin karte, aake le jao", "answer": {"in_stock": true, "price_inr": 465, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "arre malak nahit, nantar phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Haa, maal pada hai, rate puch ke batata hun.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "हाँ भाई, बोलो, स्टॉक में है, १५५० का पड़ेगा, बताइए क्या करना है.", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Induro plier", "said": "पक्कड पडलीये पाचशे पंचेचाळीस आहे चारशे नव्वद नाही", "answer": {"in_stock": true, "price_inr": 545, "delivery": null, "alternative": null}}
{"product": "AutoPowerz LED headlight", "said": "Haan bola, kay pahije? sahashe tees fix, light aahe, milel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 630, "delivery": true, "alternative": null}}
{"product": "Rico 1500W immersion rod", "said": "haan ji, mil jayega, 850 ka hai, isse kam nahin hoga, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "abhi khatm hai, sham tak aa jayega, rate 160 hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ahe ajun 19 piece padlet rate sheth sangtil", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "kya hai na ha hai... ek minute, nahi, wo kal khatam ho gaya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "बोला ना, स्टॉकमध्ये आहे, आता सत्तर आहे, आधी एकोणसाठ होता.", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan bola haan tar ho, don piece ahet 720 la padel", "answer": {"in_stock": true, "price_inr": 720, "delivery": null, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "अच्छा Crompton Greaves नहीं है Kirloskar वाला है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar"}}
{"product": "Wipro 22W LED batten", "said": "umm tube khatam nahi hui hai 430 ka padega delivery ka 100 extra", "answer": {"in_stock": true, "price_inr": 430, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "Sorry, finished, I can give the AmiciTools one instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AmiciTools"}}
{"product": "BMB welding machine", "said": "machine khatam ho gayi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "aatta sangta yenar nahi, sandhykali phone kara. bazarat maal kami ahe sadhya.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "haa ji boliye aath so ka milega ha stock men hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 800, "delivery": false, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Not sure, I'll have to check the godown. It's very busy since morning.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "ek minute ha... haan bola, pankha samplay, Orient wala aahe, tyacha price pavanechar hajar, kay aahe na pickup karava lagel.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "Orient"}}
{"product": "10W-30 engine oil 1 L", "said": "Sorry, finished", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "Yes, speak, no, we don't have it, actually, market is short of stock these days", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Matlab hai ji, mil jaega, pichle mahine chhiyasath tha, ab pachattar.", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "Dewalt drill machine", "said": "wo item nahi rakhte hum, Stanley wala hai, 7200 ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley"}}
{"product": "1 inch PVC elbow", "said": "Yes it's... no sorry, that's the other one, this one is not there", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Kami nahi, bharpur ahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Plug ahe, milel, rate badalla, ata 120, aadhi 140 hota, yeun nyava lagel.", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "dekho ji haan, rakha hai. pehle 365 tha, ab teen sao ho gaya, sasta hua hai. sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 300, "delivery": false, "alternative": null}}
{"product": "Minda Black Current Bc-64 horn", "said": "हॅलो? आवाज येतोय का? हां, बोला, आधी 945 ला होता, आता आहे की नाही बघावं लागेल.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "han ji, mil jayega, 300 rupaye lagenge, aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "Ji, available hai, rate 510 hai, kya hai na 9 baje tak main yahin hun", "answer": {"in_stock": true, "price_inr": 510, "delivery": null, "alternative": null}}
{"product": "Ravirandal silicone sealant", "said": "haan, hai apne paas matlab pehle 170 tha, ab 185 ho gaya", "answer": {"in_stock": true, "price_inr": 185, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "जी बोलिए, मतलब ७८० का पड़ेगा, पुट्टी पड़ी है.", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "हेल्मेट पडलंय", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "आहे की, मिळेल भाऊ", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cross Fingers bike cover for Bajaj", "said": "बघा कव्हर संपलंय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "बघा टेप पडलीये, रेट वीस आहे, big teflon roll हवं असेल तर ते पण आहे, 16", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "बघा हो आहे... नाही नाही, तो दुसरा आहे, हा नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Leebo LED headlight for Honda", "said": "है भाई। डिलीवरी का 100 एक्स्ट्रा। सामान तेरह सौ पचास का है, डिलीवरी अलग से", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "Goodyear tyre", "said": "Yes sir, boliye, bhaiya aap ruko do min, phone pe hun, pichle hafte bhi kisi ne yahi poocha tha, stock men sirf tis piece hai, best price chhattis sau pachas, ek customer khada hai, ek minute, free delivery hai, aur kuch?", "answer": {"in_stock": true, "price_inr": 3650, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "haan, stock mein hai, 660 bola kya? galti se bola, aath so bees hai", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "Haan haan, bolo, savaa teen hajar ka hai, isse kam nahi hoga, hai ji, mil jayega, kya hai na address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 3250, "delivery": true, "alternative": null}}
{"product": "Crompton 1500W immersion rod", "said": "kal Sunday tha to dukan band thi, chaar sau pachas last rate hai, awaaz kat rahi hai, zor se bolo, ha hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला साहेब, बरं बिजागरी संपलीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "Yes, available. 855, final price. We don't deliver.", "answer": {"in_stock": true, "price_inr": 855, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "हो, आहे शिल्लक. ६० ला देतो. अजून काही?", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "bara nahi hai ji, khatam. Favourite Bikerz wala rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Favourite Bikerz"}}
{"product": "Rainfun car cover", "said": "आहे की, मिळेल, पाचशे ऐंशी रुपये, बरं डिलिव्हरी फ्री आहे", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "आहे की, मिळेल, तीस पडेल, 24चा रेट जुना आहे, तो नाही", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Rs 40 only. Yes sir, we have", "answer": {"in_stock": true, "price_inr": 40, "delivery": null, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Okay so, no... wait, yes, we have one piece left, it comes to 1220, no delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": 1220, "delivery": false, "alternative": null}}
{"product": "Technolube Motul C2 400 ml chain lube", "said": "Yes sir, we have. Okay so, we'll send it, 50 for delivery. Item price is 1390, delivery is extra.", "answer": {"in_stock": true, "price_inr": 1390, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello? Yes, available. 110, final price. OK?", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "Anchor Wide 400 mm pedestal fan", "said": "हाँ बोलिए. त्योहार का सीज़न है ना. रेट दो हज़ार तीन सौ है. लड़का चाय लेने गया है. कंपनी ने फिर रेट बढ़ा दिया. हाँ जी, मिल जाएगा. लड़का भेज देता हूँ.", "answer": {"in_stock": true, "price_inr": 2300, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "टेप संपली नाही, आहे, १५ होतील, घरपोच देतो, दुकान ८ वाजेपर्यंत उघडं आहे.", "answer": {"in_stock": true, "price_inr": 15, "delivery": true, "alternative": null}}
{"product": "R-tek multimeter", "said": "हाँ जी मिल जाएगा सात सौ MRP लिखा है चार सौ तीस लगा देंगे डिलीवरी नहीं करते दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 430, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan ji, plug available hai, ab 180 hai, last month 140 tha.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Ritco tap", "said": "नल ख़त्म हो गया Goldcave वाला चलेगा उसका २२० है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Goldcave"}}
{"product": "Activa air filter", "said": "है ना, कितने चाहिए? रेट 190 है, भेजने में कोई दिक्कत नहीं", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "गिझर पडलाय, 2600 आहे, 2380 नाही, संध्याकाळपर्यंत पाठवून देतो", "answer": {"in_stock": true, "price_inr": 2600, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "assi last rate hai, ha, hai apne paas, aapko number kisne diya? acha thik hai, Diwali ka season hai na, sab yahi maang rahe hain, delivery free hai.", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Morphy Richards 2000W room heater", "said": "आत्ता नाहीये, संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Adima LED torch", "said": "ho bola, maal sampala, Aryshaa torch aahe, ekshe navvad la, gharpoch milel", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Aryshaa torch"}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan bolo aaj nahi hai, kal aayega aapko number kisne diya? acha thik hai kal Sunday tha toh dukan band thi 340 ka hai aj subah se light nahin hai company ne is mahine phir rate badhaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "hai bhai, rate abhi pata nahi, malik se puchna padega, home delivery nahin hai bhai.", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "helmet padlay, tumhala charashe panchyannav la deto, MRP pachshe pasasht aahe", "answer": {"in_stock": true, "price_inr": 495, "delivery": null, "alternative": null}}
{"product": "Usha 1000W immersion rod", "said": "ते मिळणार नाही इथे. बरं क्रॉम्प्टन वाला चालणार असेल तर सांगा. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "क्रॉम्प्टन"}}
{"product": "Philips Slimline 9W LED batten (pack of 2)", "said": "hello, haa bolo Diwali ka season hai na, sab yahi maang rahe hai ha ha, hai last time 570 tha, abhi 715 hai dukaan pe aake lena padega pichle hafte bhi kisi ne yahi poocha tha.", "answer": {"in_stock": true, "price_inr": 715, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। 100 रुपये लगेंगे। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए।", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "पहले २८५० का था, अभी है कि नहीं देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "नहीं नहीं, है ना", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ridroid LED headlight", "said": "Sold out, sir. Actually, the Speedwav one is there, you can take that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Speedwav"}}
{"product": "2 inch paint brush", "said": "No, we don't have it. We have paint roller, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "Studds Urban helmet", "said": "hello? aawaj yetoy ka? haan bola, ho, helmet ahe, aathshe pandhra rupaye.", "answer": {"in_stock": true, "price_inr": 815, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "Yes, it's there, printed price is 1580, I'll give it at 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "Basically, not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "pankha hai ki nahi, dekh ke batata hun.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Makita drill machine", "said": "मशीन पडलीये। बरं रेट शेठ सांगतील। पाठवायला माणूस नाही, इथे येऊन घ्या।", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Mihsp water pump", "said": "Not available sir", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri aahe, milel, rate malakanna vicharun sangto, aur kuch?", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Haan sir, bataiye, abhi busy hun, aadhe ghante baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "nahi aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient ABS room heater", "said": "हाँ जी, मिल जाएगा। GST मिलाकर 1520। भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": true, "price_inr": 1520, "delivery": false, "alternative": null}}
{"product": "Orient ABS room heater", "said": "Yes, bola, owner nahit, nantar call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kaltron 8 inch plier", "said": "कमी नाही, भरपूर आहे, पावणे दोनशेला देतो", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Studds Urban helmet", "said": "matlab haa, mil jayega bhai. 753 hai, 580 nahi. bhejne wala koi nahi hai, aapko aana padega", "answer": {"in_stock": true, "price_inr": 753, "delivery": false, "alternative": null}}
{"product": "Aegon angle grinder", "said": "Kay aahe na dedh hajar rupaye, mhanje machine padi hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "Shakti 220A welding machine", "said": "हॅलो? आवाज येतोय का? हां, बोला। मशीन आहे का बघावं लागेल, नंतर सांगतो। हां तर आज सकाळपासून लाईट नाही", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Nahi milega bhai, aur kuch?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "बघा आमच्याकडे आहे, 50 नगांचा बॉक्स 1290 ला, सुटा एक 30", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "Sonik G-7 helmet", "said": "umm aaj subah se light nahi hai. helmet nahi hai, khatam ho gaya. bhaiya aap ruko do min, phone pe hun. ladka chai lene gaya hai. ek customer khada hai, ek minat", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "हाँ हाँ, है, 330 का पड़ेगा, local air filter भी है, वो 295 का, डिलीवरी फ़्री है", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe ahe, tumhala 440 la deu, delivery nahi, shop varun pickup kara, kalva mag.", "answer": {"in_stock": true, "price_inr": 440, "delivery": false, "alternative": null}}
{"product": "Bosch wiper blade", "said": "नहीं है जी, ख़त्म", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "hello? aawaj yetoy ka? haan bola, ahe na, aapla rate char hazar pannas, MRP 4810 ahe, sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": 4050, "delivery": true, "alternative": null}}
{"product": "Crompton 800W room heater", "said": "Ha sir, bataiye, available hai, koi problem nahi, last month 1330 tha, ab 1750 hai, delivery ho jayegi sir", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए, मालिक नहीं हैं, बाद में फ़ोन कीजिए.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "Ho, pankha aahe, atharashe rupaye hotil.", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Benz helmet", "said": "haan sir bataiye wo model available nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Aaj subah se light nahi hai, thodi der baad call karo, haan haan, sun raha hun, bolo kya karna hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Galaxy car cover for Micra", "said": "ek minute... haan bolo. haan toh abhi khatam hai bhai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "hello? aawaj yetoy ka? haan bola wire sampli", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ रखते हैं पर अभी ख़त्म है उसकी जगह 5Ah वाला दे सकते हैं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "5Ah"}}
{"product": "Accessoreez helmet lock", "said": "ho sir, available ahe. eka nagache 280, daha nagancha box 2400 la. home delivery aahe", "answer": {"in_stock": true, "price_inr": 280, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "nahi nahi, hai na.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "helmet pada hai, apna rate char sau pachas, MRP toh 600 hai.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Modx steering cover for Xcent", "said": "Ji, available hai, haan toh aapko paune panch sau men milega, MRP 545 hai, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 475, "delivery": false, "alternative": null}}
{"product": "V-Guard Zio 3 L geyser", "said": "बरं आमच्याकडे नसतं, ऑर्डर करून मागवतो, येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fluke multimeter", "said": "Owner abhi available nahi hain, baad mein call kijiye.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हो दादा, आहे, पासष्टला पडेल, आठ वाजेपर्यंत मी इथेच आहे", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "Dow silicone sealant", "said": "Bhaiya aap ruko do minute, phone pe hun, haa haan, sun raha hun, kal Sunday tha to dukaan band thi, hai bhai, 365 hoga, delivery ho jayegi, aapko number kisne diya? acha thik hai", "answer": {"in_stock": true, "price_inr": 365, "delivery": true, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "ek minute ha haan bola te kalach sampla havells wala ghya te chalel porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "havells"}}
{"product": "Activa clutch plate", "said": "Haan bolo, plate aahe, milel, umm 900 hotil.", "answer": {"in_stock": true, "price_inr": 900, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "बॅटरी संपली नाही, आहे। 1150 ला देतो", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "Aahe, 20ch urlet, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "Can you call after some time?, boss.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "haan bola kay pahije ahe ki milel sadeteenshe la padel ajun kahi", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "No no, it's there, see, Rs 200 only.", "answer": {"in_stock": true, "price_inr": 200, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Available, sir. Price is 80. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Cheston angle grinder", "said": "एक मिनिट हां... बोला, आहे की, मिळेल, तुम्हाला एक हजार चारशेला देतो, पाठवायला माणूस नाही, इथे येऊन घ्या", "answer": {"in_stock": true, "price_inr": 1400, "delivery": false, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "Actually, it used to be 935, I'll have to check if it's there.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Kyun nahi milega, hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Yes, we have it. We can deliver, 20 rupees delivery charge. 340 for the item, plus delivery", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Xinor Nexo helmet", "said": "available hai, bhejne mein koi problem nahi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ जी, बल्ब रखा है. भेज देता, पर आज लड़का नहीं आया, आकर ले जाओ", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो कोण बोलतंय आहे की मिळेल 120 रुपयांना पडेल पत्ता पाठवा पाठवून देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan, wahi wala hai, matlab 190 plus GST.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan, stock mein hai, 60 rupaye lagenge, delivery ka 20 extra.", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "हो, बल्ब आहे, MRP 335 आहे, तुम्हाला 285 ला देतो, घरपोच देतो", "answer": {"in_stock": true, "price_inr": 285, "delivery": true, "alternative": null}}
{"product": "Vormir 200A welding machine", "said": "Namaskar, bola kay aahe na samplay, udya dupari yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Haan tar ho sir, available aahe, rate 250 aahe sir, delivery deto sir, aath vajeparyant mi ithech aahe.", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "सवा पाँच सौ का है, इससे कम नहीं होगा। जी, उपलब्ध है।", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "AmiciTools 60W soldering iron", "said": "Ho, 28 piece aahet, pachshe pannas rupayanna padel", "answer": {"in_stock": true, "price_inr": 550, "delivery": null, "alternative": null}}
{"product": "Stanley bolt cutter", "said": "माल संपला, सांगा काय करायचं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "I thought it was finished, but there's one left. See, that will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Bosch horn", "said": "hello? ha, awaaz aa rahi hai, bolo, haa, mil jayega bhai, printed MRP che sau tees, our price paach sau bees, khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 520, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello? aawaj yetoy ka? haan bola stock samplay M-seal ghya, te chalel gharapoch milel", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "M-seal"}}
{"product": "water pump seal kit", "said": "Hello? ha, awaaz aa rahi hai, bolo, hai bhai, 340 final, delivery nahi karte, aake le jao.", "answer": {"in_stock": true, "price_inr": 340, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "Haan aahe, delivery chi kalji karu naka, pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Wipro 22W LED batten", "said": "haa bhai, bolo, haan, mil jaega bhai, aapko sava chaar sau me de denge, MRP 755 hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 425, "delivery": true, "alternative": null}}
{"product": "Autoplus helmet lock", "said": "पता नहीं, गोदाम में देखना पड़ेगा, अच्छा आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "haan bhai, bolo, bhaiya aap ruko do mint, phone pe hun, haan ha, sun raha hun, awaaz kat rahi hai, zor se bolo, kal subah tak aa jayega, abhi nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol 10W-40 engine oil", "said": "han sir, wo model available hai, home delivery available hai, 50 charge, item price chah sau tees, delivery charge alag", "answer": {"in_stock": true, "price_inr": 630, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "बोला ना, स्टॉकमध्ये आहे, हां तर वीस रुपये.", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Yes, we keep that. I don't know the rate, the owner will tell you. We don't deliver. Uh, anything else?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "BMB 250A welding machine", "said": "लड़का चाय लेने गया है, आज सुबह से लाइट नहीं है, कंपनी ने फिर रेट बढ़ा दिया, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, BMB नहीं है, Vormir machine है, ठीक है?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Vormir machine"}}
{"product": "wall putty 20 kg", "said": "पुट्टी संपली नाही, आहे, तुम्हाला साडेनऊशेला देतो, हां तर 8 वाजता दुकान बंद होतं", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हो, ठेवतो आम्ही, आहे. एका नगाचे १२०, २० नगांचा बॉक्स २१३० ला", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Osram LED headlight", "said": "बावीस नग शिल्लक आहेत. पावणेचारशे होतील", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Candes 1000W immersion rod", "said": "आहे की, मिळेल. तीनशे पंधराला पडेल.", "answer": {"in_stock": true, "price_inr": 315, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "मतलब बल्ब है, मिल जाएगा, एक पीस ३१०, ६ लोगे तो २६० के हिसाब से", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Haan bola, kay pahije? bazarat maal kami aahe sadhya, porga chaha aanayla gelay, ho, cover aahe, 255 fix, theek aahe?", "answer": {"in_stock": true, "price_inr": 255, "delivery": null, "alternative": null}}
{"product": "Schneider Classy switch", "said": "acha han sir, available hai, aapko panch sau pachpan mein milega, MRP 860 hai.", "answer": {"in_stock": true, "price_inr": 555, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "Activa nahi hai, Dio wala hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dio"}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Ha sir, boliye, haan, maal pada hai, paune chaar sau final", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Raaisin car cover", "said": "हॅलो, कोण बोलतंय? आहे ना। आमचा रेट 4500, MRP 5350।", "answer": {"in_stock": true, "price_inr": 4500, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हॅलो, कोण बोलतंय? उद्या सकाळी माल येईल, आत्ता नाहीये", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 1.0 HP water pump", "said": "Ho bola nahi milanar bazarat maal kami aahe sadhya.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Vento Axial 150 mm exhaust fan", "said": "pankha sampla nahi, aahe, sadenaushe hotil, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 950, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "दिवाळीची गर्दी आहे ना, तेराशे पन्नास रुपये, बॅटरी पडलीये, बाजारात माल कमी आहे सध्या, आवाज तुटतोय, मोठ्याने बोला, पत्ता पाठवा, पाठवून देतो, कळवा मग", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "Ho, aahe aamchyakade, eka nagache ekshe daha, 10 nagancha box naushe pannas la.", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ji han rakha hai 150 ka ek, 25 loge toh 125 ke hisaab se laga dunga", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Tata Agrico plier", "said": "bagha hai na, kitne chahiye? sava do sau rupaye, thik hai?", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Bosch GWS angle grinder", "said": "हो, १६ पीस आहेत, हां तर २२०० होतील", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "Bara hai, bilkul hai, sawa nau sau mein de dunga, matlab bolo kya karna hai", "answer": {"in_stock": true, "price_inr": 925, "delivery": null, "alternative": null}}
{"product": "Vega Crux helmet", "said": "हां बोला हेल्मेट आहे का बघावं लागेल नंतर सांगतो दिवाळीची गर्दी आहे ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Aapko number kisne diya? acha theek hai, haan, ready stock hai, kya hai na aapke liye sava sau, kal Sunday tha toh dukan band thi, home delivery available hai, bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 125, "delivery": true, "alternative": null}}
{"product": "Havells Cista room heater", "said": "हाँ है... एक मिनट, नहीं, वो कल ख़त्म हो गया। Orient heater है, चलेगा?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Orient heater"}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo... nahi, wo nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "Okay so, yes yes, available. MRP 605, but our price is 385", "answer": {"in_stock": true, "price_inr": 385, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "kya hai na ladka dekh ke batayega, ruko zara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Digismart pedestal fan", "said": "Maal khatam hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हाँ बोलिए २६० का आता है लेकिन अभी ख़त्म है शाम तक माल आ जाएगा अभी नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "haan bolo... nahi, wo nahi hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ha bhaiya, bataiye, thodi der bad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mobil Super 5W-30 3.5 L engine oil", "said": "आमच्याकडे आहे, चौदाशे पन्नास रुपये, पाठवतो, वीस रुपये डिलिव्हरी चार्ज लागेल", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Yes, we keep that, MRP is 2,210, we'll give it for 1250.", "answer": {"in_stock": true, "price_inr": 1250, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ती संपली हो, म्हणजे त्याच्याऐवजी 2.5 sq mm वाला देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm"}}
{"product": "Philips 5W LED bulb", "said": "बल्ब संपला नाही, आहे। तुम्हाला 125 ला देतो, MRP 155 आहे। पाठवायला माणूस नाही, इथे येऊन घ्या।", "answer": {"in_stock": true, "price_inr": 125, "delivery": false, "alternative": null}}
{"product": "CarMobCentral car cover for Ritz", "said": "ha sir, bataiye, abhi khatam hai bhai, network issue hai, aawaz break ho rahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "nahin milega bhai, Lakshmi pump hai, 2950 ka.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Lakshmi pump"}}
{"product": "4 socket extension board", "said": "520 lagega, haan sir, wo model available hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 520, "delivery": true, "alternative": null}}
{"product": "Retina steering cover for Swift", "said": "Abhi stock me nahi hai, naya mal Monday ko ayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ho sir bola company ne parat rate vadhavle bazarat maal kami aahe sadhya mhanje ho sir available ahe ek giraik ahe ek minute 235 nahi nahi savvadonshe theek ahe", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Bajaj Maxima 600 mm ceiling fan", "said": "Yes sir, we have, actually, price is 1,500, delivery is 50 extra.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": true, "alternative": null}}
{"product": "Liqui Moly Street Race 10W-40 engine oil", "said": "ऑइल संपलं. Mobil oil आहे, १२०० ला", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Mobil oil"}}
{"product": "AutoClean wiper blade", "said": "Aahe aahe, aapla rate tinashe panchechalis, MRP pachshe sattar aahe, delivery deto", "answer": {"in_stock": true, "price_inr": 345, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "हाँ हाँ, है, मतलब आपको ३४५ में दे देंगे, MRP ३७५ है, डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 345, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "हाँ तो कवर है मिल जाएगा लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "ahe... ek minute, nahi, te kal sampala, AutoPowerz headlight aahe aamchyakade, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "AutoPowerz headlight"}}
{"product": "Pulsar 150 spark plug", "said": "Ho aahe. daha ghetle tar 150 pramane, ek piece 170.", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "कमी नाही, भरपूर आहे, दोनशे साठ रुपयांना पडेल", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Hello, good morning, yes yes, available, ₹190, we can deliver, my boy has gone for tea, your voice is breaking, speak loudly, it's festival season, everyone is asking, anything else?", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "pankha sampla nahi, aahe savva teen hazar fix", "answer": {"in_stock": true, "price_inr": 3250, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "है भाई, 1450... नहीं नहीं, 1270 है.", "answer": {"in_stock": true, "price_inr": 1270, "delivery": null, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "hai, bilkul hai. MRP chah sau panch hai, par teen sau painsath de dena.", "answer": {"in_stock": true, "price_inr": 365, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Almonard pedestal fan", "said": "Han sir, boliye, pankha hai, mil jayega, chah hajar final, address bhej do, bhej dete hain", "answer": {"in_stock": true, "price_inr": 6000, "delivery": true, "alternative": null}}
{"product": "Atomberg exhaust fan", "said": "एकोणीसशे पन्नास रुपये, कमी नाही होणार, पंखा पडलाय, ठीक आहे?", "answer": {"in_stock": true, "price_inr": 1950, "delivery": null, "alternative": null}}
{"product": "DHRUV-PRO multimeter", "said": "It's in stock. Uh, our price is 175, MRP is 315", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "मिल जाएगा है अपने पास अब दो सौ दस है पहले 190 था आपको आकर लेना पड़ेगा", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "Ho, tube milel. pavanesatshe rupaye. gharpoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 675, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "माल संपला बरं उद्या सकाळी माल येईल आत्ता नाहीये एक हजार पन्नासला देतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Daybetter 25W LED bulb", "said": "बल्ब संपला नाही, आहे, म्हणजे २६० रुपये, कमी नाही होणार", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho, aahe aamchyakade. packet 80 la aahe, sutta ek 9 la", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Crompton 3 L geyser", "said": "म्हणजे थांबा, बघून सांगतो।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "battery khatam ho gayi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Safeyura LED batten", "said": "जी बोलिए, पता नहीं, गोदाम में देखना पड़ेगा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous Audie 1200 mm ceiling fan", "said": "हो, बोला दादा। आहे की, मिळेल। आता पस्तीसशे आहे, आधी तीन हजार होता। बरं डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": true, "price_inr": 3500, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Hai hai... nahi yaar, last peice subah bik gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shakti 200A welding machine", "said": "Haa bhai, bolo, haan, stock me hai, delivery free hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Anchor LED batten", "said": "हो दादा, आहे। MRP दोनशे दहा आहे, तुम्हाला पावणेदोनशेला देतो। घरपोच नाही, तुम्हाला यावं लागेल", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Aaj subah se light nahi hai, hai bhai, Diwali ka season hai na, sab yahi maang rahe hain, matlab han haa, sun raha hun, abhi rate panch sau chalis ho gaya, pichle mahine chaar sau pachasi tha, bhaiya aap ruko do min, phone pe hun, aur kuch?", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "V-Guard Gatimaan Pro pedestal fan", "said": "haan toh hai, bilkul hai, do hajar nau sau pachas... nahi nahi, teen hajar hai", "answer": {"in_stock": true, "price_inr": 3000, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "1100 rupayanna padel battery padliye gharapoch nahi, tumhala yava lagel", "answer": {"in_stock": true, "price_inr": 1100, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "Ek mint... haan bolo, arre nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Hella horn", "said": "hello? kuch sunai nahi de raha, baad me karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हॅलो? आवाज येतोय का? हां, बोला, बल्ब संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "हॅलो कोण बोलतंय ती संपली हो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha 1250 mm ceiling fan", "said": "pehle 2950 ka tha abhi hai ki nahi dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Candes 1200W room heater", "said": "No... wait, yes, we have one piece left, Rs 1400 only", "answer": {"in_stock": true, "price_inr": 1400, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "arre haa rakhte hain par abhi khatm hai retina cover hai mere paas", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "retina cover"}}
{"product": "Accessory Mart car mat for Duster", "said": "ji sir, tell me, haan, ready stock hai, aapke liye 3510, aapko shop se pickup karna hoga", "answer": {"in_stock": true, "price_inr": 3510, "delivery": false, "alternative": null}}
{"product": "Lifelong 5 L geyser", "said": "गिझर संपलाय, Bajaj geyser आहे आमच्याकडे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj geyser"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "No sir, not there. We have 8 mm bolt, same thing.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "8 mm bolt"}}
{"product": "12V 7Ah battery", "said": "Ji sir, tell me, haan, ready stock hai, pandra sau ka hai, sorry, delivery service nahi hai.", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "बोला, काय पाहिजे? रेट 190 आहे. म्हणजे आहे की, मिळेल", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "अभी नहीं है, local glue है, तीन सौ साठ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local glue"}}
{"product": "2 inch paint brush", "said": "haan bola, kay pahije? ho, milel, sattar fix", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "कंपनीने परत रेट वाढवले, आवाज तुटतोय, मोठ्याने बोला, आज सकाळपासून लाईट नाही, काय सांगू, सकाळपासून गर्दी आहे, आज नाही, उद्या येईल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "संपलंय साहेब.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, it's there. Now 2100, before it was 2,600.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Wadhwa Creations car cover for Innova", "said": "aahe ek minute nahi te kal sampala", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ जी, बताइए. हाँ, है. रेट पूछ कर बताता हूँ. मतलब होम डिलीवरी हो जाएगी. ठीक है?", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Khaitan Avaante room heater", "said": "संपले सगळे, म्हणजे हॅवेल्स वाला आहे, अकराशे पन्नासला, डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "हॅवेल्स"}}
{"product": "Activa clutch plate", "said": "ti sampli ho, udya sakali maal yeil, aatta nahiye.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Maxx 400 mm pedestal fan", "said": "nahiye bahutek... ho ho, sapadla, ahe. sandhyakali pathavun deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "RNG Eko tyre", "said": "बोला ना, टायर संपलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ तो है भाई.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "Yes, it's there. Printed price is 200, I'll give it at 160. We don't deliver", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "रेट सहाशे साठ आहे, पण आत्ता स्टॉक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "1550 ka hai, haa, wahi wala hai", "answer": {"in_stock": true, "price_inr": 1550, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, ठेवतो आम्ही, आहे, तुम्हाला 230 ला देतो, घरपोच देतो, 9 वाजेपर्यंत मी इथेच आहे", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "Dr.Fixit silicone sealant", "said": "नाही वाटतं... थांबा, आहे, एक उरलाय, एकशे नव्वद रुपये.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "stock me sirf solah piece hai rate puch ke batata hun same day delivery kar denge", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bosch wiper blade", "said": "hello, ha boliye, matlab dekhna padega bhai, yaad nahi hai, pichle hafte bhi kisi ne yahi poocha tha", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "Hai na, kitne chahiye? 6 ka full box 1580, single piece 300 home delivery available hai", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "Ji boliye 90 final hai bhai bhejne wala koi nahi hai, aapko aana padega company ne is mahine phir rate badhaya Diwali ka season hai na, sab yahi maang rahe hain arre kya bataun, subah se bahut rush hai thik hai?", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "AutoClean wiper blade", "said": "Kay aahe na ho, ahe aamchyakade, rate sheth sangtil, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "MK 16A switch", "said": "ji available hai 220 bola kya galti se bola pone do sau hai dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 175, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "वो हम नहीं रखते, आवाज़ कट रही है, ज़ोर से बोलिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "Ek mint... haa bolo, mil jayega, hai, aapke liye unnis sau pachas, haan toh address bhej do, bhej dete hai", "answer": {"in_stock": true, "price_inr": 1950, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "plate sampli nahi, aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "ho sir, bola, 16A nahiye, 20A wala aahe.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20A"}}
{"product": "12V 7Ah battery", "said": "yes, available hai, ladka de aayega, 40 rupaye auto ka, maal ka 1200, delivery alag", "answer": {"in_stock": true, "price_inr": 1200, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "It's very busy since morning. Actually, company increased rates again. We were closed yesterday, Sunday. 4 socket is not there, we have 3 socket board. It's festival season, everyone is asking.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 socket board"}}
{"product": "Retina steering cover for Alto", "said": "Actually, 290/-. We have it, yes", "answer": {"in_stock": true, "price_inr": 290, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "1520 final ji, available hai.", "answer": {"in_stock": true, "price_inr": 1520, "delivery": null, "alternative": null}}
{"product": "Cheston 500W drill machine", "said": "Machine padi hai. 2200 plus GST. bara khud aake le jaiye", "answer": {"in_stock": true, "price_inr": 2200, "delivery": false, "alternative": null}}
{"product": "Starvin pedestal fan", "said": "haan ready stock hai pehle 1150 tha ab terah sau pachas ho gaya address bhej do bhej dete hain", "answer": {"in_stock": true, "price_inr": 1350, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hai, bilkul hai, 950 rupees, theek aahe?", "answer": {"in_stock": true, "price_inr": 950, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "अच्छा हाँ हाँ, समझ गया, लेकिन वो अभी नहीं है। local brake shoe है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "jara velane phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Hello, yes, I can hear you, we have it, yes, that will be Rs 370.", "answer": {"in_stock": true, "price_inr": 370, "delivery": null, "alternative": null}}
{"product": "Maharaja 500W room heater", "said": "आहे ना, बाराशे रुपये, येऊन न्यावं लागेल.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": false, "alternative": null}}
{"product": "Cheston 200A welding machine", "said": "Ha, kaun? acha, bolo, machine hai, mil jayegi, 3600 hoga", "answer": {"in_stock": true, "price_inr": 3600, "delivery": null, "alternative": null}}
{"product": "AutoGarh steering cover", "said": "Aamchyakade nasta te", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Atomberg Renesa ceiling fan", "said": "Haa bhaiya, bataiye, umm bas 10 piece bache hain, dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Electronicspices 60W soldering iron", "said": "आज नहीं है, कल आएगा.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "Ho bola. haan tar ahe ki, milel. aadhi ekshe vees hota, aata deedashe zala", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Mihsp water pump", "said": "४५०० होतील, हो, मोटर आहे", "answer": {"in_stock": true, "price_inr": 4500, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "वायर नहीं है, खत्म हो गई. उसकी जगह 2.5 sq mm wire दे सकते हैं. भेजने वाला कोई नहीं है, आ के ले जाओ", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.5 sq mm wire"}}
{"product": "Vista 500 ml chain lube", "said": "season time hai na, demand zyada hai, woh hum nahi rakhte, sir aaj bahut busy day hai, Technolube wala available hai, uska price 465, network issue hai, aawaz break ho rahi hai, bolo kya karna hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Technolube"}}
{"product": "Duravolt LED batten", "said": "bara tube aahe, milel. pathavto, chaalis rupaye delivery charge lagel. item ekshe pannas la, pathavnyache vegle", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "ekshe sath rupaye hotil, kay aahe na ho, kulup milel", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Diwali chi gardi ahe na, tape sampli, bazarat maal kami ahe sadhya, company ne parat rate vadhavle, kalva mag", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fluke multimeter", "said": "Ha sir, boliye, ladka chai lene gaya hai, helo? kuch sunai nahin de raha, baad me karo, awaaz kat rahi hai, zor se bolo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Favourite Bikerz FBZ horn", "said": "हैलो, कौन? मिल जाएगा, है अपने पास, एमआरपी ५९५ है, पर हम ३७५ में देते हैं, लड़का भेज देता हूँ", "answer": {"in_stock": true, "price_inr": 375, "delivery": true, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "Ladka dekh ke batayega, ruko zara, season time hai na, demand jyada hai.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हाँ, माल पड़ा है, अच्छा तीन सौ चालीस रुपये, डिलीवरी का 50 एक्स्ट्रा", "answer": {"in_stock": true, "price_inr": 340, "delivery": true, "alternative": null}}
{"product": "Dewalt angle grinder", "said": "Uh, yes... one minute, no, that got finished yesterday", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "mhanje abhi dukan band kar raha hun, kal subah phone karo, aaj subah se light nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Siron 60W soldering iron", "said": "It's in stock, price is 335, you'll have to collect it from the shop, anything else?", "answer": {"in_stock": true, "price_inr": 335, "delivery": false, "alternative": null}}
{"product": "Duro Gold angle grinder", "said": "ho sir, bola, machine ahe ki nahi mahit nahi, godown madhe baghto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "sampla. ek giraik aahe, ek minute.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "aahe na, 35 hotil, mhanje pathavayla manus nahi, ithe yeun ghya, dukan 8 vajeparyant ughda aahe", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "Metro Conti 140/70-17 tyre", "said": "आमच्याकडे आहे, ३१०० रुपये, कमी नाही होणार, घरी पाठवतो", "answer": {"in_stock": true, "price_inr": 3100, "delivery": true, "alternative": null}}
{"product": "Benz helmet", "said": "हेल्मेट आहे, मिळेल। डिलिव्हरीची काळजी करू नका, पाठवतो।", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "han bhaiya, bataiye. sheth baher gelet, te aale ki sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai, koi problem nahi, ek hajar panch sau... nahi nahin, do hajar hai", "answer": {"in_stock": true, "price_inr": 2000, "delivery": null, "alternative": null}}
{"product": "Kaiweets multimeter", "said": "nahin hai... ruko ruko, hai, ek pada hai, untis sau pachas lagega.", "answer": {"in_stock": true, "price_inr": 2950, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Mylinc 6A switch", "said": "Yes yes, available, 170, wait, I'm checking the list... it's 130", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Han, wahi wala hai rate 500 hai bhej denge, 20 rupaye delivery charge lagega.", "answer": {"in_stock": true, "price_inr": 500, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hai na, kitne chahiye? rate badh gaya, 61 se 80 ho gaya", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "Hello? Out of stock. We have the Cross Fingers one, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Cross Fingers"}}
{"product": "iBELL 250A welding machine", "said": "ho, machine milel, haan tar sat hazar pachshe... thamba, list baghto... aath hazar aathshe", "answer": {"in_stock": true, "price_inr": 8800, "delivery": null, "alternative": null}}
{"product": "AmiciFlo water pump", "said": "Hai... arre nahi, wo dusra wala hai, ye wala nahin hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kirloskar Chotu 0.5 HP water pump", "said": "माहीत नाही, गोडाऊनमध्ये बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Stock men sirf panch piece hai, rate puch ke batata hun, haan toh aapko shop se pickup karna hoga, theek hai?", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we keep that. That will be Rs 35. Actually, we don't deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello? han, awaaz aa rahi hai, bolo, 840 lagega, mil jaega, hai, sham tak pahuncha denge.", "answer": {"in_stock": true, "price_inr": 840, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe ahe, bavisshe rupaye, kami nahi honar, delivery deto, pannas charge lagel", "answer": {"in_stock": true, "price_inr": 2200, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "yes, available hai, 440 rupaye hai, nau baje dukan band hoti hai, delivery nahi hoti, pickup karna padega", "answer": {"in_stock": true, "price_inr": 440, "delivery": false, "alternative": null}}
{"product": "Viral Shopping steering cover for Sail", "said": "ho, thevto aamhi, aahe bara aata 190 ahe, aadhi 165 hota.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "अभी बिज़ी हूँ आधे घंटे बाद फ़ोन करना त्योहार का सीज़न है ना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "plug aahe, milel, bagha kal Ravivar hota mhanun dukan band hota, aaj sakali light navhti, 190 aahe, 200 nahi, ek giraik aahe, ek minute, porga chaha aanayla gelay, theek aahe?", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "म्हणजे आधी ५५० ला होता आता आहे की नाही बघावं लागेल", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Crompton pedestal fan", "said": "Pankha ahe, milel 2630... nahi nahi, battisshe", "answer": {"in_stock": true, "price_inr": 3200, "delivery": null, "alternative": null}}
{"product": "Havells 1200 mm ceiling fan", "said": "हां बोला... नाही, ते सध्या नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj LEDZ 5W LED bulb", "said": "nahi bhai, yahan nahi milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Retina steering cover for Alto", "said": "ek minute ha haan bola aaj nahi udya yeil yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "iBELL IBL drill machine", "said": "stock madhe aahe, tumhala 4850 la deu, pathavayla kahi problem nahi", "answer": {"in_stock": true, "price_inr": 4850, "delivery": true, "alternative": null}}
{"product": "Cross Fingers bike cover for Honda", "said": "milel na, aahe, charashe panchyannav, GST vegla", "answer": {"in_stock": true, "price_inr": 495, "delivery": null, "alternative": null}}
{"product": "Candes Star 1200 mm ceiling fan", "said": "haan rakhte hain, par abhi khatam hai, Bajaj fan chalega toh bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bajaj fan"}}
{"product": "16A MCB", "said": "Okay so, hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "ATC bike cover", "said": "Uh, I'm busy right now, call in half an hour", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हो वायर आहे तेराशे पन्नासला पडेल", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "RSCT 9W LED bulb", "said": "बल्ब पडलाय, बघा एका नगाचे दोनशे साठ, सहा नगांचा बॉक्स एक हजार चारशे पन्नासला.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Haan bola, kay pahije? mhanje aadhi 20 la hota, aata aahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Halonix 10W LED bulb", "said": "stock madhe ahe, rate malakanna vicharun sangto", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "हॅलो, कोण बोलतंय? बरं जरा वेळाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "CEAT 3.00-18 tyre", "said": "humare paas nahi hota mangwana padega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aatta gadbadit aahe, ardhya tasane phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vheelocityin car cover for Vento", "said": "Hai hai... nahi yaar, last pees subah bik gaya. uski jagah Rainfun cover le sakte ho.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Rainfun cover"}}
{"product": "Hella horn", "said": "Aapke liye 935. haan hai", "answer": {"in_stock": true, "price_inr": 935, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "हो दादा, आहे, सांगा काय करायचं.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Han, stock men hai. ek piece assi, das wala box loge toh 710", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji sir, tell me, season time hai na, demand jyada hai, kya hai na wo item hum keep nahi karte, network issue hai, aawaz break ho rahi hai, sir aaj bahut busy day hai, local wala hai, ek sau saath ka", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local"}}
{"product": "Polycab BLDC 1200 mm ceiling fan", "said": "एक मिनिट हां... बोला, हो, पंखा आहे, रेट २१५० आहे", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ho aahe... nahi nahi, to dusra ahe, ha nahiye", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है... रुको रुको, है, एक पड़ा है, रेट आठ सौ बीस है", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "MYRA helmet lock", "said": "Aadhi pavanedonshe la hota, ata ahe ki nahi baghava lagel.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नमस्कार, बोला, हां हां, ऐकतोय, दिवाळीची गर्दी आहे ना, आत्ता नाहीये, संध्याकाळी येईल, बरं 430 रुपये, कमी नाही होणार, पोरगा चहा आणायला गेलाय", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "रेट दस है, पर अभी स्टॉक नहीं है.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Singer Ir-08 1500W immersion rod", "said": "हाँ हाँ, है, डिलीवरी का 50 एक्स्ट्रा, 580 का है, प्लस डिलीवरी.", "answer": {"in_stock": true, "price_inr": 580, "delivery": true, "alternative": null}}
{"product": "teflon tape", "said": "है... अरे नहीं, वो दूसरा वाला है, ये वाला नहीं है, big teflon roll है, वो ले लीजिए", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "big teflon roll"}}
{"product": "Anchor 150 mm exhaust fan", "said": "udya sakali maal yeil, aatta nahiye, kay aahe na rate ek hajar pannas aahe sir.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "है भाई भाई.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Company ne is mahine phir rate badhaya nahi hai ji, khatam kal Sunday tha toh dukaan band thi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car mat for Ecosport", "said": "Khatam ho gaya tha... nahi, ek bacha hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Grand Pitstop GRL chain lube", "said": "kya hai na mhanje nahi hai, bolo to mangva deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Ha, wahi wala hai, aath so bees lagega", "answer": {"in_stock": true, "price_inr": 820, "delivery": null, "alternative": null}}
{"product": "Clouds wiper blade", "said": "haan ahe sahashe ainshi ahe, 635 nahi", "answer": {"in_stock": true, "price_inr": 680, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हो दादा, आहे, डिलिव्हरी नाही असं नाही, पाठवतो की.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Studds Urban helmet", "said": "aaj sakali light navhti. ek minute, check karun sangto. kay sangu, sakal pasun gardi aahe. haan haan, aiktoy. company ne parat rate vadhavle", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "प्लग संपलाय आत्ता नाहीये संध्याकाळी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ho dada, ahe, satashe ainshi fix.", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "rate ek sao bees hai, par abhi stock nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car cover", "said": "cover aahe ki nahi mahit nahi, godown madhe baghto.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sage Square helmet", "said": "Sorry, finished. We have Sonik helmet, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Sonik helmet"}}
{"product": "water pump seal kit", "said": "हैलो, कौन? अभी नहीं है. mechanical seal है, चलेगा? डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "Crispy LED torch", "said": "ji, available hai, teen sau assi ka milega", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Packy Poda car mat for Mahindra", "said": "संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Aptech multimeter", "said": "yes sir, boliye. season time hai na, demand jyada hai. paune do sau men de dunga. ji, available hai. delivery ho jayegi. sir aaj bahut busy day hai", "answer": {"in_stock": true, "price_inr": 175, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, haan bola, ho sir, available aahe, rate malakanna vicharun sangto, delivery deto sir.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "बघतो आणि कळवतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anchor 150 mm exhaust fan", "said": "ha, mil jayega bhai hajar mein de dunga matlab dukan aath baje tak khuli hai dukan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1000, "delivery": false, "alternative": null}}
{"product": "Philips Astra Line 20W LED batten", "said": "tube khatam nahi hui hai best price sat sau pandrah shop 8 baje tak open hai bhejne wala koi nahi hai aapko aana padega", "answer": {"in_stock": true, "price_inr": 715, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "kami nahi, bharpur ahe.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "बघा प्लग पडलाय, डिलिव्हरी नाही असं नाही, पाठवतो की", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "SKC steering cover", "said": "han, kaun? acha, bolo, ek sau pachaasi rupees ka hai, han, maal pada hai, umm address bhej do, bhej dete hain.", "answer": {"in_stock": true, "price_inr": 185, "delivery": true, "alternative": null}}
{"product": "Crompton Greaves 0.5 HP water pump", "said": "संपलं। Kirloskar वाला चालणार असेल तर सांगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Kirloskar"}}
{"product": "TVS Eurogrip 2.75-18 tyre", "said": "ho tyre ahe 1149 fix haan toh sandhyakali pathavun deto ajun kahi", "answer": {"in_stock": true, "price_inr": 1149, "delivery": true, "alternative": null}}
{"product": "Type R G124 horn", "said": "lagta hai nahi hai... haan mil gaya, hai, dekho 499 final price hai.", "answer": {"in_stock": true, "price_inr": 499, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं मिलेगा भाई, paint roller दे सकता हूँ, पैंतीस रुपये का.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "paint roller"}}
{"product": "wall putty 20 kg", "said": "No sir, not there. I can give wall putty 40 kg instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "wall putty 40 kg"}}
{"product": "Crompton 3 L geyser", "said": "Hello, good morning. Uh, Crompton is not there, we have AO Smith geyser", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "AO Smith geyser"}}
{"product": "HEWA Onyx 150 mm exhaust fan", "said": "ho, thevto aamhi, aahe, sahashe chalis, GST vegla", "answer": {"in_stock": true, "price_inr": 640, "delivery": null, "alternative": null}}
{"product": "Steelbird Sba-7 flip-up helmet", "said": "हेल्मेट संपलं आत्ता नाहीये परवापर्यंत येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Goldcave tap", "said": "hai ji mil jayega rate badh gaya 150 se ek sau pachattar ho gaya", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "AllExtreme LED headlight (pack of 2)", "said": "It's finished, sorry. One customer is here, one minute. Anything else?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Stanley 550W drill machine", "said": "मतलब मिल जाएगा, है अपने पास, एक हज़ार पाँच सौ पचास... रुकिए, लिस्ट देखता हूँ... अठारह सौ", "answer": {"in_stock": true, "price_inr": 1800, "delivery": null, "alternative": null}}
{"product": "Havells 450 mm pedestal fan", "said": "Price 4000 aahe, aahe sir, ready stock aahe.", "answer": {"in_stock": true, "price_inr": 4000, "delivery": null, "alternative": null}}
{"product": "Mastech multimeter", "said": "umm abhi confirm nahi kar sakta, godown chek karna padega.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "bijagri sampli nahi ahe 30 fix", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "जी, उपलब्ध है। सवा चार सौ का पड़ेगा। दुकान दस बजे तक खुली है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 425, "delivery": false, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Hello, yes, I can hear you. Yes yes, available. ₹25. You'll have to collect it from the shop", "answer": {"in_stock": true, "price_inr": 25, "delivery": false, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "बोला ना। बल्ब आहे, मिळेल। चारशेला पडेल। डिलिव्हरी नाही, दुकानात येऊन घ्या। कळवा मग।", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Crompton Greaves Drift 300 mm exhaust fan", "said": "Haan, kaun? acha, bolo ji haan, rakha hai arre 2050 last rate hai pichle hafte bhi kisi ne yahi poocha tha market me maal ki kami chal rahi hai awaaz kat rahi hai, zor se bolo kal Sunday tha toh dukan band thi.", "answer": {"in_stock": true, "price_inr": 2050, "delivery": null, "alternative": null}}
{"product": "JK 90/100-10 tyre", "said": "हाँ तो हाँ, है। आपके लिए 1300। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1300, "delivery": true, "alternative": null}}
{"product": "Syska 9W LED bulb (pack of 5)", "said": "Yes, we have it. Loose ones are 93 each, the full packet is 410.", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "Bajaj Bahar 300 mm exhaust fan", "said": "dekhna padega bhai, yaad nahi hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, we have 21 pieces, 180, final price, actually, there was no power since morning, it's festival season, everyone is asking, let me know.", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "bagha ahe na, 990... nahi nahi, 830", "answer": {"in_stock": true, "price_inr": 830, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "bara haan bola... nahi, te sadhya nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ भाई, बोलो, अभी नहीं है, अरे क्या बताऊँ, सुबह से बहुत भीड़ है, 1 sq mm wire रखा है, एक ग्राहक खड़ा है, एक मिनट, कंपनी ने फिर रेट बढ़ा दिया.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "12V 7Ah battery", "said": "1000 का आता है, लेकिन अभी ख़त्म है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Matlab hai bhai, delivery nahin hoti aisa nahi hai, bhej dunga.", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "शिल्लक नाही.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Auctimo car cover for Nexon", "said": "ha bhai, bolo, awaaz kat rahi hai, zor se bolo, market men maal ki kami chal rahi hai, han, hai apne paas, chah sau sath rupaye, haa, 23 piece pade hain, umm aur kuch chahiye?", "answer": {"in_stock": true, "price_inr": 660, "delivery": null, "alternative": null}}
{"product": "MK 16A switch", "said": "ho aahe, 25 nagancha box 3830 la, sutta ek 170, delivery deto", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ha bolo, ji, available hai, aapke liye 190, dekho bolo kya karna hai.", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "Nahiye bahutek... ho ho, sapadla, ahe, mhanje paachashe daha rupees", "answer": {"in_stock": true, "price_inr": 510, "delivery": null, "alternative": null}}
{"product": "Speedwav LED headlight", "said": "लाईट पडलीये. म्हणजे 1240 सांगितलं का? चुकलो, 1640 आहे. पत्ता पाठवा, पाठवून देतो.", "answer": {"in_stock": true, "price_inr": 1640, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "कुलूप संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "बरं दोनशे साठ रुपयांना पडेल। स्टॉकमध्ये आहे। घरपोच देतो", "answer": {"in_stock": true, "price_inr": 260, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
# Real Hindi answers from Amazon xPQA (CDLA-Sharing-1.0; see data/real/NOTICE-xpqa.md)
REAL_XPQA = [json.loads(line) for line in r'''
{"product": "pTron Bassbuds Vista true wireless earbuds", "said": "यह अभी उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vivo V21 5G (Sunset Dazzle, 8GB RAM, 128GB)", "said": "नहीं। यह मोबाईल अभी नहीं मिल रहा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "यह उत्पाद स्टॉक में है। कार्ट में डालके ले सकते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "अभी यह फोन स्टॉक में है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "हाँ। यह झारखंड में उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Paurush Jeevan + Perfect capsule combo pack", "said": "पौरुष जीवन और परफेक्ट केपसुल का कोम्बो पॅक नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ashwagandha, safed musli, shatavar, kaunch beej mixed pack", "said": "नहीं। अश्वगंधा,सफेद मूसली, सतावर, कौंच बीज, इन सबको मिलाकर मिला मिलाया आनुपातिक पैक नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wecool Moonwalk Mini earbuds charging case (spare)", "said": "नहीं। चार्जिंग केस अलग से नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "इसका ब्लेड अलग से नहीं मिलता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ambrose ceiling fan, 900 mm", "said": "नहीं। 900 mm में उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "XS and OS women's nightwear set, 40 inch size", "said": "नहीं। इसमें 40 इंच का नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं, ब्लैक कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rehmans Mumbai model wooden sofa set, other colour", "said": "नहीं, इसमें दूसरा कलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं, रेड कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Oomph georgette salwar suit material, stitched", "said": "नहीं। यह कपडा सिला हुआ नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anni Designer crepe dress material, stitched", "said": "नहीं। यह सलवार सूट सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kundan Sulz suit fabric, stitched", "said": "नहीं। यह सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "नहीं। यह 19 इंच की नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "हाँ। यह 19 इंच की है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "हाँ। यह सैमसंग a12 6gb 128gb का है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Pacer 1200 mm ceiling fan, 5 star rated", "said": "नहीं। यह 5 स्टार नहीं है, इस फैन की कोई स्टार रेटिंग नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ फ्री साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"L\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "L साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"M\", \"XL\" और \"XXL\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M, XL और XXL साइज़"}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं। यह सिर्फ डार्क ब्राउन(गहरा भूरा) रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डार्क ब्राउन"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ग्रे रंग"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ चारकोल ब्लू में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "चारकोल ब्लू"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ सिल्वर और ग्रे में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिल्वर और ग्रे"}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं। यह मेटल ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मेटल ग्रे रंग"}}
{"product": "Pare leather jacket, colour other than black", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "iBell Eiffel pedestal fan, other colour", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा जिसपे हल्का कॉपर रंग होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Autonest bike side luggage box, Splendor Plus (silver)", "said": "नहीं। यह काले रंग में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Tecno Spark 6 Air (black, 3GB RAM, 64GB)", "said": "नहीं। 3gb रैम और 64gb स्टोरेज के साथ नीले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "नीले रंग"}}
{"product": "SDS Fashion boys' t-shirt, other colour, 11-12 years", "said": "नहीं। यह 13-14 के बच्चों के लिए काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग, 13-14"}}
{"product": "Oomph georgette salwar suit material (XXL)", "said": "नहीं। यह फ्री साइज़ का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Nir Fashion net lehenga choli (3XL)", "said": "नहीं। यह ड्रेस 2xl है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2xl"}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "नहीं। यह सैमसंग a12 4gb 64gb का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सैमसंग a12 4gb 64gb"}}
{"product": "L&T three phase energy meter", "said": "यह एल&टी का सिंगल फेज मीटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिंगल फेज मीटर"}}
{"product": "Shree multi-output transformer, copper winding", "said": "नहीं। यह एल्युमिनियम का ट्रांसफार्मर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एल्युमिनियम का ट्रांसफार्मर"}}
{"product": "Activa 3 litre instant gas geyser", "said": "नहीं। यह इलेक्ट्रिक गाइज़र है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इलेक्ट्रिक गाइज़र"}}
{"product": "Microtek MTK3012SMU solar inverter", "said": "नहीं। यह इनवर्टर नहीं है, सिर्फ सोलर चार्ज कन्ट्रोलर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सोलर चार्ज कन्ट्रोलर"}}
{"product": "Fire-Boltt 360 Pro smart watch, metal body", "said": "नहीं। यह मेटल बॉडी नहीं है, सिलिकॉन बॉडी है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिलिकॉन बॉडी"}}
{"product": "Women's regular kurta (black & grey), cotton", "said": "नहीं। यह कॉटन नहीं है, यह रेयॉन का कपडा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "रेयॉन का कपडा"}}
{"product": "Leather Retail men's biker jacket, pure leather", "said": "नहीं। यह प्यूर लेदर नहीं है, फॉक्स लेदर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फॉक्स लेदर"}}
{"product": "Jaspo heavy duty cricket bat, wooden", "said": "नहीं। यह लकड़ी का नहीं प्लास्टिक का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "प्लास्टिक"}}
{"product": "HP all-in-one desktop PC with Intel i5", "said": "नहीं। इसमें एएमडी राइज़न है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एएमडी राइज़न"}}
{"product": "Epson all-in-one laser printer", "said": "नहीं। यह इंक प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इंक प्रिंटर"}}
{"product": "Webby wooden India political map puzzle, Hindi", "said": "नहीं। यह अंग्रेजी में लिखा हुआ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अंग्रेजी"}}
{"product": "Faber 60 cm kitchen chimney, island model", "said": "नहीं। यह वाल माउंट है। आइलैंड के लिए दुकान जाने की जरूरत है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वाल माउंट"}}
{"product": "Niyama Bluetooth thermal printer, A4 size", "said": "नहीं। यह सिर्फ 2 इंच प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2 इंच प्रिंटर"}}
{"product": "Dot bathroom tiles cleaner, 5 litre jar", "said": "नहीं। यह 500 एम एल का जार है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 एम एल का जार"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। यह 27*24*4 इंच का पेड़ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "27*24*4 इंच"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। 36\" (h) x 24\" (w) x 4\" का पैड मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "36\" x 24\" x 4\" का पैड"}}
{"product": "Bru instant coffee, pack bigger than 200 g", "said": "नहीं। ब्रू कॉफी 200 gm के पैक में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "200 gm के पैक"}}
{"product": "Katra Lysorus virucide, pack smaller than 50 ml", "said": "नहीं। यह सिर्फ 50 ml के पैक में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "50 ml के पैक"}}
{"product": "Baidyanath Nagpur Swarna Bhasma, 500 g", "said": "नहीं। यह 125mg में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "125mg"}}
{"product": "Zandu Kesari Jivan, 2 litre pack", "said": "नहीं। यह 900g के पैक में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "900g के पैक"}}
{"product": "Herbalife Shakemate protein shake, 1 kg pack", "said": "यह 600 g का पैक है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "600 g का पैक"}}
{"product": "Easyshoppingbazaar bird net, 25 x 25 ft", "said": "नहीं। यह आईटम 10 x 6 foot (60 sq ft) में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "10 x 6 foot"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 32*10 में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "32*10"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 3*10m में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3*10m"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 12 सेंटीमीटर लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 सेंटीमीटर लम्बाई"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 39 लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "39 लम्बाई"}}
{"product": "Car mat for new Tata Safari 7-seater", "said": "यह सफारी 6 सीटर के लिए है और इसका दाम 1420 रुपये है डिस्काउंट के बाद।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सफारी 6 सीटर"}}
{"product": "Renmax walkie talkie, single piece", "said": "नहीं। इस पैक में दो मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पैक में दो"}}
{"product": "Zoro waist belt, pack of 3", "said": "इस कॉम्बो में २ पीस मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "२ पीस"}}
{"product": "Queenzomy bop bag toy, set of 3", "said": "नहीं। इसमें सिर्फ एक पीस आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Status car door guard, set of 4", "said": "इसमें केवल एक पीस आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Flamingo calf support, pair (both legs)", "said": "इसमें सिर्फ एक पैर की आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पैर"}}
{"product": "Kozdiko car foot mat for Renault Kwid Climber", "said": "नहीं। यह ऐल्टो के लिए है, क्विड पे नहीं लगेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raya keyboard skin for 15.6 inch laptop", "said": "नहीं। यह 14 इंच लैपटाप के लिये उपयुक्त है, 15.6 इंच लैपटाप के लिये उपयुक्त नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Casotec silicone back cover for Tecno iA5", "said": "नहीं। यह टेक्नो केमन i ट्विन के लिए बना है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Eltron H4 headlight bulb for Bajaj Pulsar 125", "said": "नहीं। यह बजाज पल्सर 125 के हेडलाइट में फिट नहीं होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Live Tech 12A toner cartridge for HP 1020 Plus", "said": "नहीं। इसे एच पी 1020 में लगा सकते है पर एच पी 1020 प्लस में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AOW brake and clutch levers for Honda Shine SP 125", "said": "नहीं। यह होंडा शाइन एसपी 125 में फिट नहीं होता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JK Tyre Blaze BF32 front tyre for Honda 110cc", "said": "नहीं। यह होंडा 110cc में नहीं चलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Swift Dzire 2015", "said": "नहीं। यह किया सॉनेट के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car body cover for Alto LX 2008", "said": "नहीं। यह कवर शेव्रोले क्रूज़ के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shopoflux silicone key cover for Hyundai Eon", "said": "नहीं। यह हुंडई इयान के लिए परफेक्ट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Front touch glass for Huawei Y9 2019", "said": "नहीं। यह टच स्क्रीन ग्लास हुआवे y9 प्राइम 2019 में यूज़ हो सकता है, y9 2019 में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Loom Solar 20A charge controller for LiFePO4 battery", "said": "नहीं। यह सिर्फ लेड बैटरी के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Micromax X412 phone with headphones", "said": "इसमें एक चार्जर साथ में उपलब्ध है पर हेडफोन साथ में नहीं दिया जाता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tender Care plastic almirah with cover", "said": "नहीं। इसमें सिर्फ अलमीरा मिलेगा, कवर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अलमीरा"}}
{"product": "Tender Care plastic almirah with cover", "said": "हाँ। इस अलमीरा का एक रजाई जैसा कवर मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Blaxstoc Bluetooth earphones with mobile stand", "said": "हाँ। इसके साथ मोबाइल स्टैंड मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Microtek solar PCU 1235 with solar panel", "said": "इसमें सौर पैनल शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "हाँ। इसके साथ 2aa बैटरी मिलेगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "नहीं। इसके साथ बैटरी नहीं मिलेगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Omo Bikes Panda 20T kids bicycle with carrier", "said": "इस में कैरियर नहीं होगा, चेन कवर पहले से ही रहता है क्रैंक वाली साइड पर।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar soundbar with mic", "said": "नहीं। मैक साथ में उपलब्ध नहीं है। fm नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kuki georgette saree with blouse piece", "said": "हाँ। इसके साथ ब्लाउज पीस है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Aquerra 25 L water heater with remote", "said": "नहीं। इसके साथ रिमोट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Monkish Apple Watch case with band", "said": "इसमें केवल घड़ी का केस मिलेगा स्क्रीन प्रोटेक्टर के साथ पर घडी या बेंड नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "घड़ी का केस"}}
{"product": "mStick silicone AirPods cover with AirPods", "said": "नहीं। आइटम में केवल एयरपोड कवर है और एयरपोड शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एयरपोड कवर"}}
{"product": "Symphony Storm 100i tower air cooler", "said": "यह सिर्फ कूलर का कवर है, इसमें कूलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ayushka Crafts tent ceiling with side walls", "said": "नहीं। साइडर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें सिर्फ बीन बैग का कवर मिलेगा, बीन्स नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बीन बैग का कवर"}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें बीन्स साथ में नहीं मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pro-Ethic men's cotton kurta with pyjama", "said": "इसमें सिर्फ एक कुर्ता मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कुर्ता"}}
{"product": "Rain gun set with rain gun, 4-leg stand and riser", "said": "हाँ। यह सेट रेन गन के साथ आती है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JBL GT-X1500THI subwoofer with amplifier", "said": "नहीं। इसमें सिर्फ सबवूफर आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सबवूफर"}}
{"product": "Hectare mango fruit picker with pole", "said": "नहीं। इसके साथ डंडा नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajab Gajab solar zatka machine 12 kV with battery", "said": "नहीं। इसके साथ बैटरी नहीं आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Coirfit 3-seater sofa cum bed with stand", "said": "नहीं। स्टैंड साथ में नहीं आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Smartcam bulb CCTV camera with memory card", "said": "नहीं। इसमें मेमोरी कार्ड साथ में नहीं आती है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mi Pad 5 with pen", "said": "पेन साथ में नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G with 18 W adapter in box", "said": "नहीं। बॉक्स में 18 वॉट का एडाप्टर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Volo all-in-one makeup kit with blusher", "said": "इसमें ब्लशर नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme Narzo 50A with charger", "said": "हाँ। इस मोबाईल के साथ चार्जर आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with aux cable", "said": "हाँ। इसके साथ ऑक्स केबल आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with charging and aux cable", "said": "हाँ। इसके साथ चार्जर केबल और ऑक्स केबल आएंगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Noise ColorFit Ultra smart watch with charger", "said": "हाँ। इसमें चार्जिंग डॉक साथ में आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Globe Dolphin silencer for RE Classic 350 with mounting kit", "said": "हाँ। इस साइलेंसर को लगाने की पत्ती और बोल्ट साथ में आते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lovely 12V solar pedestal fan with battery and solar panel", "said": "नहीं। ये बैटरी और सोलर पेनल के साथ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dell Inspiron N5010 65W adapter with power cable", "said": "नहीं। इसमें एडेप्टर के साथ पावर केबल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wacom One graphic tablet with pen battery", "said": "नहीं। इस पेन टेब के साथ बैटरी नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sahni single folding bed with mattress", "said": "नहीं, इसके साथ बिस्तर नहीं आता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Docoss rechargeable head torch with charger", "said": "नहीं। इसमें चार्जर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 rear tyre with tube", "said": "नहीं। टायर के साथ में ट्यूब नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Veera Paridhaan kurta pyjama", "said": "कुर्ता पजामा आइटम की डिलीवरी जल्दी हो सकती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Samsung 6.5 kg top load washing machine", "said": "हाँ। गडचिरोली में डेलीवरी होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bharat Darshan wooden agarbatti stand", "said": "हाँ। इस कंपनी द्वारा विदेश के मित्र को उत्पाद भेज सकते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "हाँ। 499 तक के प्रोडक्ट पे कक्ष ऑन डेलीवेरी मिल सकता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Mecstroke weed cutter wheat-cutting attachment", "said": "गेहूं काटनेवाला मशीन 15 दिन बाद भेज जाता है जिसे सेट कर सकते हो।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Senapati men's cotton socks (pack of 12)", "said": "₹359 में 12 जोड़ी मोज़े है।", "answer": {"in_stock": true, "price_inr": 359, "delivery": null, "alternative": null}}
{"product": "Shade net 50% UV stabilised garden netting", "said": "रू 259 में 5ft x32ft है।", "answer": {"in_stock": true, "price_inr": 259, "delivery": null, "alternative": null}}
{"product": "Apis Himalaya honey, 1 litre", "said": "312/- में 35.27 औंस यानी 1 लीटर मिलेगा।", "answer": {"in_stock": true, "price_inr": 312, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G (Dynamo Black, 6GB RAM, 128GB)", "said": "812 रुपये किस्त में फोन उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar 3900 soundbar", "said": "इस उत्पाद पे लिए हुए दिन से १ साल तक की मैनुफैक्चरर वारंटी मिलती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Softspun microfiber cleaning cloths, 24 pcs", "said": "इस उत्पाद में 24 टावल पीसस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Himalaya Total Care baby pants diapers, medium", "said": "एक बॉक्स मे ७८ पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "NSSP flat lanyards with J-hook (pack of 50)", "said": "इसमें ५० पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Romsons 6-layer disposable face mask (pack of 50)", "said": "एक पैक में 25 पीस आते है और ऐसा 2 पैक आते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VBCFT GK50 battery for Moto E3", "said": "नहीं। इसमें सिर्फ तीन महीने की वारंटी मिलेगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Godrej 1.5 ton 3 star inverter split AC", "said": "एसी का इन्स्टलैशन गोदरेज वाले करेंगे और इसका इन्स्टलैशन चार्ज लगेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi Note 11 Pro (Star Blue, 6GB RAM, 128GB)", "said": "कम से कम किस्त 2222 रुपये की होगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "3 महीने का 2327 रुपये हर महीना और 6 महीने का 1163 रुपये हर महीने का किस्त होगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajjas hidden GPS tracker for bike", "said": "महिने का 200 रुपये का डाटा लागत है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cera angle cock with wall flange", "said": "एक पैकेट में 1 एंगल वाल आता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "नहीं। पुरानी बैटरी एक्सचेंज नहीं होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Canon GI-790 ink bottle (black)", "said": "नहीं। 4-5शीशी एक साथ मंगाने पर डिस्काउंट नहीं मिलेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "Cash on delivery mil sakta hai iske liye aapko 499 Tak ka product karna padega lekin yah Photo bahut kharab hai breakup bahut bekar hai", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "yes it is available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "In stock.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "In stock. Add to cart.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "Extra Accessories is not available. You want extra accessories so you can call customer care.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mecstroke weed cutter wheat and grass cutting blades", "said": "HELLO , WE WILL SEND THE WHEAT AND GRASS CUTTING BLADES TO YOU 15 DAYS AFTER THE MACHINES IS DELIVERED.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "Delivery charge is free and EMI depend on the offer available at the time of purchase with the debit or credit card.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Clovia cotton non-padded wirefree bra", "said": "Delivered on time only.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HBMallIndia anti-burst gym ball", "said": "10CM ka fark he sizes me. 65cm, 75cm aur 85cm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "Hello Sir, We are currently not offering exchange services online, we would request you to get best buyback prices from our authorized dealers in your nearby area.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
TRAIN_V1 = [json.loads(line) for line in r'''
{"product": "Splendor brake shoe", "said": "haan bolo, hai bhai, home delivery hai, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, haan, stock mein hai, 220 rupaye ka hai, home delivery hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "Yes, available. It's 80 rupees", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes sir, we have. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते। उसकी जगह 3.15 mfd capacitor दे सकते हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3.15 mfd capacitor"}}
{"product": "silicone sealant", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, मिल जाएगा। 160 लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "aahe na, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan bola, aahe na, 10 piece aahet, 80 rupaye honar, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Not sure, I'll have to check the godown. Come before 10", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 16 piece aahet, vees rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, ho, milel, pannas rupaye", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. No, we don't have it. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "khatam ho gaya, uski jagah 7W LED bulb de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7W LED bulb"}}
{"product": "4 inch door hinge", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, didshe rupaye", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 110 lagega", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है। 160 रुपये का है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Not today, it comes tomorrow. We don't deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aahe na, 11 piece aahet, rate 390 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, available. That will be Rs 160", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ, है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, stock mein hai, 1600 rupaye ka hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "available hai, rate 310 hai", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan bhai, hai bhai, 15 piece pade hain, rate 360 hai", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां बोला. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. हो, आहे. 950 ला पडेल. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan ji, mil jayega, rate 380 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए। वो हम नहीं रखते। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan, stock mein hai, 5 piece pade hain, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aahe na, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ji boliye, nahi hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, tell me. Yes, we have it", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji boliye, available hai, 18 hi bache hain, ₹15, dukaan 8 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, aahe na, 14 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan bhai, abhi nahi hai, parso aayega, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, we have it. Price is 35. We can deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "stock madhe aahe, 50 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, haan, stock mein hai, 160 lagega, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan bhai, ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "थांबा, बघून सांगतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Hello, who is this?. Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "available hai, dhai sau rupaye final, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "available hai, 2 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉक में है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. पन्नास ला पडेल. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji haan, rakha hai, 20 lagega, delivery ho jayegi, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hello, kaun?, hai bhai, 410 rupaye final", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, we have it. Price is 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई। 1100 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, ghari pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मालक नाहीत, नंतर फोन करा. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। ख़त्म हो गया। घर पर भेज देंगे। दुकान 8 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ji haan, rakha hai, 130 rupaye final, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bola, aahe na, rate 1600 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "hello, kaun?, thodi der baad call karo, dukaan 8 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bhai, abhi stock mein nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aamhi te thevat nahi, tyachya aivaji CPVC elbow deu shakto, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "CPVC elbow"}}
{"product": "12V 7Ah battery", "said": "आहे ना. 9 पीस आहेत. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan, stock mein hai, 2150 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi nahi hai, parso aayega, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आम्ही ते ठेवत नाही. local putty आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local putty"}}
{"product": "Activa air filter", "said": "है भाई। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan ji, mil jayega", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "available hai, 15 piece pade hain, 35 ka padega", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give CPVC elbow instead. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "silicone sealant", "said": "ji boliye, haan ji, mil jayega, 180 rupaye final, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन?। रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो, कोण?. नाही आहे. Pulsar 180 spark plug आहे, चालेल का?. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Pulsar 180 spark plug"}}
{"product": "Activa air filter", "said": "nahi hai, ladka bhej deta hun, dukaan 10 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, 7 piece aahet, 380 rupaye honar", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ho, milel, 1750 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai, haan hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, uski jagah local clutch plate de sakte hain, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "Yes, available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 5 hi bache hain, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji boliye, available hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "haan hai, 6 hi bache hain, ₹140", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला. आहे ना. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan hai, 830 lagega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "ji boliye, pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, thread seal tape hai, chalega?, delivery nahi karte, aake le jao, dukaan 9 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "thread seal tape"}}
{"product": "50 mm padlock", "said": "ji boliye, haan, stock mein hai, 17 hi bache hain, 150 ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 14 piece aahet, 50 rupaye, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai, parso aayega, 10 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo, hai bhai, 400 rupaye final", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, आहे. 1450 रुपये", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bhai, available hai, 610 lagega", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 5 पीस पड़े हैं। 1600 रुपये का है। घर पर भेज देंगे। दुकान 7 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "It's in stock. We have 14 pieces. Price is 180. I'll send it with our boy. Come before 10", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi stock mein nahi hai, uski jagah CPVC elbow de sakte hain, ladka bhej deta hun, 9 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "Activa air filter", "said": "नहीं है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "आहे ना. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "सध्या नाही, उद्या येईल. paint roller आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "2 inch paint brush", "said": "abhi nahi hai, parso aayega, paint roller hai, chalega?, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "fan capacitor 2.5 mfd", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "hello, kon?, sampla, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "abhi stock mein nahi hai, 7W LED bulb hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "7W LED bulb"}}
{"product": "12V 7Ah battery", "said": "It's in stock. It's 1050 rupees. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "aahe na, shambhar rupaye", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "We don't keep that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म हो गया। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji boliye, abhi nahi hai, parso aayega, uski jagah 60 mm padlock de sakte hain, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "bike headlight bulb", "said": "Hello?. It's in stock. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, 130 rupaye", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "stock madhe aahe, 410 la padel", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, ji haan, rakha hai, 330 ka padega", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan aahe, 19 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. Price is 290. Pickup only", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bolo, haan ji, mil jayega, 120 lagega", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. It's in stock. That will be Rs 100. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "It's in stock. 70 rupees. I'll send it with our boy. Come before 9", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 rupaye honar", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan hai, 30 lagega, ghar pe bhej denge, 7 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है। paint roller है, चलेगा?। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "paint roller"}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, mil jayega, rate 380 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, we have it. That will be Rs 1800. We don't deliver", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, आहे. 8 पीस आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 210 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, rate 1300 hai", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bola, thamba, baghun sangto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "nahi milega bhai, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, we have it. That will be Rs 240", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, tyachya aivaji wall putty 40 kg deu shakto, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "wall putty 40 kg"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए। स्टॉक में है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji haan, rakha hai, 75 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bhai, nahi milega bhai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, hai bhai, ₹1500", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. हो, आहे. 13 पीस आहेत. 120 रुपये. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan, stock mein hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give 3/4 inch elbow instead. Pickup only", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3/4 inch elbow"}}
{"product": "9W LED bulb", "said": "आहे ना. 3 पीस आहेत. रेट 140 आहे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 30 rupaye honar, delivery deto", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bola, ho aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan ji, mil jayega, 80 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji boliye, hai bhai, rate pachaas hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नहीं है। उसकी जगह 12V 9Ah battery दे सकते हैं। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12V 9Ah battery"}}
{"product": "teflon tape", "said": "ji boliye, haan ji, mil jayega, rate 30 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, thamba, baghun sangto, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "woh hum nahi rakhte, LED headlight bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 3 hi bache hain, ₹80, home delivery hai", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan bolo, haan ji, mil jayega, rate 50 hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "hai bhai, 14 piece pade hain, delivery ho jayegi, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "It's in stock. That will be Rs 1700", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नाही आहे. त्याच्याऐवजी 3.15 mfd capacitor देऊ शकतो. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "3.15 mfd capacitor"}}
{"product": "Fevicol SH 1 kg", "said": "bola, ho aahe, 420 rupaye honar", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "स्टॉक में है। रेट 2200 है। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hai bhai, 7 hi bache hain, ₹120, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "ho, milel, 9 ch urle aahet, 70 rupaye honar", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 1150 लगेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। ख़त्म हो गया। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "है भाई। 230 लगेगा। दुकान 10 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho, milel, 270 rupaye honar", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock madhe aahe, adichshe rupaye honar, ghari pathavto", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, rate 90 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "nahi milega bhai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 5 piece pade hain, 220 lagega", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, है। 13 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bolo, hai bhai, 1450 rupaye ka hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we have it. Price is 45. Pickup only", "answer": {"in_stock": true, "price_inr": 45, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bolo, nahi hai, uski jagah 60 mm padlock de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "60 mm padlock"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "khatam ho gaya, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो, कौन?। हाँ, है। 20 पीस पड़े हैं। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes sir, we have. We have 4 pieces. That will be Rs 90. Pickup only", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. 80 रुपये. घरी पाठवतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan ji, mil jayega, 730 ka padega", "answer": {"in_stock": true, "price_inr": 730, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "सध्या नाही, उद्या येईल. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji haan, rakha hai, rate 240 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, है। 120 लगेगा। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "available hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, halogen bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "halogen bulb"}}
{"product": "1 inch PVC elbow", "said": "हां बोला. स्टॉकमध्ये आहे. तीस रुपये", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, available. 65 rupees", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "It's in stock. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji haan, rakha hai, 100 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. It's in stock. We have 9 pieces. It's 220 rupees. We can deliver", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "nahi milega bhai, local brake shoe hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "4 socket extension board", "said": "hai bhai, 520 ka padega", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan aahe, 10 rupaye, yeun nyava lagel, 9 chya aat ya", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "pata nahi, godown mein dekhna padega, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Hello?. No, we don't have it. We have local seal kit, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Activa clutch plate", "said": "malik abhi nahi hain, baad mein phone karo, 9 baje ke pehle aana", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "malak nahit, nantar phone kara, 9 chya aat ya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, 1500 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Yes, tell me. We don't keep that. I can give local brake shoe instead. Home delivery is available. Come before 8", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "bike headlight bulb", "said": "hai bhai, 7 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Yes, tell me. Yes sir, we have. Price is 350", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Sorry, finished. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "khatam ho gaya, 2.25 mfd capacitor rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.25 mfd capacitor"}}
{"product": "Pulsar 150 spark plug", "said": "आम्ही ते ठेवत नाही. local spark plug आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "9W LED bulb", "said": "Hello?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai, ₹110, delivery nahi hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, haan aahe, 170 rupaye, ghari pathavto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 120 rupaye, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "available hai, 17 piece pade hain, rate 400 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Sorry, finished. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan bola, udya yeil, aaj nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "sampla, 8 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello, kaun?, abhi stock mein nahi hai, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 6 piece pade hain, 70 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, kaun?, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "abhi nahi hai, parso aayega, uski jagah 2.25 mfd capacitor de sakte hain, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2.25 mfd capacitor"}}
{"product": "50 mm padlock", "said": "हाँ बोलिए। स्टॉक में है। 2 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi stock mein nahi hai, local seal kit rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Pulsar 150 spark plug", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. स्टॉकमध्ये आहे. 10 पीस आहेत. तीस रुपये. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hello?. Yes, we have it. Only 6 left. It's 1100 rupees", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, है। 17 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, local air filter hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local air filter"}}
{"product": "Pulsar 150 spark plug", "said": "udya yeil, aaj nahi, Pulsar 180 spark plug aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180 spark plug"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ji haan, rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hello, kaun?, woh hum nahi rakhte, delivery nahi hai, dukaan 8 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello, kon?, sampla, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ji haan, rakha hai, 760 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bhai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। रेट 90 है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ख़त्म हो गया। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला. नाही आहे. त्याच्याऐवजी 2.5 sq mm wire देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, haan ji, mil jayega, 70 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hello, kaun?, nahi hai, 12W LED bulb hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "woh hum nahi rakhte, 1 sq mm wire rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "8 mm drill bit", "said": "haan bhai, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "sampla, dukan 7 vajta paryant ughada aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan aahe, 75 rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 75, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have. 930 rupees", "answer": {"in_stock": true, "price_inr": 930, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this?. It's in stock. Price is 260", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "No, we don't have it. I can give Jupiter clutch plate instead. Home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Jupiter clutch plate"}}
{"product": "bike headlight bulb", "said": "aamhi te thevat nahi, delivery deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, sampla, tyachya aivaji 3 inch hinge deu shakto, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3 inch hinge"}}
{"product": "wall putty 20 kg", "said": "haan hai, rate 710 hai", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahi aahe, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, 4 hi bache hain, 110 rupaye ka hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bola, udya yeil, aaj nahi, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan, stock mein hai, 370 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello, kaun?, nahi hai, delivery ho jayegi", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 240 ka padega", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए। हाँ जी, मिल जाएगा। 230 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, available hai, rate 1000 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hai bhai, 16 piece pade hain, ₹360", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Not sure, I'll have to check the godown. We are open till 7 pm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी, मिल जाएगा। 70 रुपये का है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aamhi te thevat nahi, tyachya aivaji 12V 5Ah battery deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 5Ah battery"}}
{"product": "4 socket extension board", "said": "haan bhai, hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। हाँ जी, मिल जाएगा। रेट 140 है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, available. Home delivery is available. Come before 7", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, udya yeil, aaj nahi, tyachya aivaji 12W LED bulb deu shakto, delivery deto, 9 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "9W LED bulb", "said": "ho aahe, 70 rupaye honar, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe aahe, rate 2100 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "available hai, 330 ka padega, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I can give 3 inch brush instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch brush"}}
{"product": "water pump seal kit", "said": "haan ji, mil jayega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए। नहीं है। घर पर भेज देंगे। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ji boliye, nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Out of stock. I can give local brake shoe instead. We are open till 10 pm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "50 mm padlock", "said": "woh hum nahi rakhte, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "It's in stock. Price is 400. We don't deliver", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नाही आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा। 60 लगेगा", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "We don't keep that. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई। 4 पीस पड़े हैं। रेट सौ है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, nahi aahe, tyachya aivaji LED headlight bulb deu shakto, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED headlight bulb"}}
{"product": "Activa air filter", "said": "malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "संपलं. 40 mm padlock आहे, चालेल का?. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm padlock"}}
{"product": "bike headlight bulb", "said": "Yes, we have it. That will be Rs 220. Pickup only", "answer": {"in_stock": true, "price_inr": 220, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, hai bhai, ₹270, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho aahe, 110 la padel", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji boliye, haan ji, mil jayega, 14 piece pade hain, 80 rupaye final", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "nahi aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे ना. 3 पीस आहेत. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not today, it comes tomorrow. I can give Jupiter clutch plate instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "4 inch door hinge", "said": "available hai, 80 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji haan, rakha hai, 3 hi bache hain, 320 lagega", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock madhe aahe, rate 330 aahe, porga pathavto", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही आहे. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bola, haan aahe, 80 la padel, yeun nyava lagel, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes, available. We have 18 pieces. It's 780 rupees", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes sir, we have. 10 rupees", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "nahi hai, local spark plug hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, 10W-30 engine oil 900 ml rakha hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "10W-30 engine oil 900 ml"}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ बोलिए। नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो, कोण?. संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Yes, available. It's 1450 rupees. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes sir, we have. Only 19 left. 170 rupees", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello, kon?, ho, milel, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan bola, ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए। है भाई। 16 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "malik abhi nahi hain, baad mein phone karo, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 35 rupaye honar", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, available. Price is 100. Home delivery is available. We are open till 8 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला. आम्ही ते ठेवत नाही. 12V 9Ah battery आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 9Ah battery"}}
{"product": "16A MCB", "said": "आहे ना. 180 रुपये. येऊन न्यावं लागेल. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel, 600 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we have it. 280 rupees. We are open till 7 pm", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, mil jayega, dhai sau ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हॅलो, कोण?. आहे ना. 5 पीस आहेत. रेट 200 आहे. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "रुकिए, देखकर बताता हूँ। दुकान 10 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, tell me. Yes, available. That will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो, कौन?। हाँ, है। 330 रुपये का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai bhai, 340 ka padega", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I can give mechanical seal instead. We don't deliver. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "1 inch PVC elbow", "said": "woh hum nahi rakhte, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan ji, mil jayega, 13 piece pade hain, ₹1100, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Hello?. Yes, available. We can deliver", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hello, kaun?, hai bhai, rate 340 hai", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "sampla, 60 mm padlock aahe, chalel ka?, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "9W LED bulb", "said": "नहीं है। उसकी जगह 7W LED bulb दे सकते हैं। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "7W LED bulb"}}
{"product": "2 inch paint brush", "said": "haan bolo, nahi milega bhai, home delivery hai, dukaan 7 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "haan, stock mein hai, 17 piece pade hain, ₹330", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ, है। चालीस लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
TRAIN_SET = globals().get("TRAIN_SET", "v2")
TRAIN_DATA = {"v4": TRAIN_V4 + REAL_XPQA, "v2": TRAIN_V2 + REAL_XPQA, "v1": TRAIN_V1}[TRAIN_SET]
TEST = [json.loads(line) for line in r'''
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai mil jayega, packet ka 60 lagega. aap aake le jao, bhejne wala koi nahi hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi hai abhi, 500 gram wala hai chalega toh", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 gram"}}
{"product": "16A MCB", "said": "Yes we have, 220 rupees. Delivery also possible in the evening.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho aahe, 850 la padel. ghari pathavto", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "संपली आहे, सोमवारी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी है, 90 रुपये का है", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "kaun? achha tape... ruko beta ko puchta hun, baad mein call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "original khatam, local spark plug hai chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "wall putty 20 kg", "said": "aahe na, 20 kilo cha bag 780. delivery nahi, tempo karun nyava lagel", "answer": {"in_stock": true, "price_inr": 780, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Sorry sir, that size is finished. 40 mm is there.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm"}}
{"product": "2 inch paint brush", "said": "है, 45 का एक। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan hai, 70. home delivery nahi karte", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Available hai sir, full roll 1850 ka. Ghar pe bhijwa denge", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "थांबा हं, मालक जेवायला गेलेत. अर्ध्या तासाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "10W-30 nahi rakhte, 20W-40 milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20W-40"}}
{"product": "bike headlight bulb", "said": "yes available, ₹150, but pickup only", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "हो आहे, ४५० रुपये", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "nahi hai bhai, kal shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai, 3 piece bache hain, 60 ka ek. ladka chhod dega ghar pe", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Splendor ka hai na, 260. dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "We don't keep sealant. Try M-seal, we have that.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M-seal"}}
{"product": "1 inch PVC elbow", "said": "हाँ है, बीस रुपये का एक", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hello? haan bolo... seal kit? hai, 300. delivery hoti hai", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "aahe, 390. aamhi delivery deto pan 50 rupaye extra", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Let me check... yes one piece is left. 1350 rupees.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "MCB 16 amp? nahi hai, 20 amp wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20 amp"}}
{"product": "9W LED bulb", "said": "पता नहीं, गोदाम में देखना पड़ेगा। शाम को फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही, आम्ही ॲक्टिव्हाचे पार्ट ठेवत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan hai, 350 ka. 7 baje tak aa jao, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "hai, 15 rupaye. ek kaam karo, 10 roll le lo toh 12 ka laga dunga", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
HARD = [json.loads(line) for line in r'''
{"product": "TVS Jupiter tubeless tyre 90/90-12", "said": "Jupiter ka tyre ek hi tha, woh subah bik gaya, ab dukaan mein ek bhi nahi bacha. Parso tak naya maal aayega. Activa wala tyre same size ka pada hai, MRF ka, woh chalega kya?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Activa wala tyre"}}
{"product": "Godrej mortise lock 6 lever", "said": "Pehle ka rate sawa gyarah sau tha, ab company ne badha ke barah sau kar diya hai. Ek hi piece pada hai, chahiye toh abhi rakhwa lo.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Hero Splendor chain kit", "said": "Sir, I joined only last week, so the owner will tell you the price and all. He is at the bank now and will be back after 4. Please call then or tomorrow.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia screwdriver set (6 pcs)", "said": "The set is there... sorry, that is the 4 piece set. 6 piece we don't have. 4 piece is there at 180, will you take that?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "4 piece set"}}
{"product": "Maruti Alto oil filter", "said": "Alto ka oil filter ek hi piece pada hai, 180 ka. Do chahiye the toh kal tak mangwa dunga. Ladka ghar tak chhod dega, uska 40 rupaye alag se lagega.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Stanley 300 g claw hammer", "said": "Claw hammer 300 gram is there. Rs 240 only, handle is wooden. Sir, you are breaking... yes 240. And you have to collect from the shop, we do not send anything home.", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Bajaj mixer grinder carbon brush", "said": "ब्रश तो हैं, लेकिन बजाज मिक्सर वाला नहीं, फिलिप्स वाला पड़ा है, वो दे दूँ? बजाज वाला शनिवार तक आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स वाला"}}
{"product": "Havells Crabtree 16A 3-pin socket", "said": "Last month it was 85, now the new rate is 92 after GST change. Stock is there, five-six pieces.", "answer": {"in_stock": true, "price_inr": 92, "delivery": null, "alternative": null}}
{"product": "Anchor Penta 6A two-way switch", "said": "Yes sir, available only. One piece is Rs. 38/- only, but if you take the full box of 10 then 320, so 32 each.", "answer": {"in_stock": true, "price_inr": 38, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 chain kit", "said": "चेन किट है... अरे नहीं, वो ऍवेंजर वाला है, आपके काम का नहीं। पल्सर १५० का किट तो पिछले महीने से आया ही नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch cutting wheel (pack of 5)", "said": "Cutting wheel cha pack aahe, 5 cha pack 90 la. Pan mala sang, tumhi site var kiti vaparta? Roj vaparat asal tar 25 cha box ghya, to 380 la hoto, jasta parvadto.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Syska 20W LED batten", "said": "Syska ka batten hai, 260 mein. Wipro ka 310 mein milta hai, woh chahiye ho toh bata dena, par aapne Syska kaha tha na.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Royal Enfield Bullet 350 clutch plate set", "said": "Malak aata jevayla gele aahet, mala rate pan mahit nahi ani stock pan baghayla lagel. Tumhi dupari don chya sumaras call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Nylon cable tie 8 inch (pack of 100)", "said": "पाकीट मिळेल, १२० ला... नाही नाही, आता १४० झालं, काल रेट वाढला.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Distilled water 5 L can", "said": "5 litre cha can nahiye ho, 1 litre chya bottle aahet, 25 rupaye chi ek. Chalel ka tumhala?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 litre chya bottle"}}
{"product": "Crompton 1200 mm ceiling fan", "said": "Hoy, 1200 mm cha pankha ready aahe, white colour madhe. Diwali offer chalu aahe mhanun pahile 2400 la hota, aata 2150 la deto, ani ekach piece urlay.", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Fischer 8 mm plastic wall plug (pack of 100)", "said": "होय, १०० चं पाकीट आहे, ६० ला मिळतं. सुटे हवे असतील तर १ रुपयाला एक देतो.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Crompton 9 inch exhaust fan", "said": "9 inch wala toh is baar aaya hi nahi, 12 inch wala pada hai, woh bhej dun? Delivery toh hum karte hi hain, kal shaam tak ho jayegi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12 inch wala"}}
{"product": "Eveready 3 cell torch", "said": "टॉर्च आहे, पण आजच किंमत वाढली. आधी १८० ला होता, आता दोनशे दहा झाला. एक पीस पडलाय फक्त.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Luminous 150Ah inverter battery", "said": "Malik se baat karo bhai, rate wahi batate hain. Main toh bas counter pe baitha hun, mujhe stock bhi nahi pata. Kal subah 11 ke baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Amaron 35Ah car battery", "said": "Haan 35 wali hai... matlab thi, kal tak. Kal hi do piece nikal gaye, ab ek bhi nahi bacha.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SDS plus 12 mm drill bit", "said": "12 mm cha SDS bit nahiye ajun, order dila aahe. Aala ki aamhi tumchya site var gadi pathavu, delivery cha charge nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Honda Shine chain sprocket kit", "said": "Shine cha kit aala hota, pan kalach don piece gele. Aata ek pan urla nahi. Udya sakali gaadi yeil. Pathvaychi soy nahiye, tumhi yeun neta ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cera 15 mm CP angle cock", "said": "Cera चा नाहीये, जॅग्वारचा आहे, चालेल ना? त्याची किंमत ३५० आहे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "जॅग्वारचा"}}
{"product": "Maruti Swift Dzire wiper blade (pair)", "said": "Dzire cha wiper aahe, pair madhe 380 la. Pan ekach pair urlay mhanun lavkar ghya, nahitar udya sampel.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Anchor Roma 6A switch", "said": "Roma ka 6 amp wala switch? Woh toh paintalis ka aata hai, kal hi naya maal utra hai, abhi 2 dabbe pade hain counter ke neeche.", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 front brake pad", "said": "Pulsar cha pad aahe, pan to 220 cha. 150 cha nahiye amchyakade, ani 220 cha tumchya gadi la basnar nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceiling fan regulator (step type)", "said": "Step wala regulator hai... ruko, arre woh toh electronic wala hai. Step wala toh pichhle mahine hi khatam hua, electronic wala 180 mein deta hun, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "electronic wala"}}
{"product": "UltraTech cement 50 kg bag", "said": "Cement aaj hi gaadi se utra hai, 50 kilo ka bag 395 ka. 20 bag se upar lo toh 385 laga dunga. Ghar tak nahi bhejte hum, khud tempo lekar aana padega.", "answer": {"in_stock": true, "price_inr": 395, "delivery": false, "alternative": null}}
{"product": "Bajaj Platina clutch cable (original)", "said": "Platina ka clutch cable rakha hai. Local wala nabbe ka milta hai, original Bajaj ka sawa do sau mein. Aapko original chahiye na, toh woh sawa do sau.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Wipro 20W LED tube light", "said": "Tube aahe, paune donshe la ek. Choke nako lagat, direct lavta yeto.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "आज रेट नहीं बता सकता, कल बोलना। सेठ जी बैंक गए हैं और स्टॉक की लिस्ट उन्हीं के पास है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch 10 mm hammer drill machine", "said": "Bosch ka toh stock mein hai hi nahi, Stanley ka 10 mm wala pada hai, woh 2200 mein de dunga, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley ka 10 mm wala"}}
{"product": "Bosch GWS 600 angle grinder", "said": "It is not that we don't keep Bosch, grinders are always there. Just the 600 watt one got sold today, the 850 watt is there at 4200. Shall I keep that for you?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "850 watt"}}
{"product": "Cera half inch CP bib cock", "said": "Bib cock aahe, Cera cha ek doshe pannas la. Local cha pan aahe, to ekshe vees la.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Havells 4-way MCB box", "said": "४ वे चा बॉक्स संपलाय, पण आम्ही घरपोच करतो, फक्त माल आला की.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Orient 16 inch table fan blade", "said": "Blade aaj nahi milega, jo the woh sab Sunday ko nikal gaye. Naya maal Tuesday ko aayega, 3 baje tak phone kar lena.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia 8-piece spanner set", "said": "No sir, the last Taparia set went yesterday evening. New stock will come on Thursday after 11. Whenever it comes you have to collect, we do not send things home. Other brand also we don't keep.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells musical door bell", "said": "Hoy aahe... ek minute, mi chukicha bolto, to musical nahi, buzzer aahe. Musical bell nahiye, buzzer deu ka 150 la?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "buzzer"}}
{"product": "Araldite 15 g tube", "said": "अरल्डाइट? आज सुबह आख़िरी ट्यूब चली गई। कल दोपहर २ बजे के बाद नया बॉक्स आएगा, घर तक भिजवा दूँगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley 5 m measuring tape", "said": "Aahe... nahi thamba, to Freemans cha aahe, Stanley cha nahiye. Freemans cha 5 meter cha tape 120 la deu ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Freemans"}}
{"product": "Tata Ace coolant 1 L", "said": "Mi Nilesh bolto, malak kalach gavi gele. Rate te yeun sangtil. Stock baddal mala kahi mahit nahi, ya aathvadyat parat phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr. Fixit LW+ 1 litre", "said": "LW+ one litre bottle, yes. MRP is three twenty but I will give for two ninety, final. Hundred rupees extra if you want it sent to your site.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Havells 25A DP MCB", "said": "DP 25 amp ka hai sir, 540 mein. Mera number likh lo 9822334455, kuch aur chahiye toh WhatsApp kar dena.", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Hero Splendor Plus side mirror (left)", "said": "Left cha mirror aahe, right cha nahiye. Tumhala left cha pahije na? 130 la ek.", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion 4 L (white)", "said": "White 4 litre abhi bacha nahi, cream 4 litre pada hai, usse bhi kaam chal jata hai. Uska 780 padta hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "cream 4 litre"}}
{"product": "Hero Passion Pro tail lamp", "said": "टेल लैंप पड़ा है, सवा दो सौ में दे दूँगा, बल्ब भी लगा हुआ आता है। घर तक भेजने का इंतज़ाम नहीं रखते, खुद उठाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "PVC conduit pipe 20 mm (3 m piece)", "said": "ISI conduit pipe, yes. Per piece 25 rupees, if you take full bundle of 10 then 230. Delivery I will check with the driver and tell you, not sure.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Hacksaw blade 12 inch (pack of 10)", "said": "हाँ मिल जाएगा, १० वाले पैकेट के १४० रुपये ले लेता हूँ। पर अगर आप एक-दो ब्लेड लोगे तो १८ का एक पड़ेगा।", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Philips 18W LED panel light (round)", "said": "Round 18 watt cha panel aahe, 480 la. GST bill pahije asel tar 18 tax vegla lagto.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Activa 6G CVT belt", "said": "रेट आज नाही सांगता येणार, मालक बाहेर गेलेत. उद्या फोन करा, मग स्टॉक पण बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 63A 2-pole ELCB", "said": "2 pole cha ELCB aahe ho, 1450 la dein. 4 pole cha pan aahe, to 3100 la, pan to tumhala nako asel na?", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "1500W water heater element", "said": "1500 watt is finished since morning, sir. 2000 watt is there, can you use that? Our boy can drop it at your place, no extra charge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2000 watt"}}
{"product": "Royal Enfield Classic 350 air filter (original)", "said": "Original or local? We keep both. Original RE one is 450, local is 160. Original is in stock, only 2 pieces, local is plenty.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Servo 80W-90 gear oil 1 L", "said": "Gear oil? Awaaz kat rahi hai... haan ab sunai diya. Servo ka 80W-90 hai, 3 litre ka can 900 ka aata hai aur 1 litre ka 330 ka.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "ESAB 3.15 mm welding rod (1 kg packet)", "said": "3.15 cha rod aahe ho, 1 kg cha packet 190 la. Loose pan deto, 20 rod cha 70 la. Pathvaychi soy nahiye, tumchya manus la pathva.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
'''.strip().splitlines()]
FRESH = [json.loads(line) for line in r'''
{"product": "Polycab 1 sq mm flexible wire", "said": "अरे भाई, अभी दुकान पर बहुत भीड़ है। Polycab का एक MM वाला तार है या नहीं, ये देखना पड़ेगा। आप आधे घंटे बाद दोबारा कॉल कर लो, तब तक मैं गोडाउन से पता कर लूँगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ashirvad 1/2 inch CPVC elbow", "said": "आधा इंच CPVC एल्बो Ashirvad का है, एक पीस १२ रुपये का, ५० वाला डिब्बा ५२० में मिलेगा। डिलीवरी नहीं है, आकर ले जाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 12, "delivery": false, "alternative": null}}
{"product": "25 mm PVC conduit pipe", "said": "Nahi bhaiya, 25 mm conduit pipe abhi khatam hai, agle hafte aayega.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Hitachi 4 inch cutting wheel", "said": "4 inch cha cutting wheel aahe. Ek wheel 25 rupaye, 25 wheel cha box 550 la deto. Pathvaycha asel tar courier ne pathavto, 40 rupaye lagtil. Kiti pahije tumhala?", "answer": {"in_stock": true, "price_inr": 25, "delivery": true, "alternative": null}}
{"product": "Maruti Swift clutch cable", "said": "स्विफ्टची क्लच केबल आत्ता नाहीये, पुढच्या आठवड्यात मागवून देतो.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Exide 35 Ah car battery", "said": "Seth ji abhi khana khane gaye hain, battery ka stock aur rate unko hi pata hai. Aap ek ghante baad call karo, 2 baje ke baad.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 16 amp socket", "said": "Yes, Havells 16 amp socket is available, 145 rupees each.", "answer": {"in_stock": true, "price_inr": 145, "delivery": null, "alternative": null}}
{"product": "Nerolac Excel exterior paint 20 litre", "said": "Nerolac Excel twenty litre, let me check with the godown sir. We had some, but painters took a lot this week. I will confirm and send you a WhatsApp in half an hour.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Luminous 900 VA inverter", "said": "Luminous 900 VA inverter is there sir. MRP is 7200, but we are giving it for 6400 including GST. Delivery and installation we will do free of cost.", "answer": {"in_stock": true, "price_inr": 6400, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Fevicol SH ek kilo wala abhi nahi hai. Aadha kilo ke dabbe hain, dhai sau ka ek, wo chahiye to bata do.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Fevicol SH 500 g"}}
{"product": "Havells 16 amp single pole MCB", "said": "हाँ जी, Havells का १६ एम्पियर MCB है, १९० रुपये का एक।", "answer": {"in_stock": true, "price_inr": 190, "delivery": null, "alternative": null}}
{"product": "Wipro 18 watt round LED panel", "said": "18 watt cha Wipro panel ata nahiye, company kadun maal aala nahi don athavde zale.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "MRF Nylogrip 2.75-18 tyre", "said": "Malak baher gelet, tyre cha stock tech baghtat. Sandhyakali saha nantar phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 clutch plate", "said": "Nahi ho, Pulsar chi clutch plate sampli aahe, udya yeil.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sintex 500 litre water tank", "said": "हो, सिंटेक्सची ५०० लिटरची टाकी आहे, साडेतीन हजार. घरपोच करतो.", "answer": {"in_stock": true, "price_inr": 3500, "delivery": true, "alternative": null}}
{"product": "Crompton 15 litre geyser", "said": "Crompton cha 15 litre geyser aahe, 6500... thamba thamba, chukla, to 25 litre cha rate hota. 15 litre cha 5800 aahe. Fitting sathi manus pathavto, delivery free aahe.", "answer": {"in_stock": true, "price_inr": 5800, "delivery": true, "alternative": null}}
{"product": "Finolex 1.5 sq mm wire coil", "said": "Finolex 1.5 ka coil abhi nahi hai ji.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Orient Apex 1200 mm ceiling fan", "said": "Orient Apex पंखा है। डिब्बे पर MRP तो दो हज़ार लिखा है, लेकिन आपको पौने सत्रह सौ में दे देंगे।", "answer": {"in_stock": true, "price_inr": 1675, "delivery": null, "alternative": null}}
{"product": "Bosch 13 mm impact drill", "said": "Bosch ka 13 mm drill abhi stock mein nahi hai, order karoge to 4 din lagega. Uski jagah Dewalt ka hai 3200 ka, wo le lo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Dewalt 13 mm drill"}}
{"product": "Syska 12 watt LED bulb", "said": "Syska cha 12 watt bulb aahe na, MRP 199 aahe pan aamhi 150 la deto. Don ghetle tar 280 la deto.", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "Polycab 4 sq mm wire coil", "said": "4 sq mm Polycab ka maal nahi hai bhaiya, itna mota wire hum rakhte hi nahi. Kasba Peth mein try karo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bajaj 1200 mm ceiling fan", "said": "Bajaj ka 1200 mm wala fan hai na, MRP to 2450 likha hai dabbe pe, lekin aapko 2100 mein de dunga. Fitting ka mistri alag se lagega.", "answer": {"in_stock": true, "price_inr": 2100, "delivery": null, "alternative": null}}
{"product": "Legrand 25 amp RCCB", "said": "लेग्रँडचा २५ अँपिअरचा RCCB संपलाय. हॅवेल्सचा आहे, तो अठराशेला पडेल.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells 25 amp RCCB"}}
{"product": "Philips 20 watt LED batten", "said": "20 watt LED batten Philips ka hai. Single piece saade teen sau, 4 ka pack 1300 mein. Delivery chahiye to Kothrud tak 50 rupaye extra lagenge.", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "Anchor 32 amp DP MCB", "said": "Anchor cha 32 amp DP nahiye. Legrand cha aahe tyach rating cha, to chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Legrand 32 amp DP MCB"}}
{"product": "Amaron 5 Ah bike battery", "said": "Ho, Amaron chi 5 Ah battery aahe, solashe pannas la. Ghari yeun basvun deto, Hadapsar madhe asal tar tyache vegle 100 rupaye lagtil.", "answer": {"in_stock": true, "price_inr": 1650, "delivery": true, "alternative": null}}
{"product": "Honda Activa air filter", "said": "Activa cha air filter aahe, 180 rupaye. Delivery nahi karat aamhi, tumhala dukanat yava lagel.", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "TVS Apache rear brake pad", "said": "अपाचे के पीछे वाले ब्रेक पैड ख़त्म हो गए हैं, नया माल सोमवार को आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Jaquar pillar cock", "said": "Jaquar ka pillar cock... ek minute, godown mein dekhna padega. Main check karke aapko wapas call karta hoon.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion white 10 litre", "said": "Tractor emulsion white 10 litre ka dabba hai, 2350 lagega. Delivery hum nahi karte bhai, aapko khud aake le jaana padega, dukaan 10 se 8 khuli hai.", "answer": {"in_stock": true, "price_inr": 2350, "delivery": false, "alternative": null}}
{"product": "Ambuja cement 50 kg bag", "said": "Ambuja ka bag hai. Pehle 380 tha, ab 395 ho gaya hai, GST mila ke. Tempo se bhej denge, 10 bag se upar delivery free hai.", "answer": {"in_stock": true, "price_inr": 395, "delivery": true, "alternative": null}}
{"product": "Usha 400 mm table fan", "said": "Usha ka table fan khatam ho gaya. Crompton ka 400 mm wala hai, wo chalega to bolo.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Crompton 400 mm table fan"}}
{"product": "Kirloskar 0.5 HP water pump", "said": "किर्लोस्करचा अर्धा HP पंप आहे. आधी ४२०० होता, आता ४५०० झालाय. डिलिव्हरी नाही, दुकानातून घेऊन जावा लागेल.", "answer": {"in_stock": true, "price_inr": 4500, "delivery": false, "alternative": null}}
{"product": "Exide 150 Ah inverter battery", "said": "सेठ जी अभी बैंक गए हैं, बैटरी का रेट वही बताएँगे। आप शाम ५ बजे के बाद फ़ोन कीजिए।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Royal Enfield Classic 350 clutch lever", "said": "मालक आज गावाला गेलेत, बुलेटचे पार्ट्स तेच बघतात. उद्या सकाळी दहा नंतर फोन करा.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Supreme 4 inch PVC pipe", "said": "४ इंच का Supreme पाइप अभी नहीं है, Finolex का है उसी साइज़ में, वो चलेगा तो बताइए।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Finolex 4 inch PVC pipe"}}
{"product": "Taparia 8 inch combination plier", "said": "टपारियाची ८ इंची पकड आहे. छापील किंमत २२० आहे, पण आम्ही तुम्हाला पावणे दोनशेला देतो.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Supreme 1 inch PVC elbow", "said": "Ho, 1 inch cha elbow aahe, 18 rupaye la ek.", "answer": {"in_stock": true, "price_inr": 18, "delivery": null, "alternative": null}}
{"product": "Jindal 1 inch GI pipe", "said": "एक इंची GI पाइप जिंदालचा... थांबा, गोदामात बघावं लागेल. मी तुम्हाला परत फोन करतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 9 watt LED bulb", "said": "9 watt Philips hai, 110 ka... nahi nahi sorry, rate badh gaya hai, ab 120 ka hai. Ghar pe bhijwa denge, 20 rupaye delivery lagega.", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "Bajaj 3 litre instant geyser", "said": "बजाज का ३ लीटर गीज़र नहीं है। Racold का ३ लीटर वाला है, वो साढ़े तीन हज़ार का पड़ेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Racold 3 litre instant geyser"}}
{"product": "Anchor Roma 6 amp switch", "said": "Haan ji, Anchor Roma 6 amp switch hai, 32 rupaye ka ek.", "answer": {"in_stock": true, "price_inr": 32, "delivery": null, "alternative": null}}
{"product": "Godrej 7 lever padlock", "said": "Main to yahan naya hoon bhaiya. Godrej ke taale wala counter malik dekhte hain, wo shaadi mein gaye hain, kal subah 11 baje ke baad phone karna, wo rate bhi bata denge.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Roff tile adhesive 20 kg bag", "said": "Roff chi bag... ek minute, godown madhe aahe ka baghun sangto. Tumhi daha minitani parat phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Legrand Myrius 6 amp 2 way switch", "said": "See, Legrand Myrius 2 way switch 6 amp we have. Single piece is 78, a box of 20 I can give you at 1400. No home delivery sir, you will have to collect it from the shop, we are near Swargate.", "answer": {"in_stock": true, "price_inr": 78, "delivery": false, "alternative": null}}
{"product": "15 mm brass bib cock", "said": "Pitalacha 15 mm bib cock sampla aahe. Stainless steel cha aahe tyach size cha, tyachi kimmat 350.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "stainless steel 15 mm bib cock"}}
{"product": "Dr. Fixit LW+ 1 litre", "said": "डॉ. फिक्सिट LW+ एक लीटर वाला है। पहले २१० का था, अब २३५ का हो गया है। घर पर भिजवा देंगे, कोई चार्ज नहीं।", "answer": {"in_stock": true, "price_inr": 235, "delivery": true, "alternative": null}}
{"product": "Stanley 5 m measuring tape", "said": "Stanley का ५ मीटर टेप है, ३२० का... नहीं, माफ़ कीजिए, ३५० का है, नया रेट आया है। भेजना हो तो ३० रुपये डिलीवरी लगेगी।", "answer": {"in_stock": true, "price_inr": 350, "delivery": true, "alternative": null}}
{"product": "3M double sided foam tape 24 mm", "said": "The 24 mm 3M foam tape is finished sir. Only the 12 mm roll is there, if that works I can keep one aside for you.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3M foam tape 12 mm"}}
{"product": "Asian Paints Apcolite enamel 1 litre", "said": "Apcolite enamel ek litre aahe. Junya stock cha 420 hota, navin stock aala to 460 cha aahe, ata fakt navinch shillak aahe.", "answer": {"in_stock": true, "price_inr": 460, "delivery": null, "alternative": null}}
{"product": "Bosch spark plug for Maruti Alto", "said": "Sir, actually the person who handles spares is not here today, I am only sitting at the counter. Please call tomorrow after 11, he will tell you.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Jaquar 2 in 1 wall mixer", "said": "Jaquar wall mixer 2 in 1 is available. Price is 3450... sorry, 3650, I was looking at the old list. We can send it with our boy, 80 rupees delivery charge.", "answer": {"in_stock": true, "price_inr": 3650, "delivery": true, "alternative": null}}
{"product": "Motul 7100 10W-50 1 litre", "said": "No sir, Motul 7100 is out of stock since last week, the distributor has not supplied yet.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Castrol Activ 4T 20W-40 1 litre", "said": "Castrol Activ 20W-40 ek litre wala hai, chaar sau pachaas ka ek bottle. Teen bottle pade hain abhi, dukaan raat 9 baje tak khuli hai.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "M8 nut bolt", "said": "M8 ka nut bolt hai, ek piece 6 rupaye ka. Poora dabba 100 piece ka saade chaar sau mein de dunga agar zyada chahiye.", "answer": {"in_stock": true, "price_inr": 6, "delivery": null, "alternative": null}}
{"product": "Taparia 6 piece screwdriver set", "said": "Taparia screwdriver set, I am not sure if it is there, my brother keeps those in the cupboard and he has the key. He will be back by 6, please call then.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cera wash basin", "said": "सेराचं पांढरं वॉश बेसिन आहे, चौदाशेला एक. दोन घेतलेत तर सत्ताविसशेला देईन. घरपोच पाठवतो, कोथरूडला शंभर रुपये लागतील.", "answer": {"in_stock": true, "price_inr": 1400, "delivery": true, "alternative": null}}
{"product": "8 inch tower bolt", "said": "8 inch tower bolt nahiye, 6 inch cha aahe, to deu ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "6 inch tower bolt"}}
{"product": "Birla White wall putty 20 kg", "said": "Birla White putty chi 20 kilo chi bag aahe, saadesatshe la padel. Gharpoch karto aamhi, Warje paryant kahi charge nahi.", "answer": {"in_stock": true, "price_inr": 750, "delivery": true, "alternative": null}}
{"product": "Hero Splendor original brake shoe", "said": "Splendor ka brake shoe? Haan rakha hai. Local wala bhi hai 150 ka, par aapko original chahiye na, wo Hero ki genuine packing mein hai, 280 ka padega.", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
ASR = [json.loads(line) for line in r'''
{"product": "Havells 16A MCB single pole", "said": "हाँ जी है हैवल्स का सोलह एम्पीयर वाला एमसीबी एक का दो सौ अस्सी रुपय पड़ेगा डिलिवरी हो जाएगी शाम तक", "answer": {"in_stock": true, "price_inr": 280, "delivery": true, "alternative": null}}
{"product": "Finolex 1 inch PVC pipe", "said": "अं फिनोलेक्स का एक इंच पाइप अभी खतम है सुप्रीम का एक इंच पाइप है वो चलेगा तो बताओ", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सुप्रीम का एक इंच पाइप"}}
{"product": "Crompton 1200 mm ceiling fan", "said": "क्रॉम्टन का बारह सौ एमएम पंखा है जी पौने दो हजार का है लेकिन आके ले जाओ डिलीवरी नहीं करते", "answer": {"in_stock": true, "price_inr": 1750, "delivery": false, "alternative": null}}
{"product": "Ceiling fan capacitor 2.5 mfd", "said": "हां कैपेस्टर ढाई एमएफडी वाला है पैंतालिस रुपए का लड़का घर दे जाएगा", "answer": {"in_stock": true, "price_inr": 45, "delivery": true, "alternative": null}}
{"product": "Honda Activa rear brake shoe", "said": "एक्टिवा का ब्रेक शू है ओरिजिनल होंडा वाला साढ़े तीन सौ का", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "Hero Splendor clutch plate", "said": "स्प्लेंडर की क्लच प्लेट अभी नहीं है कल परसों आएगा माल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Birla White wall putty 40 kg", "said": "बिरला वाइट पुट्टी चालीस किलो एम आर पी नौ सौ बीस है आपको सवा आठ सौ में दे देंगे गाड़ी से भेज देंगे", "answer": {"in_stock": true, "price_inr": 825, "delivery": true, "alternative": null}}
{"product": "Philips 20W LED batten", "said": "फिलिप्स का बीस वाट बैटन नहीं है अभी विप्रो का बीस वाट बैटन है दो सौ दस का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "विप्रो का बीस वाट बैटन"}}
{"product": "1/2 inch brass ball valve", "said": "आधा इंच वाला बॉलवाल पीतल का है एक सौ साठ नहीं नहीं एक सौ अस्सी का है", "answer": {"in_stock": true, "price_inr": 180, "delivery": null, "alternative": null}}
{"product": "Bosch 13 mm drill machine", "said": "बॉश का ड्रिल मशीन तेरह एम एम वाला है जी 3400 का पड़ेगा होम डिलीवरी फ्री है", "answer": {"in_stock": true, "price_inr": 3400, "delivery": true, "alternative": null}}
{"product": "Anchor 4 socket extension board", "said": "देखो ऐंकर का चार सॉकेट वाला एक्सटेंशन बोर्ड है साढ़े चार सौ का दुकान पे आके ले लो", "answer": {"in_stock": true, "price_inr": 450, "delivery": false, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion 20 L", "said": "एशियन पेंट्स ट्रैक्टर इमल्शन बीस लीटर अं मैं चेक करके बताता हूँ थोड़ी देर में कॉल करता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "UltraTech cement bag 50 kg", "said": "अल्ट्राटेक सीमेंट है जी एक बैग का तीन सौ अस्सी रूपए है गाड़ी से भिजवा देंगे साइट पे", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "Roff tile adhesive 20 kg", "said": "रॉफ का टाइल एडहेसिव बीस किलो खतम हो गया मायकेम का बीस किलो टाइल एडहेसिव है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मायकेम का बीस किलो टाइल एडहेसिव"}}
{"product": "Legrand 32A MCB double pole", "said": "हेलो कौन बोल रहा है ये तो मेडिकल स्टोर है रॉन्ग नंबर है भाई", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Polycab 1.5 sq mm wire, 90 m coil", "said": "पोलिकैब डेढ़ एमएम का बंडल है नब्बे मीटर का एक बंडल ग्यारह सौ पचास का", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Anchor Roma 6A switch", "said": "एंकर रोमा का छः एम्पियर स्विच है डिब्बा बीस पीस का पांच सौ का है एक पीस पच्चीस रुपये", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Havells 9W LED bulb, box of 10", "said": "हैवल्स नौ वाट बल्ब का डब्बा दस पीस वाला है एक का अस्सी रूपए डब्बा सात सौ पचास का पड़ेगा", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "Honda Activa air filter", "said": "एक्टिवा का एयर फिल्टर हैं जी एक सौ बीस का पर डिलीवरी नहीं होगी खुद आके लेजाना", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "Philips 12V 35W headlight bulb for Splendor", "said": "फिलिप्स वाला हेडलाइट बल्ब खतम है ओसराम का बारह वोल्ट पैंतीस वाट बल्ब है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ओसराम का बारह वोल्ट पैंतीस वाट बल्ब"}}
{"product": "Supreme CPVC 3/4 inch pipe, 3 m length", "said": "सुप्रीम सी पी वी सी पौन इंच पाइप है तीन मीटर लेंथ का पौने चार सौ का", "answer": {"in_stock": true, "price_inr": 375, "delivery": null, "alternative": null}}
{"product": "Usha 1200 mm ceiling fan", "said": "उषा के पंखे देखने पड़ेंगे स्टॉक में है या नहीं दोपहर में बताता हूं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Jaquar 1 inch ball valve", "said": "जैगुआर का नहीं रखते हम ज़ोलोटो का एक इंच बॉल वाल्व है चार सौ बीस का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ज़ोलोटो का एक इंच बॉल वाल्व"}}
{"product": "Fevicol SH 1 kg", "said": "फेविकोल एस एच एक किलो है जी तीन सौ दस एमआरपी है तीन सौ में दे दूंगा", "answer": {"in_stock": true, "price_inr": 300, "delivery": null, "alternative": null}}
{"product": "Taparia screwdriver set", "said": "टपरिया का स्क्रू ड्राइवर सेट है सवा पाँच सौ का लड़का शाम को घर पहुंचा देगा", "answer": {"in_stock": true, "price_inr": 525, "delivery": true, "alternative": null}}
{"product": "Exide battery for Hero Splendor", "said": "एक्साइड की स्प्लेंडर वाली बैटरी है उन्नीस सौ की आके ले जाओ यहीं फिट कर देंगे", "answer": {"in_stock": true, "price_inr": 1900, "delivery": false, "alternative": null}}
{"product": "Berger wall primer 4 L", "said": "बर्जर का वॉल प्राइमर चार लीटर नहीं है अभी एशियन का चार लीटर प्राइमर है पौने छह सौ का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एशियन का चार लीटर प्राइमर"}}
{"product": "Ashirvad 1 inch PVC elbow", "said": "आशिर्वाद का एक इंच एल्बो है बारह रूपए पीस सौ पीस का डिब्बा ग्यारह सौ", "answer": {"in_stock": true, "price_inr": 12, "delivery": null, "alternative": null}}
{"product": "V-Guard voltage stabilizer for fridge", "said": "वी गार्ड का फ्रिज वाला स्टेबलाइजर है दो हज़ार दो सौ नहीं रुको दो हज़ार चार सौ का है डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 2400, "delivery": true, "alternative": null}}
{"product": "Hero Splendor chain sprocket kit", "said": "हाँ जी चेन स्प्रॉकेट किट स्प्लेंडर का है साडे आठ सौ रुपिया", "answer": {"in_stock": true, "price_inr": 850, "delivery": null, "alternative": null}}
{"product": "Havells 32A MCB", "said": "हो आहे हॅवेल्सचा बत्तीस अँपियर एमसीबी साडेचारशे रुपये पोरगा घरी आणून देईल", "answer": {"in_stock": true, "price_inr": 450, "delivery": true, "alternative": null}}
{"product": "Finolex 4 inch PVC pipe", "said": "फिनोलेक्सचा चार इंच पाईप संपलाय सुप्रीमचा चार इंच पाईप आहे चालेल का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सुप्रीमचा चार इंच पाईप"}}
{"product": "Bajaj 1200 mm ceiling fan", "said": "बजाजचा पंखा आहे सतराशे पन्नास रुपये पण डिलेवरी नाही दुकानात येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 1750, "delivery": false, "alternative": null}}
{"product": "ACC cement bag 50 kg", "said": "ए सी सी सिमेंट आहे एक गोणी तीनशे नव्वद गाडी पाठवतो साईटवर", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "Honda Activa clutch shoe", "said": "हा ऍक्टिवाचे क्लच शू आहेत सव्वापाचशे", "answer": {"in_stock": true, "price_inr": 525, "delivery": null, "alternative": null}}
{"product": "JK wall putty 40 kg", "said": "जेके पुट्टी नाही आत्ता बिर्लाची चाळीस किलो पुट्टी आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बिर्लाची चाळीस किलो पुट्टी"}}
{"product": "Crompton 15W LED batten", "said": "मालक बाहेर गेलेत संध्याकाळी फोन करा ते सांगतील", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fan capacitor 3.15 mfd", "said": "तीन पॉईंट पंधरा चा कपॅसिटर आहे पंचावन्न रुपये", "answer": {"in_stock": true, "price_inr": 55, "delivery": null, "alternative": null}}
{"product": "DeWalt 13 mm drill machine", "said": "डिवॉल्टचं ड्रिल मशीन आहे तीन हजार आठशे नाही नाही तीन हजार सहाशे", "answer": {"in_stock": true, "price_inr": 3600, "delivery": null, "alternative": null}}
{"product": "3/4 inch brass ball valve", "said": "पाऊण इंचाचा बॉल व्हॉल्व आहे दोनशे वीस एमआरपी आहे पण तुम्हाला पावणेदोनशे ला देतो", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Laticrete tile adhesive 20 kg", "said": "लॅटिक्रीट टाइल ॲडेसिव्ह आहे वीस किलोची बॅग पाचशे साठ घरी पोचवून देऊ", "answer": {"in_stock": true, "price_inr": 560, "delivery": true, "alternative": null}}
{"product": "Hero Splendor brake shoe", "said": "स्प्लेंडरचे ब्रेक शू आहेत दीडशे रुपये येऊन घेऊन जा", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "Goldmedal extension board", "said": "गोल्डमेडलचा एक्स्टेंशन बोर्ड संपला अँकरचा चार सॉकेटचा बोर्ड आहे तीनशे ऐंशी", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अँकरचा चार सॉकेटचा बोर्ड"}}
{"product": "Asian Paints Apex 10 L", "said": "एशियन एपेक्स दहा लिटर बघून सांगतो थोड्या वेळाने फोन करतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Philips 20W LED batten, box of 10", "said": "फिलिप्स वीस वॅट बॅटन आहे एक दोनशे चाळीसला दहाचा बॉक्स बावीसशे", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "Legrand 6A MCB", "said": "हाँ Legrand का 6 amp MCB है 210 रुपये एक का delivery कल हो जाएगी", "answer": {"in_stock": true, "price_inr": 210, "delivery": true, "alternative": null}}
{"product": "Polycab 2.5 sq mm wire coil", "said": "Polycab का ढाई mm वाला coil नहीं है Havells का ढाई mm coil है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Havells का ढाई mm coil"}}
{"product": "Orient 1200 mm ceiling fan", "said": "Orient का 1200 mm fan है 1850 का MRP 2100 है delivery नहीं है shop पे आके ले जाओ", "answer": {"in_stock": true, "price_inr": 1850, "delivery": false, "alternative": null}}
{"product": "Wipro 20W LED batten", "said": "Wipro चा 20 watt batten आहे 190 रुपये पोरगा delivery करेल", "answer": {"in_stock": true, "price_inr": 190, "delivery": true, "alternative": null}}
{"product": "NGK spark plug for Splendor", "said": "Splendor का spark plug है NGK वाला पचासी नहीं पंचानवे रुपये", "answer": {"in_stock": true, "price_inr": 95, "delivery": null, "alternative": null}}
{"product": "Birla tile adhesive 20 kg", "said": "अं हाँ जी tile adhesive Birla का है 20 kg bag 480 का boy घर पे दे जाएगा", "answer": {"in_stock": true, "price_inr": 480, "delivery": true, "alternative": null}}
{"product": "Cera 1/2 inch ball valve", "said": "check करके बताता हूं stock में है या नहीं call back करता हूं", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Hitachi 10 mm drill machine", "said": "Hitachi वाला नहीं है अभी Bosch का 10 mm drill machine है 2800 का", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Bosch का 10 mm drill machine"}}
{"product": "Nerolac Excel exterior paint 20 L", "said": "Nerolac Excel वीस लिटर आहे 7400 MRP आहे 6900 ला देतो", "answer": {"in_stock": true, "price_inr": 6900, "delivery": null, "alternative": null}}
{"product": "Exide inverter battery 150 Ah", "said": "Exide का 150 Ah battery है ग्यारह हज़ार पांच सौ installation के साथ delivery free", "answer": {"in_stock": true, "price_inr": 11500, "delivery": true, "alternative": null}}
{"product": "Anchor 16A socket", "said": "haan ji anchor ka 16 amp socket hai ek ka 95 rupay ladka ghar de jayega", "answer": {"in_stock": true, "price_inr": 95, "delivery": true, "alternative": null}}
{"product": "Astral 1 inch PVC pipe", "said": "astral nahi hai bhai supreme ka 1 inch pipe hai chalega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "supreme ka 1 inch pipe"}}
{"product": "Hero Splendor rear shock absorber (one piece)", "said": "splendor ka shocker hai pair ka 1400 ek ka sade saat sau", "answer": {"in_stock": true, "price_inr": 750, "delivery": null, "alternative": null}}
{"product": "JK Super cement bag", "said": "jk super cement khatam ho gaya kal maal aayega kal phone karna", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 25A MCB", "said": "dekho havells ka 25 amp mcb hai saade teen sau ka aake le jao delivery nahi hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
'''.strip().splitlines()]
SETS = {'test': TEST, 'hard': HARD, **({'fresh': FRESH} if FRESH else {}), **({'asr': ASR} if ASR else {})}
print(TRAIN_SET, len(TRAIN_DATA), "training rows;", {k: len(v) for k, v in SETS.items()}, "test calls")

In [ ]:
#@title 5. Load the base model (about 2–3 minutes)
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import importlib.metadata as md
import torch

print({p: md.version(p) for p in ("unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft")})

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=1024, load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
# LoRA: train a small add-on (about 1% of the weights) instead of the whole model.
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
#@title 6. Test the model BEFORE training
def ask(product, said):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(product, said)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_tests(label):
    FastLanguageModel.for_inference(model)
    try:
        results = evaluate(ask, SETS)
    finally:
        FastLanguageModel.for_training(model)
    show(label, results)
    return results


before = run_tests("BEFORE training")

In [ ]:
#@title 7. Train (about 10–15 minutes; v4: about 1 hour)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer


def to_text(ex):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(ex["product"], ex["said"])},
                {"role": "assistant", "content": answer_text(ex["answer"])}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


dataset = Dataset.from_list([to_text(ex) for ex in TRAIN_DATA])
print(dataset[0]["text"])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", max_length=1024,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear",
        optim="adamw_8bit", weight_decay=0.01, logging_steps=10, seed=3407,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        output_dir="outputs", report_to="none",
    ),
)
# Learn only the answers, not the questions.
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime'] / 60:.1f} min, loss {stats.metrics['train_loss']:.3f}")

In [ ]:
#@title 8. Test the model AFTER training
after = run_tests("AFTER training")
payload = results_payload(MODEL_NAME, EPOCHS, before, after)
payload.update(train_set=TRAIN_SET, train_rows=len(TRAIN_DATA))
print("\n== Before vs after (all 4 fields right)")
for name, res in payload["sets"].items():
    print(f"   {name:5s} {res['before']['all_correct']:4.0%} -> {res['after']['all_correct']:4.0%}  ({res['examples']} calls)")
print("\nRESULTS_JSON", json.dumps({k: {n: r[k] for n, r in payload["sets"].items()} for k in ("before", "after")}))
if ON_KAGGLE:  # run_on_kaggle.py downloads this
    with open("/kaggle/working/results.json", "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=1)

In [ ]:
#@title 9. Try your own sentence
product = "Activa clutch plate"  #@param {type:"string"}
said = "haan hai, 650 ka padega, ghar pe bhej denge"  #@param {type:"string"}
FastLanguageModel.for_inference(model)
print(ask(product, said))

In [ ]:
#@title 10. Save your model: Google Drive on Colab, the Output tab on Kaggle (the Ollama file takes 10–15 minutes)
import glob, re

lora_dir = f"{WORK}/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copytree(lora_dir, f"{SAVE_DIR}/lora", dirs_exist_ok=True)
    print("Saved the LoRA add-on to", f"{SAVE_DIR}/lora")

if MAKE_OLLAMA_FILE:
    gguf_dir = f"{WORK}/{OUTPUT_NAME}"
    result = model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
    files = (result or {}).get("gguf_files") if isinstance(result, dict) else None
    files = files or glob.glob(f"{gguf_dir}*/**/*.gguf", recursive=True)
    if not files:
        raise RuntimeError("No .gguf file was made. The LoRA add-on is still saved; send Claude the error above.")
    q4 = [f for f in files if "q4_k_m" in os.path.basename(f).lower()] or sorted(files, key=os.path.getsize)
    gguf = q4[0]
    name = f"{OUTPUT_NAME}.Q4_K_M.gguf"

    modelfile = make_modelfile(name)
    modelfile_path = os.path.join(os.path.dirname(gguf), "Modelfile.wimb")  # gguf_dir itself may be cleaned up
    with open(modelfile_path, "w", encoding="utf-8") as fh:
        fh.write(modelfile)
    print(modelfile)

    if SAVE_DIR:
        shutil.copy(gguf, f"{SAVE_DIR}/{name}")
        shutil.copy(modelfile_path, f"{SAVE_DIR}/Modelfile")

if SAVE_DIR:
    for path in sorted(glob.glob(f"{SAVE_DIR}/*")):
        size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else 0
        print(f"   {path}  {size:,.0f} MB" if size else f"   {path}/")

In [ ]:
#@title 11. Test the exported GGUF file itself (CPU, up to 25 minutes; the answers Ollama would give)
import socket, subprocess, time, urllib.request

gguf_report = {"error": "not run"}
server = None
try:
    if not MAKE_OLLAMA_FILE:
        raise RuntimeError("MAKE_OLLAMA_FILE is off, so there is no GGUF file to test")
    binary = subprocess.run(
        ["bash", "-c", "find / -xdev -name llama-server -type f -perm -u+x 2>/dev/null | head -1"],
        capture_output=True, text=True).stdout.strip()
    if not binary:
        raise RuntimeError("llama-server was not found (Unsloth builds it while making the GGUF)")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    server = subprocess.Popen([binary, "-m", gguf, "--port", str(port), "--host", "127.0.0.1", "-c", "2048",
                               "-t", str(os.cpu_count() or 2), "--jinja"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    base = f"http://127.0.0.1:{port}"
    for _ in range(120):  # up to 6 minutes to load the model
        if server.poll() is not None:
            raise RuntimeError(f"llama-server stopped at start-up (exit code {server.returncode})")
        try:
            urllib.request.urlopen(base + "/health", timeout=5).read()
            break
        except Exception:
            time.sleep(3)
    else:
        raise RuntimeError("llama-server did not become ready")

    def ask_gguf(product, said):
        body = json.dumps({"messages": [{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": user_message(product, said)}],
                           "temperature": 0, "max_tokens": 96}).encode()
        req = urllib.request.Request(base + "/v1/chat/completions", body, {"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            return json.load(resp)["choices"][0]["message"]["content"].strip()

    started = time.time()
    gguf_results = evaluate(ask_gguf, SETS, deadline=started + 25 * 60)
    show("GGUF Q4_K_M", gguf_results)
    gguf_report = {"gguf": os.path.basename(gguf), "seconds": round(time.time() - started),
                   "sets": {n: {"examples": r["examples"], "of": r["of"],
                                **({k: round(r[k], 3) for k in KEYS} if r["examples"] else {}),
                                "mistakes": r["mistakes"], "replies": r["replies"]}
                            for n, r in gguf_results.items()}}
except Exception as e:  # the main results are already saved; this test must never fail the run
    print("GGUF test failed:", repr(e))
    gguf_report = {"error": repr(e)}
finally:
    if server is not None:
        server.terminate()
if ON_KAGGLE:
    with open("/kaggle/working/results_gguf.json", "w", encoding="utf-8") as fh:
        json.dump(gguf_report, fh, ensure_ascii=False, indent=1)

## Done. Next steps

**1. Save this notebook to GitHub** so Claude can read the results:
**File → Save a copy in GitHub** → `pratik9yadav/yok`, branch `main`,
path `train_colab.ipynb`.

**2. Run your model on your PC**
1. Install Ollama from ollama.com.
2. From Google Drive, download the `wimb-shop-model` folder (the `.gguf` file and `Modelfile`).
3. Open a terminal in that folder and run:
```
ollama create wimb-shop -f Modelfile
ollama run wimb-shop "Product: 16A MCB
Shopkeeper: haan hai, 220 ka, delivery nahi"
```